In [1]:

import os
import sys
from pathlib import Path

# Check the operating system
if os.name != "nt":
    raise OSError("This project is configured for Windows.")

# Check the project drive
drive = Path("D:/")

if not drive.is_dir():
    raise FileNotFoundError("D: drive was not found.")

# Define the project directory
PROJECT_DIR = drive / "NST_NSCLC_PPI_GAT"

# Define the project structure
folders = [
    "notebooks",
    "src",
    "configs",
    "data/raw/TCGA_LUAD",
    "data/raw/TCGA_LUSC",
    "data/raw/STRING",
    "data/raw/GEO",
    "data/interim",
    "data/processed",
    "data/splits",
    "models/checkpoints",
    "results/metrics",
    "results/figures",
    "results/biomarkers",
    "results/validation",
    "logs"
]

# Create the required directories
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

for folder in folders:
    (PROJECT_DIR / folder).mkdir(
        parents=True,
        exist_ok=True
    )

# Verify the directory structure
print("Project directory:", PROJECT_DIR)
print("Python version:", sys.version.split()[0])
print("Directories created:", len(folders))
print("\nProject setup completed successfully.")

Project directory: D:\NST_NSCLC_PPI_GAT
Python version: 3.14.6
Directories created: 16

Project setup completed successfully.


In [2]:

import os
import sys
import platform
import shutil
import subprocess
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

# Project directory
PROJECT_DIR = Path(r"D:\NST_NSCLC_PPI_GAT")

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}")

# Check installed package versions
def package_version(package_name):
    try:
        return version(package_name)
    except PackageNotFoundError:
        return "Not installed"

# System information
print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)

print("Operating System :", platform.platform())
print("Python Version   :", platform.python_version())
print("Python Path      :", sys.executable)
print("Architecture     :", platform.machine())
print("CPU Cores        :", os.cpu_count())

# Available disk space
disk = shutil.disk_usage(PROJECT_DIR)
free_gb = disk.free / (1024 ** 3)

print(f"D: Free Space    : {free_gb:.2f} GB")

# Installed library versions
print("\n" + "=" * 60)
print("INSTALLED LIBRARIES")
print("=" * 60)

packages = [
    "pip",
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "torch",
    "torch-geometric",
    "matplotlib",
    "jupyterlab",
    "ipykernel"
]

for package in packages:
    print(f"{package:<20}: {package_version(package)}")

# NVIDIA GPU information
print("\n" + "=" * 60)
print("GPU INFORMATION")
print("=" * 60)

nvidia_smi = shutil.which("nvidia-smi")

if nvidia_smi:
    command = [
        nvidia_smi,
        "--query-gpu=name,memory.total,driver_version",
        "--format=csv,noheader,nounits"
    ]

    try:
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=20,
            check=False
        )

        if result.returncode == 0:
            print("GPU Name, Memory (MiB), Driver Version:")
            print(result.stdout.strip())
        else:
            print("NVIDIA GPU query failed.")
            print(result.stderr.strip())

    except (OSError, subprocess.TimeoutExpired) as error:
        print(f"GPU detection error: {error}")
else:
    print("NVIDIA-SMI not found.")
    print("GPU availability requires further verification.")

print("\nEnvironment inspection completed.")

SYSTEM INFORMATION
Operating System : Windows-11-10.0.26300-SP0
Python Version   : 3.14.6
Python Path      : D:\Anaconda\python.exe
Architecture     : AMD64
CPU Cores        : 4
D: Free Space    : 163.26 GB

INSTALLED LIBRARIES
pip                 : 26.1.2
numpy               : 2.4.6
pandas              : 3.0.3
scipy               : 1.18.0
scikit-learn        : 1.9.0
torch               : 2.14.0+cpu
torch-geometric     : Not installed
matplotlib          : 3.11.0
jupyterlab          : 4.5.9
ipykernel           : 6.31.0

GPU INFORMATION
NVIDIA-SMI not found.
GPU availability requires further verification.

Environment inspection completed.


In [3]:

import sys
import shutil
import subprocess
from pathlib import Path

# Project configuration
PROJECT_DIR = Path(r"D:\NST_NSCLC_PPI_GAT")
ENV_DIR = PROJECT_DIR / "envs" / "nsclc"
KERNEL_NAME = "nst_nsclc"
KERNEL_DISPLAY_NAME = "NST NSCLC Research (Python 3.12)"

# Locate the Conda executable
conda_candidates = [
    Path(sys.prefix) / "Scripts" / "conda.exe",
    Path(sys.prefix) / "condabin" / "conda.exe"
]

conda_exe = next(
    (path for path in conda_candidates if path.is_file()),
    None
)

if conda_exe is None:
    conda_path = shutil.which("conda")
    if conda_path is None:
        raise FileNotFoundError("Conda executable was not found.")
    conda_exe = Path(conda_path)

# Create an isolated Python environment
env_python = ENV_DIR / "python.exe"

if not env_python.is_file():
    print("Creating the research environment...")

    subprocess.run(
        [
            str(conda_exe),
            "create",
            "--yes",
            "--prefix",
            str(ENV_DIR),
            "python=3.12",
            "pip",
            "ipykernel"
        ],
        check=True
    )
else:
    print("Research environment already exists.")

# Register the Jupyter kernel
subprocess.run(
    [
        str(env_python),
        "-m",
        "ipykernel",
        "install",
        "--user",
        "--name",
        KERNEL_NAME,
        "--display-name",
        KERNEL_DISPLAY_NAME
    ],
    check=True
)

# Verify the environment
result = subprocess.run(
    [str(env_python), "--version"],
    capture_output=True,
    text=True,
    check=True
)

print("\n" + "=" * 60)
print("RESEARCH ENVIRONMENT")
print("=" * 60)
print("Environment :", ENV_DIR)
print("Python      :", result.stdout.strip())
print("Kernel      :", KERNEL_DISPLAY_NAME)
print("Status      : Successfully configured")

Creating the research environment...

RESEARCH ENVIRONMENT
Environment : D:\NST_NSCLC_PPI_GAT\envs\nsclc
Python      : Python 3.12.15
Kernel      : NST NSCLC Research (Python 3.12)
Status      : Successfully configured


In [3]:

import os
import sys
import json
import platform
import subprocess
import importlib.metadata as metadata
from pathlib import Path
from datetime import datetime

# ============================================================
# 1. Project and environment verification
# ============================================================

PROJECT_DIR = Path(r"D:\NST_NSCLC_PPI_GAT")
ENV_DIR = PROJECT_DIR / "envs" / "nsclc"
EXPECTED_PYTHON = ENV_DIR / "python.exe"

if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError(
        "Incorrect kernel. Select NST NSCLC Research (Python 3.12)."
    )

for folder in ["src", "configs", "logs", "results/metrics"]:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

print("=" * 65)
print("NST NSCLC - ENVIRONMENT REPAIR AND VERIFICATION")
print("=" * 65)
print("Python :", sys.version.split()[0])
print("Kernel :", sys.executable)


# ============================================================
# 2. Package installation utilities
# ============================================================

def get_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return None


def execute(command, description):
    print(f"\n{description}")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=False
    )

    if result.returncode != 0:
        print(result.stdout[-4000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f"Failed: {description}")

    return result


# ============================================================
# 3. Repair PyTorch Geometric
# ============================================================

import torch

print("\nPyTorch Version :", torch.__version__)
print("Current PyG     :", get_version("torch-geometric"))

TARGET_PYG = "2.6.1"

if get_version("torch-geometric") != TARGET_PYG:

    result = execute(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--disable-pip-version-check",
            "--no-cache-dir",
            "--only-binary=:all:",
            "--force-reinstall",
            "--no-deps",
            f"torch-geometric=={TARGET_PYG}"
        ],
        "Installing compatible PyTorch Geometric..."
    )

    print(result.stdout.strip().splitlines()[-1])

else:
    print("Compatible PyG version already installed.")

assert get_version("torch-geometric") == TARGET_PYG

# Validate installed dependencies
execute(
    [sys.executable, "-m", "pip", "check"],
    "Checking package dependencies..."
)

print("Dependency check: PASSED")


# ============================================================
# 4. Import research libraries
# ============================================================

import numpy as np
import pandas as pd
import scipy
import sklearn
import networkx as nx
import torch_geometric

from torch import nn
from torch.nn import functional as F

from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, global_mean_pool

from captum.attr import IntegratedGradients

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef
)

print("\nResearch libraries imported successfully.")
print("PyTorch Geometric :", torch_geometric.__version__)


# ============================================================
# 5. Reproducibility and compute settings
# ============================================================

SEED = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

CPU_THREADS = min(4, os.cpu_count() or 1)
torch.set_num_threads(CPU_THREADS)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nCompute Device :", DEVICE)
print("CPU Threads    :", torch.get_num_threads())


# ============================================================
# 6. Construct a synthetic gene interaction network
# ============================================================

NUM_GENES = 24
NUM_MODALITIES = 4
NUM_REPLICATES = 3

edges = set()

for gene in range(NUM_GENES):

    for offset in (1, 3):

        neighbor = (gene + offset) % NUM_GENES

        edges.add((gene, neighbor))
        edges.add((neighbor, gene))

edge_index = torch.tensor(
    sorted(edges),
    dtype=torch.long
).t().contiguous()

graph = nx.Graph()

graph.add_nodes_from(range(NUM_GENES))
graph.add_edges_from(edges)

assert nx.is_connected(graph)
assert edge_index.shape[0] == 2

print("\nSynthetic Gene Graph")
print("Genes      :", NUM_GENES)
print("Modalities :", NUM_MODALITIES)
print("Edges      :", edge_index.shape[1])
print("Connected  :", nx.is_connected(graph))


# ============================================================
# 7. Generate synthetic multi-omics patient data
# ============================================================

samples = []

for stage in (0, 1):

    for histology in (0, 1):

        for replicate in range(NUM_REPLICATES):

            features = (
                torch.randn(NUM_GENES, NUM_MODALITIES) * 0.25
            )

            # Stage-associated expression features
            features[:6, 0] += stage * 0.9

            # Stage-associated mutation features
            features[12:16, 1] += stage * 0.5

            # Histology-associated methylation features
            features[7:11, 3] += histology * 0.6

            sample = Data(
                x=features,
                edge_index=edge_index,
                stage=torch.tensor([stage]),
                histology=torch.tensor([histology])
            )

            samples.append(sample)

patient_batch = Batch.from_data_list(
    samples
).to(DEVICE)

stages = patient_batch.stage.view(-1)
histologies = patient_batch.histology.view(-1)

assert len(samples) == 12
assert patient_batch.x.shape == (12 * NUM_GENES, 4)

print("\nSynthetic Patient Data")
print("Patients :", len(samples))
print("Stages   :", torch.bincount(stages).tolist())
print("Subtypes :", torch.bincount(histologies).tolist())


# ============================================================
# 8. Two-layer residual GAT architecture
# ============================================================

class MultiOmicsGAT(nn.Module):

    def __init__(
        self,
        input_dim=4,
        hidden_dim=32,
        dropout=0.10
    ):
        super().__init__()

        self.input_projection = nn.Linear(
            input_dim,
            hidden_dim
        )

        self.gat1 = GATConv(
            in_channels=hidden_dim,
            out_channels=8,
            heads=4,
            concat=True,
            dropout=dropout
        )

        self.gat2 = GATConv(
            in_channels=hidden_dim,
            out_channels=hidden_dim,
            heads=1,
            concat=False,
            dropout=dropout
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 16),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(16, 2)
        )

    def forward(self, data):

        x = F.relu(
            self.input_projection(data.x)
        )

        h1 = F.elu(
            self.gat1(x, data.edge_index)
        )

        x = self.norm1(x + h1)

        h2 = F.elu(
            self.gat2(x, data.edge_index)
        )

        x = self.norm2(x + h2)

        embeddings = global_mean_pool(
            x,
            data.batch
        )

        logits = self.classifier(
            embeddings
        )

        return logits, embeddings


# ============================================================
# 9. Class-conditional MMD loss
# ============================================================

def gaussian_mmd(x, y):

    combined = torch.cat([x, y], dim=0)

    distances = torch.cdist(
        combined,
        combined
    ).square()

    positive = distances.detach()
    positive = positive[positive > 0]

    if positive.numel() > 0:
        bandwidth = positive.median().clamp_min(1e-6)
    else:
        bandwidth = distances.new_tensor(1.0)

    kernel = torch.exp(
        -distances / (2.0 * bandwidth)
    )

    n = x.shape[0]

    k_xx = kernel[:n, :n].mean()
    k_yy = kernel[n:, n:].mean()
    k_xy = kernel[:n, n:].mean()

    return k_xx + k_yy - 2.0 * k_xy


def conditional_mmd(
    embeddings,
    stage_labels,
    histology_labels
):

    losses = []

    for stage in (0, 1):

        group_luad = embeddings[
            (stage_labels == stage) &
            (histology_labels == 0)
        ]

        group_lusc = embeddings[
            (stage_labels == stage) &
            (histology_labels == 1)
        ]

        if (
            group_luad.shape[0] > 0
            and group_lusc.shape[0] > 0
        ):
            losses.append(
                gaussian_mmd(
                    group_luad,
                    group_lusc
                )
            )

    if not losses:
        return embeddings.sum() * 0.0

    return torch.stack(losses).mean()


# ============================================================
# 10. Train and verify model gradients
# ============================================================

model = MultiOmicsGAT().to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

LAMBDA_MMD = 0.05
NUM_EPOCHS = 12

training_history = []

print("\n" + "=" * 65)
print("GRAPH LEARNING TRAINING TEST")
print("=" * 65)

for epoch in range(1, NUM_EPOCHS + 1):

    model.train()
    optimizer.zero_grad()

    logits, embeddings = model(
        patient_batch
    )

    classification_loss = F.cross_entropy(
        logits,
        stages
    )

    alignment_loss = conditional_mmd(
        embeddings,
        stages,
        histologies
    )

    total_loss = (
        classification_loss
        + LAMBDA_MMD * alignment_loss
    )

    if not torch.isfinite(total_loss).item():
        raise RuntimeError("Non-finite training loss detected.")

    total_loss.backward()

    for parameter in model.parameters():
        if parameter.grad is not None:
            if not torch.isfinite(parameter.grad).all().item():
                raise RuntimeError("Non-finite gradient detected.")

    nn.utils.clip_grad_norm_(
        model.parameters(),
        max_norm=1.0
    )

    optimizer.step()

    training_history.append(
        float(total_loss.item())
    )

    if epoch in (1, 4, 8, 12):
        print(
            f"Epoch {epoch:02d} | "
            f"Loss: {total_loss.item():.5f} | "
            f"CE: {classification_loss.item():.5f} | "
            f"MMD: {alignment_loss.item():.5f}"
        )

assert logits.shape == (len(samples), 2)
assert embeddings.shape == (len(samples), 32)

print("\nGAT forward pass  : PASSED")
print("GAT backward pass : PASSED")
print("MMD computation   : PASSED")
print("Optimizer update  : PASSED")


# ============================================================
# 11. Integrated Gradients verification
# ============================================================

model.eval()

single_patient = samples[0].x.unsqueeze(0).to(DEVICE)

def attribution_forward(features):

    graphs = [
        Data(
            x=features[i],
            edge_index=edge_index.to(features.device)
        )
        for i in range(features.shape[0])
    ]

    graph_batch = Batch.from_data_list(
        graphs
    ).to(features.device)

    prediction, _ = model(graph_batch)

    return prediction


integrated_gradients = IntegratedGradients(
    attribution_forward
)

attributions = integrated_gradients.attribute(
    inputs=single_patient,
    baselines=torch.zeros_like(single_patient),
    target=1,
    n_steps=8,
    internal_batch_size=4
)

assert attributions.shape == single_patient.shape
assert torch.isfinite(attributions).all().item()

print("\nIntegrated Gradients : PASSED")
print("Attribution Shape    :", tuple(attributions.shape))


# ============================================================
# 12. Statistical evaluation verification
# ============================================================

test_labels = np.array([0, 0, 1, 1])
test_scores = np.array([0.20, 0.35, 0.70, 0.85])
test_predictions = (test_scores >= 0.5).astype(int)

test_auc = roc_auc_score(
    test_labels,
    test_scores
)

test_pr_auc = average_precision_score(
    test_labels,
    test_scores
)

test_mcc = matthews_corrcoef(
    test_labels,
    test_predictions
)

assert all(
    np.isfinite(metric)
    for metric in (test_auc, test_pr_auc, test_mcc)
)

print("\nROC-AUC Metric : PASSED")
print("PR-AUC Metric  : PASSED")
print("MCC Metric     : PASSED")


# ============================================================
# 13. Save environment and verification records
# ============================================================

report = {
    "project": "NST_NSCLC_PPI_GAT",
    "timestamp": datetime.now().isoformat(),
    "status": "PASSED",
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "device": str(DEVICE),
    "cpu_threads": CPU_THREADS,
    "test_type": "synthetic_environment_verification",
    "synthetic_patients": len(samples),
    "synthetic_genes": NUM_GENES,
    "synthetic_modalities": NUM_MODALITIES,
    "gat_layers": 2,
    "gat_attention_heads": [4, 1],
    "patient_embedding_dimension": 32,
    "conditional_mmd": "PASSED",
    "forward_backward": "PASSED",
    "integrated_gradients": "PASSED",
    "evaluation_metrics": "PASSED",
    "training_epochs": NUM_EPOCHS,
    "synthetic_loss_history": training_history
}

report_path = (
    PROJECT_DIR /
    "logs" /
    "environment_verification_fixed.json"
)

report_path.write_text(
    json.dumps(report, indent=4),
    encoding="utf-8"
)

# Save portable package versions
package_list = execute(
    [
        sys.executable,
        "-m",
        "pip",
        "list",
        "--format=freeze"
    ],
    "Saving package versions..."
)

requirements_path = (
    PROJECT_DIR /
    "configs" /
    "requirements_verified.txt"
)

requirements_path.write_text(
    package_list.stdout,
    encoding="utf-8"
)

# ============================================================
# 14. Final verification summary
# ============================================================

print("\n" + "=" * 65)
print("FINAL VERIFICATION SUMMARY")
print("=" * 65)

checks = {
    "Environment": "PASSED",
    "PyTorch Installation": "PASSED",
    "PyG Import": "PASSED",
    "Dependency Check": "PASSED",
    "Synthetic PPI Graph": "PASSED",
    "Multi-Omics Data Structure": "PASSED",
    "Two-Layer Residual GAT": "PASSED",
    "Class-Conditional MMD": "PASSED",
    "Backpropagation": "PASSED",
    "Integrated Gradients": "PASSED",
    "Statistical Evaluation": "PASSED",
    "Reproducibility Records": "PASSED"
}

for name, status in checks.items():
    print(f"{name:<32}: {status}")

print("\nVerification report:", report_path)
print("Package versions   :", requirements_path)

print("\nENVIRONMENT VERIFICATION COMPLETED SUCCESSFULLY")
print("Next stage: TCGA clinical and multi-omics data acquisition")

NST NSCLC - ENVIRONMENT REPAIR AND VERIFICATION
Python : 3.12.15
Kernel : D:\NST_NSCLC_PPI_GAT\envs\nsclc\python.exe

PyTorch Version : 2.12.1+cpu
Current PyG     : 2.8.1

Installing compatible PyTorch Geometric...

Checking package dependencies...
Dependency check: PASSED


D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Research libraries imported successfully.
PyTorch Geometric : 2.6.1

Compute Device : cpu
CPU Threads    : 4

Synthetic Gene Graph
Genes      : 24
Modalities : 4
Edges      : 96
Connected  : True

Synthetic Patient Data
Patients : 12
Stages   : [6, 6]
Subtypes : [6, 6]

GRAPH LEARNING TRAINING TEST
Epoch 01 | Loss: 0.70241 | CE: 0.67761 | MMD: 0.49598
Epoch 04 | Loss: 0.67049 | CE: 0.64812 | MMD: 0.44740
Epoch 08 | Loss: 0.66249 | CE: 0.64085 | MMD: 0.43287
Epoch 12 | Loss: 0.62972 | CE: 0.61007 | MMD: 0.39304

GAT forward pass  : PASSED
GAT backward pass : PASSED
MMD computation   : PASSED
Optimizer update  : PASSED

Integrated Gradients : PASSED
Attribution Shape    : (1, 24, 4)

ROC-AUC Metric : PASSED
PR-AUC Metric  : PASSED
MCC Metric     : PASSED

Saving package versions...

FINAL VERIFICATION SUMMARY
Environment                     : PASSED
PyTorch Installation            : PASSED
PyG Import                      : PASSED
Dependency Check                : PASSED
Synthetic PPI Gr

In [4]:

import gzip
import hashlib
import json
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Project configuration
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED_PYTHON = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

API = "https://api.gdc.cancer.gov"
PROJECTS = ["TCGA-LUAD", "TCGA-LUSC"]
PAGE_SIZE = 250
REFRESH_METADATA = False

CACHE = ROOT / "data" / "raw" / "GDC_metadata"
INTERIM = ROOT / "data" / "interim"
METRICS = ROOT / "results" / "metrics"

for directory in (CACHE, INTERIM, METRICS):
    directory.mkdir(parents=True, exist_ok=True)

DATA_TYPES = {
    "Gene Expression Quantification": "rna",
    "Gene Level Copy Number": "cnv",
    "Copy Number Segment": "cnv_segment",
    "Methylation Beta Value": "methylation",
    "Masked Somatic Mutation": "mutation_maf"
}

# Configure HTTP requests
retry = Retry(
    total=5,
    backoff_factor=1.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"]
)

session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "User-Agent": "NST-NSCLC-Research/1.0"
})

session.mount(
    "https://",
    HTTPAdapter(max_retries=retry)
)


# Cache GDC metadata responses
def get_json(endpoint, params):

    cache_key = hashlib.sha256(
        json.dumps(
            {"endpoint": endpoint, "params": params},
            sort_keys=True
        ).encode()
    ).hexdigest()[:16]

    cache_path = CACHE / f"{endpoint}_{cache_key}.json.gz"

    if cache_path.exists() and not REFRESH_METADATA:
        with gzip.open(cache_path, "rt", encoding="utf-8") as f:
            return json.load(f)

    try:
        response = session.get(
            f"{API}/{endpoint}",
            params=params,
            timeout=(15, 100)
        )

        response.raise_for_status()
        result = response.json()

    except (requests.RequestException, ValueError) as error:
        raise RuntimeError(
            f"GDC API request failed ({endpoint}). "
            f"Check internet access and API availability. {error}"
        ) from error

    if "data" not in result or "hits" not in result["data"]:
        raise RuntimeError(
            f"Unexpected GDC response: {str(result)[:350]}"
        )

    with gzip.open(cache_path, "wt", encoding="utf-8") as f:
        json.dump(result, f)

    return result


# Retrieve all pages without truncating the cohort
def collect(endpoint, filters, fields, expand=""):

    records = []
    offset = 0
    total = None

    while True:

        params = {
            "filters": json.dumps(filters, separators=(",", ":")),
            "fields": fields,
            "format": "JSON",
            "size": PAGE_SIZE,
            "from": offset,
            "sort": (
                "case_id:asc"
                if endpoint == "cases"
                else "file_id:asc"
            )
        }

        if expand:
            params["expand"] = expand

        payload = get_json(endpoint, params)["data"]
        hits = payload["hits"]

        if total is None:
            total = int(
                payload.get("pagination", {}).get("total", len(hits))
            )

            if total > 25000:
                raise RuntimeError(
                    f"Unexpectedly large metadata query: {total}"
                )

            print(f"{endpoint}: {total:,} matching records")

        records.extend(hits)
        offset += len(hits)

        print(
            f"  Retrieved {offset:,} / {total:,}",
            end="\r"
        )

        if offset >= total:
            break

        if not hits:
            raise RuntimeError(
                f"Pagination stopped early: {offset}/{total}"
            )

    print(f"  Retrieved {len(records):,} / {total:,}")

    ids = [
        item.get("case_id")
        if endpoint == "cases"
        else item.get("file_id")
        for item in records
    ]

    if len(ids) != len(set(ids)):
        raise RuntimeError(
            f"Duplicate {endpoint} identifiers detected."
        )

    return records


# Define cohort and molecular file filters
project_filter = {
    "op": "in",
    "content": {
        "field": "project.project_id",
        "value": PROJECTS
    }
}

file_project_filter = {
    "op": "in",
    "content": {
        "field": "cases.project.project_id",
        "value": PROJECTS
    }
}

print("=" * 66)
print("TCGA-LUAD / TCGA-LUSC DATA DISCOVERY")
print("=" * 66)

# Retrieve clinical and biospecimen metadata
cases_raw = collect(
    endpoint="cases",
    filters=project_filter,
    fields="case_id,submitter_id",
    expand="project,diagnoses,samples,demographic"
)

if not cases_raw:
    raise RuntimeError("No TCGA cases were returned.")

# Retrieve patient-linked molecular file metadata
file_fields = ",".join([
    "file_id",
    "file_name",
    "file_size",
    "md5sum",
    "access",
    "data_type",
    "data_format",
    "data_category",
    "experimental_strategy",
    "analysis.workflow_type",
    "cases.case_id",
    "cases.submitter_id",
    "cases.project.project_id",
    "associated_entities.entity_submitter_id",
    "associated_entities.entity_type"
])

patient_file_filter = {
    "op": "and",
    "content": [
        file_project_filter,
        {
            "op": "in",
            "content": {
                "field": "data_type",
                "value": [
                    "Gene Expression Quantification",
                    "Gene Level Copy Number",
                    "Copy Number Segment",
                    "Methylation Beta Value"
                ]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "access",
                "value": ["open"]
            }
        }
    ]
}

patient_files_raw = collect(
    endpoint="files",
    filters=patient_file_filter,
    fields=file_fields
)

# Retrieve project-level mutation file metadata
maf_filter = {
    "op": "and",
    "content": [
        {
            "op": "in",
            "content": {
                "field": "data_type",
                "value": ["Masked Somatic Mutation"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "access",
                "value": ["open"]
            }
        }
    ]
}

maf_raw = collect(
    endpoint="files",
    filters=maf_filter,
    fields=file_fields
)


# Normalize clinical stage labels
stage_pattern = re.compile(
    r"^(IV|III|II|I)(?:[A-D])?(?:[1-4])?$"
)

def stage_group(value):

    if value is None:
        return None

    stage = str(value).upper().strip()
    stage = re.sub(r"^STAGE\s*", "", stage)
    stage = stage.replace(" ", "")

    match = stage_pattern.fullmatch(stage)

    if not match:
        return None

    return (
        "Early"
        if match.group(1) in ("I", "II")
        else "Advanced"
    )


def case_stage(diagnoses):

    diagnoses = diagnoses or []

    primary = [
        diagnosis
        for diagnosis in diagnoses
        if str(
            diagnosis.get("diagnosis_is_primary_disease", "")
        ).lower() in ("true", "yes", "1")
    ]

    selected = primary or diagnoses

    priority = (
        "ajcc_pathologic_stage",
        "ajcc_clinical_stage",
        "tumor_stage"
    )

    options = [
        (
            field,
            [
                (
                    diagnosis.get(field),
                    stage_group(diagnosis.get(field))
                )
                for diagnosis in selected
            ]
        )
        for field in priority
    ]

    all_groups = {
        group
        for _, values in options
        for _, group in values
        if group is not None
    }

    if len(all_groups) > 1:
        return None, "conflicting_stage_groups", ""

    for field, values in options:

        valid = [
            (raw, group)
            for raw, group in values
            if group is not None
        ]

        if valid:
            return valid[0][1], field, str(valid[0][0])

    return None, "missing_or_unrecognized_stage", ""


def is_primary_sample(sample):

    sample_type = str(
        sample.get("sample_type", "")
    ).strip().lower()

    sample_code = str(
        sample.get("sample_type_id", "")
    ).zfill(2)

    return (
        sample_type == "primary tumor"
        or sample_code == "01"
    )


# Construct the clinical cohort inventory
clinical_records = []

for case in cases_raw:

    project = case.get("project") or {}
    project_id = project.get("project_id")
    patient_id = case.get("submitter_id")

    if project_id not in PROJECTS or not patient_id:
        raise RuntimeError(
            f"Invalid clinical project or patient ID: "
            f"{case.get('case_id')}"
        )

    group, source, original_stage = case_stage(
        case.get("diagnoses")
    )

    primary_samples = [
        sample.get("submitter_id")
        for sample in (case.get("samples") or [])
        if is_primary_sample(sample)
    ]

    primary_samples = sorted(
        set(
            sample
            for sample in primary_samples
            if sample
        )
    )

    clinical_records.append({
        "case_id": case.get("case_id"),
        "patient_id": patient_id,
        "project": project_id,
        "histology": (
            "LUAD"
            if project_id == "TCGA-LUAD"
            else "LUSC"
        ),
        "stage_group": group,
        "stage_source": source,
        "stage_original": original_stage,
        "stage_label": {
            "Early": 0,
            "Advanced": 1
        }.get(group),
        "n_primary_samples": len(primary_samples),
        "primary_sample_ids": ";".join(primary_samples)
    })

clinical_df = pd.DataFrame(clinical_records)

assert clinical_df["case_id"].is_unique
assert clinical_df["patient_id"].is_unique

case_lookup = (
    clinical_df
    .set_index("case_id")
    .to_dict("index")
)


# Identify primary-tumor file associations
sample_code_pattern = re.compile(
    r"^(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})-(\d{2})(?:[A-Z]|-|$)",
    re.IGNORECASE
)

def sample_link_status(file_record, patient_id):

    barcodes = []

    for entity in file_record.get("associated_entities") or []:

        sample_id = str(
            entity.get("entity_submitter_id") or ""
        )

        match = sample_code_pattern.match(sample_id)

        if (
            match
            and match.group(1).upper() == patient_id.upper()
        ):
            barcodes.append(
                (sample_id, match.group(2))
            )

    codes = {
        code
        for _, code in barcodes
    }

    if codes == {"01"}:
        status = "confirmed_primary"

    elif "01" in codes:
        status = "mixed_sample_types"

    elif codes:
        status = "non_primary"

    else:
        status = "unverified"

    sample_ids = ";".join(
        sorted(set(sample for sample, _ in barcodes))
    )

    return status, sample_ids


# Construct the molecular file inventory
file_records = []

for file_record in patient_files_raw:

    data_type = file_record.get("data_type")
    modality = DATA_TYPES.get(data_type)

    if not modality:
        continue

    case_references = file_record.get("cases") or []

    matching_cases = {
        reference.get("case_id")
        for reference in case_references
        if reference.get("case_id") in case_lookup
    }

    if (
        len(matching_cases) != 1
        or len(case_references) != 1
    ):
        continue

    case_id = next(iter(matching_cases))
    patient_id = case_lookup[case_id]["patient_id"]

    link_status, sample_ids = sample_link_status(
        file_record,
        patient_id
    )

    file_records.append({
        "file_id": file_record.get("file_id"),
        "case_id": case_id,
        "patient_id": patient_id,
        "project": case_lookup[case_id]["project"],
        "modality": modality,
        "data_type": data_type,
        "workflow": (
            file_record.get("analysis") or {}
        ).get("workflow_type"),
        "data_format": file_record.get("data_format"),
        "file_name": file_record.get("file_name"),
        "file_size_bytes": file_record.get("file_size"),
        "md5sum": file_record.get("md5sum"),
        "sample_link_status": link_status,
        "associated_sample_barcodes": sample_ids
    })

files_df = pd.DataFrame(file_records)

if files_df.empty:
    raise RuntimeError(
        "No patient-linked molecular files were found. "
        "Review the GDC metadata response."
    )

if not files_df["file_id"].is_unique:
    raise RuntimeError(
        "Duplicate molecular file identifiers detected."
    )


# Catalog aggregated mutation files separately
maf_records = []

for file_record in maf_raw:

    file_name = str(
        file_record.get("file_name") or ""
    )

    linked_projects = {
        (reference.get("project") or {}).get("project_id")
        for reference in file_record.get("cases") or []
    } & set(PROJECTS)

    named_projects = {
        project
        for project in PROJECTS
        if re.search(
            rf"(?<![A-Z0-9]){project.split('-')[1]}(?![A-Z0-9])",
            file_name,
            re.IGNORECASE
        )
    }

    matched_projects = linked_projects | named_projects

    maf_records.append({
        "file_id": file_record.get("file_id"),
        "file_name": file_name,
        "project_hint": (
            ";".join(sorted(matched_projects))
            if matched_projects
            else "unresolved"
        ),
        "file_size_bytes": file_record.get("file_size"),
        "workflow": (
            file_record.get("analysis") or {}
        ).get("workflow_type"),
        "data_format": file_record.get("data_format"),
        "md5sum": file_record.get("md5sum")
    })

maf_columns = [
    "file_id",
    "file_name",
    "project_hint",
    "file_size_bytes",
    "workflow",
    "data_format",
    "md5sum"
]

maf_df = pd.DataFrame(
    maf_records,
    columns=maf_columns
)


# Calculate patient-level modality coverage
availability = clinical_df.copy()

for modality in (
    "rna",
    "cnv",
    "cnv_segment",
    "methylation"
):

    confirmed_cases = set(
        files_df.loc[
            (files_df["modality"] == modality)
            & (
                files_df["sample_link_status"]
                == "confirmed_primary"
            ),
            "case_id"
        ]
    )

    availability[f"confirmed_{modality}"] = (
        availability["case_id"].isin(confirmed_cases)
    )

availability["confirmed_cnv_any"] = (
    availability["confirmed_cnv"]
    | availability["confirmed_cnv_segment"]
)

availability["stage_eligible"] = (
    availability["stage_group"].notna()
    & (availability["n_primary_samples"] > 0)
)

availability["three_omics_metadata_candidate"] = (
    availability["stage_eligible"]
    & availability["confirmed_rna"]
    & availability["confirmed_cnv_any"]
    & availability["confirmed_methylation"]
)

availability["mutation_patient_coverage"] = (
    "pending_MAF_barcode_audit"
)


# Save clinical, molecular and mutation inventories
clinical_path = INTERIM / "tcga_clinical_stage_audit.csv"
files_path = INTERIM / "tcga_omics_file_inventory.csv"
maf_path = INTERIM / "tcga_masked_maf_project_catalog.csv"
availability_path = INTERIM / "tcga_patient_metadata_coverage.csv"

clinical_df.to_csv(
    clinical_path,
    index=False
)

files_df.to_csv(
    files_path,
    index=False
)

maf_df.to_csv(
    maf_path,
    index=False
)

availability.to_csv(
    availability_path,
    index=False
)


# Generate stage-by-histology cohort statistics
cohort_table = (
    availability[
        availability["stage_eligible"]
    ]
    .groupby(
        ["histology", "stage_group"],
        dropna=False
    )
    .agg(
        patients=("case_id", "count"),
        primary_RNA=("confirmed_rna", "sum"),
        primary_CNV=("confirmed_cnv_any", "sum"),
        primary_methylation=("confirmed_methylation", "sum"),
        three_omics_candidates=(
            "three_omics_metadata_candidate",
            "sum"
        )
    )
    .reset_index()
)

cohort_table_path = (
    METRICS / "tcga_preliminary_cohort_table.csv"
)

cohort_table.to_csv(
    cohort_table_path,
    index=False
)


# Save the audit summary
summary = {
    "generated_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source": API,
    "projects": PROJECTS,
    "clinical_cases": len(clinical_df),
    "stage_eligible_cases": int(
        availability["stage_eligible"].sum()
    ),
    "stage_conflicts": int(
        (
            clinical_df["stage_source"]
            == "conflicting_stage_groups"
        ).sum()
    ),
    "cases_without_primary_sample": int(
        (clinical_df["n_primary_samples"] == 0).sum()
    ),
    "patient_linked_molecular_files": len(files_df),
    "file_link_quality": (
        files_df["sample_link_status"]
        .value_counts()
        .to_dict()
    ),
    "file_modality_counts": (
        files_df["modality"]
        .value_counts()
        .to_dict()
    ),
    "mutation_maf_catalog_files": len(maf_df),
    "unresolved_maf_project_labels": int(
        (maf_df["project_hint"] == "unresolved").sum()
    ),
    "three_omics_metadata_candidates": int(
        availability[
            "three_omics_metadata_candidate"
        ].sum()
    ),
    "four_omics_eligibility_status": (
        "not_determined_until_MAF_content_and_file_QC"
    ),
    "metadata_only": True,
    "clinical_stage_priority": [
        "ajcc_pathologic_stage",
        "ajcc_clinical_stage",
        "tumor_stage"
    ],
    "staging_policy": (
        "exclude cases with discordant "
        "early-versus-advanced staging"
    )
}

summary_path = (
    METRICS / "tcga_data_discovery_summary.json"
)

summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)


# Display results
print("\n" + "=" * 66)
print("CLINICAL AND MOLECULAR DATA AUDIT")
print("=" * 66)

print(
    f"Total clinical cases: "
    f"{len(clinical_df):,}"
)

print(
    f"Stage-eligible primary-tumor cases: "
    f"{summary['stage_eligible_cases']:,}"
)

print(
    f"Stage conflicts: "
    f"{summary['stage_conflicts']:,}"
)

print(
    f"Patient-linked molecular files: "
    f"{len(files_df):,}"
)

print(
    f"Mutation MAF catalog entries: "
    f"{len(maf_df):,}"
)

print(
    f"Preliminary three-omics candidates: "
    f"{summary['three_omics_metadata_candidates']:,}"
)

print("\nFile-to-sample linkage status:")
print(
    files_df["sample_link_status"]
    .value_counts()
    .to_string()
)

print("\nMolecular file categories:")
print(
    files_df["modality"]
    .value_counts()
    .to_string()
)

print("\nStage-by-histology cohort:")
display(cohort_table)

print("\nProject-level mutation file catalog:")
display(maf_df.head(15))

print("\nSaved files:")

for path in (
    clinical_path,
    files_path,
    maf_path,
    availability_path,
    cohort_table_path,
    summary_path
):
    print(" -", path)

print("\n" + "=" * 66)
print("TCGA METADATA DISCOVERY COMPLETED")
print("=" * 66)

print(
    "Mutation coverage remains unverified "
    "until MAF contents are examined."
)
print(
    "Molecular expression, CNV and methylation "
    "matrices have not yet been downloaded."
)

TCGA-LUAD / TCGA-LUSC DATA DISCOVERY
cases: 1,089 matching records
  Retrieved 1,089 / 1,089
files: 8,512 matching records
  Retrieved 8,512 / 8,512
files: 24,498 matching records
  Retrieved 24,498 / 24,498

CLINICAL AND MOLECULAR DATA AUDIT
Total clinical cases: 1,089
Stage-eligible primary-tumor cases: 1,014
Stage conflicts: 0
Patient-linked molecular files: 8,512
Mutation MAF catalog entries: 24,498
Preliminary three-omics candidates: 1,000

File-to-sample linkage status:
sample_link_status
confirmed_primary     4288
mixed_sample_types    2852
non_primary           1372

Molecular file categories:
modality
cnv            3130
cnv_segment    2936
methylation    1283
rna            1163

Stage-by-histology cohort:


,histology,stage_group,patients,primary_RNA,primary_CNV,primary_methylation,three_omics_candidates
0,LUAD,Advanced,111,111,109,110,109
1,LUAD,Early,403,398,401,399,396
2,LUSC,Advanced,92,91,92,91,90
3,LUSC,Early,408,406,407,408,405



Project-level mutation file catalog:


,file_id,file_name,project_hint,file_size_bytes,workflow,data_format,md5sum
0,0000e8b4-1361-41b9-b11d-bcd26d4c5ee1,74940d19-7a99-4323-9b8c-07cca52faf3c.wxs.aliqu...,unresolved,255772,Aliquot Ensemble Somatic Variant Merging and M...,MAF,ef87a17f3ba4577abad461c3e35a1df5
1,0000f086-7e12-4778-bdca-959761c1d39e,2c7dfeb1-0616-418d-96b7-d2ec8cd3f9dc.wxs.aliqu...,unresolved,10514,Aliquot Ensemble Somatic Variant Merging and M...,MAF,4358e9ded7aeff82d53e3579ea2716a4
2,0001310d-964e-4c94-bde5-0510e5b76bb9,93959bfa-3652-4474-b3e8-b6207f518ce7.wxs.aliqu...,unresolved,30113,Aliquot Ensemble Somatic Variant Merging and M...,MAF,b9c355a1b5496c2d1df9656a12031b72
3,0009c117-c085-4cbc-8558-357fe21e54ff,6bcf8a46-74e9-4a6f-85bc-81d9ab300d6d.wxs.aliqu...,unresolved,45610,Aliquot Ensemble Somatic Variant Merging and M...,MAF,1fcf2125aafca3318d4de57e86697066
4,000dcbf6-d976-41a6-9086-1ee080ea6a55,46359501-e75b-4e86-af89-36b5ada5ef46.wxs.aliqu...,unresolved,13084,Aliquot Ensemble Somatic Variant Merging and M...,MAF,34e9b36c80a29b2514193ba5500b9968
5,000f237f-f77b-46db-ac6b-00b127be25cf,543dd9b8-4fed-4913-9af1-2dd15b907c4e.wxs.aliqu...,unresolved,24886,Aliquot Ensemble Somatic Variant Merging and M...,MAF,c0a94942e9fd7923f1484f525ad32090
6,00129ada-3434-4334-a881-3aed765c8765,6f4ec3e7-ab91-4c63-b84f-fc626557f60d.wxs.aliqu...,unresolved,10892,Aliquot Ensemble Somatic Variant Merging and M...,MAF,5d0c00e158d3fe2072267bd60415ee0a
7,00187151-a31d-4bc9-a86b-a4a4c9ecd596,337bfa14-5549-44fd-96c2-ae1c733f8b55.wxs.aliqu...,unresolved,28457,Aliquot Ensemble Somatic Variant Merging and M...,MAF,a987d92e8ff6b400a94d7ee7af4184ad
8,001bba13-5739-45b9-a55e-cd47495abb29,4a3991fc-ed49-4aee-b841-e974864a706e.wxs.aliqu...,unresolved,76318,Aliquot Ensemble Somatic Variant Merging and M...,MAF,86f511348f733a3a882fac7b7afc4644
9,002278c2-89ad-4ae7-a91c-9a7e9e060e5b,725068d1-c899-4391-b060-fc9f55a2a220.wxs.aliqu...,unresolved,10548,Aliquot Ensemble Somatic Variant Merging and M...,MAF,78163c0ae4d2bdc7cf883d3e7a3433e2



Saved files:
 - D:\NST_NSCLC_PPI_GAT\data\interim\tcga_clinical_stage_audit.csv
 - D:\NST_NSCLC_PPI_GAT\data\interim\tcga_omics_file_inventory.csv
 - D:\NST_NSCLC_PPI_GAT\data\interim\tcga_masked_maf_project_catalog.csv
 - D:\NST_NSCLC_PPI_GAT\data\interim\tcga_patient_metadata_coverage.csv
 - D:\NST_NSCLC_PPI_GAT\results\metrics\tcga_preliminary_cohort_table.csv
 - D:\NST_NSCLC_PPI_GAT\results\metrics\tcga_data_discovery_summary.json

TCGA METADATA DISCOVERY COMPLETED
Mutation coverage remains unverified until MAF contents are examined.
Molecular expression, CNV and methylation matrices have not yet been downloaded.


In [5]:

import gzip
import hashlib
import json
import os
import re
import sys
import time
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# Project configuration
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED_PYTHON = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

INTERIM = ROOT / "data" / "interim"
RAW = ROOT / "data" / "raw" / "GDC_pilot"
CACHE = ROOT / "data" / "raw" / "GDC_metadata"
RESULTS = ROOT / "results" / "metrics"

for folder in (INTERIM, RAW, CACHE, RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

API = "https://api.gdc.cancer.gov"
PROJECTS = ("TCGA-LUAD", "TCGA-LUSC")

DOWNLOAD_PILOT = True
PATIENTS_PER_GROUP = 1
MAX_PILOT_MB = 650
MAX_FILE_MB = 220
REFRESH_MUTATION_METADATA = False

# Load previous cohort inventories
clinical_path = INTERIM / "tcga_clinical_stage_audit.csv"
files_path = INTERIM / "tcga_omics_file_inventory.csv"

for path in (clinical_path, files_path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Cell 05 output missing: {path}"
        )

clinical = pd.read_csv(
    clinical_path,
    dtype=str,
    keep_default_na=False
)

files = pd.read_csv(
    files_path,
    dtype=str,
    keep_default_na=False
)

required_clinical = {
    "case_id",
    "patient_id",
    "project",
    "histology",
    "stage_group",
    "n_primary_samples"
}

required_files = {
    "file_id",
    "case_id",
    "patient_id",
    "project",
    "modality",
    "workflow",
    "file_name",
    "file_size_bytes",
    "md5sum",
    "sample_link_status",
    "associated_sample_barcodes"
}

if (
    required_clinical - set(clinical.columns)
    or required_files - set(files.columns)
):
    raise ValueError(
        "Missing required Cell 05 CSV columns."
    )

if (
    clinical.case_id.duplicated().any()
    or files.file_id.duplicated().any()
):
    raise ValueError(
        "Duplicate case or file IDs in the saved inventory."
    )

clinical["n_primary_samples"] = pd.to_numeric(
    clinical["n_primary_samples"],
    errors="coerce"
).fillna(0).astype(int)

clinical = clinical[
    clinical.project.isin(PROJECTS)
    & clinical.stage_group.isin(["Early", "Advanced"])
    & (clinical.n_primary_samples > 0)
].copy()

files["file_size_bytes"] = pd.to_numeric(
    files["file_size_bytes"],
    errors="coerce"
)

files = files[
    files.case_id.isin(set(clinical.case_id))
].copy()

print("=" * 74)
print("GDC MOLECULAR FILE SELECTION AND PILOT QUALITY CONTROL")
print("=" * 74)

print(
    f"Stage-eligible patients: {len(clinical):,}; "
    f"saved molecular files: {len(files):,}"
)

# Review sample links and workflow availability
for modality in ("rna", "cnv", "methylation"):

    subset = files[
        (files.modality == modality)
        & (files.sample_link_status == "confirmed_primary")
    ]

    print(
        f"\n{modality.upper()} primary-tumor workflows "
        f"({subset.case_id.nunique()} patients):"
    )

    print(
        subset.workflow.replace(
            "", "Unspecified"
        ).value_counts().head(8).to_string()
    )

# Configure GDC requests
retries = Retry(
    total=5,
    connect=5,
    read=3,
    backoff_factor=1.5,
    status_forcelist=(429, 500, 502, 503, 504),
    allowed_methods=frozenset(["GET"])
)

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "NST-NSCLC-Research/1.0"
})

session.mount(
    "https://",
    HTTPAdapter(max_retries=retries)
)

# Correct the mutation project filter
mut_filter = {
    "op": "and",
    "content": [
        {
            "op": "in",
            "content": {
                "field": "cases.project.project_id",
                "value": list(PROJECTS)
            }
        },
        {
            "op": "in",
            "content": {
                "field": "data_type",
                "value": ["Masked Somatic Mutation"]
            }
        },
        {
            "op": "in",
            "content": {
                "field": "access",
                "value": ["open"]
            }
        }
    ]
}

fields = ",".join([
    "file_id",
    "file_name",
    "file_size",
    "md5sum",
    "access",
    "data_type",
    "data_format",
    "analysis.workflow_type",
    "cases.case_id",
    "cases.submitter_id",
    "cases.project.project_id",
    "associated_entities.entity_submitter_id"
])

cache_key = hashlib.sha256(
    json.dumps(
        {
            "filters": mut_filter,
            "fields": fields
        },
        sort_keys=True
    ).encode()
).hexdigest()[:20]

mutation_cache = (
    CACHE
    / f"tcga_luad_lusc_masked_maf_{cache_key}.json.gz"
)

# Retrieve project-filtered mutation metadata
if mutation_cache.exists() and not REFRESH_MUTATION_METADATA:

    with gzip.open(
        mutation_cache,
        "rt",
        encoding="utf-8"
    ) as handle:
        mut_files = json.load(handle)

else:

    mut_files = []
    offset = 0
    total = None

    while total is None or offset < total:

        params = {
            "filters": json.dumps(
                mut_filter,
                separators=(",", ":")
            ),
            "fields": fields,
            "format": "JSON",
            "size": 200,
            "from": offset,
            "sort": "file_id:asc"
        }

        response = session.get(
            f"{API}/files",
            params=params,
            timeout=(20, 120)
        )

        response.raise_for_status()

        payload = response.json().get("data", {})
        batch = payload.get("hits", [])

        if total is None:

            total = int(
                payload.get(
                    "pagination", {}
                ).get("total", len(batch))
            )

            print(
                f"\nProject-filtered masked MAF files: {total:,}"
            )

            if total > 10000:
                raise RuntimeError(
                    "MAF query returned too many files; "
                    "check project filtering."
                )

        if not batch and offset < total:
            raise RuntimeError(
                f"GDC returned an empty page "
                f"at offset {offset}/{total}."
            )

        mut_files.extend(batch)
        offset += len(batch)

        print(
            f"Retrieved {offset:,}/{total:,} mutation records",
            end="\r"
        )

    print()

    if len({
        item.get("file_id")
        for item in mut_files
    }) != len(mut_files):
        raise RuntimeError(
            "Duplicate mutation file UUIDs returned by GDC."
        )

    with gzip.open(
        mutation_cache,
        "wt",
        encoding="utf-8"
    ) as handle:
        json.dump(mut_files, handle)

# Validate mutation file associations
case_to_project = (
    clinical.set_index("case_id").project.to_dict()
)

mutation_records = []

for item in mut_files:

    linked = [
        str(record.get("case_id") or "")
        for record in item.get("cases") or []
    ]

    eligible = sorted(
        set(linked) & set(case_to_project)
    )

    linked_projects = {
        case_to_project[case_id]
        for case_id in eligible
    }

    if not linked_projects:
        continue

    if len(linked_projects) > 1:
        continue

    file_id = str(item.get("file_id") or "")
    filename = str(item.get("file_name") or "")

    if str(
        item.get("data_format") or ""
    ).upper() != "MAF":
        continue

    mutation_records.append({
        "file_id": file_id,
        "file_name": filename,
        "file_size_bytes": item.get("file_size"),
        "md5sum": str(item.get("md5sum") or ""),
        "workflow": str(
            (item.get("analysis") or {}).get("workflow_type")
            or ""
        ),
        "project": next(iter(linked_projects)),
        "linked_cases": len(set(linked)),
        "eligible_case_count": len(eligible),
        "case_id": (
            eligible[0]
            if len(set(linked)) == 1
            else ""
        ),
        "record_type": (
            "case_linked"
            if len(set(linked)) == 1
            else "multi_case"
        )
    })

mutation = pd.DataFrame(
    mutation_records,
    columns=[
        "file_id",
        "file_name",
        "file_size_bytes",
        "md5sum",
        "workflow",
        "project",
        "linked_cases",
        "eligible_case_count",
        "case_id",
        "record_type"
    ]
)

mutation["file_size_bytes"] = pd.to_numeric(
    mutation["file_size_bytes"],
    errors="coerce"
)

mutation_path = (
    INTERIM
    / "tcga_luad_lusc_corrected_mutation_inventory.csv"
)

mutation.to_csv(
    mutation_path,
    index=False
)

print(
    f"\nVerified project-linked MAF entries: "
    f"{len(mutation):,}"
)

if not mutation.empty:

    print(
        mutation.groupby([
            "project",
            "record_type",
            "workflow"
        ]).size().sort_values(
            ascending=False
        ).head(12).to_string()
    )

else:
    print(
        "No case-linked open MAFs found; "
        "mutation eligibility remains unresolved."
    )

# Extract primary-tumor vial identifiers
barcode_pattern = re.compile(
    r"^(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4}-01[A-Z])",
    re.IGNORECASE
)

def sample_vials(text):

    result = set()

    for token in str(text).split(";"):

        match = barcode_pattern.match(
            token.strip()
        )

        if match:
            result.add(
                match.group(1).upper()
            )

    return sorted(result)


# Rank molecular data workflows
def preference(row):

    workflow = str(row.workflow).lower()
    filename = str(row.file_name).lower()

    if row.modality == "rna":
        return (
            0
            if "star - counts" in workflow
            else 100
        )

    if row.modality == "cnv":

        if (
            "absolute" in workflow
            and "snp" in (workflow + filename)
        ):
            return 0

        if "ascat3" in workflow:
            return 1

        if (
            "ascatngs" in workflow
            or "ascat-ngs" in workflow
        ):
            return 2

        if "ascat2" in workflow:
            return 3

        return 5

    if row.modality == "methylation":

        modern = (
            0
            if (
                "sesame" in workflow
                or "harmon" in workflow
            )
            else 1
        )

        platform = (
            0
            if (
                "450" in filename
                or "450" in workflow
            )
            else (
                1
                if "epic" in filename
                else 2
            )
        )

        return modern * 10 + platform

    return 100


# Retain primary-tumor molecular files
valid = files[
    (files.sample_link_status == "confirmed_primary")
    & files.modality.isin([
        "rna",
        "cnv",
        "methylation"
    ])
    & files.file_size_bytes.gt(0)
    & files.file_id.ne("")
].copy()

valid["vials"] = (
    valid.associated_sample_barcodes.map(sample_vials)
)

valid["rank"] = valid.apply(
    preference,
    axis=1
)

# Require STAR Counts for RNA
valid = valid[
    ~(
        (valid.modality == "rna")
        & (valid["rank"] > 0)
    )
].copy()

valid["size"] = (
    valid.file_size_bytes.astype("int64")
)

# Select one file per case and modality
manifest_rows = []
gaps = []

for patient in clinical.itertuples(index=False):

    candidates = valid[
        valid.case_id == patient.case_id
    ]

    pools = {
        modality: candidates[
            candidates.modality == modality
        ]
        for modality in (
            "rna",
            "cnv",
            "methylation"
        )
    }

    missing = [
        modality
        for modality, frame in pools.items()
        if frame.empty
    ]

    if missing:

        gaps.append({
            "case_id": patient.case_id,
            "histology": patient.histology,
            "stage_group": patient.stage_group,
            "missing_modalities": ";".join(missing)
        })

        continue

    vial_scores = Counter()

    for modality, pool in pools.items():

        observed = {
            vial
            for row in pool.itertuples(index=False)
            for vial in row.vials
        }

        vial_scores.update(observed)

    preferred_vial = (
        sorted(
            vial_scores,
            key=lambda vial: (
                -vial_scores[vial],
                vial
            )
        )[0]
        if vial_scores
        else ""
    )

    for modality, pool in pools.items():

        pool = pool.copy()

        pool["vial_match"] = pool.vials.map(
            lambda vials: (
                preferred_vial in vials
                if preferred_vial
                else False
            )
        )

        pool = pool.sort_values(
            [
                "rank",
                "vial_match",
                "size",
                "file_id"
            ],
            ascending=[
                True,
                False,
                True,
                True
            ],
            kind="stable"
        )

        chosen = pool.iloc[0]

        try:
            uuid.UUID(
                str(chosen.file_id)
            )

        except (ValueError, AttributeError) as exc:
            raise ValueError(
                f"Invalid file UUID: {chosen.file_id}"
            ) from exc

        manifest_rows.append({
            "case_id": patient.case_id,
            "patient_id": patient.patient_id,
            "project": patient.project,
            "histology": patient.histology,
            "stage_group": patient.stage_group,
            "modality": modality,
            "file_id": chosen.file_id,
            "file_name": chosen.file_name,
            "file_size_bytes": int(
                chosen["file_size_bytes"]
            ),
            "md5sum": chosen.md5sum,
            "workflow": chosen.workflow,
            "sample_vials": ";".join(chosen.vials),
            "preferred_vial": preferred_vial,
            "vial_match": bool(chosen.vial_match),
            "preference_rank": int(chosen["rank"])
        })

manifest = pd.DataFrame(manifest_rows)

if manifest.empty:
    raise RuntimeError(
        "No complete three-modality cohort found."
    )

if manifest.groupby(
    "case_id"
).modality.nunique().ne(3).any():
    raise RuntimeError(
        "Three-modality selection failed consistency validation."
    )

manifest_path = (
    INTERIM
    / "tcga_primary_tumor_three_omics_manifest.csv"
)

manifest.to_csv(
    manifest_path,
    index=False
)

gaps_path = (
    INTERIM
    / "tcga_primary_tumor_selection_exclusions.csv"
)

pd.DataFrame(
    gaps,
    columns=[
        "case_id",
        "histology",
        "stage_group",
        "missing_modalities"
    ]
).to_csv(
    gaps_path,
    index=False
)

cohort = manifest.drop_duplicates("case_id")

print(
    "\nStrict primary-tumor 3-omics cohort "
    "(before molecular QC):"
)

print(
    pd.crosstab(
        cohort.histology,
        cohort.stage_group
    ).to_string()
)

print(
    f"Patients: {cohort.case_id.nunique():,}; "
    f"files: {len(manifest):,}"
)

print(
    f"Total planned size: "
    f"{manifest.file_size_bytes.sum() / (1024 ** 3):,.2f} GiB"
)

print(
    f"Cross-vial selected files: "
    f"{(~manifest.vial_match).sum():,}"
)

print(
    f"Cases excluded at this stage: {len(gaps):,}"
)

# Select technical QC samples from all four groups
pilot_patients = []

for histology in ("LUAD", "LUSC"):

    for stage in ("Early", "Advanced"):

        group = manifest[
            (manifest.histology == histology)
            & (manifest.stage_group == stage)
        ]

        grouped = group.groupby(
            "case_id"
        ).agg(
            total_bytes=(
                "file_size_bytes",
                "sum"
            ),
            vial_matches=(
                "vial_match",
                "sum"
            )
        ).reset_index()

        grouped = grouped.sort_values(
            [
                "vial_matches",
                "total_bytes",
                "case_id"
            ],
            ascending=[
                False,
                True,
                True
            ],
            kind="stable"
        )

        chosen = grouped.head(
            PATIENTS_PER_GROUP
        ).case_id.tolist()

        if not chosen:
            print(
                f"WARNING: No pilot patient "
                f"for {histology}/{stage}."
            )

        pilot_patients.extend(chosen)

pilot = manifest[
    manifest.case_id.isin(pilot_patients)
].copy()

pilot["record_type"] = "three_omics"

# Add case-linked mutation files when available
mutation_pilot_rows = []

if not mutation.empty:

    for case_id in pilot_patients:

        pool = mutation[
            (mutation.case_id == case_id)
            & (mutation.record_type == "case_linked")
        ].copy()

        if pool.empty:
            continue

        pool["priority"] = (
            pool.workflow.str.contains(
                "mutect2",
                case=False,
                na=False
            ).map({
                True: 0,
                False: 1
            })
        )

        pool = pool.sort_values(
            [
                "priority",
                "file_size_bytes",
                "file_id"
            ],
            kind="stable"
        )

        selected = pool.iloc[0]

        mutation_pilot_rows.append({
            "case_id": case_id,
            "patient_id": "",
            "project": selected.project,
            "histology": "",
            "stage_group": "",
            "modality": "mutation_maf",
            "file_id": selected.file_id,
            "file_name": selected.file_name,
            "file_size_bytes": int(
                selected.file_size_bytes
            ),
            "md5sum": selected.md5sum,
            "workflow": selected.workflow,
            "sample_vials": "",
            "preferred_vial": "",
            "vial_match": False,
            "preference_rank": int(selected.priority),
            "record_type": "mutation_case_linked"
        })

if mutation_pilot_rows:

    pilot = pd.concat(
        [
            pilot,
            pd.DataFrame(mutation_pilot_rows)
        ],
        ignore_index=True
    )

pilot = pilot.drop_duplicates("file_id")

pilot_path = (
    INTERIM
    / "tcga_gdc_pilot_download_manifest.csv"
)

pilot.to_csv(
    pilot_path,
    index=False
)

print(
    f"\nPilot cases: {len(pilot_patients)}; "
    f"pilot files: {len(pilot)}"
)

print(
    f"Pilot total expected size: "
    f"{pilot.file_size_bytes.sum() / (1024 ** 2):.1f} MiB"
)

# Calculate file checksums
def md5_file(path):

    digest = hashlib.md5()

    with open(path, "rb") as handle:

        for chunk in iter(
            lambda: handle.read(
                4 * 1024 * 1024
            ),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


# Download and verify individual GDC files
def download_file(row):

    size = int(row.file_size_bytes)

    if size > MAX_FILE_MB * 1024 ** 2:
        return (
            "skipped_file_limit",
            "file exceeds per-file budget",
            ""
        )

    basename = re.sub(
        r"[^A-Za-z0-9._-]",
        "_",
        str(row.file_name)
    )[-75:]

    target_dir = RAW / str(row.modality)
    target_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    target = (
        target_dir
        / f"{row.file_id}__{basename}"
    )

    expected_hash = (
        str(row.md5sum).lower().strip()
    )

    if (
        len(expected_hash) != 32
        or not re.fullmatch(
            r"[0-9a-f]{32}",
            expected_hash
        )
    ):
        return (
            "missing_checksum",
            "GDC MD5 unavailable",
            ""
        )

    if target.is_file():

        if (
            target.stat().st_size == size
            and md5_file(target) == expected_hash
        ):
            return (
                "cached_verified",
                "",
                str(target)
            )

        target.rename(
            target.with_name(
                target.name
                + f".invalid_{int(time.time())}"
            )
        )

    partial = target.with_name(
        target.name + ".part"
    )

    for attempt in range(1, 4):

        try:

            resume_at = (
                partial.stat().st_size
                if partial.exists()
                else 0
            )

            headers = (
                {
                    "Range": f"bytes={resume_at}-"
                }
                if 0 < resume_at < size
                else {}
            )

            if resume_at >= size and partial.exists():
                partial.unlink()
                resume_at = 0
                headers = {}

            with session.get(
                f"{API}/data/{row.file_id}",
                headers=headers,
                stream=True,
                timeout=(20, 180)
            ) as response:

                response.raise_for_status()

                append = (
                    resume_at > 0
                    and response.status_code == 206
                )

                with open(
                    partial,
                    "ab" if append else "wb"
                ) as handle:

                    for chunk in response.iter_content(
                        chunk_size=2 * 1024 * 1024
                    ):

                        if chunk:
                            handle.write(chunk)

            if partial.stat().st_size != size:
                raise IOError(
                    f"size mismatch: "
                    f"{partial.stat().st_size} != {size}"
                )

            if md5_file(partial) != expected_hash:
                raise IOError(
                    "MD5 checksum mismatch"
                )

            os.replace(
                partial,
                target
            )

            return (
                "verified",
                "",
                str(target)
            )

        except (
            requests.RequestException,
            OSError,
            IOError
        ) as exc:

            message = str(exc)

            if attempt == 3:
                return (
                    "failed",
                    message[:400],
                    ""
                )

            if (
                "MD5" in message
                and partial.exists()
            ):
                partial.unlink()

            time.sleep(2 * attempt)

    return (
        "failed",
        "retry limit exceeded",
        ""
    )


# Inspect real molecular file headers
def inspect_header(path):

    with open(path, "rb") as raw:
        is_gzip = (
            raw.read(2)
            == bytes([0x1f, 0x8b])
        )

    opener = (
        gzip.open
        if is_gzip
        else open
    )

    lines = []

    try:

        with opener(
            path,
            "rt",
            encoding="utf-8-sig",
            errors="replace"
        ) as handle:

            for _ in range(40):

                line = handle.readline()

                if not line:
                    break

                if line.strip():
                    lines.append(
                        line.rstrip("\r\n")
                    )

                if len(lines) >= 8:
                    break

    except (OSError, UnicodeError) as exc:
        return f"HEADER_ERROR: {exc}"

    if not lines:
        return "EMPTY_TEXT_FILE"

    first = next(
        (
            line
            for line in lines
            if not line.startswith("#")
        ),
        lines[0]
    )

    fields = [
        item.strip()[:45]
        for item in first.split("\t")[:9]
    ]

    return (
        f"{len(first.split(chr(9)))} columns; "
        f"first row: {' | '.join(fields)}"
    )[:550]


# Run the pilot download
results = []
remaining = MAX_PILOT_MB * 1024 ** 2

if DOWNLOAD_PILOT:

    print(
        "\nDownloading technical QC pilot "
        "(source MD5 required):"
    )

    for row in pilot.sort_values(
        [
            "modality",
            "case_id",
            "file_id"
        ]
    ).itertuples(index=False):

        if int(row.file_size_bytes) > remaining:

            status = "skipped_total_limit"
            issue = "pilot budget reached"
            path = ""

        else:

            status, issue, path = download_file(
                row
            )

        if status in (
            "verified",
            "cached_verified"
        ):
            remaining -= int(
                row.file_size_bytes
            )

        header = (
            inspect_header(path)
            if path
            else ""
        )

        results.append({
            "case_id": row.case_id,
            "modality": row.modality,
            "file_id": row.file_id,
            "status": status,
            "error": issue,
            "local_path": path,
            "file_schema": header
        })

        print(
            f"{row.modality:15} "
            f"{str(row.case_id)[:13]:13} "
            f"{status:20} "
            f"{header[:115]}"
        )

else:
    print(
        "Pilot download disabled; "
        "manifests were created."
    )

# Save quality-control results
pilot_audit = pd.DataFrame(
    results,
    columns=[
        "case_id",
        "modality",
        "file_id",
        "status",
        "error",
        "local_path",
        "file_schema"
    ]
)

audit_path = (
    RESULTS
    / "tcga_pilot_download_and_format_audit.csv"
)

pilot_audit.to_csv(
    audit_path,
    index=False
)

summary = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "eligible_stage_primary_patients": len(clinical),
    "three_omics_manifest_patients": int(
        cohort.case_id.nunique()
    ),
    "three_omics_manifest_files": len(manifest),
    "three_omics_expected_gib": round(
        float(
            manifest.file_size_bytes.sum()
            / (1024 ** 3)
        ),
        3
    ),
    "masked_maf_luad_lusc_project_filtered_count": len(
        mut_files
    ),
    "masked_maf_verified_case_linked_count": int(
        (
            mutation.record_type == "case_linked"
        ).sum()
    ),
    "masked_maf_multi_case_count": int(
        (
            mutation.record_type == "multi_case"
        ).sum()
    ),
    "pilot_files": len(pilot),
    "pilot_file_results": (
        pilot_audit.status.value_counts().to_dict()
        if not pilot_audit.empty
        else {}
    ),
    "mutation_coverage_status": "pending_MAF_content_QC",
    "molecular_preprocessing_status": "not_started",
    "trained_or_tested_on_real_data": False
}

summary_path = (
    RESULTS
    / "tcga_file_selection_and_pilot_summary.json"
)

summary_path.write_text(
    json.dumps(
        summary,
        indent=2
    ),
    encoding="utf-8"
)

# Final summary
print("\n" + "=" * 74)
print("COHORT AND PILOT SUMMARY")
print("=" * 74)

for key, value in summary.items():
    print(f"{key}: {value}")

print("\nSaved files:")

for path in (
    mutation_path,
    manifest_path,
    gaps_path,
    pilot_path,
    audit_path,
    summary_path
):
    print(" ", path)

print(
    "\nFull-cohort molecular downloads "
    "and mutation coverage remain pending."
)

GDC MOLECULAR FILE SELECTION AND PILOT QUALITY CONTROL
Stage-eligible patients: 1,014; saved molecular files: 8,353

RNA primary-tumor workflows (1006 patients):
workflow
STAR - Counts    1039

CNV primary-tumor workflows (991 patients):
workflow
ABSOLUTE LiftOver    991

METHYLATION primary-tumor workflows (1008 patients):
workflow
SeSAMe Methylation Beta Estimation    1083

Project-filtered masked MAF files: 1,167
Retrieved 1,167/1,167 mutation records

Verified project-linked MAF entries: 1,103
project    record_type  workflow                                            
TCGA-LUAD  case_linked  Aliquot Ensemble Somatic Variant Merging and Masking    560
TCGA-LUSC  case_linked  Aliquot Ensemble Somatic Variant Merging and Masking    543

Strict primary-tumor 3-omics cohort (before molecular QC):
stage_group  Advanced  Early
histology                   
LUAD              106    386
LUSC               90    400
Patients: 982; files: 2,946
Total planned size: 16.83 GiB
Cross-vial selecte

In [6]:

import gzip
import hashlib
import json
import os
import re
import shutil
import sys
import threading
import time
import uuid
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ============================================================
# 1. Project configuration
# ============================================================

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

INTERIM = ROOT / "data" / "interim"
RAW = ROOT / "data" / "raw" / "GDC_full"
RESULTS = ROOT / "results" / "metrics"

PILOT_AUDIT = (
    RESULTS / "tcga_pilot_download_and_format_audit.csv"
)

for folder in (INTERIM, RAW, RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

RUN_DOWNLOADS = True
MAX_WORKERS = 3
MAX_ATTEMPTS = 4
RETAIN_FREE_GIB = 20
REPORT_EVERY = 50

three_file = (
    INTERIM / "tcga_primary_tumor_three_omics_manifest.csv"
)

mutation_file = (
    INTERIM / "tcga_luad_lusc_corrected_mutation_inventory.csv"
)

for path in (three_file, mutation_file):
    if not path.is_file():
        raise FileNotFoundError(
            f"Required Cell 06 output not found: {path}"
        )

three = pd.read_csv(
    three_file,
    dtype=str,
    keep_default_na=False
)

mutations = pd.read_csv(
    mutation_file,
    dtype=str,
    keep_default_na=False
)

required_three = {
    "case_id", "patient_id", "project",
    "histology", "stage_group", "modality",
    "file_id", "file_name", "file_size_bytes",
    "md5sum", "workflow"
}

required_mutation = {
    "case_id", "project", "file_id",
    "file_name", "file_size_bytes",
    "md5sum", "record_type", "workflow"
}

if (
    required_three - set(three.columns)
    or required_mutation - set(mutations.columns)
):
    raise ValueError(
        "The Cell 06 manifest columns are incomplete."
    )

if three["file_id"].duplicated().any():
    raise ValueError(
        "Duplicate molecular file UUIDs detected."
    )

if three.groupby("case_id")["modality"].nunique().ne(3).any():
    raise ValueError(
        "Incomplete patient records in the three-omics manifest."
    )

if set(three.modality) != {
    "rna", "cnv", "methylation"
}:
    raise ValueError(
        "Unexpected modalities in the source manifest."
    )

cohort = three.drop_duplicates("case_id")[
    [
        "case_id",
        "patient_id",
        "project",
        "histology",
        "stage_group"
    ]
].copy()

assert cohort.case_id.is_unique

case_index = cohort.set_index("case_id")

print("=" * 70)
print("NST NSCLC - FULL MULTI-OMICS DATA ACQUISITION")
print("=" * 70)

print("Three-omics candidates:", len(cohort))
print("Source molecular files:", len(three))


# ============================================================
# 2. Select eligible whole-exome mutation files
# ============================================================

mutations = mutations[
    (mutations.record_type == "case_linked")
    & mutations.case_id.isin(set(cohort.case_id))
].copy()

def identify_assay(filename):

    name = str(filename).lower()

    if re.search(
        r"(?:^|[._-])wxs(?:[._-]|$)",
        name
    ):
        return "WXS"

    if re.search(
        r"(?:^|[._-])wgs(?:[._-]|$)",
        name
    ):
        return "WGS"

    if "targeted" in name:
        return "TARGETED"

    return "UNKNOWN"


mutations["assay"] = (
    mutations.file_name.map(identify_assay)
)

print("\nMutation assay availability:")
print(
    mutations.assay.value_counts(
        dropna=False
    ).to_string()
)

# Use a consistent WXS assay for mutation features
wxs = mutations[
    mutations.assay == "WXS"
].copy()

if wxs.empty:
    raise RuntimeError(
        "No WXS mutation files were found. "
        "Review the mutation inventory before proceeding."
    )

wxs = wxs.sort_values(
    ["case_id", "file_id"],
    kind="stable"
)

wxs = wxs.drop_duplicates(
    "case_id",
    keep="first"
)

mutation_rows = []

for row in wxs.itertuples(index=False):

    patient = case_index.loc[row.case_id]

    if str(row.project) != str(patient.project):
        raise RuntimeError(
            f"Mutation project mismatch: {row.case_id}"
        )

    mutation_rows.append({
        "case_id": row.case_id,
        "patient_id": patient.patient_id,
        "project": patient.project,
        "histology": patient.histology,
        "stage_group": patient.stage_group,
        "modality": "mutation_maf",
        "file_id": row.file_id,
        "file_name": row.file_name,
        "file_size_bytes": row.file_size_bytes,
        "md5sum": row.md5sum,
        "workflow": row.workflow,
        "assay": "WXS"
    })


# ============================================================
# 3. Construct the combined molecular manifest
# ============================================================

three["assay"] = ""

manifest = pd.concat(
    [
        three,
        pd.DataFrame(mutation_rows)
    ],
    ignore_index=True,
    sort=False
)

manifest["file_size_bytes"] = pd.to_numeric(
    manifest["file_size_bytes"],
    errors="coerce"
)

if manifest.file_id.duplicated().any():
    raise RuntimeError(
        "Duplicate UUIDs in the combined manifest."
    )

if (
    manifest.file_size_bytes.isna().any()
    or manifest.file_size_bytes.le(0).any()
):
    raise RuntimeError(
        "Invalid file sizes detected."
    )

if not manifest.md5sum.astype(str).str.fullmatch(
    r"[0-9a-fA-F]{32}"
).all():
    raise RuntimeError(
        "Invalid or missing MD5 checksums detected."
    )

for file_id in manifest.file_id:
    uuid.UUID(str(file_id))

manifest["file_size_bytes"] = (
    manifest.file_size_bytes.astype("int64")
)

manifest_path = (
    INTERIM / "tcga_full_four_omics_download_manifest.csv"
)

print("\n" + "=" * 70)
print("FULL-COHORT DOWNLOAD PLAN")
print("=" * 70)

print(f"Three-omics patients: {len(cohort):,}")
print(f"WXS mutation candidates: {len(mutation_rows):,}")
print(f"Total selected files: {len(manifest):,}")

print(
    f"Total expected download: "
    f"{manifest.file_size_bytes.sum() / (1024 ** 3):.2f} GiB"
)

candidate_ids = set(wxs.case_id)

candidate_cohort = cohort[
    cohort.case_id.isin(candidate_ids)
]

print("\nPotential four-omics cohort (metadata only):")

print(
    pd.crosstab(
        candidate_cohort.histology,
        candidate_cohort.stage_group
    ).to_string()
)


# ============================================================
# 4. Configure local file storage and pilot reuse
# ============================================================

def output_path(row):

    safe_name = re.sub(
        r"[^A-Za-z0-9._-]",
        "_",
        str(row["file_name"])
    )[-105:]

    return (
        RAW
        / str(row["modality"])
        / (str(row["file_id"]) + "__" + safe_name)
    )


manifest["local_path"] = manifest.apply(
    lambda row: str(output_path(row)),
    axis=1
)

manifest.to_csv(
    manifest_path,
    index=False
)

pilot_map = {}

if PILOT_AUDIT.exists():

    previous = pd.read_csv(
        PILOT_AUDIT,
        dtype=str,
        keep_default_na=False
    )

    for row in previous.itertuples(index=False):

        if (
            row.status in {
                "verified",
                "cached_verified"
            }
            and row.local_path
        ):
            pilot_map[str(row.file_id)] = Path(
                row.local_path
            )


# ============================================================
# 5. Check available storage
# ============================================================

if RUN_DOWNLOADS:

    free_bytes = shutil.disk_usage(ROOT).free

    existing_bytes = sum(
        int(row.file_size_bytes)
        for row in manifest.itertuples(index=False)
        if (
            Path(row.local_path).is_file()
            and Path(row.local_path).stat().st_size
            == int(row.file_size_bytes)
        )
    )

    outstanding = max(
        0,
        int(manifest.file_size_bytes.sum())
        - existing_bytes
    )

    required_free = (
        RETAIN_FREE_GIB * (1024 ** 3)
    )

    if free_bytes - outstanding < required_free:
        raise RuntimeError(
            "Insufficient disk space for molecular files "
            "and the reserved free-space margin."
        )

    print(
        f"\nAvailable storage: "
        f"{free_bytes / (1024 ** 3):.2f} GiB"
    )

    print(
        "Download mode: resumable with MD5 verification"
    )


# ============================================================
# 6. Configure worker-specific HTTP sessions
# ============================================================

_thread_local = threading.local()


def worker_session():

    if not hasattr(_thread_local, "session"):

        session = requests.Session()

        retry = Retry(
            total=2,
            connect=2,
            read=1,
            backoff_factor=1.2,
            status_forcelist=[
                429, 500, 502, 503, 504
            ],
            allowed_methods=frozenset(["GET"])
        )

        session.mount(
            "https://",
            HTTPAdapter(max_retries=retry)
        )

        session.headers.update({
            "Accept-Encoding": "identity",
            "User-Agent": "NST-NSCLC-Research/1.0"
        })

        _thread_local.session = session

    return _thread_local.session


# ============================================================
# 7. File integrity functions
# ============================================================

def md5_digest(path):

    digest = hashlib.md5()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(
                4 * 1024 * 1024
            ),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


def valid_file(path, size, checksum):

    return (
        path.is_file()
        and path.stat().st_size == size
        and md5_digest(path).lower()
        == checksum.lower()
    )


# ============================================================
# 8. Resumable GDC downloader
# ============================================================

def download_one(row):

    target = Path(row["local_path"])

    target.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    expected_size = int(
        row["file_size_bytes"]
    )

    expected_md5 = str(
        row["md5sum"]
    ).lower()

    file_id = str(
        row["file_id"]
    )

    result = {
        "case_id": str(row["case_id"]),
        "modality": str(row["modality"]),
        "file_id": file_id,
        "local_path": str(target),
        "file_size_bytes": expected_size,
        "status": "failed",
        "error": ""
    }

    try:

        # Verify previously downloaded files
        if valid_file(
            target,
            expected_size,
            expected_md5
        ):
            result["status"] = "cached_verified"
            return result

        # Preserve corrupted files for inspection
        if target.exists():

            invalid_path = target.with_name(
                target.name
                + ".invalid_"
                + uuid.uuid4().hex[:8]
            )

            target.rename(invalid_path)

        # Reuse verified pilot files
        pilot = pilot_map.get(file_id)

        if (
            pilot
            and pilot.is_file()
            and valid_file(
                pilot,
                expected_size,
                expected_md5
            )
        ):

            try:
                os.link(
                    str(pilot),
                    str(target)
                )

            except OSError:
                shutil.copy2(
                    pilot,
                    target
                )

            if valid_file(
                target,
                expected_size,
                expected_md5
            ):
                result["status"] = (
                    "pilot_reused_verified"
                )
                return result

            target.unlink(missing_ok=True)

        partial = target.with_name(
            target.name + ".part"
        )

        # Download with bounded retries
        for attempt in range(
            1,
            MAX_ATTEMPTS + 1
        ):

            try:

                existing = (
                    partial.stat().st_size
                    if partial.exists()
                    else 0
                )

                # Verify a previously completed partial file
                if (
                    existing == expected_size
                    and valid_file(
                        partial,
                        expected_size,
                        expected_md5
                    )
                ):

                    os.replace(
                        partial,
                        target
                    )

                    result["status"] = (
                        "resumed_verified"
                    )

                    return result

                if existing >= expected_size:

                    partial.unlink(
                        missing_ok=True
                    )

                    existing = 0

                headers = (
                    {
                        "Range": f"bytes={existing}-"
                    }
                    if existing
                    else {}
                )

                response_url = (
                    "https://api.gdc.cancer.gov/data/"
                    + file_id
                )

                with worker_session().get(
                    response_url,
                    headers=headers,
                    stream=True,
                    timeout=(20, 180)
                ) as response:

                    if response.status_code == 416:
                        raise IOError(
                            "HTTP 416; partial file requires restart"
                        )

                    response.raise_for_status()

                    if (
                        existing
                        and response.status_code == 206
                    ):

                        content_range = (
                            response.headers.get(
                                "Content-Range",
                                ""
                            )
                        )

                        if not content_range.startswith(
                            f"bytes {existing}-"
                        ):
                            raise IOError(
                                "Unexpected Content-Range: "
                                + content_range
                            )

                        write_mode = "ab"

                    else:
                        write_mode = "wb"

                    with open(
                        partial,
                        write_mode
                    ) as output:

                        for chunk in response.iter_content(
                            chunk_size=2 * 1024 * 1024
                        ):

                            if chunk:
                                output.write(chunk)

                downloaded = (
                    partial.stat().st_size
                )

                if downloaded != expected_size:
                    raise IOError(
                        f"Incomplete transfer: "
                        f"{downloaded}/{expected_size}"
                    )

                actual_md5 = md5_digest(
                    partial
                )

                if actual_md5.lower() != expected_md5:

                    partial.unlink(
                        missing_ok=True
                    )

                    raise IOError(
                        "MD5 mismatch; partial file discarded"
                    )

                os.replace(
                    partial,
                    target
                )

                result["status"] = (
                    "downloaded_verified"
                )

                return result

            except (
                requests.RequestException,
                OSError
            ) as error:

                result["error"] = (
                    f"attempt {attempt}: "
                    f"{str(error)[:260]}"
                )

                if (
                    "416" in str(error)
                    or "Content-Range" in str(error)
                ):
                    partial.unlink(
                        missing_ok=True
                    )

                if attempt < MAX_ATTEMPTS:
                    time.sleep(
                        min(12, 2 * attempt)
                    )

    except Exception as error:

        result["error"] = (
            "File preparation failed: "
            + str(error)[:350]
        )

    return result


# ============================================================
# 9. Execute full cohort download
# ============================================================

result_rows = []

if RUN_DOWNLOADS:

    print("\nStarting molecular file downloads...")

    event_path = (
        RESULTS / "tcga_gdc_download_events.jsonl"
    )

    records = manifest.to_dict("records")

    start_time = time.monotonic()

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = {
            executor.submit(
                download_one,
                record
            ): record["file_id"]
            for record in records
        }

        for number, future in enumerate(
            as_completed(futures),
            start=1
        ):

            outcome = future.result()

            result_rows.append(outcome)

            # Persist progress after each completed file
            event = {
                **outcome,
                "recorded_utc": datetime.now(
                    timezone.utc
                ).isoformat()
            }

            with open(
                event_path,
                "a",
                encoding="utf-8"
            ) as handle:

                handle.write(
                    json.dumps(event)
                    + "\n"
                )

            if (
                number % REPORT_EVERY == 0
                or number == len(records)
            ):

                verified_count = sum(
                    1
                    for item in result_rows
                    if item["status"].endswith(
                        "verified"
                    )
                )

                failed_count = (
                    number - verified_count
                )

                elapsed = int(
                    time.monotonic()
                    - start_time
                )

                print(
                    f"{number:,}/{len(records):,} processed | "
                    f"verified {verified_count:,} | "
                    f"failed {failed_count:,} | "
                    f"elapsed {elapsed} s",
                    flush=True
                )

else:

    print(
        "Download disabled. "
        "Generating manifests only."
    )

    for record in manifest.to_dict("records"):

        result_rows.append({
            "case_id": record["case_id"],
            "modality": record["modality"],
            "file_id": record["file_id"],
            "local_path": record["local_path"],
            "file_size_bytes": int(
                record["file_size_bytes"]
            ),
            "status": "not_downloaded",
            "error": ""
        })


# ============================================================
# 10. Save verified download records
# ============================================================

status = pd.DataFrame(
    result_rows
)

status_path = (
    RESULTS / "tcga_full_download_status.csv"
)

status.to_csv(
    status_path,
    index=False
)

verified = status[
    status.status.str.endswith(
        "verified"
    )
].copy()

print("\nDownload status counts:")

print(
    status.status.value_counts().to_string()
)


# ============================================================
# 11. Detect compressed and uncompressed file formats
# ============================================================

def file_opener(path):

    with open(path, "rb") as handle:
        signature = handle.read(2)

    if signature == b"\x1f\x8b":
        return gzip.open

    return open


# ============================================================
# 12. Verify molecular file headers
# ============================================================

def inspect_schema(row):

    path = Path(
        row["local_path"]
    )

    modality = row["modality"]

    if not str(
        row["status"]
    ).endswith("verified"):

        return "not_available", ""

    if modality == "mutation_maf":
        return "mutation_barcode_review", ""

    try:

        with file_opener(path)(
            path,
            "rt",
            encoding="utf-8-sig",
            errors="replace"
        ) as handle:

            first = ""

            for _ in range(30):

                first = handle.readline().rstrip(
                    "\r\n"
                )

                if not first:
                    break

                if (
                    first.startswith("#")
                    or not first.strip()
                ):
                    continue

                break

        columns = first.split("\t")

        if modality == "rna":

            passed = {
                "gene_id",
                "unstranded",
                "tpm_unstranded"
            }.issubset(columns)

        elif modality == "cnv":

            passed = {
                "gene_id",
                "copy_number"
            }.issubset(columns)

        elif modality == "methylation":

            passed = (
                len(columns) == 2
                and columns[0].startswith(
                    ("cg", "ch", "rs")
                )
            )

        else:
            passed = False

        result = (
            "schema_pass"
            if passed
            else "schema_review"
        )

        header = " | ".join(
            columns[:9]
        )[:250]

        return result, header

    except (
        OSError,
        UnicodeError
    ) as error:

        return (
            "schema_error",
            str(error)[:180]
        )


schema_rows = []

non_mutation = status[
    status.modality != "mutation_maf"
]

for record in non_mutation.to_dict("records"):

    schema_state, header = inspect_schema(
        record
    )

    schema_rows.append({
        "file_id": record["file_id"],
        "case_id": record["case_id"],
        "modality": record["modality"],
        "schema_status": schema_state,
        "header_excerpt": header
    })

schema_df = pd.DataFrame(
    schema_rows
)

schema_path = (
    RESULTS / "tcga_molecular_schema_qc.csv"
)

schema_df.to_csv(
    schema_path,
    index=False
)


# ============================================================
# 13. Verify mutation sample barcodes
# ============================================================

barcode_pattern = re.compile(
    r"^(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})-(\d{2})([A-Z])?",
    re.IGNORECASE
)


def verify_maf(record):

    result = {
        "file_id": record["file_id"],
        "case_id": record["case_id"],
        "maf_status": "unavailable",
        "total_variant_rows": 0,
        "matched_primary_rows": 0,
        "other_patient_rows": 0,
        "nonprimary_rows": 0,
        "invalid_barcode_rows": 0,
        "error": ""
    }

    if not str(
        record["status"]
    ).endswith("verified"):
        return result

    path = Path(
        record["local_path"]
    )

    patient_id = str(
        case_index.loc[
            record["case_id"],
            "patient_id"
        ]
    ).upper()

    try:

        with file_opener(path)(
            path,
            "rt",
            encoding="utf-8-sig",
            errors="replace"
        ) as stream:

            chunks = pd.read_csv(
                stream,
                sep="\t",
                comment="#",
                usecols=lambda column: (
                    column == "Tumor_Sample_Barcode"
                ),
                dtype=str,
                chunksize=25000,
                low_memory=False
            )

            found_field = False

            for chunk in chunks:

                if (
                    "Tumor_Sample_Barcode"
                    not in chunk.columns
                ):
                    continue

                found_field = True

                barcodes = chunk[
                    "Tumor_Sample_Barcode"
                ].fillna("").astype(str)

                result["total_variant_rows"] += (
                    len(barcodes)
                )

                for barcode, count in (
                    barcodes.value_counts().items()
                ):

                    match = barcode_pattern.match(
                        barcode.upper().strip()
                    )

                    if match is None:

                        result[
                            "invalid_barcode_rows"
                        ] += int(count)

                    elif (
                        match.group(1).upper()
                        != patient_id
                    ):

                        result[
                            "other_patient_rows"
                        ] += int(count)

                    elif match.group(2) != "01":

                        result[
                            "nonprimary_rows"
                        ] += int(count)

                    else:

                        result[
                            "matched_primary_rows"
                        ] += int(count)

        if not found_field:

            result["maf_status"] = (
                "missing_barcode_field"
            )

        elif (
            result["other_patient_rows"]
            or result["nonprimary_rows"]
            or result["invalid_barcode_rows"]
        ):

            result["maf_status"] = (
                "barcode_review"
            )

        elif result["matched_primary_rows"] > 0:

            result["maf_status"] = (
                "verified_primary_case"
            )

        else:

            result["maf_status"] = (
                "empty_or_unverifiable"
            )

    except Exception as error:

        result["maf_status"] = (
            "parse_failed"
        )

        result["error"] = str(error)[:280]

    return result


maf_rows = []

mutation_status = status[
    status.modality == "mutation_maf"
]

for record in mutation_status.to_dict("records"):

    maf_rows.append(
        verify_maf(record)
    )

maf_qc = pd.DataFrame(
    maf_rows,
    columns=[
        "file_id",
        "case_id",
        "maf_status",
        "total_variant_rows",
        "matched_primary_rows",
        "other_patient_rows",
        "nonprimary_rows",
        "invalid_barcode_rows",
        "error"
    ]
)

maf_path = (
    RESULTS / "tcga_maf_primary_barcode_qc.csv"
)

maf_qc.to_csv(
    maf_path,
    index=False
)


# ============================================================
# 14. Construct the raw-data QC cohort
# ============================================================

clean_file_ids = set(
    schema_df.loc[
        schema_df.schema_status == "schema_pass",
        "file_id"
    ]
)

three_ready = set()

three_manifest = manifest[
    manifest.modality != "mutation_maf"
]

for case_id, group in three_manifest.groupby(
    "case_id"
):

    if (
        group.modality.nunique() == 3
        and set(group.file_id).issubset(
            clean_file_ids
        )
    ):
        three_ready.add(case_id)


maf_ready = set(
    maf_qc.loc[
        maf_qc.maf_status == "verified_primary_case",
        "case_id"
    ]
)

four_ready = (
    three_ready & maf_ready
)

cohort_out = cohort.copy()

cohort_out["rna_cnv_methylation_qc_pass"] = (
    cohort_out.case_id.isin(
        three_ready
    )
)

cohort_out["mutation_maf_barcode_qc_pass"] = (
    cohort_out.case_id.isin(
        maf_ready
    )
)

cohort_out["strict_four_omics_qc_pass"] = (
    cohort_out.case_id.isin(
        four_ready
    )
)

cohort_out["mutation_missing_is_not_wildtype"] = (
    ~cohort_out.case_id.isin(
        maf_ready
    )
)

cohort_path = (
    INTERIM / "tcga_raw_qc_cohort.csv"
)

cohort_out.to_csv(
    cohort_path,
    index=False
)

verified_files = manifest.merge(
    status[
        ["file_id", "status"]
    ],
    on="file_id",
    how="left"
)

verified_files = verified_files[
    verified_files.status.str.endswith(
        "verified",
        na=False
    )
].copy()

verified_files_path = (
    INTERIM / "tcga_verified_raw_files.csv"
)

verified_files.to_csv(
    verified_files_path,
    index=False
)


# ============================================================
# 15. Save reproducible QC summary
# ============================================================

summary = {
    "generated_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "three_omics_initial_patients": int(
        len(cohort)
    ),

    "mutation_wxs_metadata_candidates": int(
        len(mutation_rows)
    ),

    "files_requested": int(
        len(manifest)
    ),

    "files_verified_by_md5": int(
        len(verified)
    ),

    "files_failed_or_missing": int(
        len(status) - len(verified)
    ),

    "download_status_counts": (
        status.status.value_counts().to_dict()
    ),

    "schema_status_counts": (
        schema_df.schema_status.value_counts().to_dict()
    ),

    "maf_barcode_status_counts": (
        maf_qc.maf_status.value_counts().to_dict()
    ),

    "three_omics_raw_qc_patients": int(
        len(three_ready)
    ),

    "four_omics_raw_qc_patients": int(
        len(four_ready)
    ),

    "external_test_set_created": False,

    "molecular_feature_selection_done": False,

    "mutation_missingness_policy": (
        "Missing assay is not a negative mutation call"
    )
}

summary_path = (
    RESULTS / "tcga_full_download_and_qc_summary.json"
)

summary_path.write_text(
    json.dumps(
        summary,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 16. Final verification report
# ============================================================

print("\n" + "=" * 70)
print("REAL TCGA DOWNLOAD AND RAW QC SUMMARY")
print("=" * 70)

for key in (
    "files_requested",
    "files_verified_by_md5",
    "files_failed_or_missing",
    "mutation_wxs_metadata_candidates",
    "three_omics_raw_qc_patients",
    "four_omics_raw_qc_patients"
):

    print(
        f"{key:40}: {summary[key]}"
    )

print("\nSchema QC:")

print(
    schema_df.schema_status.value_counts().to_string()
)

print("\nMAF barcode QC:")

print(
    maf_qc.maf_status.value_counts().to_string()
)

print("\nFour-omics sample groups:")

selected_cohort = cohort_out[
    cohort_out.strict_four_omics_qc_pass
]

print(
    pd.crosstab(
        selected_cohort.histology,
        selected_cohort.stage_group
    ).to_string()
)

print("\nSaved audit files:")

for path in (
    manifest_path,
    status_path,
    schema_path,
    maf_path,
    cohort_path,
    verified_files_path,
    summary_path
):

    print(" -", path)

print(
    "\nNext: Molecular matrix construction, "
    "probe annotation and leakage-safe preprocessing."
)

NST NSCLC - FULL MULTI-OMICS DATA ACQUISITION
Three-omics candidates: 982
Source molecular files: 2946

Mutation assay availability:
assay
WXS    1068

FULL-COHORT DOWNLOAD PLAN
Three-omics patients: 982
WXS mutation candidates: 960
Total selected files: 3,906
Total expected download: 16.92 GiB

Potential four-omics cohort (metadata only):
stage_group  Advanced  Early
histology                   
LUAD              105    379
LUSC               87    389

Available storage: 161.47 GiB
Download mode: resumable with MD5 verification

Starting molecular file downloads...
50/3,906 processed | verified 50 | failed 0 | elapsed 112 s
100/3,906 processed | verified 100 | failed 0 | elapsed 212 s
150/3,906 processed | verified 150 | failed 0 | elapsed 300 s
200/3,906 processed | verified 200 | failed 0 | elapsed 390 s
250/3,906 processed | verified 250 | failed 0 | elapsed 485 s
300/3,906 processed | verified 300 | failed 0 | elapsed 596 s
350/3,906 processed | verified 350 | failed 0 | elapsed 

In [7]:

import gzip
import hashlib
import json
import os
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from scipy import sparse
from sklearn.model_selection import train_test_split

# Project configuration
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
INTERIM = ROOT / "data" / "interim"
METRICS = ROOT / "results" / "metrics"
SPLITS = ROOT / "data" / "splits"
PROCESSED = ROOT / "data" / "processed" / "gene_level"
REFERENCE = ROOT / "data" / "raw" / "references"

EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError("Select the NST NSCLC Research (Python 3.12) kernel.")

for directory in (SPLITS, PROCESSED, REFERENCE):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
STAGE_VERSION = "gene_level_stage08_v1"
CHECKPOINT_EVERY = 25

# Load QC-approved manifests
cohort = pd.read_csv(
    INTERIM / "tcga_raw_qc_cohort.csv",
    dtype=str,
    keep_default_na=False
)

manifest = pd.read_csv(
    INTERIM / "tcga_full_four_omics_download_manifest.csv",
    dtype=str,
    keep_default_na=False
)

status = pd.read_csv(
    METRICS / "tcga_full_download_status.csv",
    dtype=str,
    keep_default_na=False
)

mafq = pd.read_csv(
    METRICS / "tcga_maf_primary_barcode_qc.csv",
    dtype=str,
    keep_default_na=False
)

schema = pd.read_csv(
    METRICS / "tcga_molecular_schema_qc.csv",
    dtype=str,
    keep_default_na=False
)

eligible = cohort.loc[
    cohort["strict_four_omics_qc_pass"].str.lower().eq("true")
    & cohort["stage_group"].isin(["Early", "Advanced"])
    & cohort["histology"].isin(["LUAD", "LUSC"])
].copy().sort_values("case_id").reset_index(drop=True)

if not eligible["case_id"].is_unique or len(eligible) < 100:
    raise RuntimeError("Invalid strict four-omics cohort.")

verified_ids = set(
    status.loc[
        status["status"].str.endswith("verified"),
        "file_id"
    ]
)

verified_maf_ids = set(
    mafq.loc[
        mafq["maf_status"].eq("verified_primary_case"),
        "file_id"
    ]
)

verified_schema_ids = set(
    schema.loc[
        schema["schema_status"].eq("schema_pass"),
        "file_id"
    ]
)

chosen = manifest[
    manifest["case_id"].isin(eligible["case_id"])
].copy()

required_modalities = {
    "rna",
    "cnv",
    "methylation",
    "mutation_maf"
}

for case_id, group in chosen.groupby("case_id"):

    if (
        set(group["modality"]) != required_modalities
        or len(group) != 4
    ):
        raise RuntimeError(
            f"Incomplete four-omics manifest: {case_id}"
        )

if chosen["case_id"].nunique() != len(eligible):
    raise RuntimeError("Manifest and cohort patient counts differ.")

if not set(chosen["file_id"]).issubset(verified_ids):
    raise RuntimeError("Selected files include unverified downloads.")

if not set(
    chosen.loc[
        chosen["modality"].eq("mutation_maf"),
        "file_id"
    ]
).issubset(verified_maf_ids):
    raise RuntimeError("Selected mutation file failed barcode QC.")

if not set(
    chosen.loc[
        ~chosen["modality"].eq("mutation_maf"),
        "file_id"
    ]
).issubset(verified_schema_ids):
    raise RuntimeError("Selected molecular file failed schema QC.")

if (
    chosen["local_path"].eq("").any()
    or not chosen["local_path"].map(
        lambda path: Path(path).is_file()
    ).all()
):
    raise FileNotFoundError("Verified molecular files are missing.")

case_order = eligible["case_id"].tolist()

case_to_row = {
    case_id: i
    for i, case_id in enumerate(case_order)
}

files_by_case = chosen.set_index(
    ["case_id", "modality"]
)

cohort_signature = hashlib.sha256(
    "\n".join(case_order).encode()
).hexdigest()

file_signature = hashlib.sha256(
    "\n".join(
        sorted(
            chosen["file_id"].astype(str)
            + ":"
            + chosen["md5sum"].astype(str)
        )
    ).encode()
).hexdigest()

print("=" * 68)
print("STAGE 08 - FOUR-OMICS GENE-LEVEL MATRIX CONSTRUCTION")
print("=" * 68)

print("QC-passed patients:", len(eligible))
print("Verified molecular files:", len(chosen))


# Lock patient-level train, validation and test splits
split_path = SPLITS / "locked_four_omics_v1.csv"

if split_path.exists():

    split = pd.read_csv(
        split_path,
        dtype=str
    )

    comparison = eligible[
        ["case_id", "histology", "stage_group"]
    ].merge(
        split,
        on="case_id",
        how="outer",
        suffixes=("_current", "_saved"),
        indicator=True
    )

    if (
        not comparison["_merge"].eq("both").all()
        or not (
            comparison["histology_current"]
            == comparison["histology_saved"]
        ).all()
        or not (
            comparison["stage_group_current"]
            == comparison["stage_group_saved"]
        ).all()
    ):
        raise RuntimeError(
            "The existing locked split belongs to a different cohort."
        )

    if set(split["split"]) != {
        "train", "validation", "test"
    }:
        raise RuntimeError("Invalid split assignments.")

else:

    stratification = (
        eligible["histology"]
        + "_"
        + eligible["stage_group"]
    )

    development, test = train_test_split(
        eligible.index.to_numpy(),
        test_size=0.15,
        stratify=stratification,
        random_state=SEED
    )

    train, validation = train_test_split(
        development,
        test_size=0.15 / 0.85,
        stratify=stratification.iloc[development],
        random_state=SEED + 1
    )

    assignments = np.empty(
        len(eligible),
        dtype=object
    )

    assignments[train] = "train"
    assignments[validation] = "validation"
    assignments[test] = "test"

    split = eligible[
        [
            "case_id",
            "patient_id",
            "histology",
            "stage_group"
        ]
    ].copy()

    split["split"] = assignments

    split.to_csv(
        split_path,
        index=False
    )

print("\nLocked patient splits:")

print(
    pd.crosstab(
        [split.histology, split.stage_group],
        split.split
    ).to_string()
)


# Read compressed or uncompressed GDC files
def file_opener(path):

    with open(path, "rb") as stream:
        compressed = stream.read(2) == b"\x1f\x8b"

    return gzip.open if compressed else open


def read_table(path, columns):

    with file_opener(path)(
        path,
        "rt",
        encoding="utf-8-sig",
        errors="replace"
    ) as stream:

        frame = pd.read_csv(
            stream,
            sep="\t",
            comment="#",
            dtype=str,
            usecols=lambda column: column in columns,
            low_memory=False
        )

    missing = set(columns) - set(frame.columns)

    if missing:
        raise ValueError(
            f"Missing columns {missing} in {path}"
        )

    return frame


# Establish the protein-coding gene reference
first_train_case = sorted(
    split.loc[
        split["split"].eq("train"),
        "case_id"
    ]
)[0]

first_rna = Path(
    files_by_case.loc[
        (first_train_case, "rna"),
        "local_path"
    ]
)

reference_rna = read_table(
    first_rna,
    [
        "gene_id",
        "gene_name",
        "gene_type",
        "unstranded",
        "tpm_unstranded"
    ]
)

reference_rna = reference_rna[
    reference_rna["gene_type"]
    .str.lower()
    .eq("protein_coding")
].copy()

reference_rna["ensembl_id"] = (
    reference_rna["gene_id"].str.replace(
        r"\.\d+$",
        "",
        regex=True
    )
)

reference_rna = reference_rna[
    reference_rna["ensembl_id"].str.fullmatch(
        r"ENSG\d+",
        na=False
    )
].copy()

collisions = reference_rna.groupby(
    "ensembl_id"
)["gene_name"].nunique()

if (collisions > 1).any():
    raise RuntimeError(
        "Conflicting symbols for identical Ensembl gene IDs."
    )

reference_rna = (
    reference_rna
    .drop_duplicates("ensembl_id")
    .sort_values("ensembl_id")
    .reset_index(drop=True)
)

gene_reference = reference_rna[
    ["ensembl_id", "gene_name"]
].rename(
    columns={"gene_name": "gene_symbol"}
)

if len(gene_reference) < 10000:
    raise RuntimeError(
        "Unexpectedly small protein-coding gene reference."
    )

gene_ids = gene_reference["ensembl_id"].tolist()

unique_symbols = gene_reference.loc[
    ~gene_reference["gene_symbol"].duplicated(
        keep=False
    )
    & gene_reference["gene_symbol"].str.len().gt(0)
]

symbol_to_index = {
    symbol: int(index)
    for index, symbol in zip(
        unique_symbols.index,
        unique_symbols["gene_symbol"]
    )
}

reference_path = PROCESSED / "gene_reference.csv"

gene_reference.to_csv(
    reference_path,
    index=False
)

print("\nProtein-coding genes:", len(gene_ids))
print("Unambiguous gene symbols:", len(symbol_to_index))


# Download the manufacturer methylation annotation
annotation_sources = [
    (
        REFERENCE / "humanmethylation450_15017482_v1-2.csv",
        "https://webdata.illumina.com/downloads/productfiles/humanmethylation450/humanmethylation450_15017482_v1-2.csv"
    ),
    (
        REFERENCE / "GPL13534_HumanMethylation450_v1.1.csv.gz",
        "https://ftp.ncbi.nlm.nih.gov/geo/platforms/GPL13nnn/GPL13534/suppl/GPL13534_HumanMethylation450_15017482_v.1.1.csv.gz"
    )
]

annotation_path = None
annotation_source = None
annotation_error = ""

for candidate, source in annotation_sources:

    if not candidate.exists():

        temporary = candidate.with_name(
            candidate.name + ".part"
        )

        try:

            with requests.get(
                source,
                timeout=(20, 120),
                stream=True
            ) as response:

                response.raise_for_status()

                content_length = int(
                    response.headers.get(
                        "Content-Length",
                        "0"
                    ) or 0
                )

                if content_length > 350 * 1024 ** 2:
                    raise RuntimeError(
                        "Annotation response is unexpectedly large."
                    )

                with open(temporary, "wb") as output:

                    total = 0

                    for piece in response.iter_content(
                        chunk_size=2 * 1024 ** 2
                    ):

                        if piece:

                            total += len(piece)

                            if total > 350 * 1024 ** 2:
                                raise RuntimeError(
                                    "Annotation download exceeded limit."
                                )

                            output.write(piece)

            open_reference = (
                gzip.open
                if candidate.suffix == ".gz"
                else open
            )

            with open_reference(
                temporary,
                "rb"
            ) as test:

                preview = test.read(
                    65536
                ).decode(
                    "utf-8-sig",
                    errors="replace"
                )

            if (
                "[Assay]" not in preview
                and "IlmnID" not in preview
            ):
                raise RuntimeError(
                    "Invalid Illumina annotation response."
                )

            os.replace(
                temporary,
                candidate
            )

        except (
            requests.RequestException,
            OSError,
            ValueError,
            RuntimeError
        ) as exc:

            annotation_error += (
                f"{candidate.name}: {exc}; "
            )

            temporary.unlink(
                missing_ok=True
            )

    if candidate.exists():

        annotation_path = candidate
        annotation_source = source
        break


# Construct a promoter-specific CpG-to-gene mapping
mapping = {}
annotation_valid = False
ambiguous_annotation = 0

if annotation_path is not None:

    try:

        with file_opener(annotation_path)(
            annotation_path,
            "rt",
            encoding="utf-8-sig",
            errors="replace"
        ) as stream:

            start = None

            for line_index, line in enumerate(stream):

                if line.strip().startswith("[Assay]"):
                    start = line_index + 1
                    break

                if line.lstrip().startswith("IlmnID,"):
                    start = line_index
                    break

                if line_index > 35:
                    break

        if start is None:
            raise ValueError(
                "Annotation header not found."
            )

        columns = {
            "IlmnID",
            "UCSC_RefGene_Name",
            "UCSC_RefGene_Group"
        }

        chunks = pd.read_csv(
            annotation_path,
            skiprows=start,
            usecols=lambda col: col in columns,
            dtype=str,
            chunksize=50000,
            on_bad_lines="skip"
        )

        for chunk in chunks:

            if not columns.issubset(chunk.columns):
                raise ValueError(
                    "Required manufacturer annotation columns missing."
                )

            rows = chunk[
                [
                    "IlmnID",
                    "UCSC_RefGene_Name",
                    "UCSC_RefGene_Group"
                ]
            ].itertuples(
                index=False,
                name=None
            )

            for probe, names, groups in rows:

                if (
                    not isinstance(probe, str)
                    or not probe.startswith("cg")
                ):
                    continue

                gene_parts = (
                    str(names).split(";")
                    if isinstance(names, str)
                    else []
                )

                group_parts = (
                    str(groups).split(";")
                    if isinstance(groups, str)
                    else []
                )

                if len(gene_parts) != len(group_parts):
                    ambiguous_annotation += 1
                    continue

                gene_indices = {
                    symbol_to_index[gene.strip()]
                    for gene, region in zip(
                        gene_parts,
                        group_parts
                    )
                    if region.strip() in {
                        "TSS200",
                        "TSS1500"
                    }
                    and gene.strip() in symbol_to_index
                }

                if gene_indices:

                    mapping.setdefault(
                        probe,
                        set()
                    ).update(gene_indices)

        if len(mapping) < 10000:
            raise ValueError(
                "Insufficient promoter CpG-to-gene mappings."
            )

        annotation_valid = True

    except (
        OSError,
        ValueError,
        pd.errors.ParserError
    ) as exc:

        annotation_error = str(exc)
        mapping.clear()


# Create the sparse promoter annotation matrix
probe_gene = None
probe_ids = []

if annotation_valid:

    probe_ids = sorted(mapping)

    probe_rows = []
    gene_columns = []

    for probe_index, probe in enumerate(probe_ids):

        for gene_index in mapping[probe]:

            probe_rows.append(
                probe_index
            )

            gene_columns.append(
                gene_index
            )

    probe_gene = sparse.csr_matrix(
        (
            np.ones(
                len(probe_rows),
                dtype=np.float32
            ),
            (
                probe_rows,
                gene_columns
            )
        ),
        shape=(
            len(probe_ids),
            len(gene_ids)
        ),
        dtype=np.float32
    )

    probe_gene.sum_duplicates()

    print(
        f"\nPromoter CpG probes: {len(probe_ids):,}"
    )

    print(
        f"Probe-gene links: {probe_gene.nnz:,}"
    )

else:

    print(
        "\nPromoter annotation unavailable."
    )

    print(
        "Methylation aggregation will remain pending."
    )

    print(
        "Reason:",
        annotation_error[:250]
    )


# Prepare memory-mapped molecular matrices
matrix_specs = {
    "rna_tpm": np.float32,
    "rna_counts": np.float32,
    "cnv_absolute": np.float32,
    "mutation_nonsynonymous": np.uint8
}

if annotation_valid:

    matrix_specs[
        "methylation_promoter_beta"
    ] = np.float32

    matrix_specs[
        "methylation_promoter_nprobes"
    ] = np.uint16

arrays = {}

for name, dtype in matrix_specs.items():

    path = PROCESSED / (name + ".npy")

    shape = (
        len(case_order),
        len(gene_ids)
    )

    if path.exists():

        array = np.load(
            path,
            mmap_mode="r+"
        )

        if (
            array.shape != shape
            or array.dtype != np.dtype(dtype)
        ):
            raise RuntimeError(
                f"Existing array schema changed: {path}"
            )

    else:

        array = np.lib.format.open_memmap(
            path,
            mode="w+",
            dtype=dtype,
            shape=shape
        )

        array[:] = (
            np.nan
            if np.issubdtype(
                dtype,
                np.floating
            )
            else 0
        )

        array.flush()

    arrays[name] = array


# Initialize resumable processing checkpoints
signature = hashlib.sha256(
    (
        STAGE_VERSION
        + cohort_signature
        + file_signature
        + "\n".join(gene_ids)
    ).encode()
).hexdigest()

annotation_checksum = None

if annotation_path is not None:

    digest = hashlib.sha256()

    with open(
        annotation_path,
        "rb"
    ) as handle:

        for piece in iter(
            lambda: handle.read(
                4 * 1024 * 1024
            ),
            b""
        ):
            digest.update(piece)

    annotation_checksum = digest.hexdigest()

checkpoint_path = (
    PROCESSED / "stage08_checkpoint.json"
)

if checkpoint_path.exists():

    checkpoint = json.loads(
        checkpoint_path.read_text(
            encoding="utf-8"
        )
    )

    if checkpoint.get("signature") != signature:
        raise RuntimeError(
            "Checkpoint belongs to another cohort or gene reference."
        )

    if (
        checkpoint.get(
            "done", {}
        ).get("methylation")
        and checkpoint.get("annotation_sha256")
        != annotation_checksum
    ):
        raise RuntimeError(
            "Methylation annotation changed after aggregation."
        )

else:

    checkpoint = {
        "signature": signature,
        "done": {},
        "audit": []
    }

completed = {
    modality: set(
        checkpoint["done"].get(
            modality,
            []
        )
    )
    for modality in [
        "rna",
        "cnv",
        "mutation_maf",
        "methylation"
    ]
}

audit = {
    (
        row["case_id"],
        row["modality"]
    ): row
    for row in checkpoint.get(
        "audit", []
    )
}


def save_progress():

    for array in arrays.values():
        array.flush()

    payload = {
        "signature": signature,
        "annotation_sha256": annotation_checksum,
        "done": {
            key: sorted(value)
            for key, value in completed.items()
        },
        "audit": list(
            audit.values()
        ),
        "updated_utc": datetime.now(
            timezone.utc
        ).isoformat()
    }

    temporary = checkpoint_path.with_suffix(
        ".json.tmp"
    )

    temporary.write_text(
        json.dumps(payload),
        encoding="utf-8"
    )

    os.replace(
        temporary,
        checkpoint_path
    )


# Define protein-altering mutation classes
nonsynonymous = {
    "Missense_Mutation",
    "Nonsense_Mutation",
    "Nonstop_Mutation",
    "Frame_Shift_Del",
    "Frame_Shift_Ins",
    "In_Frame_Del",
    "In_Frame_Ins",
    "Splice_Site",
    "Translation_Start_Site",
    "Start_Codon_Del",
    "Start_Codon_Ins",
    "Start_Codon_SNP"
}


# Process real molecular files
for modality in (
    "rna",
    "cnv",
    "mutation_maf",
    "methylation"
):

    if (
        modality == "methylation"
        and not annotation_valid
    ):
        continue

    pending = [
        case_id
        for case_id in case_order
        if case_id not in completed[modality]
    ]

    print(
        f"\n{modality.upper()}: "
        f"{len(pending):,} samples to process"
    )

    for count, case_id in enumerate(
        pending,
        start=1
    ):

        row_index = case_to_row[
            case_id
        ]

        selected = files_by_case.loc[
            (case_id, modality)
        ]

        path = Path(
            selected["local_path"]
        )

        details = {
            "case_id": case_id,
            "modality": modality
        }

        # RNA expression and count matrices
        if modality == "rna":

            frame = read_table(
                path,
                [
                    "gene_id",
                    "gene_name",
                    "gene_type",
                    "unstranded",
                    "tpm_unstranded"
                ]
            )

            frame = frame[
                frame["gene_type"]
                .str.lower()
                .eq("protein_coding")
            ].copy()

            frame["ensembl_id"] = (
                frame["gene_id"].str.replace(
                    r"\.\d+$",
                    "",
                    regex=True
                )
            )

            frame = (
                frame
                .drop_duplicates("ensembl_id")
                .set_index("ensembl_id")
            )

            tpm = pd.to_numeric(
                frame["tpm_unstranded"].reindex(
                    gene_ids
                ),
                errors="coerce"
            ).to_numpy(
                dtype=np.float32
            )

            counts = pd.to_numeric(
                frame["unstranded"].reindex(
                    gene_ids
                ),
                errors="coerce"
            ).to_numpy(
                dtype=np.float32
            )

            tpm[
                ~np.isfinite(tpm)
                | (tpm < 0)
            ] = np.nan

            counts[
                ~np.isfinite(counts)
                | (counts < 0)
            ] = np.nan

            if np.isfinite(tpm).mean() < 0.95:
                raise RuntimeError(
                    f"Low RNA gene coverage: {case_id}"
                )

            arrays["rna_tpm"][
                row_index
            ] = tpm

            arrays["rna_counts"][
                row_index
            ] = counts

            details["finite_gene_fraction"] = round(
                float(
                    np.isfinite(tpm).mean()
                ),
                5
            )

        # Absolute gene-level copy number
        elif modality == "cnv":

            frame = read_table(
                path,
                [
                    "gene_id",
                    "copy_number"
                ]
            )

            frame["ensembl_id"] = (
                frame["gene_id"].str.replace(
                    r"\.\d+$",
                    "",
                    regex=True
                )
            )

            frame = (
                frame
                .drop_duplicates("ensembl_id")
                .set_index("ensembl_id")
            )

            copy_number = pd.to_numeric(
                frame["copy_number"].reindex(
                    gene_ids
                ),
                errors="coerce"
            ).to_numpy(
                dtype=np.float32
            )

            copy_number[
                ~np.isfinite(copy_number)
                | (copy_number < 0)
            ] = np.nan

            fraction = float(
                np.isfinite(
                    copy_number
                ).mean()
            )

            if fraction < 0.60:
                raise RuntimeError(
                    f"Low CNV gene coverage: {case_id}"
                )

            arrays["cnv_absolute"][
                row_index
            ] = copy_number

            details["finite_gene_fraction"] = round(
                fraction,
                5
            )

        # Nonsynonymous gene-level mutation indicators
        elif modality == "mutation_maf":

            frame = read_table(
                path,
                [
                    "Hugo_Symbol",
                    "Variant_Classification",
                    "Tumor_Sample_Barcode"
                ]
            )

            patient_id = str(
                eligible.loc[
                    row_index,
                    "patient_id"
                ]
            ).upper()

            valid = (
                frame["Tumor_Sample_Barcode"]
                .str.upper()
                .str.startswith(
                    patient_id + "-01",
                    na=False
                )
            )

            frame = frame.loc[
                valid
                & frame[
                    "Variant_Classification"
                ].isin(nonsynonymous)
            ]

            vector = np.zeros(
                len(gene_ids),
                dtype=np.uint8
            )

            mapped = {
                symbol_to_index[name]
                for name in frame[
                    "Hugo_Symbol"
                ].dropna().unique()
                if name in symbol_to_index
            }

            if mapped:
                vector[list(mapped)] = 1

            arrays[
                "mutation_nonsynonymous"
            ][row_index] = vector

            details[
                "n_nonsynonymous_calls"
            ] = int(len(frame))

            details[
                "n_mutated_coding_genes"
            ] = int(len(mapped))

        # Promoter methylation aggregation
        elif modality == "methylation":

            with file_opener(path)(
                path,
                "rt",
                encoding="utf-8-sig",
                errors="replace"
            ) as stream:

                frame = pd.read_csv(
                    stream,
                    sep="\t",
                    header=None,
                    names=[
                        "probe",
                        "beta"
                    ],
                    usecols=[0, 1],
                    dtype={
                        "probe": str
                    },
                    low_memory=False
                )

            frame["beta"] = pd.to_numeric(
                frame["beta"],
                errors="coerce"
            )

            frame = frame.loc[
                frame["beta"].between(
                    0,
                    1,
                    inclusive="both"
                )
            ]

            frame = (
                frame
                .drop_duplicates("probe")
                .set_index("probe")
            )

            values = (
                frame["beta"]
                .reindex(probe_ids)
                .to_numpy(
                    dtype=np.float32
                )
            )

            observed = np.isfinite(
                values
            )

            summed = np.asarray(
                np.where(
                    observed,
                    values,
                    0
                ) @ probe_gene
            ).ravel()

            probe_counts = np.asarray(
                observed.astype(
                    np.float32
                ) @ probe_gene
            ).ravel()

            gene_beta = np.divide(
                summed,
                probe_counts,
                out=np.full(
                    len(gene_ids),
                    np.nan,
                    dtype=np.float32
                ),
                where=probe_counts >= 2
            )

            if observed.sum() < 1500:
                raise RuntimeError(
                    f"Insufficient mapped promoter probes: {case_id}"
                )

            arrays[
                "methylation_promoter_beta"
            ][row_index] = gene_beta.astype(
                np.float32
            )

            arrays[
                "methylation_promoter_nprobes"
            ][row_index] = np.minimum(
                probe_counts,
                65535
            ).astype(
                np.uint16
            )

            details[
                "promoter_cpg_fraction"
            ] = round(
                float(
                    observed.mean()
                ),
                5
            )

            details[
                "gene_fraction_2probes"
            ] = round(
                float(
                    np.isfinite(
                        gene_beta
                    ).mean()
                ),
                5
            )

        completed[modality].add(
            case_id
        )

        audit[
            (case_id, modality)
        ] = details

        if (
            count % CHECKPOINT_EVERY == 0
            or count == len(pending)
        ):

            save_progress()

            print(
                f"  {len(completed[modality]):,}"
                f"/{len(case_order):,} complete",
                flush=True
            )


# Save sample order and QC results
case_order_path = (
    PROCESSED / "case_order.csv"
)

eligible[
    [
        "case_id",
        "patient_id",
        "histology",
        "stage_group"
    ]
].to_csv(
    case_order_path,
    index=False
)

quality_path = (
    METRICS / "stage08_gene_level_qc.csv"
)

pd.DataFrame(
    list(audit.values())
).to_csv(
    quality_path,
    index=False
)


# Generate the stage summary
report = {
    "generated_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "patients": len(case_order),

    "protein_coding_genes": len(gene_ids),

    "split_counts": (
        split["split"]
        .value_counts()
        .to_dict()
    ),

    "annotation_source": (
        annotation_source
        if annotation_valid
        else None
    ),

    "annotation_error": (
        annotation_error
        if not annotation_valid
        else None
    ),

    "annotation_sha256": annotation_checksum,

    "promoter_probe_count": len(probe_ids),

    "ambiguous_annotation_rows": (
        ambiguous_annotation
    ),

    "completed_samples": {
        key: len(value)
        for key, value in completed.items()
    },

    "full_four_omics_gene_level_ready": all(
        len(value) == len(case_order)
        for value in completed.values()
    ),

    "raw_matrices_are_unscaled": True,

    "train_only_feature_selection": "pending",

    "train_only_imputation": "pending",

    "held_out_test_used_for_tuning": False,

    "files": {
        name: str(
            PROCESSED / (name + ".npy")
        )
        for name in arrays
    }
}

report_path = (
    METRICS / "stage08_gene_level_summary.json"
)

report_path.write_text(
    json.dumps(
        report,
        indent=2
    ),
    encoding="utf-8"
)


# Final verification report
print("\n" + "=" * 68)
print("STAGE 08 RESULTS")
print("=" * 68)

for key, value in report.items():

    if key != "files":
        print(f"{key}: {value}")

print("\nCase order:", case_order_path)
print("Gene reference:", reference_path)
print("Locked split:", split_path)
print("Quality audit:", quality_path)
print("Summary:", report_path)

print(
    "\nNo feature selection, scaling or imputation "
    "was fitted on validation/test patients."
)

STAGE 08 - FOUR-OMICS GENE-LEVEL MATRIX CONSTRUCTION
QC-passed patients: 953
Verified molecular files: 3812

Locked patient splits:
split                  test  train  validation
histology stage_group                         
LUAD      Advanced       15     73          16
          Early          57    265          56
LUSC      Advanced       13     60          13
          Early          58    269          58

Protein-coding genes: 19944
Unambiguous gene symbols: 19932

Promoter CpG probes: 115,314
Probe-gene links: 119,137

RNA: 953 samples to process
  25/953 complete
  50/953 complete
  75/953 complete
  100/953 complete
  125/953 complete
  150/953 complete
  175/953 complete
  200/953 complete
  225/953 complete
  250/953 complete
  275/953 complete
  300/953 complete
  325/953 complete
  350/953 complete
  375/953 complete
  400/953 complete
  425/953 complete
  450/953 complete
  475/953 complete
  500/953 complete
  525/953 complete
  550/953 complete
  575/953 complete
  600/

In [8]:

import gzip
import hashlib
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from torch_geometric.data import Batch, Data
from torch_geometric.nn import GATConv, global_mean_pool

# Project configuration
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

SRC = ROOT / "data" / "processed" / "gene_level"
SPLITS = ROOT / "data" / "splits"
STRING_DIR = ROOT / "data" / "raw" / "STRING"
OUT = ROOT / "data" / "processed" / "ppi_gat_v1"
METRICS = ROOT / "results" / "metrics"

for path in (STRING_DIR, OUT, METRICS):
    path.mkdir(parents=True, exist_ok=True)

VERSION = "string_v12.5_trainfit_omics_v1"
STRING_VERSION = "12.5"
MAX_GENES = 1600
MIN_PPI_SCORE = 700
PHYSICAL_SCORE = 400
CLIP_Z = 5.0
SEED = 42

# Validate input artifacts and patient order
summary = json.loads(
    (METRICS / "stage08_gene_level_summary.json").read_text(
        encoding="utf-8"
    )
)

if (
    not summary.get("full_four_omics_gene_level_ready")
    or summary.get("patients") != 953
):
    raise RuntimeError(
        "Stage 08 matrices are not confirmed complete."
    )

cases = pd.read_csv(
    SRC / "case_order.csv",
    dtype=str,
    keep_default_na=False
)

genes = pd.read_csv(
    SRC / "gene_reference.csv",
    dtype=str,
    keep_default_na=False
)

splits = pd.read_csv(
    SPLITS / "locked_four_omics_v1.csv",
    dtype=str,
    keep_default_na=False
)

if (
    not cases.case_id.is_unique
    or not genes.ensembl_id.is_unique
    or not splits.case_id.is_unique
):
    raise RuntimeError("Duplicate input identifiers detected.")

if (
    len(cases) != summary["patients"]
    or len(genes) != summary["protein_coding_genes"]
):
    raise RuntimeError("Stage 08 matrix metadata mismatch.")

if set(cases.case_id) != set(splits.case_id):
    raise RuntimeError("Locked split patient IDs differ.")

splits = cases[
    ["case_id", "histology", "stage_group"]
].merge(
    splits,
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_matrix", "_split")
)

if (
    not (
        splits.histology_matrix
        == splits.histology_split
    ).all()
    or not (
        splits.stage_group_matrix
        == splits.stage_group_split
    ).all()
):
    raise RuntimeError("Clinical labels differ from the locked split.")

if set(splits["split"]) != {
    "train", "validation", "test"
}:
    raise RuntimeError("Invalid split assignments.")

train_idx = np.flatnonzero(
    splits["split"].eq("train").to_numpy()
)

val_idx = np.flatnonzero(
    splits["split"].eq("validation").to_numpy()
)

test_idx = np.flatnonzero(
    splits["split"].eq("test").to_numpy()
)

if (
    len(train_idx),
    len(val_idx),
    len(test_idx)
) != (667, 143, 143):
    raise RuntimeError("Unexpected locked split sizes.")

modalities = {
    "rna": "rna_tpm.npy",
    "cnv": "cnv_absolute.npy",
    "mutation": "mutation_nonsynonymous.npy",
    "methylation": "methylation_promoter_beta.npy"
}

matrices = {
    name: np.load(
        SRC / filename,
        mmap_mode="r"
    )
    for name, filename in modalities.items()
}

for name, matrix in matrices.items():
    if matrix.shape != (len(cases), len(genes)):
        raise RuntimeError(
            f"Incorrect {name} matrix shape: {matrix.shape}"
        )

print("=" * 72)
print("STAGE 09 - TRAIN-ONLY PREPROCESSING AND STRING PPI GRAPHS")
print("=" * 72)

print(f"Patients: {len(cases)}")
print(f"Protein-coding genes: {len(genes)}")
print(
    f"Locked split: train={len(train_idx)}, "
    f"validation={len(val_idx)}, test={len(test_idx)}"
)

# Download and validate STRING reference data
BASE_URL = "https://stringdb-downloads.org/download"

string_files = {
    "info": (
        f"protein.info.v{STRING_VERSION}",
        f"9606.protein.info.v{STRING_VERSION}.txt.gz"
    ),
    "functional": (
        f"protein.links.v{STRING_VERSION}",
        f"9606.protein.links.v{STRING_VERSION}.txt.gz"
    ),
    "physical": (
        f"protein.physical.links.v{STRING_VERSION}",
        f"9606.protein.physical.links.v{STRING_VERSION}.txt.gz"
    )
}

retry = Retry(
    total=5,
    backoff_factor=1.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"])
)

http = requests.Session()
http.mount(
    "https://",
    HTTPAdapter(max_retries=retry)
)

http.headers["User-Agent"] = "NST-NSCLC-PPI-GAT/1.0"


def valid_gzip(path):
    if not path.is_file() or path.stat().st_size < 1000:
        return False

    try:
        with gzip.open(path, "rb") as stream:
            while stream.read(4 * 1024 * 1024):
                pass
        return True

    except (OSError, EOFError):
        return False


def fetch_string(key):
    subdir, filename = string_files[key]
    destination = STRING_DIR / filename

    if valid_gzip(destination):
        print(
            f"STRING {key}: verified cached file "
            f"({destination.stat().st_size / 2**20:.1f} MiB)"
        )
        return destination

    if destination.exists():
        invalid = destination.with_name(
            destination.name + ".invalid"
        )
        destination.replace(invalid)

    partial = destination.with_name(
        destination.name + ".part"
    )

    url = f"{BASE_URL}/{subdir}/{filename}"

    print(f"Downloading STRING {key}...")

    existing = (
        partial.stat().st_size
        if partial.exists()
        else 0
    )

    headers = (
        {"Range": f"bytes={existing}-"}
        if existing
        else {}
    )

    with http.get(
        url,
        headers=headers,
        stream=True,
        timeout=(25, 180)
    ) as response:

        response.raise_for_status()

        append = (
            existing > 0
            and response.status_code == 206
        )

        if append:
            content_range = response.headers.get(
                "Content-Range", ""
            )

            if not content_range.startswith(
                f"bytes {existing}-"
            ):
                raise RuntimeError(
                    "Unexpected STRING download range."
                )

        with open(
            partial,
            "ab" if append else "wb"
        ) as handle:

            for chunk in response.iter_content(
                chunk_size=2 * 1024 * 1024
            ):
                if chunk:
                    handle.write(chunk)

    if not valid_gzip(partial):
        raise RuntimeError(
            f"Invalid or incomplete STRING download: {url}"
        )

    os.replace(partial, destination)

    print(
        f"STRING {key}: validated "
        f"({destination.stat().st_size / 2**20:.1f} MiB)"
    )

    return destination


info_path = fetch_string("info")
functional_path = fetch_string("functional")
physical_path = fetch_string("physical")

# Map gene symbols to unique STRING protein identifiers
info = pd.read_csv(
    info_path,
    sep="\t",
    compression="gzip",
    dtype=str,
    low_memory=False
)

info.columns = [
    str(column).lstrip("#").strip()
    for column in info.columns
]

protein_col = next(
    (
        key
        for key in (
            "string_protein_id",
            "protein_external_id"
        )
        if key in info.columns
    ),
    None
)

if (
    protein_col is None
    or "preferred_name" not in info.columns
):
    raise RuntimeError(
        f"Unexpected STRING protein columns: {list(info.columns)}"
    )

info = info[
    [protein_col, "preferred_name"]
].dropna().copy()

info["preferred_name"] = (
    info["preferred_name"]
    .str.upper()
    .str.strip()
)

info = info[
    info["preferred_name"].ne("")
    & info[protein_col].str.startswith("9606.")
]

info = info.drop_duplicates(
    ["preferred_name", protein_col]
)

unique_string = info.loc[
    ~info["preferred_name"].duplicated(keep=False)
]

protein_by_symbol = dict(
    zip(
        unique_string["preferred_name"],
        unique_string[protein_col]
    )
)

symbols = (
    genes["gene_symbol"]
    .astype(str)
    .str.upper()
    .str.strip()
)

nonduplicate = ~symbols.duplicated(
    keep=False
)

mapped = (
    nonduplicate
    & symbols.isin(protein_by_symbol)
)

print(
    f"\nUnambiguously mapped STRING genes: "
    f"{mapped.sum():,}/{len(genes):,}"
)

# Calculate feature statistics using training patients only
rna = np.asarray(
    matrices["rna"][train_idx],
    dtype=np.float32
)

cnv = np.asarray(
    matrices["cnv"][train_idx],
    dtype=np.float32
)

mut = np.asarray(
    matrices["mutation"][train_idx],
    dtype=np.uint8
)

meth = np.asarray(
    matrices["methylation"][train_idx],
    dtype=np.float32
)

rna_obs = np.isfinite(rna) & (rna >= 0)
cnv_obs = np.isfinite(cnv) & (cnv >= 0)

meth_obs = (
    np.isfinite(meth)
    & (meth >= 0)
    & (meth <= 1)
)

rna_fraction = rna_obs.mean(axis=0)
cnv_fraction = cnv_obs.mean(axis=0)
meth_fraction = meth_obs.mean(axis=0)

expression_fraction = (
    rna_obs & (rna >= 1.0)
).mean(axis=0)

mutation_count = (
    mut.sum(axis=0)
    .astype(np.int32)
)

rna_log = np.where(
    rna_obs,
    np.log1p(np.maximum(rna, 0)),
    np.nan
)

cnv_log = np.log2(
    (np.clip(cnv, 0, None) + 0.5) / 2.5
)

cnv_log = np.where(
    cnv_obs,
    np.clip(cnv_log, -3, 3),
    np.nan
)

meth_values = np.where(
    meth_obs,
    meth,
    np.nan
)

rna_var = np.nanstd(
    rna_log,
    axis=0
)

cnv_var = np.nanstd(
    cnv_log,
    axis=0
)

meth_var = np.nanstd(
    meth_values,
    axis=0
)

# Filter unreliable gene-level measurements
eligible = (
    mapped.to_numpy()
    & (rna_fraction >= 0.98)
    & (cnv_fraction >= 0.85)
    & (meth_fraction >= 0.50)
)

if eligible.sum() < MAX_GENES:

    eligible = (
        mapped.to_numpy()
        & (rna_fraction >= 0.95)
        & (cnv_fraction >= 0.75)
        & (meth_fraction >= 0.20)
    )

    print(
        "Coverage filter relaxed using training data: "
        "RNA >=95%, CNV >=75%, methylation >=20%."
    )

if eligible.sum() < 500:
    raise RuntimeError(
        f"Insufficient reliable STRING-mapped genes: "
        f"{eligible.sum()}"
    )

print(
    f"Eligible genes after training-only QC: "
    f"{eligible.sum():,}"
)

# Build modality-specific candidate rankings
metrics = {
    "rna": rna_var,
    "cnv": cnv_var,
    "methylation": meth_var,
    "mutation": mutation_count.astype(float)
}

allowed = {
    "rna": (
        eligible
        & (expression_fraction >= 0.10)
    ),
    "cnv": (
        eligible
        & (cnv_var >= 0.02)
    ),
    "methylation": (
        eligible
        & (meth_var >= 0.015)
    ),
    "mutation": (
        eligible
        & (mutation_count >= 2)
    )
}


def descending_gene_rank(score, mask):
    indices = np.flatnonzero(
        mask & np.isfinite(score)
    )

    order = np.argsort(
        -score[indices],
        kind="stable"
    )

    return indices[order].tolist()


ranks = {
    name: descending_gene_rank(
        score,
        allowed[name]
    )
    for name, score in metrics.items()
}

quotas = {
    "rna": 400,
    "cnv": 330,
    "methylation": 440,
    "mutation": 230
}

selected = []
selected_set = set()
sources = {}

for modality, quota in quotas.items():

    for index in ranks[modality][:quota]:

        if index not in selected_set:
            selected_set.add(index)
            selected.append(index)
            sources[index] = modality

# Fill the remaining nodes from the ranked candidate pools
next_positions = {
    name: 0
    for name in ranks
}

while len(selected) < min(
    MAX_GENES,
    int(eligible.sum())
):

    added = False

    for modality in (
        "rna",
        "methylation",
        "cnv",
        "mutation"
    ):

        rank = ranks[modality]

        while next_positions[modality] < len(rank):

            index = rank[
                next_positions[modality]
            ]

            next_positions[modality] += 1

            if index not in selected_set:

                selected_set.add(index)
                selected.append(index)
                sources[index] = modality
                added = True
                break

        if len(selected) >= MAX_GENES:
            break

    if not added:
        break

selected_idx = np.asarray(
    sorted(selected),
    dtype=np.int64
)

if len(selected_idx) < 500:
    raise RuntimeError(
        "Feature selection retained fewer than 500 genes."
    )

selected_genes = (
    genes.iloc[selected_idx]
    .copy()
    .reset_index(drop=True)
)

selected_genes.insert(
    0,
    "node_index",
    np.arange(len(selected_idx))
)

selected_genes["selected_by"] = [
    sources[index]
    for index in selected_idx
]

selected_genes["train_rna_observed"] = (
    rna_fraction[selected_idx]
)

selected_genes["train_cnv_observed"] = (
    cnv_fraction[selected_idx]
)

selected_genes["train_methylation_observed"] = (
    meth_fraction[selected_idx]
)

selected_genes["train_rna_variable"] = (
    rna_var[selected_idx]
)

selected_genes["train_cnv_variable"] = (
    cnv_var[selected_idx]
)

selected_genes["train_methylation_variable"] = (
    meth_var[selected_idx]
)

selected_genes["train_mutated_samples"] = (
    mutation_count[selected_idx]
)

selected_genes["string_id"] = [
    protein_by_symbol[symbol]
    for symbol in symbols.iloc[selected_idx]
]

selected_genes.to_csv(
    OUT / "selected_gene_nodes.csv",
    index=False
)

print(f"\nSelected graph nodes: {len(selected_idx)}")

print(
    "Selection sources:",
    selected_genes["selected_by"]
    .value_counts()
    .to_dict()
)

# Extract STRING associations for selected proteins
protein_to_node = dict(
    zip(
        selected_genes["string_id"],
        selected_genes["node_index"]
    )
)


def parse_string_edges(path, minimum_score):

    pair_scores = {}
    scanned = 0

    chunks = pd.read_csv(
        path,
        compression="gzip",
        sep=r"\s+",
        chunksize=200000,
        usecols=[
            "protein1",
            "protein2",
            "combined_score"
        ],
        dtype={
            "protein1": str,
            "protein2": str,
            "combined_score": np.int32
        }
    )

    for chunk in chunks:

        scanned += len(chunk)

        cut = chunk.loc[
            chunk["combined_score"].ge(
                minimum_score
            )
        ]

        if cut.empty:
            continue

        a = cut["protein1"].map(
            protein_to_node
        )

        b = cut["protein2"].map(
            protein_to_node
        )

        valid = (
            a.notna()
            & b.notna()
            & a.lt(b)
        )

        if not valid.any():
            continue

        scores = cut.loc[
            valid,
            "combined_score"
        ].to_numpy(
            dtype=np.int32
        )

        for i, j, score in zip(
            a[valid].astype(int),
            b[valid].astype(int),
            scores
        ):

            pair = (
                int(i),
                int(j)
            )

            if score > pair_scores.get(pair, 0):
                pair_scores[pair] = int(score)

    if not pair_scores:

        return (
            np.empty((0, 2), dtype=np.int64),
            np.empty(0, dtype=np.float32),
            scanned
        )

    pairs = np.asarray(
        sorted(pair_scores),
        dtype=np.int64
    )

    weights = np.asarray(
        [
            pair_scores[tuple(pair)] / 1000.0
            for pair in pairs
        ],
        dtype=np.float32
    )

    return pairs, weights, scanned


def save_graph(name, pairs, weights):

    directed = (
        np.concatenate(
            [pairs, pairs[:, ::-1]],
            axis=0
        )
        if len(pairs)
        else np.empty(
            (0, 2),
            dtype=np.int64
        )
    )

    edge_index = (
        directed.T.astype(
            np.int64,
            copy=False
        )
    )

    edge_weight = np.tile(
        weights,
        2
    )

    np.savez_compressed(
        OUT / f"string_{name}_graph.npz",
        edge_index=edge_index,
        edge_weight=edge_weight
    )

    degree = (
        np.bincount(
            directed.reshape(-1),
            minlength=len(selected_idx)
        )
        if len(directed)
        else np.zeros(
            len(selected_idx),
            dtype=int
        )
    )

    report = {
        "undirected_edges": len(pairs),
        "directed_edges": edge_index.shape[1],
        "nodes_with_neighbors": int(
            (degree > 0).sum()
        ),
        "isolated_nodes": int(
            (degree == 0).sum()
        ),
        "max_degree": int(
            degree.max()
        )
    }

    return report, edge_index


print(
    "\nExtracting STRING v12.5 interactions..."
)

functional_pairs, functional_weights, scanned_f = (
    parse_string_edges(
        functional_path,
        MIN_PPI_SCORE
    )
)

physical_pairs, physical_weights, scanned_p = (
    parse_string_edges(
        physical_path,
        PHYSICAL_SCORE
    )
)

functional_summary, functional_edge_index = (
    save_graph(
        "functional_700",
        functional_pairs,
        functional_weights
    )
)

physical_summary, physical_edge_index = (
    save_graph(
        "physical_400",
        physical_pairs,
        physical_weights
    )
)

if functional_summary["undirected_edges"] < 300:
    raise RuntimeError(
        "The high-confidence STRING graph is unexpectedly sparse. "
        "Check gene-to-protein identifier mapping."
    )

print("\nFunctional graph:", functional_summary)
print("Physical graph  :", physical_summary)

# Fit preprocessing parameters on training patients only
transform_spec = {
    "rna": (
        "rna_tpm",
        "rna_log1p_tpm",
        0.10
    ),
    "cnv": (
        "cnv_absolute",
        "cnv_log2_absolute_ratio",
        0.15
    ),
    "methylation": (
        "methylation_promoter_beta",
        "methylation_promoter_beta",
        0.04
    )
}


def transform(name, subset):

    subset = np.asarray(
        subset,
        dtype=np.float32
    )

    if name == "rna":

        return np.log1p(
            np.where(
                subset >= 0,
                subset,
                np.nan
            )
        )

    if name == "cnv":

        values = np.where(
            subset >= 0,
            subset,
            np.nan
        )

        return np.clip(
            np.log2(
                (values + 0.5) / 2.5
            ),
            -3.0,
            3.0
        )

    if name == "methylation":

        return np.where(
            (subset >= 0)
            & (subset <= 1),
            subset,
            np.nan
        )

    raise ValueError(name)


# Allocate a graph-ready feature tensor
x_path = OUT / "patient_gene_features_8ch.npy"

tmp_path = (
    OUT / "patient_gene_features_8ch.partial.npy"
)

if x_path.exists():
    x_path.unlink()

x = np.lib.format.open_memmap(
    tmp_path,
    mode="w+",
    dtype=np.float32,
    shape=(
        len(cases),
        len(selected_idx),
        8
    )
)

x[:] = 0

fitted = []

for channel, (
    modality,
    (source, feature_name, minimum_scale)
) in enumerate(transform_spec.items()):

    raw = np.asarray(
        matrices[modality][
            :,
            selected_idx
        ],
        dtype=np.float32
    )

    values = transform(
        modality,
        raw
    )

    training = values[train_idx]

    with np.errstate(invalid="ignore"):

        centers = np.nanmedian(
            training,
            axis=0
        )

        q25 = np.nanpercentile(
            training,
            25,
            axis=0
        )

        q75 = np.nanpercentile(
            training,
            75,
            axis=0
        )

    centers = np.where(
        np.isfinite(centers),
        centers,
        0
    )

    scales = np.maximum(
        np.where(
            np.isfinite(q75 - q25),
            (q75 - q25) / 1.349,
            0
        ),
        minimum_scale
    )

    observed = np.isfinite(
        values
    )

    filled = np.where(
        observed,
        values,
        centers[None, :]
    )

    standardized = (
        filled - centers
    ) / scales

    x[:, :, channel] = np.clip(
        standardized,
        -CLIP_Z,
        CLIP_Z
    ).astype(np.float32)

    mask_channel = {
        "rna": 4,
        "cnv": 5,
        "methylation": 7
    }[modality]

    x[:, :, mask_channel] = (
        observed.astype(np.float32)
    )

    for j in range(
        len(selected_idx)
    ):

        fitted.append({
            "node_index": j,
            "gene_id": selected_genes[
                "ensembl_id"
            ].iloc[j],
            "modality": modality,
            "transform": feature_name,
            "train_center": float(
                centers[j]
            ),
            "train_scale": float(
                scales[j]
            ),
            "train_observed_fraction": float(
                observed[
                    train_idx,
                    j
                ].mean()
            )
        })

    print(
        f"Training-only {modality} preprocessing: PASSED"
    )

# Retain mutation indicators without continuous scaling
mutation = np.asarray(
    matrices["mutation"][
        :,
        selected_idx
    ],
    dtype=np.float32
)

if not np.isin(
    mutation,
    (0, 1)
).all():
    raise RuntimeError(
        "Mutation features are not binary."
    )

x[:, :, 2] = mutation
x[:, :, 6] = 1.0

x.flush()
del x

os.replace(
    tmp_path,
    x_path
)

scalers_path = (
    OUT / "train_only_scaler_parameters.csv"
)

pd.DataFrame(fitted).to_csv(
    scalers_path,
    index=False
)

# Verify standardized features and modality masks
x_check = np.load(
    x_path,
    mmap_mode="r"
)

expected_shape = (
    len(cases),
    len(selected_idx),
    8
)

if (
    x_check.shape != expected_shape
    or not np.isfinite(x_check).all()
):
    raise RuntimeError(
        "Invalid graph-ready patient feature tensor."
    )

if not np.isin(
    x_check[:, :, 4:],
    (0.0, 1.0)
).all():
    raise RuntimeError(
        "Modality availability masks are not binary."
    )

if not np.isin(
    x_check[:, :, 2],
    (0.0, 1.0)
).all():
    raise RuntimeError(
        "Mutation indicator channel is not binary."
    )

# Save patient labels with their original locked splits
labels = (
    cases.stage_group == "Advanced"
).astype(int).to_numpy(
    dtype=np.int64
)

histology = (
    cases.histology == "LUSC"
).astype(int).to_numpy(
    dtype=np.int64
)

labels_path = (
    OUT / "patient_labels_and_splits.csv"
)

pd.DataFrame({
    "case_id": cases.case_id,
    "histology": cases.histology,
    "stage_group": cases.stage_group,
    "stage_label": labels,
    "histology_label": histology,
    "split": splits["split"]
}).to_csv(
    labels_path,
    index=False
)

# Verify GAT compatibility using training patients only
torch.set_num_threads(
    min(
        4,
        os.cpu_count() or 1
    )
)

torch.manual_seed(SEED)

probe_ids = [
    int(train_idx[0]),
    int(train_idx[-1])
]

graph_samples = [
    Data(
        x=torch.tensor(
            np.asarray(
                x_check[index]
            ).copy()
        ),
        edge_index=torch.tensor(
            functional_edge_index.copy(),
            dtype=torch.long
        ),
        y=torch.tensor(
            [labels[index]],
            dtype=torch.long
        )
    )
    for index in probe_ids
]

batch = Batch.from_data_list(
    graph_samples
)

conv = GATConv(
    in_channels=8,
    out_channels=8,
    heads=2,
    dropout=0.1
)

classifier = torch.nn.Linear(
    16,
    2
)

embeddings = torch.nn.functional.elu(
    conv(
        batch.x,
        batch.edge_index
    )
)

patient_repr = global_mean_pool(
    embeddings,
    batch.batch
)

predictions = classifier(
    patient_repr
)

loss = torch.nn.functional.cross_entropy(
    predictions,
    batch.y.view(-1)
)

loss.backward()

gradients = [
    parameter.grad
    for parameter in conv.parameters()
    if parameter.grad is not None
]

if (
    predictions.shape != (2, 2)
    or not torch.isfinite(loss).item()
    or not gradients
    or not all(
        torch.isfinite(grad).all().item()
        for grad in gradients
    )
):
    raise RuntimeError(
        "Training-only GAT gradient test failed."
    )

print("\nGraph-ready GAT gradient test: PASSED")

# Record source file hashes
source_hashes = {}

for source in (
    SRC / "gene_reference.csv",
    SRC / "case_order.csv",
    SPLITS / "locked_four_omics_v1.csv"
):

    source_hashes[source.name] = (
        hashlib.sha256(
            source.read_bytes()
        ).hexdigest()
    )

# Save reproducibility metadata
report = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "version": VERSION,
    "n_patients": len(cases),
    "n_train": len(train_idx),
    "n_validation": len(val_idx),
    "n_test": len(test_idx),
    "input_gene_count": len(genes),
    "selected_gene_count": len(selected_idx),
    "selected_by": (
        selected_genes["selected_by"]
        .value_counts()
        .to_dict()
    ),
    "string_version": STRING_VERSION,
    "string_species": 9606,
    "string_functional_minimum_score": MIN_PPI_SCORE,
    "string_physical_minimum_score": PHYSICAL_SCORE,
    "string_functional_graph": functional_summary,
    "string_physical_graph": physical_summary,
    "feature_channels": [
        "rna_log1p_tpm_z",
        "cnv_log2_absolute_ratio_z",
        "mutation_binary",
        "promoter_methylation_beta_z",
        "rna_observed",
        "cnv_observed",
        "mutation_assayed",
        "methylation_observed"
    ],
    "tensor_shape": list(x_check.shape),
    "train_only_preprocessing": True,
    "stage_labels_used_in_gene_selection": False,
    "test_tuning_performed": False,
    "gat_training_only_sanity_check": "PASSED",
    "source_sha256": source_hashes,
    "string_reference_files": {
        key: str(
            STRING_DIR / name
        )
        for key, (_, name) in string_files.items()
    },
    "outputs": {
        "feature_tensor": str(x_path),
        "genes": str(
            OUT / "selected_gene_nodes.csv"
        ),
        "scalers": str(scalers_path),
        "patient_labels": str(labels_path)
    }
}

report_path = (
    OUT / "stage09_summary.json"
)

report_path.write_text(
    json.dumps(
        report,
        indent=2
    ),
    encoding="utf-8"
)

# Final verification
print("\n" + "=" * 72)
print("STAGE 09 FINAL SUMMARY")
print("=" * 72)

print("Patient tensor:", x_check.shape)
print("Selected genes:", len(selected_idx))

print(
    "STRING functional edges:",
    functional_summary["undirected_edges"]
)

print(
    "STRING physical edges:",
    physical_summary["undirected_edges"]
)

print("Training-only GAT gradient check: PASSED")
print("Original patient split preserved: PASSED")
print("Stage labels used for feature selection: NO")
print("Final test set used for tuning: NO")
print("Saved output folder:", OUT)

print("\nSTAGE 09 COMPLETED SUCCESSFULLY")

STAGE 09 - TRAIN-ONLY PREPROCESSING AND STRING PPI GRAPHS
Patients: 953
Protein-coding genes: 19944
Locked split: train=667, validation=143, test=143
STRING info: validated (1.9 MiB)
STRING functional: validated (59.9 MiB)
STRING physical: validated (7.0 MiB)

Unambiguously mapped STRING genes: 19,199/19,944


D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1998: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Eligible genes after training-only QC: 13,958

Selected graph nodes: 1600
Selection sources: {'methylation': 475, 'rna': 473, 'cnv': 391, 'mutation': 261}

Extracting STRING v12.5 interactions...

Functional graph: {'undirected_edges': 2827, 'directed_edges': 5654, 'nodes_with_neighbors': 1118, 'isolated_nodes': 482, 'max_degree': 180}
Physical graph  : {'undirected_edges': 2926, 'directed_edges': 5852, 'nodes_with_neighbors': 1053, 'isolated_nodes': 547, 'max_degree': 204}
Training-only rna preprocessing: PASSED
Training-only cnv preprocessing: PASSED
Training-only methylation preprocessing: PASSED

Graph-ready GAT gradient test: PASSED

STAGE 09 FINAL SUMMARY
Patient tensor: (953, 1600, 8)
Selected genes: 1600
STRING functional edges: 2827
STRING physical edges: 2926
Training-only GAT gradient check: PASSED
Original patient split preserved: PASSED
Stage labels used for feature selection: NO
Final test set used for tuning: NO
Saved output folder: D:\NST_NSCLC_PPI_GAT\data\processed\pp

In [9]:

import hashlib
import importlib.util
import json
import math
import os
import sys
import textwrap
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score
)

# Project configuration
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

SOURCE = ROOT / "data" / "processed" / "ppi_gat_v1"
STRING = ROOT / "data" / "raw" / "STRING"
SRC = ROOT / "src"
CHECKPOINTS = ROOT / "models" / "checkpoints" / "stage10"
METRICS = ROOT / "results" / "metrics" / "stage10"

for directory in (SRC, CHECKPOINTS, METRICS):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
BATCH_SIZE = 16
MAX_EPOCHS = 35
EARLY_STOP_PATIENCE = 8
MIN_EPOCHS = 10
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.001

NUM_THREADS = min(4, os.cpu_count() or 1)
torch.set_num_threads(NUM_THREADS)

feature_file = SOURCE / "patient_gene_features_8ch.npy"
node_file = SOURCE / "selected_gene_nodes.csv"
label_file = SOURCE / "patient_labels_and_splits.csv"

functional_file = (
    STRING / "9606.protein.links.v12.5.txt.gz"
)

physical_file = (
    STRING / "9606.protein.physical.links.v12.5.txt.gz"
)

for required in (
    feature_file,
    node_file,
    label_file,
    functional_file,
    physical_file
):
    if not required.is_file():
        raise FileNotFoundError(
            f"Missing Stage 09 input: {required}"
        )

# Load and verify the locked research cohort
nodes = pd.read_csv(
    node_file,
    dtype={"string_id": str}
)

patients = pd.read_csv(label_file)

N = len(nodes)
X = np.load(feature_file, mmap_mode="r")

if X.shape != (len(patients), N, 8):
    raise RuntimeError(
        f"Incorrect input tensor: {X.shape}"
    )

if N != 1600 or len(patients) != 953:
    raise RuntimeError(
        "Unexpected Stage 09 cohort or gene count."
    )

if (
    not patients.case_id.is_unique
    or not nodes.node_index.is_unique
):
    raise RuntimeError(
        "Duplicate patient or graph node identifiers."
    )

if not np.array_equal(
    nodes.node_index.to_numpy(),
    np.arange(N)
):
    raise RuntimeError(
        "Graph node indices are out of order."
    )

if set(patients.split) != {
    "train",
    "validation",
    "test"
}:
    raise RuntimeError(
        "Locked split is incomplete."
    )

train_idx = np.flatnonzero(
    patients.split.eq("train")
)

val_idx = np.flatnonzero(
    patients.split.eq("validation")
)

test_idx = np.flatnonzero(
    patients.split.eq("test")
)

if (
    len(train_idx),
    len(val_idx),
    len(test_idx)
) != (667, 143, 143):
    raise RuntimeError(
        "The original patient split has changed."
    )

if not np.isin(
    patients.stage_label,
    [0, 1]
).all():
    raise RuntimeError(
        "Invalid stage labels."
    )

if not np.isin(
    patients.histology_label,
    [0, 1]
).all():
    raise RuntimeError(
        "Invalid histology labels."
    )

print("=" * 72)
print("STAGE 10 - CORRECTED STRING NETWORK AND MODEL DEVELOPMENT")
print("=" * 72)

print("Patients:", len(patients), "| Nodes:", N)

print(
    "Train / validation / untouched test:",
    len(train_idx),
    len(val_idx),
    len(test_idx)
)


# ============================================================
# Correct the undirected STRING edge construction
# ============================================================

protein_to_node = dict(
    zip(
        nodes.string_id.astype(str),
        nodes.node_index.astype(int)
    )
)


def rebuild_links(file_path, threshold):

    scores = {}
    read_rows = 0

    chunks = pd.read_csv(
        file_path,
        compression="gzip",
        sep=r"\s+",
        usecols=[
            "protein1",
            "protein2",
            "combined_score"
        ],
        dtype={
            "protein1": str,
            "protein2": str,
            "combined_score": np.int32
        },
        chunksize=250000
    )

    for chunk in chunks:

        read_rows += len(chunk)

        chunk = chunk.loc[
            chunk.combined_score.ge(threshold)
        ]

        if chunk.empty:
            continue

        left = chunk.protein1.map(
            protein_to_node
        )

        right = chunk.protein2.map(
            protein_to_node
        )

        keep = (
            left.notna()
            & right.notna()
            & left.ne(right)
        )

        if not keep.any():
            continue

        a = left[keep].to_numpy(
            dtype=np.int32
        )

        b = right[keep].to_numpy(
            dtype=np.int32
        )

        score_array = chunk.loc[
            keep,
            "combined_score"
        ].to_numpy(
            dtype=np.int32
        )

        for lo, hi, score in zip(
            np.minimum(a, b),
            np.maximum(a, b),
            score_array
        ):

            key = (
                int(lo),
                int(hi)
            )

            if int(score) > scores.get(key, 0):
                scores[key] = int(score)

    if not scores:
        raise RuntimeError(
            f"No matching STRING edges: {file_path.name}"
        )

    pairs = np.array(
        sorted(scores),
        dtype=np.int64
    )

    weights = np.array(
        [
            scores[tuple(pair)] / 1000.0
            for pair in pairs
        ],
        dtype=np.float32
    )

    directed = np.concatenate(
        [
            pairs,
            pairs[:, ::-1]
        ],
        axis=0
    )

    edge_index = directed.T.copy()

    edge_weight = np.concatenate(
        [weights, weights]
    ).astype(np.float32)

    degree = np.bincount(
        edge_index[0],
        minlength=N
    )

    assert np.all(edge_index >= 0)
    assert np.max(edge_index) < N
    assert not np.any(
        edge_index[0] == edge_index[1]
    )

    return edge_index, edge_weight, {
        "source_rows": read_rows,
        "undirected_edges": int(len(pairs)),
        "directed_edges": int(edge_index.shape[1]),
        "nodes_with_neighbors": int(
            (degree > 0).sum()
        ),
        "isolated_nodes": int(
            (degree == 0).sum()
        ),
        "max_degree": int(degree.max()),
        "minimum_score": int(threshold)
    }


functional_ei, functional_w, functional_stats = (
    rebuild_links(
        functional_file,
        700
    )
)

physical_ei, physical_w, physical_stats = (
    rebuild_links(
        physical_file,
        400
    )
)

np.savez_compressed(
    SOURCE / "string_functional_700_corrected.npz",
    edge_index=functional_ei,
    edge_weight=functional_w
)

np.savez_compressed(
    SOURCE / "string_physical_400_corrected.npz",
    edge_index=physical_ei,
    edge_weight=physical_w
)

print("\nCorrected functional graph:", functional_stats)
print("Corrected physical graph:", physical_stats)

if functional_stats["undirected_edges"] < 2800:
    raise RuntimeError(
        "The corrected functional graph is unexpectedly sparse."
    )


# ============================================================
# Define and save the reusable model architecture
# ============================================================

MODEL_SOURCE = r'''
import torch
from torch import nn
from torch.nn import functional as F
from torch_geometric.nn import GATConv


class HistologyAlignedPPIGAT(nn.Module):

    def __init__(
        self,
        n_genes,
        hidden_dim=32,
        dropout=0.15,
        use_graph=True
    ):
        super().__init__()

        self.n_genes = int(n_genes)
        self.hidden_dim = int(hidden_dim)
        self.use_graph = bool(use_graph)

        self.dropout = nn.Dropout(dropout)

        self.modal_encoders = nn.ModuleList([
            nn.Linear(1, hidden_dim)
            for _ in range(4)
        ])

        self.gating = nn.Sequential(
            nn.Linear(8, 32),
            nn.GELU(),
            nn.Linear(32, 4)
        )

        self.gene_embedding = nn.Parameter(
            torch.zeros(n_genes, hidden_dim)
        )

        nn.init.normal_(
            self.gene_embedding,
            std=0.02
        )

        self.input_norm = nn.LayerNorm(hidden_dim)

        if use_graph:

            self.layer1 = GATConv(
                hidden_dim,
                hidden_dim // 2,
                heads=2,
                concat=True,
                edge_dim=1,
                fill_value=1.0,
                dropout=dropout
            )

            self.layer2 = GATConv(
                hidden_dim,
                hidden_dim,
                heads=1,
                concat=False,
                edge_dim=1,
                fill_value=1.0,
                dropout=dropout
            )

        else:

            self.layer1 = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU()
            )

            self.layer2 = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU()
            )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.pool_score = nn.Linear(
            hidden_dim,
            1
        )

        self.patient_norm = nn.LayerNorm(
            hidden_dim * 2
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 2)
        )

    def forward(
        self,
        features,
        edge_index=None,
        edge_attr=None
    ):

        batch_size, n_genes, n_features = features.shape

        if n_genes != self.n_genes or n_features != 8:
            raise ValueError(
                "Unexpected patient gene-feature tensor shape."
            )

        values = features[..., :4]
        observed = features[..., 4:]

        # Encode molecular modalities separately
        modal_h = torch.stack(
            [
                F.gelu(
                    layer(
                        values[..., j:j + 1]
                    )
                )
                for j, layer in enumerate(
                    self.modal_encoders
                )
            ],
            dim=2
        )

        # Prevent missing modalities from receiving gate weight
        gate_logits = self.gating(features)

        gate_logits = gate_logits.masked_fill(
            observed < 0.5,
            -1e4
        )

        gates = torch.softmax(
            gate_logits,
            dim=-1
        )

        h = (
            modal_h * gates.unsqueeze(-1)
        ).sum(dim=2)

        h = self.input_norm(
            h + self.gene_embedding.unsqueeze(0)
        )

        # Propagate information through the PPI graph
        if self.use_graph:

            if edge_index is None or edge_attr is None:
                raise ValueError(
                    "PPI edges and confidence scores are required."
                )

            flat = h.reshape(
                batch_size * n_genes,
                self.hidden_dim
            )

            h1 = F.elu(
                self.layer1(
                    flat,
                    edge_index,
                    edge_attr
                )
            )

            h = self.norm1(
                h + self.dropout(
                    h1.reshape(
                        batch_size,
                        n_genes,
                        -1
                    )
                )
            )

            flat = h.reshape(
                batch_size * n_genes,
                self.hidden_dim
            )

            h2 = F.elu(
                self.layer2(
                    flat,
                    edge_index,
                    edge_attr
                )
            )

            h = self.norm2(
                h + self.dropout(
                    h2.reshape(
                        batch_size,
                        n_genes,
                        -1
                    )
                )
            )

        else:

            h = self.norm1(
                h + self.dropout(
                    self.layer1(h)
                )
            )

            h = self.norm2(
                h + self.dropout(
                    self.layer2(h)
                )
            )

        # Combine attention and mean graph pooling
        attention = torch.softmax(
            self.pool_score(h).squeeze(-1),
            dim=1
        )

        attentive_pool = (
            attention.unsqueeze(-1) * h
        ).sum(dim=1)

        patient_embedding = self.patient_norm(
            torch.cat(
                [
                    attentive_pool,
                    h.mean(dim=1)
                ],
                dim=-1
            )
        )

        logits = self.classifier(
            patient_embedding
        )

        return logits, patient_embedding


def conditional_mmd(
    embeddings,
    stages,
    histologies
):

    losses = []

    for stage in (0, 1):

        a = embeddings[
            (stages == stage)
            & (histologies == 0)
        ]

        b = embeddings[
            (stages == stage)
            & (histologies == 1)
        ]

        if a.shape[0] < 2 or b.shape[0] < 2:
            continue

        merged = torch.cat(
            [a, b],
            dim=0
        )

        distances = torch.cdist(
            merged,
            merged
        ).square()

        detached = distances.detach()
        positive = detached[detached > 0]

        if positive.numel():
            median = positive.median().clamp_min(1e-6)
        else:
            median = distances.new_tensor(1.0)

        mmd = distances.new_tensor(0.0)
        na = a.shape[0]

        # Multi-bandwidth Gaussian MMD
        for multiplier in (0.5, 1.0, 2.0):

            K = torch.exp(
                -distances
                / (2.0 * median * multiplier)
            )

            mmd = (
                mmd
                + K[:na, :na].mean()
                + K[na:, na:].mean()
                - 2.0 * K[:na, na:].mean()
            )

        losses.append(
            mmd / 3.0
        )

    if losses:
        return torch.stack(losses).mean()

    return embeddings.sum() * 0.0
'''

model_path = SRC / "nsclc_ppi_gat.py"

model_path.write_text(
    textwrap.dedent(MODEL_SOURCE).strip() + "\n",
    encoding="utf-8"
)

spec = importlib.util.spec_from_file_location(
    "nsclc_ppi_gat",
    model_path
)

module = importlib.util.module_from_spec(spec)

sys.modules[spec.name] = module
spec.loader.exec_module(module)

Model = module.HistologyAlignedPPIGAT
conditional_mmd = module.conditional_mmd


# ============================================================
# Load training and validation data only
# ============================================================

x_train = torch.from_numpy(
    np.asarray(
        X[train_idx]
    ).copy()
)

x_val = torch.from_numpy(
    np.asarray(
        X[val_idx]
    ).copy()
)

y_train = torch.as_tensor(
    patients.stage_label.iloc[
        train_idx
    ].to_numpy(
        dtype=np.int64
    )
)

h_train = torch.as_tensor(
    patients.histology_label.iloc[
        train_idx
    ].to_numpy(
        dtype=np.int64
    )
)

y_val = torch.as_tensor(
    patients.stage_label.iloc[
        val_idx
    ].to_numpy(
        dtype=np.int64
    )
)

h_val = torch.as_tensor(
    patients.histology_label.iloc[
        val_idx
    ].to_numpy(
        dtype=np.int64
    )
)

if (
    not torch.isfinite(x_train).all()
    or not torch.isfinite(x_val).all()
):
    raise RuntimeError(
        "Non-finite model input found."
    )

if not torch.all(
    (x_train[..., 4:] == 0)
    | (x_train[..., 4:] == 1)
):
    raise RuntimeError(
        "Invalid modality-observation masks."
    )

if len(set(y_val.tolist())) < 2:
    raise RuntimeError(
        "Validation set contains only one stage class."
    )

print("\nTraining feature tensor:", tuple(x_train.shape))
print("Validation feature tensor:", tuple(x_val.shape))
print("Test features not loaded or evaluated.")


# ============================================================
# Prepare shared PPI topology for batched patient graphs
# ============================================================

base_ei = torch.from_numpy(
    functional_ei
).long()

base_ea = torch.from_numpy(
    functional_w
).float()

edge_cache = {}


def edges_for_batch(batch_size):

    if batch_size not in edge_cache:

        offsets = (
            torch.arange(batch_size)
            .view(batch_size, 1, 1) * N
        )

        edge_index = (
            base_ei.unsqueeze(0) + offsets
        ).permute(
            1, 0, 2
        ).reshape(
            2, -1
        ).contiguous()

        attributes = (
            base_ea.repeat(batch_size)
            .unsqueeze(1)
            .contiguous()
        )

        edge_cache[batch_size] = (
            edge_index,
            attributes
        )

    return edge_cache[batch_size]


# ============================================================
# Balanced sampling across stage and histology groups
# ============================================================

def balanced_batches(seed):

    rng = np.random.default_rng(seed)

    groups = {}
    cursors = {}

    for stage in (0, 1):

        for histology in (0, 1):

            key = (stage, histology)

            ids = np.flatnonzero(
                (y_train.numpy() == stage)
                & (h_train.numpy() == histology)
            )

            if len(ids) < 4:
                raise RuntimeError(
                    f"Too few samples for stratum {key}."
                )

            groups[key] = rng.permutation(ids)
            cursors[key] = 0

    steps = math.ceil(
        len(y_train) / BATCH_SIZE
    )

    for _ in range(steps):

        picked = []

        for key in sorted(groups):

            needed = BATCH_SIZE // 4

            while needed:

                array = groups[key]
                start = cursors[key]

                if start >= len(array):

                    groups[key] = rng.permutation(
                        array
                    )

                    cursors[key] = 0
                    continue

                take = min(
                    needed,
                    len(array) - start
                )

                picked.extend(
                    array[
                        start:start + take
                    ].tolist()
                )

                cursors[key] += take
                needed -= take

        rng.shuffle(picked)

        yield torch.tensor(
            picked,
            dtype=torch.long
        )


# ============================================================
# Validation predictions and classification metrics
# ============================================================

@torch.inference_mode()
def validation_predictions(model):

    model.eval()

    probabilities = []
    embeddings = []

    for first in range(
        0,
        len(x_val),
        BATCH_SIZE
    ):

        subset = x_val[
            first:first + BATCH_SIZE
        ]

        if model.use_graph:

            ei, ea = edges_for_batch(
                len(subset)
            )

            logits, embedding = model(
                subset,
                ei,
                ea
            )

        else:

            logits, embedding = model(
                subset
            )

        probabilities.extend(
            torch.softmax(
                logits,
                dim=1
            )[:, 1].cpu().numpy().tolist()
        )

        embeddings.append(
            embedding.cpu()
        )

    return (
        np.asarray(
            probabilities,
            dtype=float
        ),
        torch.cat(
            embeddings,
            dim=0
        )
    )


def metrics_at_threshold(
    y,
    probabilities,
    threshold
):

    labels = (
        probabilities >= threshold
    ).astype(int)

    tn = int(
        np.sum(
            (y == 0) & (labels == 0)
        )
    )

    fp = int(
        np.sum(
            (y == 0) & (labels == 1)
        )
    )

    fn = int(
        np.sum(
            (y == 1) & (labels == 0)
        )
    )

    tp = int(
        np.sum(
            (y == 1) & (labels == 1)
        )
    )

    return {
        "mcc": float(
            matthews_corrcoef(
                y,
                labels
            )
        ),
        "f1_advanced": float(
            f1_score(
                y,
                labels,
                zero_division=0
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y,
                labels
            )
        ),
        "sensitivity_advanced": (
            tp / (tp + fn)
            if tp + fn
            else 0.0
        ),
        "specificity_early": (
            tn / (tn + fp)
            if tn + fp
            else 0.0
        ),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }


def optimal_validation_threshold(
    y,
    probabilities
):

    unique = np.unique(
        np.clip(
            probabilities,
            0,
            1
        )
    )

    cutoffs = np.unique(
        np.r_[
            0.0,
            (
                unique[:-1]
                + unique[1:]
            ) / 2,
            0.5,
            1.0
        ]
    )

    scored = [
        (
            metrics_at_threshold(
                y,
                probabilities,
                float(threshold)
            )["mcc"],
            float(threshold)
        )
        for threshold in cutoffs
    ]

    best = max(
        scored,
        key=lambda item: (
            item[0],
            -abs(item[1] - 0.5)
        )
    )

    return best[1]


# ============================================================
# Experiment configuration and reproducibility
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(
                4 * 1024 * 1024
            ),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


fingerprint_data = {
    "features_sha256": sha256_file(feature_file),
    "nodes_sha256": sha256_file(node_file),
    "labels_sha256": sha256_file(label_file),
    "graph_sha256": hashlib.sha256(
        functional_ei.tobytes()
        + functional_w.tobytes()
    ).hexdigest(),
    "model_sha256": sha256_file(model_path),
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "early_stop_patience": EARLY_STOP_PATIENCE
}

experiments = [
    {
        "name": "Graph_free_fusion",
        "use_graph": False,
        "mmd_weight": 0.0
    },
    {
        "name": "PPI_GAT_no_MMD",
        "use_graph": True,
        "mmd_weight": 0.0
    },
    {
        "name": "PPI_GAT_MMD",
        "use_graph": True,
        "mmd_weight": 0.02
    }
]


def save_torch(
    path,
    object_to_save
):

    temporary = path.with_name(
        path.name + ".tmp"
    )

    torch.save(
        object_to_save,
        temporary
    )

    os.replace(
        temporary,
        path
    )


# ============================================================
# Train, validate and checkpoint each experiment
# ============================================================

def train_experiment(config):

    name = config["name"]

    run_dir = CHECKPOINTS / name

    run_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    result_file = (
        METRICS / f"{name}_summary.json"
    )

    predictions_file = (
        METRICS / f"{name}_validation_predictions.csv"
    )

    history_file = (
        METRICS / f"{name}_epoch_history.csv"
    )

    best_file = (
        run_dir / "best_model.pt"
    )

    last_file = (
        run_dir / "last_training.pt"
    )

    signature = hashlib.sha256(
        json.dumps(
            {
                **fingerprint_data,
                **config
            },
            sort_keys=True
        ).encode()
    ).hexdigest()

    # Reuse a completed run with matching inputs
    if result_file.exists():

        existing = json.loads(
            result_file.read_text(
                encoding="utf-8"
            )
        )

        if existing.get("signature") != signature:
            raise RuntimeError(
                f"Existing {name} results use different settings."
            )

        if (
            best_file.exists()
            and predictions_file.exists()
        ):

            print(
                f"{name}: completed run reused"
            )

            return existing

    torch.manual_seed(SEED)

    model = Model(
        n_genes=N,
        hidden_dim=32,
        dropout=0.15,
        use_graph=config["use_graph"]
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=3,
            min_lr=1e-5
        )
    )

    start_epoch = 1
    best_ap = -1.0
    best_epoch = 0
    bad_epochs = 0
    history = []

    # Resume interrupted training
    if last_file.exists():

        state = torch.load(
            last_file,
            map_location="cpu",
            weights_only=True
        )

        if state.get("signature") != signature:
            raise RuntimeError(
                f"Checkpoint signature mismatch: {name}"
            )

        model.load_state_dict(
            state["model"]
        )

        optimizer.load_state_dict(
            state["optimizer"]
        )

        scheduler.load_state_dict(
            state["scheduler"]
        )

        torch.set_rng_state(
            state["rng_state"]
        )

        start_epoch = (
            int(state["epoch"]) + 1
        )

        best_ap = float(
            state["best_ap"]
        )

        best_epoch = int(
            state["best_epoch"]
        )

        bad_epochs = int(
            state["bad_epochs"]
        )

        history = state["history"]

        print(
            f"{name}: resuming from epoch {start_epoch}"
        )

    for epoch in range(
        start_epoch,
        MAX_EPOCHS + 1
    ):

        model.train()

        batch_losses = []
        batch_ce = []
        batch_mmd = []

        for indices in balanced_batches(
            SEED + 1000 * epoch
        ):

            xb = x_train[indices]
            yb = y_train[indices]
            hb = h_train[indices]

            optimizer.zero_grad(
                set_to_none=True
            )

            if model.use_graph:

                ei, ea = edges_for_batch(
                    len(indices)
                )

                logits, embedding = model(
                    xb,
                    ei,
                    ea
                )

            else:

                logits, embedding = model(
                    xb
                )

            classification_loss = (
                torch.nn.functional.cross_entropy(
                    logits,
                    yb
                )
            )

            if config["mmd_weight"]:

                alignment_loss = conditional_mmd(
                    embedding,
                    yb,
                    hb
                )

            else:

                alignment_loss = (
                    embedding.sum() * 0.0
                )

            total_loss = (
                classification_loss
                + float(
                    config["mmd_weight"]
                ) * alignment_loss
            )

            if not torch.isfinite(
                total_loss
            ).item():

                raise RuntimeError(
                    f"Non-finite loss: {name}, epoch {epoch}"
                )

            total_loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=2.0,
                error_if_nonfinite=True
            )

            optimizer.step()

            batch_losses.append(
                float(
                    total_loss.detach()
                )
            )

            batch_ce.append(
                float(
                    classification_loss.detach()
                )
            )

            batch_mmd.append(
                float(
                    alignment_loss.detach()
                )
            )

        # Validation predictions
        p_val, _ = validation_predictions(
            model
        )

        y_reference = y_val.numpy()

        auprc = float(
            average_precision_score(
                y_reference,
                p_val
            )
        )

        auc = float(
            roc_auc_score(
                y_reference,
                p_val
            )
        )

        if (
            not np.isfinite(auprc)
            or not np.isfinite(auc)
        ):
            raise RuntimeError(
                "Non-finite validation metrics."
            )

        scheduler.step(auprc)

        progress = {
            "epoch": epoch,
            "train_loss": float(
                np.mean(batch_losses)
            ),
            "train_ce": float(
                np.mean(batch_ce)
            ),
            "train_mmd": float(
                np.mean(batch_mmd)
            ),
            "val_auprc": auprc,
            "val_roc_auc": auc,
            "val_mcc_at_0_5": (
                metrics_at_threshold(
                    y_reference,
                    p_val,
                    0.5
                )["mcc"]
            ),
            "lr": float(
                optimizer.param_groups[0]["lr"]
            )
        }

        history.append(progress)

        # Save the best validation checkpoint
        if auprc > best_ap + 1e-4:

            best_ap = auprc
            best_epoch = epoch
            bad_epochs = 0

            save_torch(
                best_file,
                {
                    "model": model.state_dict(),
                    "config": config,
                    "signature": signature,
                    "epoch": epoch,
                    "validation_auprc": auprc
                }
            )

        else:

            bad_epochs += 1

        # Save the latest training state
        save_torch(
            last_file,
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": epoch,
                "best_ap": best_ap,
                "best_epoch": best_epoch,
                "bad_epochs": bad_epochs,
                "history": history,
                "rng_state": torch.get_rng_state(),
                "signature": signature
            }
        )

        pd.DataFrame(
            history
        ).to_csv(
            history_file,
            index=False
        )

        if (
            epoch == 1
            or epoch % 5 == 0
            or bad_epochs >= EARLY_STOP_PATIENCE
        ):

            print(
                f"{name} | epoch {epoch:02d} | "
                f"loss={progress['train_loss']:.4f} | "
                f"val AP={auprc:.4f} | "
                f"val AUC={auc:.4f} | "
                f"best AP={best_ap:.4f}",
                flush=True
            )

        if (
            epoch >= MIN_EPOCHS
            and bad_epochs >= EARLY_STOP_PATIENCE
        ):
            break

    if not best_file.exists():
        raise RuntimeError(
            f"Best checkpoint missing: {name}"
        )

    # Restore the best model
    best = torch.load(
        best_file,
        map_location="cpu",
        weights_only=True
    )

    model.load_state_dict(
        best["model"]
    )

    p_best, embeddings = validation_predictions(
        model
    )

    y_reference = y_val.numpy()

    threshold = optimal_validation_threshold(
        y_reference,
        p_best
    )

    val_metrics = {
        "auprc": float(
            average_precision_score(
                y_reference,
                p_best
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                y_reference,
                p_best
            )
        ),
        "threshold_selected_on_validation": float(
            threshold
        ),
        **{
            f"at_selected_{key}": value
            for key, value in metrics_at_threshold(
                y_reference,
                p_best,
                threshold
            ).items()
        },
        "mcc_at_0_5": (
            metrics_at_threshold(
                y_reference,
                p_best,
                0.5
            )["mcc"]
        ),
        "histology_conditional_mmd_validation": float(
            conditional_mmd(
                embeddings,
                y_val,
                h_val
            ).item()
        )
    }

    # Save validation probabilities for reproducibility
    pd.DataFrame({
        "case_id": patients.case_id.iloc[
            val_idx
        ].to_numpy(),
        "stage_label": y_reference,
        "histology_label": h_val.numpy(),
        "prob_advanced": p_best
    }).to_csv(
        predictions_file,
        index=False
    )

    result = {
        "experiment": name,
        "signature": signature,
        "parameters": int(
            sum(
                parameter.numel()
                for parameter in model.parameters()
            )
        ),
        "use_ppi_graph": bool(
            config["use_graph"]
        ),
        "mmd_weight": float(
            config["mmd_weight"]
        ),
        "best_epoch": int(
            best_epoch
        ),
        "epochs_completed": len(
            history
        ),
        "validation": val_metrics,
        "checkpoint": str(
            best_file
        ),
        "training_history": str(
            history_file
        ),
        "validation_predictions": str(
            predictions_file
        ),
        "test_set_evaluated": False
    }

    temporary = result_file.with_suffix(
        ".json.tmp"
    )

    temporary.write_text(
        json.dumps(
            result,
            indent=2
        ),
        encoding="utf-8"
    )

    os.replace(
        temporary,
        result_file
    )

    return result


# ============================================================
# Run the predeclared model comparisons
# ============================================================

results = []

for config in experiments:

    print("\n" + "=" * 72)
    print("TRAINING:", config["name"])
    print("=" * 72)

    results.append(
        train_experiment(config)
    )


# Compare models using validation performance
comparison = pd.DataFrame([
    {
        "model": result["experiment"],
        "params": result["parameters"],
        "best_epoch": result["best_epoch"],
        "validation_PR_AUC": (
            result["validation"]["auprc"]
        ),
        "validation_ROC_AUC": (
            result["validation"]["roc_auc"]
        ),
        "validation_MCC": (
            result["validation"]["at_selected_mcc"]
        ),
        "validation_threshold": (
            result["validation"][
                "threshold_selected_on_validation"
            ]
        ),
        "validation_histology_MMD": (
            result["validation"][
                "histology_conditional_mmd_validation"
            ]
        )
    }
    for result in results
])

comparison = comparison.sort_values(
    [
        "validation_PR_AUC",
        "validation_ROC_AUC"
    ],
    ascending=False
)

comparison.to_csv(
    METRICS / "stage10_validation_comparison.csv",
    index=False
)

selected = comparison.iloc[0]["model"]

summary = {
    "generated_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "patients": len(patients),
    "train": len(train_idx),
    "validation": len(val_idx),
    "test_untouched": len(test_idx),
    "selected_genes": N,
    "functional_graph_corrected": functional_stats,
    "physical_graph_corrected": physical_stats,
    "model_selected_on_validation": selected,
    "experiments": results,
    "test_set_evaluated": False
}

(
    METRICS / "stage10_summary.json"
).write_text(
    json.dumps(
        summary,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 72)
print("STAGE 10 FINAL VALIDATION RESULTS")
print("=" * 72)

print(
    comparison.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

print("\nSelected on validation only:", selected)
print("Untouched test patients:", len(test_idx))
print("Model code:", model_path)
print("Checkpoint directory:", CHECKPOINTS)
print("Metrics directory:", METRICS)

print("\nSTAGE 10 COMPLETED SUCCESSFULLY")

STAGE 10 - CORRECTED STRING NETWORK AND MODEL DEVELOPMENT
Patients: 953 | Nodes: 1600
Train / validation / untouched test: 667 143 143

Corrected functional graph: {'source_rows': 10826054, 'undirected_edges': 2827, 'directed_edges': 5654, 'nodes_with_neighbors': 1118, 'isolated_nodes': 482, 'max_degree': 90, 'minimum_score': 700}
Corrected physical graph: {'source_rows': 1296490, 'undirected_edges': 2926, 'directed_edges': 5852, 'nodes_with_neighbors': 1053, 'isolated_nodes': 547, 'max_degree': 102, 'minimum_score': 400}

Training feature tensor: (667, 1600, 8)
Validation feature tensor: (143, 1600, 8)
Test features not loaded or evaluated.


C:\Users\M R Computer\AppData\Local\Temp\ipykernel_4540\3789839022.py:714: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  y_train = torch.as_tensor(



TRAINING: Graph_free_fusion
Graph_free_fusion | epoch 01 | loss=0.6919 | val AP=0.2713 | val AUC=0.5127 | best AP=0.2713
Graph_free_fusion | epoch 05 | loss=0.6470 | val AP=0.3499 | val AUC=0.6503 | best AP=0.3499
Graph_free_fusion | epoch 10 | loss=0.1534 | val AP=0.2673 | val AUC=0.6177 | best AP=0.3499
Graph_free_fusion | epoch 13 | loss=0.0659 | val AP=0.2657 | val AUC=0.6152 | best AP=0.3499

TRAINING: PPI_GAT_no_MMD
PPI_GAT_no_MMD | epoch 01 | loss=0.6909 | val AP=0.2887 | val AUC=0.5423 | best AP=0.2887
PPI_GAT_no_MMD | epoch 05 | loss=0.6709 | val AP=0.3090 | val AUC=0.5820 | best AP=0.3090
PPI_GAT_no_MMD | epoch 10 | loss=0.3632 | val AP=0.2576 | val AUC=0.5920 | best AP=0.3254
PPI_GAT_no_MMD | epoch 15 | loss=0.0878 | val AP=0.2587 | val AUC=0.5511 | best AP=0.3254

TRAINING: PPI_GAT_MMD
PPI_GAT_MMD | epoch 01 | loss=0.6956 | val AP=0.2832 | val AUC=0.5357 | best AP=0.2832
PPI_GAT_MMD | epoch 05 | loss=0.6800 | val AP=0.3031 | val AUC=0.5950 | best AP=0.3031
PPI_GAT_MMD | ep

In [10]:

import hashlib
import json
import os
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    matthews_corrcoef,
    roc_auc_score
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from threadpoolctl import threadpool_limits

# Project paths and experiment settings
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) Jupyter kernel."
    )

INPUT = ROOT / "data" / "processed" / "ppi_gat_v1"
LOCKED = ROOT / "data" / "splits" / "locked_four_omics_v1.csv"
OUTPUT = ROOT / "results" / "metrics" / "stage11"
MODELS = ROOT / "models" / "checkpoints" / "stage11"

OUTPUT.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

SEED = 42
N_FOLDS = 4
N_BOOTSTRAP = 600
SVD_COMPONENTS = 64
C_GRID = (0.003, 0.03, 0.3)
CLASS_WEIGHTS = (None, "balanced")
THREADS = min(4, os.cpu_count() or 1)

warnings.filterwarnings(
    "once",
    category=ConvergenceWarning
)

# Load Stage 09 features and locked patient assignments
features = np.load(
    INPUT / "patient_gene_features_8ch.npy",
    mmap_mode="r"
)

patients = pd.read_csv(
    INPUT / "patient_labels_and_splits.csv",
    dtype={"case_id": str}
)

locked = pd.read_csv(
    LOCKED,
    dtype={"case_id": str}
)

if (
    len(patients) != 953
    or features.shape != (953, 1600, 8)
    or not patients.case_id.is_unique
    or not locked.case_id.is_unique
):
    raise RuntimeError(
        "Unexpected cohort, feature dimensions, or duplicate patient IDs."
    )

check = patients.merge(
    locked[
        ["case_id", "histology", "stage_group", "split"]
    ],
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_current", "_locked")
)

for column in ("histology", "stage_group", "split"):

    if not check[
        f"{column}_current"
    ].equals(
        check[f"{column}_locked"]
    ):
        raise RuntimeError(
            f"Locked patient {column} assignments have changed."
        )

tr_idx = np.flatnonzero(
    patients.split.eq("train").to_numpy()
)

va_idx = np.flatnonzero(
    patients.split.eq("validation").to_numpy()
)

te_idx = np.flatnonzero(
    patients.split.eq("test").to_numpy()
)

if (
    len(tr_idx),
    len(va_idx),
    len(te_idx)
) != (667, 143, 143):
    raise RuntimeError(
        "Original locked split sizes have changed."
    )

# Load training and validation features only
train = np.asarray(
    features[tr_idx],
    dtype=np.float32
).copy()

val = np.asarray(
    features[va_idx],
    dtype=np.float32
).copy()

if (
    not np.isfinite(train).all()
    or not np.isfinite(val).all()
):
    raise RuntimeError(
        "Non-finite training or validation features."
    )

y_tr = patients.stage_label.iloc[
    tr_idx
].to_numpy(
    dtype=np.int64
)

y_va = patients.stage_label.iloc[
    va_idx
].to_numpy(
    dtype=np.int64
)

h_tr = patients.histology_label.iloc[
    tr_idx
].to_numpy(
    dtype=np.int64
)

h_va = patients.histology_label.iloc[
    va_idx
].to_numpy(
    dtype=np.int64
)

if (
    set(np.unique(y_tr)) != {0, 1}
    or set(np.unique(y_va)) != {0, 1}
):
    raise RuntimeError(
        "Both stage classes are required."
    )

print("=" * 74)
print("STAGE 11 - CROSS-VALIDATED BASELINES AND ERROR ANALYSIS")
print("=" * 74)

print(
    f"Patients: train={len(tr_idx)}, "
    f"validation={len(va_idx)}, "
    f"untouched test={len(te_idx)}"
)

print(
    f"Advanced-stage prevalence: "
    f"train={y_tr.mean():.3f}, "
    f"validation={y_va.mean():.3f}"
)

print(
    "Validation PR-AUC chance reference:",
    round(float(y_va.mean()), 4)
)

# Construct modality-specific feature views
CHANNELS = {
    "RNA_L2": (0, 4),
    "CNV_L2": (1, 5),
    "Mutation_L2": (2,),
    "Methylation_L2": (3, 7),
    "MultiOmics_L2": (0, 1, 2, 3, 4, 5, 7),
    "MultiOmics_SVD64_L2": (0, 1, 2, 3, 4, 5, 7),
    "MultiOmics_RandomForest": (0, 1, 2, 3, 4, 5, 7),
    "Missingness_only_L2": (4, 5, 7)
}


def view_matrix(tensor, channel_tuple):

    return np.concatenate(
        [
            tensor[:, :, channel]
            for channel in channel_tuple
        ],
        axis=1
    ).astype(
        np.float32,
        copy=False
    )


view_train = {}
view_val = {}

for name, channels in CHANNELS.items():

    view_train[name] = view_matrix(
        train,
        channels
    )

    view_val[name] = view_matrix(
        val,
        channels
    )

# Histology-only and missingness-only negative controls
view_train["Histology_only_L2"] = (
    h_tr[:, None].astype(np.float32)
)

view_val["Histology_only_L2"] = (
    h_va[:, None].astype(np.float32)
)

# Stratify CV folds jointly by stage and histology
strata = 2 * y_tr + h_tr

cv = list(
    StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SEED
    ).split(
        np.arange(len(y_tr)),
        strata
    )
)

for inner_tr, inner_va in cv:

    if set(
        np.unique(
            strata[inner_va]
        )
    ) != {0, 1, 2, 3}:
        raise RuntimeError(
            "A CV fold is missing a histology-stage group."
        )

print("\nTraining CV:", N_FOLDS, "stratified folds")

# Fit SVD separately inside each CV training fold
svd_fold_data = []

print("Preparing fold-specific SVD representations...")

with threadpool_limits(limits=THREADS):

    for fold, (inner_tr, inner_va) in enumerate(cv):

        base = view_train[
            "MultiOmics_SVD64_L2"
        ]

        svd = TruncatedSVD(
            n_components=SVD_COMPONENTS,
            n_iter=4,
            random_state=SEED + fold
        )

        transformed_train = svd.fit_transform(
            base[inner_tr]
        )

        transformed_val = svd.transform(
            base[inner_va]
        )

        svd_fold_data.append(
            (
                transformed_train,
                transformed_val
            )
        )

print("Fold-specific SVD: PASSED")


# Define model candidates
def logistic_estimator(C, weight):

    return LogisticRegression(
        solver="liblinear",
        C=float(C),
        class_weight=weight,
        max_iter=2000,
        tol=1e-4,
        random_state=SEED
    )


def family_candidates(name):

    if name == "MultiOmics_RandomForest":

        return [
            {
                "kind": "forest",
                "trees": 200,
                "max_depth": 8,
                "min_leaf": 5
            }
        ]

    return [
        {
            "kind": "logistic",
            "C": C,
            "class_weight": weight
        }
        for C in C_GRID
        for weight in CLASS_WEIGHTS
    ]


# Generate out-of-fold predictions
def fold_predictions(name, candidate):

    data = view_train[name]

    oof = np.full(
        len(y_tr),
        np.nan,
        dtype=np.float64
    )

    fold_ap = []
    fold_auc = []

    for fold, (inner_tr, inner_va) in enumerate(cv):

        if name == "MultiOmics_SVD64_L2":

            x_fit, x_predict = (
                svd_fold_data[fold]
            )

        else:

            x_fit = data[inner_tr]
            x_predict = data[inner_va]

        if candidate["kind"] == "forest":

            estimator = RandomForestClassifier(
                n_estimators=candidate["trees"],
                max_depth=candidate["max_depth"],
                min_samples_leaf=candidate["min_leaf"],
                max_features="sqrt",
                class_weight="balanced_subsample",
                n_jobs=THREADS,
                random_state=SEED + fold
            )

        else:

            estimator = logistic_estimator(
                candidate["C"],
                candidate["class_weight"]
            )

        estimator.fit(
            x_fit,
            y_tr[inner_tr]
        )

        probabilities = (
            estimator.predict_proba(
                x_predict
            )[:, 1]
        )

        if not np.isfinite(probabilities).all():

            raise RuntimeError(
                "Non-finite out-of-fold probabilities."
            )

        oof[inner_va] = probabilities

        fold_ap.append(
            float(
                average_precision_score(
                    y_tr[inner_va],
                    probabilities
                )
            )
        )

        fold_auc.append(
            float(
                roc_auc_score(
                    y_tr[inner_va],
                    probabilities
                )
            )
        )

    if not np.isfinite(oof).all():
        raise RuntimeError(
            "Incomplete training-only OOF predictions."
        )

    return oof, fold_ap, fold_auc


# Select the decision threshold using training OOF predictions
def threshold_from_oof(y, probabilities):

    thresholds = np.unique(
        np.r_[
            np.linspace(
                0.025,
                0.975,
                191
            ),
            0.5
        ]
    )

    candidates = [
        (
            float(
                matthews_corrcoef(
                    y,
                    probabilities >= threshold
                )
            ),
            float(threshold)
        )
        for threshold in thresholds
    ]

    return max(
        candidates,
        key=lambda pair: (
            pair[0],
            -abs(pair[1] - 0.5)
        )
    )[1]


# Calculate classification and probability metrics
def evaluate(y, probabilities, threshold):

    predicted = (
        probabilities >= threshold
    ).astype(np.int64)

    tp = int(
        (
            (y == 1)
            & (predicted == 1)
        ).sum()
    )

    fn = int(
        (
            (y == 1)
            & (predicted == 0)
        ).sum()
    )

    tn = int(
        (
            (y == 0)
            & (predicted == 0)
        ).sum()
    )

    fp = int(
        (
            (y == 0)
            & (predicted == 1)
        ).sum()
    )

    return {
        "PR_AUC": float(
            average_precision_score(
                y,
                probabilities
            )
        ),
        "ROC_AUC": float(
            roc_auc_score(
                y,
                probabilities
            )
        ),
        "MCC": float(
            matthews_corrcoef(
                y,
                predicted
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y,
                predicted
            )
        ),
        "recall_advanced": (
            float(tp / (tp + fn))
            if tp + fn
            else 0.0
        ),
        "specificity_early": (
            float(tn / (tn + fp))
            if tn + fp
            else 0.0
        ),
        "Brier_score_uncalibrated": float(
            brier_score_loss(
                y,
                probabilities
            )
        ),
        "threshold_from_training_OOF": float(
            threshold
        ),
        "tp": tp,
        "fn": fn,
        "tn": tn,
        "fp": fp
    }


# Prepare stratified patient-bootstrap samples
rng = np.random.default_rng(
    SEED + 500
)

bootstrap_indices = []

val_strata = (
    2 * y_va + h_va
)

for _ in range(N_BOOTSTRAP):

    draws = []

    for group in sorted(
        np.unique(val_strata)
    ):

        population = np.flatnonzero(
            val_strata == group
        )

        draws.append(
            rng.choice(
                population,
                size=len(population),
                replace=True
            )
        )

    bootstrap_indices.append(
        np.concatenate(draws)
    )


def interval(values):

    return [
        float(value)
        for value in np.percentile(
            values,
            [2.5, 97.5]
        )
    ]


def validation_intervals(
    probabilities,
    threshold
):

    pr = []
    auc = []
    mcc = []

    for indices in bootstrap_indices:

        y = y_va[indices]
        scores = probabilities[indices]

        pr.append(
            average_precision_score(
                y,
                scores
            )
        )

        auc.append(
            roc_auc_score(
                y,
                scores
            )
        )

        mcc.append(
            matthews_corrcoef(
                y,
                scores >= threshold
            )
        )

    return {
        "PR_AUC_95pct": interval(pr),
        "ROC_AUC_95pct": interval(auc),
        "MCC_95pct": interval(mcc)
    }


# Load the previous graph-free reference for comparison
old_reference = None

stage10_file = (
    ROOT
    / "results"
    / "metrics"
    / "stage10"
    / "Graph_free_fusion_validation_predictions.csv"
)

if stage10_file.exists():

    old = pd.read_csv(
        stage10_file,
        dtype={"case_id": str}
    )

    merged = patients.iloc[
        va_idx
    ][
        ["case_id", "stage_label"]
    ].merge(
        old[
            [
                "case_id",
                "stage_label",
                "prob_advanced"
            ]
        ],
        on="case_id",
        how="left",
        validate="one_to_one",
        suffixes=(
            "_current",
            "_old"
        )
    )

    if (
        len(old) == len(va_idx)
        and merged.prob_advanced.notna().all()
        and (
            merged.stage_label_current
            == merged.stage_label_old
        ).all()
    ):

        old_reference = (
            merged.prob_advanced.to_numpy(
                dtype=np.float64
            )
        )

        print(
            "\nStage 10 graph-free reference PR-AUC:",
            round(
                average_precision_score(
                    y_va,
                    old_reference
                ),
                4
            )
        )

    else:

        print(
            "\nStage 10 reference mismatch; "
            "paired comparison skipped."
        )


# Fingerprint the training and validation inputs
digest = hashlib.sha256()

for array in (
    train,
    val,
    y_tr,
    y_va,
    h_tr,
    h_va
):
    digest.update(
        array.tobytes()
    )

digest.update(
    "|".join(
        patients.case_id.astype(str)
    ).encode()
)

digest.update(
    str(
        (
            SEED,
            N_FOLDS,
            SVD_COMPONENTS,
            C_GRID,
            CLASS_WEIGHTS,
            N_BOOTSTRAP
        )
    ).encode()
)

signature = digest.hexdigest()


# Create final models for refitting on all training patients
def final_estimator(name, candidate):

    if candidate["kind"] == "forest":

        return RandomForestClassifier(
            n_estimators=candidate["trees"],
            max_depth=candidate["max_depth"],
            min_samples_leaf=candidate["min_leaf"],
            max_features="sqrt",
            class_weight="balanced_subsample",
            n_jobs=THREADS,
            random_state=SEED
        )

    classifier = logistic_estimator(
        candidate["C"],
        candidate["class_weight"]
    )

    if name == "MultiOmics_SVD64_L2":

        return Pipeline([
            (
                "svd",
                TruncatedSVD(
                    n_components=SVD_COMPONENTS,
                    n_iter=4,
                    random_state=SEED
                )
            ),
            (
                "classifier",
                classifier
            )
        ])

    return classifier


# Compare models using training-only cross-validation
results = []

for name in view_train:

    print("\n" + "-" * 74)
    print("Evaluating:", name, flush=True)

    result_path = (
        OUTPUT / f"{name}_summary.json"
    )

    prediction_path = (
        OUTPUT
        / f"{name}_validation_predictions.csv"
    )

    model_path = (
        MODELS / f"{name}.joblib"
    )

    # Reuse verified completed experiments
    if (
        result_path.exists()
        and prediction_path.exists()
        and model_path.exists()
    ):

        cached = json.loads(
            result_path.read_text(
                encoding="utf-8"
            )
        )

        if cached.get("signature") == signature:

            print(
                "Matching completed experiment reused."
            )

            results.append(cached)
            continue

    candidate_results = []
    best = None
    best_oof = None

    for config in family_candidates(name):

        with threadpool_limits(
            limits=THREADS
        ):

            oof, fold_ap, fold_auc = (
                fold_predictions(
                    name,
                    config
                )
            )

        candidate_result = {
            "configuration": config,
            "cv_mean_PR_AUC": float(
                np.mean(fold_ap)
            ),
            "cv_std_PR_AUC": float(
                np.std(
                    fold_ap,
                    ddof=1
                )
            ),
            "cv_mean_ROC_AUC": float(
                np.mean(fold_auc)
            ),
            "cv_pooled_OOF_PR_AUC": float(
                average_precision_score(
                    y_tr,
                    oof
                )
            ),
            "fold_PR_AUC": fold_ap,
            "fold_ROC_AUC": fold_auc
        }

        candidate_results.append(
            candidate_result
        )

        quality = (
            candidate_result[
                "cv_mean_PR_AUC"
            ],
            candidate_result[
                "cv_mean_ROC_AUC"
            ]
        )

        if (
            best is None
            or quality > (
                best["cv_mean_PR_AUC"],
                best["cv_mean_ROC_AUC"]
            )
        ):

            best = candidate_result
            best_oof = oof.copy()

    # Determine the threshold before evaluating validation
    threshold = threshold_from_oof(
        y_tr,
        best_oof
    )

    # Refit the CV-selected configuration on all training patients
    with threadpool_limits(
        limits=THREADS
    ):

        estimator = final_estimator(
            name,
            best["configuration"]
        )

        estimator.fit(
            view_train[name],
            y_tr
        )

        p_val = estimator.predict_proba(
            view_val[name]
        )[:, 1]

    if not np.isfinite(p_val).all():
        raise RuntimeError(
            f"Non-finite validation predictions: {name}"
        )

    validation = evaluate(
        y_va,
        p_val,
        threshold
    )

    validation.update(
        validation_intervals(
            p_val,
            threshold
        )
    )

    # Evaluate each histological subtype separately
    subgroup = {}

    for group, code in (
        ("LUAD", 0),
        ("LUSC", 1)
    ):

        keep = (
            h_va == code
        )

        subgroup[group] = {
            "patients": int(
                keep.sum()
            ),
            "advanced": int(
                y_va[keep].sum()
            ),
            **evaluate(
                y_va[keep],
                p_val[keep],
                threshold
            )
        }

    # Paired exploratory comparison with Stage 10
    paired = None

    if old_reference is not None:

        differences = [
            average_precision_score(
                y_va[indices],
                p_val[indices]
            )
            - average_precision_score(
                y_va[indices],
                old_reference[indices]
            )
            for indices in bootstrap_indices
        ]

        paired = {
            "PR_AUC_difference_vs_stage10_graphfree": float(
                validation["PR_AUC"]
                - average_precision_score(
                    y_va,
                    old_reference
                )
            ),
            "exploratory_paired_95pct_interval": interval(
                differences
            )
        }

    # Save validation predictions
    prediction_table = pd.DataFrame({
        "case_id": patients.case_id.iloc[
            va_idx
        ].to_numpy(),
        "histology": patients.histology.iloc[
            va_idx
        ].to_numpy(),
        "stage_label": y_va,
        "prob_advanced": p_val,
        "pred_advanced_oof_threshold": (
            p_val >= threshold
        ).astype(int)
    })

    prediction_table.to_csv(
        prediction_path,
        index=False
    )

    # Save the fitted model
    temporary_model = model_path.with_name(
        model_path.name + ".tmp"
    )

    joblib.dump(
        estimator,
        temporary_model,
        compress=3
    )

    os.replace(
        temporary_model,
        model_path
    )

    result = {
        "family": name,
        "signature": signature,
        "selected_by": (
            "training_stratified_4fold_CV_PR_AUC_only"
        ),
        "selected_hyperparameters": best[
            "configuration"
        ],
        "training_CV": {
            key: value
            for key, value in best.items()
            if key != "configuration"
        },
        "training_OOF_threshold": float(
            threshold
        ),
        "all_CV_candidates": candidate_results,
        "validation": validation,
        "validation_by_histology": subgroup,
        "paired_stage10_exploratory": paired,
        "model_path": str(
            model_path
        ),
        "validation_predictions": str(
            prediction_path
        ),
        "test_set_evaluated": False
    }

    temporary_result = result_path.with_suffix(
        ".json.tmp"
    )

    temporary_result.write_text(
        json.dumps(
            result,
            indent=2
        ),
        encoding="utf-8"
    )

    os.replace(
        temporary_result,
        result_path
    )

    results.append(result)

    print(
        f"CV PR-AUC="
        f"{best['cv_mean_PR_AUC']:.4f} "
        f"+/- {best['cv_std_PR_AUC']:.4f} | "
        f"Val PR-AUC={validation['PR_AUC']:.4f} | "
        f"Val AUC={validation['ROC_AUC']:.4f} | "
        f"Val MCC={validation['MCC']:.4f} | "
        f"OOF threshold={threshold:.3f}",
        flush=True
    )


# Build the complete model comparison
comparison = pd.DataFrame([
    {
        "family": result["family"],
        "CV_PR_AUC": result[
            "training_CV"
        ]["cv_mean_PR_AUC"],
        "CV_PR_AUC_SD": result[
            "training_CV"
        ]["cv_std_PR_AUC"],
        "CV_ROC_AUC": result[
            "training_CV"
        ]["cv_mean_ROC_AUC"],
        "val_PR_AUC": result[
            "validation"
        ]["PR_AUC"],
        "val_ROC_AUC": result[
            "validation"
        ]["ROC_AUC"],
        "val_MCC": result[
            "validation"
        ]["MCC"],
        "val_recall_advanced": result[
            "validation"
        ]["recall_advanced"],
        "val_Brier": result[
            "validation"
        ]["Brier_score_uncalibrated"],
        "train_OOF_threshold": result[
            "training_OOF_threshold"
        ]
    }
    for result in results
])

comparison = comparison.sort_values(
    [
        "CV_PR_AUC",
        "CV_ROC_AUC"
    ],
    ascending=False
)

comparison_path = (
    OUTPUT / "stage11_baseline_comparison.csv"
)

comparison.to_csv(
    comparison_path,
    index=False
)

# Keep diagnostic negative controls out of model selection
predictive_comparison = comparison.loc[
    ~comparison.family.isin([
        "Missingness_only_L2",
        "Histology_only_L2"
    ])
]

selected = str(
    predictive_comparison.iloc[0]["family"]
)

missingness = next(
    result
    for result in results
    if result["family"] == "Missingness_only_L2"
)

# Save the reproducible experiment summary
summary = {
    "generated_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "data_signature": signature,
    "train": len(tr_idx),
    "validation": len(va_idx),
    "untouched_test": len(te_idx),
    "validation_advanced_prevalence": float(
        y_va.mean()
    ),
    "CV_strata": "histology_by_stage",
    "CV_folds": N_FOLDS,
    "best_baseline_selected_using_training_CV_only": selected,
    "missingness_only_validation_PR_AUC": (
        missingness["validation"]["PR_AUC"]
    ),
    "stage09_unsupervised_gene_selection_fitted_on_full_train": True,
    "CV_caveat": (
        "Stage09 unsupervised gene selection and scaling "
        "were fitted on the complete training partition. "
        "Inner-CV results are not fully nested; validation "
        "patients were not used for feature fitting."
    ),
    "probability_calibration_assessed_by_brier_not_refitted": True,
    "no_test_features_loaded_or_evaluated": True
}

summary_path = (
    OUTPUT / "stage11_summary.json"
)

summary_path.write_text(
    json.dumps(
        summary,
        indent=2
    ),
    encoding="utf-8"
)

# Display final results
print("\n" + "=" * 74)
print("STAGE 11 FINAL BASELINE AND DIAGNOSTIC RESULTS")
print("=" * 74)

print(
    comparison.to_string(
        index=False,
        float_format=lambda number: f"{number:.4f}"
    )
)

print(
    "\nBest predictive baseline by training CV:",
    selected
)

print(
    "Missingness-only validation PR-AUC:",
    round(
        missingness["validation"]["PR_AUC"],
        4
    )
)

print(
    "Validation chance-level PR-AUC:",
    round(
        float(y_va.mean()),
        4
    )
)

print(
    "Untouched test patients:",
    len(te_idx)
)

print("Saved models:", MODELS)
print("Saved analyses:", OUTPUT)

print(
    "\nSTAGE 11 COMPLETED - TEST SET UNTOUCHED"
)

STAGE 11 - CROSS-VALIDATED BASELINES AND ERROR ANALYSIS
Patients: train=667, validation=143, untouched test=143
Advanced-stage prevalence: train=0.199, validation=0.203
Validation PR-AUC chance reference: 0.2028

Training CV: 4 stratified folds
Preparing fold-specific SVD representations...
Fold-specific SVD: PASSED

Stage 10 graph-free reference PR-AUC: 0.3499

--------------------------------------------------------------------------
Evaluating: RNA_L2
CV PR-AUC=0.2866 +/- 0.0572 | Val PR-AUC=0.3129 | Val AUC=0.6246 | Val MCC=0.1076 | OOF threshold=0.450

--------------------------------------------------------------------------
Evaluating: CNV_L2
CV PR-AUC=0.2688 +/- 0.0201 | Val PR-AUC=0.3493 | Val AUC=0.6428 | Val MCC=0.1673 | OOF threshold=0.270

--------------------------------------------------------------------------
Evaluating: Mutation_L2
CV PR-AUC=0.2248 +/- 0.0142 | Val PR-AUC=0.2053 | Val AUC=0.4752 | Val MCC=-0.0451 | OOF threshold=0.345

--------------------------------

In [11]:
import os
import sys
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score, matthews_corrcoef, balanced_accuracy_score, brier_score_loss
from threadpoolctl import threadpool_limits

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"
if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError("Select the NST NSCLC Research (Python 3.12) kernel.")
INPUT = ROOT / "data" / "processed" / "ppi_gat_v1"
OUT = ROOT / "results" / "metrics" / "stage12"
MODELS = ROOT / "models" / "checkpoints" / "stage12"
for folder in (OUT, MODELS):
    folder.mkdir(parents=True, exist_ok=True)

SEED = 42
FOLDS = 4
ALPHA = 0.45
GENE_COUNTS = [64, 160, 320]
C_GRID = [0.03, 0.3]
WEIGHTS = [None, "balanced"]
N_BOOTSTRAP = 400
THREADS = min(4, os.cpu_count() or 1)
VERSION = "stage12_crossvalidated_ppi_diffusion_v1"

paths = {
    "tensor": INPUT / "patient_gene_features_8ch.npy",
    "genes": INPUT / "selected_gene_nodes.csv",
    "patients": INPUT / "patient_labels_and_splits.csv",
    "functional": INPUT / "string_functional_700_corrected.npz",
    "physical": INPUT / "string_physical_400_corrected.npz"
}
for path in paths.values():
    if not path.is_file():
        raise FileNotFoundError(f"Missing Stage 10 artifact: {path}")

patients = pd.read_csv(paths["patients"], dtype={"case_id": str})
genes = pd.read_csv(paths["genes"], dtype={"ensembl_id": str, "gene_symbol": str})
loaded = np.load(paths["tensor"], mmap_mode="r")
n_patients, n_genes, channels = loaded.shape
if channels != 8 or n_patients != len(patients) or n_genes != len(genes):
    raise ValueError("Input array and metadata dimensions differ.")
if patients["case_id"].duplicated().any() or genes["node_index"].duplicated().any():
    raise ValueError("Duplicate identifiers in input metadata.")
if not np.array_equal(genes["node_index"].to_numpy(), np.arange(n_genes)):
    raise ValueError("Graph node ordering differs from feature tensor.")

indices = {s: np.flatnonzero(patients["split"].eq(s).to_numpy()) for s in ("train", "validation", "test")}
if [len(indices[s]) for s in ("train", "validation", "test")] != [667, 143, 143]:
    raise ValueError("The locked 667/143/143 patient split has changed.")
train_idx, val_idx, test_idx = (indices[s] for s in ("train", "validation", "test"))
Xtr = np.asarray(loaded[train_idx], dtype=np.float32).copy()
Xva = np.asarray(loaded[val_idx], dtype=np.float32).copy()
if not np.isfinite(Xtr).all() or not np.isfinite(Xva).all():
    raise ValueError("Non-finite training or validation features.")
if not np.isin(Xtr[..., 4:], [0, 1]).all() or not np.isin(Xva[..., 4:], [0, 1]).all():
    raise ValueError("Expected binary modality-availability masks.")
Rtr, Mtr = Xtr[..., :4], Xtr[..., 4:]
Rva, Mva = Xva[..., :4], Xva[..., 4:]
ytr = patients["stage_label"].to_numpy(dtype=np.int64)[train_idx]
yva = patients["stage_label"].to_numpy(dtype=np.int64)[val_idx]
htr = patients["histology_label"].to_numpy(dtype=np.int64)[train_idx]
hva = patients["histology_label"].to_numpy(dtype=np.int64)[val_idx]
if any(set(np.unique(v)) != {0, 1} for v in (ytr, yva, htr, hva)):
    raise ValueError("Stage and histology classes must both be present.")
strata = 2 * ytr + htr
folds = list(StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED).split(np.arange(len(ytr)), strata))
if any(set(strata[idx]) != {0, 1, 2, 3} for _, idx in folds):
    raise ValueError("A CV fold lacks a stage-histology stratum.")

print("=" * 72)
print("STAGE 12 - HISTOLOGY-ADJUSTED GENE SIGNAL AND PPI CONTROL")
print("=" * 72)
print(f"Patients: train={len(ytr)}, validation={len(yva)}, sealed test={len(test_idx)}")
print(f"Nodes: {n_genes}; advanced prevalence: train={ytr.mean():.3f}, val={yva.mean():.3f}")

# Audit feature availability within training-stage and histology groups.
audit_rows = []
for hist in (0, 1):
    for stage in (0, 1):
        group = (htr == hist) & (ytr == stage)
        row = {"histology": "LUSC" if hist else "LUAD", "stage": "Advanced" if stage else "Early", "patients": int(group.sum())}
        for i, name in enumerate(("RNA", "CNV", "Mutation_assayed", "Methylation")):
            fraction = Mtr[group, :, i].mean(axis=1)
            row[f"{name}_observed_mean"] = float(fraction.mean())
            row[f"{name}_observed_median"] = float(np.median(fraction))
        audit_rows.append(row)
coverage = pd.DataFrame(audit_rows)
coverage.to_csv(OUT / "training_availability_by_stage_histology.csv", index=False)
print("\nTraining-only modality availability:")
print(coverage.to_string(index=False, float_format=lambda value: f"{value:.3f}"))

# Test stage differences in assay completeness within each histology.
missingness_tests = []
for hist in (0, 1):
    for mod, name in ((0, "RNA"), (1, "CNV"), (3, "Methylation")):
        early = Mtr[(htr == hist) & (ytr == 0), :, mod].mean(axis=1)
        advanced = Mtr[(htr == hist) & (ytr == 1), :, mod].mean(axis=1)
        p = float(mannwhitneyu(early, advanced, alternative="two-sided").pvalue)
        missingness_tests.append({"histology": "LUSC" if hist else "LUAD",
                                  "modality": name,
                                  "early_mean": float(early.mean()),
                                  "advanced_mean": float(advanced.mean()),
                                  "raw_p": p})
corrected = multipletests([row["raw_p"] for row in missingness_tests], method="fdr_bh")[1]
for row, q in zip(missingness_tests, corrected):
    row["BH_FDR_q"] = float(q)
pd.DataFrame(missingness_tests).to_csv(OUT / "training_missingness_stage_tests.csv", index=False)
print("\nTraining-only stage-versus-availability audit (exploratory):")
print(pd.DataFrame(missingness_tests).to_string(index=False, float_format=lambda x: f"{x:.4g}"))

# Verify graph symmetry, weights and topology.
def load_graph(path):
    with np.load(path, allow_pickle=False) as bundle:
        edge_index = np.asarray(bundle["edge_index"], dtype=np.int64)
        weight = np.asarray(bundle["edge_weight"], dtype=np.float32)
    if edge_index.ndim != 2 or edge_index.shape[0] != 2 or len(weight) != edge_index.shape[1]:
        raise ValueError(f"Invalid edge format: {path}")
    if np.min(edge_index) < 0 or np.max(edge_index) >= n_genes or not np.isfinite(weight).all():
        raise ValueError(f"Out-of-range edge or confidence: {path}")
    graph = sparse.csr_matrix((weight, (edge_index[0], edge_index[1])), shape=(n_genes, n_genes), dtype=np.float32)
    graph.sum_duplicates()
    graph.setdiag(0)
    graph.eliminate_zeros()
    if (graph - graph.T).nnz:
        raise ValueError(f"Graph is not symmetric: {path}")
    if graph.nnz < 100:
        raise ValueError(f"Graph unexpectedly sparse: {path}")
    return graph

functional = load_graph(paths["functional"])
physical = load_graph(paths["physical"])
perm = np.random.default_rng(SEED + 202).permutation(n_genes)
coo = functional.tocoo()
randomized = sparse.csr_matrix((coo.data, (perm[coo.row], perm[coo.col])), shape=coo.shape, dtype=np.float32)
assert randomized.nnz == functional.nnz
assert (randomized - randomized.T).nnz == 0

graphs = {"functional": functional, "physical": physical, "randomized_functional": randomized}
print("\nGraph edges (undirected):", {name: graph.nnz // 2 for name, graph in graphs.items()})
print("Functional isolated nodes:", int((functional.getnnz(axis=0) == 0).sum()))

# Graph diffusion is per-patient, masked by observed modalities.
def smooth_features(raw, masks, adjacency, alpha=ALPHA):
    result = np.empty_like(raw, dtype=np.float32)
    for mod in range(4):
        value = raw[:, :, mod]
        observed = masks[:, :, mod]
        numerator = (adjacency @ (value * observed).T).T
        denominator = (adjacency @ observed.T).T
        neighbor = np.divide(numerator, denominator, out=np.zeros_like(numerator), where=denominator > 1e-8)
        result[:, :, mod] = np.where(
            denominator > 1e-8,
            (1.0 - alpha) * value + alpha * neighbor,
            value
        )
    if not np.isfinite(result).all():
        raise RuntimeError("Diffusion generated invalid values.")
    return np.clip(result, -5, 5)

smooth_train = {}
smooth_val = {}
for graph_name, adjacency in graphs.items():
    smooth_train[graph_name] = smooth_features(Rtr, Mtr, adjacency)
    smooth_val[graph_name] = smooth_features(Rva, Mva, adjacency)
    print(f"Graph diffusion prepared: {graph_name}")

# Score genes using stage effects estimated separately within LUAD and LUSC.
def gene_ranking(subset):
    y = ytr[subset]
    h = htr[subset]
    raw = Rtr[subset]
    observed = Mtr[subset]
    score_matrix = np.zeros((n_genes, 4), dtype=np.float64)
    for mod in range(4):
        hist_effects = []
        for hist in (0, 1):
            summaries = []
            for stage in (0, 1):
                selected = (h == hist) & (y == stage)
                xx = raw[selected, :, mod].astype(np.float64)
                mask = observed[selected, :, mod].astype(np.float64)
                n = mask.sum(axis=0)
                total = (xx * mask).sum(axis=0)
                average = total / np.maximum(n, 1.0)
                total2 = ((xx ** 2) * mask).sum(axis=0)
                variance = np.maximum((total2 - total * total / np.maximum(n, 1)) / np.maximum(n - 1, 1), 0.0)
                summaries.append((n, average, variance))
            n0, mean0, var0 = summaries[0]
            n1, mean1, var1 = summaries[1]
            se = np.sqrt(var0 / np.maximum(n0, 1) + var1 / np.maximum(n1, 1) + 0.08)
            effect = np.where((n0 >= 5) & (n1 >= 5), (mean1 - mean0) / se, 0.0)
            hist_effects.append(np.clip(effect, -8.0, 8.0))
        t0, t1 = hist_effects
        score_matrix[:, mod] = 0.65 * np.abs((t0 + t1) / 2) + 0.35 * np.maximum(np.abs(t0), np.abs(t1))
    overall = score_matrix.max(axis=1) + 0.10 * score_matrix.mean(axis=1)
    ranked = np.argsort(-overall, kind="stable")
    return ranked, overall, score_matrix

fold_rankings = [gene_ranking(inner_tr)[0] for inner_tr, _ in folds]
full_rank, full_scores, full_modal_scores = gene_ranking(np.arange(len(ytr)))
print("\nHistology-adjusted supervised ranking: fold-wise fit PASSED")

# Examine selection stability across overlapping training folds.
stability_records = []
for k in GENE_COUNTS:
    counts = np.zeros(n_genes, dtype=np.int32)
    top_sets = []
    for rank in fold_rankings:
        top = set(rank[:k].tolist())
        top_sets.append(top)
        counts[list(top)] += 1
    overlaps = [len(a & b) / max(1, len(a | b))
                for i, a in enumerate(top_sets) for b in top_sets[i + 1:]]
    stability_records.append({"selected_genes": k,
                              "fold_mean_jaccard": float(np.mean(overlaps)),
                              "fold_min_jaccard": float(np.min(overlaps))})
    genes[f"selected_in_folds_top{k}"] = counts
pd.DataFrame(stability_records).to_csv(OUT / "training_gene_selection_stability.csv", index=False)
genes[["node_index", "ensembl_id", "gene_symbol"]
      + [f"selected_in_folds_top{k}" for k in GENE_COUNTS]].to_csv(
    OUT / "training_gene_selection_frequency.csv", index=False
)
print("Fold-overlap gene selection stability:")
print(pd.DataFrame(stability_records).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# Construct compact feature views. Retain raw signal and missingness indicators.
def design(raw, observed, smoothed, selected_nodes, histology):
    pieces = [raw[:, selected_nodes, :], observed[:, selected_nodes, :][:, :, [0, 1, 3]]]
    if smoothed is not None:
        pieces.append(smoothed[:, selected_nodes, :])
    combined = np.concatenate(pieces, axis=2).reshape(len(raw), -1)
    combined = np.concatenate([combined, histology[:, None].astype(np.float32)], axis=1)
    return np.ascontiguousarray(combined, dtype=np.float32)

# Cache only fold-local feature views; no validation data during candidate selection.
fold_views = {}
for fold_no, (inner_tr, inner_va) in enumerate(folds):
    for k in GENE_COUNTS:
        selected_nodes = fold_rankings[fold_no][:k]
        for mode in ("graphfree", "functional", "physical"):
            smooth = smooth_train.get(mode) if mode != "graphfree" else None
            fold_views[(fold_no, k, mode)] = (
                design(Rtr[inner_tr], Mtr[inner_tr], None if smooth is None else smooth[inner_tr], selected_nodes, htr[inner_tr]),
                design(Rtr[inner_va], Mtr[inner_va], None if smooth is None else smooth[inner_va], selected_nodes, htr[inner_va])
            )

# Track candidate experiments and safely reuse completed results.
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

fingerprint = hashlib.sha256(
    json.dumps({"version": VERSION, "seed": SEED, "folds": FOLDS, "alpha": ALPHA,
                "genes": GENE_COUNTS, "C": C_GRID, "weight": WEIGHTS,
                "source_hashes": {key: sha256_file(path) for key, path in paths.items()}}, sort_keys=True).encode()
).hexdigest()

def estimator(C, weight):
    return Pipeline([
        ("standardize", StandardScaler()),
        ("classifier", LogisticRegression(solver="liblinear", C=float(C), class_weight=weight, max_iter=1000, random_state=SEED))
    ])

def cv_evaluate(mode, k, C, weight, use_cache=True):
    identifier = f"{mode}_k{k}_C{str(C).replace('.', 'p')}_w{weight or 'none'}"
    cache_json = OUT / "candidates" / (identifier + ".json")
    cache_npz = OUT / "candidates" / (identifier + ".npz")
    cache_json.parent.mkdir(parents=True, exist_ok=True)
    if use_cache and cache_json.exists() and cache_npz.exists():
        saved = json.loads(cache_json.read_text(encoding="utf-8"))
        if saved.get("fingerprint") == fingerprint:
            with np.load(cache_npz, allow_pickle=False) as npz:
                return saved, npz["oof"]
    probabilities = np.full(len(ytr), np.nan, dtype=np.float64)
    fold_AP, fold_AUC = [], []
    for f, (inner_tr, inner_va) in enumerate(folds):
        if mode == "randomized_functional":
            selected = fold_rankings[f][:k]
            Xfit = design(Rtr[inner_tr], Mtr[inner_tr], smooth_train[mode][inner_tr], selected, htr[inner_tr])
            Xpred = design(Rtr[inner_va], Mtr[inner_va], smooth_train[mode][inner_va], selected, htr[inner_va])
        else:
            Xfit, Xpred = fold_views[(f, k, mode)]
        model = estimator(C, weight)
        model.fit(Xfit, ytr[inner_tr])
        p = model.predict_proba(Xpred)[:, 1]
        probabilities[inner_va] = p
        fold_AP.append(float(average_precision_score(ytr[inner_va], p)))
        fold_AUC.append(float(roc_auc_score(ytr[inner_va], p)))
    if not np.isfinite(probabilities).all():
        raise RuntimeError(f"Incomplete CV predictions for {identifier}")
    summary = {
        "identifier": identifier, "mode": mode, "genes": int(k), "C": float(C), "class_weight": weight,
        "fingerprint": fingerprint,
        "CV_mean_PR_AUC": float(np.mean(fold_AP)),
        "CV_std_PR_AUC": float(np.std(fold_AP, ddof=1)),
        "CV_mean_ROC_AUC": float(np.mean(fold_AUC)),
        "CV_pooled_PR_AUC": float(average_precision_score(ytr, probabilities)),
        "fold_PR_AUC": fold_AP, "fold_ROC_AUC": fold_AUC
    }
    np.savez_compressed(cache_npz, oof=probabilities)
    temp = cache_json.with_suffix(".json.tmp")
    temp.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    os.replace(temp, cache_json)
    return summary, probabilities

all_candidates = []
with threadpool_limits(limits=THREADS):
    for mode in ("graphfree", "functional", "physical"):
        for k in GENE_COUNTS:
            for C in C_GRID:
                for weight in WEIGHTS:
                    candidate, _ = cv_evaluate(mode, k, C, weight)
                    all_candidates.append(candidate)
        candidates_mode = [r for r in all_candidates if r["mode"] == mode]
        current = max(candidates_mode, key=lambda r: (r["CV_mean_PR_AUC"], r["CV_mean_ROC_AUC"], -r["genes"]))
        print(f"{mode:11} best CV AP={current['CV_mean_PR_AUC']:.4f} +/- {current['CV_std_PR_AUC']:.4f} | k={current['genes']} C={current['C']} weight={current['class_weight']}", flush=True)

pd.DataFrame(all_candidates).drop(columns=["fold_PR_AUC", "fold_ROC_AUC"]).to_csv(OUT / "training_cv_all_candidates.csv", index=False)

best_candidates = {
    mode: max((row for row in all_candidates if row["mode"] == mode),
              key=lambda row: (row["CV_mean_PR_AUC"], row["CV_mean_ROC_AUC"], -row["genes"]))
    for mode in ("graphfree", "functional", "physical")
}

# Random graph uses precisely the functional model's selected hyperparameters.
ref = best_candidates["functional"]
with threadpool_limits(limits=THREADS):
    random_candidate, _ = cv_evaluate(
        "randomized_functional", ref["genes"], ref["C"], ref["class_weight"]
    )
best_candidates["randomized_functional"] = random_candidate
print(f"Randomized functional graph (fixed hyperparameters): CV AP={random_candidate['CV_mean_PR_AUC']:.4f}")

# Optimize threshold using OOF predictions from training patients only.
def select_oof_threshold(labels, scores):
    grid = np.unique(np.r_[np.linspace(0.025, 0.975, 191), 0.5])
    values = [(float(matthews_corrcoef(labels, scores >= t)), float(t)) for t in grid]
    return max(values, key=lambda row: (row[0], -abs(row[1] - 0.5)))[1]

def logit(values):
    clipped = np.clip(values, 1e-6, 1 - 1e-6)
    return np.log(clipped / (1 - clipped)).reshape(-1, 1)

def calc_metrics(y, p, threshold):
    predicted = p >= threshold
    tp = int(((y == 1) & predicted).sum())
    fn = int(((y == 1) & ~predicted).sum())
    tn = int(((y == 0) & ~predicted).sum())
    fp = int(((y == 0) & predicted).sum())
    return {
        "PR_AUC": float(average_precision_score(y, p)),
        "ROC_AUC": float(roc_auc_score(y, p)),
        "MCC": float(matthews_corrcoef(y, predicted)),
        "balanced_accuracy": float(balanced_accuracy_score(y, predicted)),
        "sensitivity": tp / (tp + fn) if tp + fn else 0.0,
        "specificity": tn / (tn + fp) if tn + fp else 0.0,
        "Brier": float(brier_score_loss(y, p)),
        "TP": tp, "FP": fp, "TN": tn, "FN": fn
    }

rng = np.random.default_rng(SEED + 650)
validation_strata = 2 * yva + hva
bootstrap_indices = [
    np.concatenate([
        rng.choice(np.flatnonzero(validation_strata == group),
                   size=int((validation_strata == group).sum()), replace=True)
        for group in sorted(np.unique(validation_strata))
    ]) for _ in range(N_BOOTSTRAP)
]

def confidence_intervals(p, threshold):
    scores = [calc_metrics(yva[i], p[i], threshold) for i in bootstrap_indices]
    return {f"{field}_95CI": [float(v) for v in np.percentile([r[field] for r in scores], [2.5, 97.5])]
            for field in ("PR_AUC", "ROC_AUC", "MCC")}

# Select the final model by CV only and evaluate each predeclared graph experiment.
model_summaries = []
for mode, chosen in best_candidates.items():
    identifier = chosen["identifier"]
    _, oof = cv_evaluate(mode, chosen["genes"], chosen["C"], chosen["class_weight"])
    calibrator = LogisticRegression(C=1.0, solver="lbfgs", max_iter=500)
    calibrator.fit(logit(oof), ytr)
    oof_calibrated = calibrator.predict_proba(logit(oof))[:, 1]
    threshold = select_oof_threshold(ytr, oof_calibrated)
    nodes_chosen = full_rank[:chosen["genes"]]
    smtr = None if mode == "graphfree" else smooth_train[mode]
    smva = None if mode == "graphfree" else smooth_val[mode]
    features_train = design(Rtr, Mtr, smtr, nodes_chosen, htr)
    features_val = design(Rva, Mva, smva, nodes_chosen, hva)
    classifier = estimator(chosen["C"], chosen["class_weight"])
    with threadpool_limits(limits=THREADS):
        classifier.fit(features_train, ytr)
        val_prob_uncalibrated = classifier.predict_proba(features_val)[:, 1]
    val_prob = calibrator.predict_proba(logit(val_prob_uncalibrated))[:, 1]
    metrics = calc_metrics(yva, val_prob, threshold)
    metrics["Brier_uncalibrated"] = float(brier_score_loss(yva, val_prob_uncalibrated))
    metrics["threshold_training_OOF"] = threshold
    metrics.update(confidence_intervals(val_prob, threshold))
    metrics["by_histology"] = {
        name: {"n": int((hva == code).sum()), **calc_metrics(yva[hva == code], val_prob[hva == code], threshold)}
        for name, code in (("LUAD", 0), ("LUSC", 1))
    }
    pd.DataFrame({"case_id": patients.case_id.iloc[val_idx].to_numpy(),
                  "histology": patients.histology.iloc[val_idx].to_numpy(),
                  "stage_label": yva, "prob_advanced": val_prob,
                  "uncalibrated_probability": val_prob_uncalibrated}).to_csv(
        OUT / f"{mode}_validation_predictions.csv", index=False
    )
    pd.DataFrame({"case_id": patients.case_id.iloc[train_idx].to_numpy(),
                  "stage_label": ytr, "histology_label": htr,
                  "oof_probability": oof, "oof_calibrated_probability": oof_calibrated}).to_csv(
        OUT / f"{mode}_training_OOF_predictions.csv", index=False
    )
    gene_table = genes.iloc[nodes_chosen].copy()
    gene_table["training_stage_score"] = full_scores[nodes_chosen]
    for m, name in enumerate(("rna", "cnv", "mutation", "methylation")):
        gene_table[f"{name}_training_score"] = full_modal_scores[nodes_chosen, m]
    gene_table.to_csv(OUT / f"{mode}_training_selected_genes.csv", index=False)
    save_file = MODELS / f"{mode}_trained.joblib"
    temp = save_file.with_name(save_file.name + ".tmp")
    joblib.dump({"classifier": classifier, "calibrator": calibrator,
                 "selected_node_indices": nodes_chosen,
                 "graph_mode": mode, "alpha": ALPHA,
                 "training_OOF_threshold": threshold,
                 "fingerprint": fingerprint}, temp, compress=3)
    os.replace(temp, save_file)
    model_summaries.append({"mode": mode, "CV_mean_PR_AUC": chosen["CV_mean_PR_AUC"],
                            "CV_std_PR_AUC": chosen["CV_std_PR_AUC"],
                            "gene_count": chosen["genes"], "C": chosen["C"],
                            "class_weight": chosen["class_weight"], "validation": metrics,
                            "model_path": str(save_file), "test_evaluated": False})
    print(f"Validation {mode:24} AP={metrics['PR_AUC']:.4f} AUC={metrics['ROC_AUC']:.4f} MCC={metrics['MCC']:.4f} Brier={metrics['Brier']:.4f}", flush=True)

comparison = pd.DataFrame([
    {"mode": row["mode"], "CV_PR_AUC": row["CV_mean_PR_AUC"],
     "CV_PR_AUC_SD": row["CV_std_PR_AUC"], "genes": row["gene_count"],
     "validation_PR_AUC": row["validation"]["PR_AUC"],
     "validation_ROC_AUC": row["validation"]["ROC_AUC"],
     "validation_MCC": row["validation"]["MCC"],
     "validation_Brier": row["validation"]["Brier"],
     "OOF_threshold": row["validation"]["threshold_training_OOF"]}
    for row in model_summaries
]).sort_values("CV_PR_AUC", ascending=False)
comparison.to_csv(OUT / "stage12_graph_comparison.csv", index=False)

# Read the earlier SVD baseline only for descriptive comparison.
svd_reference = None
old_prediction_file = ROOT / "results" / "metrics" / "stage11" / "MultiOmics_SVD64_L2_validation_predictions.csv"
if old_prediction_file.is_file():
    previous = pd.read_csv(old_prediction_file, dtype={"case_id": str})
    frame = patients.iloc[val_idx][["case_id", "stage_label"]].merge(
        previous[["case_id", "stage_label", "prob_advanced"]],
        on="case_id", how="left", validate="one_to_one", suffixes=("_current", "_previous")
    )
    if (len(previous) == len(val_idx) and frame["prob_advanced"].notna().all()
            and frame["stage_label_current"].equals(frame["stage_label_previous"])):
        p_old = frame["prob_advanced"].to_numpy(dtype=np.float64)
        svd_reference = {"PR_AUC": float(average_precision_score(yva, p_old)),
                         "ROC_AUC": float(roc_auc_score(yva, p_old))}
        print("Prior Stage 11 SVD validation reference:", svd_reference)
    else:
        print("Earlier SVD prediction file mismatched cohort; comparison skipped.")

selected_mode = max(
    (row for row in model_summaries if row["mode"] != "randomized_functional"),
    key=lambda row: row["CV_mean_PR_AUC"]
)["mode"]

report = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "version": VERSION, "fingerprint": fingerprint,
    "patients": {"training": len(train_idx), "validation": len(val_idx), "sealed_test": len(test_idx)},
    "feature_selection": "stage-associated, histology-adjusted, refitted within each CV fold",
    "graph_smoothing": "observed-neighbor-weighted, raw features retained, fixed alpha",
    "random_graph_control": "fixed permutation of gene assignments, functional-selected hyperparameters",
    "selected_model_based_only_on_training_CV": selected_mode,
    "models": model_summaries,
    "earlier_SVD_validation_reference": svd_reference,
    "training_feature_selection_stability": stability_records,
    "methodological_caveat": (
        "Stage 09 upstream gene filtering and scaling were fitted on the complete 667-patient training "
        "partition, so these training CV scores are not fully nested. Fold-specific Stage 12 supervised "
        "ranking and model scaling use fold-training patients only. Validation has been examined "
        "in earlier stages and is exploratory, not an untouched confirmation set."
    ),
    "test_features_loaded": False,
    "test_set_evaluated": False
}
(OUT / "stage12_summary.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("\n" + "=" * 72)
print("STAGE 12 FINAL RESULTS")
print("=" * 72)
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print("\nCV-selected model:", selected_mode)
print("Untouched test patients (features not loaded):", len(test_idx))
print("Outputs:", OUT)
print("STAGE 12 COMPLETED")

STAGE 12 - HISTOLOGY-ADJUSTED GENE SIGNAL AND PPI CONTROL
Patients: train=667, validation=143, sealed test=143
Nodes: 1600; advanced prevalence: train=0.199, val=0.203

Training-only modality availability:
histology    stage  patients  RNA_observed_mean  RNA_observed_median  CNV_observed_mean  CNV_observed_median  Mutation_assayed_observed_mean  Mutation_assayed_observed_median  Methylation_observed_mean  Methylation_observed_median
     LUAD    Early       265              1.000                1.000              0.995                0.999                           1.000                             1.000                      0.881                        0.996
     LUAD Advanced        73              1.000                1.000              0.994                0.999                           1.000                             1.000                      0.882                        0.996
     LUSC    Early       269              1.000                1.000              0.997              

D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\scipy\stats\_mannwhitneyu.py:157: RuntimeWarning: invalid value encountered in sqrt
  s = xp.sqrt(n1*n2/12 * ((n + 1) - tie_term/(n*(n-1))))



Graph edges (undirected): {'functional': 2827, 'physical': 2926, 'randomized_functional': 2827}
Functional isolated nodes: 482
Graph diffusion prepared: functional
Graph diffusion prepared: physical
Graph diffusion prepared: randomized_functional

Histology-adjusted supervised ranking: fold-wise fit PASSED
Fold-overlap gene selection stability:
 selected_genes  fold_mean_jaccard  fold_min_jaccard
             64              0.267             0.185
            160              0.298             0.236
            320              0.337             0.262
graphfree   best CV AP=0.3240 +/- 0.0394 | k=320 C=0.03 weight=balanced
functional  best CV AP=0.3104 +/- 0.0563 | k=320 C=0.03 weight=balanced
physical    best CV AP=0.2983 +/- 0.0413 | k=320 C=0.03 weight=balanced
Randomized functional graph (fixed hyperparameters): CV AP=0.3242
Validation graphfree                AP=0.2369 AUC=0.5384 MCC=0.1048 Brier=0.1690
Validation functional               AP=0.2496 AUC=0.5496 MCC=0.0509 Brier=0.1

In [12]:

import hashlib
import json
import os
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import networkx as nx
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import mannwhitneyu
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    matthews_corrcoef,
    roc_auc_score
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.multitest import multipletests
from threadpoolctl import threadpool_limits

# ============================================================
# 1. Configuration and input verification
# ============================================================

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

INPUT = ROOT / "data" / "processed" / "ppi_gat_v1"
OUTPUT = ROOT / "results" / "metrics" / "stage13"
MODEL_DIR = ROOT / "models" / "checkpoints" / "stage13"

OUTPUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
FOLDS = 4
THREADS = min(4, os.cpu_count() or 1)
N_BOOTSTRAP = 500
GRAPH_BLEND = 0.30

C_VALUES = (0.003, 0.03, 0.3, 3.0)
JOINT_C_VALUES = (0.003, 0.03, 0.3)
CLASS_WEIGHTS = (None, "balanced")

VERSION = "stage13_multiview_latent_01"

feature_path = INPUT / "patient_gene_features_8ch.npy"
label_path = INPUT / "patient_labels_and_splits.csv"
gene_path = INPUT / "selected_gene_nodes.csv"
graph_path = INPUT / "string_functional_700_corrected.npz"
physical_path = INPUT / "string_physical_400_corrected.npz"
locked_path = ROOT / "data" / "splits" / "locked_four_omics_v1.csv"

for path in (
    feature_path,
    label_path,
    gene_path,
    graph_path,
    physical_path,
    locked_path
):
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing Stage 09/10 artifact: {path}"
        )

patients = pd.read_csv(
    label_path,
    dtype={"case_id": str}
)

locked = pd.read_csv(
    locked_path,
    dtype={"case_id": str}
)

nodes = pd.read_csv(gene_path)

X = np.load(
    feature_path,
    mmap_mode="r"
)

if (
    X.shape != (953, 1600, 8)
    or len(patients) != 953
    or len(nodes) != 1600
):
    raise RuntimeError(
        f"Unexpected Stage 09 input dimensions: {X.shape}"
    )

if (
    not patients.case_id.is_unique
    or not locked.case_id.is_unique
    or not nodes.node_index.is_unique
):
    raise RuntimeError(
        "Duplicate patient or gene identifiers."
    )

if not np.array_equal(
    nodes.node_index.to_numpy(),
    np.arange(1600)
):
    raise RuntimeError(
        "Gene node ordering mismatch."
    )

comparison = patients[
    ["case_id", "histology", "stage_group", "split"]
].merge(
    locked[
        ["case_id", "histology", "stage_group", "split"]
    ],
    on="case_id",
    validate="one_to_one",
    how="left",
    suffixes=("_saved", "_locked")
)

if len(comparison) != len(patients):
    raise RuntimeError(
        "Locked cohort mismatch."
    )

for field in (
    "histology",
    "stage_group",
    "split"
):
    if not comparison[
        f"{field}_saved"
    ].equals(
        comparison[f"{field}_locked"]
    ):
        raise RuntimeError(
            f"Locked {field} mismatch."
        )

train_idx = np.flatnonzero(
    patients.split.eq("train").to_numpy()
)

val_idx = np.flatnonzero(
    patients.split.eq("validation").to_numpy()
)

test_idx = np.flatnonzero(
    patients.split.eq("test").to_numpy()
)

if (
    len(train_idx),
    len(val_idx),
    len(test_idx)
) != (667, 143, 143):
    raise RuntimeError(
        "Unexpected locked split sizes."
    )

# Load only training and validation features
Xtr = np.asarray(
    X[train_idx],
    dtype=np.float32
).copy()

Xva = np.asarray(
    X[val_idx],
    dtype=np.float32
).copy()

ytr = patients.stage_label.iloc[
    train_idx
].to_numpy(dtype=np.int32)

yva = patients.stage_label.iloc[
    val_idx
].to_numpy(dtype=np.int32)

htr = patients.histology_label.iloc[
    train_idx
].to_numpy(dtype=np.int32)

hva = patients.histology_label.iloc[
    val_idx
].to_numpy(dtype=np.int32)

if (
    not np.isfinite(Xtr).all()
    or not np.isfinite(Xva).all()
):
    raise RuntimeError(
        "Non-finite patient features detected."
    )

if (
    not np.isin(
        Xtr[:, :, 4:],
        (0, 1)
    ).all()
    or not np.isin(
        Xva[:, :, 4:],
        (0, 1)
    ).all()
):
    raise RuntimeError(
        "Invalid modality observation masks."
    )

print("=" * 75)
print(
    "STAGE 13 - MULTIVIEW LATENT SIGNAL "
    "AND PPI-PRIOR EVALUATION"
)
print("=" * 75)

print(
    f"Train={len(train_idx)}, "
    f"validation={len(val_idx)}, "
    f"sealed test={len(test_idx)}"
)

print(
    f"Nodes={len(nodes)}; "
    f"train advanced prevalence={ytr.mean():.3f}"
)


# ============================================================
# 2. Load confidence-weighted STRING networks
# ============================================================

NGENES = len(nodes)


def graph_from_file(path):

    with np.load(path) as graph:
        ei = graph["edge_index"].astype(np.int64)
        weights = graph["edge_weight"].astype(np.float32)

    if (
        ei.ndim != 2
        or ei.shape[0] != 2
        or len(weights) != ei.shape[1]
        or not np.isfinite(weights).all()
        or (weights <= 0).any()
        or ei.min() < 0
        or ei.max() >= NGENES
    ):
        raise RuntimeError(
            f"Invalid STRING graph: {path}"
        )

    adjacency = sparse.csr_matrix(
        (
            weights,
            (ei[0], ei[1])
        ),
        shape=(NGENES, NGENES)
    )

    adjacency.sum_duplicates()

    asymmetry = adjacency - adjacency.T

    if np.max(
        np.abs(asymmetry.data),
        initial=0
    ) > 1e-5:
        raise RuntimeError(
            f"Asymmetric graph: {path}"
        )

    adjacency.setdiag(0)
    adjacency.eliminate_zeros()

    return adjacency, ei, weights


A_functional, functional_ei, functional_w = (
    graph_from_file(graph_path)
)

A_physical, _, _ = graph_from_file(
    physical_path
)


# ============================================================
# 3. Construct degree-preserving randomized PPI control
# ============================================================

G = nx.from_scipy_sparse_array(
    A_functional,
    create_using=nx.Graph()
)

original_degrees = np.array([
    G.degree(i)
    for i in range(NGENES)
])

G_null = G.copy()

edge_count = G.number_of_edges()

if edge_count < 100:
    raise RuntimeError(
        "PPI graph is unexpectedly sparse."
    )

nx.double_edge_swap(
    G_null,
    nswap=3 * edge_count,
    max_tries=100 * edge_count,
    seed=SEED + 13
)

new_degrees = np.array([
    G_null.degree(i)
    for i in range(NGENES)
])

if (
    G_null.number_of_edges() != edge_count
    or not np.array_equal(
        original_degrees,
        new_degrees
    )
):
    raise RuntimeError(
        "Randomized graph changed gene degrees."
    )

original_pairs = np.array(
    sorted(G.edges()),
    dtype=np.int64
)

original_weights = np.asarray(
    A_functional[
        original_pairs[:, 0],
        original_pairs[:, 1]
    ]
).ravel()

if len(original_weights) != edge_count:
    raise RuntimeError(
        "Undirected STRING score extraction failed."
    )

rng = np.random.default_rng(
    SEED + 13
)

rng.shuffle(
    original_weights
)

null_pairs = np.array(
    sorted(G_null.edges()),
    dtype=np.int64
)

A_null = sparse.csr_matrix(
    (
        np.r_[
            original_weights,
            original_weights
        ],
        (
            np.r_[
                null_pairs[:, 0],
                null_pairs[:, 1]
            ],
            np.r_[
                null_pairs[:, 1],
                null_pairs[:, 0]
            ]
        )
    ),
    shape=(NGENES, NGENES)
)

A_null.sum_duplicates()

null_degree = np.array(
    (A_null > 0).sum(axis=1)
).ravel()

if not np.array_equal(
    original_degrees,
    null_degree
):
    raise RuntimeError(
        "Randomized graph degree mismatch."
    )

sparse.save_npz(
    OUTPUT / "degree_preserving_randomized_graph.npz",
    A_null
)

adjacencies = {
    "functional": A_functional,
    "physical": A_physical,
    "random": A_null
}

print(
    "\nFunctional edges:",
    edge_count
)

print(
    "Functional isolated genes:",
    int((original_degrees == 0).sum())
)

print(
    "Degree-preserving random control:",
    G_null.number_of_edges(),
    "edges | degree check: PASSED"
)


# ============================================================
# 4. Missingness and histology audit
# ============================================================

quality_rows = []

for hist_name, hist_id in [
    ("LUAD", 0),
    ("LUSC", 1)
]:

    for channel, label in (
        (4, "RNA"),
        (5, "CNV"),
        (7, "Methylation")
    ):

        observed = Xtr[
            :,
            :,
            channel
        ].mean(axis=1)

        early = observed[
            (htr == hist_id)
            & (ytr == 0)
        ]

        advanced = observed[
            (htr == hist_id)
            & (ytr == 1)
        ]

        if np.ptp(
            np.r_[early, advanced]
        ) < 1e-10:

            p = 1.0

        else:

            with warnings.catch_warnings():

                warnings.simplefilter(
                    "ignore",
                    RuntimeWarning
                )

                p = float(
                    mannwhitneyu(
                        early,
                        advanced,
                        alternative="two-sided"
                    ).pvalue
                )

            if not np.isfinite(p):
                p = 1.0

        quality_rows.append({
            "histology": hist_name,
            "modality": label,
            "early_mean_observed": float(
                early.mean()
            ),
            "advanced_mean_observed": float(
                advanced.mean()
            ),
            "raw_p": p
        })

quality = pd.DataFrame(
    quality_rows
)

quality["BH_FDR_q"] = multipletests(
    quality.raw_p.to_numpy(),
    method="fdr_bh"
)[1]

quality.to_csv(
    OUTPUT / "stage13_missingness_audit.csv",
    index=False
)

print(
    "\nTraining-only missingness audit "
    "(exploratory):"
)

print(
    quality.round(4).to_string(
        index=False
    )
)


# ============================================================
# 5. Define stratified training-only CV
# ============================================================

strata = 2 * ytr + htr

folds = list(
    StratifiedKFold(
        n_splits=FOLDS,
        shuffle=True,
        random_state=SEED
    ).split(
        np.arange(len(ytr)),
        strata
    )
)

for fit_ids, hold_ids in folds:

    if set(
        np.unique(
            strata[hold_ids]
        )
    ) != {0, 1, 2, 3}:

        raise RuntimeError(
            "CV fold missing a histology-stage stratum."
        )

print(
    "\nTraining-only CV folds:",
    FOLDS
)


# ============================================================
# 6. Predeclare model configurations
# ============================================================

EXPERIMENTS = [
    {
        "name": "Joint_SVD64_Reference",
        "kind": "joint",
        "dim": 64,
        "mods": (0, 1, 2, 3),
        "graph": "none",
        "hist": "none"
    },
    {
        "name": "MultiView_SVD16",
        "kind": "multi",
        "dim": 16,
        "mods": (0, 1, 2, 3),
        "graph": "none",
        "hist": "none"
    },
    {
        "name": "MultiView_SVD32",
        "kind": "multi",
        "dim": 32,
        "mods": (0, 1, 2, 3),
        "graph": "none",
        "hist": "none"
    },
    {
        "name": "MultiView_SVD32_Histology",
        "kind": "multi",
        "dim": 32,
        "mods": (0, 1, 2, 3),
        "graph": "none",
        "hist": "offset"
    },
    {
        "name": "MultiView_SVD32_HistologyInteractions",
        "kind": "multi",
        "dim": 32,
        "mods": (0, 1, 2, 3),
        "graph": "none",
        "hist": "interaction"
    },
    {
        "name": "MultiView_SVD32_NoMethylation",
        "kind": "multi",
        "dim": 32,
        "mods": (0, 1, 2),
        "graph": "none",
        "hist": "none"
    },
    {
        "name": "Hybrid_FunctionalPPI",
        "kind": "multi",
        "dim": 16,
        "mods": (0, 1, 2, 3),
        "graph": "functional",
        "hist": "none"
    },
    {
        "name": "Hybrid_PhysicalPPI",
        "kind": "multi",
        "dim": 16,
        "mods": (0, 1, 2, 3),
        "graph": "physical",
        "hist": "none"
    },
    {
        "name": "Hybrid_DegreeMatchedNull",
        "kind": "multi",
        "dim": 16,
        "mods": (0, 1, 2, 3),
        "graph": "random",
        "hist": "none"
    }
]

embedding_cache = {}


# ============================================================
# 7. Training-fold modality preprocessing
# ============================================================

def fold_center_scale(fit_arr, fit_mask):

    counts = fit_mask.sum(
        axis=0
    )

    centers = (
        fit_arr * fit_mask
    ).sum(
        axis=0
    ) / np.maximum(
        counts,
        1
    )

    variance = (
        (
            (fit_arr - centers)
            * fit_mask
        ) ** 2
    ).sum(
        axis=0
    ) / np.maximum(
        counts,
        1
    )

    scales = np.maximum(
        np.sqrt(
            np.maximum(
                variance,
                0.0
            )
        ),
        0.50
    )

    return (
        centers.astype(np.float32),
        scales.astype(np.float32)
    )


def transformed_modality(
    tensor,
    modality,
    center,
    scale
):

    values = tensor[
        :,
        :,
        modality
    ]

    observed = tensor[
        :,
        :,
        4 + modality
    ]

    processed = np.clip(
        (values - center) / scale,
        -5.0,
        5.0
    ) * observed

    return (
        np.asarray(
            processed,
            dtype=np.float32
        ),
        observed
    )


# ============================================================
# 8. Confidence-weighted PPI signal aggregation
# ============================================================

def neighbor_aggregation(
    features,
    observed,
    matrix
):

    numerator = np.asarray(
        features @ matrix.T,
        dtype=np.float32
    )

    denominator = np.asarray(
        observed @ matrix.T,
        dtype=np.float32
    )

    neighbor = np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator),
        where=denominator > 1e-8
    )

    neighbor = np.where(
        denominator > 1e-8,
        neighbor,
        features
    )

    combined = (
        (1.0 - GRAPH_BLEND) * features
        + GRAPH_BLEND * neighbor
    )

    return np.asarray(
        np.clip(
            combined,
            -5,
            5
        ) * observed,
        dtype=np.float32
    )


# ============================================================
# 9. Fit fold-specific latent representations
# ============================================================

def fit_latent(
    fit_array,
    eval_array,
    components,
    seed
):

    svd = TruncatedSVD(
        n_components=components,
        n_iter=3,
        n_oversamples=6,
        random_state=seed
    )

    z_fit = svd.fit_transform(
        fit_array
    )

    z_eval = svd.transform(
        eval_array
    )

    standardizer = StandardScaler()

    z_fit = np.clip(
        standardizer.fit_transform(
            z_fit
        ),
        -6,
        6
    ).astype(np.float32)

    z_eval = np.clip(
        standardizer.transform(
            z_eval
        ),
        -6,
        6
    ).astype(np.float32)

    return (
        z_fit,
        z_eval,
        (svd, standardizer)
    )


def add_histology(
    z_fit,
    z_eval,
    steps,
    spec,
    hist_fit,
    hist_eval
):

    if spec["hist"] in (
        "offset",
        "interaction"
    ):

        hf = (
            2 * hist_fit - 1
        ).reshape(
            -1,
            1
        ).astype(np.float32)

        he = (
            2 * hist_eval - 1
        ).reshape(
            -1,
            1
        ).astype(np.float32)

        if spec["hist"] == "interaction":

            z_fit = np.concatenate(
                (
                    z_fit,
                    hf,
                    0.5 * z_fit * hf
                ),
                axis=1
            )

            z_eval = np.concatenate(
                (
                    z_eval,
                    he,
                    0.5 * z_eval * he
                ),
                axis=1
            )

        else:

            z_fit = np.concatenate(
                (z_fit, hf),
                axis=1
            )

            z_eval = np.concatenate(
                (z_eval, he),
                axis=1
            )

    if (
        not np.isfinite(z_fit).all()
        or not np.isfinite(z_eval).all()
    ):
        raise RuntimeError(
            f"Non-finite representations for {spec['name']}"
        )

    return z_fit, z_eval, steps


def build_embeddings(
    spec,
    fold_number,
    fit_ids,
    hold_ids,
    final=False
):

    key = (
        "final" if final else fold_number,
        spec["kind"],
        spec["dim"],
        spec["mods"],
        spec["graph"]
    )

    if key in embedding_cache:

        z_fit, z_eval, steps = (
            embedding_cache[key]
        )

        return add_histology(
            z_fit,
            z_eval,
            steps,
            spec,
            hist_fit=htr[fit_ids],
            hist_eval=(
                hva
                if final
                else htr[hold_ids]
            )
        )

    fit_x = Xtr[fit_ids]

    eval_x = (
        Xva
        if final
        else Xtr[hold_ids]
    )

    hist_fit = htr[fit_ids]

    hist_eval = (
        hva
        if final
        else htr[hold_ids]
    )

    parts_fit = []
    parts_eval = []
    steps = []

    if spec["kind"] == "joint":

        feature_channels = (
            0, 1, 2, 3, 4, 5, 7
        )

        xf = np.concatenate(
            [
                fit_x[:, :, k]
                for k in feature_channels
            ],
            axis=1
        )

        xe = np.concatenate(
            [
                eval_x[:, :, k]
                for k in feature_channels
            ],
            axis=1
        )

        svd = TruncatedSVD(
            n_components=spec["dim"],
            n_iter=4,
            random_state=SEED + (
                0 if final else fold_number
            )
        )

        zf = svd.fit_transform(
            xf
        ).astype(np.float32)

        ze = svd.transform(
            xe
        ).astype(np.float32)

        steps.append({
            "kind": "joint",
            "channels": feature_channels,
            "svd": svd
        })

        parts_fit.append(zf)
        parts_eval.append(ze)

    else:

        for j in spec["mods"]:

            fit_mask = fit_x[
                :,
                :,
                4 + j
            ]

            center, scale = fold_center_scale(
                fit_x[:, :, j],
                fit_mask
            )

            xf, obs_f = transformed_modality(
                fit_x,
                j,
                center,
                scale
            )

            xe, obs_e = transformed_modality(
                eval_x,
                j,
                center,
                scale
            )

            local_step = {
                "kind": "modality",
                "modality": j,
                "center": center,
                "scale": scale,
                "latent": []
            }

            # Preserve original molecular information
            zf, ze, pair = fit_latent(
                xf,
                xe,
                spec["dim"],
                SEED + 111 * j + (
                    0 if final else fold_number
                )
            )

            parts_fit.append(zf)
            parts_eval.append(ze)

            local_step["latent"].append({
                "source": "raw",
                "svd": pair[0],
                "scaler": pair[1]
            })

            # Add PPI-informed latent factors
            if (
                spec["graph"] != "none"
                and j != 2
            ):

                matrix = adjacencies[
                    spec["graph"]
                ]

                gf = neighbor_aggregation(
                    xf,
                    obs_f,
                    matrix
                )

                ge = neighbor_aggregation(
                    xe,
                    obs_e,
                    matrix
                )

                zf, ze, pair = fit_latent(
                    gf,
                    ge,
                    spec["dim"],
                    SEED + 111 * j + (
                        0 if final else fold_number
                    ) + 999
                )

                parts_fit.append(zf)
                parts_eval.append(ze)

                local_step["latent"].append({
                    "source": spec["graph"],
                    "svd": pair[0],
                    "scaler": pair[1]
                })

            steps.append(local_step)

        # Include a normalized mutation-burden feature
        if 2 in spec["mods"]:

            burden_fit = np.log1p(
                fit_x[:, :, 2].sum(
                    axis=1
                )
            ).reshape(-1, 1)

            burden_eval = np.log1p(
                eval_x[:, :, 2].sum(
                    axis=1
                )
            ).reshape(-1, 1)

            standardizer = StandardScaler()

            parts_fit.append(
                standardizer.fit_transform(
                    burden_fit
                ).astype(np.float32)
            )

            parts_eval.append(
                standardizer.transform(
                    burden_eval
                ).astype(np.float32)
            )

            steps.append({
                "kind": "mutation_burden",
                "scaler": standardizer
            })

    z_fit = np.concatenate(
        parts_fit,
        axis=1
    ).astype(np.float32)

    z_eval = np.concatenate(
        parts_eval,
        axis=1
    ).astype(np.float32)

    embedding_cache[key] = (
        z_fit,
        z_eval,
        steps
    )

    return add_histology(
        z_fit,
        z_eval,
        steps,
        spec,
        hist_fit,
        hist_eval
    )


# ============================================================
# 10. Training-only probability calibration
# ============================================================

def sigmoid_calibrator(
    oof,
    truth
):

    clipped = np.clip(
        oof,
        1e-6,
        1 - 1e-6
    )

    logits = np.log(
        clipped / (1 - clipped)
    ).reshape(-1, 1)

    calibrator = LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        random_state=SEED
    )

    calibrator.fit(
        logits,
        truth
    )

    # Avoid reversing an unsuitable probability ranking
    if calibrator.coef_[0, 0] <= 0:
        return None

    return calibrator


def apply_calibrator(
    model,
    scores
):

    if model is None:
        return scores

    clipped = np.clip(
        scores,
        1e-6,
        1 - 1e-6
    )

    logits = np.log(
        clipped / (1 - clipped)
    ).reshape(-1, 1)

    return model.predict_proba(
        logits
    )[:, 1]


def best_oof_threshold(
    truth,
    probabilities
):

    thresholds = np.arange(
        0.04,
        0.961,
        0.01
    )

    scores = [
        (
            matthews_corrcoef(
                truth,
                probabilities >= threshold
            ),
            float(threshold)
        )
        for threshold in thresholds
    ]

    return max(
        scores,
        key=lambda pair: (
            pair[0],
            -abs(pair[1] - 0.5)
        )
    )[1]


# ============================================================
# 11. Classification and calibration metrics
# ============================================================

def binary_metrics(
    truth,
    probabilities,
    threshold
):

    labels = (
        probabilities >= threshold
    ).astype(np.int32)

    tp = int(
        (
            (truth == 1)
            & (labels == 1)
        ).sum()
    )

    tn = int(
        (
            (truth == 0)
            & (labels == 0)
        ).sum()
    )

    fp = int(
        (
            (truth == 0)
            & (labels == 1)
        ).sum()
    )

    fn = int(
        (
            (truth == 1)
            & (labels == 0)
        ).sum()
    )

    return {
        "AP": float(
            average_precision_score(
                truth,
                probabilities
            )
        ),
        "ROC_AUC": float(
            roc_auc_score(
                truth,
                probabilities
            )
        ),
        "MCC": float(
            matthews_corrcoef(
                truth,
                labels
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                truth,
                labels
            )
        ),
        "advanced_recall": (
            float(tp / (tp + fn))
            if tp + fn
            else 0.0
        ),
        "advanced_precision": (
            float(tp / (tp + fp))
            if tp + fp
            else 0.0
        ),
        "early_specificity": (
            float(tn / (tn + fp))
            if tn + fp
            else 0.0
        ),
        "Brier": float(
            brier_score_loss(
                truth,
                probabilities
            )
        ),
        "threshold": float(threshold),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }


# ============================================================
# 12. Experiment fingerprint and bootstrap samples
# ============================================================

def sha(path):

    digest = hashlib.sha256()

    with open(path, "rb") as stream:

        for piece in iter(
            lambda: stream.read(
                4 * 1024 * 1024
            ),
            b""
        ):
            digest.update(piece)

    return digest.hexdigest()


signature = hashlib.sha256(
    json.dumps(
        {
            "version": VERSION,
            "feature": sha(feature_path),
            "patients": sha(label_path),
            "nodes": sha(gene_path),
            "graph": sha(graph_path),
            "physical": sha(physical_path),
            "blend": GRAPH_BLEND,
            "seed": SEED,
            "folds": FOLDS,
            "c_values": C_VALUES,
            "joint_c_values": JOINT_C_VALUES,
            "weights": CLASS_WEIGHTS,
            "experiments": EXPERIMENTS
        },
        sort_keys=True
    ).encode()
).hexdigest()

rng = np.random.default_rng(
    SEED + 3013
)

val_strata = 2 * yva + hva

bootstrap = [
    np.concatenate(
        [
            rng.choice(
                ids,
                len(ids),
                replace=True
            )
            for group in np.unique(
                val_strata
            )
            for ids in [
                np.flatnonzero(
                    val_strata == group
                )
            ]
        ]
    )
    for _ in range(N_BOOTSTRAP)
]


# ============================================================
# 13. Load the Stage 11 validation reference
# ============================================================

reference_path = (
    ROOT
    / "results"
    / "metrics"
    / "stage11"
    / "MultiOmics_SVD64_L2_validation_predictions.csv"
)

reference_scores = None

if reference_path.exists():

    old = pd.read_csv(
        reference_path,
        dtype={"case_id": str}
    )

    aligned = patients.iloc[
        val_idx
    ][
        ["case_id", "stage_label"]
    ].merge(
        old[
            [
                "case_id",
                "stage_label",
                "prob_advanced"
            ]
        ],
        on="case_id",
        validate="one_to_one",
        how="left",
        suffixes=("_now", "_old")
    )

    if (
        len(old) == len(val_idx)
        and aligned.prob_advanced.notna().all()
        and (
            aligned.stage_label_now
            == aligned.stage_label_old
        ).all()
    ):

        reference_scores = (
            aligned.prob_advanced.to_numpy(
                dtype=float
            )
        )

        print(
            "\nStage 11 SVD reference validation AP:",
            round(
                float(
                    average_precision_score(
                        yva,
                        reference_scores
                    )
                ),
                4
            )
        )


# ============================================================
# 14. Cross-validated model selection and fitting
# ============================================================

results = []

with threadpool_limits(limits=THREADS):

    for spec in EXPERIMENTS:

        name = spec["name"]

        result_path = (
            OUTPUT / f"{name}_summary.json"
        )

        model_path = (
            MODEL_DIR / f"{name}.joblib"
        )

        predictions_path = (
            OUTPUT
            / f"{name}_validation_predictions.csv"
        )

        oof_path = (
            OUTPUT
            / f"{name}_training_oof.csv"
        )

        # Reuse fully completed matching experiments
        if all(
            path.exists()
            for path in (
                result_path,
                model_path,
                predictions_path,
                oof_path
            )
        ):

            result = json.loads(
                result_path.read_text(
                    encoding="utf-8"
                )
            )

            if result.get(
                "signature"
            ) != signature:

                raise RuntimeError(
                    f"Existing {name} files have "
                    "a different data/config signature."
                )

            results.append(result)

            print(
                f"{name:<39} "
                "reused completed result",
                flush=True
            )

            continue

        # Prepare fold-wise latent representations
        fold_data = []

        for fold_index, (
            fit_ids,
            hold_ids
        ) in enumerate(folds):

            zfit, zhold, _ = build_embeddings(
                spec,
                fold_index,
                fit_ids,
                hold_ids
            )

            fold_data.append(
                (
                    fit_ids,
                    hold_ids,
                    zfit,
                    zhold
                )
            )

        selected_candidate = None
        selected_quality = None
        selected_oof = None
        all_candidates = []

        grid = (
            JOINT_C_VALUES
            if spec["kind"] == "joint"
            else C_VALUES
        )

        for C in grid:

            for weight in CLASS_WEIGHTS:

                oof = np.full(
                    len(ytr),
                    np.nan,
                    dtype=float
                )

                ap_folds = []
                auc_folds = []

                for (
                    fit_ids,
                    hold_ids,
                    zfit,
                    zhold
                ) in fold_data:

                    lr = LogisticRegression(
                        C=C,
                        class_weight=weight,
                        solver="liblinear",
                        max_iter=2500,
                        random_state=SEED
                    )

                    lr.fit(
                        zfit,
                        ytr[fit_ids]
                    )

                    probabilities = (
                        lr.predict_proba(
                            zhold
                        )[:, 1]
                    )

                    oof[hold_ids] = probabilities

                    ap_folds.append(
                        average_precision_score(
                            ytr[hold_ids],
                            probabilities
                        )
                    )

                    auc_folds.append(
                        roc_auc_score(
                            ytr[hold_ids],
                            probabilities
                        )
                    )

                if not np.isfinite(
                    oof
                ).all():

                    raise RuntimeError(
                        "Incomplete training-only "
                        "OOF predictions."
                    )

                quality_value = (
                    float(
                        np.mean(ap_folds)
                    ),
                    float(
                        np.mean(auc_folds)
                    )
                )

                all_candidates.append({
                    "C": C,
                    "class_weight": weight,
                    "mean_CV_AP": quality_value[0],
                    "sd_CV_AP": float(
                        np.std(
                            ap_folds,
                            ddof=1
                        )
                    ),
                    "mean_CV_AUC": quality_value[1],
                    "pooled_OOF_AP": float(
                        average_precision_score(
                            ytr,
                            oof
                        )
                    )
                })

                if (
                    selected_quality is None
                    or quality_value > selected_quality
                ):

                    selected_candidate = (
                        C,
                        weight
                    )

                    selected_quality = (
                        quality_value
                    )

                    selected_oof = (
                        oof.copy()
                    )

        # Fit probability calibration on training OOF
        calibrator = sigmoid_calibrator(
            selected_oof,
            ytr
        )

        oof_calibrated = apply_calibrator(
            calibrator,
            selected_oof
        )

        threshold = best_oof_threshold(
            ytr,
            oof_calibrated
        )

        # Refit selected transformation on all training patients
        full_train_indices = np.arange(
            len(ytr)
        )

        ztrain, zval, fitted_transform = (
            build_embeddings(
                spec,
                42,
                full_train_indices,
                np.arange(len(yva)),
                final=True
            )
        )

        estimator = LogisticRegression(
            C=selected_candidate[0],
            class_weight=selected_candidate[1],
            solver="liblinear",
            max_iter=2500,
            random_state=SEED
        )

        estimator.fit(
            ztrain,
            ytr
        )

        p_raw = estimator.predict_proba(
            zval
        )[:, 1]

        p_val = apply_calibrator(
            calibrator,
            p_raw
        )

        validation_metrics = binary_metrics(
            yva,
            p_val,
            threshold
        )

        # Histology-specific validation performance
        by_histology = {}

        for label, code in (
            ("LUAD", 0),
            ("LUSC", 1)
        ):

            select = (
                hva == code
            )

            by_histology[label] = {
                "patients": int(
                    select.sum()
                ),
                "advanced": int(
                    yva[select].sum()
                ),
                **binary_metrics(
                    yva[select],
                    p_val[select],
                    threshold
                )
            }

        # Stratified bootstrap uncertainty
        confidence_intervals = {}

        for metric, metric_function in (
            ("AP", average_precision_score),
            ("ROC_AUC", roc_auc_score)
        ):

            values = [
                metric_function(
                    yva[index],
                    p_val[index]
                )
                for index in bootstrap
            ]

            confidence_intervals[
                f"{metric}_stratified_95pct"
            ] = [
                float(value)
                for value in np.percentile(
                    values,
                    [2.5, 97.5]
                )
            ]

        if reference_scores is not None:

            differences = [
                average_precision_score(
                    yva[index],
                    p_val[index]
                )
                - average_precision_score(
                    yva[index],
                    reference_scores[index]
                )
                for index in bootstrap
            ]

            confidence_intervals[
                "AP_difference_vs_stage11_SVD"
            ] = float(
                validation_metrics["AP"]
                - average_precision_score(
                    yva,
                    reference_scores
                )
            )

            confidence_intervals[
                "AP_difference_exploratory_95pct"
            ] = [
                float(value)
                for value in np.percentile(
                    differences,
                    [2.5, 97.5]
                )
            ]

        # Save training-only OOF predictions
        pd.DataFrame({
            "case_id": patients.case_id.iloc[
                train_idx
            ].to_numpy(),
            "histology_label": htr,
            "stage_label": ytr,
            "oof_probability_raw": selected_oof,
            "oof_probability_calibrated": oof_calibrated
        }).to_csv(
            oof_path,
            index=False
        )

        # Save independent validation predictions
        pd.DataFrame({
            "case_id": patients.case_id.iloc[
                val_idx
            ].to_numpy(),
            "histology_label": hva,
            "stage_label": yva,
            "prob_raw": p_raw,
            "prob_calibrated": p_val,
            "predicted_advanced": (
                p_val >= threshold
            ).astype(int)
        }).to_csv(
            predictions_path,
            index=False
        )

        # Save the fitted preprocessing and classifier
        temporary_model = model_path.with_name(
            model_path.name + ".tmp"
        )

        joblib.dump(
            {
                "signature": signature,
                "spec": spec,
                "preprocessing": fitted_transform,
                "estimator": estimator,
                "calibrator": calibrator,
                "threshold": threshold,
                "gene_nodes": len(nodes)
            },
            temporary_model,
            compress=3
        )

        os.replace(
            temporary_model,
            model_path
        )

        result = {
            "name": name,
            "signature": signature,
            "selected_by": (
                "training_4fold_stage_histology_stratified_CV_AP"
            ),
            "selected_C": selected_candidate[0],
            "selected_class_weight": selected_candidate[1],
            "CV_mean_AP": selected_quality[0],
            "CV_mean_AUC": selected_quality[1],
            "CV_candidates": all_candidates,
            "oof_threshold": threshold,
            "oof_AP": float(
                average_precision_score(
                    ytr,
                    selected_oof
                )
            ),
            "oof_calibrated_Brier": float(
                brier_score_loss(
                    ytr,
                    oof_calibrated
                )
            ),
            "validation": validation_metrics,
            "by_histology": by_histology,
            "uncertainty": confidence_intervals,
            "model_path": str(model_path),
            "validation_predictions": str(
                predictions_path
            ),
            "training_oof": str(oof_path),
            "test_set_evaluated": False
        }

        temporary_result = result_path.with_suffix(
            ".json.tmp"
        )

        temporary_result.write_text(
            json.dumps(
                result,
                indent=2
            ),
            encoding="utf-8"
        )

        os.replace(
            temporary_result,
            result_path
        )

        results.append(result)

        print(
            f"{name:<39} "
            f"CV AP={selected_quality[0]:.4f} | "
            f"Val AP={validation_metrics['AP']:.4f} | "
            f"Val AUC={validation_metrics['ROC_AUC']:.4f} | "
            f"MCC={validation_metrics['MCC']:.4f} | "
            f"Brier={validation_metrics['Brier']:.4f}",
            flush=True
        )


# ============================================================
# 15. Compare all model configurations
# ============================================================

summary_table = pd.DataFrame([
    {
        "name": result["name"],
        "CV_AP": result["CV_mean_AP"],
        "CV_AUC": result["CV_mean_AUC"],
        "validation_AP": result[
            "validation"
        ]["AP"],
        "validation_ROC_AUC": result[
            "validation"
        ]["ROC_AUC"],
        "validation_MCC": result[
            "validation"
        ]["MCC"],
        "validation_Brier": result[
            "validation"
        ]["Brier"],
        "validation_advanced_recall": result[
            "validation"
        ]["advanced_recall"],
        "OOF_threshold": result[
            "oof_threshold"
        ]
    }
    for result in results
])

summary_table = summary_table.sort_values(
    ["CV_AP", "CV_AUC"],
    ascending=False
)

summary_table.to_csv(
    OUTPUT / "stage13_cv_validation_comparison.csv",
    index=False
)


# ============================================================
# 16. Compare biological graphs against the randomized graph
# ============================================================

paired_graph_comparison = {}

null_prediction_file = (
    OUTPUT
    / "Hybrid_DegreeMatchedNull_validation_predictions.csv"
)

if null_prediction_file.exists():

    null_df = pd.read_csv(
        null_prediction_file,
        dtype={"case_id": str}
    )

    expected_ids = patients.case_id.iloc[
        val_idx
    ].to_numpy()

    if not np.array_equal(
        null_df.case_id.to_numpy(),
        expected_ids
    ):
        raise RuntimeError(
            "Random-graph validation patient order mismatch."
        )

    p_null = null_df.prob_calibrated.to_numpy(
        dtype=float
    )

    for network in (
        "Hybrid_FunctionalPPI",
        "Hybrid_PhysicalPPI"
    ):

        network_file = (
            OUTPUT
            / f"{network}_validation_predictions.csv"
        )

        if not network_file.exists():
            continue

        network_df = pd.read_csv(
            network_file,
            dtype={"case_id": str}
        )

        if not np.array_equal(
            network_df.case_id.to_numpy(),
            expected_ids
        ):
            raise RuntimeError(
                f"{network} validation patient order mismatch."
            )

        p_network = (
            network_df.prob_calibrated.to_numpy(
                dtype=float
            )
        )

        differences = [
            average_precision_score(
                yva[index],
                p_network[index]
            )
            - average_precision_score(
                yva[index],
                p_null[index]
            )
            for index in bootstrap
        ]

        paired_graph_comparison[network] = {
            "validation_AP_difference_vs_degree_matched_null": float(
                average_precision_score(
                    yva,
                    p_network
                )
                - average_precision_score(
                    yva,
                    p_null
                )
            ),
            "paired_stratified_bootstrap_95pct": [
                float(value)
                for value in np.percentile(
                    differences,
                    [2.5, 97.5]
                )
            ],
            "interpretation": (
                "exploratory; validation already used "
                "in previous stages"
            )
        }

(
    OUTPUT / "stage13_graph_vs_null_comparison.json"
).write_text(
    json.dumps(
        paired_graph_comparison,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 17. Save reproducibility summary
# ============================================================

preferred = summary_table.loc[
    ~summary_table.name.eq(
        "Hybrid_DegreeMatchedNull"
    ),
    "name"
].iloc[0]

report = {
    "generated_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "signature": signature,
    "training_patients": len(train_idx),
    "validation_patients": len(val_idx),
    "sealed_test_patients": len(test_idx),
    "training_CV_folds": FOLDS,
    "graph_blend": GRAPH_BLEND,
    "graph_null": (
        "degree_preserving_double_edge_swap"
    ),
    "paired_graph_vs_null": paired_graph_comparison,
    "selected_model_by_training_CV_only": preferred,
    "selected_model_validation_AP": float(
        summary_table.loc[
            summary_table.name.eq(preferred),
            "validation_AP"
        ].iloc[0]
    ),
    "test_features_loaded": False,
    "caveats": [
        (
            "Stage09 gene nodes and initial standardization "
            "were fit on all training samples; inner CV is "
            "not fully nested."
        ),
        (
            "Out-of-fold calibration and threshold fitting "
            "use the same OOF targets; threshold performance "
            "may be optimistic."
        ),
        (
            "Repeated analyses on the same validation set "
            "create adaptive validation-selection risk."
        ),
        (
            "STRING confidence scores represent association "
            "confidence, not biological effect sizes."
        )
    ]
}

(
    OUTPUT / "stage13_summary.json"
).write_text(
    json.dumps(
        report,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 18. Final validation results
# ============================================================

print("\n" + "=" * 75)
print(
    "STAGE 13 FINAL CROSS-VALIDATION "
    "AND VALIDATION RESULTS"
)
print("=" * 75)

print(
    summary_table.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

print(
    "\nBest by training CV "
    "(randomized control excluded):",
    preferred
)

if reference_scores is not None:

    print(
        "Stage 11 SVD reference validation AP:",
        round(
            average_precision_score(
                yva,
                reference_scores
            ),
            4
        )
    )

print(
    "\nPaired biological graph vs randomized graph:"
)

for model_name, comparison_result in (
    paired_graph_comparison.items()
):

    print(
        model_name,
        comparison_result
    )

print(
    "\nSealed test patients:",
    len(test_idx)
)

print(
    "Saved metrics:",
    OUTPUT
)

print(
    "Saved models:",
    MODEL_DIR
)

print(
    "\nSTAGE 13 COMPLETED - TEST SET UNTOUCHED"
)

STAGE 13 - MULTIVIEW LATENT SIGNAL AND PPI-PRIOR EVALUATION
Train=667, validation=143, sealed test=143
Nodes=1600; train advanced prevalence=0.199

Functional edges: 2827
Functional isolated genes: 482
Degree-preserving random control: 2827 edges | degree check: PASSED

Training-only missingness audit (exploratory):
histology    modality  early_mean_observed  advanced_mean_observed  raw_p  BH_FDR_q
     LUAD         RNA               1.0000                  1.0000 1.0000    1.0000
     LUAD         CNV               0.9951                  0.9940 0.8074    1.0000
     LUAD Methylation               0.8809                  0.8821 0.9078    1.0000
     LUSC         RNA               1.0000                  1.0000 1.0000    1.0000
     LUSC         CNV               0.9967                  0.9964 0.9600    1.0000
     LUSC Methylation               0.7746                  0.6769 0.1065    0.6389

Training-only CV folds: 4

Stage 11 SVD reference validation AP: 0.3721
Joint_SVD64_Reference

D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full

MultiView_SVD16                         CV AP=0.3308 | Val AP=0.2939 | Val AUC=0.5684 | MCC=0.0473 | Brier=0.1600


D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full

MultiView_SVD32                         CV AP=0.3324 | Val AP=0.3076 | Val AUC=0.6158 | MCC=0.0880 | Brier=0.1569
MultiView_SVD32_Histology               CV AP=0.3335 | Val AP=0.3079 | Val AUC=0.6183 | MCC=0.0880 | Brier=0.1568
MultiView_SVD32_HistologyInteractions   CV AP=0.3271 | Val AP=0.3358 | Val AUC=0.6044 | MCC=0.2479 | Brier=0.1566
MultiView_SVD32_NoMethylation           CV AP=0.3324 | Val AP=0.3076 | Val AUC=0.6158 | MCC=0.0880 | Brier=0.1569


D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full

Hybrid_FunctionalPPI                    CV AP=0.3531 | Val AP=0.3075 | Val AUC=0.6025 | MCC=0.0475 | Brier=0.1581


D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full

Hybrid_PhysicalPPI                      CV AP=0.3497 | Val AP=0.2849 | Val AUC=0.5962 | MCC=0.0708 | Brier=0.1593


D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full_var
D:\NST_NSCLC_PPI_GAT\envs\nsclc\Lib\site-packages\sklearn\decomposition\_truncated_svd.py:273: RuntimeWarning: invalid value encountered in divide
  self.explained_variance_ratio_ = exp_var / full

Hybrid_DegreeMatchedNull                CV AP=0.3489 | Val AP=0.2813 | Val AUC=0.5938 | MCC=0.0934 | Brier=0.1590

STAGE 13 FINAL CROSS-VALIDATION AND VALIDATION RESULTS
                                 name  CV_AP  CV_AUC  validation_AP  validation_ROC_AUC  validation_MCC  validation_Brier  validation_advanced_recall  OOF_threshold
                 Hybrid_FunctionalPPI 0.3531  0.6344         0.3075              0.6025          0.0475            0.1581                      0.0345         0.3900
                   Hybrid_PhysicalPPI 0.3497  0.6305         0.2849              0.5962          0.0708            0.1593                      0.1724         0.3000
             Hybrid_DegreeMatchedNull 0.3489  0.6287         0.2813              0.5938          0.0934            0.1590                      0.0690         0.3600
            MultiView_SVD32_Histology 0.3335  0.6189         0.3079              0.6183          0.0880            0.1568                      0.0345         0.3800
     

In [13]:

import hashlib
import json
import os
import shutil
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    matthews_corrcoef,
    roc_auc_score,
    balanced_accuracy_score
)
from sklearn.model_selection import StratifiedKFold
from threadpoolctl import threadpool_limits

# Configuration
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

OLD = ROOT / "data" / "processed" / "ppi_gat_v1"
NEW = ROOT / "data" / "processed" / "ppi_gat_v2"
GENE = ROOT / "data" / "processed" / "gene_level"
SPLITS = ROOT / "data" / "splits"
RESULTS = ROOT / "results" / "metrics" / "stage14"
MODELS = ROOT / "models" / "checkpoints" / "stage14"

for directory in (NEW, RESULTS, MODELS):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
FOLDS = 4
THREADS = min(4, os.cpu_count() or 1)
BOOTSTRAPS = 500
C_GRID = (0.003, 0.03, 0.3)
WEIGHTS = (None, "balanced")
DIFFUSION = 0.30
VERSION = "stage14_fix_missing_methylation_values_v1"

# Load reference tables and original arrays
old_file = OLD / "patient_gene_features_8ch.npy"
beta_file = GENE / "methylation_promoter_beta.npy"
ref_file = GENE / "gene_reference.csv"
scaler_file = OLD / "train_only_scaler_parameters.csv"
node_file = OLD / "selected_gene_nodes.csv"
label_file = OLD / "patient_labels_and_splits.csv"
order_file = GENE / "case_order.csv"
split_file = SPLITS / "locked_four_omics_v1.csv"

fg_file = OLD / "string_functional_700_corrected.npz"
pg_file = OLD / "string_physical_400_corrected.npz"

null_file = (
    ROOT
    / "results"
    / "metrics"
    / "stage13"
    / "degree_preserving_randomized_graph.npz"
)

for path in (
    old_file,
    beta_file,
    ref_file,
    scaler_file,
    node_file,
    label_file,
    order_file,
    split_file,
    fg_file,
    pg_file,
    null_file
):
    if not path.is_file():
        raise FileNotFoundError(
            f"Required input not found: {path}"
        )

nodes = pd.read_csv(
    node_file,
    dtype={"ensembl_id": str}
)

reference = pd.read_csv(
    ref_file,
    dtype={"ensembl_id": str}
)

patients = pd.read_csv(
    label_file,
    dtype={"case_id": str}
)

case_order = pd.read_csv(
    order_file,
    dtype={"case_id": str}
)

locked = pd.read_csv(
    split_file,
    dtype={"case_id": str}
)

scalers = pd.read_csv(scaler_file)

old = np.load(old_file, mmap_mode="r")
beta_source = np.load(beta_file, mmap_mode="r")

if (
    old.shape != (953, 1600, 8)
    or beta_source.shape != (953, 19944)
):
    raise RuntimeError(
        f"Unexpected raw array shapes: "
        f"{old.shape}, {beta_source.shape}"
    )

if (
    len(nodes) != 1600
    or len(patients) != 953
    or not nodes.ensembl_id.is_unique
    or not reference.ensembl_id.is_unique
    or not patients.case_id.is_unique
    or not locked.case_id.is_unique
):
    raise RuntimeError(
        "Duplicate identifiers or inconsistent source tables."
    )

if not np.array_equal(
    nodes.node_index.to_numpy(dtype=int),
    np.arange(1600)
):
    raise RuntimeError(
        "Gene node indices are not ordered from zero."
    )

if not np.array_equal(
    patients.case_id.astype(str),
    case_order.case_id.astype(str)
):
    raise RuntimeError(
        "Stage 08 and Stage 09 patient orders differ."
    )

check = patients[
    ["case_id", "histology", "stage_group", "split"]
].merge(
    locked[
        ["case_id", "histology", "stage_group", "split"]
    ],
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_now", "_locked")
)

if (
    len(check) != 953
    or any(
        not check[f"{x}_now"].equals(
            check[f"{x}_locked"]
        )
        for x in (
            "histology",
            "stage_group",
            "split"
        )
    )
):
    raise RuntimeError(
        "Locked patient assignments have changed."
    )

train_ids = np.flatnonzero(
    patients.split.eq("train").to_numpy()
)

val_ids = np.flatnonzero(
    patients.split.eq("validation").to_numpy()
)

test_ids = np.flatnonzero(
    patients.split.eq("test").to_numpy()
)

if (
    len(train_ids),
    len(val_ids),
    len(test_ids)
) != (667, 143, 143):
    raise RuntimeError(
        "Expected locked 667/143/143 partition was not found."
    )

# Map selected genes to Stage 08
ref_index = pd.Index(
    reference.ensembl_id.astype(str)
)

selected_columns = ref_index.get_indexer(
    nodes.ensembl_id.astype(str)
)

if (selected_columns < 0).any():
    raise RuntimeError(
        "Some selected Ensembl genes are not in Stage 08."
    )

meth_params = scalers.loc[
    scalers.modality.eq("methylation")
].copy()

if (
    len(meth_params) != len(nodes)
    or meth_params.node_index.duplicated().any()
):
    raise RuntimeError(
        "Stage 09 methylation scaler table is incomplete."
    )

meth_params = (
    meth_params
    .sort_values("node_index")
    .reset_index(drop=True)
)

if (
    not np.array_equal(
        meth_params.node_index.to_numpy(dtype=int),
        np.arange(1600)
    )
    or not np.array_equal(
        meth_params.gene_id.astype(str),
        nodes.ensembl_id.astype(str)
    )
):
    raise RuntimeError(
        "Methylation scaler gene ordering is inconsistent."
    )

centers = meth_params.train_center.to_numpy(
    dtype=np.float32
)

scales = meth_params.train_scale.to_numpy(
    dtype=np.float32
)

if (
    not np.isfinite(centers).all()
    or not np.isfinite(scales).all()
    or (scales < 0.0399).any()
):
    raise RuntimeError(
        "Invalid training-only methylation scaling parameters."
    )

print("=" * 73)
print(
    "STAGE 14 - REPAIR METHYLATION CHANNEL "
    "AND REASSESS FOUR-OMICS"
)
print("=" * 73)

print(
    f"Locked cohorts: train={len(train_ids)}, "
    f"validation={len(val_ids)}, "
    f"sealed test={len(test_ids)}"
)

print(
    "Original methylation channel 3 nonzero values:",
    int(np.count_nonzero(old[:, :, 3]))
)

# Reconstruct methylation using frozen training scalers
raw_beta = np.asarray(
    beta_source[:, selected_columns],
    dtype=np.float32
)

observed = (
    np.isfinite(raw_beta)
    & (raw_beta >= 0)
    & (raw_beta <= 1)
)

reconstructed = np.clip(
    (
        np.where(
            observed,
            raw_beta,
            centers[None, :]
        )
        - centers[None, :]
    ) / scales[None, :],
    -5.0,
    5.0
).astype(np.float32)

reconstructed[~observed] = 0.0

if (
    np.count_nonzero(
        reconstructed[train_ids]
    ) < 1000
    or not np.isfinite(reconstructed).all()
):
    raise RuntimeError(
        "Recovered methylation channel has insufficient signal."
    )

original_mask = np.asarray(
    old[:, :, 7]
)

mask_disagreements = int(
    np.count_nonzero(
        original_mask
        != observed.astype(np.float32)
    )
)

print(
    "Methylation mask discrepancies:",
    mask_disagreements
)

if mask_disagreements:
    raise RuntimeError(
        "Saved methylation mask disagrees with Stage 08 data."
    )

# Verify train-only scaler provenance
rng = np.random.default_rng(SEED)

check_indices = rng.choice(
    len(nodes),
    size=30,
    replace=False
)

for index in check_indices:

    values = raw_beta[
        train_ids,
        index
    ]

    values = values[
        np.isfinite(values)
        & (values >= 0)
        & (values <= 1)
    ]

    expected_center = (
        float(np.median(values))
        if len(values)
        else 0.0
    )

    if (
        abs(
            expected_center
            - float(centers[index])
        ) > 1e-4
    ):
        raise RuntimeError(
            "Saved methylation scaler is inconsistent "
            "with training data."
        )

# Write the repaired feature tensor without modifying v1
v2_file = NEW / "patient_gene_features_8ch.npy"

v2_temp = (
    NEW / "patient_gene_features_8ch.partial.npy"
)

if v2_temp.exists():
    v2_temp.unlink()

fixed = np.lib.format.open_memmap(
    v2_temp,
    mode="w+",
    dtype=np.float32,
    shape=old.shape
)

fixed[:] = old[:]

# Correct channel 3: promoter methylation
fixed[:, :, 3] = reconstructed

fixed.flush()
del fixed

candidate = np.load(
    v2_temp,
    mmap_mode="r"
)

if (
    not np.array_equal(
        candidate[:, :, 2],
        old[:, :, 2]
    )
    or not np.array_equal(
        candidate[:, :, 7],
        old[:, :, 7]
    )
    or not np.isfinite(candidate).all()
    or float(
        np.std(
            candidate[train_ids, :, 3]
        )
    ) < 0.03
):
    raise RuntimeError(
        "Repaired array failed channel-integrity checks."
    )

del candidate

os.replace(
    v2_temp,
    v2_file
)

for filename in (
    "selected_gene_nodes.csv",
    "patient_labels_and_splits.csv",
    "string_functional_700_corrected.npz",
    "string_physical_400_corrected.npz",
    "train_only_scaler_parameters.csv"
):
    shutil.copy2(
        OLD / filename,
        NEW / filename
    )

x_v2 = np.load(
    v2_file,
    mmap_mode="r"
)

print(
    "Restored methylation train SD:",
    round(
        float(
            np.std(
                x_v2[train_ids, :, 3]
            )
        ),
        4
    )
)

print(
    "Restored methylation observed fraction:",
    round(
        float(
            np.mean(
                x_v2[train_ids, :, 7]
            )
        ),
        4
    )
)

print(
    "Original mutation indicators preserved: PASSED"
)

# Load training and validation data for experiments
x_train = np.asarray(
    x_v2[train_ids],
    dtype=np.float32
).copy()

x_val = np.asarray(
    x_v2[val_ids],
    dtype=np.float32
).copy()

x_legacy_train = np.asarray(
    old[train_ids],
    dtype=np.float32
).copy()

x_legacy_val = np.asarray(
    old[val_ids],
    dtype=np.float32
).copy()

y_train = patients.stage_label.iloc[
    train_ids
].to_numpy(dtype=np.int32)

y_val = patients.stage_label.iloc[
    val_ids
].to_numpy(dtype=np.int32)

h_train = patients.histology_label.iloc[
    train_ids
].to_numpy(dtype=np.int32)

h_val = patients.histology_label.iloc[
    val_ids
].to_numpy(dtype=np.int32)

# Audit molecular feature variability
quality_rows = []

for j, modality in enumerate(
    (
        "RNA",
        "CNV",
        "Mutation",
        "Methylation"
    )
):

    for histology, code in (
        ("LUAD", 0),
        ("LUSC", 1)
    ):

        for stage, label in (
            ("Early", 0),
            ("Advanced", 1)
        ):

            selected = (
                (h_train == code)
                & (y_train == label)
            )

            modality_values = (
                x_train[selected, :, j]
            )

            quality_rows.append({
                "modality": modality,
                "histology": histology,
                "stage": stage,
                "patients": int(selected.sum()),
                "gene_feature_std": float(
                    np.std(modality_values)
                ),
                "fraction_nonzero": float(
                    np.mean(
                        np.abs(modality_values) > 1e-7
                    )
                ),
                "observed_fraction": float(
                    np.mean(
                        x_train[
                            selected,
                            :,
                            4 + j
                        ]
                    )
                )
            })

quality = pd.DataFrame(
    quality_rows
)

quality.to_csv(
    RESULTS / "stage14_modality_signal_audit.csv",
    index=False
)

print(
    "\nTraining-only molecular-channel audit:"
)

print(
    quality.round(4).to_string(
        index=False
    )
)

# Load functional and physical STRING networks
def load_graph(path):

    with np.load(path) as pack:

        edge_index = np.asarray(
            pack["edge_index"],
            dtype=np.int64
        )

        edge_weight = np.asarray(
            pack["edge_weight"],
            dtype=np.float32
        )

    if (
        edge_index.shape[0] != 2
        or edge_index.min() < 0
        or edge_index.max() >= len(nodes)
        or len(edge_weight)
        != edge_index.shape[1]
    ):
        raise RuntimeError(
            f"Invalid graph data: {path}"
        )

    graph = sparse.csr_matrix(
        (
            edge_weight,
            (
                edge_index[0],
                edge_index[1]
            )
        ),
        shape=(
            len(nodes),
            len(nodes)
        ),
        dtype=np.float32
    )

    graph.setdiag(0)
    graph.eliminate_zeros()

    difference = graph - graph.T

    if (
        difference.nnz
        and np.max(
            np.abs(difference.data)
        ) > 1e-5
    ):
        raise RuntimeError(
            f"Asymmetric graph: {path}"
        )

    return graph


functional = load_graph(fg_file)
physical = load_graph(pg_file)

random_graph = sparse.load_npz(
    null_file
).astype(
    np.float32
).tocsr()

if random_graph.shape != functional.shape:
    raise RuntimeError(
        "Randomized graph dimensions differ."
    )

functional_degrees = np.asarray(
    (functional > 0).sum(axis=1)
).ravel()

random_degrees = np.asarray(
    (random_graph > 0).sum(axis=1)
).ravel()

if not np.array_equal(
    functional_degrees,
    random_degrees
):
    raise RuntimeError(
        "Random graph does not preserve STRING gene degrees."
    )

print(
    "\nVerified functional edges:",
    functional.nnz // 2
)

print(
    "Verified physical edges:",
    physical.nnz // 2
)

print(
    "Verified degree-matched random edges:",
    random_graph.nnz // 2
)

# Construct comparable representations
CHANNELS = (
    0, 1, 2, 3, 4, 5, 7
)


def flatten_view(x, channels):

    return np.concatenate(
        [
            x[:, :, j]
            for j in channels
        ],
        axis=1
    ).astype(np.float32)


def diffuse_view(x, adjacency):

    parts = []

    for j in range(4):

        signal = np.asarray(
            x[:, :, j],
            dtype=np.float32
        )

        mask = np.asarray(
            x[:, :, 4 + j],
            dtype=np.float32
        )

        numerator = np.asarray(
            (
                adjacency
                @ (signal * mask).T
            ).T,
            dtype=np.float32
        )

        denominator = np.asarray(
            (
                adjacency @ mask.T
            ).T,
            dtype=np.float32
        )

        neighbor = np.divide(
            numerator,
            denominator,
            out=np.zeros_like(numerator),
            where=denominator > 1e-8
        )

        neighbor = np.where(
            denominator > 1e-8,
            neighbor,
            signal
        )

        signal = (
            (1.0 - DIFFUSION) * signal
            + DIFFUSION * neighbor
        ) * mask

        parts.append(
            np.clip(
                signal,
                -5,
                5
            ).astype(np.float32)
        )

    parts.extend(
        [
            x[:, :, j]
            for j in (4, 5, 7)
        ]
    )

    result = np.concatenate(
        parts,
        axis=1
    ).astype(np.float32)

    if not np.isfinite(result).all():
        raise RuntimeError(
            "Non-finite diffused molecular features."
        )

    return result


views = {
    "legacy": (
        flatten_view(
            x_legacy_train,
            CHANNELS
        ),
        flatten_view(
            x_legacy_val,
            CHANNELS
        )
    ),

    "four": (
        flatten_view(
            x_train,
            CHANNELS
        ),
        flatten_view(
            x_val,
            CHANNELS
        )
    ),

    "three": (
        flatten_view(
            x_train,
            (0, 1, 2, 4, 5)
        ),
        flatten_view(
            x_val,
            (0, 1, 2, 4, 5)
        )
    ),

    "methyl": (
        flatten_view(
            x_train,
            (3, 7)
        ),
        flatten_view(
            x_val,
            (3, 7)
        )
    )
}

for name, graph in (
    ("functional", functional),
    ("physical", physical),
    ("randomized", random_graph)
):

    views[name] = (
        diffuse_view(
            x_train,
            graph
        ),
        diffuse_view(
            x_val,
            graph
        )
    )

print(
    "Graph-diffused molecular representations prepared."
)

# Predeclare models and negative controls
specifications = [
    (
        "Legacy_Bugged_SVD64",
        "legacy",
        64,
        False,
        True
    ),
    (
        "Repaired_FourOmics_SVD64",
        "four",
        64,
        False,
        False
    ),
    (
        "Repaired_FourOmics_SVD128",
        "four",
        128,
        False,
        False
    ),
    (
        "Repaired_ThreeOmics_SVD64",
        "three",
        64,
        False,
        False
    ),
    (
        "Repaired_MethylationOnly_SVD32",
        "methyl",
        32,
        False,
        True
    ),
    (
        "Repaired_FourOmics_Histology_SVD64",
        "four",
        64,
        True,
        False
    ),
    (
        "Repaired_FunctionalPPI_SVD64",
        "functional",
        64,
        False,
        False
    ),
    (
        "Repaired_PhysicalPPI_SVD64",
        "physical",
        64,
        False,
        False
    ),
    (
        "Repaired_RandomizedPPI_SVD64",
        "randomized",
        64,
        False,
        True
    )
]

# Create training-only CV folds
strata = (
    2 * y_train + h_train
)

folds = list(
    StratifiedKFold(
        n_splits=FOLDS,
        shuffle=True,
        random_state=SEED
    ).split(
        np.arange(len(y_train)),
        strata
    )
)

if any(
    len(
        np.unique(
            strata[hold]
        )
    ) != 4
    for _, hold in folds
):
    raise RuntimeError(
        "A CV fold is missing a stage-histology stratum."
    )

# Fingerprint experiment inputs
def file_hash(path):

    digest = hashlib.sha256()

    with open(path, "rb") as stream:

        for chunk in iter(
            lambda: stream.read(
                4 * 1024 * 1024
            ),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


signature = hashlib.sha256(
    json.dumps(
        {
            "version": VERSION,
            "old_features": file_hash(old_file),
            "beta_source": file_hash(beta_file),
            "scalers": file_hash(scaler_file),
            "nodes": file_hash(node_file),
            "patients": file_hash(label_file),
            "functional": file_hash(fg_file),
            "physical": file_hash(pg_file),
            "randomized": file_hash(null_file),
            "cv_folds": FOLDS,
            "C": C_GRID,
            "weights": WEIGHTS,
            "diffusion": DIFFUSION,
            "experiments": specifications
        },
        sort_keys=True
    ).encode()
).hexdigest()

fold_cache = {}
final_cache = {}

# Fit SVD within each training fold
def projected(
    view,
    dimension,
    fold_number,
    histology=False,
    final=False
):

    key = (
        view,
        dimension,
        fold_number,
        final
    )

    cache = (
        final_cache
        if final
        else fold_cache
    )

    if key not in cache:

        fit_x, val_x = views[view]

        if final:

            xi = fit_x
            xv = val_x

        else:

            fit_ids, hold_ids = folds[
                fold_number
            ]

            xi = fit_x[fit_ids]
            xv = fit_x[hold_ids]

        svd = TruncatedSVD(
            n_components=dimension,
            n_iter=4,
            random_state=SEED + (
                0
                if final
                else fold_number
            )
        )

        with warnings.catch_warnings():

            warnings.filterwarnings(
                "ignore",
                message="invalid value encountered in divide",
                category=RuntimeWarning
            )

            z_train = svd.fit_transform(
                xi
            ).astype(np.float32)

        z_held = svd.transform(
            xv
        ).astype(np.float32)

        if (
            not np.isfinite(z_train).all()
            or not np.isfinite(z_held).all()
        ):
            raise RuntimeError(
                f"Invalid SVD factors: "
                f"{view}, fold {fold_number}"
            )

        cache[key] = (
            z_train,
            z_held,
            svd
        )

    z_train, z_held, svd = cache[key]

    if not histology:

        return (
            z_train,
            z_held,
            svd
        )

    if final:

        ht = h_train
        hv = h_val

    else:

        fit_ids, hold_ids = folds[
            fold_number
        ]

        ht = h_train[fit_ids]
        hv = h_train[hold_ids]

    z_train = np.column_stack(
        [
            z_train,
            2 * ht - 1
        ]
    ).astype(np.float32)

    z_held = np.column_stack(
        [
            z_held,
            2 * hv - 1
        ]
    ).astype(np.float32)

    return (
        z_train,
        z_held,
        svd
    )


# Classification metrics
def metrics(y, p, threshold):

    predicted = (
        p >= threshold
    ).astype(np.int32)

    tp = int(
        (
            (y == 1)
            & (predicted == 1)
        ).sum()
    )

    tn = int(
        (
            (y == 0)
            & (predicted == 0)
        ).sum()
    )

    fp = int(
        (
            (y == 0)
            & (predicted == 1)
        ).sum()
    )

    fn = int(
        (
            (y == 1)
            & (predicted == 0)
        ).sum()
    )

    return {
        "AP": float(
            average_precision_score(
                y,
                p
            )
        ),
        "ROC_AUC": float(
            roc_auc_score(
                y,
                p
            )
        ),
        "MCC": float(
            matthews_corrcoef(
                y,
                predicted
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y,
                predicted
            )
        ),
        "Brier": float(
            brier_score_loss(
                y,
                p
            )
        ),
        "advanced_recall": (
            float(tp / (tp + fn))
            if tp + fn
            else 0.0
        ),
        "advanced_precision": (
            float(tp / (tp + fp))
            if tp + fp
            else 0.0
        ),
        "early_specificity": (
            float(tn / (tn + fp))
            if tn + fp
            else 0.0
        ),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "threshold": float(threshold)
    }


def choose_threshold(y, scores):

    thresholds = np.unique(
        np.r_[
            np.arange(
                0.02,
                0.981,
                0.01
            ),
            0.5
        ]
    )

    best = max(
        (
            (
                matthews_corrcoef(
                    y,
                    scores >= threshold
                ),
                float(threshold)
            )
            for threshold in thresholds
        ),
        key=lambda pair: (
            pair[0],
            -abs(pair[1] - 0.5)
        )
    )

    return float(best[1])


# Generate stratified bootstrap samples
rng = np.random.default_rng(
    SEED + 414
)

validation_strata = (
    2 * y_val + h_val
)

boot_indices = []

for _ in range(BOOTSTRAPS):

    boot_indices.append(
        np.concatenate(
            [
                rng.choice(
                    idx,
                    len(idx),
                    replace=True
                )
                for group in range(4)
                for idx in [
                    np.flatnonzero(
                        validation_strata == group
                    )
                ]
            ]
        )
    )


# Train and validate the controlled experiments
results = []
validation_predictions = {}

with threadpool_limits(
    limits=THREADS
):

    for (
        name,
        view,
        dimension,
        histology,
        is_control
    ) in specifications:

        result_file = (
            RESULTS / f"{name}_summary.json"
        )

        pred_file = (
            RESULTS
            / f"{name}_validation_predictions.csv"
        )

        model_file = (
            MODELS / f"{name}.joblib"
        )

        # Reuse completed experiments
        if all(
            path.exists()
            for path in (
                result_file,
                pred_file,
                model_file
            )
        ):

            prior = json.loads(
                result_file.read_text(
                    encoding="utf-8"
                )
            )

            if prior.get("signature") != signature:
                raise RuntimeError(
                    f"Cached experiment signature mismatch: {name}"
                )

            pred = pd.read_csv(
                pred_file,
                dtype={"case_id": str}
            )

            if not np.array_equal(
                pred.case_id.to_numpy(),
                patients.case_id.iloc[
                    val_ids
                ].to_numpy()
            ):
                raise RuntimeError(
                    f"Cached validation case mismatch: {name}"
                )

            validation_predictions[name] = (
                pred.prob_advanced.to_numpy(
                    dtype=float
                )
            )

            results.append(prior)

            print(
                f"{name}: reused verified run",
                flush=True
            )

            continue

        # Prepare fold-specific SVD factors
        fold_data = []

        for fold in range(FOLDS):

            fit_ids, hold_ids = folds[
                fold
            ]

            z_fit, z_hold, _ = projected(
                view,
                dimension,
                fold,
                histology=histology,
                final=False
            )

            fold_data.append(
                (
                    fit_ids,
                    hold_ids,
                    z_fit,
                    z_hold
                )
            )

        best_choice = None
        best_quality = (
            -np.inf,
            -np.inf
        )

        best_oof = None

        # Hyperparameters selected from training CV only
        for C in C_GRID:

            for weight in WEIGHTS:

                probabilities = np.full(
                    len(y_train),
                    np.nan
                )

                fold_ap = []
                fold_auc = []

                for (
                    fit_ids,
                    hold_ids,
                    z_fit,
                    z_hold
                ) in fold_data:

                    lr = LogisticRegression(
                        C=C,
                        class_weight=weight,
                        solver="liblinear",
                        max_iter=2000,
                        tol=1e-4,
                        random_state=SEED
                    )

                    lr.fit(
                        z_fit,
                        y_train[fit_ids]
                    )

                    p = lr.predict_proba(
                        z_hold
                    )[:, 1]

                    probabilities[
                        hold_ids
                    ] = p

                    fold_ap.append(
                        average_precision_score(
                            y_train[hold_ids],
                            p
                        )
                    )

                    fold_auc.append(
                        roc_auc_score(
                            y_train[hold_ids],
                            p
                        )
                    )

                if not np.isfinite(
                    probabilities
                ).all():
                    raise RuntimeError(
                        "Incomplete out-of-fold probabilities."
                    )

                quality_score = (
                    float(
                        np.mean(fold_ap)
                    ),
                    float(
                        np.mean(fold_auc)
                    )
                )

                if quality_score > best_quality:

                    best_choice = (
                        float(C),
                        weight
                    )

                    best_quality = (
                        quality_score
                    )

                    best_oof = (
                        probabilities.copy()
                    )

        threshold = choose_threshold(
            y_train,
            best_oof
        )

        # Refit the selected model on all training patients
        z_fit, z_val, svd = projected(
            view,
            dimension,
            0,
            histology=histology,
            final=True
        )

        classifier = LogisticRegression(
            C=best_choice[0],
            class_weight=best_choice[1],
            solver="liblinear",
            max_iter=2000,
            tol=1e-4,
            random_state=SEED
        )

        classifier.fit(
            z_fit,
            y_train
        )

        p_val = classifier.predict_proba(
            z_val
        )[:, 1]

        validation_predictions[name] = (
            p_val
        )

        validation = metrics(
            y_val,
            p_val,
            threshold
        )

        subgroup = {}

        for title, code in (
            ("LUAD", 0),
            ("LUSC", 1)
        ):

            idx = np.flatnonzero(
                h_val == code
            )

            subgroup[title] = {
                "n": len(idx),
                "advanced": int(
                    y_val[idx].sum()
                ),
                **metrics(
                    y_val[idx],
                    p_val[idx],
                    threshold
                )
            }

        # Save model and preprocessing
        model_payload = {
            "signature": signature,
            "representation": view,
            "channels": CHANNELS,
            "components": dimension,
            "histology_offset": histology,
            "svd": svd,
            "classifier": classifier,
            "threshold_from_training_OOF": threshold,
            "C": best_choice[0],
            "class_weight": best_choice[1]
        }

        temp_model = model_file.with_suffix(
            ".joblib.tmp"
        )

        joblib.dump(
            model_payload,
            temp_model,
            compress=3
        )

        os.replace(
            temp_model,
            model_file
        )

        pd.DataFrame({
            "case_id": patients.case_id.iloc[
                val_ids
            ].to_numpy(),
            "histology_label": h_val,
            "stage_label": y_val,
            "prob_advanced": p_val,
            "predicted_advanced": (
                p_val >= threshold
            ).astype(np.int32)
        }).to_csv(
            pred_file,
            index=False
        )

        result = {
            "name": name,
            "signature": signature,
            "selected_on": (
                "training_fourfold_stage_histology_CV_AP"
            ),
            "is_diagnostic_control": is_control,
            "CV_AP": best_quality[0],
            "CV_AUC": best_quality[1],
            "C": best_choice[0],
            "class_weight": best_choice[1],
            "OOF_threshold": threshold,
            "OOF_AP": float(
                average_precision_score(
                    y_train,
                    best_oof
                )
            ),
            "validation": validation,
            "subgroup": subgroup,
            "test_evaluated": False
        }

        temporary_result = result_file.with_suffix(
            ".json.tmp"
        )

        temporary_result.write_text(
            json.dumps(
                result,
                indent=2
            ),
            encoding="utf-8"
        )

        os.replace(
            temporary_result,
            result_file
        )

        results.append(result)

        print(
            f"{name:35} "
            f"CV AP={best_quality[0]:.4f} | "
            f"Val AP={validation['AP']:.4f} "
            f"AUC={validation['ROC_AUC']:.4f} "
            f"MCC={validation['MCC']:.4f} "
            f"Recall={validation['advanced_recall']:.3f}",
            flush=True
        )


# Paired validation comparisons
comparisons = {}

for label, target, control in (
    (
        "Restored_methylation_vs_legacy",
        "Repaired_FourOmics_SVD64",
        "Legacy_Bugged_SVD64"
    ),
    (
        "Four_omics_vs_three_omics",
        "Repaired_FourOmics_SVD64",
        "Repaired_ThreeOmics_SVD64"
    ),
    (
        "Functional_PPI_vs_randomized",
        "Repaired_FunctionalPPI_SVD64",
        "Repaired_RandomizedPPI_SVD64"
    ),
    (
        "Physical_PPI_vs_randomized",
        "Repaired_PhysicalPPI_SVD64",
        "Repaired_RandomizedPPI_SVD64"
    )
):

    p_a = validation_predictions[
        target
    ]

    p_b = validation_predictions[
        control
    ]

    deltas = [
        float(
            average_precision_score(
                y_val[ix],
                p_a[ix]
            )
            - average_precision_score(
                y_val[ix],
                p_b[ix]
            )
        )
        for ix in boot_indices
    ]

    comparisons[label] = {
        "delta_validation_AP": float(
            average_precision_score(
                y_val,
                p_a
            )
            - average_precision_score(
                y_val,
                p_b
            )
        ),
        "exploratory_paired_stratified_95pct_CI": [
            float(value)
            for value in np.percentile(
                deltas,
                [2.5, 97.5]
            )
        ]
    }


# Prepare the final comparison table
summary = pd.DataFrame([
    {
        "model": result["name"],
        "CV_AP": result["CV_AP"],
        "CV_AUC": result["CV_AUC"],
        "validation_AP": (
            result["validation"]["AP"]
        ),
        "validation_ROC_AUC": (
            result["validation"]["ROC_AUC"]
        ),
        "validation_MCC": (
            result["validation"]["MCC"]
        ),
        "validation_advanced_recall": (
            result["validation"]["advanced_recall"]
        ),
        "validation_Brier": (
            result["validation"]["Brier"]
        ),
        "OOF_threshold": (
            result["OOF_threshold"]
        )
    }
    for result in results
]).sort_values(
    [
        "CV_AP",
        "CV_AUC"
    ],
    ascending=False
)

summary.to_csv(
    RESULTS / "stage14_corrected_multimodal_comparison.csv",
    index=False
)

# Exclude diagnostic controls from model selection
eligible_models = summary.loc[
    ~summary.model.isin(
        [
            "Legacy_Bugged_SVD64",
            "Repaired_RandomizedPPI_SVD64",
            "Repaired_MethylationOnly_SVD32"
        ]
    )
]

selected_model = str(
    eligible_models.iloc[0].model
)

report = {
    "generated_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "version": VERSION,
    "signature": signature,

    "old_channel3_nonzero": int(
        np.count_nonzero(
            old[:, :, 3]
        )
    ),

    "new_channel3_nonzero": int(
        np.count_nonzero(
            x_v2[:, :, 3]
        )
    ),

    "new_channel3_training_std": float(
        np.std(
            x_v2[train_ids, :, 3]
        )
    ),

    "methylation_observed_training_fraction": float(
        observed[train_ids].mean()
    ),

    "methylation_mask_disagreement": (
        mask_disagreements
    ),

    "patient_counts": {
        "train": len(train_ids),
        "validation": len(val_ids),
        "sealed_test": len(test_ids)
    },

    "selected_using_training_CV_only": (
        selected_model
    ),

    "paired_exploratory_comparisons": (
        comparisons
    ),

    "corrected_feature_file": str(
        v2_file
    ),

    "test_labels_used_for_modeling": False,

    "limitations": [
        (
            "Stage09 gene list and original RNA/CNV "
            "scalers were fitted on the complete "
            "training partition, so inner CV "
            "is not fully nested."
        ),
        (
            "Repeated use of the same validation "
            "set from Stages 10-13 creates "
            "adaptive-selection risk."
        ),
        (
            "Threshold selected using OOF targets "
            "may be optimistic and is not "
            "external calibration."
        ),
        (
            "Gene-level methylation uses promoter "
            "CpG aggregation; platform and probe "
            "annotations require further audit."
        )
    ]
}

(
    RESULTS / "stage14_summary.json"
).write_text(
    json.dumps(
        report,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 73)
print(
    "STAGE 14 FINAL - "
    "REPAIRED FOUR-OMICS COMPARISON"
)
print("=" * 73)

print(
    summary.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

print(
    "\nPAIRED EXPLORATORY VALIDATION DIFFERENCES"
)

for key, value in comparisons.items():
    print(key, value)

print(
    "\nModel selected by training CV:",
    selected_model
)

print(
    "Corrected four-omics features:",
    v2_file
)

print("Reports:", RESULTS)
print("Model checkpoints:", MODELS)

print(
    "Sealed test patients:",
    len(test_ids),
    "(not evaluated)"
)

print("\nSTAGE 14 COMPLETED")

STAGE 14 - REPAIR METHYLATION CHANNEL AND REASSESS FOUR-OMICS
Locked cohorts: train=667, validation=143, sealed test=143
Original methylation channel 3 nonzero values: 0
Methylation mask discrepancies: 0
Restored methylation train SD: 1.003
Restored methylation observed fraction: 0.8198
Original mutation indicators preserved: PASSED

Training-only molecular-channel audit:
   modality histology    stage  patients  gene_feature_std  fraction_nonzero  observed_fraction
        RNA      LUAD    Early       265            1.0579            0.9194             1.0000
        RNA      LUAD Advanced        73            1.0699            0.9181             1.0000
        RNA      LUSC    Early       269            1.1310            0.9239             1.0000
        RNA      LUSC Advanced        60            1.1579            0.9232             1.0000
        CNV      LUAD    Early       265            1.2019            0.6404             0.9951
        CNV      LUAD Advanced        73         

In [14]:

import hashlib
import json
import os
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests

from scipy.stats import chi2_contingency
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ============================================================
# 1. Configuration
# ============================================================

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

RAW = ROOT / "data" / "processed" / "gene_level"
INTERIM = ROOT / "data" / "interim"
SPLIT_DIR = ROOT / "data" / "splits"
OUTPUT = ROOT / "results" / "metrics" / "stage15"
MODEL_DIR = ROOT / "models" / "checkpoints" / "stage15"

OUTPUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_GENES = 1600
N_FOLDS = 4
REPEATS = 2
LATENT_DIM = 48

C_VALUES = (0.01, 0.1, 1.0)
WEIGHTS = (None, "balanced")

THREADS = min(4, os.cpu_count() or 1)
FETCH_PLATFORM_METADATA = True

VERSION = "stage15_fold_nested_raw_omics_v1"

# ============================================================
# 2. Verify patient order and locked partitions
# ============================================================

cases = pd.read_csv(
    RAW / "case_order.csv",
    dtype=str,
    keep_default_na=False
)

genes = pd.read_csv(
    RAW / "gene_reference.csv",
    dtype=str,
    keep_default_na=False
)

splits = pd.read_csv(
    SPLIT_DIR / "locked_four_omics_v1.csv",
    dtype=str,
    keep_default_na=False
)

if (
    not cases.case_id.is_unique
    or not splits.case_id.is_unique
    or not genes.ensembl_id.is_unique
):
    raise RuntimeError("Duplicate patient or gene identifiers.")

if len(cases) != 953 or len(genes) != 19944:
    raise RuntimeError("Unexpected Stage 08 cohort dimensions.")

lock = cases[
    ["case_id", "histology", "stage_group"]
].merge(
    splits[
        ["case_id", "histology", "stage_group", "split"]
    ],
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_source", "_locked")
)

if not (
    lock.histology_source.eq(lock.histology_locked).all()
    and lock.stage_group_source.eq(
        lock.stage_group_locked
    ).all()
):
    raise RuntimeError("Clinical labels or patient order mismatch.")

train_indices = np.flatnonzero(
    lock.split.eq("train").to_numpy()
)

val_indices = np.flatnonzero(
    lock.split.eq("validation").to_numpy()
)

test_indices = np.flatnonzero(
    lock.split.eq("test").to_numpy()
)

if (
    len(train_indices),
    len(val_indices),
    len(test_indices)
) != (667, 143, 143):
    raise RuntimeError("Locked partition sizes changed.")

y = (
    lock.stage_group_source.iloc[train_indices]
    .eq("Advanced")
    .astype(int)
    .to_numpy()
)

h = (
    lock.histology_source.iloc[train_indices]
    .eq("LUSC")
    .astype(int)
    .to_numpy()
)

case_ids = (
    lock.case_id.iloc[train_indices]
    .astype(str)
    .to_numpy()
)

if set(y) != {0, 1} or set(h) != {0, 1}:
    raise RuntimeError("Incomplete stage or histology labels.")

# ============================================================
# 3. Load the original Stage 08 molecular matrices
# ============================================================

source_paths = {
    "rna": RAW / "rna_tpm.npy",
    "cnv": RAW / "cnv_absolute.npy",
    "mutation": RAW / "mutation_nonsynonymous.npy",
    "methylation": RAW / "methylation_promoter_beta.npy"
}

source = {
    name: np.load(path, mmap_mode="r")
    for name, path in source_paths.items()
}

for name, array in source.items():

    if array.shape != (953, 19944):
        raise RuntimeError(
            f"Invalid {name} matrix shape: {array.shape}"
        )

print("=" * 73)
print("STAGE 15 - FOLD-NESTED FOUR-OMICS AND METHYLATION QUALITY")
print("=" * 73)

print(
    f"Train={len(y)}, validation={len(val_indices)}, "
    f"sealed test={len(test_indices)}"
)

print(
    f"Raw protein-coding genes={len(genes)}; "
    f"training advanced prevalence={y.mean():.3f}"
)

# ============================================================
# 4. Audit methylation completeness in training patients
# ============================================================

meth_train = np.asarray(
    source["methylation"][train_indices],
    dtype=np.float32
)

coverage = (
    np.isfinite(meth_train)
    & (meth_train >= 0)
    & (meth_train <= 1)
).mean(axis=1)

quality = np.where(
    coverage >= 0.90,
    "high_90plus",
    np.where(
        coverage >= 0.50,
        "moderate_50to90",
        "low_under50"
    )
)

site = (
    pd.Series(case_ids)
    .str.extract(
        r"^TCGA-([A-Z0-9]{2})-",
        expand=False
    )
    .fillna("unknown")
    .to_numpy()
)

audit = pd.DataFrame({
    "case_id": case_ids,
    "histology": np.where(h == 0, "LUAD", "LUSC"),
    "stage_group": np.where(y == 0, "Early", "Advanced"),
    "tissue_source_site": site,
    "promoter_gene_coverage": coverage,
    "coverage_band": quality
})

summary_qc = (
    audit.groupby(
        ["histology", "stage_group"],
        observed=True
    )
    .agg(
        patients=("case_id", "size"),
        mean_coverage=("promoter_gene_coverage", "mean"),
        median_coverage=("promoter_gene_coverage", "median"),
        low_coverage=(
            "coverage_band",
            lambda s: int((s == "low_under50").sum())
        ),
        high_coverage=(
            "coverage_band",
            lambda s: int((s == "high_90plus").sum())
        )
    )
    .reset_index()
)

print("\nTraining-only methylation coverage:")

print(
    summary_qc.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

chi_rows = []

for histology in ("LUAD", "LUSC"):

    subset = audit.loc[
        audit.histology.eq(histology)
    ]

    table = pd.crosstab(
        subset.stage_group,
        subset.coverage_band
    )

    if table.shape[1] >= 2:
        p = float(
            chi2_contingency(
                table,
                correction=False
            )[1]
        )
    else:
        p = 1.0

    chi_rows.append({
        "histology": histology,
        "coverage_stage_chisquare_p": p,
        "coverage_groups": int(table.shape[1])
    })

print("\nCoverage-versus-stage audit:")
print(pd.DataFrame(chi_rows).to_string(index=False))

# ============================================================
# 5. Retrieve GDC methylation platform metadata
# ============================================================

platform_cache = (
    OUTPUT / "gdc_methylation_platform_training.csv"
)

if platform_cache.exists():

    platform = pd.read_csv(
        platform_cache,
        dtype=str,
        keep_default_na=False
    )

else:

    platform = pd.DataFrame(
        columns=["file_id", "platform"]
    )

    if FETCH_PLATFORM_METADATA:

        manifest = pd.read_csv(
            INTERIM / "tcga_full_four_omics_download_manifest.csv",
            dtype=str,
            keep_default_na=False
        )

        candidate = (
            manifest.loc[
                manifest.modality.eq("methylation")
                & manifest.case_id.isin(set(case_ids)),
                ["case_id", "file_id"]
            ]
            .drop_duplicates("file_id")
        )

        if len(candidate) != len(case_ids):
            raise RuntimeError(
                "Training methylation file association is incomplete."
            )

        session = requests.Session()

        retry = Retry(
            total=3,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET"]
        )

        session.mount(
            "https://",
            HTTPAdapter(max_retries=retry)
        )

        found = []

        for start in range(0, len(candidate), 100):

            identifiers = (
                candidate.file_id.iloc[start:start + 100]
                .tolist()
            )

            query = {
                "op": "in",
                "content": {
                    "field": "file_id",
                    "value": identifiers
                }
            }

            params = {
                "filters": json.dumps(query),
                "fields": "file_id,platform",
                "size": 150,
                "format": "JSON"
            }

            try:

                response = session.get(
                    "https://api.gdc.cancer.gov/files",
                    params=params,
                    timeout=(15, 75)
                )

                response.raise_for_status()

                for item in response.json()["data"]["hits"]:

                    found.append({
                        "file_id": item.get("file_id", ""),
                        "platform": (
                            item.get("platform") or "unreported"
                        )
                    })

            except (
                requests.RequestException,
                KeyError,
                ValueError
            ) as error:

                print(
                    "Platform metadata unavailable for batch",
                    start // 100 + 1,
                    str(error)[:120]
                )

        platform = (
            pd.DataFrame(
                found,
                columns=["file_id", "platform"]
            )
            .drop_duplicates("file_id")
        )

        platform = candidate.merge(
            platform,
            on="file_id",
            how="left"
        )

        platform["platform"] = (
            platform["platform"].fillna("unreported")
        )

        platform.to_csv(
            platform_cache,
            index=False
        )

if "case_id" in platform.columns:

    audit = audit.merge(
        platform[
            ["case_id", "platform"]
        ].drop_duplicates("case_id"),
        on="case_id",
        how="left",
        validate="one_to_one"
    )

else:
    audit["platform"] = "unreported"

audit["platform"] = (
    audit["platform"].fillna("unreported")
)

print("\nGDC methylation platform by histology and stage:")

print(
    pd.crosstab(
        [
            audit.histology,
            audit.stage_group
        ],
        audit.platform
    ).to_string()
)

audit.to_csv(
    OUTPUT / "training_methylation_platform_coverage_audit.csv",
    index=False
)

summary_qc.to_csv(
    OUTPUT / "methylation_coverage_by_stage_histology.csv",
    index=False
)

pd.DataFrame(chi_rows).to_csv(
    OUTPUT / "methylation_coverage_stage_association.csv",
    index=False
)

# ============================================================
# 6. Define modality-specific molecular transformations
# ============================================================

def safe_transform(name, values):

    values = np.asarray(
        values,
        dtype=np.float32
    )

    if name in ("rna", "cnv"):

        observed = (
            np.isfinite(values)
            & (values >= 0)
        )

        clean = np.where(
            observed,
            values,
            0
        )

        if name == "rna":

            result = np.log1p(clean)

        else:

            result = np.clip(
                np.log2((clean + 0.5) / 2.5),
                -3,
                3
            )

    elif name == "methylation":

        observed = (
            np.isfinite(values)
            & (values >= 0)
            & (values <= 1)
        )

        result = np.where(
            observed,
            values,
            0
        )

    else:

        observed = np.isfinite(values)

        result = (
            np.where(observed, values, 0) > 0
        ).astype(np.float32)

    return (
        np.asarray(result, dtype=np.float32),
        observed
    )


def masked_statistics(values, observed):

    count = observed.sum(
        axis=0
    ).astype(np.float64)

    first = np.asarray(
        (values * observed).sum(axis=0),
        dtype=np.float64
    )

    second = np.asarray(
        ((values * observed) ** 2).sum(axis=0),
        dtype=np.float64
    )

    mean = first / np.maximum(
        count,
        1
    )

    variance = np.maximum(
        second / np.maximum(count, 1)
        - mean ** 2,
        0
    )

    return count, np.sqrt(variance)


# ============================================================
# 7. Select genes within each training fold
# ============================================================

def choose_genes(fit_local):

    global_ids = train_indices[fit_local]

    rna_raw = np.asarray(
        source["rna"][global_ids],
        dtype=np.float32
    )

    cnv_raw = np.asarray(
        source["cnv"][global_ids],
        dtype=np.float32
    )

    mutation_raw = np.asarray(
        source["mutation"][global_ids],
        dtype=np.float32
    )

    rna, rna_observed = safe_transform(
        "rna",
        rna_raw
    )

    cnv, cnv_observed = safe_transform(
        "cnv",
        cnv_raw
    )

    mutation, _ = safe_transform(
        "mutation",
        mutation_raw
    )

    rna_count, rna_sd = masked_statistics(
        rna,
        rna_observed
    )

    cnv_count, cnv_sd = masked_statistics(
        cnv,
        cnv_observed
    )

    expressed = np.mean(
        rna_raw >= 1,
        axis=0
    )

    eligible = (
        (rna_count / len(fit_local) >= 0.95)
        & (cnv_count / len(fit_local) >= 0.75)
        & (expressed >= 0.10)
    )

    if eligible.sum() < N_GENES:

        eligible = (
            (rna_count / len(fit_local) >= 0.85)
            & (cnv_count / len(fit_local) >= 0.65)
        )

    indices = np.flatnonzero(
        eligible
    )

    if len(indices) < N_GENES:
        raise RuntimeError(
            f"Only {len(indices)} genes passed fold-specific QC."
        )

    def percentile_ranks(scores):

        ranks = np.empty(
            len(scores),
            dtype=np.float64
        )

        order = np.argsort(
            scores,
            kind="stable"
        )

        ranks[order] = np.arange(
            len(scores)
        )

        return ranks / max(
            len(scores) - 1,
            1
        )

    mutation_prevalence = mutation.mean(
        axis=0
    )

    mutation_variance = (
        mutation_prevalence
        * (1.0 - mutation_prevalence)
    )

    score = (
        0.45 * percentile_ranks(rna_sd[indices])
        + 0.35 * percentile_ranks(cnv_sd[indices])
        + 0.20 * percentile_ranks(
            mutation_variance[indices]
        )
    )

    selected = indices[
        np.argsort(
            -score,
            kind="stable"
        )[:N_GENES]
    ]

    return np.sort(selected)


# ============================================================
# 8. Fit fold-specific normalization and missingness masks
# ============================================================

def prepare_fold(fit_local, held_local, selected):

    fit_global = train_indices[
        fit_local
    ]

    held_global = (
        train_indices[held_local]
        if held_local is not None
        else fit_global
    )

    xfit = np.zeros(
        (len(fit_global), N_GENES, 8),
        dtype=np.float32
    )

    xhold = np.zeros(
        (len(held_global), N_GENES, 8),
        dtype=np.float32
    )

    parameters = {}

    for name, channel, mask_channel, floor in (
        ("rna", 0, 4, 0.10),
        ("cnv", 1, 5, 0.15),
        ("methylation", 3, 7, 0.04)
    ):

        train_raw = np.asarray(
            source[name][
                np.ix_(
                    fit_global,
                    selected
                )
            ],
            dtype=np.float32
        )

        held_raw = np.asarray(
            source[name][
                np.ix_(
                    held_global,
                    selected
                )
            ],
            dtype=np.float32
        )

        train_values, train_valid = safe_transform(
            name,
            train_raw
        )

        held_values, held_valid = safe_transform(
            name,
            held_raw
        )

        with warnings.catch_warnings():

            warnings.simplefilter(
                "ignore",
                RuntimeWarning
            )

            masked = np.where(
                train_valid,
                train_values,
                np.nan
            )

            median = np.nanmedian(
                masked,
                axis=0
            )

            q25, q75 = np.nanpercentile(
                masked,
                [25, 75],
                axis=0
            )

        median = np.where(
            np.isfinite(median),
            median,
            0
        )

        dispersion = np.where(
            np.isfinite(q75 - q25),
            (q75 - q25) / 1.349,
            0
        )

        scale = np.maximum(
            dispersion,
            floor
        ).astype(np.float32)

        for target, values, valid in (
            (xfit, train_values, train_valid),
            (xhold, held_values, held_valid)
        ):

            standardized = (
                np.where(
                    valid,
                    values,
                    median
                ) - median
            ) / scale

            target[:, :, channel] = np.where(
                valid,
                np.clip(
                    standardized,
                    -5,
                    5
                ),
                0
            )

            target[:, :, mask_channel] = (
                valid.astype(np.float32)
            )

        parameters[name] = {
            "median": median.astype(np.float32),
            "scale": scale
        }

    for target, global_ids in (
        (xfit, fit_global),
        (xhold, held_global)
    ):

        raw_mutation = np.asarray(
            source["mutation"][
                np.ix_(
                    global_ids,
                    selected
                )
            ],
            dtype=np.float32
        )

        target[:, :, 2] = (
            raw_mutation > 0
        ).astype(np.float32)

        target[:, :, 6] = 1.0

    if (
        not np.isfinite(xfit).all()
        or not np.isfinite(xhold).all()
    ):
        raise RuntimeError(
            "Non-finite fold-specific molecular tensor."
        )

    return xfit, xhold, parameters


# ============================================================
# 9. Predeclare molecular-value and missingness controls
# ============================================================

SPECS = {
    "ThreeOmics": {
        "channels": (0, 1, 2, 4, 5)
    },
    "ThreePlusMethylationMask": {
        "channels": (0, 1, 2, 4, 5, 7)
    },
    "FourValuesNoMethylationMask": {
        "channels": (0, 1, 2, 3, 4, 5)
    },
    "FourOmicsWithMask": {
        "channels": (0, 1, 2, 3, 4, 5, 7)
    },
    "FourOmicsCoverageGated": {
        "channels": (0, 1, 2, 3, 4, 5, 7),
        "gate": True
    },
    "ThreeOmicsPlusCoverage": {
        "channels": (0, 1, 2, 4, 5),
        "quality": True
    },
    "RNA_CNV": {
        "channels": (0, 1, 4, 5)
    },
    "MethylationOnly": {
        "channels": (3, 7),
        "control": True
    },
    "CoverageHistologyOnly": {
        "control": True,
        "quality_only": True
    }
}


def make_view(tensor, spec, histology):

    if spec.get("quality_only"):

        fractions = np.column_stack([
            tensor[:, :, j].mean(axis=1)
            for j in (4, 5, 7)
        ])

        return np.column_stack([
            fractions,
            histology
        ]).astype(np.float32)

    channels = spec["channels"]

    parts = []

    quality_fraction = (
        tensor[:, :, 7].mean(axis=1)
    )

    for channel in channels:

        values = tensor[
            :,
            :,
            channel
        ]

        if (
            channel == 3
            and spec.get("gate")
        ):

            gate = np.clip(
                (quality_fraction - 0.50) / 0.40,
                0,
                1
            )

            values = (
                values * gate[:, None]
            )

        parts.append(values)

    return np.concatenate(
        parts,
        axis=1
    ).astype(np.float32)


# ============================================================
# 10. Fit latent representation inside the training fold
# ============================================================

def latent_fit(
    xfit,
    xhold,
    spec,
    histfit,
    histhold,
    seed
):

    train_view = make_view(
        xfit,
        spec,
        histfit
    )

    held_view = make_view(
        xhold,
        spec,
        histhold
    )

    # Remove columns constant in the fitting patients.
    active = (
        np.var(
            train_view,
            axis=0
        ) > 1e-9
    )

    if not np.any(active):
        raise RuntimeError(
            "All features are constant within a training fold."
        )

    train_view = train_view[:, active]
    held_view = held_view[:, active]

    if spec.get("quality_only"):

        svd = None
        zfit = train_view
        zhold = held_view

    else:

        dimension = min(
            LATENT_DIM,
            train_view.shape[0] - 2,
            train_view.shape[1] - 1
        )

        if dimension < 5:
            raise RuntimeError(
                "Insufficient nonconstant latent features."
            )

        svd = TruncatedSVD(
            n_components=dimension,
            n_iter=4,
            random_state=seed
        )

        with warnings.catch_warnings():

            warnings.simplefilter(
                "ignore",
                RuntimeWarning
            )

            zfit = svd.fit_transform(
                train_view
            )

        zhold = svd.transform(
            held_view
        )

    scaler = StandardScaler()

    zfit = np.clip(
        scaler.fit_transform(zfit),
        -8,
        8
    ).astype(np.float32)

    zhold = np.clip(
        scaler.transform(zhold),
        -8,
        8
    ).astype(np.float32)

    if spec.get("quality"):

        qfit = (
            xfit[:, :, 7].mean(axis=1)
        )

        qhold = (
            xhold[:, :, 7].mean(axis=1)
        )

        qmean = qfit.mean()
        qstd = max(
            qfit.std(),
            0.05
        )

        zfit = np.column_stack([
            zfit,
            (qfit - qmean) / qstd
        ]).astype(np.float32)

        zhold = np.column_stack([
            zhold,
            (qhold - qmean) / qstd
        ]).astype(np.float32)

    else:

        qmean = None
        qstd = None

    if (
        not np.isfinite(zfit).all()
        or not np.isfinite(zhold).all()
    ):
        raise RuntimeError(
            "Non-finite fold-specific latent representation."
        )

    fitted = {
        "active_features": active,
        "svd": svd,
        "latent_scaler": scaler,
        "quality_mean": qmean,
        "quality_scale": qstd
    }

    return zfit, zhold, fitted


# ============================================================
# 11. Repeated stratified training-only cross-validation
# ============================================================

strata = (
    2 * y + h
)

folds = []

for repetition in range(REPEATS):

    cv = StratifiedKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=SEED + repetition * 37
    )

    for fold_number, (
        fit,
        hold
    ) in enumerate(
        cv.split(
            np.arange(len(y)),
            strata
        )
    ):

        if set(strata[hold]) != {
            0, 1, 2, 3
        }:
            raise RuntimeError(
                "A fold is missing a histology-stage group."
            )

        folds.append((
            repetition,
            fold_number,
            fit,
            hold
        ))

print(
    "\nTraining-only nested preprocessing folds:",
    len(folds)
)

# ============================================================
# 12. Construct independent fold-specific molecular tensors
# ============================================================

fold_prepared = []
fold_selected_genes = []

selection_counts = np.zeros(
    len(genes),
    dtype=np.int32
)

for repetition, fold_number, fit, hold in folds:

    selected = choose_genes(
        fit
    )

    fold_selected_genes.append(
        selected
    )

    selection_counts[selected] += 1

    xfit, xhold, _ = prepare_fold(
        fit,
        hold,
        selected
    )

    fold_prepared.append((
        repetition,
        fold_number,
        fit,
        hold,
        xfit,
        xhold
    ))

    print(
        f"Fold {len(fold_prepared):02d}/{len(folds)} | "
        f"Genes={len(selected)} | "
        f"Methylation observed={xfit[:, :, 7].mean():.3f}",
        flush=True
    )


# ============================================================
# 13. Estimate gene-selection stability
# ============================================================

selection_overlap = []

for i in range(len(fold_selected_genes)):

    for j in range(
        i + 1,
        len(fold_selected_genes)
    ):

        left = fold_selected_genes[i]
        right = fold_selected_genes[j]

        common = len(
            np.intersect1d(
                left,
                right,
                assume_unique=True
            )
        )

        jaccard = (
            common / (2 * N_GENES - common)
        )

        selection_overlap.append(
            jaccard
        )

selection_frequency = pd.DataFrame({
    "ensembl_id": genes.ensembl_id,
    "selected_folds": selection_counts,
    "selection_frequency": (
        selection_counts / len(folds)
    )
})

selection_frequency.to_csv(
    OUTPUT / "training_fold_gene_selection_frequency.csv",
    index=False
)

stability = {
    "mean_fold_pairwise_jaccard": float(
        np.mean(selection_overlap)
    ),
    "minimum_fold_pairwise_jaccard": float(
        np.min(selection_overlap)
    ),
    "genes_selected_all_folds": int(
        (selection_counts == len(folds)).sum()
    ),
    "genes_selected_half_or_more_folds": int(
        (selection_counts >= len(folds) / 2).sum()
    ),
    "genes_selected_at_least_once": int(
        (selection_counts > 0).sum()
    )
}

print("\nFold-wise gene selection stability:")
print(stability)


# ============================================================
# 14. Configure cross-validated classifiers
# ============================================================

def make_classifier(C, weight):

    return LogisticRegression(
        C=float(C),
        class_weight=weight,
        solver="liblinear",
        max_iter=2500,
        random_state=SEED
    )


results = []
oof_by_name = {}

source_stats = {
    name: {
        "size": path.stat().st_size,
        "mtime_ns": path.stat().st_mtime_ns
    }
    for name, path in source_paths.items()
}

cache_signature = hashlib.sha256(
    json.dumps(
        {
            "version": VERSION,
            "train_cases": case_ids.tolist(),
            "train_stage": y.tolist(),
            "train_histology": h.tolist(),
            "file_stats": source_stats,
            "repeats": REPEATS,
            "folds": N_FOLDS,
            "seed": SEED,
            "genes": N_GENES,
            "dimension": LATENT_DIM,
            "models": SPECS,
            "C_values": C_VALUES,
            "weights": WEIGHTS
        },
        sort_keys=True
    ).encode()
).hexdigest()


# ============================================================
# 15. Train and compare nine predeclared model configurations
# ============================================================

with threadpool_limits(limits=THREADS):

    for name, spec in SPECS.items():

        result_file = (
            OUTPUT / f"{name}_CV_summary.json"
        )

        oof_file = (
            OUTPUT / f"{name}_training_oof.npy"
        )

        # Reuse matching completed model evaluations.
        if (
            result_file.exists()
            and oof_file.exists()
        ):

            prior = json.loads(
                result_file.read_text(
                    encoding="utf-8"
                )
            )

            if prior.get("signature") != cache_signature:
                raise RuntimeError(
                    f"Cached results use different inputs: {name}"
                )

            cached_oof = np.load(
                oof_file
            )

            if cached_oof.shape != (
                REPEATS,
                len(y)
            ):
                raise RuntimeError(
                    f"Invalid cached OOF shape: {name}"
                )

            results.append(prior)
            oof_by_name[name] = cached_oof

            print(
                f"{name:<34} "
                f"cached CV AP={prior['CV_AP_mean']:.4f}",
                flush=True
            )

            continue

        latent_folds = []

        for (
            repetition,
            fold_number,
            fit,
            hold,
            xfit,
            xhold
        ) in fold_prepared:

            zfit, zhold, _ = latent_fit(
                xfit,
                xhold,
                spec,
                h[fit],
                h[hold],
                SEED + repetition * 100 + fold_number
            )

            latent_folds.append((
                repetition,
                fit,
                hold,
                zfit,
                zhold
            ))

        best = None

        for C in C_VALUES:

            for weight in WEIGHTS:

                scores = np.full(
                    (REPEATS, len(y)),
                    np.nan,
                    dtype=np.float64
                )

                fold_ap = []
                fold_auc = []

                hist_ap = {
                    "LUAD": [],
                    "LUSC": []
                }

                for (
                    repetition,
                    fit,
                    hold,
                    zfit,
                    zhold
                ) in latent_folds:

                    classifier = make_classifier(
                        C,
                        weight
                    )

                    classifier.fit(
                        zfit,
                        y[fit]
                    )

                    probabilities = (
                        classifier.predict_proba(
                            zhold
                        )[:, 1]
                    )

                    scores[
                        repetition,
                        hold
                    ] = probabilities

                    fold_ap.append(
                        float(
                            average_precision_score(
                                y[hold],
                                probabilities
                            )
                        )
                    )

                    fold_auc.append(
                        float(
                            roc_auc_score(
                                y[hold],
                                probabilities
                            )
                        )
                    )

                    for label, code in (
                        ("LUAD", 0),
                        ("LUSC", 1)
                    ):

                        group = (
                            h[hold] == code
                        )

                        hist_ap[label].append(
                            float(
                                average_precision_score(
                                    y[hold][group],
                                    probabilities[group]
                                )
                            )
                        )

                if not np.isfinite(
                    scores
                ).all():
                    raise RuntimeError(
                        f"Incomplete out-of-fold probabilities: {name}"
                    )

                quality_score = (
                    float(
                        np.mean(fold_ap)
                    ),
                    float(
                        np.mean(fold_auc)
                    )
                )

                if (
                    best is None
                    or quality_score > best["score"]
                ):

                    best = {
                        "score": quality_score,
                        "C": float(C),
                        "weight": weight,
                        "AP_SD": float(
                            np.std(
                                fold_ap,
                                ddof=1
                            )
                        ),
                        "hist_ap": {
                            key: float(np.mean(values))
                            for key, values in hist_ap.items()
                        },
                        "oof": scores.copy()
                    }

        oof_by_name[name] = (
            best["oof"]
        )

        np.save(
            oof_file,
            best["oof"]
        )

        result = {
            "name": name,
            "signature": cache_signature,
            "CV_AP_mean": best["score"][0],
            "CV_AP_SD": best["AP_SD"],
            "CV_AUC_mean": best["score"][1],
            "LUAD_CV_AP": best["hist_ap"]["LUAD"],
            "LUSC_CV_AP": best["hist_ap"]["LUSC"],
            "C": best["C"],
            "class_weight": best["weight"],
            "fold_nested_gene_selection": True,
            "validation_or_test_evaluated": False
        }

        temporary = result_file.with_suffix(
            ".json.tmp"
        )

        temporary.write_text(
            json.dumps(
                result,
                indent=2
            ),
            encoding="utf-8"
        )

        os.replace(
            temporary,
            result_file
        )

        results.append(result)

        print(
            f"{name:<34} "
            f"CV AP={result['CV_AP_mean']:.4f} "
            f"+/- {result['CV_AP_SD']:.4f} | "
            f"AUC={result['CV_AUC_mean']:.4f} | "
            f"LUAD={result['LUAD_CV_AP']:.4f} | "
            f"LUSC={result['LUSC_CV_AP']:.4f}",
            flush=True
        )


# ============================================================
# 16. Save the complete cross-validation comparison
# ============================================================

comparison = pd.DataFrame([
    {
        key: result[key]
        for key in (
            "name",
            "CV_AP_mean",
            "CV_AP_SD",
            "CV_AUC_mean",
            "LUAD_CV_AP",
            "LUSC_CV_AP",
            "C",
            "class_weight"
        )
    }
    for result in results
])

comparison = comparison.sort_values(
    [
        "CV_AP_mean",
        "CV_AUC_mean"
    ],
    ascending=False
)

comparison_path = (
    OUTPUT / "stage15_fold_nested_model_comparison.csv"
)

comparison.to_csv(
    comparison_path,
    index=False
)


# ============================================================
# 17. Paired exploratory comparisons of molecular contributions
# ============================================================

pairs = [
    (
        "Methylation_mask_vs_three",
        "ThreePlusMethylationMask",
        "ThreeOmics"
    ),
    (
        "Methylation_values_vs_three",
        "FourOmicsWithMask",
        "ThreeOmics"
    ),
    (
        "Four_values_vs_three_plus_mask",
        "FourOmicsWithMask",
        "ThreePlusMethylationMask"
    ),
    (
        "Coverage_gating_vs_four",
        "FourOmicsCoverageGated",
        "FourOmicsWithMask"
    )
]

pair_results = {}

bootstrap_rng = np.random.default_rng(
    SEED + 1515
)

for label, model_a, model_b in pairs:

    pa = oof_by_name[
        model_a
    ].mean(axis=0)

    pb = oof_by_name[
        model_b
    ].mean(axis=0)

    delta = float(
        average_precision_score(y, pa)
        - average_precision_score(y, pb)
    )

    sampled = []

    for _ in range(400):

        indices = np.concatenate([
            bootstrap_rng.choice(
                np.flatnonzero(
                    strata == group
                ),
                size=int(
                    (strata == group).sum()
                ),
                replace=True
            )
            for group in range(4)
        ])

        sampled.append(
            float(
                average_precision_score(
                    y[indices],
                    pa[indices]
                )
                - average_precision_score(
                    y[indices],
                    pb[indices]
                )
            )
        )

    pair_results[label] = {
        "OOF_AP_difference": delta,
        "exploratory_paired_95pct_CI": (
            np.percentile(
                sampled,
                [2.5, 97.5]
            ).tolist()
        )
    }


# ============================================================
# 18. Fit the selected predictive model on all training patients
# ============================================================

# Diagnostic controls are excluded from final model selection.
controls = {
    "CoverageHistologyOnly",
    "MethylationOnly",
    "ThreePlusMethylationMask",
    "ThreeOmicsPlusCoverage"
}

eligible_models = comparison.loc[
    ~comparison.name.isin(controls)
]

selected_name = str(
    eligible_models.iloc[0]["name"]
)

selected_result = next(
    result
    for result in results
    if result["name"] == selected_name
)

full_train_local = np.arange(
    len(y)
)

selected_gene_ids = choose_genes(
    full_train_local
)

full_x, _, parameters = prepare_fold(
    full_train_local,
    None,
    selected_gene_ids
)

with threadpool_limits(limits=THREADS):

    zfull, _, latent_model = latent_fit(
        full_x,
        full_x,
        SPECS[selected_name],
        h,
        h,
        SEED
    )

    final_classifier = make_classifier(
        selected_result["C"],
        selected_result["class_weight"]
    )

    final_classifier.fit(
        zfull,
        y
    )

final_model_path = (
    MODEL_DIR / "stage15_training_only_selected_model.joblib"
)

model_temp = final_model_path.with_name(
    final_model_path.name + ".tmp"
)

joblib.dump(
    {
        "version": VERSION,
        "model": selected_name,
        "signature": cache_signature,
        "selected_gene_positions": selected_gene_ids,
        "selected_ensembl_ids": (
            genes.ensembl_id.iloc[
                selected_gene_ids
            ].tolist()
        ),
        "fold_fit_preprocessing": parameters,
        "latent_transform": latent_model,
        "classifier": final_classifier,
        "spec": SPECS[selected_name],
        "validation_test_not_used": True
    },
    model_temp,
    compress=3
)

os.replace(
    model_temp,
    final_model_path
)


# ============================================================
# 19. Save the final research audit
# ============================================================

report = {
    "generated_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "signature": cache_signature,
    "train": len(y),
    "validation_not_loaded": len(val_indices),
    "test_not_loaded": len(test_indices),
    "folds": N_FOLDS,
    "repeats": REPEATS,
    "protein_coding_genes": len(genes),
    "selected_per_fold": N_GENES,
    "gene_selection_stability": stability,
    "coverage_stage_audit": chi_rows,
    "selected_by_training_CV": selected_name,
    "paired_exploratory": pair_results,
    "saved_model": str(final_model_path),
    "limitations": [
        "CV performance is internal, not external validation.",
        "The best model's selected CV score may be optimistic.",
        "Repeated use of previous validation data limits independence.",
        "HM450 promoter annotation may not cover all methylation platforms.",
        "Coverage bands are technical proxies, not platform identifiers."
    ]
}

summary_path = (
    OUTPUT / "stage15_summary.json"
)

summary_path.write_text(
    json.dumps(
        report,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 20. Final results
# ============================================================

print("\n" + "=" * 73)
print("STAGE 15 FINAL - TRAINING-ONLY FOLD-NESTED RESULTS")
print("=" * 73)

print(
    comparison.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

print("\nPaired exploratory OOF differences:")

for name, value in pair_results.items():
    print(name, value)

print("\nSelected on training CV:", selected_name)
print("Selected gene stability:", stability)

print("\nSaved reports:", OUTPUT)
print("Saved model:", final_model_path)

print(
    "\nValidation features and test features evaluated: NO"
)

print("\nSTAGE 15 COMPLETED")

STAGE 15 - FOLD-NESTED FOUR-OMICS AND METHYLATION QUALITY
Train=667, validation=143, sealed test=143
Raw protein-coding genes=19944; training advanced prevalence=0.199

Training-only methylation coverage:
histology stage_group  patients  mean_coverage  median_coverage  low_coverage  high_coverage
     LUAD    Advanced        73         0.6692           0.7546             9              0
     LUAD       Early       265         0.6683           0.7545            33              0
     LUSC    Advanced        60         0.5123           0.7504            21              0
     LUSC       Early       269         0.5888           0.7525            64              0

Coverage-versus-stage audit:
histology  coverage_stage_chisquare_p  coverage_groups
     LUAD                    0.977302                2
     LUSC                    0.072907                2

GDC methylation platform by histology and stage:
platform               Illumina Human Methylation 27  Illumina Human Methylation 450


In [15]:

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib
import json
import os
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

# ============================================================
# 1. Research configuration
# ============================================================

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

GENE_DIR = ROOT / "data" / "processed" / "gene_level"
SPLIT_PATH = (
    ROOT / "data" / "splits" / "locked_four_omics_v1.csv"
)
PLATFORM_PATH = (
    ROOT / "results" / "metrics" / "stage15"
    / "training_methylation_platform_coverage_audit.csv"
)
OUT = ROOT / "results" / "metrics" / "stage16"
MODELS = ROOT / "models" / "checkpoints" / "stage16"

for directory in (OUT, MODELS):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
GENE_COUNT = 1200
LATENT_COMPONENTS = 40
OUTER_FOLDS = 4
OUTER_REPEATS = 2
INNER_FOLDS = 3
C_GRID = (0.03, 0.3)
CLASS_WEIGHTS = (None, "balanced")
BOOTSTRAPS = 500
THREADS = min(4, os.cpu_count() or 1)

VERSION = "stage16_nested_platform_sensitivity_v1"

# ============================================================
# 2. Verify patient identities and locked cohort
# ============================================================

cases = pd.read_csv(
    GENE_DIR / "case_order.csv",
    dtype=str,
    keep_default_na=False
)

splits = pd.read_csv(
    SPLIT_PATH,
    dtype=str,
    keep_default_na=False
)

genes = pd.read_csv(
    GENE_DIR / "gene_reference.csv",
    dtype=str,
    keep_default_na=False
)

platform_table = pd.read_csv(
    PLATFORM_PATH,
    dtype=str,
    keep_default_na=False
)

if any(
    frame["case_id"].duplicated().any()
    for frame in (cases, splits, platform_table)
):
    raise RuntimeError(
        "Duplicate patient identifiers in a reference table."
    )

if genes["ensembl_id"].duplicated().any():
    raise RuntimeError("Duplicate gene identifiers.")

lock = cases[
    ["case_id", "histology", "stage_group"]
].merge(
    splits[
        ["case_id", "histology", "stage_group", "split"]
    ],
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_case", "_split")
)

if not (
    lock.histology_case.eq(lock.histology_split).all()
    and lock.stage_group_case.eq(
        lock.stage_group_split
    ).all()
):
    raise RuntimeError(
        "Clinical labels differ from the locked split."
    )

train_global = np.flatnonzero(
    lock["split"].eq("train").to_numpy()
)

val_global = np.flatnonzero(
    lock["split"].eq("validation").to_numpy()
)

test_global = np.flatnonzero(
    lock["split"].eq("test").to_numpy()
)

if (
    len(train_global),
    len(val_global),
    len(test_global)
) != (667, 143, 143):
    raise RuntimeError(
        "Locked patient counts changed."
    )

train_table = (
    lock.iloc[train_global]
    .reset_index(drop=True)[
        ["case_id", "histology_case", "stage_group_case"]
    ]
    .rename(
        columns={
            "histology_case": "histology",
            "stage_group_case": "stage_group"
        }
    )
)

train_table = train_table.merge(
    platform_table[
        [
            "case_id",
            "platform",
            "tissue_source_site",
            "promoter_gene_coverage"
        ]
    ],
    on="case_id",
    how="left",
    validate="one_to_one"
)

if train_table["platform"].isna().any():
    raise RuntimeError(
        "Missing GDC platform metadata for training patients."
    )

platform_str = train_table["platform"].astype(str)

train_table["platform_group"] = np.select(
    [
        platform_str.str.contains(
            "450", case=False, regex=False
        ),
        platform_str.str.contains(
            "27", case=False, regex=False
        )
    ],
    ["HM450", "HM27"],
    default="UNKNOWN"
)

if (
    train_table["platform_group"] == "UNKNOWN"
).any():
    raise RuntimeError(
        "Unrecognized methylation platform. "
        "Review GDC metadata before continuing."
    )

train_table["stage_label"] = (
    train_table["stage_group"]
    .eq("Advanced")
    .astype(np.int32)
)

train_table["histology_label"] = (
    train_table["histology"]
    .eq("LUSC")
    .astype(np.int32)
)

train_table["promoter_gene_coverage"] = (
    pd.to_numeric(
        train_table["promoter_gene_coverage"],
        errors="coerce"
    )
)

if train_table["promoter_gene_coverage"].isna().any():
    raise RuntimeError(
        "Missing Stage 15 methylation coverage data."
    )

# ============================================================
# 3. Load only the 667 training patients' molecular data
# ============================================================

raw = {}

raw_names = {
    "rna": "rna_tpm.npy",
    "cnv": "cnv_absolute.npy",
    "mutation": "mutation_nonsynonymous.npy",
    "methylation": "methylation_promoter_beta.npy"
}

for name, filename in raw_names.items():

    matrix = np.load(
        GENE_DIR / filename,
        mmap_mode="r"
    )

    if matrix.shape != (len(cases), len(genes)):
        raise RuntimeError(
            f"Unexpected {name} matrix dimensions: "
            f"{matrix.shape}"
        )

    raw[name] = np.ascontiguousarray(
        matrix[train_global],
        dtype=np.float32
    )

    del matrix

y = train_table["stage_label"].to_numpy()
h = train_table["histology_label"].to_numpy()
platform = train_table["platform_group"].to_numpy()

print("=" * 73)
print(
    "STAGE 16 - PLATFORM-AWARE "
    "FULLY NESTED TRAINING CROSS-VALIDATION"
)
print("=" * 73)

print(
    f"Training patients={len(y)}; "
    f"validation={len(val_global)} (not loaded); "
    f"sealed test={len(test_global)} (not loaded)"
)

print(
    "Methylation platforms:",
    train_table.platform_group.value_counts().to_dict()
)

print("\nTraining platform and stage distribution:")

print(
    pd.crosstab(
        [
            train_table.histology,
            train_table.stage_group
        ],
        train_table.platform_group
    ).to_string()
)

print("\nMethylation promoter-gene coverage by platform:")

print(
    train_table.groupby(
        "platform_group"
    )["promoter_gene_coverage"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(4)
    .to_string()
)

# ============================================================
# 4. Audit stage-platform association within histology
# ============================================================

platform_audit = []

for hist in ("LUAD", "LUSC"):

    group = train_table[
        train_table.histology.eq(hist)
    ]

    table = pd.crosstab(
        group.stage_group,
        group.platform_group
    ).reindex(
        index=["Early", "Advanced"],
        columns=["HM27", "HM450"],
        fill_value=0
    )

    odds, pvalue = fisher_exact(
        table.to_numpy()
    )

    platform_audit.append({
        "histology": hist,
        "early_HM27": int(
            table.loc["Early", "HM27"]
        ),
        "advanced_HM27": int(
            table.loc["Advanced", "HM27"]
        ),
        "early_HM450": int(
            table.loc["Early", "HM450"]
        ),
        "advanced_HM450": int(
            table.loc["Advanced", "HM450"]
        ),
        "fisher_odds_ratio": float(odds),
        "fisher_p": float(pvalue)
    })

pd.DataFrame(
    platform_audit
).to_csv(
    OUT / "platform_stage_association_training.csv",
    index=False
)

train_table.to_csv(
    OUT / "training_platform_audit.csv",
    index=False
)

print(
    "\nStage-platform association within histology "
    "(exploratory):"
)

print(
    pd.DataFrame(platform_audit)
    .round(4)
    .to_string(index=False)
)

# ============================================================
# 5. Predeclare molecular representations and cohorts
# ============================================================

SPECS = {
    "RNA_CNV": (
        0, 1, 4, 5
    ),
    "RNA_CNV_MUT": (
        0, 1, 2, 4, 5
    ),
    "RNA_CNV_METH": (
        0, 1, 3, 4, 5, 7
    ),
    "FOUR_OMICS": (
        0, 1, 2, 3, 4, 5, 7
    ),
    "RNA_CNV_METH_MASK": (
        0, 1, 4, 5, 7
    )
}

COHORTS = {
    "Pooled": {
        "ids": np.arange(len(y)),
        "models": [
            "RNA_CNV",
            "RNA_CNV_MUT",
            "RNA_CNV_METH",
            "FOUR_OMICS"
        ]
    },
    "HM450Only": {
        "ids": np.flatnonzero(
            platform == "HM450"
        ),
        "models": list(SPECS)
    }
}

# ============================================================
# 6. Select genes using RNA/CNV training variation only
# ============================================================

def masked_sd(values, observed):

    count = observed.sum(
        axis=0
    ).astype(np.float64)

    mean = np.sum(
        np.where(
            observed,
            values,
            0.0
        ),
        axis=0,
        dtype=np.float64
    ) / np.maximum(count, 1)

    second = np.sum(
        np.where(
            observed,
            values * values,
            0.0
        ),
        axis=0,
        dtype=np.float64
    ) / np.maximum(count, 1)

    return np.sqrt(
        np.maximum(
            second - mean * mean,
            0.0
        )
    )


def select_genes(fit_ids):

    rna = raw["rna"][fit_ids]
    cnv = raw["cnv"][fit_ids]

    rna_observed = (
        np.isfinite(rna)
        & (rna >= 0)
    )

    cnv_observed = (
        np.isfinite(cnv)
        & (cnv >= 0)
    )

    rna_values = np.log1p(
        np.where(
            rna_observed,
            rna,
            0.0
        )
    )

    cnv_values = np.clip(
        np.log2(
            (
                np.where(
                    cnv_observed,
                    cnv,
                    0.0
                ) + 0.5
            ) / 2.5
        ),
        -3.0,
        3.0
    )

    rna_variability = masked_sd(
        rna_values,
        rna_observed
    )

    cnv_variability = masked_sd(
        cnv_values,
        cnv_observed
    )

    expressed = np.mean(
        rna_observed & (rna >= 1.0),
        axis=0
    )

    eligible = (
        (rna_observed.mean(axis=0) >= 0.95)
        & (cnv_observed.mean(axis=0) >= 0.75)
        & (expressed >= 0.10)
    )

    if np.count_nonzero(eligible) < GENE_COUNT:

        eligible = (
            (rna_observed.mean(axis=0) >= 0.85)
            & (cnv_observed.mean(axis=0) >= 0.65)
        )

    indices = np.flatnonzero(
        eligible
    )

    if indices.size < GENE_COUNT:
        raise RuntimeError(
            "Insufficient eligible genes within "
            "a nested training fold."
        )

    def rank(values):

        order = np.argsort(
            values,
            kind="stable"
        )

        ranked = np.empty(
            values.size,
            dtype=np.float64
        )

        ranked[order] = (
            np.arange(values.size)
            / max(values.size - 1, 1)
        )

        return ranked

    score = (
        0.55 * rank(
            rna_variability[indices]
        )
        + 0.45 * rank(
            cnv_variability[indices]
        )
    )

    chosen = indices[
        np.argsort(
            -score,
            kind="stable"
        )[:GENE_COUNT]
    ]

    return np.sort(
        chosen
    ).astype(np.int32)


# ============================================================
# 7. Fit modality preprocessing on training folds only
# ============================================================

def prepare_tensors(fit_ids, hold_ids):

    selected = select_genes(
        fit_ids
    )

    xf = np.zeros(
        (
            len(fit_ids),
            len(selected),
            8
        ),
        dtype=np.float32
    )

    xh = np.zeros(
        (
            len(hold_ids),
            len(selected),
            8
        ),
        dtype=np.float32
    )

    transforms = {}

    for (
        name,
        value_channel,
        mask_channel,
        floor
    ) in (
        ("rna", 0, 4, 0.10),
        ("cnv", 1, 5, 0.15),
        ("methylation", 3, 7, 0.04)
    ):

        a = np.asarray(
            raw[name][
                np.ix_(
                    fit_ids,
                    selected
                )
            ],
            dtype=np.float32
        )

        b = np.asarray(
            raw[name][
                np.ix_(
                    hold_ids,
                    selected
                )
            ],
            dtype=np.float32
        )

        if name == "rna":

            oa = (
                np.isfinite(a)
                & (a >= 0)
            )

            ob = (
                np.isfinite(b)
                & (b >= 0)
            )

            a = np.log1p(
                np.where(oa, a, 0.0)
            )

            b = np.log1p(
                np.where(ob, b, 0.0)
            )

        elif name == "cnv":

            oa = (
                np.isfinite(a)
                & (a >= 0)
            )

            ob = (
                np.isfinite(b)
                & (b >= 0)
            )

            a = np.clip(
                np.log2(
                    (
                        np.where(
                            oa,
                            a,
                            0.0
                        ) + 0.5
                    ) / 2.5
                ),
                -3.0,
                3.0
            )

            b = np.clip(
                np.log2(
                    (
                        np.where(
                            ob,
                            b,
                            0.0
                        ) + 0.5
                    ) / 2.5
                ),
                -3.0,
                3.0
            )

        else:

            oa = (
                np.isfinite(a)
                & (a >= 0)
                & (a <= 1)
            )

            ob = (
                np.isfinite(b)
                & (b >= 0)
                & (b <= 1)
            )

        with warnings.catch_warnings():

            warnings.simplefilter(
                "ignore",
                RuntimeWarning
            )

            seen = np.where(
                oa,
                a,
                np.nan
            )

            center = np.nanmedian(
                seen,
                axis=0
            )

            q25, q75 = np.nanpercentile(
                seen,
                [25, 75],
                axis=0
            )

        center = np.where(
            np.isfinite(center),
            center,
            0.0
        ).astype(np.float32)

        scale = np.maximum(
            np.nan_to_num(
                (q75 - q25) / 1.349,
                nan=0.0
            ),
            floor
        ).astype(np.float32)

        for tensor, values, observed in (
            (xf, a, oa),
            (xh, b, ob)
        ):

            tensor[:, :, value_channel] = (
                np.where(
                    observed,
                    np.clip(
                        (
                            np.where(
                                observed,
                                values,
                                center
                            ) - center
                        ) / scale,
                        -5.0,
                        5.0
                    ),
                    0.0
                )
            )

            tensor[:, :, mask_channel] = (
                observed.astype(np.float32)
            )

        transforms[name] = {
            "center": center,
            "scale": scale
        }

    for tensor, ids in (
        (xf, fit_ids),
        (xh, hold_ids)
    ):

        mutation = np.asarray(
            raw["mutation"][
                np.ix_(
                    ids,
                    selected
                )
            ],
            dtype=np.float32
        )

        if (
            not np.isfinite(mutation).all()
            or not np.isin(
                mutation,
                (0, 1)
            ).all()
        ):
            raise RuntimeError(
                "Mutation matrix contains unexpected values."
            )

        tensor[:, :, 2] = mutation
        tensor[:, :, 6] = 1.0

    if (
        not np.isfinite(xf).all()
        or not np.isfinite(xh).all()
    ):
        raise RuntimeError(
            "Non-finite fold-transformed tensor."
        )

    return xf, xh, selected, transforms


# ============================================================
# 8. Fold-specific SVD and latent scaling
# ============================================================

def latent_projection(
    xf,
    xh,
    channels,
    seed
):

    a = np.concatenate(
        [
            xf[:, :, channel]
            for channel in channels
        ],
        axis=1
    )

    b = np.concatenate(
        [
            xh[:, :, channel]
            for channel in channels
        ],
        axis=1
    )

    active = (
        np.var(
            a,
            axis=0
        ) > 1e-9
    )

    if active.sum() < LATENT_COMPONENTS:
        raise RuntimeError(
            "Too few nonconstant features "
            "for latent projection."
        )

    a = a[:, active]
    b = b[:, active]

    components = min(
        LATENT_COMPONENTS,
        a.shape[0] - 2,
        a.shape[1] - 1
    )

    svd = TruncatedSVD(
        n_components=components,
        n_iter=4,
        random_state=seed
    )

    with warnings.catch_warnings():

        warnings.simplefilter(
            "ignore",
            RuntimeWarning
        )

        za = svd.fit_transform(
            a
        )

    zb = svd.transform(
        b
    )

    scaler = StandardScaler()

    za = np.clip(
        scaler.fit_transform(za),
        -7.0,
        7.0
    ).astype(np.float32)

    zb = np.clip(
        scaler.transform(zb),
        -7.0,
        7.0
    ).astype(np.float32)

    if (
        not np.isfinite(za).all()
        or not np.isfinite(zb).all()
    ):
        raise RuntimeError(
            "Invalid latent representation."
        )

    return za, zb, {
        "active": active,
        "svd": svd,
        "scaler": scaler
    }


# ============================================================
# 9. Inner-CV hyperparameter selection
# ============================================================

def classifier(c, weight):

    return LogisticRegression(
        C=c,
        class_weight=weight,
        solver="liblinear",
        max_iter=2000,
        random_state=SEED
    )


def select_hyperparameters(
    inner_representations
):

    best = None

    for c in C_GRID:

        for weight in CLASS_WEIGHTS:

            fold_ap = []
            fold_auc = []

            for (
                fit_ids,
                val_ids,
                zfit,
                zval
            ) in inner_representations:

                model = classifier(
                    c,
                    weight
                )

                model.fit(
                    zfit,
                    y[fit_ids]
                )

                scores = model.predict_proba(
                    zval
                )[:, 1]

                fold_ap.append(
                    average_precision_score(
                        y[val_ids],
                        scores
                    )
                )

                fold_auc.append(
                    roc_auc_score(
                        y[val_ids],
                        scores
                    )
                )

            quality = (
                float(np.mean(fold_ap)),
                float(np.mean(fold_auc))
            )

            if (
                best is None
                or quality > best["quality"]
            ):

                best = {
                    "C": float(c),
                    "weight": weight,
                    "quality": quality
                }

    return best


# ============================================================
# 10. Calculate a training-only data fingerprint
# ============================================================

sha = hashlib.sha256()

sha.update(
    VERSION.encode()
)

sha.update(
    json.dumps(
        {
            "seed": SEED,
            "genes": GENE_COUNT,
            "latent": LATENT_COMPONENTS,
            "outer": OUTER_FOLDS,
            "repeats": OUTER_REPEATS,
            "inner": INNER_FOLDS,
            "c": C_GRID,
            "weights": CLASS_WEIGHTS,
            "specs": SPECS
        },
        sort_keys=True
    ).encode()
)

sha.update(
    "\n".join(
        train_table.case_id.tolist()
    ).encode()
)

sha.update(
    "\n".join(
        genes.ensembl_id.astype(str)
    ).encode()
)

sha.update(
    train_table.platform_group.to_numpy(
        dtype="U5"
    ).tobytes()
)

sha.update(
    y.tobytes()
)

for name in raw_names:

    sha.update(
        np.ascontiguousarray(
            raw[name]
        ).view(np.uint8)
    )

signature = sha.hexdigest()


# ============================================================
# 11. Fully nested, repeated outer cross-validation
# ============================================================

all_predictions = []
all_summaries = []
fold_gene_sets = {}

with threadpool_limits(limits=THREADS):

    for cohort_name, cfg in COHORTS.items():

        cohort_ids = cfg["ids"]

        cohort_strata = (
            2 * y[cohort_ids]
            + h[cohort_ids]
        )

        if len(set(cohort_strata)) != 4:
            raise RuntimeError(
                f"Incomplete stage/histology strata "
                f"in {cohort_name}."
            )

        print("\n" + "=" * 73)
        print(
            f"NESTED CV: {cohort_name}, "
            f"n={len(cohort_ids)}, "
            f"advanced prevalence="
            f"{y[cohort_ids].mean():.3f}"
        )
        print("=" * 73)

        for rep in range(
            OUTER_REPEATS
        ):

            outer_cv = StratifiedKFold(
                n_splits=OUTER_FOLDS,
                shuffle=True,
                random_state=SEED + 1000 * rep
            )

            outer_splits = outer_cv.split(
                cohort_ids,
                cohort_strata
            )

            for fold, (
                outer_fit_local,
                outer_hold_local
            ) in enumerate(
                outer_splits,
                start=1
            ):

                outer_fit = cohort_ids[
                    outer_fit_local
                ]

                outer_hold = cohort_ids[
                    outer_hold_local
                ]

                cache_files = {
                    name: (
                        OUT
                        / f"{cohort_name}_{name}"
                        f"_rep{rep}_fold{fold}.npz"
                    )
                    for name in cfg["models"]
                }

                # Reuse complete matching outer-fold results
                if all(
                    path.exists()
                    for path in cache_files.values()
                ):

                    with np.load(
                        cache_files[
                            cfg["models"][0]
                        ],
                        allow_pickle=False
                    ) as stored:

                        fold_gene_sets[
                            (cohort_name, rep, fold)
                        ] = stored[
                            "selected_genes"
                        ].astype(np.int32)

                    for name, path in cache_files.items():

                        with np.load(
                            path,
                            allow_pickle=False
                        ) as saved:

                            if (
                                str(
                                    saved["signature"].item()
                                ) != signature
                            ):
                                raise RuntimeError(
                                    "Cached outer-fold signature changed."
                                )

                            if not np.array_equal(
                                saved["train_row_ids"],
                                outer_hold
                            ):
                                raise RuntimeError(
                                    "Cached outer-fold membership changed."
                                )

                            for i, probability in zip(
                                outer_hold,
                                saved["probabilities"]
                            ):

                                all_predictions.append({
                                    "cohort": cohort_name,
                                    "model": name,
                                    "repeat": rep,
                                    "fold": fold,
                                    "train_row": int(i),
                                    "case_id": str(
                                        train_table.case_id.iloc[i]
                                    ),
                                    "stage_label": int(y[i]),
                                    "histology_label": int(h[i]),
                                    "platform": str(platform[i]),
                                    "prob_advanced": float(probability)
                                })

                            all_summaries.append({
                                "cohort": cohort_name,
                                "model": name,
                                "repeat": rep,
                                "fold": fold,
                                "outer_AP": float(
                                    saved["outer_ap"].item()
                                ),
                                "outer_AUC": float(
                                    saved["outer_auc"].item()
                                ),
                                "inner_AP_selected": float(
                                    saved["inner_ap"].item()
                                ),
                                "chosen_C": float(
                                    saved["selected_c"].item()
                                ),
                                "chosen_weight": str(
                                    saved["selected_weight"].item()
                                )
                            })

                    print(
                        f"  repeat {rep+1}/{OUTER_REPEATS} "
                        f"fold {fold}: reused",
                        flush=True
                    )

                    continue

                # Inner CV uses only the outer-training patients
                inner_strata = (
                    2 * y[outer_fit]
                    + h[outer_fit]
                )

                inner_cv = StratifiedKFold(
                    n_splits=INNER_FOLDS,
                    shuffle=True,
                    random_state=(
                        SEED + 1000 * rep
                        + fold + 123
                    )
                )

                prepared = []

                for j, (
                    inner_fit_local,
                    inner_hold_local
                ) in enumerate(
                    inner_cv.split(
                        outer_fit,
                        inner_strata
                    )
                ):

                    fit_ids = outer_fit[
                        inner_fit_local
                    ]

                    held_ids = outer_fit[
                        inner_hold_local
                    ]

                    xfit, xhold, _, _ = prepare_tensors(
                        fit_ids,
                        held_ids
                    )

                    prepared.append(
                        (
                            fit_ids,
                            held_ids,
                            xfit,
                            xhold
                        )
                    )

                # Refit preprocessing on the full outer-training fold
                (
                    x_outer_fit,
                    x_outer_hold,
                    selected_outer,
                    _
                ) = prepare_tensors(
                    outer_fit,
                    outer_hold
                )

                fold_gene_sets[
                    (cohort_name, rep, fold)
                ] = selected_outer

                for name in cfg["models"]:

                    cache_path = cache_files[
                        name
                    ]

                    if cache_path.exists():

                        with np.load(
                            cache_path,
                            allow_pickle=False
                        ) as saved:

                            if (
                                str(
                                    saved["signature"].item()
                                ) != signature
                            ):
                                raise RuntimeError(
                                    f"Cached configuration mismatch: "
                                    f"{cache_path}"
                                )

                            if not np.array_equal(
                                saved["train_row_ids"],
                                outer_hold
                            ):
                                raise RuntimeError(
                                    "Cached outer holdout mismatch."
                                )

                            prob = (
                                saved["probabilities"]
                                .astype(float)
                            )

                            inner_ap = float(
                                saved["inner_ap"].item()
                            )

                            c = float(
                                saved["selected_c"].item()
                            )

                            weight = str(
                                saved["selected_weight"].item()
                            )

                            ap = float(
                                saved["outer_ap"].item()
                            )

                            auc = float(
                                saved["outer_auc"].item()
                            )

                    else:

                        representations = []

                        for j, (
                            fit_ids,
                            held_ids,
                            xf,
                            xh
                        ) in enumerate(prepared):

                            a, b, _ = latent_projection(
                                xf,
                                xh,
                                SPECS[name],
                                SEED
                                + 100 * rep
                                + 10 * fold
                                + j
                            )

                            representations.append(
                                (
                                    fit_ids,
                                    held_ids,
                                    a,
                                    b
                                )
                            )

                        chosen = select_hyperparameters(
                            representations
                        )

                        a, b, _ = latent_projection(
                            x_outer_fit,
                            x_outer_hold,
                            SPECS[name],
                            SEED + 100 * rep + fold
                        )

                        model = classifier(
                            chosen["C"],
                            chosen["weight"]
                        )

                        model.fit(
                            a,
                            y[outer_fit]
                        )

                        prob = model.predict_proba(
                            b
                        )[:, 1]

                        ap = float(
                            average_precision_score(
                                y[outer_hold],
                                prob
                            )
                        )

                        auc = float(
                            roc_auc_score(
                                y[outer_hold],
                                prob
                            )
                        )

                        inner_ap = chosen[
                            "quality"
                        ][0]

                        c = chosen["C"]
                        weight = str(
                            chosen["weight"]
                        )

                        temporary = cache_path.with_name(
                            cache_path.name
                            + ".partial.npz"
                        )

                        np.savez_compressed(
                            temporary,
                            signature=signature,
                            train_row_ids=outer_hold,
                            probabilities=prob,
                            outer_ap=ap,
                            outer_auc=auc,
                            inner_ap=inner_ap,
                            selected_c=c,
                            selected_weight=weight,
                            selected_genes=selected_outer
                        )

                        os.replace(
                            temporary,
                            cache_path
                        )

                    for i, probability in zip(
                        outer_hold,
                        prob
                    ):

                        all_predictions.append({
                            "cohort": cohort_name,
                            "model": name,
                            "repeat": rep,
                            "fold": fold,
                            "train_row": int(i),
                            "case_id": str(
                                train_table.case_id.iloc[i]
                            ),
                            "stage_label": int(y[i]),
                            "histology_label": int(h[i]),
                            "platform": str(platform[i]),
                            "prob_advanced": float(probability)
                        })

                    all_summaries.append({
                        "cohort": cohort_name,
                        "model": name,
                        "repeat": rep,
                        "fold": fold,
                        "outer_AP": ap,
                        "outer_AUC": auc,
                        "inner_AP_selected": inner_ap,
                        "chosen_C": c,
                        "chosen_weight": weight
                    })

                    print(
                        f"  {cohort_name:<9} "
                        f"r{rep+1} f{fold} "
                        f"{name:<20} "
                        f"inner AP={inner_ap:.3f} | "
                        f"outer AP={ap:.3f}",
                        flush=True
                    )


# ============================================================
# 12. Verify complete outer-fold predictions
# ============================================================

predictions = pd.DataFrame(
    all_predictions
)

fold_details = pd.DataFrame(
    all_summaries
)

for cohort_name, cfg in COHORTS.items():

    for name in cfg["models"]:

        subset = predictions[
            predictions.cohort.eq(cohort_name)
            & predictions.model.eq(name)
        ]

        for repeat in range(
            OUTER_REPEATS
        ):

            cases_in_repeat = subset[
                subset.repeat.eq(repeat)
            ]

            if (
                len(cases_in_repeat) != len(cfg["ids"])
                or cases_in_repeat.train_row.duplicated().any()
                or set(cases_in_repeat.train_row)
                != set(cfg["ids"])
            ):
                raise RuntimeError(
                    "Outer CV omitted or duplicated patients."
                )

predictions.to_csv(
    OUT
    / "stage16_outer_holdout_predictions_training_only.csv",
    index=False
)

fold_details.to_csv(
    OUT / "stage16_outer_fold_metrics.csv",
    index=False
)


# ============================================================
# 13. Aggregate unbiased-within-protocol outer-fold metrics
# ============================================================

comparisons = []

for cohort_name, cfg in COHORTS.items():

    for name in cfg["models"]:

        folds_here = fold_details[
            fold_details.cohort.eq(cohort_name)
            & fold_details.model.eq(name)
        ]

        prediction_here = predictions[
            predictions.cohort.eq(cohort_name)
            & predictions.model.eq(name)
        ]

        per_repeat = []

        for rep in range(
            OUTER_REPEATS
        ):

            subset = prediction_here[
                prediction_here.repeat.eq(rep)
            ]

            per_repeat.append(
                average_precision_score(
                    subset.stage_label,
                    subset.prob_advanced
                )
            )

        hist_ap = {}

        for title, index in (
            ("LUAD", 0),
            ("LUSC", 1)
        ):

            ap_values = []

            for rep in range(
                OUTER_REPEATS
            ):

                s = prediction_here[
                    prediction_here.repeat.eq(rep)
                    & prediction_here.histology_label.eq(index)
                ]

                ap_values.append(
                    average_precision_score(
                        s.stage_label,
                        s.prob_advanced
                    )
                )

            hist_ap[title] = float(
                np.mean(ap_values)
            )

        comparisons.append({
            "cohort": cohort_name,
            "model": name,
            "patients": len(cfg["ids"]),
            "advanced_prevalence": float(
                y[cfg["ids"]].mean()
            ),
            "outer_mean_AP": float(
                folds_here.outer_AP.mean()
            ),
            "outer_AP_SD": float(
                folds_here.outer_AP.std(
                    ddof=1
                )
            ),
            "outer_mean_AUC": float(
                folds_here.outer_AUC.mean()
            ),
            "repeat_pooled_AP": float(
                np.mean(per_repeat)
            ),
            "LUAD_repeat_AP": hist_ap["LUAD"],
            "LUSC_repeat_AP": hist_ap["LUSC"],
            "selected_C_mode": str(
                Counter(
                    folds_here.chosen_C
                ).most_common(1)[0][0]
            ),
            "selected_weight_mode": str(
                Counter(
                    folds_here.chosen_weight
                ).most_common(1)[0][0]
            )
        })

results = pd.DataFrame(
    comparisons
).sort_values(
    ["cohort", "outer_mean_AP"],
    ascending=[True, False]
)

results.to_csv(
    OUT / "stage16_nested_cohort_comparison.csv",
    index=False
)


# ============================================================
# 14. Exploratory paired molecular contribution analysis
# ============================================================

rng = np.random.default_rng(
    SEED + 1616
)

paired = []

for cohort_name, cfg in COHORTS.items():

    base = predictions[
        predictions.cohort.eq(cohort_name)
        & predictions.model.eq("RNA_CNV")
    ]

    reference = (
        base.groupby(
            "train_row"
        )
        .prob_advanced
        .mean()
        .sort_index()
    )

    ids = reference.index.to_numpy(
        dtype=int
    )

    truth = y[ids]

    subgroup = (
        2 * truth + h[ids]
    )

    for name in cfg["models"]:

        if name == "RNA_CNV":
            continue

        current = predictions[
            predictions.cohort.eq(cohort_name)
            & predictions.model.eq(name)
        ].groupby(
            "train_row"
        )

        scores = (
            current.prob_advanced.mean()
            .reindex(ids)
            .to_numpy()
        )

        if not np.isfinite(scores).all():
            raise RuntimeError(
                "Invalid paired outer-fold predictions."
            )

        delta = float(
            average_precision_score(
                truth,
                scores
            )
            - average_precision_score(
                truth,
                reference.to_numpy()
            )
        )

        bootstrap_values = []

        for _ in range(BOOTSTRAPS):

            sampled = np.concatenate([
                rng.choice(
                    np.flatnonzero(
                        subgroup == group
                    ),
                    size=int(
                        np.sum(
                            subgroup == group
                        )
                    ),
                    replace=True
                )
                for group in (0, 1, 2, 3)
            ])

            bootstrap_values.append(
                float(
                    average_precision_score(
                        truth[sampled],
                        scores[sampled]
                    )
                    - average_precision_score(
                        truth[sampled],
                        reference.to_numpy()[sampled]
                    )
                )
            )

        paired.append({
            "cohort": cohort_name,
            "comparison": (
                f"{name} minus RNA_CNV"
            ),
            "exploratory_mean_repeated_OOF_AP_delta": (
                delta
            ),
            "bootstrap_CI_low": float(
                np.percentile(
                    bootstrap_values,
                    2.5
                )
            ),
            "bootstrap_CI_high": float(
                np.percentile(
                    bootstrap_values,
                    97.5
                )
            )
        })

paired_df = pd.DataFrame(
    paired
)

paired_df.to_csv(
    OUT / "stage16_paired_exploratory_differences.csv",
    index=False
)


# ============================================================
# 15. Save four training-only reference models
# ============================================================

for cohort_name, cfg in COHORTS.items():

    chosen_ids = cfg["ids"]

    for name in (
        "RNA_CNV",
        "FOUR_OMICS"
    ):

        target = (
            MODELS
            / f"{cohort_name}_{name}_training_only.joblib"
        )

        records = fold_details[
            fold_details.cohort.eq(cohort_name)
            & fold_details.model.eq(name)
        ]

        c = float(
            Counter(
                records.chosen_C
            ).most_common(1)[0][0]
        )

        weight_str = Counter(
            records.chosen_weight
        ).most_common(1)[0][0]

        weight = (
            None
            if weight_str == "None"
            else weight_str
        )

        xf, xref, selected, params = (
            prepare_tensors(
                chosen_ids,
                chosen_ids
            )
        )

        zfit, _, fitted = latent_projection(
            xf,
            xref,
            SPECS[name],
            SEED
        )

        model = classifier(
            c,
            weight
        )

        model.fit(
            zfit,
            y[chosen_ids]
        )

        payload = {
            "signature": signature,
            "cohort": cohort_name,
            "model": name,
            "training_case_ids": (
                train_table.case_id.iloc[
                    chosen_ids
                ].tolist()
            ),
            "selected_gene_indices": selected,
            "selected_gene_ids": (
                genes.ensembl_id.iloc[
                    selected
                ].tolist()
            ),
            "preprocessing": params,
            "latent": fitted,
            "classifier": model,
            "channels": SPECS[name],
            "class_threshold_frozen": False,
            "validation_test_evaluated": False
        }

        temporary = target.with_name(
            target.name + ".tmp"
        )

        joblib.dump(
            payload,
            temporary,
            compress=3
        )

        os.replace(
            temporary,
            target
        )


# ============================================================
# 16. Audit outer-fold gene-selection stability
# ============================================================

stability = {}

for cohort_name in COHORTS:

    gene_lists = [
        set(value.tolist())
        for (
            name,
            _,
            _
        ), value in fold_gene_sets.items()
        if name == cohort_name
    ]

    if len(gene_lists) >= 2:

        overlaps = [
            len(left & right)
            / len(left | right)
            for i, left in enumerate(
                gene_lists
            )
            for right in gene_lists[
                i + 1:
            ]
        ]

        stability[cohort_name] = {
            "mean_outer_gene_Jaccard": float(
                np.mean(overlaps)
            ),
            "minimum_outer_gene_Jaccard": float(
                np.min(overlaps)
            )
        }


# ============================================================
# 17. Save reproducibility report
# ============================================================

report = {
    "generated_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "signature": signature,
    "training_patients": len(y),
    "validation_features_loaded": False,
    "test_features_loaded": False,
    "outer_folds": OUTER_FOLDS,
    "outer_repeats": OUTER_REPEATS,
    "inner_folds": INNER_FOLDS,
    "genes_per_outer_training": GENE_COUNT,
    "platform_counts": (
        train_table.platform_group
        .value_counts()
        .to_dict()
    ),
    "outer_gene_stability": stability,
    "cohort_details": {
        name: {
            "patients": len(cfg["ids"]),
            "advanced_prevalence": float(
                y[cfg["ids"]].mean()
            )
        }
        for name, cfg in COHORTS.items()
    },
    "limitations": [
        (
            "HM450-only analysis uses a different sample "
            "mix and is not external validation."
        ),
        (
            "Previous experiments informed the model arms; "
            "comparisons remain exploratory."
        ),
        (
            "Selecting the highest-scoring outer-CV model "
            "would introduce additional selection bias."
        ),
        (
            "Patient-level bootstraps of correlated repeated "
            "OOF predictions are descriptive."
        ),
        (
            "Gene selection uses RNA/CNV variation to isolate "
            "the contribution of additional modalities."
        ),
        (
            "Technical-center and histology confounding "
            "require further investigation."
        )
    ]
}

(
    OUT / "stage16_summary.json"
).write_text(
    json.dumps(
        report,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 18. Final report
# ============================================================

print("\n" + "=" * 73)
print(
    "STAGE 16 FINAL - "
    "FULLY NESTED TRAINING-ONLY RESULTS"
)
print("=" * 73)

print(
    results.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

print("\nPAIRED EXPLORATORY OMICS CONTRIBUTIONS")

print(
    paired_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

print("\nGene selection stability:", stability)

print("\nSaved results:", OUT)

print(
    "Saved training-only reference models:",
    MODELS
)

print(
    "\nValidation features evaluated: NO | "
    "Sealed test features evaluated: NO"
)

print("\nSTAGE 16 COMPLETED")

STAGE 16 - PLATFORM-AWARE FULLY NESTED TRAINING CROSS-VALIDATION
Training patients=667; validation=143 (not loaded); sealed test=143 (not loaded)
Methylation platforms: {'HM450': 540, 'HM27': 127}

Training platform and stage distribution:
platform_group         HM27  HM450
histology stage_group             
LUAD      Advanced        9     64
          Early          33    232
LUSC      Advanced       21     39
          Early          64    205

Methylation promoter-gene coverage by platform:
                count    mean  median     min     max
platform_group                                       
HM27              127  0.0668  0.0673  0.0620  0.0685
HM450             540  0.7529  0.7546  0.7148  0.7568

Stage-platform association within histology (exploratory):
histology  early_HM27  advanced_HM27  early_HM450  advanced_HM450  fisher_odds_ratio  fisher_p
     LUAD          33              9          232              64             1.0115    1.0000
     LUSC          64             2

In [16]:

from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import t as student_t, spearmanr, mannwhitneyu
from statsmodels.stats.multitest import multipletests

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

RAW = ROOT / "data" / "processed" / "gene_level"
SPLIT = ROOT / "data" / "splits" / "locked_four_omics_v1.csv"
CLINICAL = ROOT / "data" / "interim" / "tcga_clinical_stage_audit.csv"
PLATFORM = (
    ROOT / "results" / "metrics" / "stage15"
    / "gdc_methylation_platform_training.csv"
)
OUT = ROOT / "results" / "metrics" / "stage17_root_cause_audit"

OUT.mkdir(parents=True, exist_ok=True)

# Verify clinical labels and locked training indices
cases = pd.read_csv(
    RAW / "case_order.csv",
    dtype=str
).reset_index(drop=True)

genes = pd.read_csv(
    RAW / "gene_reference.csv",
    dtype=str
).reset_index(drop=True)

locked = pd.read_csv(SPLIT, dtype=str)

if (
    not cases.case_id.is_unique
    or not locked.case_id.is_unique
    or not genes.ensembl_id.is_unique
):
    raise RuntimeError("Duplicate patient or gene identifiers.")

labels = cases[
    ["case_id", "histology", "stage_group"]
].merge(
    locked[
        ["case_id", "histology", "stage_group", "split"]
    ],
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_source", "_locked")
)

if (
    not labels.histology_source.equals(
        labels.histology_locked
    )
    or not labels.stage_group_source.equals(
        labels.stage_group_locked
    )
):
    raise RuntimeError("Clinical labels do not match the locked split.")

if len(labels) != 953 or len(genes) != 19944:
    raise RuntimeError("Unexpected Stage 08 cohort dimensions.")

train_index = np.flatnonzero(
    labels.split.eq("train").to_numpy()
)

if len(train_index) != 667:
    raise RuntimeError("Training cohort must contain 667 patients.")

train_labels = labels.iloc[
    train_index
].copy()

if "patient_id" not in cases.columns:
    raise RuntimeError(
        "Stage 08 case_order.csv lacks TCGA patient barcodes."
    )

train_labels["patient_id"] = cases.iloc[
    train_index
].patient_id.to_numpy()

train_labels["site"] = (
    train_labels.patient_id.str.extract(
        r"^TCGA-([A-Za-z0-9]{2})-",
        expand=False
    ).fillna("UNKNOWN")
)

train_labels["advanced"] = (
    train_labels.stage_group_source.eq("Advanced")
).astype(int)

train_labels["histology"] = (
    train_labels.histology_source
)

print("=" * 74)
print("STAGE 17 - TRAINING-ONLY ROOT CAUSE AUDIT")
print("=" * 74)

print(
    "Training patients:",
    len(train_index),
    "| Advanced:",
    int(train_labels.advanced.sum())
)

print("Validation/test molecular features: NOT READ")

# Inspect original clinical stage-source fields
if CLINICAL.exists():

    clinical = pd.read_csv(
        CLINICAL,
        dtype=str,
        keep_default_na=False,
        low_memory=False
    )

    stage_cols = [
        column
        for column in clinical.columns
        if any(
            term in column.lower()
            for term in ("stage", "ajcc", "tnm")
        )
    ]

    print("\nOriginal clinical-stage fields:")
    print(stage_cols[:16])

    subset = None

    training_case_ids = set(
        train_labels.case_id.astype(str)
    )

    training_barcodes = set(
        train_labels.patient_id.astype(str)
    )

    for key, values in (
        ("case_id", training_case_ids),
        ("patient_id", training_barcodes),
        ("submitter_id", training_barcodes)
    ):

        if key in clinical.columns:

            matching = clinical[key].isin(values)

            if int(matching.sum()) >= 50:

                subset = clinical.loc[
                    matching
                ]

                print(
                    "Matched training clinical rows:",
                    len(subset),
                    "| ID field:",
                    key
                )

                break

    if subset is not None:

        for column in stage_cols[:8]:

            counts = (
                subset[column]
                .value_counts(dropna=False)
                .head(8)
                .to_dict()
            )

            print(column, counts)

    else:

        print(
            "Training clinical IDs could not be matched; "
            "stage-source values require manual review."
        )

else:

    print(
        "\nOriginal clinical-stage audit unavailable."
    )

# Join previously retrieved methylation platform metadata
train_labels["platform"] = "UNREPORTED"

if PLATFORM.exists():

    pmeta = pd.read_csv(
        PLATFORM,
        dtype=str,
        keep_default_na=False
    )

    if {"case_id", "platform"}.issubset(
        pmeta.columns
    ):

        pmeta = (
            pmeta[
                ["case_id", "platform"]
            ]
            .drop_duplicates("case_id")
        )

        raw_platform = (
            train_labels[
                ["case_id"]
            ]
            .merge(
                pmeta,
                on="case_id",
                how="left",
                validate="one_to_one"
            )["platform"]
            .fillna("")
        )

        train_labels["platform"] = np.select(
            [
                raw_platform.str.contains(
                    "450", case=False
                ),
                raw_platform.str.contains(
                    "27", case=False
                ),
                raw_platform.str.contains(
                    "EPIC", case=False
                )
            ],
            ["HM450", "HM27", "EPIC"],
            default="UNREPORTED"
        )

print("\nPlatform distribution:")
print(
    train_labels.platform.value_counts().to_string()
)

print("\nPlatform by histology and stage:")
print(
    pd.crosstab(
        [
            train_labels.histology,
            train_labels.stage_group_source
        ],
        train_labels.platform
    ).to_string()
)

# Audit TCGA tissue source sites
site_table = (
    train_labels
    .groupby(
        ["histology", "site"],
        observed=True
    )
    .agg(
        n=("case_id", "size"),
        advanced=("advanced", "sum")
    )
    .reset_index()
)

site_table["advanced_fraction"] = (
    site_table.advanced / site_table.n
)

site_table.to_csv(
    OUT / "tissue_source_site_stage_training.csv",
    index=False
)

print("\nTissue source sites with >=10 training patients:")

print(
    site_table.loc[
        site_table.n >= 10
    ]
    .sort_values(
        "n",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

# Quantify promoter-gene availability by platform
meth_source = np.load(
    RAW / "methylation_promoter_beta.npy",
    mmap_mode="r"
)

if meth_source.shape != (953, len(genes)):
    raise RuntimeError("Unexpected methylation matrix shape.")

meth_raw = np.asarray(
    meth_source[train_index],
    dtype=np.float32
)

meth_valid = (
    np.isfinite(meth_raw)
    & (meth_raw >= 0)
    & (meth_raw <= 1)
)

train_labels["methylation_coverage"] = (
    meth_valid.mean(axis=1)
)

print("\nPromoter-gene coverage by platform:")

print(
    train_labels.groupby(
        "platform"
    ).methylation_coverage.agg(
        ["count", "mean", "median", "min", "max"]
    ).round(4).to_string()
)

coverage_stats = {}

for platform in ("HM27", "HM450"):

    indices = np.flatnonzero(
        train_labels.platform.eq(
            platform
        ).to_numpy()
    )

    if len(indices) >= 10:

        fractions = meth_valid[
            indices
        ].mean(axis=0)

        coverage_stats[platform] = {
            "patients": len(indices),
            "genes_80pct": int(
                (fractions >= 0.80).sum()
            ),
            "genes_95pct": int(
                (fractions >= 0.95).sum()
            )
        }

if all(
    name in coverage_stats
    for name in ("HM27", "HM450")
):

    hm27_coverage = meth_valid[
        train_labels.platform.eq(
            "HM27"
        ).to_numpy()
    ].mean(axis=0)

    hm450_coverage = meth_valid[
        train_labels.platform.eq(
            "HM450"
        ).to_numpy()
    ].mean(axis=0)

    coverage_stats[
        "shared_genes_80pct_both"
    ] = int(
        (
            (hm27_coverage >= 0.80)
            & (hm450_coverage >= 0.80)
        ).sum()
    )

    coverage_stats[
        "shared_genes_95pct_both"
    ] = int(
        (
            (hm27_coverage >= 0.95)
            & (hm450_coverage >= 0.95)
        ).sum()
    )

print("\nCross-platform comparable gene counts:")
print(coverage_stats)

# Compare stage-associated molecular effects across histologies
y = train_labels.advanced.to_numpy(
    dtype=int
)

h = train_labels.histology.eq(
    "LUSC"
).to_numpy(dtype=int)


def group_effects(
    matrix,
    row_filter,
    required_fraction
):

    effects = []
    pvalues = []
    retained = []

    for hist in (0, 1):

        early = matrix[
            row_filter
            & (h == hist)
            & (y == 0)
        ]

        advanced = matrix[
            row_filter
            & (h == hist)
            & (y == 1)
        ]

        if (
            len(early) < 10
            or len(advanced) < 10
        ):
            return None

        with warnings.catch_warnings():

            warnings.simplefilter(
                "ignore",
                RuntimeWarning
            )

            n0 = np.isfinite(
                early
            ).sum(axis=0)

            n1 = np.isfinite(
                advanced
            ).sum(axis=0)

            mu0 = np.nanmean(
                early, axis=0
            )

            mu1 = np.nanmean(
                advanced, axis=0
            )

            v0 = np.nanvar(
                early, axis=0, ddof=1
            )

            v1 = np.nanvar(
                advanced, axis=0, ddof=1
            )

        eligible = (
            (n0 >= max(
                10,
                int(np.ceil(
                    len(early) * required_fraction
                ))
            ))
            & (n1 >= max(
                10,
                int(np.ceil(
                    len(advanced) * required_fraction
                ))
            ))
        )

        pooled = np.sqrt(
            np.maximum(
                (v0 + v1) / 2.0,
                0
            )
        )

        effect = np.divide(
            mu1 - mu0,
            pooled,
            out=np.full_like(
                mu0, np.nan
            ),
            where=pooled > 1e-8
        )

        se2 = (
            v0 / np.maximum(n0, 1)
            + v1 / np.maximum(n1, 1)
        )

        t_stat = np.divide(
            mu1 - mu0,
            np.sqrt(se2),
            out=np.zeros_like(mu0),
            where=se2 > 1e-12
        )

        df = se2 ** 2 / (
            v0 ** 2 / (
                np.maximum(n0 - 1, 1)
                * np.maximum(n0, 1) ** 2
            )
            + v1 ** 2 / (
                np.maximum(n1 - 1, 1)
                * np.maximum(n1, 1) ** 2
            )
            + 1e-30
        )

        p = 2 * student_t.sf(
            np.abs(t_stat),
            df=np.maximum(df, 1)
        )

        good = (
            eligible
            & np.isfinite(effect)
            & np.isfinite(p)
        )

        effect[~good] = np.nan
        p[~good] = np.nan

        effects.append(effect)
        pvalues.append(p)
        retained.append(good)

    return effects, pvalues, retained


def summarize_effects(
    modality,
    matrix,
    row_filter,
    minimum_fraction
):

    result = group_effects(
        matrix,
        row_filter,
        minimum_fraction
    )

    if result is None:

        print(
            modality,
            ": insufficient stage/histology samples"
        )

        return {
            "modality": modality,
            "skipped": "insufficient samples"
        }

    effects, pvalues, good = result

    qvalues = []

    for j in range(2):

        current = np.full(
            len(genes),
            np.nan
        )

        current[
            good[j]
        ] = multipletests(
            pvalues[j][good[j]],
            method="fdr_bh"
        )[1]

        qvalues.append(current)

    both = (
        good[0] & good[1]
    )

    if both.sum() > 100:

        correlation = float(
            spearmanr(
                effects[0][both],
                effects[1][both]
            ).statistic
        )

    else:
        correlation = float("nan")

    valid0 = np.flatnonzero(
        np.isfinite(
            effects[0]
        )
    )

    valid1 = np.flatnonzero(
        np.isfinite(
            effects[1]
        )
    )

    top_luad = valid0[
        np.argsort(
            -np.abs(
                effects[0][valid0]
            ),
            kind="stable"
        )[:100]
    ]

    top_lusc = valid1[
        np.argsort(
            -np.abs(
                effects[1][valid1]
            ),
            kind="stable"
        )[:100]
    ]

    shared = np.intersect1d(
        top_luad,
        top_lusc
    )

    concordant = int(
        (
            np.sign(
                effects[0][shared]
            )
            == np.sign(
                effects[1][shared]
            )
        ).sum()
    )

    gene_results = pd.DataFrame({
        "ensembl_id": genes.ensembl_id,
        "gene_symbol": genes.gene_symbol,
        "LUAD_stage_effect_d": effects[0],
        "LUSC_stage_effect_d": effects[1],
        "LUAD_stage_q": qvalues[0],
        "LUSC_stage_q": qvalues[1]
    })

    gene_results.to_csv(
        OUT / f"{modality}_stage_gene_effects_train.csv",
        index=False
    )

    answer = {
        "modality": modality,
        "genes_tested_both": int(
            both.sum()
        ),
        "LUAD_FDR_below_0_05": int(
            np.nansum(
                qvalues[0] < 0.05
            )
        ),
        "LUSC_FDR_below_0_05": int(
            np.nansum(
                qvalues[1] < 0.05
            )
        ),
        "FDR_significant_both": int(
            (
                (qvalues[0] < 0.05)
                & (qvalues[1] < 0.05)
            ).sum()
        ),
        "cross_histology_effect_spearman": correlation,
        "common_top100_effect_genes": int(
            len(shared)
        ),
        "common_top100_same_direction": concordant
    }

    print("\n", modality, answer)

    return answer


# RNA stage effects
rna_source = np.load(
    RAW / "rna_tpm.npy",
    mmap_mode="r"
)

rna_raw = np.asarray(
    rna_source[train_index],
    dtype=np.float32
)

rna = np.where(
    np.isfinite(rna_raw)
    & (rna_raw >= 0),
    np.log1p(
        np.maximum(
            rna_raw, 0
        )
    ),
    np.nan
)

del rna_raw

rna_report = summarize_effects(
    "rna",
    rna,
    np.ones(len(y), dtype=bool),
    0.95
)

del rna, rna_source

# CNV stage effects
cnv_source = np.load(
    RAW / "cnv_absolute.npy",
    mmap_mode="r"
)

cnv_raw = np.asarray(
    cnv_source[train_index],
    dtype=np.float32
)

cnv = np.where(
    np.isfinite(cnv_raw)
    & (cnv_raw >= 0),
    np.log2(
        (
            np.maximum(
                cnv_raw, 0
            ) + 0.5
        ) / 2.5
    ),
    np.nan
)

del cnv_raw

cnv_report = summarize_effects(
    "cnv",
    cnv,
    np.ones(len(y), dtype=bool),
    0.85
)

del cnv, cnv_source

# Methylation stage effects, restricted to HM450
meth_report = summarize_effects(
    "methylation_HM450",
    np.where(
        meth_valid,
        meth_raw,
        np.nan
    ),
    train_labels.platform.eq(
        "HM450"
    ).to_numpy(),
    0.70
)

del meth_raw, meth_valid

# Mutation-burden audit
mutation_source = np.load(
    RAW / "mutation_nonsynonymous.npy",
    mmap_mode="r"
)

mutation_burden = np.asarray(
    mutation_source[
        train_index
    ]
).sum(axis=1)

mutation_report = []

for histology, hist_value in (
    ("LUAD", 0),
    ("LUSC", 1)
):

    early = mutation_burden[
        (h == hist_value)
        & (y == 0)
    ]

    advanced = mutation_burden[
        (h == hist_value)
        & (y == 1)
    ]

    row = {
        "histology": histology,
        "early_median_mutated_genes": float(
            np.median(early)
        ),
        "advanced_median_mutated_genes": float(
            np.median(advanced)
        ),
        "mann_whitney_p": float(
            mannwhitneyu(
                early,
                advanced
            ).pvalue
        )
    }

    mutation_report.append(row)

print("\nMutation burden by histology:")
print(pd.DataFrame(mutation_report).to_string(index=False))

# Save reproducible diagnostic summary
report = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "training_cases": len(train_index),
    "advanced_cases": int(y.sum()),
    "validation_and_test_features_loaded": False,
    "methylation_cross_platform_coverage": coverage_stats,
    "rna_stage_effects": rna_report,
    "cnv_stage_effects": cnv_report,
    "methylation_HM450_stage_effects": meth_report,
    "mutation_burden": mutation_report,
    "caveat": (
        "Exploratory training-only associations; "
        "not independent validation or causal evidence. "
        "Clinical stage provenance requires separate review."
    )
}

(
    OUT / "stage17_diagnostic_summary.json"
).write_text(
    json.dumps(
        report,
        indent=2
    ),
    encoding="utf-8"
)

train_labels[
    [
        "case_id",
        "histology",
        "stage_group_source",
        "site",
        "platform",
        "methylation_coverage"
    ]
].to_csv(
    OUT / "training_sample_technical_audit.csv",
    index=False
)

print("\n" + "=" * 74)
print("STAGE 17 FINAL DIAGNOSTIC SUMMARY")
print("=" * 74)

print(
    "Methylation platform gene coverage:",
    coverage_stats
)

print(
    "RNA cross-histology stage effect:",
    rna_report.get(
        "cross_histology_effect_spearman"
    )
)

print(
    "CNV cross-histology stage effect:",
    cnv_report.get(
        "cross_histology_effect_spearman"
    )
)

print(
    "Methylation HM450 cross-histology stage effect:",
    meth_report.get(
        "cross_histology_effect_spearman"
    )
)

print("Saved diagnostic reports:", OUT)
print("TRAINING ONLY; VALIDATION/TEST NOT READ")

STAGE 17 - TRAINING-ONLY ROOT CAUSE AUDIT
Training patients: 667 | Advanced: 133
Validation/test molecular features: NOT READ

Original clinical-stage fields:
['stage_group', 'stage_source', 'stage_original', 'stage_label']
Matched training clinical rows: 667 | ID field: case_id
stage_group {'Early': 534, 'Advanced': 133}
stage_source {'ajcc_pathologic_stage': 667}
stage_original {'Stage IB': 190, 'Stage IA': 148, 'Stage IIB': 106, 'Stage IIIA': 90, 'Stage IIA': 83, 'Stage IV': 23, 'Stage IIIB': 18, 'Stage I': 5}
stage_label {'0.0': 534, '1.0': 133}

Platform distribution:
platform
HM450    540
HM27     127

Platform by histology and stage:
platform                      HM27  HM450
histology stage_group_source             
LUAD      Advanced               9     64
          Early                 33    232
LUSC      Advanced              21     39
          Early                 64    205

Tissue source sites with >=10 training patients:
histology site  n  advanced  advanced_fraction
  

In [17]:

import os
import sys
import hashlib
import json
from pathlib import Path
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency
from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold
)
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import average_precision_score, roc_auc_score
from threadpoolctl import threadpool_limits

# ============================================================
# 1. Configuration
# ============================================================

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

RAW = ROOT / "data" / "processed" / "gene_level"
SPLIT_PATH = (
    ROOT / "data" / "splits" / "locked_four_omics_v1.csv"
)
OUTPUT = ROOT / "results" / "metrics" / "stage18"
CHECKPOINT = ROOT / "models" / "checkpoints" / "stage18"

OUTPUT.mkdir(parents=True, exist_ok=True)
CHECKPOINT.mkdir(parents=True, exist_ok=True)

SEED = 42

# Fixed using the previous training-CV evidence
FIXED_C = 0.03

K_RNA = 160
K_CNV = 80

SVD_RNA = 600
SVD_CNV = 300
SVD_DIM = 48

PATIENT_REPEATS = 2
N_SPLITS = 4
THREADS = min(4, os.cpu_count() or 1)

VERSION = "stage18_site_aware_histology_specific_v1"

# ============================================================
# 2. Verify original cohort and locked splits
# ============================================================

cases = pd.read_csv(
    RAW / "case_order.csv",
    dtype=str,
    keep_default_na=False
)

reference = pd.read_csv(
    RAW / "gene_reference.csv",
    dtype=str,
    keep_default_na=False
)

locked = pd.read_csv(
    SPLIT_PATH,
    dtype=str,
    keep_default_na=False
)

if (
    not cases.case_id.is_unique
    or not locked.case_id.is_unique
    or not reference.ensembl_id.is_unique
):
    raise RuntimeError(
        "Duplicate patient or gene identifiers."
    )

check = cases[
    ["case_id", "histology", "stage_group"]
].merge(
    locked[
        ["case_id", "histology", "stage_group", "split"]
    ],
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_a", "_b")
)

if (
    not check.histology_a.equals(check.histology_b)
    or not check.stage_group_a.equals(
        check.stage_group_b
    )
):
    raise RuntimeError(
        "Clinical labels disagree with the locked split."
    )

tr = np.flatnonzero(
    check.split.eq("train").to_numpy()
)

va = np.flatnonzero(
    check.split.eq("validation").to_numpy()
)

te = np.flatnonzero(
    check.split.eq("test").to_numpy()
)

if (
    (len(tr), len(va), len(te)) != (667, 143, 143)
    or len(reference) != 19944
):
    raise RuntimeError(
        "Unexpected Stage 08 source cohort."
    )

y = (
    check.stage_group_a.iloc[tr]
    .eq("Advanced")
    .to_numpy(dtype=np.int8)
)

h = (
    check.histology_a.iloc[tr]
    .eq("LUSC")
    .to_numpy(dtype=np.int8)
)

patient_barcodes = (
    cases.patient_id.iloc[tr]
    .astype(str)
    .to_numpy()
)

site = (
    pd.Series(patient_barcodes)
    .str.extract(
        r"^TCGA-([A-Za-z0-9]{2})-",
        expand=False
    )
    .to_numpy()
)

if pd.isna(site).any():
    raise RuntimeError(
        "Invalid TCGA participant barcode."
    )

site = site.astype(str)
n = len(tr)

print("=" * 72)
print(
    "STAGE 18 - SITE-ROBUST, HISTOLOGY-SPECIFIC "
    "RNA/CNV MODELING"
)
print("=" * 72)

print(
    "Training:",
    n,
    "| Advanced:",
    int(y.sum())
)

print(
    "Validation/test untouched:",
    len(va),
    len(te)
)

print(
    "Distinct tissue-source sites:",
    len(set(site))
)

# ============================================================
# 3. Load original training molecular measurements
# ============================================================

rna_path = RAW / "rna_tpm.npy"
cnv_path = RAW / "cnv_absolute.npy"

rna_source = np.load(
    rna_path,
    mmap_mode="r"
)

cnv_source = np.load(
    cnv_path,
    mmap_mode="r"
)

if not (
    rna_source.shape
    == cnv_source.shape
    == (953, len(reference))
):
    raise RuntimeError(
        "Unexpected molecular matrix dimensions."
    )

# Only the 667 training rows are materialized
rna = np.asarray(
    rna_source[tr],
    dtype=np.float32
)

cnv = np.asarray(
    cnv_source[tr],
    dtype=np.float32
)

rna = np.where(
    np.isfinite(rna) & (rna >= 0),
    np.log1p(np.maximum(rna, 0)),
    np.nan
).astype(np.float32)

cnv = np.where(
    np.isfinite(cnv) & (cnv >= 0),
    np.clip(
        np.log2(
            (np.maximum(cnv, 0) + 0.5) / 2.5
        ),
        -3,
        3
    ),
    np.nan
).astype(np.float32)

del rna_source, cnv_source

# ============================================================
# 4. Test tissue-source-site association with stage
# ============================================================

site_rows = []

for code, label in (
    (0, "LUAD"),
    (1, "LUSC")
):

    take = np.flatnonzero(
        h == code
    )

    site_names, site_indices = np.unique(
        site[take],
        return_inverse=True
    )

    table = np.zeros(
        (len(site_names), 2),
        dtype=np.int32
    )

    np.add.at(
        table,
        (site_indices, y[take]),
        1
    )

    observed_chi = chi2_contingency(
        table,
        correction=False
    )[0]

    rng = np.random.default_rng(
        SEED + 200 + code
    )

    exceed = 0

    # Permutation test handles sparse site categories
    for _ in range(399):

        permuted = rng.permutation(
            y[take]
        )

        shuffled = np.zeros_like(
            table
        )

        np.add.at(
            shuffled,
            (site_indices, permuted),
            1
        )

        simulated_chi = chi2_contingency(
            shuffled,
            correction=False
        )[0]

        exceed += simulated_chi >= observed_chi

    site_rows.append({
        "histology": label,
        "patients": len(take),
        "sites": len(site_names),
        "chi_square": float(observed_chi),
        "permutation_p": float(
            (exceed + 1) / 400
        )
    })

site_audit = pd.DataFrame(
    site_rows
)

site_audit.to_csv(
    OUTPUT / "training_site_stage_association.csv",
    index=False
)

print(
    "\nTissue-source-site association with stage:"
)

print(
    site_audit.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

# ============================================================
# 5. Build patient-wise and site-disjoint folds
# ============================================================

strata = 2 * y + h

folds = []

for repetition in range(
    PATIENT_REPEATS
):

    splitter = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=SEED + 101 * repetition
    )

    for fold, (fit, hold) in enumerate(
        splitter.split(
            np.arange(n),
            strata
        ),
        start=1
    ):

        folds.append({
            "regime": "patient",
            "repeat": repetition + 1,
            "fold": fold,
            "fit": fit,
            "hold": hold
        })

# Choose a label-balanced group split without examining model scores
site_choices = []

candidate_seeds = (
    42, 49, 61, 73, 89, 97,
    109, 127, 151, 167, 181, 191
)

for seed in candidate_seeds:

    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    trial = list(
        splitter.split(
            np.arange(n),
            strata,
            groups=site
        )
    )

    acceptable = True
    imbalance = 0.0

    for fit, hold in trial:

        if (
            set(np.unique(strata[fit]))
            != {0, 1, 2, 3}
            or set(np.unique(strata[hold]))
            != {0, 1, 2, 3}
        ):
            acceptable = False
            break

        if set(site[fit]) & set(site[hold]):
            raise RuntimeError(
                "Tissue-source-site overlap detected."
            )

        imbalance += (
            abs(len(hold) / n - 0.25)
            + abs(y[hold].mean() - y.mean())
        )

    if acceptable:
        site_choices.append(
            (imbalance, seed, trial)
        )

if not site_choices:
    raise RuntimeError(
        "Could not construct four-stratum site-disjoint folds."
    )

_, group_seed, site_trial = min(
    site_choices,
    key=lambda item: (
        item[0],
        item[1]
    )
)

for fold, (fit, hold) in enumerate(
    site_trial,
    start=1
):

    folds.append({
        "regime": "site",
        "repeat": 1,
        "fold": fold,
        "fit": fit,
        "hold": hold
    })

print(
    "\nSite-held-out fold seed:",
    group_seed
)

print(
    "Tissue-source-site overlap: NONE"
)

print(
    "Patient-level folds:",
    PATIENT_REPEATS * N_SPLITS
)

print(
    "Site-held-out folds:",
    N_SPLITS
)

# ============================================================
# 6. Predeclare model configurations
# ============================================================

SPECS = {
    "Pooled_Variance_SVD": {
        "pool": "pooled",
        "selection": "variance",
        "rna": SVD_RNA,
        "cnv": SVD_CNV,
        "svd": SVD_DIM
    },

    "Pooled_SiteAdjusted_RNA_CNV": {
        "pool": "pooled",
        "selection": "site",
        "rna": K_RNA,
        "cnv": K_CNV
    },

    "Pooled_SiteAdjusted_Interaction": {
        "pool": "pooled",
        "selection": "site",
        "rna": K_RNA,
        "cnv": K_CNV,
        "interaction": True
    },

    "Separate_RNA_SiteAdjusted": {
        "pool": "separate",
        "selection": "site",
        "rna": K_RNA,
        "cnv": 0
    },

    "Separate_RNA_CNV_NoSiteAdjustment": {
        "pool": "separate",
        "selection": "stage",
        "rna": K_RNA,
        "cnv": K_CNV
    },

    "Separate_RNA_CNV_SiteAdjusted": {
        "pool": "separate",
        "selection": "site",
        "rna": K_RNA,
        "cnv": K_CNV
    },

    "Histology_TSS_Only_Control": {
        "pool": "control"
    }
}

# ============================================================
# 7. Fit fold-specific feature statistics
# ============================================================

def stat_block(matrix):

    finite = np.isfinite(
        matrix
    )

    counts = finite.sum(
        axis=0
    )

    clean = np.where(
        finite,
        matrix,
        0
    ).astype(np.float32)

    mean = (
        clean.sum(
            axis=0,
            dtype=np.float64
        )
        / np.maximum(counts, 1)
    )

    squared_sum = (
        clean.astype(np.float64) ** 2
    ).sum(axis=0)

    standard_deviation = np.sqrt(
        np.maximum(
            squared_sum / np.maximum(counts, 1)
            - mean ** 2,
            0
        )
    )

    scale = np.maximum(
        standard_deviation,
        0.08
    )

    standardized = np.clip(
        (
            np.where(
                finite,
                matrix,
                mean[None, :]
            )
            - mean[None, :]
        ) / scale[None, :],
        -5,
        5
    ).astype(np.float32)

    return {
        "mean": mean.astype(np.float32),
        "scale": scale.astype(np.float32),
        "z": standardized,
        "sd": standard_deviation,
        "coverage": counts / len(matrix)
    }


# ============================================================
# 8. Compute within-site stage association
# ============================================================

def within_group_correlation(
    z,
    yfit,
    group_ids
):

    numerator = np.zeros(
        z.shape[1],
        dtype=np.float64
    )

    sum_square_x = np.zeros(
        z.shape[1],
        dtype=np.float64
    )

    sum_square_y = 0.0

    for group in np.unique(
        group_ids
    ):

        ix = np.flatnonzero(
            group_ids == group
        )

        if len(ix) < 2:
            continue

        yc = (
            yfit[ix].astype(np.float64)
            - float(yfit[ix].mean())
        )

        if not np.any(yc):
            continue

        xx = z[ix].astype(np.float64)

        xx -= xx.mean(
            axis=0,
            keepdims=True
        )

        numerator += yc @ xx

        sum_square_x += (
            xx * xx
        ).sum(axis=0)

        sum_square_y += float(
            yc @ yc
        )

    if sum_square_y < 3.0:
        raise RuntimeError(
            "Insufficient within-site stage variation "
            "for site-adjusted ranking."
        )

    return (
        np.abs(numerator)
        / np.sqrt(
            np.maximum(
                sum_square_x * sum_square_y,
                1e-15
            )
        )
    )


# ============================================================
# 9. Prepare training-only statistics for each fitting group
# ============================================================

def get_prepared(
    fit_ids,
    cache
):

    key = fit_ids.tobytes()

    if key not in cache:

        r = stat_block(
            rna[fit_ids]
        )

        c = stat_block(
            cnv[fit_ids]
        )

        expression_fraction = np.mean(
            rna[fit_ids] >= np.log(2),
            axis=0
        )

        cache[key] = (
            r,
            c,
            expression_fraction
        )

    return cache[key]


# ============================================================
# 10. Select genes using training-only information
# ============================================================

def select_indices(
    prepared,
    fit_ids,
    spec
):

    r, c, expressed = prepared

    eligible_rna = (
        (r["coverage"] >= 0.95)
        & (expressed >= 0.10)
        & (r["sd"] >= 0.02)
    )

    eligible_cnv = (
        (c["coverage"] >= 0.85)
        & (c["sd"] >= 0.02)
    )

    mode = spec["selection"]

    if mode == "variance":

        scores_r = r["sd"]
        scores_c = c["sd"]

    else:

        if mode == "site":

            if spec["pool"] == "separate":

                groups = site[
                    fit_ids
                ]

            else:

                groups = np.array([
                    f"{hist}:{tss}"
                    for hist, tss in zip(
                        h[fit_ids],
                        site[fit_ids]
                    )
                ])

        elif spec["pool"] == "separate":

            groups = np.zeros(
                len(fit_ids),
                dtype=int
            )

        else:

            groups = h[
                fit_ids
            ]

        scores_r = within_group_correlation(
            r["z"],
            y[fit_ids],
            groups
        )

        scores_c = within_group_correlation(
            c["z"],
            y[fit_ids],
            groups
        )

    def pick(scores, valid, count):

        if count == 0:
            return np.empty(
                0,
                dtype=int
            )

        if valid.sum() < count:
            raise RuntimeError(
                "Too few reliable genes for "
                f"requested selection size {count}."
            )

        ranked = np.argsort(
            -np.where(
                valid,
                scores,
                -np.inf
            ),
            kind="stable"
        )

        return np.sort(
            ranked[:count]
        )

    selected_rna = pick(
        scores_r,
        eligible_rna,
        spec["rna"]
    )

    selected_cnv = pick(
        scores_c,
        eligible_cnv,
        spec["cnv"]
    )

    return (
        selected_rna,
        selected_cnv
    )


# ============================================================
# 11. Create processed features using fitted parameters
# ============================================================

def features_for(
    prepared,
    ids,
    selected_rna,
    selected_cnv
):

    r, c, _ = prepared

    parts = []

    for matrix, statistics, chosen in (
        (rna, r, selected_rna),
        (cnv, c, selected_cnv)
    ):

        if not len(chosen):
            continue

        raw = np.asarray(
            matrix[
                np.ix_(ids, chosen)
            ],
            dtype=np.float32
        )

        filled = np.where(
            np.isfinite(raw),
            raw,
            statistics["mean"][chosen][None, :]
        )

        scaled = np.clip(
            (
                filled - statistics["mean"][chosen]
            ) / statistics["scale"][chosen],
            -5,
            5
        )

        parts.append(
            scaled
        )

    return np.concatenate(
        parts,
        axis=1
    ).astype(np.float32)


# ============================================================
# 12. Fit a molecular classifier
# ============================================================

def fit_one(
    fit_ids,
    spec,
    cache
):

    prepared = get_prepared(
        fit_ids,
        cache
    )

    selected_rna, selected_cnv = (
        select_indices(
            prepared,
            fit_ids,
            spec
        )
    )

    z = features_for(
        prepared,
        fit_ids,
        selected_rna,
        selected_cnv
    )

    projection = None
    latent_scaler = None

    if "svd" in spec:

        projection = TruncatedSVD(
            n_components=spec["svd"],
            n_iter=4,
            random_state=SEED
        )

        z = projection.fit_transform(
            z
        )

        latent_scaler = StandardScaler()

        z = latent_scaler.fit_transform(
            z
        )

    # Allow different stage coefficients by histology
    if spec.get("interaction"):

        sign = (
            2 * h[fit_ids] - 1
        ).astype(np.float32)[:, None]

        z = np.column_stack([
            z,
            0.5 * z * sign,
            sign
        ])

    elif spec["pool"] == "pooled":

        z = np.column_stack([
            z,
            2 * h[fit_ids] - 1
        ])

    classifier = LogisticRegression(
        C=FIXED_C,
        solver="liblinear",
        class_weight=None,
        max_iter=2000,
        random_state=SEED
    )

    classifier.fit(
        z,
        y[fit_ids]
    )

    r, c, _ = prepared

    return {
        "rna_gene_indices": selected_rna,
        "cnv_gene_indices": selected_cnv,
        "rna_mean": r["mean"][selected_rna],
        "rna_scale": r["scale"][selected_rna],
        "cnv_mean": c["mean"][selected_cnv],
        "cnv_scale": c["scale"][selected_cnv],
        "svd": projection,
        "latent_scaler": latent_scaler,
        "classifier": classifier
    }


# ============================================================
# 13. Predict with frozen preprocessing and classifier
# ============================================================

def predict_one(
    fitted_model,
    hold_ids,
    spec
):

    def transform(
        matrix,
        selected,
        mean,
        scale
    ):

        if not len(selected):
            return np.empty(
                (len(hold_ids), 0),
                dtype=np.float32
            )

        raw = np.asarray(
            matrix[
                np.ix_(
                    hold_ids,
                    selected
                )
            ],
            dtype=np.float32
        )

        filled = np.where(
            np.isfinite(raw),
            raw,
            mean[None, :]
        )

        return np.clip(
            (
                filled - mean
            ) / scale,
            -5,
            5
        ).astype(np.float32)

    parts = [
        transform(
            rna,
            fitted_model["rna_gene_indices"],
            fitted_model["rna_mean"],
            fitted_model["rna_scale"]
        ),

        transform(
            cnv,
            fitted_model["cnv_gene_indices"],
            fitted_model["cnv_mean"],
            fitted_model["cnv_scale"]
        )
    ]

    z = np.concatenate(
        parts,
        axis=1
    )

    if fitted_model["svd"] is not None:

        z = fitted_model["svd"].transform(
            z
        )

        z = fitted_model[
            "latent_scaler"
        ].transform(z)

    if spec.get("interaction"):

        sign = (
            2 * h[hold_ids] - 1
        ).astype(np.float32)[:, None]

        z = np.column_stack([
            z,
            0.5 * z * sign,
            sign
        ])

    elif spec["pool"] == "pooled":

        z = np.column_stack([
            z,
            2 * h[hold_ids] - 1
        ])

    return fitted_model[
        "classifier"
    ].predict_proba(z)[:, 1]


# ============================================================
# 14. Evaluate pooled and histology-specific strategies
# ============================================================

def fit_predict(
    fit,
    hold,
    spec,
    cache
):

    if spec["pool"] == "control":

        fit_categories = np.column_stack([
            h[fit].astype(str),
            site[fit]
        ])

        hold_categories = np.column_stack([
            h[hold].astype(str),
            site[hold]
        ])

        encoder = OneHotEncoder(
            handle_unknown="ignore"
        )

        a = encoder.fit_transform(
            fit_categories
        )

        b = encoder.transform(
            hold_categories
        )

        classifier = LogisticRegression(
            C=FIXED_C,
            solver="liblinear",
            max_iter=2000,
            random_state=SEED
        )

        classifier.fit(
            a,
            y[fit]
        )

        return (
            classifier.predict_proba(b)[:, 1],
            []
        )

    if spec["pool"] == "pooled":

        model = fit_one(
            fit,
            spec,
            cache
        )

        probabilities = predict_one(
            model,
            hold,
            spec
        )

        selections = [
            (
                "Pooled",
                model["rna_gene_indices"],
                model["cnv_gene_indices"]
            )
        ]

        return probabilities, selections

    probabilities = np.full(
        len(hold),
        np.nan
    )

    selections = []

    for code, label in (
        (0, "LUAD"),
        (1, "LUSC")
    ):

        hist_fit = fit[
            h[fit] == code
        ]

        hist_hold_positions = np.flatnonzero(
            h[hold] == code
        )

        if len(hist_hold_positions) == 0:
            continue

        if len(np.unique(y[hist_fit])) != 2:
            raise RuntimeError(
                "Histology-specific training fold "
                "contains only one stage class."
            )

        model = fit_one(
            hist_fit,
            spec,
            cache
        )

        probabilities[
            hist_hold_positions
        ] = predict_one(
            model,
            hold[hist_hold_positions],
            spec
        )

        selections.append(
            (
                label,
                model["rna_gene_indices"],
                model["cnv_gene_indices"]
            )
        )

    if not np.isfinite(
        probabilities
    ).all():
        raise RuntimeError(
            "Incomplete histology-specific predictions."
        )

    return probabilities, selections


# ============================================================
# 15. Create experiment fingerprint for resumable folds
# ============================================================

input_paths = (
    rna_path,
    cnv_path,
    RAW / "case_order.csv",
    RAW / "gene_reference.csv",
    SPLIT_PATH
)

fingerprint = hashlib.sha256(
    json.dumps(
        {
            "version": VERSION,
            "seed": SEED,
            "C": FIXED_C,
            "folds": N_SPLITS,
            "repeats": PATIENT_REPEATS,
            "specs": SPECS,
            "training_cases": (
                cases.case_id.iloc[tr].tolist()
            ),
            "stages": y.tolist(),
            "histology": h.tolist(),
            "sites": site.tolist(),
            "input_stats": {
                str(path): (
                    path.stat().st_size,
                    path.stat().st_mtime_ns
                )
                for path in input_paths
            }
        },
        sort_keys=True
    ).encode()
).hexdigest()

score_rows = []
gene_rows = []

OOF = {}

for regime in (
    "patient",
    "site"
):

    repetitions = (
        PATIENT_REPEATS
        if regime == "patient"
        else 1
    )

    for name in SPECS:

        OOF[(regime, name)] = np.full(
            (repetitions, n),
            np.nan,
            dtype=np.float64
        )


# ============================================================
# 16. Run the two cross-validation designs
# ============================================================

with threadpool_limits(
    limits=THREADS
):

    for task in folds:

        regime = task["regime"]
        repetition = task["repeat"]
        fold = task["fold"]

        fit = task["fit"]
        hold = task["hold"]

        checkpoint_file = (
            OUTPUT
            / f"_checkpoint_{regime}_r{repetition}_f{fold}.joblib"
        )

        if checkpoint_file.exists():

            cached = joblib.load(
                checkpoint_file
            )

            if (
                cached["signature"] != fingerprint
                or not np.array_equal(
                    cached["hold"],
                    hold
                )
                or set(cached["predictions"])
                != set(SPECS)
            ):
                raise RuntimeError(
                    f"Incompatible checkpoint: {checkpoint_file}"
                )

            current_scores = cached["scores"]
            current_genes = cached["genes"]
            current_predictions = cached[
                "predictions"
            ]

            print(
                f"{regime.upper()} "
                f"r{repetition} f{fold}: "
                "verified checkpoint reused",
                flush=True
            )

        else:

            cache = {}

            current_scores = []
            current_genes = []
            current_predictions = {}

            print(
                f"\n{regime.upper()} "
                f"repeat {repetition} fold {fold} | "
                f"train={len(fit)} | "
                f"held={len(hold)} | "
                f"advanced={int(y[hold].sum())} | "
                f"sites={len(set(site[hold]))}",
                flush=True
            )

            for name, spec in SPECS.items():

                p_hold, selected = fit_predict(
                    fit,
                    hold,
                    spec,
                    cache
                )

                if np.unique(
                    y[hold]
                ).size < 2:
                    raise RuntimeError(
                        "Held-out fold contains one stage class."
                    )

                row = {
                    "regime": regime,
                    "repeat": repetition,
                    "fold": fold,
                    "model": name,
                    "n_train": len(fit),
                    "n_held": len(hold),
                    "n_advanced": int(y[hold].sum()),
                    "held_prevalence": float(
                        y[hold].mean()
                    ),
                    "AP": float(
                        average_precision_score(
                            y[hold],
                            p_hold
                        )
                    ),
                    "ROC_AUC": float(
                        roc_auc_score(
                            y[hold],
                            p_hold
                        )
                    )
                }

                row["AP_above_prevalence"] = (
                    row["AP"]
                    - row["held_prevalence"]
                )

                current_scores.append(
                    row
                )

                current_predictions[name] = (
                    np.asarray(
                        p_hold,
                        dtype=np.float64
                    )
                )

                for histology, r_idx, c_idx in selected:

                    for modality, indices in (
                        ("RNA", r_idx),
                        ("CNV", c_idx)
                    ):

                        for index in indices:

                            current_genes.append({
                                "regime": regime,
                                "repeat": repetition,
                                "fold": fold,
                                "model": name,
                                "histology": histology,
                                "modality": modality,
                                "ensembl_id": str(
                                    reference.ensembl_id.iloc[index]
                                ),
                                "gene_symbol": str(
                                    reference.gene_symbol.iloc[index]
                                )
                            })

                print(
                    f"  {name:<42} "
                    f"AP={row['AP']:.4f} "
                    f"AUC={row['ROC_AUC']:.4f}",
                    flush=True
                )

            temporary = checkpoint_file.with_name(
                checkpoint_file.name + ".tmp"
            )

            joblib.dump(
                {
                    "signature": fingerprint,
                    "hold": hold,
                    "scores": current_scores,
                    "genes": current_genes,
                    "predictions": current_predictions
                },
                temporary,
                compress=3
            )

            os.replace(
                temporary,
                checkpoint_file
            )

            del cache

        score_rows.extend(
            current_scores
        )

        gene_rows.extend(
            current_genes
        )

        for name, probabilities in (
            current_predictions.items()
        ):

            OOF[
                (regime, name)
            ][repetition - 1, hold] = probabilities


# ============================================================
# 17. Save fold-level metrics and selected genes
# ============================================================

scores = pd.DataFrame(
    score_rows
)

scores.to_csv(
    OUTPUT / "stage18_fold_level_metrics.csv",
    index=False
)

gene_table = pd.DataFrame(
    gene_rows
)

gene_table.to_csv(
    OUTPUT / "stage18_fold_selected_genes.csv",
    index=False
)

# ============================================================
# 18. Evaluate gene-selection stability
# ============================================================

stability_rows = []

group_columns = [
    "regime",
    "model",
    "histology",
    "modality"
]

for (
    regime,
    name,
    histology,
    modality
), subset in gene_table.groupby(
    group_columns,
    sort=True
):

    selected_sets = [
        set(part.ensembl_id)
        for _, part in subset.groupby(
            ["repeat", "fold"]
        )
    ]

    overlaps = [
        len(a & b) / len(a | b)
        for i, a in enumerate(
            selected_sets
        )
        for b in selected_sets[i + 1:]
        if len(a | b)
    ]

    stability_rows.append({
        "regime": regime,
        "model": name,
        "histology": histology,
        "modality": modality,
        "n_fold_lists": len(selected_sets),
        "mean_pairwise_gene_Jaccard": (
            float(np.mean(overlaps))
            if overlaps
            else float("nan")
        ),
        "minimum_gene_Jaccard": (
            float(np.min(overlaps))
            if overlaps
            else float("nan")
        )
    })

stability_table = pd.DataFrame(
    stability_rows
)

stability_table.to_csv(
    OUTPUT / "stage18_gene_selection_stability.csv",
    index=False
)


# ============================================================
# 19. Summarize patient and site CV performance
# ============================================================

summary_rows = []

for regime in (
    "patient",
    "site"
):

    for name in SPECS:

        piece = scores[
            scores.regime.eq(regime)
            & scores.model.eq(name)
        ]

        probabilities = OOF[
            (regime, name)
        ]

        if not np.isfinite(
            probabilities
        ).all():
            raise RuntimeError(
                "Incomplete out-of-fold prediction matrix."
            )

        pooled_ap = []
        pooled_auc = []

        for repeat_probabilities in probabilities:

            pooled_ap.append(
                average_precision_score(
                    y,
                    repeat_probabilities
                )
            )

            pooled_auc.append(
                roc_auc_score(
                    y,
                    repeat_probabilities
                )
            )

        mean_probabilities = (
            probabilities.mean(axis=0)
        )

        summary_rows.append({
            "regime": regime,
            "model": name,
            "fold_mean_AP": float(
                piece.AP.mean()
            ),
            "fold_AP_SD": float(
                piece.AP.std(ddof=1)
            ),
            "fold_mean_AP_gain": float(
                piece.AP_above_prevalence.mean()
            ),
            "fold_mean_AUC": float(
                piece.ROC_AUC.mean()
            ),
            "pooled_OOF_AP": float(
                np.mean(pooled_ap)
            ),
            "pooled_OOF_AUC": float(
                np.mean(pooled_auc)
            ),
            "LUAD_OOF_AP": float(
                average_precision_score(
                    y[h == 0],
                    mean_probabilities[h == 0]
                )
            ),
            "LUSC_OOF_AP": float(
                average_precision_score(
                    y[h == 1],
                    mean_probabilities[h == 1]
                )
            )
        })

        np.save(
            OUTPUT
            / f"{regime}_{name}_training_only_OOF.npy",
            probabilities
        )

summary_table = pd.DataFrame(
    summary_rows
).sort_values(
    ["regime", "fold_mean_AP_gain"],
    ascending=[True, False]
)

summary_table.to_csv(
    OUTPUT / "stage18_patient_vs_site_cv.csv",
    index=False
)


# ============================================================
# 20. Select using site-held-out training CV
# ============================================================

molecular = summary_table[
    summary_table.regime.eq("site")
    & ~summary_table.model.eq(
        "Histology_TSS_Only_Control"
    )
]

selected_name = str(
    molecular.sort_values(
        [
            "fold_mean_AP_gain",
            "pooled_OOF_AP"
        ],
        ascending=False
    ).iloc[0].model
)

# ============================================================
# 21. Refit selected model on all training patients
# ============================================================

all_train = np.arange(n)

selected_spec = SPECS[
    selected_name
]

final_cache = {}

if selected_spec["pool"] == "pooled":

    final_models = {
        "Pooled": fit_one(
            all_train,
            selected_spec,
            final_cache
        )
    }

else:

    final_models = {}

    for code, label in (
        (0, "LUAD"),
        (1, "LUSC")
    ):

        ids = all_train[
            h == code
        ]

        final_models[label] = fit_one(
            ids,
            selected_spec,
            final_cache
        )

final_model_path = (
    CHECKPOINT
    / "stage18_selected_training_model.joblib"
)

joblib.dump(
    {
        "version": VERSION,
        "model_name": selected_name,
        "fixed_C": FIXED_C,
        "gene_reference": str(
            RAW / "gene_reference.csv"
        ),
        "training_patient_ids": (
            cases.case_id.iloc[tr].tolist()
        ),
        "spec": selected_spec,
        "models": final_models,
        "fit_on_training_only": True
    },
    final_model_path,
    compress=3
)


# ============================================================
# 22. Save reproducibility and limitation report
# ============================================================

report = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "stage": VERSION,
    "train_n": len(tr),
    "validation_n_not_loaded": len(va),
    "test_n_not_loaded": len(te),
    "train_advanced": int(y.sum()),
    "unique_sites": len(set(site)),
    "site_split_seed": group_seed,
    "selected_model_by_site_disjoint_training_CV": (
        selected_name
    ),
    "selection_criterion": (
        "site_fold_mean_AP_minus_fold_prevalence"
    ),
    "site_association": site_rows,
    "strict_patient_train_only": True,
    "important_caveats": [
        (
            "Selecting among model families using site-CV "
            "makes the winning site-CV score optimistic."
        ),
        (
            "Site-disjoint CV is an internal transfer "
            "stress test, not external validation."
        ),
        (
            "Stage16 training-CV results informed the "
            "fixed regularization and model choices."
        ),
        (
            "Biomarker claims require independent "
            "replication and clinical-covariate audits."
        )
    ]
}

(
    OUTPUT / "stage18_summary.json"
).write_text(
    json.dumps(
        report,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 23. Final diagnostic comparison
# ============================================================

print("\n" + "=" * 72)
print(
    "STAGE 18 FINAL - "
    "PATIENT-WISE VS SITE-HELD-OUT RESULTS"
)
print("=" * 72)

print(
    summary_table.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

print(
    "\nSite-held-out gene selection stability:"
)

print(
    stability_table[
        stability_table.regime.eq("site")
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.3f}"
    )
)

print(
    "\nSite-held-out CV selected molecular model:",
    selected_name
)

print(
    "Saved training-only model:",
    final_model_path
)

print(
    "Untouched validation/test molecular features: NOT LOADED"
)

print("\nSTAGE 18 COMPLETED")

STAGE 18 - SITE-ROBUST, HISTOLOGY-SPECIFIC RNA/CNV MODELING
Training: 667 | Advanced: 133
Validation/test untouched: 143 143
Distinct tissue-source sites: 63

Tissue-source-site association with stage:
histology  patients  sites  chi_square  permutation_p
     LUAD       338     31     52.8188         0.0125
     LUSC       329     32     59.9128         0.0050

Site-held-out fold seed: 42
Tissue-source-site overlap: NONE
Patient-level folds: 8
Site-held-out folds: 4

PATIENT repeat 1 fold 1 | train=500 | held=167 | advanced=33 | sites=48
  Pooled_Variance_SVD                        AP=0.2452 AUC=0.5787
  Pooled_SiteAdjusted_RNA_CNV                AP=0.2710 AUC=0.6269
  Pooled_SiteAdjusted_Interaction            AP=0.2452 AUC=0.5972
  Separate_RNA_SiteAdjusted                  AP=0.3761 AUC=0.6454
  Separate_RNA_CNV_NoSiteAdjustment          AP=0.3865 AUC=0.6036
  Separate_RNA_CNV_SiteAdjusted              AP=0.3672 AUC=0.6420
  Histology_TSS_Only_Control                 AP=0.2607 AUC=

In [18]:

import hashlib
import json
import os
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests
from scipy import sparse
from scipy.stats import rankdata
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits

# ============================================================
# 1. Project configuration
# ============================================================

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

RAW = ROOT / "data" / "processed" / "gene_level"
SPLIT_FILE = (
    ROOT / "data" / "splits" / "locked_four_omics_v1.csv"
)
REFERENCE_DIR = ROOT / "data" / "raw" / "references"
OUT = ROOT / "results" / "metrics" / "stage19"
MODELS = ROOT / "models" / "checkpoints" / "stage19"

for directory in (REFERENCE_DIR, OUT, MODELS):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
FOLDS = 4
N_BOOT = 500
THREADS = min(4, os.cpu_count() or 1)

VERSION = "stage19_reactome_site_robust_v1"
REACTOME_URL = (
    "https://reactome.org/download/current/"
    "ReactomePathways.gmt.zip"
)
ARCHIVE = REFERENCE_DIR / "ReactomePathways.gmt.zip"

# ============================================================
# 2. Verify the original training cohort
# ============================================================

cases = pd.read_csv(
    RAW / "case_order.csv",
    dtype=str,
    keep_default_na=False
)

genes = pd.read_csv(
    RAW / "gene_reference.csv",
    dtype=str,
    keep_default_na=False
)

locked = pd.read_csv(
    SPLIT_FILE,
    dtype=str,
    keep_default_na=False
)

if (
    not cases.case_id.is_unique
    or not genes.ensembl_id.is_unique
    or not locked.case_id.is_unique
):
    raise RuntimeError("Duplicate patient or gene identifiers.")

merged = cases[
    ["case_id", "patient_id", "histology", "stage_group"]
].merge(
    locked[
        ["case_id", "histology", "stage_group", "split"]
    ],
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_raw", "_locked")
)

if (
    not merged.histology_raw.equals(
        merged.histology_locked
    )
    or not merged.stage_group_raw.equals(
        merged.stage_group_locked
    )
):
    raise RuntimeError(
        "Locked patient labels or order have changed."
    )

train_ids = np.flatnonzero(
    merged.split.eq("train")
)

val_ids = np.flatnonzero(
    merged.split.eq("validation")
)

test_ids = np.flatnonzero(
    merged.split.eq("test")
)

if (
    len(train_ids),
    len(val_ids),
    len(test_ids)
) != (667, 143, 143):
    raise RuntimeError("Unexpected patient split sizes.")

if len(genes) != 19944:
    raise RuntimeError("Unexpected gene reference size.")

labels = merged.iloc[
    train_ids
].reset_index(drop=True)

y = labels.stage_group_raw.eq(
    "Advanced"
).to_numpy(dtype=np.int64)

h = labels.histology_raw.eq(
    "LUSC"
).to_numpy(dtype=np.int64)

site = labels.patient_id.str.extract(
    r"^TCGA-([A-Za-z0-9]{2})-",
    expand=False
)

if site.isna().any():
    raise RuntimeError("Missing tissue-source-site code.")

site = site.to_numpy(dtype=str)
strata = 2 * y + h

if set(strata) != {0, 1, 2, 3}:
    raise RuntimeError(
        "A stage-histology stratum is missing."
    )

print("=" * 73)
print(
    "STAGE 19 - REACTOME SINGLE-SAMPLE "
    "BIOLOGICAL REPRESENTATION"
)
print("=" * 73)

print(
    f"Training={len(y)} | Advanced={y.sum()} | "
    f"Validation={len(val_ids)} untouched | "
    f"Test={len(test_ids)} untouched"
)

print(
    "Distinct tissue-source sites:",
    len(np.unique(site))
)

# ============================================================
# 3. Download and verify Reactome gene sets
# ============================================================

if (
    not ARCHIVE.is_file()
    or not zipfile.is_zipfile(ARCHIVE)
):

    print("\nDownloading the official Reactome pathway library...")

    response = requests.get(
        REACTOME_URL,
        timeout=(25, 120)
    )

    response.raise_for_status()

    partial = ARCHIVE.with_name(
        ARCHIVE.name + ".part"
    )

    partial.write_bytes(
        response.content
    )

    if not zipfile.is_zipfile(partial):
        raise RuntimeError(
            "Reactome pathway download is not a valid ZIP."
        )

    os.replace(
        partial,
        ARCHIVE
    )

with zipfile.ZipFile(ARCHIVE) as archive:

    if archive.testzip() is not None:
        raise RuntimeError(
            "Reactome pathway ZIP integrity check failed."
        )

    names = [
        filename
        for filename in archive.namelist()
        if filename.lower().endswith(".gmt")
    ]

    if not names:
        raise RuntimeError(
            f"No GMT file in Reactome archive: {archive.namelist()}"
        )

    gmt_data = archive.read(
        names[0]
    ).decode(
        "utf-8-sig",
        errors="replace"
    )

# ============================================================
# 4. Map Reactome pathway genes to TCGA Ensembl genes
# ============================================================

symbol_series = (
    genes.gene_symbol
    .astype(str)
    .str.upper()
    .str.strip()
)

unique_symbols = ~symbol_series.duplicated(
    keep=False
)

symbol_map = {
    symbol: i
    for i, symbol in enumerate(symbol_series)
    if unique_symbols.iat[i]
    and symbol not in ("", "NAN")
}

ensembl_map = {
    str(gene_id).upper().split(".")[0]: i
    for i, gene_id in enumerate(
        genes.ensembl_id
    )
}

pathways = []

for line in gmt_data.splitlines():

    fields = line.rstrip(
        "\r\n"
    ).split("\t")

    if len(fields) < 8:
        continue

    # Reactome GMT may have a third metadata column
    start = (
        3
        if "reactome pathway" in fields[2].lower()
        else 2
    )

    matched = set()

    for item in fields[start:]:

        token = item.strip().upper()

        if token in symbol_map:
            matched.add(
                symbol_map[token]
            )

        elif token in ensembl_map:
            matched.add(
                ensembl_map[token]
            )

    if 10 <= len(matched) <= 300:

        pathways.append(
            (
                fields[0],
                fields[1],
                tuple(sorted(matched))
            )
        )

# Remove duplicated gene-set memberships
deduplicated = {}

for name, identifier, members in pathways:

    if members not in deduplicated:
        deduplicated[members] = (
            name,
            identifier,
            members
        )

pathways = list(
    deduplicated.values()
)

if len(pathways) < 100:
    raise RuntimeError(
        f"Only {len(pathways)} pathways mapped. "
        "Check Reactome GMT format and gene identifiers."
    )

rows = []
columns = []
weights = []
metadata = []

for j, (name, identifier, members) in enumerate(
    pathways
):

    for gene_index in members:

        rows.append(j)
        columns.append(gene_index)
        weights.append(
            1.0 / len(members)
        )

    metadata.append({
        "pathway_index": j,
        "pathway_name": name,
        "reactome_id": identifier,
        "mapped_genes": len(members)
    })

W = sparse.csr_matrix(
    (
        np.asarray(
            weights,
            dtype=np.float32
        ),
        (rows, columns)
    ),
    shape=(
        len(pathways),
        len(genes)
    ),
    dtype=np.float32
)

if not np.allclose(
    np.asarray(
        W.sum(axis=1)
    ).ravel(),
    1.0,
    atol=1e-5
):
    raise RuntimeError(
        "Pathway gene-membership weights are invalid."
    )

sparse.save_npz(
    OUT / "reactome_gene_pathway_weights.npz",
    W
)

pd.DataFrame(
    metadata
).to_csv(
    OUT / "reactome_pathway_reference.csv",
    index=False
)

print(
    f"\nMapped unique Reactome pathways: "
    f"{len(pathways)}"
)

print(
    f"Unambiguous gene symbols: "
    f"{len(symbol_map)}"
)

# ============================================================
# 5. Load original molecular measurements: train only
# ============================================================

rna_file = RAW / "rna_tpm.npy"
cnv_file = RAW / "cnv_absolute.npy"

rna_source = np.load(
    rna_file,
    mmap_mode="r"
)

cnv_source = np.load(
    cnv_file,
    mmap_mode="r"
)

expected_shape = (
    len(cases),
    len(genes)
)

if (
    rna_source.shape != expected_shape
    or cnv_source.shape != expected_shape
):
    raise RuntimeError(
        "Raw RNA and CNV matrix dimensions are inconsistent."
    )

rna = np.asarray(
    rna_source[train_ids],
    dtype=np.float32
).copy()

cnv = np.asarray(
    cnv_source[train_ids],
    dtype=np.float32
).copy()

del rna_source, cnv_source

# ============================================================
# 6. RNA single-sample pathway scoring
# ============================================================

rna_ok = (
    np.isfinite(rna)
    & (rna >= 0)
)

if rna_ok.mean() < 0.95:
    raise RuntimeError(
        "Unexpectedly low RNA feature completeness."
    )

rna = np.where(
    rna_ok,
    np.log1p(
        np.maximum(
            rna,
            0.0
        )
    ),
    0.0
).astype(np.float32)

# Average ranks handle tied zero-expression genes
ranks = rankdata(
    rna,
    axis=1,
    method="average"
).astype(np.float32)

ranks = (
    ranks - (len(genes) + 1) / 2
) / len(genes)

RNA_PATH = np.asarray(
    (W @ ranks.T).T,
    dtype=np.float32
)

del rna, rna_ok, ranks

print(
    "Single-sample RNA pathway representation:",
    RNA_PATH.shape
)

# ============================================================
# 7. CNV signed and absolute pathway-burden features
# ============================================================

cnv_ok = (
    np.isfinite(cnv)
    & (cnv >= 0)
)

cnv_log = np.clip(
    np.log2(
        (
            np.where(
                cnv_ok,
                cnv,
                2.0
            ) + 0.5
        ) / 2.5
    ),
    -3,
    3
).astype(np.float32)

cnv_valid = cnv_ok.astype(
    np.float32
)

cnv_coverage = np.asarray(
    (W @ cnv_valid.T).T,
    dtype=np.float32
)

CNV_SIGNED = np.asarray(
    (
        W @ (
            cnv_log * cnv_valid
        ).T
    ).T,
    dtype=np.float32
)

CNV_ABSOLUTE = np.asarray(
    (
        W @ (
            np.abs(cnv_log) * cnv_valid
        ).T
    ).T,
    dtype=np.float32
)

CNV_SIGNED = np.divide(
    CNV_SIGNED,
    cnv_coverage,
    out=np.zeros_like(
        CNV_SIGNED
    ),
    where=cnv_coverage >= 0.7
)

CNV_ABSOLUTE = np.divide(
    CNV_ABSOLUTE,
    cnv_coverage,
    out=np.zeros_like(
        CNV_ABSOLUTE
    ),
    where=cnv_coverage >= 0.7
)

del cnv, cnv_ok, cnv_log, cnv_valid, cnv_coverage

if (
    not np.isfinite(RNA_PATH).all()
    or not np.isfinite(CNV_SIGNED).all()
    or not np.isfinite(CNV_ABSOLUTE).all()
    or np.std(RNA_PATH) < 1e-5
):
    raise RuntimeError(
        "Pathway feature matrices are invalid."
    )

FEATURES = {
    "rna": RNA_PATH,
    "rna_cnv": np.column_stack(
        [
            RNA_PATH,
            CNV_SIGNED,
            CNV_ABSOLUTE
        ]
    ).astype(np.float32)
}

print(
    "RNA + CNV pathway representation:",
    FEATURES["rna_cnv"].shape
)

# ============================================================
# 8. Prespecified predictive models
# ============================================================

EXPERIMENTS = [
    {
        "name": "RNA_pathway_L2",
        "view": "rna",
        "model": "logit",
        "c": 0.1,
        "site_weight": False,
        "interaction": False
    },
    {
        "name": "RNA_pathway_L2_sitebalanced",
        "view": "rna",
        "model": "logit",
        "c": 0.1,
        "site_weight": True,
        "interaction": False
    },
    {
        "name": "RNA_pathway_histology_interaction",
        "view": "rna",
        "model": "logit",
        "c": 0.03,
        "site_weight": True,
        "interaction": True
    },
    {
        "name": "RNA_CNV_pathway_L2",
        "view": "rna_cnv",
        "model": "logit",
        "c": 0.1,
        "site_weight": False,
        "interaction": False
    },
    {
        "name": "RNA_CNV_pathway_sitebalanced",
        "view": "rna_cnv",
        "model": "logit",
        "c": 0.1,
        "site_weight": True,
        "interaction": False
    },
    {
        "name": "RNA_pathway_RBF_SVM",
        "view": "rna",
        "model": "svm",
        "c": 0.5,
        "site_weight": True,
        "interaction": False
    },
    {
        "name": "RNA_pathway_gradient_boosting",
        "view": "rna",
        "model": "hgb",
        "c": None,
        "site_weight": True,
        "interaction": False
    },
    {
        "name": "RNA_CNV_pathway_gradient_boosting",
        "view": "rna_cnv",
        "model": "hgb",
        "c": None,
        "site_weight": True,
        "interaction": False
    }
]

# ============================================================
# 9. Training-only site-balanced sample weights
# ============================================================

def make_site_weight(fit):

    _, inverse, counts = np.unique(
        site[fit],
        return_inverse=True,
        return_counts=True
    )

    weight = 1.0 / np.sqrt(
        counts[inverse].astype(float)
    )

    weight = np.minimum(
        weight,
        np.median(weight) * 3.0
    )

    return (
        weight / weight.mean()
    ).astype(np.float64)


# ============================================================
# 10. Fit a model and its training-only transformations
# ============================================================

def fit_model(config, fit):

    z = FEATURES[
        config["view"]
    ][fit]

    scaler = StandardScaler()

    z = np.clip(
        scaler.fit_transform(z),
        -7.0,
        7.0
    ).astype(np.float32)

    projection = None

    if config["model"] == "hgb":

        dimension = min(
            48,
            z.shape[0] - 2,
            z.shape[1] - 1
        )

        projection = TruncatedSVD(
            n_components=dimension,
            n_iter=3,
            random_state=SEED
        )

        z = projection.fit_transform(z)

    if config["interaction"]:

        hist = (
            h[fit] * 2 - 1
        ).astype(
            np.float32
        ).reshape(-1, 1)

        z = np.column_stack(
            [
                z,
                hist,
                0.5 * z * hist
            ]
        ).astype(np.float32)

    elif config["model"] == "logit":

        z = np.column_stack(
            [
                z,
                2 * h[fit] - 1
            ]
        ).astype(np.float32)

    if config["model"] == "hgb":

        estimator = HistGradientBoostingClassifier(
            max_iter=130,
            learning_rate=0.04,
            max_leaf_nodes=7,
            min_samples_leaf=18,
            l2_regularization=3.0,
            early_stopping=False,
            random_state=SEED
        )

    elif config["model"] == "svm":

        estimator = SVC(
            C=config["c"],
            kernel="rbf",
            gamma="scale",
            class_weight="balanced",
            random_state=SEED
        )

    else:

        estimator = LogisticRegression(
            C=config["c"],
            solver="liblinear",
            class_weight="balanced",
            max_iter=2500,
            random_state=SEED
        )

    weights = (
        make_site_weight(fit)
        if config["site_weight"]
        else None
    )

    estimator.fit(
        z,
        y[fit],
        sample_weight=weights
    )

    return {
        "config": config,
        "scaler": scaler,
        "projection": projection,
        "estimator": estimator
    }


# ============================================================
# 11. Predict using frozen transformations
# ============================================================

def predict_model(fitted, hold):

    config = fitted["config"]

    z = FEATURES[
        config["view"]
    ][hold]

    z = np.clip(
        fitted["scaler"].transform(z),
        -7.0,
        7.0
    ).astype(np.float32)

    if fitted["projection"] is not None:

        z = fitted["projection"].transform(z)

    if config["interaction"]:

        hist = (
            h[hold] * 2 - 1
        ).astype(
            np.float32
        ).reshape(-1, 1)

        z = np.column_stack(
            [
                z,
                hist,
                0.5 * z * hist
            ]
        ).astype(np.float32)

    elif config["model"] == "logit":

        z = np.column_stack(
            [
                z,
                2 * h[hold] - 1
            ]
        ).astype(np.float32)

    if config["model"] == "svm":

        # SVM decision scores are not calibrated probabilities
        return fitted[
            "estimator"
        ].decision_function(
            z
        ).astype(float)

    return fitted[
        "estimator"
    ].predict_proba(
        z
    )[:, 1].astype(float)


# ============================================================
# 12. Define site-held-out and patient-wise CV
# ============================================================

site_cv = list(
    StratifiedGroupKFold(
        n_splits=FOLDS,
        shuffle=True,
        random_state=SEED
    ).split(
        np.arange(len(y)),
        strata,
        groups=site
    )
)

patient_cv = list(
    StratifiedKFold(
        n_splits=FOLDS,
        shuffle=True,
        random_state=SEED
    ).split(
        np.arange(len(y)),
        strata
    )
)

for fit, hold in site_cv:

    if set(site[fit]) & set(site[hold]):
        raise RuntimeError(
            "Tissue-source-site overlap in site CV."
        )

    if (
        set(strata[fit]) != {0, 1, 2, 3}
        or set(strata[hold]) != {0, 1, 2, 3}
    ):
        raise RuntimeError(
            "A site CV fold lacks a stage-histology group."
        )

print("\nSite-held-out CV folds:", len(site_cv))
print("Patient-wise CV folds:", len(patient_cv))
print("Site overlap in site-held-out CV: NONE")

# ============================================================
# 13. Create reproducibility signature
# ============================================================

signature_data = {
    "version": VERSION,
    "case_ids": labels.case_id.tolist(),
    "stage": y.tolist(),
    "histology": h.tolist(),
    "sites": site.tolist(),
    "gene_ref_sha256": hashlib.sha256(
        (
            RAW / "gene_reference.csv"
        ).read_bytes()
    ).hexdigest(),
    "reactome_sha256": hashlib.sha256(
        ARCHIVE.read_bytes()
    ).hexdigest(),
    "RNA_size_mtime": (
        rna_file.stat().st_size,
        rna_file.stat().st_mtime_ns
    ),
    "CNV_size_mtime": (
        cnv_file.stat().st_size,
        cnv_file.stat().st_mtime_ns
    ),
    "experiments": EXPERIMENTS
}

signature = hashlib.sha256(
    json.dumps(
        signature_data,
        sort_keys=True
    ).encode()
).hexdigest()

fold_rows = []
predictions = {}

# ============================================================
# 14. Run all model comparisons with resumable checkpoints
# ============================================================

with threadpool_limits(
    limits=THREADS
):

    for regime, folds in (
        ("site", site_cv),
        ("patient", patient_cv)
    ):

        for config in EXPERIMENTS:

            predictions[
                (regime, config["name"])
            ] = np.full(
                len(y),
                np.nan,
                dtype=float
            )

        for fold_number, (
            fit,
            hold
        ) in enumerate(
            folds,
            start=1
        ):

            checkpoint = (
                OUT
                / f"{regime}_fold{fold_number}_checkpoint.joblib"
            )

            if checkpoint.exists():

                saved = joblib.load(
                    checkpoint
                )

                if (
                    saved.get("signature") != signature
                    or not np.array_equal(
                        saved.get("hold"),
                        hold
                    )
                    or set(saved["scores"])
                    != {
                        c["name"]
                        for c in EXPERIMENTS
                    }
                ):
                    raise RuntimeError(
                        f"Incompatible cached fold: {checkpoint}"
                    )

                scores = saved["scores"]

                print(
                    f"{regime.upper()} fold {fold_number}: "
                    "cached results reused",
                    flush=True
                )

            else:

                scores = {}

                print(
                    f"\n{regime.upper()} fold {fold_number}: "
                    f"train={len(fit)}, "
                    f"held={len(hold)}, "
                    f"advanced={y[hold].sum()}, "
                    f"held sites={len(set(site[hold]))}",
                    flush=True
                )

                for config in EXPERIMENTS:

                    model = fit_model(
                        config,
                        fit
                    )

                    score = predict_model(
                        model,
                        hold
                    )

                    if not np.isfinite(
                        score
                    ).all():
                        raise RuntimeError(
                            f"Invalid predictions: "
                            f"{config['name']}"
                        )

                    scores[
                        config["name"]
                    ] = score

                    ap = average_precision_score(
                        y[hold],
                        score
                    )

                    auc = roc_auc_score(
                        y[hold],
                        score
                    )

                    print(
                        f"  {config['name']:<41} "
                        f"AP={ap:.4f} "
                        f"AUC={auc:.4f}",
                        flush=True
                    )

                temporary = checkpoint.with_name(
                    checkpoint.name + ".tmp"
                )

                joblib.dump(
                    {
                        "signature": signature,
                        "hold": hold,
                        "scores": scores
                    },
                    temporary,
                    compress=3
                )

                os.replace(
                    temporary,
                    checkpoint
                )

            for config in EXPERIMENTS:

                name = config["name"]
                score = scores[name]

                predictions[
                    (regime, name)
                ][hold] = score

                fold_rows.append({
                    "regime": regime,
                    "fold": fold_number,
                    "name": name,
                    "patients": len(hold),
                    "advanced": int(
                        y[hold].sum()
                    ),
                    "fold_prevalence": float(
                        y[hold].mean()
                    ),
                    "AP": float(
                        average_precision_score(
                            y[hold],
                            score
                        )
                    ),
                    "AUC": float(
                        roc_auc_score(
                            y[hold],
                            score
                        )
                    )
                })

# ============================================================
# 15. Histology positive-control experiment
# ============================================================

# Check whether pathway features preserve LUAD/LUSC biology
positive = np.full(
    len(h),
    np.nan,
    dtype=float
)

with threadpool_limits(
    limits=THREADS
):

    for fit, hold in site_cv:

        scaler = StandardScaler()

        zfit = scaler.fit_transform(
            RNA_PATH[fit]
        )

        zhold = scaler.transform(
            RNA_PATH[hold]
        )

        classifier = LogisticRegression(
            C=0.1,
            solver="liblinear",
            class_weight="balanced",
            random_state=SEED,
            max_iter=2500
        )

        classifier.fit(
            zfit,
            h[fit]
        )

        positive[hold] = (
            classifier.predict_proba(
                zhold
            )[:, 1]
        )

positive_auc = float(
    roc_auc_score(
        h,
        positive
    )
)

positive_ap = float(
    average_precision_score(
        h,
        positive
    )
)

# ============================================================
# 16. Summarize both cross-validation regimes
# ============================================================

fold_df = pd.DataFrame(
    fold_rows
)

fold_df.to_csv(
    OUT / "stage19_fold_level_metrics.csv",
    index=False
)

summary = []

for regime in (
    "site",
    "patient"
):

    for config in EXPERIMENTS:

        name = config["name"]

        score = predictions[
            (regime, name)
        ]

        if not np.isfinite(score).all():
            raise RuntimeError(
                f"Incomplete OOF predictions: {regime}, {name}"
            )

        subset = fold_df.loc[
            fold_df.regime.eq(regime)
            & fold_df.name.eq(name)
        ]

        summary.append({
            "regime": regime,
            "model": name,
            "mean_fold_AP": float(
                subset.AP.mean()
            ),
            "mean_fold_AP_gain": float(
                (
                    subset.AP
                    - subset.fold_prevalence
                ).mean()
            ),
            "mean_fold_AUC": float(
                subset.AUC.mean()
            ),
            "pooled_OOF_AP": float(
                average_precision_score(
                    y,
                    score
                )
            ),
            "pooled_OOF_AUC": float(
                roc_auc_score(
                    y,
                    score
                )
            ),
            "LUAD_OOF_AP": float(
                average_precision_score(
                    y[h == 0],
                    score[h == 0]
                )
            ),
            "LUSC_OOF_AP": float(
                average_precision_score(
                    y[h == 1],
                    score[h == 1]
                )
            )
        })

        pd.DataFrame({
            "case_id": labels.case_id,
            "stage_label": y,
            "histology_label": h,
            "tissue_source_site": site,
            "oof_score": score
        }).to_csv(
            OUT / f"{regime}_{name}_OOF.csv",
            index=False
        )

comparison = pd.DataFrame(
    summary
).sort_values(
    [
        "regime",
        "mean_fold_AP_gain"
    ],
    ascending=[True, False]
)

comparison.to_csv(
    OUT / "stage19_site_vs_patient_comparison.csv",
    index=False
)

# ============================================================
# 17. Select the model using site-disjoint CV only
# ============================================================

site_leader = comparison.loc[
    comparison.regime.eq("site")
].sort_values(
    [
        "mean_fold_AP_gain",
        "mean_fold_AUC"
    ],
    ascending=False
).iloc[0]

best_name = str(
    site_leader.model
)

best_config = next(
    config
    for config in EXPERIMENTS
    if config["name"] == best_name
)

# ============================================================
# 18. Compare with the earlier Stage 18 reference
# ============================================================

paired_reference = None

reference_path = (
    ROOT
    / "results"
    / "metrics"
    / "stage18"
    / "site_Pooled_Variance_SVD_training_only_OOF.npy"
)

if reference_path.exists():

    old_scores = np.load(
        reference_path
    )

    if (
        old_scores.shape == (1, len(y))
        and np.isfinite(old_scores).all()
    ):

        old_scores = old_scores[0]

        new_scores = predictions[
            ("site", best_name)
        ]

        rng = np.random.default_rng(
            SEED + 19
        )

        differences = []

        for _ in range(N_BOOT):

            indices = np.concatenate([
                rng.choice(
                    np.flatnonzero(
                        strata == group
                    ),
                    size=int(
                        (strata == group).sum()
                    ),
                    replace=True
                )
                for group in (0, 1, 2, 3)
            ])

            differences.append(
                average_precision_score(
                    y[indices],
                    new_scores[indices]
                )
                - average_precision_score(
                    y[indices],
                    old_scores[indices]
                )
            )

        paired_reference = {
            "best_model_delta_pooled_AP": float(
                average_precision_score(
                    y,
                    new_scores
                )
                - average_precision_score(
                    y,
                    old_scores
                )
            ),
            "exploratory_bootstrap_interval": (
                np.percentile(
                    differences,
                    [2.5, 97.5]
                ).tolist()
            ),
            "reference": (
                "Stage18 site-held-out pooled-variance SVD"
            )
        }

# ============================================================
# 19. Refit selected model on all training patients
# ============================================================

with threadpool_limits(
    limits=THREADS
):

    final_fit = fit_model(
        best_config,
        np.arange(len(y))
    )

model_path = (
    MODELS
    / "stage19_site_cv_selected_training_model.joblib"
)

temporary_model = model_path.with_name(
    model_path.name + ".tmp"
)

joblib.dump(
    {
        "signature": signature,
        "config": best_config,
        "pipeline": final_fit,
        "pathway_weights": str(
            OUT / "reactome_gene_pathway_weights.npz"
        ),
        "gene_reference": str(
            RAW / "gene_reference.csv"
        ),
        "training_only": True
    },
    temporary_model,
    compress=3
)

os.replace(
    temporary_model,
    model_path
)

# ============================================================
# 20. Save the complete research audit
# ============================================================

report = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "signature": signature,
    "training_patients": len(y),
    "validation_not_evaluated": len(val_ids),
    "sealed_test_not_evaluated": len(test_ids),
    "Reactome_source": REACTOME_URL,
    "Reactome_pathways": len(pathways),
    "site_cv_selected_model": best_name,
    "site_cv_selected_mean_AP": float(
        site_leader.mean_fold_AP
    ),
    "site_cv_selected_mean_AUC": float(
        site_leader.mean_fold_AUC
    ),
    "histology_positive_control_site_OOF_AUC": (
        positive_auc
    ),
    "histology_positive_control_site_OOF_AP": (
        positive_ap
    ),
    "paired_exploratory_stage18": paired_reference,
    "limitations": [
        "The mean-rank method is singscore-inspired, "
        "not identical to the singscore R package.",
        "Repeated model selection can inflate CV scores.",
        "Site-disjoint CV is not independent external validation.",
        "No preprocessing or fitting uses validation/test patients.",
        "The pathway approach is exploratory; improvement "
        "is not guaranteed."
    ]
}

(
    OUT / "stage19_summary.json"
).write_text(
    json.dumps(
        report,
        indent=2
    ),
    encoding="utf-8"
)

# ============================================================
# 21. Final results
# ============================================================

print("\n" + "=" * 73)
print(
    "STAGE 19 FINAL - "
    "SITE-ROBUST PATHWAY MODEL COMPARISON"
)
print("=" * 73)

print(
    comparison.to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

print(
    "\nHistology positive control, "
    f"site-held-out AUC={positive_auc:.4f}, "
    f"AP={positive_ap:.4f}"
)

print(
    "Selected by site-disjoint training CV:",
    best_name
)

print(
    "Exploratory paired Stage18 comparison:",
    paired_reference
)

print("Saved results:", OUT)
print("Saved frozen training model:", model_path)

print(
    "Validation/test molecular features evaluated: NO"
)

print("\nSTAGE 19 COMPLETED")

STAGE 19 - REACTOME SINGLE-SAMPLE BIOLOGICAL REPRESENTATION
Training=667 | Advanced=133 | Validation=143 untouched | Test=143 untouched
Distinct tissue-source sites: 63


Mapped unique Reactome pathways: 1613
Unambiguous gene symbols: 19932
Single-sample RNA pathway representation: (667, 1613)
RNA + CNV pathway representation: (667, 4839)

Site-held-out CV folds: 4
Patient-wise CV folds: 4
Site overlap in site-held-out CV: NONE

SITE fold 1: train=498, held=169, advanced=33, held sites=13
  RNA_pathway_L2                            AP=0.2791 AUC=0.5947
  RNA_pathway_L2_sitebalanced               AP=0.2750 AUC=0.6018
  RNA_pathway_histology_interaction         AP=0.2374 AUC=0.5354
  RNA_CNV_pathway_L2                        AP=0.2522 AUC=0.5885
  RNA_CNV_pathway_sitebalanced              AP=0.2579 AUC=0.6000
  RNA_pathway_RBF_SVM                       AP=0.3205 AUC=0.5818
  RNA_pathway_gradient_boosting             AP=0.1914 AUC=0.4886
  RNA_CNV_pathway_gradient_boosting         AP=0.33

In [19]:

import hashlib
import json
import os
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from scipy import sparse
from scipy.stats import rankdata

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from threadpoolctl import threadpool_limits


# ============================================================
# 1. Project configuration
# ============================================================

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

RAW = ROOT / "data" / "processed" / "gene_level"

LOCKED = (
    ROOT / "data" / "splits"
    / "locked_four_omics_v1.csv"
)

STAGE19 = (
    ROOT / "results" / "metrics" / "stage19"
)

OUT = (
    ROOT / "results" / "metrics" / "stage20"
)

MODEL_DIR = (
    ROOT / "models" / "checkpoints" / "stage20"
)

for directory in (OUT, MODEL_DIR):
    directory.mkdir(
        parents=True,
        exist_ok=True
    )

SEED = 42
THREADS = min(4, os.cpu_count() or 1)

OUTER_FOLDS = 4
INNER_FOLDS = 3

N_BOOT = 600

VERSION = "stage20_site_nuisance_nested_cv_v1"


# ============================================================
# 2. Predeclared candidate architectures
# ============================================================

CANDIDATES = [
    {
        "name": "Reference_RBF_C05",
        "family": "svm",
        "mode": "raw",
        "C": 0.5
    },
    {
        "name": "Reference_RBF_C15",
        "family": "svm",
        "mode": "raw",
        "C": 1.5
    },
    {
        "name": "HistCenter_RBF_C05",
        "family": "svm",
        "mode": "hist",
        "C": 0.5
    },
    {
        "name": "SitePC3_Half_RBF_C05",
        "family": "svm",
        "mode": "site3half",
        "C": 0.5
    },
    {
        "name": "SitePC3_Full_RBF_C05",
        "family": "svm",
        "mode": "site3full",
        "C": 0.5
    },
    {
        "name": "SitePC6_Half_RBF_C05",
        "family": "svm",
        "mode": "site6half",
        "C": 0.5
    },
    {
        "name": "HistSitePC3_Half_RBF_C05",
        "family": "svm",
        "mode": "histsite3half",
        "C": 0.5
    },
    {
        "name": "HistSitePC3_Half_RBF_C15",
        "family": "svm",
        "mode": "histsite3half",
        "C": 1.5
    },
    {
        "name": "SiteStable512_RBF_C05",
        "family": "svm",
        "mode": "stable512",
        "C": 0.5
    },
    {
        "name": "SiteStable512_RBF_C15",
        "family": "svm",
        "mode": "stable512",
        "C": 1.5
    },
    {
        "name": "Reference_Linear_C01",
        "family": "logit",
        "mode": "raw",
        "C": 0.1
    },
    {
        "name": "SitePC3_Linear_C01",
        "family": "logit",
        "mode": "site3half",
        "C": 0.1
    }
]


# ============================================================
# 3. Input integrity and sealed partition
# ============================================================

cases = pd.read_csv(
    RAW / "case_order.csv",
    dtype=str,
    keep_default_na=False
)

genes = pd.read_csv(
    RAW / "gene_reference.csv",
    dtype=str,
    keep_default_na=False
)

locked = pd.read_csv(
    LOCKED,
    dtype=str,
    keep_default_na=False
)

if (
    not cases.case_id.is_unique
    or not locked.case_id.is_unique
    or not genes.ensembl_id.is_unique
):
    raise RuntimeError(
        "Duplicate patient or gene identifiers."
    )

joined = cases[
    [
        "case_id",
        "patient_id",
        "histology",
        "stage_group"
    ]
].merge(
    locked[
        [
            "case_id",
            "histology",
            "stage_group",
            "split"
        ]
    ],
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_matrix", "_locked")
)

if (
    not joined.histology_matrix.equals(
        joined.histology_locked
    )
    or not joined.stage_group_matrix.equals(
        joined.stage_group_locked
    )
):
    raise RuntimeError(
        "Clinical labels do not match the locked patient split."
    )

train_ids = np.flatnonzero(
    joined.split.eq("train").to_numpy()
)

val_ids = np.flatnonzero(
    joined.split.eq("validation").to_numpy()
)

test_ids = np.flatnonzero(
    joined.split.eq("test").to_numpy()
)

if (
    len(train_ids),
    len(val_ids),
    len(test_ids)
) != (667, 143, 143):
    raise RuntimeError(
        "Locked 667/143/143 partition changed."
    )

if len(genes) != 19944:
    raise RuntimeError(
        "Unexpected Stage 08 gene reference length."
    )

labels = joined.iloc[
    train_ids
].reset_index(drop=True)

y = labels.stage_group_matrix.eq(
    "Advanced"
).to_numpy(dtype=np.int64)

h = labels.histology_matrix.eq(
    "LUSC"
).to_numpy(dtype=np.int64)

site = labels.patient_id.str.extract(
    r"^TCGA-([A-Za-z0-9]{2})-",
    expand=False
)

if site.isna().any():
    raise RuntimeError(
        "Some training TCGA tissue-source-site codes are missing."
    )

site = site.to_numpy(dtype=str)

strata = 2 * y + h

if set(strata) != {0, 1, 2, 3}:
    raise RuntimeError(
        "Missing training stage-histology stratum."
    )

print("=" * 76)
print(
    "STAGE 20 - NESTED SITE-GENERALIZATION "
    "AND UNWANTED-VARIATION CONTROL"
)
print("=" * 76)

print(
    f"Training={len(y)}, "
    f"validation={len(val_ids)} (not read), "
    f"sealed test={len(test_ids)} (not read)"
)

print(
    f"Training sites={len(np.unique(site))}; "
    f"advanced prevalence={y.mean():.4f}"
)


# ============================================================
# 4. Rebuild Stage 19 Reactome pathway scores
# ============================================================

W_path = (
    STAGE19
    / "reactome_gene_pathway_weights.npz"
)

metadata_path = (
    STAGE19
    / "reactome_pathway_reference.csv"
)

rna_path = RAW / "rna_tpm.npy"

for needed in (
    W_path,
    metadata_path,
    rna_path
):
    if not needed.is_file():
        raise FileNotFoundError(
            f"Missing Stage19/Stage08 input: {needed}"
        )

W = sparse.load_npz(
    W_path
).tocsr().astype(np.float32)

pathway_metadata = pd.read_csv(
    metadata_path
)

if (
    W.shape[1] != len(genes)
    or W.shape[0] != len(pathway_metadata)
    or W.shape[0] < 100
    or not np.allclose(
        np.asarray(
            W.sum(axis=1)
        ).ravel(),
        1.0,
        atol=1e-5
    )
):
    raise RuntimeError(
        "Stage19 Reactome pathway matrix is inconsistent."
    )

source = np.load(
    rna_path,
    mmap_mode="r"
)

if source.shape != (
    len(cases),
    len(genes)
):
    raise RuntimeError(
        "Stage08 RNA matrix does not match reference genes."
    )

# Materialize only the locked training patients
rna = np.asarray(
    source[train_ids],
    dtype=np.float32
).copy()

del source

observed = (
    np.isfinite(rna)
    & (rna >= 0)
)

if observed.mean() < 0.95:
    raise RuntimeError(
        "RNA observation coverage unexpectedly low."
    )

rna = np.where(
    observed,
    np.log1p(
        np.maximum(rna, 0.0)
    ),
    0.0
)

ranks = rankdata(
    rna,
    axis=1,
    method="average"
).astype(np.float32)

ranks = (
    ranks - (len(genes) + 1.0) / 2.0
) / len(genes)

X = np.asarray(
    (W @ ranks.T).T,
    dtype=np.float32
)

del rna, observed, ranks

if (
    X.shape != (len(y), W.shape[0])
    or not np.isfinite(X).all()
):
    raise RuntimeError(
        "Pathway score dimensions or values invalid."
    )

print(
    "Stage19 RNA pathway features reconstructed:",
    X.shape
)


# ============================================================
# 5. Define site-disjoint nested CV
# ============================================================

outer = list(
    StratifiedGroupKFold(
        n_splits=OUTER_FOLDS,
        shuffle=True,
        random_state=SEED
    ).split(
        np.arange(len(y)),
        strata,
        site
    )
)

for fit, hold in outer:

    if set(site[fit]) & set(site[hold]):
        raise RuntimeError(
            "Outer site leakage."
        )

    if (
        set(strata[fit]) != {0, 1, 2, 3}
        or set(strata[hold]) != {0, 1, 2, 3}
    ):
        raise RuntimeError(
            "A site-held-out fold is missing "
            "a stage-histology group."
        )


def choose_inner_groups(
    outer_fit,
    outer_number
):

    options = []

    for seed in (
        42, 59, 73, 89,
        101, 127, 151, 173
    ):

        trial = list(
            StratifiedGroupKFold(
                n_splits=INNER_FOLDS,
                shuffle=True,
                random_state=seed
            ).split(
                np.arange(
                    len(outer_fit)
                ),
                strata[outer_fit],
                site[outer_fit]
            )
        )

        valid = True
        deviation = 0.0

        for a, b in trial:

            fit_idx = outer_fit[a]
            hold_idx = outer_fit[b]

            if (
                set(strata[fit_idx])
                != {0, 1, 2, 3}
                or set(strata[hold_idx])
                != {0, 1, 2, 3}
                or bool(
                    set(site[fit_idx])
                    & set(site[hold_idx])
                )
            ):

                valid = False
                break

            deviation += abs(
                len(hold_idx)
                / len(outer_fit)
                - 1 / INNER_FOLDS
            )

            deviation += abs(
                y[hold_idx].mean()
                - y[outer_fit].mean()
            )

        if valid:

            options.append(
                (
                    deviation,
                    seed,
                    trial
                )
            )

    if not options:
        raise RuntimeError(
            "No valid training-only inner site CV "
            f"for outer fold {outer_number}."
        )

    _, seed, selected = min(
        options,
        key=lambda entry: (
            entry[0],
            entry[1]
        )
    )

    return seed, [
        (
            outer_fit[a],
            outer_fit[b]
        )
        for a, b in selected
    ]


# ============================================================
# 6. Estimate unwanted site-associated variation
# ============================================================

def site_residuals(
    z,
    y_fit,
    h_fit
):

    # Covariates of interest are removed only
    # while estimating the site nuisance directions.
    design = np.column_stack([
        np.ones(len(y_fit)),
        y_fit.astype(float),
        h_fit.astype(float)
    ])

    coef = np.linalg.lstsq(
        design,
        z,
        rcond=None
    )[0]

    return z - design @ coef


def nuisance_basis(
    z,
    y_fit,
    h_fit,
    site_fit,
    k
):

    residual = site_residuals(
        z,
        y_fit,
        h_fit
    )

    centroids = []

    for category in np.unique(
        site_fit
    ):

        mask = (
            site_fit == category
        )

        if mask.sum() >= 3:

            centroids.append(
                residual[mask].mean(
                    axis=0
                ) * np.sqrt(
                    mask.sum()
                )
            )

    if len(centroids) <= k + 1:

        return np.zeros(
            (z.shape[1], 0),
            dtype=np.float32
        )

    centered = np.stack(
        centroids
    ).astype(np.float64)

    centered -= centered.mean(
        axis=0,
        keepdims=True
    )

    _, _, vh = np.linalg.svd(
        centered,
        full_matrices=False
    )

    return vh[:k].T.astype(
        np.float32
    )


# ============================================================
# 7. Select pathways relatively stable across sites
# ============================================================

def stable_site_features(
    z,
    y_fit,
    h_fit,
    site_fit,
    n_keep=512
):

    residual = site_residuals(
        z,
        y_fit,
        h_fit
    )

    site_ss = np.zeros(
        z.shape[1],
        dtype=np.float64
    )

    total_ss = np.sum(
        residual ** 2,
        axis=0,
        dtype=np.float64
    )

    for category in np.unique(
        site_fit
    ):

        rows = (
            site_fit == category
        )

        site_ss += (
            rows.sum()
            * residual[rows].mean(
                axis=0
            ) ** 2
        )

    ratio = site_ss / np.maximum(
        total_ss,
        1e-8
    )

    ratio[
        total_ss < 1e-4
    ] = np.inf

    selected = np.argsort(
        ratio,
        kind="stable"
    )[:min(n_keep, z.shape[1])]

    return np.sort(
        selected
    )


# ============================================================
# 8. Fit preprocessing only on fitting patients
# ============================================================

def fit_preprocessing(
    fit,
    mode
):

    scaler = StandardScaler()

    z = np.clip(
        scaler.fit_transform(
            X[fit]
        ),
        -7,
        7
    ).astype(np.float32)

    state = {
        "scaler": scaler,
        "hist_means": None,
        "basis": None,
        "strength": 0.0,
        "columns": None,
        "mode": mode
    }

    # Histology-conditional centering
    if mode.startswith("hist"):

        means = {}

        for code in (0, 1):

            if not np.any(
                h[fit] == code
            ):
                raise RuntimeError(
                    "Missing histology in preprocessing fit."
                )

            means[code] = z[
                h[fit] == code
            ].mean(axis=0)

            z[
                h[fit] == code
            ] -= means[code]

        state["hist_means"] = means

    # Pathway stability filtering
    if mode == "stable512":

        selected = stable_site_features(
            z,
            y[fit],
            h[fit],
            site[fit]
        )

        state["columns"] = selected
        z = z[:, selected]

    # Source-site nuisance subspace projection
    if (
        "site" in mode
        and mode != "stable512"
    ):

        k = (
            6
            if "site6" in mode
            else 3
        )

        strength = (
            1.0
            if "full" in mode
            else 0.50
        )

        basis = nuisance_basis(
            z,
            y[fit],
            h[fit],
            site[fit],
            k
        )

        if basis.shape[1]:

            z = (
                z
                - strength
                * (
                    (z @ basis)
                    @ basis.T
                )
            )

        state["basis"] = basis
        state["strength"] = strength

    if not np.isfinite(z).all():
        raise RuntimeError(
            "Non-finite training feature after correction."
        )

    return z.astype(
        np.float32
    ), state


# ============================================================
# 9. Transform held-out patients without refitting
# ============================================================

def frozen_transform(
    ids,
    state
):

    z = np.clip(
        state["scaler"].transform(
            X[ids]
        ),
        -7,
        7
    ).astype(np.float32)

    if state["hist_means"] is not None:

        for code, mean in (
            state["hist_means"].items()
        ):

            z[
                h[ids] == code
            ] -= mean

    if state["columns"] is not None:

        z = z[
            :,
            state["columns"]
        ]

    basis = state["basis"]

    if (
        basis is not None
        and basis.shape[1]
    ):

        z = (
            z
            - state["strength"]
            * (
                (z @ basis)
                @ basis.T
            )
        )

    if not np.isfinite(z).all():
        raise RuntimeError(
            "Non-finite transformed held-out features."
        )

    return z.astype(np.float32)


# ============================================================
# 10. Site-balanced fitting weights
# ============================================================

def weights_by_site(
    fit
):

    _, inverse, counts = np.unique(
        site[fit],
        return_inverse=True,
        return_counts=True
    )

    weight = (
        1.0
        / np.sqrt(
            counts[inverse].astype(float)
        )
    )

    weight = np.minimum(
        weight,
        np.median(weight) * 3.0
    )

    return (
        weight / weight.mean()
    ).astype(np.float64)


# ============================================================
# 11. Fit and predict a candidate model
# ============================================================

def fit_predict(
    fit,
    hold,
    candidate,
    cached=None
):

    key = candidate["mode"]

    if (
        cached is not None
        and key in cached
    ):

        zfit, state = cached[key]

    else:

        zfit, state = fit_preprocessing(
            fit,
            key
        )

        if cached is not None:

            cached[key] = (
                zfit,
                state
            )

    zhold = frozen_transform(
        hold,
        state
    )

    if candidate["family"] == "svm":

        estimator = SVC(
            C=float(candidate["C"]),
            kernel="rbf",
            gamma="scale",
            class_weight="balanced",
            random_state=SEED
        )

    else:

        estimator = LogisticRegression(
            C=float(candidate["C"]),
            solver="liblinear",
            class_weight="balanced",
            max_iter=2500,
            random_state=SEED
        )

    estimator.fit(
        zfit,
        y[fit],
        sample_weight=weights_by_site(fit)
    )

    if candidate["family"] == "svm":

        scores = estimator.decision_function(
            zhold
        )

    else:

        scores = estimator.predict_proba(
            zhold
        )[:, 1]

    if not np.isfinite(
        scores
    ).all():

        raise RuntimeError(
            "Non-finite site-held-out model score."
        )

    return (
        np.asarray(
            scores,
            dtype=float
        ),
        state,
        estimator
    )


# ============================================================
# 12. Reproducibility fingerprint
# ============================================================

def signature_of(
    path
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as handle:

        for chunk in iter(
            lambda: handle.read(
                4 * 1024 * 1024
            ),
            b""
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()


signature = hashlib.sha256(
    json.dumps(
        {
            "version": VERSION,
            "cohort": labels.case_id.tolist(),
            "stage": y.tolist(),
            "histology": h.tolist(),
            "sites": site.tolist(),
            "source_rna": signature_of(
                rna_path
            ),
            "source_gene_ref": signature_of(
                RAW / "gene_reference.csv"
            ),
            "pathways": signature_of(
                W_path
            ),
            "configs": CANDIDATES,
            "outer": OUTER_FOLDS,
            "inner": INNER_FOLDS
        },
        sort_keys=True
    ).encode()
).hexdigest()


# ============================================================
# 13. Run nested site-held-out cross-validation
# ============================================================

nested_scores = np.full(
    len(y),
    np.nan,
    dtype=float
)

reference_scores = np.full(
    len(y),
    np.nan,
    dtype=float
)

fold_rows = []
inner_rows = []

reference = CANDIDATES[0]

with threadpool_limits(
    limits=THREADS
):

    for number, (
        outer_fit,
        outer_hold
    ) in enumerate(
        outer,
        start=1
    ):

        checkpoint = (
            OUT
            / f"outer_fold{number}_checkpoint.joblib"
        )

        if checkpoint.exists():

            saved = joblib.load(
                checkpoint
            )

            if (
                saved.get("signature")
                != signature
                or not np.array_equal(
                    saved.get("outer_hold"),
                    outer_hold
                )
            ):

                raise RuntimeError(
                    f"Incompatible fold checkpoint: {checkpoint}"
                )

            chosen = saved["winner"]

            p_nested = saved[
                "nested_scores"
            ]

            p_reference = saved[
                "reference_scores"
            ]

            fold_rows.append(
                saved["metrics"]
            )

            inner_rows.extend(
                saved["inner_rows"]
            )

            print(
                f"Outer {number}: resumed "
                f"{chosen['name']}",
                flush=True
            )

        else:

            inner_seed, inner_folds = (
                choose_inner_groups(
                    outer_fit,
                    number
                )
            )

            performance = {
                c["name"]: []
                for c in CANDIDATES
            }

            inner_for_outer = []

            print(
                f"\nOUTER SITE FOLD {number}: "
                f"fit={len(outer_fit)}, "
                f"held={len(outer_hold)}, "
                f"advanced={int(y[outer_hold].sum())}, "
                f"inner seed={inner_seed}",
                flush=True
            )

            for j, (
                inner_fit,
                inner_hold
            ) in enumerate(
                inner_folds,
                start=1
            ):

                cached_preprocessing = {}

                for candidate in CANDIDATES:

                    pred, _, _ = fit_predict(
                        inner_fit,
                        inner_hold,
                        candidate,
                        cached_preprocessing
                    )

                    ap = float(
                        average_precision_score(
                            y[inner_hold],
                            pred
                        )
                    )

                    auc = float(
                        roc_auc_score(
                            y[inner_hold],
                            pred
                        )
                    )

                    ap_gain = (
                        ap
                        - float(
                            y[inner_hold].mean()
                        )
                    )

                    performance[
                        candidate["name"]
                    ].append(
                        (
                            ap_gain,
                            auc
                        )
                    )

                    inner_for_outer.append({
                        "outer_fold": number,
                        "inner_fold": j,
                        "candidate": candidate["name"],
                        "held_patients": len(inner_hold),
                        "AP": ap,
                        "AUC": auc,
                        "AP_gain": ap_gain
                    })

            rankings = []

            for candidate in CANDIDATES:

                qualities = performance[
                    candidate["name"]
                ]

                rankings.append(
                    (
                        float(
                            np.mean([
                                x[0]
                                for x in qualities
                            ])
                        ),
                        float(
                            np.mean([
                                x[1]
                                for x in qualities
                            ])
                        ),
                        candidate
                    )
                )

            # Outer fold outcomes do not influence model selection.
            _, _, chosen = max(
                rankings,
                key=lambda item: (
                    item[0],
                    item[1]
                )
            )

            outer_cache = {}

            p_nested, _, _ = fit_predict(
                outer_fit,
                outer_hold,
                chosen,
                outer_cache
            )

            p_reference, _, _ = fit_predict(
                outer_fit,
                outer_hold,
                reference,
                outer_cache
            )

            metrics = {
                "fold": number,
                "winner": chosen["name"],
                "held_patients": len(outer_hold),
                "advanced": int(
                    y[outer_hold].sum()
                ),
                "prevalence": float(
                    y[outer_hold].mean()
                ),
                "nested_AP": float(
                    average_precision_score(
                        y[outer_hold],
                        p_nested
                    )
                ),
                "nested_AUC": float(
                    roc_auc_score(
                        y[outer_hold],
                        p_nested
                    )
                ),
                "reference_AP": float(
                    average_precision_score(
                        y[outer_hold],
                        p_reference
                    )
                ),
                "reference_AUC": float(
                    roc_auc_score(
                        y[outer_hold],
                        p_reference
                    )
                )
            }

            saved = {
                "signature": signature,
                "outer_hold": outer_hold,
                "winner": chosen,
                "nested_scores": p_nested,
                "reference_scores": p_reference,
                "metrics": metrics,
                "inner_rows": inner_for_outer
            }

            temporary = checkpoint.with_name(
                checkpoint.name + ".tmp"
            )

            joblib.dump(
                saved,
                temporary,
                compress=3
            )

            os.replace(
                temporary,
                checkpoint
            )

            fold_rows.append(
                metrics
            )

            inner_rows.extend(
                inner_for_outer
            )

            print(
                f"  Inner-selected={chosen['name']} | "
                f"outer AP={metrics['nested_AP']:.4f} "
                f"AUC={metrics['nested_AUC']:.4f} | "
                f"Stage19 reference AP="
                f"{metrics['reference_AP']:.4f}",
                flush=True
            )

        nested_scores[
            outer_hold
        ] = p_nested

        reference_scores[
            outer_hold
        ] = p_reference


if (
    not np.isfinite(
        nested_scores
    ).all()
    or not np.isfinite(
        reference_scores
    ).all()
):
    raise RuntimeError(
        "Nested site-held-out predictions incomplete."
    )


# ============================================================
# 14. Verify the original Stage 19 SVM reference
# ============================================================

reference_path = (
    STAGE19
    / "site_RNA_pathway_RBF_SVM_OOF.csv"
)

stage19_match = None

if reference_path.exists():

    reference_df = pd.read_csv(
        reference_path,
        dtype={"case_id": str}
    )

    expected_ids = (
        labels.case_id.astype(str).to_numpy()
    )

    if (
        len(reference_df)
        != len(expected_ids)
        or not np.array_equal(
            reference_df.case_id.astype(str),
            expected_ids
        )
    ):

        raise RuntimeError(
            "Stage19 reference patient order changed."
        )

    historic = (
        reference_df.oof_score.to_numpy(
            dtype=float
        )
    )

    stage19_match = float(
        np.max(
            np.abs(
                historic - reference_scores
            )
        )
    )

    if stage19_match > 1e-4:

        raise RuntimeError(
            "Stage19 baseline reproduction differs by "
            f"{stage19_match:.6f}. "
            "Do not compare until feature or CV "
            "alignment is resolved."
        )

    print(
        "\nStage19 SVM baseline reproduction: "
        "PASSED "
        f"(max error {stage19_match:.3g})"
    )


# ============================================================
# 15. Save fold-level evaluation
# ============================================================

fold_table = pd.DataFrame(
    fold_rows
).sort_values("fold")

inner_table = pd.DataFrame(
    inner_rows
)

fold_table.to_csv(
    OUT / "nested_site_heldout_fold_results.csv",
    index=False
)

inner_table.to_csv(
    OUT / "nested_inner_site_cv_results.csv",
    index=False
)

pd.DataFrame({
    "case_id": labels.case_id,
    "histology": labels.histology_matrix,
    "advanced_stage": y,
    "site": site,
    "nested_score": nested_scores,
    "stage19_reference_score": reference_scores
}).to_csv(
    OUT / "nested_site_heldout_OOF_predictions.csv",
    index=False
)


# ============================================================
# 16. Exploratory site-cluster bootstrap uncertainty
# ============================================================

rng = np.random.default_rng(
    SEED + 2020
)

site_names = np.unique(
    site
)

boot_deltas = []

for _ in range(N_BOOT):

    sampled_sites = rng.choice(
        site_names,
        size=len(site_names),
        replace=True
    )

    sampled_idx = np.concatenate([
        np.flatnonzero(
            site == item
        )
        for item in sampled_sites
    ])

    if (
        len(np.unique(
            y[sampled_idx]
        )) < 2
    ):
        continue

    boot_deltas.append(
        average_precision_score(
            y[sampled_idx],
            nested_scores[sampled_idx]
        )
        - average_precision_score(
            y[sampled_idx],
            reference_scores[sampled_idx]
        )
    )

if len(boot_deltas) < 100:
    raise RuntimeError(
        "Insufficient valid site-cluster bootstrap draws."
    )


# ============================================================
# 17. Assemble the nested-CV research report
# ============================================================

winner_counts = dict(
    Counter(
        fold_table.winner
    )
)

result = {
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "signature": signature,
    "training_cases": len(y),
    "sites": len(site_names),

    "validation_features_loaded": False,
    "test_features_loaded": False,

    "nested_outer_mean_AP": float(
        fold_table.nested_AP.mean()
    ),
    "nested_outer_mean_AUC": float(
        fold_table.nested_AUC.mean()
    ),

    "reference_outer_mean_AP": float(
        fold_table.reference_AP.mean()
    ),
    "reference_outer_mean_AUC": float(
        fold_table.reference_AUC.mean()
    ),

    "nested_pooled_OOF_AP": float(
        average_precision_score(
            y,
            nested_scores
        )
    ),

    "reference_pooled_OOF_AP": float(
        average_precision_score(
            y,
            reference_scores
        )
    ),

    "nested_pooled_OOF_AUC": float(
        roc_auc_score(
            y,
            nested_scores
        )
    ),

    "reference_pooled_OOF_AUC": float(
        roc_auc_score(
            y,
            reference_scores
        )
    ),

    "nested_LUAD_OOF_AP": float(
        average_precision_score(
            y[h == 0],
            nested_scores[h == 0]
        )
    ),

    "nested_LUSC_OOF_AP": float(
        average_precision_score(
            y[h == 1],
            nested_scores[h == 1]
        )
    ),

    "nested_delta_pooled_AP_vs_stage19": float(
        average_precision_score(
            y,
            nested_scores
        )
        - average_precision_score(
            y,
            reference_scores
        )
    ),

    "exploratory_site_cluster_bootstrap_delta_AP_95pct": [
        float(value)
        for value in np.percentile(
            boot_deltas,
            [2.5, 97.5]
        )
    ],

    "winning_candidate_by_outer_fold": winner_counts,

    "stage19_reference_max_error": stage19_match,

    "limitations": [
        (
            "Nested folds prevent using current outer "
            "held-out outcomes for model choice, but "
            "Stage19 results informed Stage20 candidate design."
        ),
        (
            "Site-held-out internal CV is not independent "
            "external validation."
        ),
        (
            "Subspace removal is an experimental "
            "RUV-inspired approximation, not RUV-III."
        ),
        (
            "Site-cluster bootstrap is exploratory "
            "and cannot remove selection bias."
        ),
        (
            "SVM decision scores are not "
            "probability-calibrated."
        )
    ]
}


# ============================================================
# 18. Select a deployable training-only model
# ============================================================

mean_inner = (
    inner_table
    .groupby(
        "candidate",
        as_index=False
    )
    .agg(
        mean_inner_AP_gain=(
            "AP_gain",
            "mean"
        ),
        mean_inner_AUC=(
            "AUC",
            "mean"
        )
    )
    .sort_values(
        [
            "mean_inner_AP_gain",
            "mean_inner_AUC"
        ],
        ascending=False
    )
)

final_name = str(
    mean_inner.iloc[0].candidate
)

final_candidate = next(
    c
    for c in CANDIDATES
    if c["name"] == final_name
)

with threadpool_limits(
    limits=THREADS
):

    _, state = fit_preprocessing(
        np.arange(
            len(y)
        ),
        final_candidate["mode"]
    )

    Ztrain = frozen_transform(
        np.arange(
            len(y)
        ),
        state
    )

    if final_candidate["family"] == "svm":

        final_estimator = SVC(
            C=float(
                final_candidate["C"]
            ),
            kernel="rbf",
            gamma="scale",
            class_weight="balanced",
            random_state=SEED
        )

    else:

        final_estimator = LogisticRegression(
            C=float(
                final_candidate["C"]
            ),
            solver="liblinear",
            class_weight="balanced",
            random_state=SEED,
            max_iter=2500
        )

    final_estimator.fit(
        Ztrain,
        y,
        sample_weight=weights_by_site(
            np.arange(
                len(y)
            )
        )
    )


# ============================================================
# 19. Save the frozen model
# ============================================================

model_path = (
    MODEL_DIR
    / "stage20_training_only_model.joblib"
)

temporary = model_path.with_name(
    model_path.name + ".tmp"
)

joblib.dump(
    {
        "version": VERSION,
        "signature": signature,
        "candidate": final_candidate,
        "preprocessing": state,
        "estimator": final_estimator,
        "pathway_reference": str(W_path),
        "training_only": True,
        "probabilities_calibrated": False
    },
    temporary,
    compress=3
)

os.replace(
    temporary,
    model_path
)

mean_inner.to_csv(
    OUT / "nested_inner_candidate_ranking.csv",
    index=False
)

result[
    "deployment_candidate_selected_from_training_inner_CV"
] = final_name

result[
    "frozen_training_model"
] = str(
    model_path
)

(
    OUT / "stage20_summary.json"
).write_text(
    json.dumps(
        result,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 20. Final reporting
# ============================================================

print("\n" + "=" * 76)
print(
    "STAGE 20 FINAL - NESTED SITE-DISJOINT RESULTS"
)
print("=" * 76)

print(
    fold_table.to_string(
        index=False,
        float_format=lambda v: f"{v:.4f}"
    )
)

print(
    "\nModel selections by outer fold:",
    winner_counts
)

print(
    "Nested mean AP/AUC:",
    f"{result['nested_outer_mean_AP']:.4f} / "
    f"{result['nested_outer_mean_AUC']:.4f}"
)

print(
    "Stage19 reference mean AP/AUC:",
    f"{result['reference_outer_mean_AP']:.4f} / "
    f"{result['reference_outer_mean_AUC']:.4f}"
)

print(
    "Nested pooled AP:",
    f"{result['nested_pooled_OOF_AP']:.4f}"
)

print(
    "Stage19 reference pooled AP:",
    f"{result['reference_pooled_OOF_AP']:.4f}"
)

print(
    "Site-bootstrap paired AP delta 95% interval:",
    result[
        "exploratory_site_cluster_bootstrap_delta_AP_95pct"
    ]
)

print(
    "Training-only deployment selection:",
    final_name
)

print(
    "Saved model:",
    model_path
)

print(
    "Validation/test features used: NO"
)

print(
    "STAGE 20 COMPLETED"
)

STAGE 20 - NESTED SITE-GENERALIZATION AND UNWANTED-VARIATION CONTROL
Training=667, validation=143 (not read), sealed test=143 (not read)
Training sites=63; advanced prevalence=0.1994
Stage19 RNA pathway features reconstructed: (667, 1613)

OUTER SITE FOLD 1: fit=498, held=169, advanced=33, inner seed=73
  Inner-selected=SitePC6_Half_RBF_C05 | outer AP=0.2821 AUC=0.5751 | Stage19 reference AP=0.3205

OUTER SITE FOLD 2: fit=500, held=167, advanced=34, inner seed=42
  Inner-selected=Reference_RBF_C05 | outer AP=0.2132 AUC=0.4920 | Stage19 reference AP=0.2132

OUTER SITE FOLD 3: fit=502, held=165, advanced=33, inner seed=42
  Inner-selected=HistSitePC3_Half_RBF_C15 | outer AP=0.2154 AUC=0.4798 | Stage19 reference AP=0.2565

OUTER SITE FOLD 4: fit=501, held=166, advanced=33, inner seed=42
  Inner-selected=SitePC6_Half_RBF_C05 | outer AP=0.3291 AUC=0.6685 | Stage19 reference AP=0.3344

Stage19 SVM baseline reproduction: PASSED (max error 0)

STAGE 20 FINAL - NESTED SITE-DISJOINT RESULTS
 fol

In [20]:

import hashlib
import json
import os
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import requests
from scipy import sparse
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

# ============================================================
# 1. Project configuration
# ============================================================

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

RAW = ROOT / "data" / "processed" / "gene_level"
STAGE19 = ROOT / "results" / "metrics" / "stage19"
OUT = ROOT / "results" / "metrics" / "stage21"
MODEL_DIR = ROOT / "models" / "checkpoints" / "stage21"
CLINICAL_DIR = ROOT / "data" / "raw" / "clinical_references"

for directory in (OUT, MODEL_DIR, CLINICAL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
THREADS = min(4, os.cpu_count() or 1)

OUTER_FOLDS = 4
INNER_FOLDS = 3

VERSION = "stage21_site_heldout_pfi_survival_v1"

CDR_ID = "1b5f413e-a8d1-4d10-92eb-7c4ae739ed81"
CDR_MD5 = "a4591b2dcee39591f59e5e25a6ce75fa"

CDR_PATH = (
    CLINICAL_DIR / "TCGA-CDR-SupplementalTableS1.xlsx"
)

CDR_SOURCES = [
    f"https://api.gdc.cancer.gov/data/{CDR_ID}",
    (
        "https://huggingface.co/datasets/gabrielaltay/"
        "tcga-pancanatlas-cdr/resolve/main/source/"
        "TCGA-CDR-SupplementalTableS1.xlsx"
    )
]

# Prespecified prognostic models
SPECS = [
    {
        "name": "Clinical_Cox",
        "kind": "clinical",
        "k": 0,
        "alpha": 0.003,
        "clinical": True
    },
    {
        "name": "RNA_Pathway_SVD16_Cox",
        "kind": "svd",
        "k": 16,
        "alpha": 0.03,
        "clinical": False
    },
    {
        "name": "Clinical_RNA_SVD16_Cox",
        "kind": "svd",
        "k": 16,
        "alpha": 0.03,
        "clinical": True
    },
    {
        "name": "Clinical_RNA_SVD32_Cox",
        "kind": "svd",
        "k": 32,
        "alpha": 0.03,
        "clinical": True
    },
    {
        "name": "Clinical_RNA_ScoreTop16_Cox",
        "kind": "score",
        "k": 16,
        "alpha": 0.03,
        "clinical": True
    },
    {
        "name": "Clinical_RNA_ScoreTop32_Cox",
        "kind": "score",
        "k": 32,
        "alpha": 0.03,
        "clinical": True
    },
    {
        "name": "Clinical_RNA_ScoreTop32_StrongRidge",
        "kind": "score",
        "k": 32,
        "alpha": 0.10,
        "clinical": True
    }
]

# ============================================================
# 2. Verify the original patient cohort
# ============================================================

cases = pd.read_csv(
    RAW / "case_order.csv",
    dtype=str,
    keep_default_na=False
)

genes = pd.read_csv(
    RAW / "gene_reference.csv",
    dtype=str,
    keep_default_na=False
)

locked = pd.read_csv(
    ROOT / "data" / "splits" / "locked_four_omics_v1.csv",
    dtype=str,
    keep_default_na=False
)

if not (
    cases.case_id.is_unique
    and genes.ensembl_id.is_unique
    and locked.case_id.is_unique
):
    raise RuntimeError(
        "Duplicate TCGA patient or gene identifiers."
    )

merged = cases[
    [
        "case_id",
        "patient_id",
        "histology",
        "stage_group"
    ]
].merge(
    locked[
        [
            "case_id",
            "histology",
            "stage_group",
            "split"
        ]
    ],
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_matrix", "_locked")
)

for name in ("histology", "stage_group"):

    if not merged[
        f"{name}_matrix"
    ].equals(
        merged[f"{name}_locked"]
    ):
        raise RuntimeError(
            f"Locked clinical {name} mismatch."
        )

train_indices = np.flatnonzero(
    merged.split.eq("train").to_numpy()
)

val_indices = np.flatnonzero(
    merged.split.eq("validation").to_numpy()
)

test_indices = np.flatnonzero(
    merged.split.eq("test").to_numpy()
)

if (
    len(train_indices),
    len(val_indices),
    len(test_indices)
) != (667, 143, 143):
    raise RuntimeError(
        "Locked cohort sizes changed."
    )

if len(genes) != 19944:
    raise RuntimeError(
        "Stage 08 gene reference count changed."
    )

training = (
    merged.iloc[train_indices]
    .reset_index(drop=True)
    .copy()
)

training["patient_id"] = (
    training.patient_id.astype(str)
    .str.upper()
    .str.slice(0, 12)
)

if not training.patient_id.is_unique:
    raise RuntimeError(
        "Duplicated TCGA patient barcodes."
    )

site = training.patient_id.str.extract(
    r"^TCGA-([A-Z0-9]{2})-",
    expand=False
)

if site.isna().any():
    raise RuntimeError(
        "Malformed TCGA participant ID."
    )

# ============================================================
# 3. Download the verified TCGA-CDR survival reference
# ============================================================

valid_download = (
    CDR_PATH.is_file()
    and hashlib.md5(
        CDR_PATH.read_bytes()
    ).hexdigest() == CDR_MD5
)

if not valid_download:

    temporary = CDR_PATH.with_name(
        CDR_PATH.name + ".part"
    )

    if temporary.exists():
        temporary.unlink()

    last_problem = ""

    for url in CDR_SOURCES:

        try:

            print(
                "Downloading TCGA-CDR clinical reference:",
                url.split("/")[2],
                flush=True
            )

            with requests.get(
                url,
                stream=True,
                timeout=(25, 120)
            ) as response:

                response.raise_for_status()

                digest = hashlib.md5()

                with temporary.open("wb") as output:

                    for chunk in response.iter_content(
                        chunk_size=2 ** 20
                    ):

                        if chunk:
                            output.write(chunk)
                            digest.update(chunk)

            if digest.hexdigest() != CDR_MD5:
                raise RuntimeError(
                    "Downloaded workbook checksum mismatch."
                )

            os.replace(
                temporary,
                CDR_PATH
            )

            break

        except (
            requests.RequestException,
            RuntimeError,
            OSError
        ) as exc:

            last_problem = str(exc)

            if temporary.exists():
                temporary.unlink()

    else:

        raise RuntimeError(
            "The verified TCGA-CDR workbook could not "
            "be downloaded. Download "
            "TCGA-CDR-SupplementalTableS1.xlsx from "
            "the NCI GDC PanCan-Clinical-2018 page into "
            + str(CDR_PATH)
            + ". Last error: "
            + last_problem
        )

if hashlib.md5(
    CDR_PATH.read_bytes()
).hexdigest() != CDR_MD5:

    raise RuntimeError(
        "TCGA-CDR clinical workbook checksum mismatch."
    )

# ============================================================
# 4. Read standardized PFI and OS endpoints
# ============================================================

cdr = pd.read_excel(
    CDR_PATH,
    sheet_name="TCGA-CDR",
    engine="openpyxl"
)

required = (
    "bcr_patient_barcode",
    "type",
    "PFI",
    "PFI.time",
    "OS",
    "OS.time",
    "Redaction",
    "age_at_initial_pathologic_diagnosis",
    "gender"
)

missing = sorted(
    set(required) - set(cdr.columns)
)

if missing:
    raise RuntimeError(
        f"TCGA-CDR missing columns: {missing}"
    )

cdr["patient_id"] = (
    cdr.bcr_patient_barcode.astype(str)
    .str.upper()
    .str.slice(0, 12)
)

# Retain only previously locked training patients
cdr = cdr.loc[
    cdr.patient_id.isin(
        set(training.patient_id)
    )
    & cdr.type.astype(str)
    .str.upper()
    .isin(("LUAD", "LUSC"))
].copy()

if not cdr.patient_id.is_unique:
    raise RuntimeError(
        "Duplicate TCGA-CDR patient barcode."
    )

survival = training[
    [
        "case_id",
        "patient_id",
        "histology_matrix",
        "stage_group_matrix"
    ]
].merge(
    cdr,
    on="patient_id",
    how="left",
    validate="one_to_one"
)

survival["site"] = (
    site.to_numpy(dtype=str)
)

survival["histology_code"] = (
    survival.histology_matrix.eq("LUSC")
    .astype(int)
)

survival["stage_binary"] = (
    survival.stage_group_matrix.eq("Advanced")
    .astype(int)
)

for endpoint in ("PFI", "OS"):

    survival[endpoint] = pd.to_numeric(
        survival[endpoint],
        errors="coerce"
    )

    survival[f"{endpoint}.time"] = pd.to_numeric(
        survival[f"{endpoint}.time"],
        errors="coerce"
    )

# ============================================================
# 5. Verify pathological stage at diagnosis
# ============================================================

clinical_stage_path = (
    ROOT
    / "data"
    / "interim"
    / "tcga_clinical_stage_audit.csv"
)

if not clinical_stage_path.exists():
    raise FileNotFoundError(
        "Original Stage 17 clinical source required: "
        + str(clinical_stage_path)
    )

clinical_stage = pd.read_csv(
    clinical_stage_path,
    dtype=str,
    keep_default_na=False
)

if not {
    "case_id",
    "stage_original"
}.issubset(
    clinical_stage.columns
):
    raise RuntimeError(
        "Original pathological-stage fields are missing."
    )

clinical_stage = (
    clinical_stage[
        ["case_id", "stage_original"]
    ]
    .drop_duplicates("case_id")
)

if not clinical_stage.case_id.is_unique:
    raise RuntimeError(
        "Conflicting pathological stage records."
    )

survival = survival.merge(
    clinical_stage,
    on="case_id",
    how="left",
    validate="one_to_one"
)


def stage_major(value):

    match = re.search(
        r"STAGE\s*(IV|III|II|I)",
        str(value).upper()
    )

    if not match:
        return np.nan

    return {
        "I": 1,
        "II": 2,
        "III": 3,
        "IV": 4
    }[match.group(1)]


survival["stage_major"] = (
    survival.stage_original.map(
        stage_major
    )
)

if survival.stage_major.isna().any():
    raise RuntimeError(
        "Unknown AJCC stage; review clinical mappings."
    )

if (
    survival.stage_binary.to_numpy()
    != (
        survival.stage_major.to_numpy() >= 3
    )
).any():

    raise RuntimeError(
        "Stage group inconsistent with original AJCC stage."
    )

# ============================================================
# 6. Build baseline clinical covariates
# ============================================================

survival["age_years"] = pd.to_numeric(
    survival.age_at_initial_pathologic_diagnosis,
    errors="coerce"
)

survival.loc[
    ~survival.age_years.between(18, 105),
    "age_years"
] = np.nan

sex = (
    survival.gender.astype(str)
    .str.lower()
    .str.strip()
)

survival["male"] = np.where(
    sex.eq("male"),
    1.0,
    np.where(
        sex.eq("female"),
        0.0,
        np.nan
    )
)

survival["redacted"] = (
    survival.Redaction.astype(str)
    .str.strip()
    .str.lower()
    .eq("redacted")
)

for endpoint in ("PFI", "OS"):

    survival[f"{endpoint}_valid"] = (
        survival[endpoint].isin((0, 1))
        & np.isfinite(
            survival[f"{endpoint}.time"]
        )
        & (
            survival[f"{endpoint}.time"] > 0
        )
        & ~survival.redacted
    )

# Save only the necessary training clinical fields
survival[
    [
        "case_id",
        "patient_id",
        "histology_matrix",
        "stage_group_matrix",
        "site",
        "stage_original",
        "stage_major",
        "age_years",
        "male",
        "PFI",
        "PFI.time",
        "PFI_valid",
        "OS",
        "OS.time",
        "OS_valid",
        "redacted"
    ]
].to_csv(
    OUT / "stage21_training_clinical_endpoints.csv",
    index=False
)

print("=" * 75)
print(
    "STAGE 21 - SITE-HELD-OUT, "
    "CENSOR-AWARE NSCLC PROGNOSIS"
)
print("=" * 75)

print(
    "Original split: train=667, "
    "validation=143 untouched, test=143 sealed"
)

for endpoint in ("PFI", "OS"):

    take = survival[
        f"{endpoint}_valid"
    ]

    print(
        f"{endpoint}: eligible={take.sum()}, "
        f"events={int(survival.loc[take, endpoint].sum())}, "
        f"censored={int(take.sum() - survival.loc[take, endpoint].sum())}"
    )

    print(
        pd.crosstab(
            survival.loc[
                take,
                "histology_matrix"
            ],
            survival.loc[
                take,
                endpoint
            ],
            rownames=["Histology"],
            colnames=["Event (0=censored, 1=event)"]
        ).to_string()
    )

primary_ids = np.flatnonzero(
    survival.PFI_valid.to_numpy()
)

if (
    len(primary_ids) < 400
    or int(
        survival.loc[
            primary_ids,
            "PFI"
        ].sum()
    ) < 60
):

    raise RuntimeError(
        "Insufficient PFI cases/events "
        "for the prespecified survival study."
    )

# ============================================================
# 7. Reconstruct original Reactome RNA pathway features
# ============================================================

W = sparse.load_npz(
    STAGE19 / "reactome_gene_pathway_weights.npz"
).tocsr().astype(
    np.float32
)

if (
    W.shape[1] != len(genes)
    or W.shape[0] < 100
):
    raise RuntimeError(
        "Reactome mapping dimensions mismatch."
    )

rna_source = np.load(
    RAW / "rna_tpm.npy",
    mmap_mode="r"
)

if rna_source.shape != (
    len(cases),
    len(genes)
):
    raise RuntimeError(
        "Stage 08 RNA dimensions changed."
    )

rna = np.asarray(
    rna_source[train_indices],
    dtype=np.float32
).copy()

del rna_source

observed = (
    np.isfinite(rna)
    & (rna >= 0)
)

if observed.mean() < 0.95:
    raise RuntimeError(
        "Unexpectedly low RNA completeness."
    )

rna = np.where(
    observed,
    np.log1p(
        np.maximum(rna, 0)
    ),
    0.0
)

ranks = rankdata(
    rna,
    axis=1,
    method="average"
).astype(
    np.float32
)

ranks = (
    ranks - (len(genes) + 1) / 2.0
) / len(genes)

Xrna = np.asarray(
    (W @ ranks.T).T,
    dtype=np.float32
)

del ranks, rna, observed

if (
    not np.isfinite(Xrna).all()
    or Xrna.shape[0] != len(survival)
):
    raise RuntimeError(
        "Invalid training pathway representation."
    )

print(
    "Training Reactome RNA features:",
    Xrna.shape
)

# ============================================================
# 8. Ridge-penalized Cox proportional hazards regression
# ============================================================

def fit_ridge_cox(
    x,
    times,
    events,
    alpha=0.03,
    clinical_dim=0
):

    x = np.asarray(
        x,
        dtype=np.float64
    )

    times = np.asarray(
        times,
        dtype=np.float64
    )

    events = np.asarray(
        events,
        dtype=np.int64
    )

    if (
        not np.isfinite(x).all()
        or not np.isfinite(times).all()
        or set(np.unique(events)) - {0, 1}
        or events.sum() < 8
    ):

        raise RuntimeError(
            "Invalid Cox training data."
        )

    # Descending time order gives cumulative risk sets
    order = np.argsort(
        -times,
        kind="stable"
    )

    z = x[order]
    t = times[order]
    e = events[order]

    boundaries = np.r_[
        0,
        np.flatnonzero(
            t[1:] != t[:-1]
        ) + 1
    ]

    ends = np.r_[
        boundaries[1:] - 1,
        len(t) - 1
    ]

    event_counts = np.add.reduceat(
        e,
        boundaries
    )

    relevant = event_counts > 0

    ends = ends[relevant]

    event_counts = (
        event_counts[relevant]
        .astype(float)
    )

    n_events = max(
        int(events.sum()),
        1
    )

    penalty = np.full(
        z.shape[1],
        float(alpha)
    )

    if clinical_dim:
        penalty[:clinical_dim] = (
            float(alpha) * 0.15
        )

    def objective(beta):

        eta = z @ beta

        offset = np.max(eta)

        expeta = np.exp(
            eta - offset
        )

        cumul_w = np.cumsum(
            expeta
        )

        cumul_x = np.cumsum(
            expeta[:, None] * z,
            axis=0
        )

        risk_sums = np.maximum(
            cumul_w[ends],
            1e-300
        )

        loss = (
            np.sum(
                event_counts
                * (
                    offset
                    + np.log(risk_sums)
                )
            )
            - np.sum(
                eta[e == 1]
            )
        ) / n_events

        gradient = (
            np.sum(
                event_counts[:, None]
                * (
                    cumul_x[ends]
                    / risk_sums[:, None]
                ),
                axis=0
            )
            - np.sum(
                z[e == 1],
                axis=0
            )
        ) / n_events

        penalty_loss = (
            0.5
            * (
                penalty * beta * beta
            ).sum()
        )

        penalty_gradient = (
            penalty * beta
        )

        return (
            float(
                loss + penalty_loss
            ),
            np.asarray(
                gradient + penalty_gradient,
                dtype=float
            )
        )

    optimized = minimize(
        objective,
        np.zeros(x.shape[1]),
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter": 300,
            "ftol": 1e-9,
            "gtol": 1e-6
        }
    )

    if (
        not np.isfinite(
            optimized.x
        ).all()
        or not np.isfinite(
            optimized.fun
        )
    ):
        raise RuntimeError(
            "Cox optimization diverged."
        )

    return optimized.x.astype(
        np.float64
    )


# ============================================================
# 9. Harrell concordance index
# ============================================================

def c_index(
    time,
    event,
    risk
):

    time = np.asarray(
        time,
        dtype=float
    )

    event = np.asarray(
        event,
        dtype=int
    )

    risk = np.asarray(
        risk,
        dtype=float
    )

    earlier_event = (
        (time[:, None] < time[None, :])
        & (event[:, None] == 1)
    )

    comparable = int(
        earlier_event.sum()
    )

    if comparable < 10:
        return np.nan, comparable

    difference = (
        risk[:, None]
        - risk[None, :]
    )

    concordance = (
        np.sum(
            (difference > 1e-10)
            & earlier_event
        )
        + 0.5
        * np.sum(
            (np.abs(difference) <= 1e-10)
            & earlier_event
        )
    )

    return (
        float(
            concordance / comparable
        ),
        comparable
    )


# ============================================================
# 10. Select prognostic pathways using training-fold Cox scores
# ============================================================

def clinical_adjusted_scores(
    z,
    time,
    event,
    baseline_risk
):

    x = np.asarray(
        z,
        dtype=float
    )

    order = np.argsort(
        -time,
        kind="stable"
    )

    t = time[order]

    e = event[
        order
    ].astype(int)

    risk = baseline_risk[
        order
    ]

    x = x[order]

    w = np.exp(
        risk - np.max(risk)
    )

    csum_w = np.cumsum(w)

    csum_x = np.cumsum(
        w[:, None] * x,
        axis=0
    )

    csum_xx = np.cumsum(
        w[:, None] * x * x,
        axis=0
    )

    starts = np.r_[
        0,
        np.flatnonzero(
            t[1:] != t[:-1]
        ) + 1
    ]

    ends = np.r_[
        starts[1:] - 1,
        len(t) - 1
    ]

    d = np.add.reduceat(
        e,
        starts
    )

    valid = d > 0

    ends = ends[valid]
    d = d[valid]

    expected = (
        csum_x[ends]
        / np.maximum(
            csum_w[ends, None],
            1e-300
        )
    )

    second = (
        csum_xx[ends]
        / np.maximum(
            csum_w[ends, None],
            1e-300
        )
    )

    u = (
        (e[:, None] * x).sum(axis=0)
        - (
            d[:, None] * expected
        ).sum(axis=0)
    )

    variance = (
        d[:, None]
        * np.maximum(
            second - expected * expected,
            0
        )
    ).sum(axis=0)

    return (
        np.abs(u)
        / np.sqrt(
            np.maximum(
                variance,
                1e-10
            )
        )
    )


# ============================================================
# 11. Fold-specific clinical preprocessing
# ============================================================

def get_clinical(
    fit,
    hold
):

    age = survival.age_years.to_numpy(
        dtype=float
    )

    sex = survival.male.to_numpy(
        dtype=float
    )

    stage = survival.stage_major.to_numpy(
        dtype=float
    )

    hist = survival.histology_code.to_numpy(
        dtype=float
    )

    valid_age = np.isfinite(
        age[fit]
    )

    med_age = (
        float(
            np.median(
                age[fit][valid_age]
            )
        )
        if valid_age.any()
        else 65.0
    )

    def block(idx):

        a = np.where(
            np.isfinite(
                age[idx]
            ),
            age[idx],
            med_age
        )

        return np.column_stack([
            (a - med_age) / 10.0,
            (
                ~np.isfinite(age[idx])
            ).astype(float),
            np.where(
                np.isfinite(sex[idx]),
                sex[idx],
                0.5
            ),
            (
                ~np.isfinite(sex[idx])
            ).astype(float),
            hist[idx],
            (stage[idx] == 2).astype(float),
            (stage[idx] == 3).astype(float),
            (stage[idx] == 4).astype(float)
        ]).astype(float)

    a = block(fit)
    b = block(hold)

    scaler = StandardScaler().fit(a)

    return (
        scaler.transform(a),
        scaler.transform(b),
        (med_age, scaler)
    )


# ============================================================
# 12. Fit and predict a survival model
# ============================================================

def train_survival_model(
    fit,
    hold,
    spec,
    endpoint="PFI"
):

    fit = np.asarray(
        fit,
        dtype=int
    )

    hold = np.asarray(
        hold,
        dtype=int
    )

    time = survival[
        f"{endpoint}.time"
    ].to_numpy(
        dtype=float
    )

    events = survival[
        endpoint
    ].to_numpy(
        dtype=int
    )

    clinical_fit, clinical_hold, clinical_state = (
        get_clinical(
            fit,
            hold
        )
    )

    state = {
        "spec": spec,
        "clinical": clinical_state,
        "pathway_scaler": None,
        "selection": None,
        "projection": None,
        "latent_scaler": None
    }

    dim_clinical = (
        clinical_fit.shape[1]
        if spec["clinical"]
        else 0
    )

    if spec["kind"] != "clinical":

        molecular_scaler = (
            StandardScaler().fit(
                Xrna[fit]
            )
        )

        zfit = np.clip(
            molecular_scaler.transform(
                Xrna[fit]
            ),
            -7,
            7
        )

        zhold = np.clip(
            molecular_scaler.transform(
                Xrna[hold]
            ),
            -7,
            7
        )

        state["pathway_scaler"] = (
            molecular_scaler
        )

        if spec["kind"] == "svd":

            projection = TruncatedSVD(
                n_components=spec["k"],
                n_iter=4,
                random_state=SEED
            ).fit(zfit)

            zfit = projection.transform(
                zfit
            )

            zhold = projection.transform(
                zhold
            )

            state["projection"] = projection

        elif spec["kind"] == "score":

            baseline_beta = fit_ridge_cox(
                clinical_fit,
                time[fit],
                events[fit],
                alpha=0.003
            )

            strength = clinical_adjusted_scores(
                zfit,
                time[fit],
                events[fit],
                clinical_fit @ baseline_beta
            )

            selected = np.sort(
                np.argsort(
                    -strength,
                    kind="stable"
                )[:spec["k"]]
            )

            zfit = zfit[
                :,
                selected
            ]

            zhold = zhold[
                :,
                selected
            ]

            state["selection"] = selected

        latent_scaler = (
            StandardScaler().fit(
                zfit
            )
        )

        zfit = np.clip(
            latent_scaler.transform(zfit),
            -8,
            8
        )

        zhold = np.clip(
            latent_scaler.transform(zhold),
            -8,
            8
        )

        state["latent_scaler"] = (
            latent_scaler
        )

        if spec["clinical"]:

            zfit = np.column_stack([
                clinical_fit,
                zfit
            ])

            zhold = np.column_stack([
                clinical_hold,
                zhold
            ])

    else:

        zfit = clinical_fit
        zhold = clinical_hold

    coef = fit_ridge_cox(
        zfit,
        time[fit],
        events[fit],
        spec["alpha"],
        clinical_dim=dim_clinical
    )

    state["coefficients"] = coef

    return (
        zhold @ coef,
        state
    )


# ============================================================
# 13. Construct patient-disjoint and site-disjoint folds
# ============================================================

def candidate_splits(
    ids,
    groups,
    event,
    histology,
    k,
    seeds
):

    classification = (
        event[ids] * 2
        + histology[ids]
    )

    options = []

    for seed in seeds:

        try:

            splits = list(
                StratifiedGroupKFold(
                    n_splits=k,
                    shuffle=True,
                    random_state=seed
                ).split(
                    np.arange(len(ids)),
                    classification,
                    groups[ids]
                )
            )

        except ValueError:
            continue

        valid = True
        deviation = 0.0
        complete = []

        for local_fit, local_hold in splits:

            fit = ids[local_fit]
            hold = ids[local_hold]

            if (
                set(groups[fit])
                & set(groups[hold])
                or event[fit].sum() < 30
                or event[hold].sum() < 8
                or len(
                    np.unique(event[hold])
                ) != 2
                or set(
                    np.unique(
                        histology[hold]
                    )
                ) != {0, 1}
            ):

                valid = False
                break

            deviation += abs(
                len(hold) / len(ids)
                - 1 / k
            )

            deviation += abs(
                event[hold].mean()
                - event[ids].mean()
            )

            complete.append(
                (fit, hold)
            )

        if valid:

            options.append(
                (
                    deviation,
                    seed,
                    complete
                )
            )

    if not options:
        raise RuntimeError(
            "Cannot construct site-disjoint folds "
            "with sufficient outcome events."
        )

    return min(
        options,
        key=lambda item: (
            item[0],
            item[1]
        )
    )[1:]


# ============================================================
# 14. Prepare eligible PFI cases and outer site-held-out CV
# ============================================================

site_all = survival.site.to_numpy(
    dtype=str
)

event_all = survival.PFI.to_numpy(
    dtype=float
)

time_all = survival[
    "PFI.time"
].to_numpy(
    dtype=float
)

hist_all = survival.histology_code.to_numpy(
    dtype=int
)

outer_seed, outer = candidate_splits(
    primary_ids,
    site_all,
    np.nan_to_num(
        event_all
    ).astype(int),
    hist_all,
    OUTER_FOLDS,
    (
        42, 49, 61, 73,
        89, 97, 109, 127
    )
)

print(
    f"PFI outer site folds={len(outer)}; "
    f"split seed={outer_seed}"
)

# ============================================================
# 15. Fingerprint the experiment
# ============================================================

fingerprint = hashlib.sha256(
    json.dumps(
        {
            "version": VERSION,
            "cases": survival.case_id.tolist(),
            "PFI_time": np.nan_to_num(
                time_all,
                nan=-1
            ).tolist(),
            "PFI_event": np.nan_to_num(
                event_all,
                nan=-1
            ).tolist(),
            "histology": (
                survival.histology_matrix.tolist()
            ),
            "source_stage": (
                survival.stage_original.astype(str).tolist()
            ),
            "site": site_all.tolist(),
            "sources": {
                str(path): (
                    path.stat().st_size,
                    path.stat().st_mtime_ns
                )
                for path in (
                    CDR_PATH,
                    RAW / "rna_tpm.npy",
                    STAGE19 / "reactome_gene_pathway_weights.npz",
                    clinical_stage_path
                )
            },
            "specifications": SPECS,
            "outer_seed": outer_seed
        },
        sort_keys=True
    ).encode()
).hexdigest()

fold_results = []
inner_results = []
fold_outputs = []

clinical_spec = SPECS[0]

# ============================================================
# 16. Nested site-held-out Cox modeling
# ============================================================

with threadpool_limits(limits=THREADS):

    for j, (
        outer_fit,
        outer_held
    ) in enumerate(
        outer,
        start=1
    ):

        checkpoint = (
            OUT / f"pfi_outer_fold{j}.joblib"
        )

        if checkpoint.exists():

            cached = joblib.load(
                checkpoint
            )

            if (
                cached.get("fingerprint")
                != fingerprint
                or not np.array_equal(
                    cached.get("held"),
                    outer_held
                )
            ):

                raise RuntimeError(
                    f"Incompatible saved fold: {checkpoint}"
                )

            fold_result = cached[
                "fold_result"
            ]

            inner = cached["inner"]

            predictions = cached[
                "predictions"
            ]

            print(
                f"Outer fold {j}: checkpoint reused",
                flush=True
            )

        else:

            inner_seed, inner_folds = candidate_splits(
                outer_fit,
                site_all,
                np.nan_to_num(
                    event_all
                ).astype(int),
                hist_all,
                INNER_FOLDS,
                (
                    42, 59, 73,
                    89, 101, 127
                )
            )

            inner = []
            candidate_quality = {}

            print(
                f"\nPFI OUTER {j}: "
                f"fit={len(outer_fit)}, "
                f"held={len(outer_held)}, "
                f"events={int(event_all[outer_held].sum())}; "
                f"inner seed={inner_seed}",
                flush=True
            )

            for spec in SPECS:

                scores = []

                for k, (
                    fit,
                    held
                ) in enumerate(
                    inner_folds,
                    start=1
                ):

                    risk, _ = train_survival_model(
                        fit,
                        held,
                        spec,
                        endpoint="PFI"
                    )

                    concordance, pairs = c_index(
                        time_all[held],
                        event_all[held],
                        risk
                    )

                    scores.append(
                        concordance
                    )

                    inner.append({
                        "outer_fold": j,
                        "inner_fold": k,
                        "model": spec["name"],
                        "C_index": concordance,
                        "comparable_pairs": pairs
                    })

                candidate_quality[
                    spec["name"]
                ] = float(
                    np.nanmean(scores)
                )

            # Select by inner site-CV only
            winner = max(
                SPECS,
                key=lambda spec: (
                    candidate_quality[
                        spec["name"]
                    ],
                    -SPECS.index(spec)
                )
            )

            risk_best, selected_state = (
                train_survival_model(
                    outer_fit,
                    outer_held,
                    winner,
                    endpoint="PFI"
                )
            )

            risk_clinical, _ = (
                train_survival_model(
                    outer_fit,
                    outer_held,
                    clinical_spec,
                    endpoint="PFI"
                )
            )

            ci_best, comparable = c_index(
                time_all[outer_held],
                event_all[outer_held],
                risk_best
            )

            ci_clinical, _ = c_index(
                time_all[outer_held],
                event_all[outer_held],
                risk_clinical
            )

            fold_result = {
                "fold": j,
                "held_patients": len(outer_held),
                "PFI_events": int(
                    event_all[outer_held].sum()
                ),
                "sites_held": len(
                    set(site_all[outer_held])
                ),
                "winner": winner["name"],
                "nested_PFI_C": ci_best,
                "clinical_PFI_C": ci_clinical,
                "difference": (
                    ci_best - ci_clinical
                ),
                "comparable_pairs": comparable
            }

            predictions = {
                "index": outer_held,
                "PFI_risk": risk_best,
                "clinical_risk": risk_clinical
            }

            # Secondary OS evaluation
            valid_os = survival.OS_valid.to_numpy(
                dtype=bool
            )

            os_fit = outer_fit[
                valid_os[outer_fit]
            ]

            os_hold = outer_held[
                valid_os[outer_held]
            ]

            fold_result["OS_events_held"] = int(
                survival.OS.iloc[
                    os_hold
                ].sum()
            )

            if (
                len(os_fit) >= 100
                and survival.OS.iloc[
                    os_fit
                ].sum() >= 25
                and len(os_hold) >= 15
                and survival.OS.iloc[
                    os_hold
                ].sum() >= 5
            ):

                os_t = survival[
                    "OS.time"
                ].to_numpy(
                    dtype=float
                )

                os_e = survival.OS.to_numpy(
                    dtype=float
                )

                os_best, _ = train_survival_model(
                    os_fit,
                    os_hold,
                    winner,
                    endpoint="OS"
                )

                os_base, _ = train_survival_model(
                    os_fit,
                    os_hold,
                    clinical_spec,
                    endpoint="OS"
                )

                fold_result["nested_OS_C"] = (
                    c_index(
                        os_t[os_hold],
                        os_e[os_hold],
                        os_best
                    )[0]
                )

                fold_result["clinical_OS_C"] = (
                    c_index(
                        os_t[os_hold],
                        os_e[os_hold],
                        os_base
                    )[0]
                )

            else:

                fold_result["nested_OS_C"] = np.nan
                fold_result["clinical_OS_C"] = np.nan

            temporary = checkpoint.with_name(
                checkpoint.name + ".tmp"
            )

            joblib.dump(
                {
                    "fingerprint": fingerprint,
                    "held": outer_held,
                    "inner": inner,
                    "fold_result": fold_result,
                    "predictions": predictions
                },
                temporary,
                compress=3
            )

            os.replace(
                temporary,
                checkpoint
            )

            print(
                f"  selected={winner['name']} | "
                f"PFI C={ci_best:.4f} "
                f"vs clinical C={ci_clinical:.4f} | "
                f"OS C={fold_result['nested_OS_C']:.4f}",
                flush=True
            )

        fold_results.append(
            fold_result
        )

        inner_results.extend(
            inner
        )

        fold_outputs.append(
            predictions
        )

# ============================================================
# 17. Save fold-level prognostic evaluations
# ============================================================

fold_table = pd.DataFrame(
    fold_results
).sort_values("fold")

inner_table = pd.DataFrame(
    inner_results
)

fold_table.to_csv(
    OUT / "site_nested_pfi_os_fold_results.csv",
    index=False
)

inner_table.to_csv(
    OUT / "inner_site_survival_model_comparison.csv",
    index=False
)

# ============================================================
# 18. Select and refit the training-only prognostic model
# ============================================================

inner_ranking = (
    inner_table.groupby(
        "model",
        as_index=False
    )
    .C_index.mean()
    .rename(
        columns={
            "C_index": "mean_inner_C"
        }
    )
    .sort_values(
        "mean_inner_C",
        ascending=False
    )
)

inner_ranking.to_csv(
    OUT / "training_inner_model_ranking.csv",
    index=False
)

final_name = str(
    inner_ranking.iloc[0].model
)

final_spec = next(
    spec
    for spec in SPECS
    if spec["name"] == final_name
)

with threadpool_limits(limits=THREADS):

    _, final_state = train_survival_model(
        primary_ids,
        primary_ids,
        final_spec,
        endpoint="PFI"
    )

final_path = (
    MODEL_DIR
    / "stage21_pfi_frozen_training_model.joblib"
)

temporary = final_path.with_name(
    final_path.name + ".tmp"
)

joblib.dump(
    {
        "version": VERSION,
        "fingerprint": fingerprint,
        "source": "Liu et al 2018 TCGA-CDR",
        "endpoint": "PFI",
        "specification": final_spec,
        "fitted_components": final_state,
        "eligible_training_case_ids": (
            survival.case_id.iloc[
                primary_ids
            ].tolist()
        ),
        "validation_test_unused": True
    },
    temporary,
    compress=3
)

os.replace(
    temporary,
    final_path
)

# ============================================================
# 19. Save reproducibility summary
# ============================================================

summary = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "fingerprint": fingerprint,

    "train_total": len(survival),

    "PFI_eligible_train": len(primary_ids),

    "PFI_events_train": int(
        survival.PFI.iloc[
            primary_ids
        ].sum()
    ),

    "validation_not_evaluated": len(val_indices),

    "test_not_evaluated": len(test_indices),

    "outer_site_seed": outer_seed,

    "PFI_nested_mean_C": float(
        fold_table.nested_PFI_C.mean()
    ),

    "PFI_clinical_mean_C": float(
        fold_table.clinical_PFI_C.mean()
    ),

    "PFI_mean_delta_C": float(
        fold_table.difference.mean()
    ),

    "OS_nested_mean_C": float(
        fold_table.nested_OS_C.mean()
    ),

    "OS_clinical_mean_C": float(
        fold_table.clinical_OS_C.mean()
    ),

    "outer_winners": (
        fold_table.winner.value_counts().to_dict()
    ),

    "final_training_inner_selected_model": final_name,

    "limitations": [
        (
            "Time-to-event prognosis is a different "
            "research endpoint from stage classification."
        ),
        (
            "TCGA-CDR uses a frozen clinical "
            "follow-up snapshot from 2018."
        ),
        (
            "Pathological stage at diagnosis is a "
            "predictor, not the outcome."
        ),
        (
            "Harrell's C-index can be affected "
            "by censoring; time-dependent "
            "calibration is not yet evaluated."
        ),
        (
            "Internal site-held-out CV is not "
            "independent external validation."
        ),
        (
            "Repeated experiments on the same "
            "TCGA cohort create adaptive "
            "method-development risk."
        ),
        (
            "Molecular incremental value requires "
            "improvement over the clinical-only "
            "model in matched held-out folds."
        )
    ]
}

(
    OUT / "stage21_summary.json"
).write_text(
    json.dumps(
        summary,
        indent=2
    ),
    encoding="utf-8"
)

# ============================================================
# 20. Final reporting
# ============================================================

print("\n" + "=" * 75)

print(
    "STAGE 21 FINAL - "
    "SITE-HELD-OUT PFI AND OS PROGNOSIS"
)

print("=" * 75)

print(
    fold_table.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print("\nTraining inner model ranking:")

print(
    inner_ranking.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print(
    "\nPFI nested mean C-index:",
    round(
        summary["PFI_nested_mean_C"],
        4
    )
)

print(
    "PFI clinical-only mean C-index:",
    round(
        summary["PFI_clinical_mean_C"],
        4
    )
)

print(
    "Mean incremental PFI C-index:",
    round(
        summary["PFI_mean_delta_C"],
        4
    )
)

print(
    "OS secondary mean C-index:",
    round(
        summary["OS_nested_mean_C"],
        4
    )
)

print(
    "OS clinical-only mean C-index:",
    round(
        summary["OS_clinical_mean_C"],
        4
    )
)

print(
    "Final PFI model from training inner CV:",
    final_name
)

print(
    "Saved model:",
    final_path
)

print(
    "Validation/test molecular features "
    "and labels used for optimization: NO"
)

print("STAGE 21 COMPLETED")

STAGE 21 - SITE-HELD-OUT, CENSOR-AWARE NSCLC PROGNOSIS
Original split: train=667, validation=143 untouched, test=143 sealed
PFI: eligible=657, events=225, censored=432
Event (0=censored, 1=event)  0.0  1.0
Histology                            
LUAD                         197  135
LUSC                         235   90
OS: eligible=657, events=250, censored=407
Event (0=censored, 1=event)  0.0  1.0
Histology                            
LUAD                         215  117
LUSC                         192  133
Training Reactome RNA features: (667, 1613)
PFI outer site folds=4; split seed=42

PFI OUTER 1: fit=491, held=166, events=56; inner seed=73
  selected=Clinical_RNA_SVD16_Cox | PFI C=0.6260 vs clinical C=0.6489 | OS C=0.5639

PFI OUTER 2: fit=494, held=163, events=55; inner seed=42
  selected=Clinical_RNA_SVD16_Cox | PFI C=0.6630 vs clinical C=0.6421 | OS C=0.5914

PFI OUTER 3: fit=493, held=164, events=57; inner seed=59
  selected=Clinical_Cox | PFI C=0.6660 vs clinical C=0.6660 |

In [21]:

import json
import os
import random
import sys
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import joblib
import networkx as nx
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

from scipy import sparse
from sklearn.model_selection import StratifiedGroupKFold

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

RAW = ROOT / "data" / "processed" / "gene_level"
OLD = ROOT / "data" / "processed" / "ppi_gat_v1"
S21 = ROOT / "results" / "metrics" / "stage21"
OUT = ROOT / "results" / "metrics" / "stage22"
CKPT = ROOT / "models" / "checkpoints" / "stage22"

SEED = 42
N_GENE = 512
EPOCHS = 18
PATIENCE = 5
BATCH = 32
THREADS = min(4, os.cpu_count() or 1)

VERSION = "stage22_pfi_graph_attention_pilot_v1"

EXPERIMENTS = (
    "graphfree",
    "functional",
    "degree_preserving_null"
)


# Load the locked cohort and Stage 21 survival endpoints.
def load_project():

    required = (
        RAW / "case_order.csv",
        RAW / "gene_reference.csv",
        OLD / "selected_gene_nodes.csv",
        OLD / "string_functional_700_corrected.npz",
        S21 / "stage21_training_clinical_endpoints.csv"
    )

    for path in required:
        if not path.is_file():
            raise FileNotFoundError(str(path))

    cases = pd.read_csv(
        RAW / "case_order.csv",
        dtype={"case_id": str}
    )

    reference = pd.read_csv(
        RAW / "gene_reference.csv",
        dtype={"ensembl_id": str}
    )

    nodes = (
        pd.read_csv(
            OLD / "selected_gene_nodes.csv",
            dtype={"ensembl_id": str}
        )
        .sort_values("node_index")
    )

    locked = pd.read_csv(
        ROOT / "data" / "splits"
        / "locked_four_omics_v1.csv",
        dtype={"case_id": str}
    )

    assigned = cases[
        ["case_id", "patient_id", "histology", "stage_group"]
    ].merge(
        locked[
            ["case_id", "histology", "stage_group", "split"]
        ],
        on="case_id",
        validate="one_to_one",
        how="left",
        suffixes=("_a", "_b")
    )

    if (
        not assigned.histology_a.equals(assigned.histology_b)
        or not assigned.stage_group_a.equals(
            assigned.stage_group_b
        )
    ):
        raise RuntimeError(
            "Locked patient labels changed."
        )

    tr = np.flatnonzero(
        assigned.split.eq("train").to_numpy()
    )

    va = np.flatnonzero(
        assigned.split.eq("validation").to_numpy()
    )

    te = np.flatnonzero(
        assigned.split.eq("test").to_numpy()
    )

    if (
        len(tr),
        len(va),
        len(te)
    ) != (667, 143, 143):
        raise RuntimeError(
            "Locked split sizes changed."
        )

    clinical = pd.read_csv(
        S21 / "stage21_training_clinical_endpoints.csv",
        dtype={"case_id": str}
    )

    if (
        len(clinical) != len(tr)
        or not np.array_equal(
            clinical.case_id.to_numpy(),
            assigned.case_id.iloc[tr].to_numpy()
        )
    ):
        raise RuntimeError(
            "Stage 21 clinical alignment mismatch."
        )

    valid = (
        clinical.PFI_valid
        .astype(str)
        .str.lower()
        .eq("true")
        .to_numpy()
    )

    eligible = np.flatnonzero(valid)

    times = pd.to_numeric(
        clinical["PFI.time"],
        errors="coerce"
    ).to_numpy(dtype=np.float64)

    events = pd.to_numeric(
        clinical.PFI,
        errors="coerce"
    ).to_numpy(dtype=np.float64)

    if (
        len(eligible) < 600
        or int(events[eligible].sum()) < 180
        or not np.isfinite(times[eligible]).all()
        or (times[eligible] <= 0).any()
    ):
        raise RuntimeError(
            "PFI endpoint or event count changed."
        )

    histology = clinical.histology_matrix.eq(
        "LUSC"
    ).to_numpy(dtype=int)

    sites = clinical.site.astype(str).to_numpy()

    if not np.array_equal(
        nodes.node_index.to_numpy(dtype=int),
        np.arange(1600)
    ):
        raise RuntimeError(
            "Selected gene node order changed."
        )

    source_indices = pd.Index(
        reference.ensembl_id.astype(str)
    ).get_indexer(
        nodes.ensembl_id.astype(str)
    )

    if (source_indices < 0).any():
        raise RuntimeError(
            "Selected genes missing from Stage 08."
        )

    # Reconstruct the corrected functional PPI network.
    with np.load(
        OLD / "string_functional_700_corrected.npz"
    ) as packed:

        edge_index = np.asarray(
            packed["edge_index"],
            dtype=np.int64
        )

        weights = np.asarray(
            packed["edge_weight"],
            dtype=np.float32
        )

    if (
        edge_index.shape[0] != 2
        or edge_index.max() >= 1600
        or edge_index.min() < 0
        or not np.isfinite(weights).all()
    ):
        raise RuntimeError(
            "Invalid STRING functional graph."
        )

    adjacency = sparse.csr_matrix(
        (
            weights,
            (edge_index[0], edge_index[1])
        ),
        shape=(1600, 1600)
    )

    adjacency.setdiag(0)
    adjacency.eliminate_zeros()
    adjacency.sum_duplicates()

    asymmetry = adjacency - adjacency.T

    if (
        asymmetry.nnz
        and np.max(
            np.abs(asymmetry.data)
        ) > 1e-4
    ):
        raise RuntimeError(
            "Asymmetric STRING graph."
        )

    # Select a fixed, connectivity-guided subgraph.
    degrees = np.asarray(
        (adjacency > 0).sum(axis=0)
    ).ravel()

    selected = np.sort(
        np.lexsort(
            (np.arange(1600), -degrees)
        )[:N_GENE]
    )

    positions = source_indices[selected]

    subgraph = adjacency[
        selected, :
    ][:, selected].tocsr()

    subgraph.setdiag(0)
    subgraph.eliminate_zeros()

    edge_count = subgraph.nnz // 2

    if edge_count < 300:
        raise RuntimeError(
            "Selected PPI graph is unexpectedly sparse."
        )

    graph = nx.from_scipy_sparse_array(
        subgraph,
        create_using=nx.Graph()
    )

    graph_null = graph.copy()

    nx.double_edge_swap(
        graph_null,
        nswap=edge_count * 3,
        max_tries=edge_count * 100,
        seed=SEED + 22
    )

    if any(
        graph.degree(i) != graph_null.degree(i)
        for i in range(N_GENE)
    ):
        raise RuntimeError(
            "Randomized graph changed node degrees."
        )

    true_edges = np.array(
        list(graph.edges()),
        dtype=np.int64
    )

    true_weights = np.array(
        [
            graph[u][v]["weight"]
            for u, v in graph.edges()
        ],
        dtype=np.float32
    )

    null_edges = np.array(
        list(graph_null.edges()),
        dtype=np.int64
    )

    null_weights = true_weights.copy()

    rng = np.random.default_rng(
        SEED + 22
    )

    rng.shuffle(null_weights)

    def make_edges(edges, values):

        source = np.r_[
            edges[:, 0],
            edges[:, 1],
            np.arange(N_GENE)
        ].astype(np.int64)

        target = np.r_[
            edges[:, 1],
            edges[:, 0],
            np.arange(N_GENE)
        ].astype(np.int64)

        scaled = np.r_[
            values / values.max(),
            values / values.max(),
            np.ones(N_GENE)
        ].astype(np.float32)

        return (
            torch.from_numpy(source),
            torch.from_numpy(target),
            torch.from_numpy(scaled)
        )

    self_indices = torch.arange(
        N_GENE,
        dtype=torch.long
    )

    graphs = {
        "graphfree": (
            self_indices,
            self_indices,
            torch.ones(N_GENE)
        ),
        "functional": make_edges(
            true_edges,
            true_weights
        ),
        "degree_preserving_null": make_edges(
            null_edges,
            null_weights
        )
    }

    print(
        "Selected graph nodes:",
        N_GENE
    )

    print(
        "Functional undirected edges:",
        edge_count
    )

    print(
        "Degree-preserving randomized control: PASSED"
    )

    # Read only the locked training rows from Stage 08.
    rna = np.load(
        RAW / "rna_tpm.npy",
        mmap_mode="r"
    )

    cnv = np.load(
        RAW / "cnv_absolute.npy",
        mmap_mode="r"
    )

    if (
        rna.shape != (953, 19944)
        or cnv.shape != (953, 19944)
    ):
        raise RuntimeError(
            "Stage 08 source matrix dimensions changed."
        )

    r = np.asarray(
        rna[np.ix_(tr, positions)],
        dtype=np.float32
    )

    c = np.asarray(
        cnv[np.ix_(tr, positions)],
        dtype=np.float32
    )

    r_valid = (
        np.isfinite(r) & (r >= 0)
    )

    c_valid = (
        np.isfinite(c) & (c >= 0)
    )

    r = np.where(
        r_valid,
        np.log1p(
            np.maximum(r, 0)
        ),
        np.nan
    )

    c = np.where(
        c_valid,
        np.clip(
            np.log2(
                (
                    np.maximum(c, 0) + 0.5
                ) / 2.5
            ),
            -3,
            3
        ),
        np.nan
    )

    raw_features = np.stack(
        (r, c),
        axis=-1
    ).astype(np.float32)

    if (
        np.isfinite(
            raw_features[:, :, 0]
        ).mean() <= 0.95
    ):
        raise RuntimeError(
            "Unexpected RNA missingness."
        )

    return (
        raw_features,
        clinical,
        times,
        events,
        histology,
        sites,
        eligible,
        graphs
    )


# Fit molecular and clinical scaling on the fitting fold.
def prepare_features(
    raw,
    clinical,
    fit,
    held
):

    xfit = raw[fit]
    xheld = raw[held]

    with np.errstate(all="ignore"):

        center = np.nanmean(
            xfit,
            axis=0
        )

        center = np.nan_to_num(
            center,
            nan=0.0
        )

        scale = np.nanstd(
            xfit,
            axis=0
        )

        scale = np.maximum(
            np.nan_to_num(
                scale,
                nan=0.0
            ),
            0.25
        )

    def molecular(x):

        filled = np.where(
            np.isfinite(x),
            x,
            center
        )

        return np.clip(
            (filled - center) / scale,
            -5,
            5
        ).astype(np.float32)

    age = pd.to_numeric(
        clinical.age_years,
        errors="coerce"
    ).to_numpy(dtype=float)

    male = pd.to_numeric(
        clinical.male,
        errors="coerce"
    ).to_numpy(dtype=float)

    stage = pd.to_numeric(
        clinical.stage_major,
        errors="coerce"
    ).to_numpy(dtype=int)

    histology = (
        clinical.histology_matrix.eq("LUSC")
        .to_numpy(dtype=float)
    )

    observed_ages = age[fit][
        np.isfinite(age[fit])
    ]

    median_age = (
        float(np.median(observed_ages))
        if len(observed_ages)
        else 65.0
    )

    def clinical_view(indices):

        a = np.where(
            np.isfinite(age[indices]),
            age[indices],
            median_age
        )

        return np.column_stack([
            (a - median_age) / 10,
            (~np.isfinite(age[indices])).astype(float),
            np.where(
                np.isfinite(male[indices]),
                male[indices],
                0.5
            ),
            (~np.isfinite(male[indices])).astype(float),
            histology[indices],
            (stage[indices] == 2).astype(float),
            (stage[indices] == 3).astype(float),
            (stage[indices] == 4).astype(float)
        ]).astype(np.float32)

    clinical_fit = clinical_view(fit)
    clinical_hold = clinical_view(held)

    mean = clinical_fit.mean(axis=0)

    sd = np.maximum(
        clinical_fit.std(axis=0),
        0.30
    )

    clinical_fit = np.clip(
        (clinical_fit - mean) / sd,
        -5,
        5
    )

    clinical_hold = np.clip(
        (clinical_hold - mean) / sd,
        -5,
        5
    )

    return (
        torch.from_numpy(molecular(xfit)),
        torch.from_numpy(molecular(xheld)),
        torch.from_numpy(
            clinical_fit.astype(np.float32)
        ),
        torch.from_numpy(
            clinical_hold.astype(np.float32)
        )
    )


# Single-layer, edge-aware multi-head graph attention.
class SurvivalGAT(nn.Module):

    def __init__(self, n_nodes):

        super().__init__()

        self.gene_embed = nn.Embedding(
            n_nodes,
            4
        )

        self.input = nn.Linear(
            6,
            16
        )

        self.proj = nn.Linear(
            16,
            16,
            bias=False
        )

        self.att_source = nn.Parameter(
            torch.empty(2, 8)
        )

        self.att_target = nn.Parameter(
            torch.empty(2, 8)
        )

        self.att_edge = nn.Parameter(
            torch.zeros(2)
        )

        self.norm = nn.LayerNorm(16)

        self.gate = nn.Linear(
            16,
            1
        )

        self.clinical_head = nn.Linear(
            8,
            1
        )

        self.molecular_head = nn.Sequential(
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(16, 1)
        )

        nn.init.xavier_uniform_(
            self.att_source
        )

        nn.init.xavier_uniform_(
            self.att_target
        )

    def forward(
        self,
        x,
        clinical,
        graph
    ):

        src, dst, weight = graph

        batch, n, _ = x.shape

        embeddings = (
            self.gene_embed.weight
            .unsqueeze(0)
            .expand(batch, -1, -1)
        )

        z = torch.cat(
            (x, embeddings),
            dim=-1
        )

        hidden = F.relu(
            self.input(z)
        )

        q = self.proj(
            hidden
        ).reshape(
            batch,
            n,
            2,
            8
        )

        source_logits = (
            q * self.att_source
        ).sum(-1)

        target_logits = (
            q * self.att_target
        ).sum(-1)

        logits = F.leaky_relu(
            source_logits[:, src]
            + target_logits[:, dst]
            + weight.view(1, -1, 1)
            * self.att_edge.view(1, 1, 2),
            negative_slope=0.2
        )

        index = dst.view(
            1,
            -1,
            1
        ).expand(
            batch,
            -1,
            2
        )

        maximum = torch.full(
            (batch, n, 2),
            -torch.inf,
            dtype=logits.dtype,
            device=logits.device
        )

        maximum = maximum.scatter_reduce(
            1,
            index,
            logits,
            reduce="amax",
            include_self=True
        )

        exponent = torch.exp(
            logits - maximum.gather(
                1,
                index
            )
        )

        denominator = torch.zeros(
            (batch, n, 2),
            dtype=logits.dtype,
            device=logits.device
        ).scatter_add(
            1,
            index,
            exponent
        )

        attention = F.dropout(
            exponent / denominator.gather(
                1,
                index
            ).clamp_min(1e-9),
            p=0.10,
            training=self.training
        )

        messages = (
            q[:, src]
            * attention.unsqueeze(-1)
        )

        message_indices = dst.view(
            1,
            -1,
            1,
            1
        ).expand(
            batch,
            -1,
            2,
            8
        )

        aggregated = torch.zeros_like(
            q
        ).scatter_add(
            1,
            message_indices,
            messages
        ).reshape(
            batch,
            n,
            16
        )

        z = self.norm(
            hidden + F.relu(aggregated)
        )

        node_attention = torch.softmax(
            self.gate(z).squeeze(-1),
            dim=1
        )

        molecular_summary = torch.cat(
            (
                (
                    z * node_attention.unsqueeze(-1)
                ).sum(1),
                z.mean(1)
            ),
            dim=1
        )

        # Clinical prediction plus a regularized molecular branch.
        return (
            self.clinical_head(
                clinical
            ).squeeze(-1)
            + 0.20
            * self.molecular_head(
                molecular_summary
            ).squeeze(-1)
        )


# Breslow Cox loss using complete training-fold risk sets.
def cox_loss(
    risk,
    times,
    events
):

    order = torch.argsort(
        times,
        descending=True,
        stable=True
    )

    t = times[order]
    e = events[order]
    r = risk[order]

    if e.sum() == 0:
        raise RuntimeError(
            "Training fold has no PFI events."
        )

    cumulative = torch.logcumsumexp(
        r,
        dim=0
    )

    # Extend tied event times to the end of their risk set.
    ends = torch.searchsorted(
        -t.contiguous(),
        -t.contiguous(),
        right=True
    ) - 1

    return (
        e * (
            cumulative[ends] - r
        )
    ).sum() / e.sum()


# Harrell C-index for higher-risk / earlier-event predictions.
def c_index(
    times,
    events,
    risk
):

    comparable = (
        (times[:, None] < times[None, :])
        & (events[:, None] == 1)
    )

    n_pairs = int(
        comparable.sum()
    )

    if n_pairs < 10:
        return np.nan

    delta = (
        risk[:, None] - risk[None, :]
    )

    concordant = (
        (
            (delta > 1e-9)
            & comparable
        ).sum()
        + 0.5 * (
            (np.abs(delta) <= 1e-9)
            & comparable
        ).sum()
    )

    return float(
        concordant / n_pairs
    )


def set_seed(seed):

    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)


# Use an inner site fold for epoch selection, then refit.
def train_model(
    raw,
    clinical,
    times,
    events,
    train_ids,
    prediction_ids,
    graph,
    seed,
    max_epochs,
    select_epochs
):

    set_seed(seed)

    Xfit, Xhold, Cfit, Chold = prepare_features(
        raw,
        clinical,
        train_ids,
        prediction_ids
    )

    Tfit = torch.from_numpy(
        times[train_ids].astype(np.float32)
    )

    Efit = torch.from_numpy(
        events[train_ids].astype(np.float32)
    )

    model = SurvivalGAT(
        Xfit.shape[1]
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.001,
        weight_decay=0.01
    )

    best_c = -np.inf
    best_epoch = 1

    for epoch in range(
        1,
        max_epochs + 1
    ):

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        indices = torch.randperm(
            len(train_ids)
        )

        risk_parts = []

        for batch in indices.split(BATCH):

            risk_parts.append(
                model(
                    Xfit[batch],
                    Cfit[batch],
                    graph
                )
            )

        # Concatenate predictions before the Cox loss so
        # every training patient belongs to the risk set.
        risk = torch.cat(
            risk_parts
        )

        loss = cox_loss(
            risk,
            Tfit[indices],
            Efit[indices]
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Non-finite neural Cox loss."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            2.0
        )

        optimizer.step()

        if select_epochs:

            model.eval()

            with torch.no_grad():

                held_risk = torch.cat([
                    model(x, c, graph)
                    for x, c in zip(
                        Xhold.split(BATCH),
                        Chold.split(BATCH)
                    )
                ]).numpy()

            score = c_index(
                times[prediction_ids],
                events[prediction_ids],
                held_risk
            )

            if (
                np.isfinite(score)
                and score > best_c + 1e-5
            ):

                best_c = score
                best_epoch = epoch

            if (
                epoch >= 6
                and epoch - best_epoch >= PATIENCE
            ):
                break

    if select_epochs:

        return (
            best_epoch,
            best_c,
            None
        )

    model.eval()

    with torch.no_grad():

        predictions = torch.cat([
            model(x, c, graph)
            for x, c in zip(
                Xhold.split(BATCH),
                Chold.split(BATCH)
            )
        ]).numpy()

    return (
        max_epochs,
        None,
        predictions
    )


# Run matched Stage 21 site-held-out folds.
def run_stage22():

    OUT.mkdir(
        parents=True,
        exist_ok=True
    )

    CKPT.mkdir(
        parents=True,
        exist_ok=True
    )

    torch.set_num_threads(
        THREADS
    )

    (
        raw,
        clinical,
        times,
        events,
        histology,
        sites,
        eligible,
        graphs
    ) = load_project()

    fingerprint = hashlib.sha256(
        json.dumps(
            {
                "version": VERSION,
                "eligible": eligible.tolist(),
                "case_ids": clinical.case_id.tolist(),
                "times": np.nan_to_num(
                    times,
                    nan=-1
                ).tolist(),
                "events": np.nan_to_num(
                    events,
                    nan=-1
                ).tolist(),
                "gene_file_mtime": (
                    OLD / "selected_gene_nodes.csv"
                ).stat().st_mtime_ns,
                "graph_file_mtime": (
                    OLD
                    / "string_functional_700_corrected.npz"
                ).stat().st_mtime_ns,
                "config": [
                    N_GENE,
                    EPOCHS,
                    PATIENCE,
                    BATCH,
                    SEED,
                    EXPERIMENTS
                ]
            },
            sort_keys=True
        ).encode()
    ).hexdigest()

    rows = []

    oof = {
        name: np.full(
            len(clinical),
            np.nan
        )
        for name in EXPERIMENTS
    }

    clinical_oof = np.full(
        len(clinical),
        np.nan
    )

    print("=" * 74)
    print("STAGE 22 - CLINICAL-INTEGRATED PPI-GAT SURVIVAL")
    print("=" * 74)

    print(
        f"PFI eligible={len(eligible)}, "
        f"events={int(events[eligible].sum())}, "
        "test untouched"
    )

    for fold in range(1, 5):

        reference = joblib.load(
            S21 / f"pfi_outer_fold{fold}.joblib"
        )

        held = np.asarray(
            reference["predictions"]["index"],
            dtype=int
        )

        fit = np.setdiff1d(
            eligible,
            held
        )

        clinical_risk = np.asarray(
            reference["predictions"]["clinical_risk"],
            dtype=float
        )

        if (
            len(held) != len(clinical_risk)
            or not np.isin(
                held,
                eligible
            ).all()
            or set(sites[fit]) & set(sites[held])
        ):
            raise RuntimeError(
                f"Stage 21 fold {fold} mismatch."
            )

        clinical_oof[held] = clinical_risk

        print(
            f"\nSITE FOLD {fold}: "
            f"fit={len(fit)}, "
            f"held={len(held)}, "
            f"PFI events={int(events[held].sum())}",
            flush=True
        )

        # Pick one inner site-disjoint fold using outcomes
        # from the outer training portion only.
        inner_strata = (
            2 * events[fit].astype(int)
            + histology[fit]
        )

        splitter = StratifiedGroupKFold(
            n_splits=4,
            shuffle=True,
            random_state=SEED + fold
        )

        inner_choices = []

        for index, (
            local_fit,
            local_held
        ) in enumerate(
            splitter.split(
                fit,
                inner_strata,
                sites[fit]
            )
        ):

            inner_fit = fit[local_fit]
            inner_hold = fit[local_held]

            if (
                len(np.unique(events[inner_hold])) == 2
                and events[inner_hold].sum() >= 10
                and not (
                    set(sites[inner_fit])
                    & set(sites[inner_hold])
                )
            ):

                deviation = (
                    abs(
                        events[inner_hold].mean()
                        - events[fit].mean()
                    )
                    + abs(
                        len(inner_hold) / len(fit)
                        - 0.25
                    )
                )

                inner_choices.append(
                    (
                        deviation,
                        index,
                        inner_fit,
                        inner_hold
                    )
                )

        if not inner_choices:
            raise RuntimeError(
                "No valid inner site fold for early stopping."
            )

        (
            _,
            _,
            inner_fit,
            inner_hold
        ) = min(
            inner_choices,
            key=lambda item: (
                item[0],
                item[1]
            )
        )

        row = {
            "fold": fold,
            "held_patients": len(held),
            "events": int(
                events[held].sum()
            ),
            "clinical_C": c_index(
                times[held],
                events[held],
                clinical_risk
            )
        }

        for name in EXPERIMENTS:

            checkpoint = (
                OUT
                / f"fold{fold}_{name}_result.joblib"
            )

            if checkpoint.exists():

                saved = joblib.load(
                    checkpoint
                )

                if (
                    saved.get("fingerprint") != fingerprint
                    or not np.array_equal(
                        saved["held_index"],
                        held
                    )
                ):
                    raise RuntimeError(
                        "Incompatible saved Stage 22 fold."
                    )

                risk = np.asarray(
                    saved["risk"],
                    dtype=float
                )

                selected_epoch = saved["epoch"]

                print(
                    f"  {name}: cached epoch={selected_epoch}",
                    flush=True
                )

            else:

                # Inner site fold determines the training duration.
                (
                    selected_epoch,
                    inner_c,
                    _
                ) = train_model(
                    raw,
                    clinical,
                    times,
                    events,
                    inner_fit,
                    inner_hold,
                    graphs[name],
                    SEED + fold,
                    max_epochs=EPOCHS,
                    select_epochs=True
                )

                # Refit using all outer training patients.
                (
                    _,
                    _,
                    risk
                ) = train_model(
                    raw,
                    clinical,
                    times,
                    events,
                    fit,
                    held,
                    graphs[name],
                    SEED + fold,
                    max_epochs=selected_epoch,
                    select_epochs=False
                )

                temporary = checkpoint.with_name(
                    checkpoint.name + ".tmp"
                )

                joblib.dump(
                    {
                        "fingerprint": fingerprint,
                        "held_index": held,
                        "risk": risk,
                        "epoch": selected_epoch,
                        "inner_C": inner_c
                    },
                    temporary,
                    compress=3
                )

                os.replace(
                    temporary,
                    checkpoint
                )

                print(
                    f"  {name}: "
                    f"inner C={inner_c:.4f}, "
                    f"epoch={selected_epoch}",
                    flush=True
                )

            oof[name][held] = risk

            row[f"{name}_C"] = c_index(
                times[held],
                events[held],
                risk
            )

            print(
                f"    outer PFI C="
                f"{row[f'{name}_C']:.4f}, "
                f"clinical={row['clinical_C']:.4f}",
                flush=True
            )

        rows.append(row)

    if (
        not np.isfinite(
            clinical_oof[eligible]
        ).all()
        or any(
            not np.isfinite(
                oof[name][eligible]
            ).all()
            for name in EXPERIMENTS
        )
    ):
        raise RuntimeError(
            "Incomplete site-held-out predictions."
        )

    table = pd.DataFrame(rows)

    table.to_csv(
        OUT / "stage22_graph_survival_site_folds.csv",
        index=False
    )

    pd.DataFrame({
        "case_id": clinical.case_id.iloc[eligible],
        "PFI.time": times[eligible],
        "PFI": events[eligible],
        "site": sites[eligible],
        "clinical_risk": clinical_oof[eligible],
        **{
            f"{name}_risk": oof[name][eligible]
            for name in EXPERIMENTS
        }
    }).to_csv(
        OUT / "stage22_training_only_oof_predictions.csv",
        index=False
    )

    report = {
        "timestamp": datetime.now(
            timezone.utc
        ).isoformat(),
        "version": VERSION,
        "eligible_patients": len(eligible),
        "events": int(
            events[eligible].sum()
        ),
        "nodes": N_GENE,
        "outer_folds": len(rows),
        "mean_clinical_C": float(
            table.clinical_C.mean()
        ),
        "model_mean_C": {
            name: float(
                table[f"{name}_C"].mean()
            )
            for name in EXPERIMENTS
        },
        "increment_over_clinical": {
            name: float(
                (
                    table[f"{name}_C"]
                    - table.clinical_C
                ).mean()
            )
            for name in EXPERIMENTS
        },
        "functional_minus_null_C": float(
            (
                table.functional_C
                - table.degree_preserving_null_C
            ).mean()
        ),
        "test_evaluated": False,
        "caveats": [
            "Stage 09 selected the original 1600 genes "
            "using training-wide stage information.",
            "This experiment is not fully nested "
            "at the original gene-selection level.",
            "The clinical baseline is from matched "
            "Stage 21 held-out folds.",
            "Harrell C-index does not establish calibration.",
            "Internal site-held-out evaluation "
            "is not independent external validation."
        ]
    }

    (
        OUT / "stage22_summary.json"
    ).write_text(
        json.dumps(
            report,
            indent=2
        ),
        encoding="utf-8"
    )

    print("\n" + "=" * 74)
    print("STAGE 22 FINAL - PPI-GAT PFI SITE-HELD-OUT")
    print("=" * 74)

    print(
        table.to_string(
            index=False,
            float_format=lambda v: f"{v:.4f}"
        )
    )

    print(
        "\nMean clinical-only C-index:",
        round(
            report["mean_clinical_C"],
            4
        )
    )

    print(
        "Mean neural model C-indices:",
        report["model_mean_C"]
    )

    print(
        "Increment over clinical-only:",
        report["increment_over_clinical"]
    )

    print(
        "Functional minus randomized PPI:",
        round(
            report["functional_minus_null_C"],
            4
        )
    )

    print(
        "Validation/test data loaded: NO"
    )

    print("Results:", OUT)
    print("STAGE 22 COMPLETED")


run_stage22()

Selected graph nodes: 512
Functional undirected edges: 1994
Degree-preserving randomized control: PASSED
STAGE 22 - CLINICAL-INTEGRATED PPI-GAT SURVIVAL
PFI eligible=657, events=225, test untouched

SITE FOLD 1: fit=491, held=166, PFI events=56
  graphfree: inner C=0.5000, epoch=17
    outer PFI C=0.4890, clinical=0.6489
  functional: inner C=0.5008, epoch=18
    outer PFI C=0.4883, clinical=0.6489
  degree_preserving_null: inner C=0.4996, epoch=17
    outer PFI C=0.4886, clinical=0.6489

SITE FOLD 2: fit=494, held=163, PFI events=55
  graphfree: inner C=0.6296, epoch=7
    outer PFI C=0.5964, clinical=0.6421
  functional: inner C=0.6293, epoch=6
    outer PFI C=0.5949, clinical=0.6421
  degree_preserving_null: inner C=0.6293, epoch=5
    outer PFI C=0.5953, clinical=0.6421

SITE FOLD 3: fit=493, held=164, PFI events=57
  graphfree: inner C=0.6126, epoch=15
    outer PFI C=0.5715, clinical=0.6660
  functional: inner C=0.6173, epoch=18
    outer PFI C=0.5750, clinical=0.6660
  degree_pr

In [22]:

import hashlib
import json
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import networkx as nx
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from scipy import sparse
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

# ============================================================
# 1. Project configuration
# ============================================================

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select NST NSCLC Research (Python 3.12) kernel."
    )

RAW = ROOT / "data" / "processed" / "gene_level"
PPI = ROOT / "data" / "processed" / "ppi_gat_v1"
S21 = ROOT / "results" / "metrics" / "stage21"
OUT = ROOT / "results" / "metrics" / "stage23"
MODEL_DIR = ROOT / "models" / "checkpoints" / "stage23"

OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_NODES = 512
MAX_EPOCHS = 28
PATIENCE = 7
BATCH_SIZE = 32
LR = 0.002

L2_RESIDUAL = 0.02
L2_SITE = 0.02

GAMMAS = (0.0, 0.25, 0.50, 1.0)
MIN_GAIN = 0.005
THREADS = min(4, os.cpu_count() or 1)

VERSION = "stage23_clinically_anchored_residual_gat_v1"

MODES = (
    "graphfree",
    "functional",
    "degree_preserving_null"
)


def seed_all(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)


# ============================================================
# 2. Survival evaluation
# ============================================================

def concordance(t, e, r):

    t = np.asarray(t, dtype=float)
    e = np.asarray(e, dtype=int)
    r = np.asarray(r, dtype=float)

    comparable = (
        (t[:, None] < t[None, :])
        & (e[:, None] == 1)
    )

    n_pairs = int(comparable.sum())

    if n_pairs < 10:
        return float("nan")

    difference = r[:, None] - r[None, :]

    concordant = np.count_nonzero(
        comparable & (difference > 1e-10)
    )

    tied = np.count_nonzero(
        comparable
        & (np.abs(difference) <= 1e-10)
    )

    return float(
        (concordant + 0.5 * tied) / n_pairs
    )


# ============================================================
# 3. Reconstruct Stage 21 clinical Cox baseline
# ============================================================

def ridge_cox(x, times, events):

    x = np.asarray(x, dtype=np.float64)
    times = np.asarray(times, dtype=np.float64)
    events = np.asarray(events, dtype=np.int64)

    order = np.argsort(
        -times,
        kind="stable"
    )

    z = x[order]
    t = times[order]
    e = events[order]

    starts = np.r_[
        0,
        np.flatnonzero(t[1:] != t[:-1]) + 1
    ]

    ends = np.r_[
        starts[1:] - 1,
        len(t) - 1
    ]

    event_counts = np.add.reduceat(
        e,
        starts
    )

    valid = event_counts > 0

    ends = ends[valid]

    event_counts = event_counts[
        valid
    ].astype(float)

    penalty = np.full(
        z.shape[1],
        0.003 * 0.15
    )

    n_events = max(
        int(events.sum()),
        1
    )

    def objective(beta):

        eta = z @ beta
        shift = np.max(eta)

        weights = np.exp(
            eta - shift
        )

        cumulative_w = np.cumsum(
            weights
        )

        cumulative_x = np.cumsum(
            weights[:, None] * z,
            axis=0
        )

        risk_sums = np.maximum(
            cumulative_w[ends],
            1e-300
        )

        loss = (
            np.sum(
                event_counts
                * (
                    shift
                    + np.log(risk_sums)
                )
            )
            - np.sum(eta[e == 1])
        ) / n_events

        gradient = (
            np.sum(
                event_counts[:, None]
                * cumulative_x[ends]
                / risk_sums[:, None],
                axis=0
            )
            - np.sum(z[e == 1], axis=0)
        ) / n_events

        return (
            float(
                loss
                + 0.5 * (
                    penalty * beta * beta
                ).sum()
            ),
            gradient + penalty * beta
        )

    result = minimize(
        objective,
        np.zeros(z.shape[1]),
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter": 300,
            "ftol": 1e-9,
            "gtol": 1e-6
        }
    )

    if not np.isfinite(
        result.x
    ).all():
        raise RuntimeError(
            "Clinical Cox optimization failed."
        )

    return result.x


def fold_clinical(frame, fit, held):

    age = pd.to_numeric(
        frame.age_years,
        errors="coerce"
    ).to_numpy(dtype=float)

    sex = pd.to_numeric(
        frame.male,
        errors="coerce"
    ).to_numpy(dtype=float)

    stage = pd.to_numeric(
        frame.stage_major,
        errors="coerce"
    ).to_numpy(dtype=float)

    histology = (
        frame.histology_matrix.eq("LUSC")
        .to_numpy(dtype=float)
    )

    available_age = age[fit][
        np.isfinite(age[fit])
    ]

    median_age = (
        float(np.median(available_age))
        if len(available_age)
        else 65.0
    )

    def block(ids):

        a = np.where(
            np.isfinite(age[ids]),
            age[ids],
            median_age
        )

        return np.column_stack([
            (a - median_age) / 10.0,
            (~np.isfinite(age[ids])).astype(float),
            np.where(
                np.isfinite(sex[ids]),
                sex[ids],
                0.5
            ),
            (~np.isfinite(sex[ids])).astype(float),
            histology[ids],
            (stage[ids] == 2).astype(float),
            (stage[ids] == 3).astype(float),
            (stage[ids] == 4).astype(float)
        ])

    x_fit = block(fit)
    x_held = block(held)

    scaler = StandardScaler().fit(
        x_fit
    )

    return (
        scaler.transform(x_fit),
        scaler.transform(x_held)
    )


def clinical_anchor(frame, times, events, fit, held):

    x_fit, x_held = fold_clinical(
        frame,
        fit,
        held
    )

    coefficients = ridge_cox(
        x_fit,
        times[fit],
        events[fit]
    )

    return (
        (x_fit @ coefficients).astype(np.float32),
        (x_held @ coefficients).astype(np.float32)
    )


# ============================================================
# 4. Load verified TCGA cohort and PFI endpoints
# ============================================================

def load_data():

    cases = pd.read_csv(
        RAW / "case_order.csv",
        dtype={"case_id": str}
    )

    genes = pd.read_csv(
        RAW / "gene_reference.csv",
        dtype={"ensembl_id": str}
    )

    locked = pd.read_csv(
        ROOT / "data" / "splits"
        / "locked_four_omics_v1.csv",
        dtype={"case_id": str}
    )

    joined = cases[
        [
            "case_id",
            "histology",
            "stage_group"
        ]
    ].merge(
        locked[
            [
                "case_id",
                "histology",
                "stage_group",
                "split"
            ]
        ],
        on="case_id",
        how="left",
        validate="one_to_one",
        suffixes=("_a", "_b")
    )

    if (
        not joined.histology_a.equals(
            joined.histology_b
        )
        or not joined.stage_group_a.equals(
            joined.stage_group_b
        )
    ):
        raise RuntimeError(
            "Locked labels or patient order changed."
        )

    tr = np.flatnonzero(
        joined.split.eq("train").to_numpy()
    )

    va = np.flatnonzero(
        joined.split.eq("validation").to_numpy()
    )

    te = np.flatnonzero(
        joined.split.eq("test").to_numpy()
    )

    if (
        (len(tr), len(va), len(te))
        != (667, 143, 143)
        or len(genes) != 19944
    ):
        raise RuntimeError(
            "Unexpected cohort dimensions."
        )

    frame = pd.read_csv(
        S21 / "stage21_training_clinical_endpoints.csv",
        dtype={"case_id": str}
    )

    if (
        len(frame) != len(tr)
        or not np.array_equal(
            frame.case_id.to_numpy(),
            joined.case_id.iloc[tr].to_numpy()
        )
    ):
        raise RuntimeError(
            "Stage 21 survival patient order differs."
        )

    eligible = np.flatnonzero(
        frame.PFI_valid.astype(str)
        .str.lower()
        .eq("true")
        .to_numpy()
    )

    times = pd.to_numeric(
        frame["PFI.time"],
        errors="coerce"
    ).to_numpy(dtype=float)

    events = pd.to_numeric(
        frame.PFI,
        errors="coerce"
    ).to_numpy(dtype=float)

    histology = (
        frame.histology_matrix.eq("LUSC")
        .to_numpy(dtype=np.int64)
    )

    site = frame.site.astype(str).to_numpy()

    if (
        len(eligible) != 657
        or int(events[eligible].sum()) != 225
    ):
        raise RuntimeError(
            "Stage 21 PFI eligibility or event count changed."
        )

    # Load the existing STRING network.
    nodes = pd.read_csv(
        PPI / "selected_gene_nodes.csv",
        dtype={"ensembl_id": str}
    ).sort_values("node_index")

    if (
        len(nodes) != 1600
        or not np.array_equal(
            nodes.node_index,
            np.arange(1600)
        )
    ):
        raise RuntimeError(
            "Stage 09 node indices are inconsistent."
        )

    mapping = pd.Index(
        genes.ensembl_id.astype(str)
    ).get_indexer(
        nodes.ensembl_id.astype(str)
    )

    if (mapping < 0).any():
        raise RuntimeError(
            "Graph genes are missing from Stage 08."
        )

    with np.load(
        PPI / "string_functional_700_corrected.npz"
    ) as archive:

        edges = archive[
            "edge_index"
        ].astype(np.int64)

        weights = archive[
            "edge_weight"
        ].astype(np.float32)

    adjacency = sparse.csr_matrix(
        (
            weights,
            (edges[0], edges[1])
        ),
        shape=(1600, 1600)
    )

    adjacency.setdiag(0)
    adjacency.eliminate_zeros()

    if np.max(
        np.abs(
            (adjacency - adjacency.T).data
        ),
        initial=0
    ) > 1e-4:
        raise RuntimeError(
            "Functional PPI graph is asymmetric."
        )

    degrees = np.asarray(
        (adjacency > 0).sum(axis=0)
    ).ravel()

    selection = np.sort(
        np.lexsort(
            (np.arange(1600), -degrees)
        )[:N_NODES]
    )

    positions = mapping[
        selection
    ]

    adjacency = adjacency[
        selection, :
    ][:, selection].tocsr()

    graph = nx.from_scipy_sparse_array(
        adjacency
    )

    graph_null = graph.copy()

    nx.double_edge_swap(
        graph_null,
        nswap=3 * graph.number_of_edges(),
        max_tries=100 * graph.number_of_edges(),
        seed=SEED + 23
    )

    if any(
        graph.degree(i) != graph_null.degree(i)
        for i in range(N_NODES)
    ):
        raise RuntimeError(
            "Degree-preserving null graph mismatch."
        )

    def graph_tensors(g, values):

        pairs = np.asarray(
            sorted(g.edges()),
            dtype=np.int64
        )

        if not len(pairs):
            raise RuntimeError(
                "The selected PPI graph is empty."
            )

        source = np.r_[
            pairs[:, 0],
            pairs[:, 1],
            np.arange(N_NODES)
        ].astype(np.int64)

        target = np.r_[
            pairs[:, 1],
            pairs[:, 0],
            np.arange(N_NODES)
        ].astype(np.int64)

        confidence = np.r_[
            values,
            values,
            np.ones(N_NODES)
        ].astype(np.float32)

        return tuple(
            torch.as_tensor(a)
            for a in (
                source,
                target,
                confidence
            )
        )

    true_pairs = sorted(
        graph.edges()
    )

    true_weights = np.array(
        [
            graph[u][v].get("weight", 1.0)
            for u, v in true_pairs
        ],
        dtype=np.float32
    )

    true_weights /= max(
        float(true_weights.max()),
        1e-8
    )

    null_weights = true_weights.copy()

    np.random.default_rng(
        SEED + 23
    ).shuffle(null_weights)

    node_index = torch.arange(
        N_NODES
    )

    graphs = {
        "graphfree": (
            node_index,
            node_index,
            torch.ones(N_NODES)
        ),
        "functional": graph_tensors(
            graph,
            true_weights
        ),
        "degree_preserving_null": graph_tensors(
            graph_null,
            null_weights
        )
    }

    # Load training-only RNA and CNV measurements.
    rna = np.load(
        RAW / "rna_tpm.npy",
        mmap_mode="r"
    )

    cnv = np.load(
        RAW / "cnv_absolute.npy",
        mmap_mode="r"
    )

    if (
        rna.shape != cnv.shape
        or rna.shape != (953, 19944)
    ):
        raise RuntimeError(
            "Stage 08 RNA/CNV dimensions mismatch."
        )

    r = np.asarray(
        rna[np.ix_(tr, positions)],
        dtype=np.float32
    )

    c = np.asarray(
        cnv[np.ix_(tr, positions)],
        dtype=np.float32
    )

    r = np.where(
        np.isfinite(r) & (r >= 0),
        np.log1p(np.maximum(r, 0)),
        np.nan
    )

    c = np.where(
        np.isfinite(c) & (c >= 0),
        np.clip(
            np.log2(
                (np.maximum(c, 0) + 0.5) / 2.5
            ),
            -3,
            3
        ),
        np.nan
    )

    raw_features = np.stack(
        (r, c),
        axis=-1
    ).astype(np.float32)

    return (
        frame,
        times,
        events,
        histology,
        site,
        eligible,
        raw_features,
        graphs,
        graph.number_of_edges()
    )


# ============================================================
# 5. Fold-specific molecular preprocessing
# ============================================================

def normalize_molecular(raw, fit, held):

    fitting = raw[fit]

    observed = np.isfinite(
        fitting
    )

    count = observed.sum(
        axis=0
    ).clip(min=1)

    mean = np.nansum(
        fitting,
        axis=0
    ) / count

    deviation = np.where(
        observed,
        fitting - mean,
        0
    )

    sd = np.sqrt(
        np.nansum(
            deviation ** 2,
            axis=0
        ) / count
    )

    sd = np.maximum(
        sd,
        0.25
    )

    def transform(ids):

        x = raw[ids]

        valid = np.isfinite(
            x
        )

        scaled = np.where(
            valid,
            (x - mean) / sd,
            0
        ).clip(-5, 5)

        combined = np.concatenate(
            (
                scaled,
                valid.astype(np.float32)
            ),
            axis=-1
        )

        return torch.tensor(
            combined,
            dtype=torch.float32
        )

    return (
        transform(fit),
        transform(held)
    )


# ============================================================
# 6. New clinical-anchored molecular residual GAT
# ============================================================

class ResidualSurvivalGAT(nn.Module):

    def __init__(self, n_genes):

        super().__init__()

        self.gene_embed = nn.Embedding(
            n_genes,
            4
        )

        self.embed = nn.Linear(
            8,
            12
        )

        self.project = nn.Linear(
            12,
            16,
            bias=False
        )

        self.src_att = nn.Parameter(
            torch.empty(2, 8)
        )

        self.dst_att = nn.Parameter(
            torch.empty(2, 8)
        )

        self.edge_bias = nn.Parameter(
            torch.zeros(2)
        )

        self.self_transform = nn.Linear(
            12,
            12
        )

        self.merge = nn.Linear(
            16,
            12
        )

        self.norm = nn.LayerNorm(
            12
        )

        # Histology-conditioned node attention.
        self.hist_query = nn.Embedding(
            2,
            12
        )

        self.att_pool = nn.Linear(
            12,
            1
        )

        self.head = nn.Sequential(
            nn.LayerNorm(24),
            nn.Linear(24, 12),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(12, 1)
        )

        nn.init.xavier_uniform_(
            self.src_att
        )

        nn.init.xavier_uniform_(
            self.dst_att
        )

        nn.init.normal_(
            self.gene_embed.weight,
            std=0.05
        )

        # The initial residual is exactly zero.
        nn.init.zeros_(
            self.head[-1].weight
        )

        nn.init.zeros_(
            self.head[-1].bias
        )

    def forward(self, x, hist, graph):

        src, dst, weights = graph

        batch, n, _ = x.shape

        gene = (
            self.gene_embed.weight
            .unsqueeze(0)
            .expand(batch, -1, -1)
        )

        hidden = F.gelu(
            self.embed(
                torch.cat(
                    (x, gene),
                    dim=-1
                )
            )
        )

        q = self.project(
            hidden
        ).reshape(
            batch,
            n,
            2,
            8
        )

        attention_logits = F.leaky_relu(
            (
                q[:, src] * self.src_att
            ).sum(-1)
            + (
                q[:, dst] * self.dst_att
            ).sum(-1)
            + weights.view(1, -1, 1)
            * self.edge_bias.view(1, 1, 2),
            negative_slope=0.2
        )

        index = dst.view(
            1, -1, 1
        ).expand(
            batch, -1, 2
        )

        maximum = torch.full(
            (batch, n, 2),
            -torch.inf,
            dtype=attention_logits.dtype
        )

        maximum = maximum.scatter_reduce(
            1,
            index,
            attention_logits,
            reduce="amax",
            include_self=True
        )

        exponent = torch.exp(
            attention_logits
            - maximum.gather(1, index)
        )

        denominator = torch.zeros_like(
            maximum
        ).scatter_add(
            1,
            index,
            exponent
        )

        attention = F.dropout(
            exponent
            / denominator.gather(
                1,
                index
            ).clamp_min(1e-8),
            p=0.08,
            training=self.training
        )

        messages = (
            q[:, src]
            * attention.unsqueeze(-1)
        )

        message_index = dst.view(
            1, -1, 1, 1
        ).expand(
            batch, -1, 2, 8
        )

        aggregated = torch.zeros_like(
            q
        ).scatter_add(
            1,
            message_index,
            messages
        ).reshape(
            batch, n, 16
        )

        z = F.gelu(
            self.norm(
                self.self_transform(hidden)
                + self.merge(aggregated)
            )
        )

        # Histology-conditioned attention pooling.
        scores = (
            self.att_pool(z).squeeze(-1)
            + (
                z
                * self.hist_query(hist)[:, None, :]
            ).sum(-1) / np.sqrt(12)
        )

        weighted = (
            z
            * scores.softmax(
                dim=1
            ).unsqueeze(-1)
        ).sum(dim=1)

        pooled = torch.cat(
            (
                weighted,
                z.mean(dim=1)
            ),
            dim=1
        )

        return self.head(
            pooled
        ).squeeze(-1).clamp(
            -2.0,
            2.0
        )


# ============================================================
# 7. Full-risk-set Cox loss
# ============================================================

def cox_loss(log_risk, time, event):

    order = torch.argsort(
        time,
        descending=True,
        stable=True
    )

    r = log_risk[order]
    t = time[order]
    e = event[order]

    ends = torch.searchsorted(
        -t.contiguous(),
        -t.contiguous(),
        right=True
    ) - 1

    cumulative = torch.logcumsumexp(
        r,
        dim=0
    )

    return (
        e * (
            cumulative[ends] - r
        )
    ).sum() / e.sum().clamp_min(1.0)


def predict_residual(
    model,
    X,
    hist,
    graph
):

    model.eval()

    with torch.no_grad():

        predictions = [
            model(
                a,
                b,
                graph
            )
            for a, b in zip(
                X.split(BATCH_SIZE),
                hist.split(BATCH_SIZE)
            )
        ]

    return torch.cat(
        predictions
    ).numpy()


# ============================================================
# 8. Fit the molecular residual with frozen clinical risk
# ============================================================

def fit_residual(
    raw,
    frame,
    times,
    events,
    hist,
    site,
    fit,
    held,
    graph,
    seed,
    max_epochs,
    search=False
):

    seed_all(seed)

    xtr, xte = normalize_molecular(
        raw,
        fit,
        held
    )

    htr = torch.as_tensor(
        hist[fit],
        dtype=torch.long
    )

    hte = torch.as_tensor(
        hist[held],
        dtype=torch.long
    )

    clinical_train, clinical_held = (
        clinical_anchor(
            frame,
            times,
            events,
            fit,
            held
        )
    )

    baseline_c = concordance(
        times[held],
        events[held],
        clinical_held
    )

    model = ResidualSurvivalGAT(
        xtr.shape[1]
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0.025
    )

    ttr = torch.as_tensor(
        times[fit],
        dtype=torch.float32
    )

    etr = torch.as_tensor(
        events[fit],
        dtype=torch.float32
    )

    clinical_tensor = torch.as_tensor(
        clinical_train,
        dtype=torch.float32
    )

    _, site_indices = np.unique(
        site[fit],
        return_inverse=True
    )

    site_codes = torch.as_tensor(
        site_indices,
        dtype=torch.long
    )

    n_sites = int(
        site_codes.max().item() + 1
    )

    site_counts = torch.bincount(
        site_codes,
        minlength=n_sites
    ).float().clamp_min(1.0)

    # The clinical-only model is always a valid candidate.
    best = (
        float(baseline_c),
        1,
        0.0
    )

    last_improvement = 1

    for epoch in range(
        1,
        max_epochs + 1
    ):

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        order = torch.randperm(
            len(fit)
        )

        residual = torch.cat([
            model(
                xtr[part],
                htr[part],
                graph
            )
            for part in order.split(
                BATCH_SIZE
            )
        ])

        risk = (
            clinical_tensor[order]
            + residual
        )

        site_sums = torch.zeros(
            n_sites,
            dtype=residual.dtype
        ).index_add(
            0,
            site_codes[order],
            residual
        )

        site_penalty = (
            site_sums
            / site_counts
        ).square().mean()

        loss = (
            cox_loss(
                risk,
                ttr[order],
                etr[order]
            )
            + L2_RESIDUAL
            * residual.square().mean()
            + L2_SITE
            * site_penalty
        )

        if not torch.isfinite(
            loss
        ):
            raise RuntimeError(
                "Non-finite neural Cox loss."
            )

        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(),
            2.0
        )

        optimizer.step()

        if search:

            held_residual = predict_residual(
                model,
                xte,
                hte,
                graph
            )

            for gamma in GAMMAS[1:]:

                score = concordance(
                    times[held],
                    events[held],
                    clinical_held
                    + gamma * held_residual
                )

                if (
                    np.isfinite(score)
                    and score > best[0] + 1e-5
                ):

                    best = (
                        float(score),
                        epoch,
                        float(gamma)
                    )

                    last_improvement = epoch

            if (
                epoch >= 9
                and epoch - last_improvement
                >= PATIENCE
            ):
                break

    if search:

        # Reject weak inner-CV improvements.
        if best[0] - baseline_c < MIN_GAIN:

            return (
                1,
                0.0,
                baseline_c,
                baseline_c
            )

        return (
            best[1],
            best[2],
            baseline_c,
            best[0]
        )

    return (
        clinical_train,
        clinical_held,
        predict_residual(
            model,
            xte,
            hte,
            graph
        ),
        model.state_dict()
    )


# ============================================================
# 9. Select an inner site-disjoint fold
# ============================================================

def choose_inner(
    fit,
    histology,
    events,
    site,
    outer_fold
):

    strata = (
        2 * events[fit].astype(int)
        + histology[fit]
    )

    candidates = []

    for seed in (
        42 + outer_fold,
        42,
        59,
        73,
        89,
        127
    ):

        splitter = StratifiedGroupKFold(
            n_splits=4,
            shuffle=True,
            random_state=seed
        )

        for j, (a, b) in enumerate(
            splitter.split(
                np.arange(len(fit)),
                strata,
                site[fit]
            )
        ):

            inner_fit = fit[a]
            inner_held = fit[b]

            if (
                len(
                    np.unique(
                        events[inner_held]
                    )
                ) == 2
                and events[
                    inner_held
                ].sum() >= 12
                and not (
                    set(site[inner_fit])
                    & set(site[inner_held])
                )
            ):

                deviation = (
                    abs(
                        events[inner_held].mean()
                        - events[fit].mean()
                    )
                    + abs(
                        len(inner_held) / len(fit)
                        - 0.25
                    )
                )

                candidates.append(
                    (
                        deviation,
                        seed,
                        j,
                        inner_fit,
                        inner_held
                    )
                )

    if not candidates:
        raise RuntimeError(
            "Unable to construct an inner site-disjoint split."
        )

    _, _, _, tr, held = min(
        candidates,
        key=lambda item: (
            item[0],
            item[1],
            item[2]
        )
    )

    return tr, held


# ============================================================
# 10. Matched Stage 21 outer-fold evaluation
# ============================================================

def run_stage23():

    torch.set_num_threads(
        THREADS
    )

    (
        frame,
        times,
        events,
        histology,
        site,
        eligible,
        raw,
        graphs,
        edge_count
    ) = load_data()

    print("=" * 72)
    print(
        "STAGE 23 - CLINICAL-ANCHORED RESIDUAL SURVIVAL GAT"
    )
    print("=" * 72)

    print(
        f"Eligible={len(eligible)}, "
        f"events={int(events[eligible].sum())}, "
        f"selected genes={N_NODES}, "
        f"PPI edges={edge_count}"
    )

    print(
        "Validation/test: untouched"
    )

    input_files = (
        PPI / "string_functional_700_corrected.npz",
        RAW / "rna_tpm.npy",
        RAW / "cnv_absolute.npy",
        S21 / "stage21_training_clinical_endpoints.csv"
    )

    fingerprint = hashlib.sha256(
        json.dumps(
            {
                "version": VERSION,
                "cases": frame.case_id.tolist(),
                "PFI": np.nan_to_num(
                    events, nan=-1
                ).tolist(),
                "time": np.nan_to_num(
                    times, nan=-1
                ).tolist(),
                "settings": (
                    N_NODES,
                    MAX_EPOCHS,
                    PATIENCE,
                    BATCH_SIZE,
                    LR,
                    L2_RESIDUAL,
                    L2_SITE,
                    GAMMAS,
                    MIN_GAIN
                ),
                "input_stats": {
                    str(p): (
                        p.stat().st_size,
                        p.stat().st_mtime_ns
                    )
                    for p in input_files
                }
            },
            sort_keys=True
        ).encode()
    ).hexdigest()

    oof = {
        mode: np.full(
            len(frame),
            np.nan
        )
        for mode in MODES
    }

    clinical_oof = np.full(
        len(frame),
        np.nan
    )

    rows = []
    selected_parameters = []

    for fold in range(1, 5):

        previous = joblib.load(
            S21 / f"pfi_outer_fold{fold}.joblib"
        )

        held = np.asarray(
            previous["predictions"]["index"],
            dtype=int
        )

        fit = np.setdiff1d(
            eligible,
            held
        )

        reference_clinical = np.asarray(
            previous["predictions"]["clinical_risk"],
            dtype=float
        )

        if (
            len(held) != len(reference_clinical)
            or not np.isin(
                held,
                eligible
            ).all()
            or set(site[fit]) & set(site[held])
        ):
            raise RuntimeError(
                "Stage 21 fold alignment or site overlap problem."
            )

        # Verify exact clinical baseline reproduction.
        _, reconstructed = clinical_anchor(
            frame,
            times,
            events,
            fit,
            held
        )

        baseline_error = float(
            np.max(
                np.abs(
                    reconstructed - reference_clinical
                )
            )
        )

        if baseline_error > 1e-3:
            raise RuntimeError(
                "Clinical Cox anchor does not reproduce "
                f"Stage 21 fold {fold}: {baseline_error}"
            )

        clinical_oof[held] = (
            reference_clinical
        )

        inner_fit, inner_held = choose_inner(
            fit,
            histology,
            events,
            site,
            fold
        )

        baseline_c = concordance(
            times[held],
            events[held],
            reference_clinical
        )

        row = {
            "fold": fold,
            "held": len(held),
            "events": int(
                events[held].sum()
            ),
            "clinical_C": baseline_c,
            "baseline_max_abs_error": (
                baseline_error
            )
        }

        print(
            f"\nOUTER FOLD {fold}: "
            f"fit={len(fit)}, "
            f"held={len(held)}, "
            f"events={int(events[held].sum())}, "
            f"clinical C={baseline_c:.4f}",
            flush=True
        )

        for mode in MODES:

            checkpoint = (
                OUT / f"fold{fold}_{mode}.joblib"
            )

            if checkpoint.exists():

                saved = joblib.load(
                    checkpoint
                )

                if (
                    saved.get("signature")
                    != fingerprint
                    or not np.array_equal(
                        saved["held"],
                        held
                    )
                ):
                    raise RuntimeError(
                        f"Incompatible checkpoint: {checkpoint}"
                    )

                predicted = np.asarray(
                    saved["risk"],
                    dtype=float
                )

                epoch = saved["epoch"]
                gamma = saved["gamma"]

            else:

                (
                    epoch,
                    gamma,
                    inner_baseline,
                    inner_best
                ) = fit_residual(
                    raw,
                    frame,
                    times,
                    events,
                    histology,
                    site,
                    inner_fit,
                    inner_held,
                    graphs[mode],
                    seed=SEED + fold,
                    max_epochs=MAX_EPOCHS,
                    search=True
                )

                if gamma == 0:

                    predicted = (
                        reference_clinical.copy()
                    )

                    model_weights = None

                else:

                    (
                        _,
                        outer_baseline,
                        residual,
                        model_weights
                    ) = fit_residual(
                        raw,
                        frame,
                        times,
                        events,
                        histology,
                        site,
                        fit,
                        held,
                        graphs[mode],
                        seed=SEED + fold,
                        max_epochs=epoch,
                        search=False
                    )

                    if np.max(
                        np.abs(
                            outer_baseline
                            - reference_clinical
                        )
                    ) > 1e-3:
                        raise RuntimeError(
                            "Outer clinical anchor mismatch."
                        )

                    predicted = (
                        outer_baseline
                        + gamma * residual
                    )

                temporary = checkpoint.with_name(
                    checkpoint.name + ".tmp"
                )

                joblib.dump(
                    {
                        "signature": fingerprint,
                        "held": held,
                        "risk": predicted,
                        "epoch": epoch,
                        "gamma": gamma,
                        "weights": model_weights,
                        "inner_clinical_C": inner_baseline,
                        "inner_selected_C": inner_best
                    },
                    temporary,
                    compress=3
                )

                os.replace(
                    temporary,
                    checkpoint
                )

            oof[mode][held] = (
                predicted
            )

            row[f"{mode}_C"] = concordance(
                times[held],
                events[held],
                predicted
            )

            row[f"{mode}_gamma"] = gamma
            row[f"{mode}_epoch"] = epoch

            selected_parameters.append({
                "fold": fold,
                "mode": mode,
                "gamma": gamma,
                "epoch": epoch
            })

            print(
                f"  {mode:<22} "
                f"gamma={gamma:.2f} "
                f"epoch={epoch:2d} "
                f"outer C={row[f'{mode}_C']:.4f}",
                flush=True
            )

        rows.append(row)

    # ========================================================
    # 11. Save fold-level results and OOF predictions
    # ========================================================

    table = pd.DataFrame(
        rows
    )

    table.to_csv(
        OUT / "stage23_residual_gat_site_folds.csv",
        index=False
    )

    pd.DataFrame({
        "case_id": frame.case_id.iloc[
            eligible
        ].to_numpy(),
        "PFI.time": times[eligible],
        "PFI": events[eligible],
        "site": site[eligible],
        "clinical_risk": clinical_oof[eligible],
        **{
            mode: oof[mode][eligible]
            for mode in MODES
        }
    }).to_csv(
        OUT / "stage23_training_oof_predictions.csv",
        index=False
    )

    summary = {
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "signature": fingerprint,
        "clinical_mean_C": float(
            table.clinical_C.mean()
        ),
        "neural_mean_C": {
            mode: float(
                table[f"{mode}_C"].mean()
            )
            for mode in MODES
        },
        "functional_minus_null": float(
            (
                table.functional_C
                - table.degree_preserving_null_C
            ).mean()
        ),
        "functional_minus_graphfree": float(
            (
                table.functional_C
                - table.graphfree_C
            ).mean()
        ),
        "fold_gamma": selected_parameters,
        "limitations": [
            "Stage09 node selection used complete training partition stage labels.",
            "A single inner held-out site fold tunes epochs and residual mixing.",
            "This is an architecture feasibility study, not unbiased external validation.",
            "Cross-fold risk-score scales may vary; report fold-mean C-index.",
            "Functional PPI benefit requires superiority over the degree-preserving null."
        ]
    }

    (
        OUT / "stage23_summary.json"
    ).write_text(
        json.dumps(
            summary,
            indent=2
        ),
        encoding="utf-8"
    )

    # ========================================================
    # 12. Final results
    # ========================================================

    print("\n" + "=" * 72)
    print(
        "STAGE 23 FINAL - CLINICAL-ANCHORED RESIDUAL GAT"
    )
    print("=" * 72)

    print(
        table.to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

    print(
        "Clinical mean C:",
        round(
            summary["clinical_mean_C"],
            4
        )
    )

    print(
        "Neural mean C:",
        summary["neural_mean_C"]
    )

    print(
        "Functional minus null:",
        round(
            summary["functional_minus_null"],
            4
        )
    )

    print(
        "Functional minus graphfree:",
        round(
            summary["functional_minus_graphfree"],
            4
        )
    )

    print(
        "Saved:",
        OUT
    )

    print(
        "Validation/test evaluated: NO"
    )

    print("STAGE 23 COMPLETED")


run_stage23()

STAGE 23 - CLINICAL-ANCHORED RESIDUAL SURVIVAL GAT
Eligible=657, events=225, selected genes=512, PPI edges=1994
Validation/test: untouched

OUTER FOLD 1: fit=491, held=166, events=56, clinical C=0.6489
  graphfree              gamma=0.00 epoch= 1 outer C=0.6489
  functional             gamma=0.00 epoch= 1 outer C=0.6489
  degree_preserving_null gamma=0.00 epoch= 1 outer C=0.6489

OUTER FOLD 2: fit=494, held=163, events=55, clinical C=0.6421
  graphfree              gamma=0.00 epoch= 1 outer C=0.6421
  functional             gamma=0.00 epoch= 1 outer C=0.6421
  degree_preserving_null gamma=1.00 epoch=28 outer C=0.6428

OUTER FOLD 3: fit=493, held=164, events=57, clinical C=0.6660
  graphfree              gamma=0.00 epoch= 1 outer C=0.6660
  functional             gamma=0.00 epoch= 1 outer C=0.6660
  degree_preserving_null gamma=0.00 epoch= 1 outer C=0.6660

OUTER FOLD 4: fit=493, held=164, events=57, clinical C=0.5696
  graphfree              gamma=0.00 epoch= 1 outer C=0.5696
  functio

In [23]:

import hashlib
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from scipy import sparse
from scipy.optimize import minimize
from scipy.stats import rankdata

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


# ============================================================
# 1. Configuration
# ============================================================

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")

RAW = ROOT / "data" / "processed" / "gene_level"
S19 = ROOT / "results" / "metrics" / "stage19"
S21 = ROOT / "results" / "metrics" / "stage21"

OUT = ROOT / "results" / "metrics" / "stage24"
MODELS = ROOT / "models" / "checkpoints" / "stage24"

OUT.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

SEED = 42
THREADS = min(4, os.cpu_count() or 1)

GAMMAS = (0.0, 0.25, 0.50, 1.0)

MIN_GAIN = 0.005
BOOTSTRAPS = 300

VERSION = "stage24_consensus_pathway_offset_cox_v1"


# ============================================================
# 2. Predeclare candidate survival architectures
# ============================================================

SPECS = [
    {
        "name": "Clinical_Only",
        "kind": "none",
        "k": 0,
        "alpha": 0.0,
        "interact": False
    },
    {
        "name": "Offset_SVD8",
        "kind": "svd",
        "k": 8,
        "alpha": 0.10,
        "interact": False
    },
    {
        "name": "Offset_SVD16",
        "kind": "svd",
        "k": 16,
        "alpha": 0.10,
        "interact": False
    },
    {
        "name": "Offset_Score8",
        "kind": "score",
        "k": 8,
        "alpha": 0.10,
        "interact": False
    },
    {
        "name": "Offset_Score16",
        "kind": "score",
        "k": 16,
        "alpha": 0.10,
        "interact": False
    },
    {
        "name": "Offset_SiteConsensus8",
        "kind": "consensus",
        "k": 8,
        "alpha": 0.10,
        "interact": False
    },
    {
        "name": "Offset_SiteConsensus16",
        "kind": "consensus",
        "k": 16,
        "alpha": 0.10,
        "interact": False
    },
    {
        "name": "Offset_SiteConsensus8_Strong",
        "kind": "consensus",
        "k": 8,
        "alpha": 0.30,
        "interact": False
    },
    {
        "name": "Offset_SiteConsensus8_HistInteraction",
        "kind": "consensus",
        "k": 8,
        "alpha": 0.30,
        "interact": True
    }
]


# ============================================================
# 3. Verify training patients and survival endpoints
# ============================================================

clinical = pd.read_csv(
    S21 / "stage21_training_clinical_endpoints.csv",
    dtype={"case_id": str}
)

cases = pd.read_csv(
    RAW / "case_order.csv",
    dtype={"case_id": str}
)

locked = pd.read_csv(
    ROOT / "data" / "splits" / "locked_four_omics_v1.csv",
    dtype={"case_id": str}
)

genes = pd.read_csv(
    RAW / "gene_reference.csv",
    dtype={"ensembl_id": str}
)

W = sparse.load_npz(
    S19 / "reactome_gene_pathway_weights.npz"
).tocsr().astype(np.float32)

metadata = pd.read_csv(
    S19 / "reactome_pathway_reference.csv"
)

if not (
    cases.case_id.is_unique
    and locked.case_id.is_unique
    and clinical.case_id.is_unique
):
    raise RuntimeError(
        "Duplicate TCGA case identifiers."
    )

joined = cases[
    ["case_id", "histology", "stage_group"]
].merge(
    locked[
        ["case_id", "histology", "stage_group", "split"]
    ],
    on="case_id",
    validate="one_to_one",
    how="left",
    suffixes=("_raw", "_locked")
)

if not (
    joined.histology_raw.equals(joined.histology_locked)
    and joined.stage_group_raw.equals(
        joined.stage_group_locked
    )
):
    raise RuntimeError(
        "Locked clinical stage/histology mismatch."
    )

tr = np.flatnonzero(
    joined.split.eq("train").to_numpy()
)

va = np.flatnonzero(
    joined.split.eq("validation").to_numpy()
)

te = np.flatnonzero(
    joined.split.eq("test").to_numpy()
)

if (
    len(tr),
    len(va),
    len(te)
) != (667, 143, 143):
    raise RuntimeError(
        "Locked cohort dimensions changed."
    )

if (
    len(clinical) != len(tr)
    or not np.array_equal(
        clinical.case_id.to_numpy(),
        joined.case_id.iloc[tr].to_numpy()
    )
):
    raise RuntimeError(
        "Stage 21 PFI clinical records are out of order."
    )

if (
    W.shape != (len(metadata), len(genes))
    or len(genes) != 19944
):
    raise RuntimeError(
        "Reactome mapping/gene dimensions changed."
    )

eligible = np.flatnonzero(
    clinical.PFI_valid.astype(str)
    .str.lower()
    .eq("true")
    .to_numpy()
)

time_all = pd.to_numeric(
    clinical["PFI.time"],
    errors="coerce"
).to_numpy(float)

event_all = pd.to_numeric(
    clinical.PFI,
    errors="coerce"
).to_numpy(float)

hist_all = (
    clinical.histology_matrix.eq("LUSC")
    .to_numpy(int)
)

site_all = (
    clinical.site.astype(str).to_numpy()
)

if (
    len(eligible),
    int(event_all[eligible].sum())
) != (657, 225):
    raise RuntimeError(
        "Expected Stage 21 PFI cohort of 657 patients "
        "and 225 events."
    )

if (
    not np.isfinite(time_all[eligible]).all()
    or not np.isin(
        event_all[eligible],
        [0, 1]
    ).all()
):
    raise RuntimeError(
        "PFI event or time is invalid."
    )


# ============================================================
# 4. Reconstruct single-sample Reactome RNA scores
# ============================================================

rna = np.load(
    RAW / "rna_tpm.npy",
    mmap_mode="r"
)

if rna.shape != (
    len(cases),
    len(genes)
):
    raise RuntimeError(
        "Raw RNA shape mismatch."
    )

values = np.asarray(
    rna[tr],
    dtype=np.float32
).copy()

if np.mean(
    np.isfinite(values) & (values >= 0)
) < 0.95:
    raise RuntimeError(
        "Unexpectedly low RNA completeness."
    )

values = np.log1p(
    np.where(
        np.isfinite(values) & (values >= 0),
        values,
        0
    )
).astype(np.float32)

rank = rankdata(
    values,
    axis=1,
    method="average"
).astype(np.float32)

rank = (
    rank - (len(genes) + 1) / 2.0
) / len(genes)

X = np.asarray(
    (W @ rank.T).T,
    dtype=np.float32
)

del values, rank, rna

if not np.isfinite(X).all():
    raise RuntimeError(
        "Non-finite Reactome pathway scores."
    )

print("=" * 76)
print(
    "STAGE 24 - SITE-CONSENSUS CLINICAL-OFFSET PATHWAY SURVIVAL"
)
print("=" * 76)

print(
    f"Training={len(tr)}, "
    f"PFI eligible={len(eligible)}, "
    f"events={int(event_all[eligible].sum())}"
)

print(
    f"Pathways={X.shape[1]}; "
    f"validation={len(va)} and test={len(te)} "
    "molecular features untouched"
)


# ============================================================
# 5. Reproduce Stage 21 clinical preprocessing
# ============================================================

def clinical_matrix(fit, held):

    ages = pd.to_numeric(
        clinical.age_years,
        errors="coerce"
    ).to_numpy(float)

    males = pd.to_numeric(
        clinical.male,
        errors="coerce"
    ).to_numpy(float)

    stages = pd.to_numeric(
        clinical.stage_major,
        errors="coerce"
    ).to_numpy(float)

    available = ages[fit][
        np.isfinite(ages[fit])
    ]

    median = (
        float(np.median(available))
        if len(available)
        else 65.0
    )

    def block(ids):

        a = np.where(
            np.isfinite(ages[ids]),
            ages[ids],
            median
        )

        return np.column_stack([
            (a - median) / 10.0,
            (~np.isfinite(ages[ids])).astype(float),
            np.where(
                np.isfinite(males[ids]),
                males[ids],
                0.5
            ),
            (~np.isfinite(males[ids])).astype(float),
            hist_all[ids],
            (stages[ids] == 2).astype(float),
            (stages[ids] == 3).astype(float),
            (stages[ids] == 4).astype(float)
        ])

    a = block(fit)
    b = block(held)

    scaler = StandardScaler().fit(a)

    return (
        scaler.transform(a),
        scaler.transform(b),
        {
            "age_median": median,
            "scaler": scaler
        }
    )


# ============================================================
# 6. Penalized Cox loss with optional fixed clinical offset
# ============================================================

def fit_cox(
    z,
    t,
    e,
    penalty,
    offset=None
):

    z = np.asarray(
        z,
        dtype=np.float64
    )

    t = np.asarray(
        t,
        dtype=np.float64
    )

    e = np.asarray(
        e,
        dtype=np.int64
    )

    offset = (
        np.zeros(len(t))
        if offset is None
        else np.asarray(
            offset,
            dtype=np.float64
        )
    )

    if (
        z.ndim != 2
        or z.shape[0] != len(t)
        or e.sum() < 10
    ):
        raise RuntimeError(
            "Invalid Cox input dimensions/events."
        )

    order = np.argsort(
        -t,
        kind="stable"
    )

    z = z[order]
    t = t[order]
    e = e[order]
    offset = offset[order]

    boundaries = np.r_[
        0,
        np.flatnonzero(
            t[1:] != t[:-1]
        ) + 1
    ]

    ends = np.r_[
        boundaries[1:] - 1,
        len(t) - 1
    ]

    event_counts = np.add.reduceat(
        e,
        boundaries
    )

    active = event_counts > 0

    ends = ends[active]

    event_counts = (
        event_counts[active]
        .astype(np.float64)
    )

    n_events = float(e.sum())

    def objective(beta):

        eta = z @ beta + offset

        shift = eta.max()

        w = np.exp(
            eta - shift
        )

        cw = np.cumsum(w)

        cx = np.cumsum(
            w[:, None] * z,
            axis=0
        )

        denom = np.maximum(
            cw[ends],
            1e-300
        )

        loss = (
            np.sum(
                event_counts
                * (
                    shift + np.log(denom)
                )
            )
            - eta[e == 1].sum()
        ) / n_events

        grad = (
            np.sum(
                event_counts[:, None]
                * cx[ends]
                / denom[:, None],
                axis=0
            )
            - z[e == 1].sum(axis=0)
        ) / n_events

        return (
            float(
                loss
                + 0.5
                * penalty
                * (beta @ beta)
            ),
            grad + penalty * beta
        )

    result = minimize(
        objective,
        np.zeros(z.shape[1]),
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter": 250,
            "ftol": 1e-9,
            "gtol": 1e-6
        }
    )

    if (
        not np.isfinite(result.x).all()
        or not np.isfinite(result.fun)
    ):
        raise RuntimeError(
            "Cox optimization diverged."
        )

    return result.x


# ============================================================
# 7. Survival concordance and frozen clinical anchor
# ============================================================

def cindex(time, event, risk):

    comparable = (
        (time[:, None] < time[None, :])
        & (event[:, None] == 1)
    )

    n = int(
        comparable.sum()
    )

    if n < 10:
        return float("nan")

    dif = (
        risk[:, None] - risk[None, :]
    )

    concordant = np.count_nonzero(
        comparable & (dif > 1e-10)
    )

    tied = np.count_nonzero(
        comparable
        & (np.abs(dif) <= 1e-10)
    )

    return float(
        (concordant + 0.5 * tied) / n
    )


def fit_anchor(fit, held):

    a, b, state = clinical_matrix(
        fit,
        held
    )

    beta = fit_cox(
        a,
        time_all[fit],
        event_all[fit],
        penalty=0.003 * 0.15
    )

    return (
        (a @ beta),
        (b @ beta),
        {
            "beta": beta,
            **state
        }
    )


# ============================================================
# 8. Clinically adjusted pathway survival scores
# ============================================================

def cox_pathway_z(
    z,
    time,
    event,
    offset
):

    idx = np.argsort(
        -time,
        kind="stable"
    )

    zz = z[idx].astype(
        np.float64
    )

    t = time[idx]

    e = event[idx].astype(int)

    off = offset[idx]

    w = np.exp(
        off - off.max()
    )

    cw = np.cumsum(w)

    cx = np.cumsum(
        w[:, None] * zz,
        axis=0
    )

    cx2 = np.cumsum(
        w[:, None] * zz * zz,
        axis=0
    )

    starts = np.r_[
        0,
        np.flatnonzero(
            t[1:] != t[:-1]
        ) + 1
    ]

    ends = np.r_[
        starts[1:] - 1,
        len(t) - 1
    ]

    d = np.add.reduceat(
        e,
        starts
    )

    active = d > 0

    ends = ends[active]
    d = d[active]

    avg = (
        cx[ends]
        / np.maximum(
            cw[ends, None],
            1e-300
        )
    )

    avg2 = (
        cx2[ends]
        / np.maximum(
            cw[ends, None],
            1e-300
        )
    )

    u = (
        (e[:, None] * zz).sum(axis=0)
        - (d[:, None] * avg).sum(axis=0)
    )

    v = (
        d[:, None]
        * np.maximum(
            avg2 - avg * avg,
            0
        )
    ).sum(axis=0)

    return (
        u
        / np.sqrt(
            np.maximum(v, 1e-8)
        )
    )


# ============================================================
# 9. Site-balanced consensus groups
# ============================================================

def site_subgroups(fit, n_groups=3):

    unique, inverse, counts = np.unique(
        site_all[fit],
        return_inverse=True,
        return_counts=True
    )

    site_events = np.bincount(
        inverse,
        weights=event_all[fit],
        minlength=len(unique)
    )

    order = np.argsort(
        -counts,
        kind="stable"
    )

    assignments = np.empty(
        len(unique),
        dtype=int
    )

    group_sizes = np.zeros(
        n_groups,
        dtype=float
    )

    group_events = np.zeros(
        n_groups,
        dtype=float
    )

    for j in order:

        g = min(
            range(n_groups),
            key=lambda k: (
                group_sizes[k]
                + 1.8 * group_events[k],
                k
            )
        )

        assignments[j] = g

        group_sizes[g] += counts[j]

        group_events[g] += site_events[j]

    groups = assignments[inverse]

    if any(
        (groups == g).sum() < 30
        or event_all[
            fit[groups == g]
        ].sum() < 10
        for g in range(n_groups)
    ):
        return None

    return groups


# ============================================================
# 10. Fit a pathway residual model within the fitting fold
# ============================================================

def fit_candidate(
    fit,
    held,
    spec,
    anchor_fit,
    anchor_held
):

    if spec["kind"] == "none":

        return (
            np.zeros(len(held)),
            {
                "kind": "none",
                "coef": np.zeros(0)
            }
        )

    scaler = StandardScaler().fit(
        X[fit]
    )

    fit_z = np.clip(
        scaler.transform(X[fit]),
        -7,
        7
    ).astype(np.float32)

    held_z = np.clip(
        scaler.transform(X[held]),
        -7,
        7
    ).astype(np.float32)

    state = {
        "kind": spec["kind"],
        "scaler": scaler,
        "selected": None,
        "svd": None,
        "latent_scaler": None,
        "interact": spec["interact"]
    }

    if spec["kind"] == "svd":

        svd = TruncatedSVD(
            n_components=spec["k"],
            n_iter=4,
            random_state=SEED
        )

        fit_z = svd.fit_transform(
            fit_z
        )

        held_z = svd.transform(
            held_z
        )

        state["svd"] = svd

    else:

        score = cox_pathway_z(
            fit_z,
            time_all[fit],
            event_all[fit],
            anchor_fit
        )

        if spec["kind"] == "consensus":

            groups = site_subgroups(fit)

            if groups is not None:

                signs = []

                for group in range(3):

                    m = groups == group

                    subscore = cox_pathway_z(
                        fit_z[m],
                        time_all[fit][m],
                        event_all[fit][m],
                        anchor_fit[m]
                    )

                    signs.append(
                        np.sign(subscore)
                        == np.sign(score)
                    )

                agreement = np.mean(
                    np.stack(signs),
                    axis=0
                )

                score *= agreement ** 2

        selected = np.sort(
            np.argsort(
                -np.abs(score),
                kind="stable"
            )[:spec["k"]]
        )

        fit_z = fit_z[:, selected]

        held_z = held_z[:, selected]

        state["selected"] = selected

    latent = StandardScaler().fit(
        fit_z
    )

    fit_z = np.clip(
        latent.transform(fit_z),
        -7,
        7
    )

    held_z = np.clip(
        latent.transform(held_z),
        -7,
        7
    )

    state["latent_scaler"] = latent

    if spec["interact"]:

        sign_fit = (
            hist_all[fit] * 2 - 1
        )[:, None]

        sign_held = (
            hist_all[held] * 2 - 1
        )[:, None]

        fit_z = np.column_stack([
            fit_z,
            0.5 * fit_z * sign_fit
        ])

        held_z = np.column_stack([
            held_z,
            0.5 * held_z * sign_held
        ])

    beta = fit_cox(
        fit_z,
        time_all[fit],
        event_all[fit],
        spec["alpha"],
        offset=anchor_fit
    )

    state["coef"] = beta

    return (
        held_z @ beta,
        state
    )


# ============================================================
# 11. Reuse exact Stage 21 site-held-out folds
# ============================================================

outer = []

for fold in range(1, 5):

    saved = joblib.load(
        S21 / f"pfi_outer_fold{fold}.joblib"
    )

    held = np.asarray(
        saved["predictions"]["index"],
        dtype=int
    )

    fit = np.setdiff1d(
        eligible,
        held
    )

    if (
        set(site_all[fit])
        & set(site_all[held])
    ):
        raise RuntimeError(
            "Stage 21 outer folds are not site disjoint."
        )

    outer.append((
        fold,
        fit,
        held,
        np.asarray(
            saved["predictions"]["clinical_risk"]
        )
    ))

if set(
    np.concatenate(
        [fold[2] for fold in outer]
    )
) != set(eligible):
    raise RuntimeError(
        "Stage 21 outer folds do not partition "
        "the eligible PFI patients."
    )


# ============================================================
# 12. Construct inner site-disjoint folds
# ============================================================

def inner_site_folds(fit):

    strata = (
        2 * event_all[fit].astype(int)
        + hist_all[fit]
    )

    choices = []

    for seed in (
        42, 59, 73, 89,
        101, 127, 151, 173
    ):

        try:

            folds = list(
                StratifiedGroupKFold(
                    n_splits=3,
                    shuffle=True,
                    random_state=seed
                ).split(
                    np.arange(len(fit)),
                    strata,
                    site_all[fit]
                )
            )

        except ValueError:
            continue

        valid = all(
            event_all[
                fit[b]
            ].sum() >= 15
            and event_all[
                fit[a]
            ].sum() >= 30
            and set(
                site_all[fit[a]]
            ).isdisjoint(
                site_all[fit[b]]
            )
            and len(
                np.unique(
                    event_all[fit[b]]
                )
            ) == 2
            for a, b in folds
        )

        if valid:

            deviation = sum(
                abs(
                    len(b) / len(fit) - 1 / 3
                )
                + abs(
                    event_all[fit[b]].mean()
                    - event_all[fit].mean()
                )
                for a, b in folds
            )

            choices.append((
                deviation,
                seed,
                [
                    (fit[a], fit[b])
                    for a, b in folds
                ]
            ))

    if not choices:
        raise RuntimeError(
            "Cannot make event-balanced inner site folds."
        )

    return min(
        choices,
        key=lambda entry: (
            entry[0],
            entry[1]
        )
    )[2]


# ============================================================
# 13. Build reproducibility signature
# ============================================================

source_stats = {
    str(p): (
        p.stat().st_size,
        p.stat().st_mtime_ns
    )
    for p in (
        RAW / "rna_tpm.npy",
        S19 / "reactome_gene_pathway_weights.npz",
        S21 / "stage21_training_clinical_endpoints.csv"
    )
}

signature = hashlib.sha256(
    json.dumps(
        {
            "version": VERSION,
            "cases": clinical.case_id.tolist(),
            "eligible": eligible.tolist(),
            "times": np.nan_to_num(
                time_all,
                nan=-1
            ).tolist(),
            "events": np.nan_to_num(
                event_all,
                nan=-1
            ).tolist(),
            "sources": source_stats,
            "models": SPECS,
            "gammas": GAMMAS,
            "threshold": MIN_GAIN
        },
        sort_keys=True
    ).encode()
).hexdigest()


# ============================================================
# 14. Nested model selection and matched outer evaluation
# ============================================================

fold_results = []
inner_results = []

oof_clinical = np.full(
    len(clinical),
    np.nan
)

oof_selected = np.full(
    len(clinical),
    np.nan
)

with threadpool_limits(limits=THREADS):

    for fold, fit, held, historic_anchor in outer:

        checkpoint = (
            OUT / f"outer_fold{fold}_verified.joblib"
        )

        if checkpoint.exists():

            saved = joblib.load(
                checkpoint
            )

            if (
                saved.get("signature") != signature
                or not np.array_equal(
                    saved["held"],
                    held
                )
            ):
                raise RuntimeError(
                    f"Incompatible existing result: {checkpoint}"
                )

            current = saved["summary"]

            inner_rows = saved["inner"]

            pred = saved["prediction"]

        else:

            # Reproduce the original clinical-only risk.
            _, anchor_held, _ = fit_anchor(
                fit,
                held
            )

            max_error = float(
                np.max(
                    np.abs(
                        anchor_held - historic_anchor
                    )
                )
            )

            if max_error > 1e-3:
                raise RuntimeError(
                    "Stage 21 clinical anchor mismatch, "
                    f"fold={fold}: {max_error}"
                )

            inner = inner_site_folds(
                fit
            )

            candidates = {
                (
                    spec["name"],
                    gamma
                ): []
                for spec in SPECS
                for gamma in GAMMAS
            }

            inner_rows = []

            print(
                f"\nOUTER {fold}: fit={len(fit)}, "
                f"held={len(held)}, "
                f"events={int(event_all[held].sum())}",
                flush=True
            )

            for j, (inf, inh) in enumerate(
                inner,
                1
            ):

                anchor_fit, anchor_hold, _ = (
                    fit_anchor(
                        inf,
                        inh
                    )
                )

                base_c = cindex(
                    time_all[inh],
                    event_all[inh],
                    anchor_hold
                )

                for spec in SPECS:

                    pred_mol, _ = fit_candidate(
                        inf,
                        inh,
                        spec,
                        anchor_fit,
                        anchor_hold
                    )

                    for gamma in GAMMAS:

                        c = cindex(
                            time_all[inh],
                            event_all[inh],
                            anchor_hold
                            + gamma * pred_mol
                        )

                        gain = c - base_c

                        candidates[
                            (spec["name"], gamma)
                        ].append(
                            gain
                        )

                        inner_rows.append({
                            "outer_fold": fold,
                            "inner_fold": j,
                            "model": spec["name"],
                            "gamma": gamma,
                            "C": c,
                            "clinical_C": base_c,
                            "gain": gain
                        })

            ranking = sorted(
                [
                    (
                        float(
                            np.mean(gains)
                        ),
                        float(
                            np.std(gains)
                        ),
                        name,
                        gamma
                    )
                    for (
                        name,
                        gamma
                    ), gains in candidates.items()
                ],
                key=lambda z: (
                    -z[0],
                    z[1],
                    z[2],
                    z[3]
                )
            )

            best_gain, _, winning_name, gamma = (
                ranking[0]
            )

            if best_gain < MIN_GAIN:

                winning_name = "Clinical_Only"

                gamma = 0.0

                best_gain = 0.0

            winning = next(
                s
                for s in SPECS
                if s["name"] == winning_name
            )

            anchor_fit, anchor_held, anchor_state = (
                fit_anchor(
                    fit,
                    held
                )
            )

            mol_pred, molecular_state = (
                fit_candidate(
                    fit,
                    held,
                    winning,
                    anchor_fit,
                    anchor_held
                )
            )

            pred = (
                anchor_held
                + gamma * mol_pred
            )

            current = {
                "fold": fold,
                "n_held": len(held),
                "events": int(
                    event_all[held].sum()
                ),
                "model": winning_name,
                "gamma": gamma,
                "inner_gain": best_gain,
                "clinical_C": cindex(
                    time_all[held],
                    event_all[held],
                    anchor_held
                ),
                "selected_C": cindex(
                    time_all[held],
                    event_all[held],
                    pred
                ),
                "max_anchor_error": float(
                    np.max(
                        np.abs(
                            anchor_held
                            - historic_anchor
                        )
                    )
                )
            }

            current["delta_C"] = (
                current["selected_C"]
                - current["clinical_C"]
            )

            temp = checkpoint.with_name(
                checkpoint.name + ".tmp"
            )

            joblib.dump(
                {
                    "signature": signature,
                    "held": held,
                    "summary": current,
                    "inner": inner_rows,
                    "prediction": pred,
                    "molecular_state": molecular_state,
                    "clinical_state": anchor_state
                },
                temp,
                compress=3
            )

            os.replace(
                temp,
                checkpoint
            )

        fold_results.append(
            current
        )

        inner_results.extend(
            inner_rows
        )

        oof_selected[held] = pred

        oof_clinical[held] = (
            historic_anchor
        )

        print(
            f"  fold {fold}: {current['model']}, "
            f"gamma={current['gamma']:.2f}, "
            f"C={current['selected_C']:.4f}, "
            f"clinical={current['clinical_C']:.4f}",
            flush=True
        )


# ============================================================
# 15. Verify complete out-of-fold evaluation
# ============================================================

if not np.isfinite(
    oof_selected[eligible]
).all():
    raise RuntimeError(
        "Incomplete PFI outer-fold predictions."
    )

table = pd.DataFrame(
    fold_results
).sort_values("fold")

inner_table = pd.DataFrame(
    inner_results
)

table.to_csv(
    OUT / "stage24_nested_site_fold_comparison.csv",
    index=False
)

inner_table.to_csv(
    OUT / "stage24_inner_candidate_scores.csv",
    index=False
)

pd.DataFrame({
    "case_id": clinical.case_id.iloc[eligible],
    "site": site_all[eligible],
    "PFI_time": time_all[eligible],
    "PFI_event": event_all[eligible],
    "clinical_risk": oof_clinical[eligible],
    "chosen_risk": oof_selected[eligible]
}).to_csv(
    OUT / "stage24_training_site_oof.csv",
    index=False
)


# ============================================================
# 16. Exploratory site-cluster bootstrap uncertainty
# ============================================================

rng = np.random.default_rng(
    SEED + 24
)

interval = []

for _ in range(BOOTSTRAPS):

    gains = []

    for fold, fit, held, _ in outer:

        groups = np.unique(
            site_all[held]
        )

        sampled = rng.choice(
            groups,
            size=len(groups),
            replace=True
        )

        ix = np.concatenate([
            held[
                site_all[held] == group
            ]
            for group in sampled
        ])

        a = cindex(
            time_all[ix],
            event_all[ix],
            oof_selected[ix]
        )

        b = cindex(
            time_all[ix],
            event_all[ix],
            oof_clinical[ix]
        )

        if (
            np.isfinite(a)
            and np.isfinite(b)
        ):
            gains.append(
                a - b
            )

    if len(gains) == len(outer):

        interval.append(
            np.mean(gains)
        )

if len(interval) < 100:
    raise RuntimeError(
        "Too few valid site-bootstrap replicates."
    )


# ============================================================
# 17. Select final training-only model
# ============================================================

ranking = (
    inner_table
    .groupby(
        ["model", "gamma"],
        as_index=False
    )["gain"]
    .mean()
    .sort_values(
        "gain",
        ascending=False
    )
)

ranking.to_csv(
    OUT / "stage24_inner_model_ranking.csv",
    index=False
)

final_name = str(
    ranking.iloc[0]["model"]
)

final_gamma = float(
    ranking.iloc[0]["gamma"]
)

if (
    float(ranking.iloc[0]["gain"])
    < MIN_GAIN
):

    final_name = "Clinical_Only"

    final_gamma = 0.0

final_spec = next(
    spec
    for spec in SPECS
    if spec["name"] == final_name
)

# Fit the frozen clinical offset on all eligible training patients.
with threadpool_limits(limits=THREADS):

    _, _, clinical_state = fit_anchor(
        eligible,
        eligible
    )

    if final_gamma > 0:

        anchor, _, _ = fit_anchor(
            eligible,
            eligible
        )

        _, mol_state = fit_candidate(
            eligible,
            eligible,
            final_spec,
            anchor,
            anchor
        )

    else:

        mol_state = {
            "kind": "none",
            "coef": np.zeros(0)
        }


# ============================================================
# 18. Save final fitted model and research summary
# ============================================================

model_path = (
    MODELS
    / "stage24_frozen_clinical_pathway_model.joblib"
)

tmp = model_path.with_name(
    model_path.name + ".tmp"
)

joblib.dump(
    {
        "version": VERSION,
        "signature": signature,
        "spec": final_spec,
        "gamma": final_gamma,
        "clinical_state": clinical_state,
        "molecular_state": mol_state,
        "pathway_mapping": str(
            S19 / "reactome_gene_pathway_weights.npz"
        ),
        "training_case_ids": (
            clinical.case_id.iloc[eligible].tolist()
        ),
        "validation_test_unseen": True
    },
    tmp,
    compress=3
)

os.replace(
    tmp,
    model_path
)

summary = {
    "timestamp": datetime.now(
        timezone.utc
    ).isoformat(),
    "version": VERSION,
    "baseline_fold_mean_C": float(
        table.clinical_C.mean()
    ),
    "chosen_fold_mean_C": float(
        table.selected_C.mean()
    ),
    "mean_delta_C": float(
        table.delta_C.mean()
    ),
    "exploratory_site_bootstrap_95pct": (
        np.percentile(
            interval,
            [2.5, 97.5]
        ).tolist()
    ),
    "model_usage": (
        table.model.value_counts().to_dict()
    ),
    "final_training_only_selection": {
        "model": final_name,
        "gamma": final_gamma
    },
    "limitations": [
        "Internal site-CV only; not external validation.",
        "Stage21-23 results informed Stage24 hypotheses.",
        "Model selection does not guarantee improvement.",
        "Clinical offset uses stage, age, sex and histology.",
        "PFI C-index measures ranking, not calibration."
    ]
}

(
    OUT / "stage24_summary.json"
).write_text(
    json.dumps(
        summary,
        indent=2
    ),
    encoding="utf-8"
)


# ============================================================
# 19. Final results
# ============================================================

print("\n" + "=" * 76)

print(
    "STAGE 24 FINAL - CONSENSUS PATHWAY RESIDUAL SURVIVAL"
)

print("=" * 76)

print(
    table.to_string(
        index=False,
        float_format=lambda z: f"{z:.4f}"
    )
)

print(
    "Clinical mean C:",
    round(
        summary["baseline_fold_mean_C"],
        4
    )
)

print(
    "Chosen mean C:",
    round(
        summary["chosen_fold_mean_C"],
        4
    )
)

print(
    "Mean increment:",
    round(
        summary["mean_delta_C"],
        4
    )
)

print(
    "Site-cluster bootstrap exploratory 95%:",
    summary["exploratory_site_bootstrap_95pct"]
)

print(
    "Outer-fold model choices:",
    summary["model_usage"]
)

print(
    "Frozen full-training model:",
    final_name,
    "gamma",
    final_gamma
)

print(
    "Saved:",
    OUT,
    model_path
)

print(
    "Validation/test molecular features and "
    "survival outcomes evaluated: NO"
)

print("STAGE 24 COMPLETED")

STAGE 24 - SITE-CONSENSUS CLINICAL-OFFSET PATHWAY SURVIVAL
Training=667, PFI eligible=657, events=225
Pathways=1613; validation=143 and test=143 molecular features untouched

OUTER 1: fit=491, held=166, events=56
  fold 1: Offset_Score16, gamma=0.50, C=0.6412, clinical=0.6489

OUTER 2: fit=494, held=163, events=55
  fold 2: Offset_SVD8, gamma=1.00, C=0.6596, clinical=0.6421

OUTER 3: fit=493, held=164, events=57
  fold 3: Offset_SVD8, gamma=1.00, C=0.7075, clinical=0.6660

OUTER 4: fit=493, held=164, events=57
  fold 4: Offset_SVD16, gamma=0.50, C=0.5919, clinical=0.5696

STAGE 24 FINAL - CONSENSUS PATHWAY RESIDUAL SURVIVAL
 fold  n_held  events          model  gamma  inner_gain  clinical_C  selected_C  max_anchor_error  delta_C
    1     166      56 Offset_Score16 0.5000      0.0168      0.6489      0.6412            0.0000  -0.0078
    2     163      55    Offset_SVD8 1.0000      0.0277      0.6421      0.6596            0.0000   0.0174
    3     164      57    Offset_SVD8 1.0000    

In [24]:

import hashlib
import json
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import networkx as nx
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

from scipy import sparse
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler

# Configuration
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
RAW = ROOT / "data" / "processed" / "gene_level"
S19 = ROOT / "results" / "metrics" / "stage19"
S21 = ROOT / "results" / "metrics" / "stage21"
S24 = ROOT / "results" / "metrics" / "stage24"
OUT = ROOT / "results" / "metrics" / "stage25"

OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
N_PATHWAYS = 256
NEIGHBORS = 4
INNER_FOLDS = 3
EPOCH_CHECKPOINTS = (8, 16, 24)
GAMMAS = (0.25, 0.5, 1.0)
MIN_GRAPH_GAIN = 0.005
LR = 0.0015
THREADS = min(4, os.cpu_count() or 1)

VERSION = "stage25_clinical_offset_pathway_overlap_gat_v1"
MODES = ("graphfree", "reactome_overlap", "degree_null")

expected = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != expected.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

torch.set_num_threads(THREADS)

# Load the locked training partition
cases = pd.read_csv(
    RAW / "case_order.csv",
    dtype={"case_id": str}
)

locked = pd.read_csv(
    ROOT / "data" / "splits" / "locked_four_omics_v1.csv",
    dtype={"case_id": str}
)

clinical = pd.read_csv(
    S21 / "stage21_training_clinical_endpoints.csv",
    dtype={"case_id": str}
)

genes = pd.read_csv(
    RAW / "gene_reference.csv",
    dtype={"ensembl_id": str}
)

joined = cases[
    ["case_id", "histology", "stage_group"]
].merge(
    locked[
        ["case_id", "histology", "stage_group", "split"]
    ],
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_raw", "_locked")
)

if (
    not joined.case_id.is_unique
    or not locked.case_id.is_unique
    or not joined.histology_raw.equals(joined.histology_locked)
    or not joined.stage_group_raw.equals(joined.stage_group_locked)
):
    raise RuntimeError("Locked patient labels/order invalid.")

tr = np.flatnonzero(joined.split.eq("train"))
va = np.flatnonzero(joined.split.eq("validation"))
te = np.flatnonzero(joined.split.eq("test"))

if (len(tr), len(va), len(te)) != (667, 143, 143):
    raise RuntimeError("Cohort partition changed.")

if len(genes) != 19944:
    raise RuntimeError("Gene reference changed.")

if (
    len(clinical) != len(tr)
    or not np.array_equal(
        clinical.case_id.to_numpy(),
        joined.case_id.iloc[tr].to_numpy()
    )
):
    raise RuntimeError("Stage 21 clinical alignment changed.")

eligible = np.flatnonzero(
    clinical.PFI_valid.astype(str)
    .str.lower()
    .eq("true")
    .to_numpy()
)

time = pd.to_numeric(
    clinical["PFI.time"],
    errors="coerce"
).to_numpy(float)

event = pd.to_numeric(
    clinical.PFI,
    errors="coerce"
).to_numpy(float)

hist = clinical.histology_matrix.eq(
    "LUSC"
).to_numpy(int)

site = clinical.site.astype(str).to_numpy()

if (len(eligible), int(event[eligible].sum())) != (657, 225):
    raise RuntimeError(
        "Stage 21 PFI eligibility or event count changed."
    )

if (
    not np.isfinite(time[eligible]).all()
    or not np.isin(event[eligible], (0, 1)).all()
):
    raise RuntimeError("Invalid PFI survival labels.")

# Reconstruct Stage 19 Reactome pathway scores
W = sparse.load_npz(
    S19 / "reactome_gene_pathway_weights.npz"
).tocsr().astype(np.float32)

meta = pd.read_csv(
    S19 / "reactome_pathway_reference.csv"
)

if (
    W.shape != (len(meta), len(genes))
    or W.shape[0] < N_PATHWAYS
):
    raise RuntimeError(
        "Stage 19 Reactome mapping changed."
    )

raw_rna = np.load(
    RAW / "rna_tpm.npy",
    mmap_mode="r"
)

if raw_rna.shape != (953, len(genes)):
    raise RuntimeError("Unexpected raw RNA dimensions.")

values = np.asarray(
    raw_rna[tr],
    dtype=np.float32
).copy()

valid = np.isfinite(values) & (values >= 0)

if valid.mean() < 0.95:
    raise RuntimeError("RNA observation rate is too low.")

values = np.log1p(
    np.where(valid, values, 0)
).astype(np.float32)

ranks = rankdata(
    values,
    method="average",
    axis=1
).astype(np.float32)

ranks = (
    ranks - (len(genes) + 1) / 2
) / len(genes)

X = np.asarray(
    (W @ ranks.T).T,
    dtype=np.float32
)

del values, ranks, raw_rna

if not np.isfinite(X).all():
    raise RuntimeError("Non-finite Reactome features.")

print("=" * 78)
print(
    "STAGE 25 - CLINICAL-ANCHORED PATHWAY GRAPH "
    "RESIDUAL SURVIVAL"
)
print("=" * 78)

print(
    f"PFI eligible={len(eligible)}, "
    f"events={int(event[eligible].sum())}, "
    f"pathways={X.shape[1]}"
)

print(
    f"Matched Stage21/24 site-held-out folds; "
    f"validation={len(va)}, test={len(te)} untouched"
)


# Survival evaluation and Cox regression
def c_index(t, e, risk):

    pair = (
        (t[:, None] < t[None, :])
        & (e[:, None] == 1)
    )

    count = int(pair.sum())

    if count < 10:
        return np.nan

    delta = risk[:, None] - risk[None, :]

    concordant = (
        pair & (delta > 1e-10)
    ).sum()

    tied = (
        pair & (np.abs(delta) <= 1e-10)
    ).sum()

    return float(
        (concordant + 0.5 * tied) / count
    )


def cox_fit(z, t, e, ridge, offset=None):

    z = np.asarray(z, dtype=np.float64)

    offset = (
        np.zeros(len(t))
        if offset is None
        else np.asarray(offset, dtype=np.float64)
    )

    order = np.argsort(-t, kind="stable")

    a = z[order]
    tt = t[order]
    ee = e[order].astype(int)
    off = offset[order]

    starts = np.r_[
        0,
        np.flatnonzero(tt[1:] != tt[:-1]) + 1
    ]

    ends = np.r_[
        starts[1:] - 1,
        len(tt) - 1
    ]

    num = np.add.reduceat(ee, starts)

    ends = ends[num > 0]
    num = num[num > 0].astype(float)

    def objective(beta):

        risk = a @ beta + off
        shift = np.max(risk)

        weight = np.exp(risk - shift)
        cum = np.cumsum(weight)

        cumx = np.cumsum(
            weight[:, None] * a,
            axis=0
        )

        denom = np.maximum(
            cum[ends],
            1e-300
        )

        loss = (
            np.sum(
                num * (shift + np.log(denom))
            )
            - risk[ee == 1].sum()
        ) / ee.sum()

        grad = (
            np.sum(
                num[:, None]
                * cumx[ends]
                / denom[:, None],
                axis=0
            )
            - a[ee == 1].sum(axis=0)
        ) / ee.sum()

        penalty = 0.5 * ridge * (beta @ beta)

        return (
            float(loss + penalty),
            grad + ridge * beta
        )

    solution = minimize(
        objective,
        np.zeros(z.shape[1]),
        jac=True,
        method="L-BFGS-B",
        options={
            "maxiter": 250,
            "ftol": 1e-9,
            "gtol": 1e-6
        }
    )

    if (
        not np.isfinite(solution.x).all()
        or not np.isfinite(solution.fun)
    ):
        raise RuntimeError(
            "Cox optimizer returned non-finite output."
        )

    return solution.x


# Frozen Stage 21 clinical predictor
def clinical_offset(fit, hold):

    age = pd.to_numeric(
        clinical.age_years,
        errors="coerce"
    ).to_numpy(float)

    male = pd.to_numeric(
        clinical.male,
        errors="coerce"
    ).to_numpy(float)

    stage = pd.to_numeric(
        clinical.stage_major,
        errors="coerce"
    ).to_numpy(float)

    available = age[fit][
        np.isfinite(age[fit])
    ]

    median = (
        float(np.median(available))
        if len(available)
        else 65.0
    )

    def block(ids):

        a = np.where(
            np.isfinite(age[ids]),
            age[ids],
            median
        )

        return np.column_stack((
            (a - median) / 10.0,
            (~np.isfinite(age[ids])).astype(float),
            np.where(
                np.isfinite(male[ids]),
                male[ids],
                0.5
            ),
            (~np.isfinite(male[ids])).astype(float),
            hist[ids],
            (stage[ids] == 2).astype(float),
            (stage[ids] == 3).astype(float),
            (stage[ids] == 4).astype(float)
        ))

    a = block(fit)
    b = block(hold)

    scaler = StandardScaler().fit(a)

    a = scaler.transform(a)
    b = scaler.transform(b)

    beta = cox_fit(
        a,
        time[fit],
        event[fit],
        ridge=0.003 * 0.15
    )

    return a @ beta, b @ beta


# Build pathway-overlap graphs using curated gene memberships
def build_graphs(selected):

    membership = W[selected].copy().tocsr()
    membership.data[:] = 1.0

    common = (
        membership @ membership.T
    ).toarray()

    size = np.asarray(
        membership.sum(axis=1)
    ).ravel()

    denominator = np.maximum(
        size[:, None] + size[None, :] - common,
        1
    )

    similarity = common / denominator

    np.fill_diagonal(
        similarity,
        0.0
    )

    adjacency = np.zeros_like(
        similarity,
        dtype=np.float32
    )

    for i in range(len(selected)):

        neighbors = np.argsort(
            -similarity[i],
            kind="stable"
        )[:NEIGHBORS]

        neighbors = neighbors[
            similarity[i, neighbors] >= 0.02
        ]

        adjacency[i, neighbors] = (
            similarity[i, neighbors]
        )

    adjacency = np.maximum(
        adjacency,
        adjacency.T
    )

    graph = nx.from_numpy_array(
        adjacency
    )

    edge_count = graph.number_of_edges()

    if edge_count < 100:
        raise RuntimeError(
            "Insufficient pathway graph connectivity."
        )

    randomized = graph.copy()

    swap_counts = (
        2 * edge_count,
        edge_count,
        max(10, edge_count // 4)
    )

    for attempts in swap_counts:

        try:

            nx.double_edge_swap(
                randomized,
                nswap=attempts,
                max_tries=attempts * 90,
                seed=SEED + 25
            )

            break

        except nx.NetworkXAlgorithmError:
            continue

    if any(
        graph.degree(i) != randomized.degree(i)
        for i in range(len(selected))
    ):
        raise RuntimeError(
            "Randomized graph changed node degrees."
        )

    if set(graph.edges()) == set(randomized.edges()):
        raise RuntimeError(
            "Randomization did not change graph topology."
        )

    original_edges = sorted(graph.edges())
    null_edges = sorted(randomized.edges())

    weights = np.asarray(
        [
            graph[u][v]["weight"]
            for u, v in original_edges
        ],
        dtype=np.float32
    )

    null_weights = np.random.default_rng(
        SEED + 25
    ).permutation(weights)

    def tensorize(edges, values):

        pairs = np.asarray(
            edges,
            dtype=np.int64
        )

        src = np.r_[
            pairs[:, 0],
            pairs[:, 1],
            np.arange(len(selected))
        ]

        dst = np.r_[
            pairs[:, 1],
            pairs[:, 0],
            np.arange(len(selected))
        ]

        w = np.r_[
            values,
            values,
            np.ones(len(selected))
        ].astype(np.float32)

        return (
            torch.as_tensor(src),
            torch.as_tensor(dst),
            torch.as_tensor(w)
        )

    diagonal = torch.arange(
        len(selected)
    )

    graphs = {
        "graphfree": (
            diagonal,
            diagonal,
            torch.ones(len(selected))
        ),

        "reactome_overlap": tensorize(
            original_edges,
            weights
        ),

        "degree_null": tensorize(
            null_edges,
            null_weights
        )
    }

    return graphs, edge_count


# Fold-specific SVD8 baseline and graph features
def prepare_fold(fit, hold):

    anchor_fit, anchor_hold = (
        clinical_offset(fit, hold)
    )

    pathway_scaler = StandardScaler().fit(
        X[fit]
    )

    a = np.clip(
        pathway_scaler.transform(X[fit]),
        -7,
        7
    ).astype(np.float32)

    b = np.clip(
        pathway_scaler.transform(X[hold]),
        -7,
        7
    ).astype(np.float32)

    svd = TruncatedSVD(
        n_components=8,
        n_iter=4,
        random_state=SEED
    )

    za = svd.fit_transform(a)
    zb = svd.transform(b)

    latent = StandardScaler().fit(za)

    za = np.clip(
        latent.transform(za),
        -7,
        7
    )

    zb = np.clip(
        latent.transform(zb),
        -7,
        7
    )

    beta = cox_fit(
        za,
        time[fit],
        event[fit],
        ridge=0.10,
        offset=anchor_fit
    )

    risk_fit = anchor_fit + za @ beta
    risk_hold = anchor_hold + zb @ beta

    # No survival labels are used for pathway node selection.
    variability = np.var(
        X[fit],
        axis=0
    )

    selected = np.sort(
        np.argsort(
            -variability,
            kind="stable"
        )[:N_PATHWAYS]
    )

    graphs, edge_count = build_graphs(
        selected
    )

    return {
        "xtrain": torch.as_tensor(a[:, selected]),
        "xhold": torch.as_tensor(b[:, selected]),

        "htrain": torch.as_tensor(
            hist[fit],
            dtype=torch.long
        ),

        "hhold": torch.as_tensor(
            hist[hold],
            dtype=torch.long
        ),

        "fit_risk": risk_fit,
        "held_risk": risk_hold,
        "anchor_held": anchor_hold,

        "graphs": graphs,
        "edges": edge_count,

        "fit_time": time[fit],
        "fit_event": event[fit],
        "fit_site": site[fit],

        "selected": selected
    }


# Clinical-anchored pathway graph attention residual encoder
class PathwayResidualGAT(nn.Module):

    def __init__(self, n):

        super().__init__()

        self.identity = nn.Embedding(
            n,
            3
        )

        self.in_proj = nn.Linear(
            4,
            12
        )

        self.q_proj = nn.Linear(
            12,
            12,
            bias=False
        )

        self.a_src = nn.Parameter(
            torch.empty(2, 6)
        )

        self.a_dst = nn.Parameter(
            torch.empty(2, 6)
        )

        self.edge_scale = nn.Parameter(
            torch.zeros(2)
        )

        self.mix = nn.Linear(
            12,
            12
        )

        self.norm = nn.LayerNorm(
            12
        )

        self.hist_query = nn.Embedding(
            2,
            12
        )

        self.pool_gate = nn.Linear(
            12,
            1
        )

        self.head = nn.Sequential(
            nn.Linear(24, 12),
            nn.Tanh(),
            nn.Linear(12, 1)
        )

        nn.init.normal_(
            self.identity.weight,
            std=0.04
        )

        nn.init.xavier_uniform_(
            self.a_src
        )

        nn.init.xavier_uniform_(
            self.a_dst
        )

        # Start with zero molecular correction.
        nn.init.zeros_(
            self.head[-1].weight
        )

        nn.init.zeros_(
            self.head[-1].bias
        )

    def forward(self, x, h, graph):

        src, dst, edge_weight = graph

        batch, nodes = x.shape

        identity = (
            self.identity.weight
            .unsqueeze(0)
            .expand(batch, -1, -1)
        )

        z = F.gelu(
            self.in_proj(
                torch.cat(
                    (
                        x.unsqueeze(-1),
                        identity
                    ),
                    dim=-1
                )
            )
        )

        q = self.q_proj(z).reshape(
            batch,
            nodes,
            2,
            6
        )

        logits = F.leaky_relu(
            (
                q[:, src] * self.a_src
            ).sum(-1)
            + (
                q[:, dst] * self.a_dst
            ).sum(-1)
            + edge_weight.view(1, -1, 1)
            * self.edge_scale.view(1, 1, 2),
            negative_slope=0.2
        )

        index = dst.view(
            1,
            -1,
            1
        ).expand(
            batch,
            -1,
            2
        )

        maximum = torch.full(
            (batch, nodes, 2),
            -torch.inf,
            dtype=logits.dtype
        )

        maximum = maximum.scatter_reduce(
            1,
            index,
            logits,
            reduce="amax",
            include_self=True
        )

        exponent = torch.exp(
            logits - maximum.gather(
                1,
                index
            )
        )

        denominator = torch.zeros_like(
            maximum
        ).scatter_add(
            1,
            index,
            exponent
        )

        attention = F.dropout(
            exponent
            / denominator.gather(
                1,
                index
            ).clamp_min(1e-9),
            p=0.08,
            training=self.training
        )

        messages = (
            q[:, src]
            * attention.unsqueeze(-1)
        )

        message_index = dst.view(
            1,
            -1,
            1,
            1
        ).expand(
            batch,
            -1,
            2,
            6
        )

        aggregated = torch.zeros_like(
            q
        ).scatter_add(
            1,
            message_index,
            messages
        ).reshape(
            batch,
            nodes,
            12
        )

        z = self.norm(
            z + F.gelu(
                self.mix(aggregated)
            )
        )

        pool_logit = (
            self.pool_gate(z).squeeze(-1)
        )

        pool_logit = (
            pool_logit
            + (
                z
                * self.hist_query(h)[:, None, :]
            ).sum(-1) / np.sqrt(12)
        )

        pooled = torch.cat(
            (
                (
                    z
                    * torch.softmax(
                        pool_logit,
                        dim=1
                    ).unsqueeze(-1)
                ).sum(1),
                z.mean(1)
            ),
            dim=1
        )

        return (
            0.8
            * torch.tanh(
                self.head(pooled).squeeze(-1)
            )
        )


# Full-risk-set Cox partial likelihood
def torch_cox(risk, t, e):

    order = torch.argsort(
        t,
        descending=True,
        stable=True
    )

    t = t[order]
    e = e[order]
    risk = risk[order]

    ends = torch.searchsorted(
        -t.contiguous(),
        -t.contiguous(),
        right=True
    ) - 1

    return (
        e
        * (
            torch.logcumsumexp(
                risk,
                dim=0
            )[ends]
            - risk
        )
    ).sum() / e.sum().clamp_min(1.0)


# Train residual branches using only fitting-fold patients
def train_gat(
    prepared,
    mode,
    epochs,
    seed,
    collect=EPOCH_CHECKPOINTS
):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = PathwayResidualGAT(
        N_PATHWAYS
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=0.025
    )

    xtrain = prepared["xtrain"]
    xhold = prepared["xhold"]

    htrain = prepared["htrain"]
    hhold = prepared["hhold"]

    graph = prepared["graphs"][mode]

    baseline = torch.as_tensor(
        prepared["fit_risk"],
        dtype=torch.float32
    )

    t = torch.as_tensor(
        prepared["fit_time"],
        dtype=torch.float32
    )

    e = torch.as_tensor(
        prepared["fit_event"],
        dtype=torch.float32
    )

    _, inverse = np.unique(
        prepared["fit_site"],
        return_inverse=True
    )

    group_index = torch.as_tensor(
        inverse,
        dtype=torch.long
    )

    group_size = torch.bincount(
        group_index
    ).float().clamp_min(1)

    snapshots = {}

    for epoch in range(
        1,
        epochs + 1
    ):

        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        residual = model(
            xtrain,
            htrain,
            graph
        )

        site_average = (
            torch.zeros(
                len(group_size)
            ).index_add(
                0,
                group_index,
                residual
            )
            / group_size
        )

        loss = (
            torch_cox(
                baseline + residual,
                t,
                e
            )
            + 0.02
            * residual.square().mean()
            + 0.01
            * site_average.square().mean()
        )

        if not torch.isfinite(loss):
            raise RuntimeError(
                "Non-finite neural survival loss."
            )

        loss.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(),
            2.0
        )

        optimizer.step()

        if epoch in collect:

            model.eval()

            with torch.no_grad():

                predictions = [
                    model(
                        x,
                        h,
                        graph
                    )
                    for x, h in zip(
                        xhold.split(64),
                        hhold.split(64)
                    )
                ]

            snapshots[epoch] = (
                torch.cat(predictions)
                .cpu()
                .numpy()
                .astype(float)
            )

    return snapshots


# Inner site-held-out CV for model selection
def choose_inner(fit):

    strata = (
        2 * event[fit].astype(int)
        + hist[fit]
    )

    options = []

    for seed in (
        42, 59, 73, 89,
        101, 127, 151, 173
    ):

        try:

            splits = list(
                StratifiedGroupKFold(
                    n_splits=INNER_FOLDS,
                    shuffle=True,
                    random_state=seed
                ).split(
                    np.arange(len(fit)),
                    strata,
                    site[fit]
                )
            )

        except ValueError:
            continue

        valid_split = True
        imbalance = 0.0

        for a, b in splits:

            if (
                event[fit[b]].sum() < 15
                or event[fit[a]].sum() < 30
                or set(site[fit[a]]) & set(site[fit[b]])
                or len(np.unique(event[fit[b]])) != 2
            ):

                valid_split = False
                break

            imbalance += (
                abs(
                    len(b) / len(fit)
                    - 1 / INNER_FOLDS
                )
                + abs(
                    event[fit[b]].mean()
                    - event[fit].mean()
                )
            )

        if valid_split:

            options.append((
                imbalance,
                seed,
                [
                    (fit[a], fit[b])
                    for a, b in splits
                ]
            ))

    if not options:
        raise RuntimeError(
            "No valid inner site-held-out partition."
        )

    return min(
        options,
        key=lambda item: (
            item[0],
            item[1]
        )
    )[2]


# Choose the best training-only strategy
def choose_candidate(inner_scores):

    ranking = sorted(
        [
            (
                float(np.mean(values)),
                family,
                epoch,
                gamma
            )
            for (
                family,
                epoch,
                gamma
            ), values in inner_scores.items()
        ],
        key=lambda row: (
            -row[0],
            row[1] not in ("clinical", "svd8"),
            row[1],
            row[2],
            row[3]
        )
    )

    best = ranking[0]

    safe = max(
        (
            row
            for row in ranking
            if row[1] in ("clinical", "svd8")
        ),
        key=lambda row: row[0]
    )

    if (
        best[1] not in ("clinical", "svd8")
        and best[0] < safe[0] + MIN_GRAPH_GAIN
    ):
        best = safe

    return {
        "family": best[1],
        "epoch": int(best[2]),
        "gamma": float(best[3]),
        "inner_C": float(best[0]),
        "inner_base_C": float(safe[0])
    }


# Fingerprint the experiment
source_paths = [
    RAW / "rna_tpm.npy",
    S19 / "reactome_gene_pathway_weights.npz",
    S21 / "stage21_training_clinical_endpoints.csv",
    S24 / "stage24_training_site_oof.csv"
]

source_paths += [
    S21 / f"pfi_outer_fold{i}.joblib"
    for i in range(1, 5)
]

source_paths += [
    S24 / f"outer_fold{i}_verified.joblib"
    for i in range(1, 5)
]

signature = hashlib.sha256(
    json.dumps(
        {
            "version": VERSION,
            "patients": clinical.case_id.tolist(),
            "PFI_time": np.nan_to_num(
                time,
                nan=-1
            ).tolist(),
            "PFI_event": np.nan_to_num(
                event,
                nan=-1
            ).tolist(),
            "seed": SEED,
            "nodes": N_PATHWAYS,
            "neighbors": NEIGHBORS,
            "epochs": EPOCH_CHECKPOINTS,
            "gammas": GAMMAS,
            "files": {
                str(path): (
                    path.stat().st_size,
                    path.stat().st_mtime_ns
                )
                for path in source_paths
            }
        },
        sort_keys=True
    ).encode()
).hexdigest()


# Run matched outer site-held-out evaluation
rows = []
inner_rows = []

oof = {
    name: np.full(
        len(clinical),
        np.nan
    )
    for name in (
        "clinical",
        "svd8",
        "stage24",
        "selected",
        "graphfree",
        "reactome_overlap",
        "degree_null"
    )
}

for fold in range(1, 5):

    previous21 = joblib.load(
        S21 / f"pfi_outer_fold{fold}.joblib"
    )

    previous24 = joblib.load(
        S24 / f"outer_fold{fold}_verified.joblib"
    )

    hold = np.asarray(
        previous21["predictions"]["index"],
        dtype=int
    )

    fit = np.setdiff1d(
        eligible,
        hold
    )

    if (
        not np.array_equal(
            hold,
            np.asarray(
                previous24["held"],
                dtype=int
            )
        )
        or set(site[fit]) & set(site[hold])
        or not np.isin(hold, eligible).all()
    ):
        raise RuntimeError(
            "Stage 21/24 matched outer folds do not align."
        )

    clinical_reference = np.asarray(
        previous21["predictions"]["clinical_risk"],
        dtype=float
    )

    stage24_reference = np.asarray(
        previous24["prediction"],
        dtype=float
    )

    checkpoint = (
        OUT / f"fold{fold}_pathway_graph.joblib"
    )

    if checkpoint.exists():

        saved = joblib.load(checkpoint)

        if (
            saved.get("signature") != signature
            or not np.array_equal(
                saved["held"],
                hold
            )
        ):
            raise RuntimeError(
                "Incompatible saved Stage 25 fold."
            )

        result = saved["result"]
        current_inner = saved["inner"]
        fold_predictions = saved["predictions"]

        print(
            f"Fold {fold}: verified checkpoint reused",
            flush=True
        )

    else:

        prepared = prepare_fold(
            fit,
            hold
        )

        anchor_error = float(
            np.max(
                np.abs(
                    prepared["anchor_held"]
                    - clinical_reference
                )
            )
        )

        if anchor_error > 1e-3:
            raise RuntimeError(
                "Stage 21 clinical baseline mismatch: "
                f"fold={fold}, error={anchor_error}"
            )

        inner_folds = choose_inner(fit)

        performance = {
            ("clinical", 0, 0.0): [],
            ("svd8", 0, 0.0): []
        }

        for mode in (
            "graphfree",
            "reactome_overlap"
        ):

            for epoch in EPOCH_CHECKPOINTS:

                for gamma in GAMMAS:

                    performance[
                        (mode, epoch, gamma)
                    ] = []

        current_inner = []

        print(
            f"\nOUTER {fold}: "
            f"train={len(fit)}, "
            f"held={len(hold)}, "
            f"events={int(event[hold].sum())}, "
            f"graph edges={prepared['edges']}",
            flush=True
        )

        for inner_number, (
            inner_fit,
            inner_hold
        ) in enumerate(
            inner_folds,
            1
        ):

            inner_prepared = prepare_fold(
                inner_fit,
                inner_hold
            )

            candidate_scores = {
                ("clinical", 0, 0.0): (
                    inner_prepared["anchor_held"]
                ),
                ("svd8", 0, 0.0): (
                    inner_prepared["held_risk"]
                )
            }

            for mode in MODES:

                snapshots = train_gat(
                    inner_prepared,
                    mode,
                    max(EPOCH_CHECKPOINTS),
                    seed=(
                        SEED
                        + 100 * fold
                        + inner_number
                    )
                )

                # Randomized topology is an ablation,
                # not a model-selection candidate.
                if mode != "degree_null":

                    for epoch, residual in snapshots.items():

                        for gamma in GAMMAS:

                            candidate_scores[
                                (mode, epoch, gamma)
                            ] = (
                                inner_prepared["held_risk"]
                                + gamma * residual
                            )

            for key, risk in candidate_scores.items():

                score = c_index(
                    time[inner_hold],
                    event[inner_hold],
                    risk
                )

                if not np.isfinite(score):
                    raise RuntimeError(
                        "Invalid inner C-index."
                    )

                performance[key].append(
                    score
                )

                current_inner.append({
                    "outer_fold": fold,
                    "inner_fold": inner_number,
                    "family": key[0],
                    "epoch": key[1],
                    "gamma": key[2],
                    "C_index": score
                })

        winner = choose_candidate(
            performance
        )

        # Identical epoch and gamma across graph controls.
        probe_epoch = (
            winner["epoch"]
            if winner["epoch"]
            else 16
        )

        probe_gamma = (
            winner["gamma"]
            if winner["gamma"]
            else 0.25
        )

        fold_predictions = {
            "clinical": clinical_reference,
            "svd8": prepared["held_risk"],
            "stage24": stage24_reference
        }

        for mode in MODES:

            snapshots = train_gat(
                prepared,
                mode,
                probe_epoch,
                seed=SEED + 100 * fold + 9,
                collect=(probe_epoch,)
            )

            fold_predictions[mode] = (
                prepared["held_risk"]
                + probe_gamma
                * snapshots[probe_epoch]
            )

        if winner["family"] == "clinical":

            fold_predictions["selected"] = (
                clinical_reference
            )

        elif winner["family"] == "svd8":

            fold_predictions["selected"] = (
                prepared["held_risk"]
            )

        else:

            fold_predictions["selected"] = (
                fold_predictions[
                    winner["family"]
                ]
            )

        result = {
            "fold": fold,
            "n_held": len(hold),
            "events": int(event[hold].sum()),
            "pathway_nodes": N_PATHWAYS,
            "overlap_edges": prepared["edges"],
            "selection": winner["family"],
            "epoch": probe_epoch,
            "gamma": probe_gamma,
            "inner_selected_C": winner["inner_C"],
            "anchor_max_error": anchor_error
        }

        for name, risk in fold_predictions.items():

            result[f"{name}_C"] = c_index(
                time[hold],
                event[hold],
                risk
            )

        result["selected_minus_stage24"] = (
            result["selected_C"]
            - result["stage24_C"]
        )

        result["reactome_minus_null"] = (
            result["reactome_overlap_C"]
            - result["degree_null_C"]
        )

        temporary = checkpoint.with_name(
            checkpoint.name + ".tmp"
        )

        joblib.dump(
            {
                "signature": signature,
                "held": hold,
                "result": result,
                "inner": current_inner,
                "predictions": fold_predictions
            },
            temporary,
            compress=3
        )

        os.replace(
            temporary,
            checkpoint
        )

        print(
            f"  chosen={winner['family']}, "
            f"epoch={probe_epoch}, "
            f"gamma={probe_gamma}; "
            f"selected C={result['selected_C']:.4f}, "
            f"Stage24 C={result['stage24_C']:.4f}",
            flush=True
        )

    for name, risk in fold_predictions.items():

        oof[name][hold] = risk

    rows.append(result)
    inner_rows.extend(current_inner)


# Save predictions and results
if any(
    not np.isfinite(
        oof[name][eligible]
    ).all()
    for name in oof
):
    raise RuntimeError(
        "Incomplete Stage 25 outer predictions."
    )

result_table = (
    pd.DataFrame(rows)
    .sort_values("fold")
)

result_table.to_csv(
    OUT / "stage25_matched_outer_site_results.csv",
    index=False
)

pd.DataFrame(
    inner_rows
).to_csv(
    OUT / "stage25_inner_candidate_scores.csv",
    index=False
)

pd.DataFrame({
    "case_id": clinical.case_id.iloc[eligible],
    "PFI_time": time[eligible],
    "PFI_event": event[eligible],
    "site": site[eligible],
    **{
        name: oof[name][eligible]
        for name in oof
    }
}).to_csv(
    OUT / "stage25_training_only_oof.csv",
    index=False
)


# Exploratory patient-site cluster bootstrap
rng_boot = np.random.default_rng(
    SEED + 2525
)

bootstrap_stage24 = []
bootstrap_graph_null = []

for _ in range(400):

    stage_deltas = []
    graph_deltas = []

    for fold in range(1, 5):

        previous = joblib.load(
            S21 / f"pfi_outer_fold{fold}.joblib"
        )

        held = np.asarray(
            previous["predictions"]["index"],
            dtype=int
        )

        sites_in_fold = np.unique(
            site[held]
        )

        sampled_sites = rng_boot.choice(
            sites_in_fold,
            size=len(sites_in_fold),
            replace=True
        )

        indices = np.concatenate([
            held[
                site[held] == category
            ]
            for category in sampled_sites
        ])

        if (
            len(indices) < 2
            or len(np.unique(event[indices])) != 2
        ):
            break

        a = c_index(
            time[indices],
            event[indices],
            oof["selected"][indices]
        )

        b = c_index(
            time[indices],
            event[indices],
            oof["stage24"][indices]
        )

        c = c_index(
            time[indices],
            event[indices],
            oof["reactome_overlap"][indices]
        )

        d = c_index(
            time[indices],
            event[indices],
            oof["degree_null"][indices]
        )

        if not all(
            np.isfinite(value)
            for value in (a, b, c, d)
        ):
            break

        stage_deltas.append(a - b)
        graph_deltas.append(c - d)

    if len(stage_deltas) == 4:

        bootstrap_stage24.append(
            np.mean(stage_deltas)
        )

        bootstrap_graph_null.append(
            np.mean(graph_deltas)
        )

if len(bootstrap_stage24) < 100:
    raise RuntimeError(
        "Insufficient valid site-bootstrap samples."
    )


# Research summary
summary = {
    "version": VERSION,

    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "clinical_mean_C": float(
        result_table.clinical_C.mean()
    ),

    "svd8_mean_C": float(
        result_table.svd8_C.mean()
    ),

    "stage24_mean_C": float(
        result_table.stage24_C.mean()
    ),

    "stage25_selected_mean_C": float(
        result_table.selected_C.mean()
    ),

    "graphfree_probe_mean_C": float(
        result_table.graphfree_C.mean()
    ),

    "reactome_graph_probe_mean_C": float(
        result_table.reactome_overlap_C.mean()
    ),

    "degree_null_probe_mean_C": float(
        result_table.degree_null_C.mean()
    ),

    "selected_minus_stage24": float(
        result_table.selected_minus_stage24.mean()
    ),

    "biological_graph_minus_null": float(
        result_table.reactome_minus_null.mean()
    ),

    "outer_choices": (
        result_table.selection.value_counts().to_dict()
    ),

    "exploratory_selected_minus_stage24_95pct_CI": (
        np.percentile(
            bootstrap_stage24,
            [2.5, 97.5]
        ).tolist()
    ),

    "exploratory_biological_minus_null_95pct_CI": (
        np.percentile(
            bootstrap_graph_null,
            [2.5, 97.5]
        ).tolist()
    ),

    "validation_test_evaluated": False,

    "limitations": [
        "Pathway-overlap edges are not direct PPIs.",
        "Stage24 remains the incumbent comparison.",
        "The same training cohort was used for earlier research development.",
        "Inner CV selects Stage25 configuration; held-out outer results assess it.",
        "Site-disjoint internal CV is not independent external validation.",
        "Bootstrap intervals are exploratory after repeated model development."
    ]
}

(
    OUT / "stage25_summary.json"
).write_text(
    json.dumps(
        summary,
        indent=2
    ),
    encoding="utf-8"
)

print("\n" + "=" * 78)
print(
    "STAGE 25 FINAL - REACTOME PATHWAY GAT VS STAGE24"
)
print("=" * 78)

columns = [
    "fold",
    "selection",
    "epoch",
    "gamma",
    "clinical_C",
    "svd8_C",
    "stage24_C",
    "selected_C",
    "graphfree_C",
    "reactome_overlap_C",
    "degree_null_C",
    "selected_minus_stage24",
    "reactome_minus_null"
]

print(
    result_table[columns].to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)

for key in (
    "clinical_mean_C",
    "svd8_mean_C",
    "stage24_mean_C",
    "stage25_selected_mean_C",
    "biological_graph_minus_null"
):

    print(
        f"{key}: {summary[key]:.4f}"
    )

print(
    "Outer-fold choices:",
    summary["outer_choices"]
)

print(
    "Exploratory Stage25 minus Stage24 CI:",
    summary[
        "exploratory_selected_minus_stage24_95pct_CI"
    ]
)

print(
    "Exploratory biological graph minus null CI:",
    summary[
        "exploratory_biological_minus_null_95pct_CI"
    ]
)

print("Saved results:", OUT)

print(
    "Validation/test molecular features and outcomes evaluated: NO"
)

print("STAGE 25 COMPLETED")

STAGE 25 - CLINICAL-ANCHORED PATHWAY GRAPH RESIDUAL SURVIVAL
PFI eligible=657, events=225, pathways=1613
Matched Stage21/24 site-held-out folds; validation=143, test=143 untouched

OUTER 1: train=491, held=166, events=56, graph edges=607
  chosen=clinical, epoch=16, gamma=0.25; selected C=0.6489, Stage24 C=0.6412

OUTER 2: train=494, held=163, events=55, graph edges=607
  chosen=svd8, epoch=16, gamma=0.25; selected C=0.6596, Stage24 C=0.6596

OUTER 3: train=493, held=164, events=57, graph edges=609
  chosen=svd8, epoch=16, gamma=0.25; selected C=0.7075, Stage24 C=0.7075

OUTER 4: train=493, held=164, events=57, graph edges=602
  chosen=svd8, epoch=16, gamma=0.25; selected C=0.5804, Stage24 C=0.5919

STAGE 25 FINAL - REACTOME PATHWAY GAT VS STAGE24
 fold selection  epoch  gamma  clinical_C  svd8_C  stage24_C  selected_C  graphfree_C  reactome_overlap_C  degree_null_C  selected_minus_stage24  reactome_minus_null
    1  clinical     16 0.2500      0.6489  0.6720     0.6412      0.6489    

In [25]:

import json
import hashlib
import os
import sys
from pathlib import Path
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd

from scipy import sparse
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


# Project configuration
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
RAW = ROOT / "data" / "processed" / "gene_level"
S19 = ROOT / "results" / "metrics" / "stage19"
S21 = ROOT / "results" / "metrics" / "stage21"
S25 = ROOT / "results" / "metrics" / "stage25"
OUT = ROOT / "results" / "metrics" / "stage26"
MODELS = ROOT / "models" / "checkpoints" / "stage26"

SEED = 42
THREADS = min(4, os.cpu_count() or 1)
MIN_EXTENSION_GAIN = 0.010
N_BOOT = 400

VERSION = "stage26_histology_conditioned_multiomic_residual_v1"

SPECS = {
    "Clinical_Cox": {
        "rna": False, "cnv": False,
        "hist": False, "alpha": 0
    },
    "RNA_SVD8": {
        "rna": True, "cnv": False,
        "hist": False, "alpha": 0.10
    },
    "RNA_SVD8_HistShrink": {
        "rna": True, "cnv": False,
        "hist": True, "alpha": 0.10
    },
    "RNA_SVD8_CNV4": {
        "rna": True, "cnv": True,
        "hist": False, "alpha": 0.10
    },
    "RNA_SVD8_CNV4_HistShrink": {
        "rna": True, "cnv": True,
        "hist": True, "alpha": 0.10
    }
}

expected = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != expected.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

for directory in (OUT, MODELS):
    directory.mkdir(parents=True, exist_ok=True)


# Verify locked training cohort
cases = pd.read_csv(
    RAW / "case_order.csv",
    dtype=str,
    keep_default_na=False
)

locked = pd.read_csv(
    ROOT / "data" / "splits" / "locked_four_omics_v1.csv",
    dtype=str,
    keep_default_na=False
)

clinical = pd.read_csv(
    S21 / "stage21_training_clinical_endpoints.csv",
    dtype={"case_id": str}
)

genes = pd.read_csv(
    RAW / "gene_reference.csv",
    dtype=str,
    keep_default_na=False
)

if (
    not cases.case_id.is_unique
    or not locked.case_id.is_unique
    or not clinical.case_id.is_unique
):
    raise RuntimeError("Duplicate patient identifiers.")

joined = cases[
    ["case_id", "histology", "stage_group"]
].merge(
    locked[
        ["case_id", "histology", "stage_group", "split"]
    ],
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_raw", "_locked")
)

if (
    not joined.histology_raw.equals(joined.histology_locked)
    or not joined.stage_group_raw.equals(
        joined.stage_group_locked
    )
):
    raise RuntimeError("Locked clinical labels changed.")

tr = np.flatnonzero(joined.split.eq("train"))
va = np.flatnonzero(joined.split.eq("validation"))
te = np.flatnonzero(joined.split.eq("test"))

if (
    len(tr), len(va), len(te), len(genes)
) != (667, 143, 143, 19944):
    raise RuntimeError("Unexpected cohort dimensions.")

if (
    len(clinical) != 667
    or not np.array_equal(
        clinical.case_id,
        joined.case_id.iloc[tr]
    )
):
    raise RuntimeError("Stage 21 patient alignment mismatch.")

eligible = np.flatnonzero(
    clinical.PFI_valid.astype(str)
    .str.lower()
    .eq("true")
)

times = pd.to_numeric(
    clinical["PFI.time"],
    errors="coerce"
).to_numpy(float)

events = pd.to_numeric(
    clinical.PFI,
    errors="coerce"
).to_numpy(float)

hist = clinical.histology_matrix.eq(
    "LUSC"
).to_numpy(int)

site = clinical.site.astype(str).to_numpy()

if (
    len(eligible),
    int(events[eligible].sum())
) != (657, 225):
    raise RuntimeError(
        "PFI eligibility or event count changed."
    )

if (
    not np.isfinite(times[eligible]).all()
    or not np.isin(
        events[eligible],
        [0, 1]
    ).all()
):
    raise RuntimeError("Invalid survival endpoints.")


# Reconstruct training-only Reactome RNA features
W = sparse.load_npz(
    S19 / "reactome_gene_pathway_weights.npz"
).tocsr().astype(np.float32)

if W.shape != (1613, 19944):
    raise RuntimeError("Reactome mapping changed.")

source_rna = np.load(
    RAW / "rna_tpm.npy",
    mmap_mode="r"
)

source_cnv = np.load(
    RAW / "cnv_absolute.npy",
    mmap_mode="r"
)

if (
    source_rna.shape != source_cnv.shape
    or source_rna.shape != (953, 19944)
):
    raise RuntimeError("Original molecular matrices changed.")

rna = np.asarray(
    source_rna[tr],
    dtype=np.float32
).copy()

cnv = np.asarray(
    source_cnv[tr],
    dtype=np.float32
).copy()

del source_rna, source_cnv

valid = np.isfinite(rna) & (rna >= 0)

if valid.mean() < 0.95:
    raise RuntimeError("Unexpected RNA missingness.")

rna = np.log1p(
    np.where(valid, rna, 0)
).astype(np.float32)

ranks = rankdata(
    rna,
    axis=1,
    method="average"
).astype(np.float32)

ranks = (
    ranks - (len(genes) + 1) / 2.0
) / len(genes)

XR = np.asarray(
    (W @ ranks.T).T,
    dtype=np.float32
)

del rna, ranks, valid


# Construct signed and absolute CNV pathway features
valid = np.isfinite(cnv) & (cnv >= 0)

cv = np.clip(
    np.log2(
        (
            np.where(valid, cnv, 2.0) + 0.5
        ) / 2.5
    ),
    -3,
    3
).astype(np.float32)

coverage = np.asarray(
    (
        W @ valid.astype(np.float32).T
    ).T,
    dtype=np.float32
)

signed = np.asarray(
    (W @ (cv * valid).T).T,
    dtype=np.float32
)

absolute = np.asarray(
    (W @ (np.abs(cv) * valid).T).T,
    dtype=np.float32
)

for z in (signed, absolute):

    np.divide(
        z,
        coverage,
        out=z,
        where=coverage >= 0.7
    )

    z[coverage < 0.7] = 0

XC = np.column_stack([
    signed,
    absolute
]).astype(np.float32)

del valid, cnv, cv, coverage, signed, absolute

if (
    not np.isfinite(XR).all()
    or not np.isfinite(XC).all()
):
    raise RuntimeError("Invalid molecular pathway scores.")

print("=" * 72)
print(
    "STAGE 26 - HISTOLOGY-CONDITIONED "
    "MULTI-OMIC RESIDUAL COX"
)
print("=" * 72)

print(
    f"Training=667, PFI eligible={len(eligible)}, "
    f"events={int(events[eligible].sum())}"
)

print(
    f"RNA pathways={XR.shape[1]}; "
    f"CNV signed/absolute={XC.shape[1]}"
)

print(
    "Validation/test molecular features "
    "and outcomes: NOT READ"
)


# Harrell's concordance index
def cindex(t, e, r):

    comparable = (
        (t[:, None] < t[None, :])
        & (e[:, None] == 1)
    )

    denominator = int(comparable.sum())

    if denominator < 10:
        return np.nan

    difference = r[:, None] - r[None, :]

    concordant = (
        comparable & (difference > 1e-10)
    ).sum()

    tied = (
        comparable
        & (np.abs(difference) <= 1e-10)
    ).sum()

    return float(
        (concordant + 0.5 * tied) / denominator
    )


# Penalized Cox regression with optional clinical offset
def coxfit(z, t, e, penalty, offset=None):

    z = np.asarray(z, dtype=np.float64)
    t = np.asarray(t, dtype=np.float64)
    e = np.asarray(e, dtype=np.int64)

    off = (
        np.zeros(len(t))
        if offset is None
        else np.asarray(offset, dtype=np.float64)
    )

    order = np.argsort(
        -t,
        kind="stable"
    )

    z = z[order]
    t = t[order]
    e = e[order]
    off = off[order]

    starts = np.r_[
        0,
        np.flatnonzero(
            t[1:] != t[:-1]
        ) + 1
    ]

    ends = np.r_[
        starts[1:] - 1,
        len(t) - 1
    ]

    counts = np.add.reduceat(
        e,
        starts
    )

    ends = ends[counts > 0]
    counts = counts[counts > 0].astype(float)

    penalty = np.broadcast_to(
        np.asarray(penalty, dtype=float),
        (z.shape[1],)
    )

    n_events = e.sum()

    def objective(beta):

        eta = z @ beta + off

        shift = eta.max()

        weights = np.exp(
            eta - shift
        )

        cumulative = np.cumsum(
            weights
        )

        cumulative_x = np.cumsum(
            weights[:, None] * z,
            axis=0
        )

        denominator = np.maximum(
            cumulative[ends],
            1e-300
        )

        loss = (
            np.sum(
                counts * (
                    shift + np.log(denominator)
                )
            )
            - eta[e == 1].sum()
        ) / n_events

        gradient = (
            np.sum(
                counts[:, None]
                * cumulative_x[ends]
                / denominator[:, None],
                axis=0
            )
            - z[e == 1].sum(axis=0)
        ) / n_events

        return (
            float(
                loss
                + 0.5 * np.dot(
                    penalty,
                    beta * beta
                )
            ),
            gradient + penalty * beta
        )

    result = minimize(
        objective,
        np.zeros(z.shape[1]),
        jac=True,
        method="L-BFGS-B",
        options={
            "maxiter": 250,
            "ftol": 1e-9,
            "gtol": 1e-6
        }
    )

    if (
        not np.isfinite(result.x).all()
        or not np.isfinite(result.fun)
    ):
        raise RuntimeError("Cox optimization failed.")

    return result.x


# Fold-specific clinical baseline preprocessing
def clinical_view(fit, held):

    age = pd.to_numeric(
        clinical.age_years,
        errors="coerce"
    ).to_numpy(float)

    male = pd.to_numeric(
        clinical.male,
        errors="coerce"
    ).to_numpy(float)

    stage = pd.to_numeric(
        clinical.stage_major,
        errors="coerce"
    ).to_numpy(float)

    available = age[fit][
        np.isfinite(age[fit])
    ]

    median = (
        float(np.median(available))
        if len(available)
        else 65.0
    )

    def block(ids):

        a = np.where(
            np.isfinite(age[ids]),
            age[ids],
            median
        )

        return np.column_stack([
            (a - median) / 10.0,
            (~np.isfinite(age[ids])).astype(float),
            np.where(
                np.isfinite(male[ids]),
                male[ids],
                0.5
            ),
            (~np.isfinite(male[ids])).astype(float),
            hist[ids],
            (stage[ids] == 2).astype(float),
            (stage[ids] == 3).astype(float),
            (stage[ids] == 4).astype(float)
        ])

    scaler = StandardScaler().fit(
        block(fit)
    )

    return (
        scaler.transform(block(fit)),
        scaler.transform(block(held)),
        {
            "age_median": median,
            "scaler": scaler
        }
    )


# Fit clinical offset and molecular representations
def build_fold(fit, held):

    a, b, clinical_state = clinical_view(
        fit,
        held
    )

    clinical_beta = coxfit(
        a,
        times[fit],
        events[fit],
        0.003 * 0.15
    )

    anchor_fit = a @ clinical_beta
    anchor_hold = b @ clinical_beta

    state = {
        "clinical": {
            **clinical_state,
            "beta": clinical_beta
        }
    }

    latent = {}

    for key, matrix, n_components in (
        ("rna", XR, 8),
        ("cnv", XC, 4)
    ):

        scaler = StandardScaler().fit(
            matrix[fit]
        )

        zfit = np.clip(
            scaler.transform(matrix[fit]),
            -7,
            7
        ).astype(np.float32)

        zhold = np.clip(
            scaler.transform(matrix[held]),
            -7,
            7
        ).astype(np.float32)

        svd = TruncatedSVD(
            n_components=n_components,
            n_iter=4,
            random_state=SEED
        )

        zfit = svd.fit_transform(
            zfit
        )

        zhold = svd.transform(
            zhold
        )

        latent_scaler = StandardScaler().fit(
            zfit
        )

        zfit = np.clip(
            latent_scaler.transform(zfit),
            -7,
            7
        )

        zhold = np.clip(
            latent_scaler.transform(zhold),
            -7,
            7
        )

        latent[key] = (
            zfit,
            zhold
        )

        state[key] = {
            "scaler": scaler,
            "svd": svd,
            "latent_scaler": latent_scaler
        }

    return (
        anchor_fit,
        anchor_hold,
        latent,
        state
    )


# Clinical-anchored residual architectures
def predict_spec(
    spec,
    fit,
    held,
    anchor_fit,
    anchor_hold,
    latent
):

    if not spec["rna"]:

        return (
            anchor_hold.copy(),
            {"coef": np.zeros(0)}
        )

    rfit, rhold = latent["rna"]

    fit_parts = [rfit]
    hold_parts = [rhold]

    penalties = [
        np.full(
            8,
            spec["alpha"]
        )
    ]

    if spec["cnv"]:

        cfit, chold = latent["cnv"]

        fit_parts.append(cfit)
        hold_parts.append(chold)

        penalties.append(
            np.full(
                4,
                spec["alpha"] * 2
            )
        )

    if spec["hist"]:

        sign_fit = (
            2 * hist[fit] - 1
        )[:, None]

        sign_hold = (
            2 * hist[held] - 1
        )[:, None]

        fit_parts.append(
            0.5 * rfit * sign_fit
        )

        hold_parts.append(
            0.5 * rhold * sign_hold
        )

        penalties.append(
            np.full(
                8,
                spec["alpha"] * 4
            )
        )

    xfit = np.column_stack(
        fit_parts
    )

    xhold = np.column_stack(
        hold_parts
    )

    beta = coxfit(
        xfit,
        times[fit],
        events[fit],
        np.concatenate(penalties),
        offset=anchor_fit
    )

    return (
        anchor_hold + xhold @ beta,
        {"coef": beta}
    )


# Site-disjoint inner cross-validation
def inner_folds(fit):

    strata = (
        2 * events[fit].astype(int)
        + hist[fit]
    )

    choices = []

    for seed in (
        42, 59, 73, 89,
        101, 127, 151, 173
    ):

        trial = list(
            StratifiedGroupKFold(
                n_splits=3,
                shuffle=True,
                random_state=seed
            ).split(
                np.arange(len(fit)),
                strata,
                site[fit]
            )
        )

        valid = True
        imbalance = 0.0

        for a, b in trial:

            if (
                events[fit[b]].sum() < 15
                or events[fit[a]].sum() < 30
                or set(site[fit[a]]) & set(site[fit[b]])
                or len(np.unique(events[fit[b]])) < 2
            ):

                valid = False
                break

            imbalance += (
                abs(
                    len(b) / len(fit) - 1 / 3
                )
                + abs(
                    events[fit[b]].mean()
                    - events[fit].mean()
                )
            )

        if valid:

            choices.append((
                imbalance,
                seed,
                [
                    (fit[a], fit[b])
                    for a, b in trial
                ]
            ))

    if not choices:
        raise RuntimeError(
            "No valid inner site-held-out CV."
        )

    return min(
        choices,
        key=lambda x: (x[0], x[1])
    )[2]


# Reuse exact Stage 21 and Stage 25 outer folds
outer = []

for fold in range(1, 5):

    previous21 = joblib.load(
        S21 / f"pfi_outer_fold{fold}.joblib"
    )

    previous25 = joblib.load(
        S25 / f"fold{fold}_pathway_graph.joblib"
    )

    held = np.asarray(
        previous21["predictions"]["index"],
        dtype=int
    )

    fit = np.setdiff1d(
        eligible,
        held
    )

    if (
        not np.array_equal(
            held,
            np.asarray(previous25["held"])
        )
        or set(site[fit]) & set(site[held])
    ):
        raise RuntimeError(
            "Stage 21/25 outer-fold mismatch."
        )

    outer.append((
        fold,
        fit,
        held,
        previous21,
        previous25
    ))

all_held = np.concatenate([
    item[2] for item in outer
])

if not np.array_equal(
    np.sort(all_held),
    np.sort(eligible)
):
    raise RuntimeError(
        "Outer folds do not partition eligible patients."
    )


# Reproducibility fingerprint
source_paths = [
    RAW / "rna_tpm.npy",
    RAW / "cnv_absolute.npy",
    S19 / "reactome_gene_pathway_weights.npz",
    S21 / "stage21_training_clinical_endpoints.csv"
]

source_paths += [
    S21 / f"pfi_outer_fold{i}.joblib"
    for i in range(1, 5)
]

source_paths += [
    S25 / f"fold{i}_pathway_graph.joblib"
    for i in range(1, 5)
]

signature = hashlib.sha256(
    json.dumps(
        {
            "version": VERSION,
            "specifications": SPECS,
            "patient_ids": clinical.case_id.tolist(),
            "eligible": eligible.tolist(),
            "times": np.nan_to_num(
                times, nan=-1
            ).tolist(),
            "events": np.nan_to_num(
                events, nan=-1
            ).tolist(),
            "sources": {
                str(p): (
                    p.stat().st_size,
                    p.stat().st_mtime_ns
                )
                for p in source_paths
            },
            "minimum_extension_gain": MIN_EXTENSION_GAIN
        },
        sort_keys=True
    ).encode()
).hexdigest()


# Nested site-held-out evaluation
rows = []
inner_rows = []

oof = {
    name: np.full(
        len(clinical),
        np.nan
    )
    for name in (
        "clinical",
        "rna_svd8",
        "selected"
    )
}

with threadpool_limits(limits=THREADS):

    for fold, fit, held, historical21, historical25 in outer:

        checkpoint = (
            OUT / f"fold{fold}_verified.joblib"
        )

        if checkpoint.exists():

            saved = joblib.load(
                checkpoint
            )

            if (
                saved.get("signature") != signature
                or not np.array_equal(
                    saved["held"],
                    held
                )
            ):
                raise RuntimeError(
                    f"Incompatible checkpoint: {checkpoint}"
                )

            row = saved["row"]
            current_inner = saved["inner"]
            predictions = saved["predictions"]

            print(
                f"OUTER {fold}: checkpoint reused",
                flush=True
            )

        else:

            (
                anchor_fit,
                anchor_hold,
                latent,
                state
            ) = build_fold(
                fit,
                held
            )

            reference_clinical = np.asarray(
                historical21[
                    "predictions"
                ]["clinical_risk"],
                dtype=float
            )

            reference_svd8 = np.asarray(
                historical25[
                    "predictions"
                ]["svd8"],
                dtype=float
            )

            baseline_rna, _ = predict_spec(
                SPECS["RNA_SVD8"],
                fit,
                held,
                anchor_fit,
                anchor_hold,
                latent
            )

            clinical_error = float(
                np.max(
                    np.abs(
                        reference_clinical - anchor_hold
                    )
                )
            )

            svd8_error = float(
                np.max(
                    np.abs(
                        reference_svd8 - baseline_rna
                    )
                )
            )

            if (
                clinical_error > 1e-3
                or svd8_error > 1e-3
            ):
                raise RuntimeError(
                    f"Reference reproduction failed "
                    f"at fold {fold}: "
                    f"clinical={clinical_error}, "
                    f"SVD8={svd8_error}"
                )

            performance = {
                name: []
                for name in SPECS
            }

            current_inner = []

            print(
                f"\nOUTER {fold}: "
                f"train={len(fit)}, "
                f"held={len(held)}, "
                f"events={int(events[held].sum())}",
                flush=True
            )

            for j, (
                inner_fit,
                inner_held
            ) in enumerate(
                inner_folds(fit),
                1
            ):

                (
                    inner_anchor_fit,
                    inner_anchor_hold,
                    inner_latent,
                    _
                ) = build_fold(
                    inner_fit,
                    inner_held
                )

                for name, spec in SPECS.items():

                    prediction, _ = predict_spec(
                        spec,
                        inner_fit,
                        inner_held,
                        inner_anchor_fit,
                        inner_anchor_hold,
                        inner_latent
                    )

                    score = cindex(
                        times[inner_held],
                        events[inner_held],
                        prediction
                    )

                    if not np.isfinite(score):
                        raise RuntimeError(
                            "Invalid inner C-index."
                        )

                    performance[name].append(
                        score
                    )

                    current_inner.append({
                        "outer_fold": fold,
                        "inner_fold": j,
                        "model": name,
                        "C_index": score
                    })

            means = {
                name: float(np.mean(scores))
                for name, scores in performance.items()
            }

            simple = max(
                (
                    "Clinical_Cox",
                    "RNA_SVD8"
                ),
                key=lambda name: means[name]
            )

            extended = max(
                (
                    name
                    for name in SPECS
                    if name not in (
                        "Clinical_Cox",
                        "RNA_SVD8"
                    )
                ),
                key=lambda name: means[name]
            )

            gain = (
                means[extended] - means[simple]
            )

            winner = (
                extended
                if gain >= MIN_EXTENSION_GAIN
                else simple
            )

            chosen, _ = predict_spec(
                SPECS[winner],
                fit,
                held,
                anchor_fit,
                anchor_hold,
                latent
            )

            predictions = {
                "clinical": anchor_hold,
                "rna_svd8": baseline_rna,
                "selected": chosen
            }

            row = {
                "fold": fold,
                "held": len(held),
                "events": int(events[held].sum()),
                "winner": winner,
                "inner_best_gain_vs_simple": gain,
                "clinical_C": cindex(
                    times[held],
                    events[held],
                    anchor_hold
                ),
                "rna_svd8_C": cindex(
                    times[held],
                    events[held],
                    baseline_rna
                ),
                "selected_C": cindex(
                    times[held],
                    events[held],
                    chosen
                ),
                "LUAD_selected_C": cindex(
                    times[held][hist[held] == 0],
                    events[held][hist[held] == 0],
                    chosen[hist[held] == 0]
                ),
                "LUSC_selected_C": cindex(
                    times[held][hist[held] == 1],
                    events[held][hist[held] == 1],
                    chosen[hist[held] == 1]
                ),
                "clinical_ref_error": clinical_error,
                "svd8_ref_error": svd8_error
            }

            row["delta_C_vs_SVD8"] = (
                row["selected_C"]
                - row["rna_svd8_C"]
            )

            temporary = checkpoint.with_name(
                checkpoint.name + ".tmp"
            )

            joblib.dump(
                {
                    "signature": signature,
                    "held": held,
                    "row": row,
                    "inner": current_inner,
                    "predictions": predictions
                },
                temporary,
                compress=3
            )

            os.replace(
                temporary,
                checkpoint
            )

            print(
                f"  selected={winner} | "
                f"clinical={row['clinical_C']:.4f} | "
                f"SVD8={row['rna_svd8_C']:.4f} | "
                f"selected={row['selected_C']:.4f}",
                flush=True
            )

        rows.append(row)
        inner_rows.extend(current_inner)

        for name, risk in predictions.items():
            oof[name][held] = risk


# Save matched evaluation results
if any(
    not np.isfinite(
        values[eligible]
    ).all()
    for values in oof.values()
):
    raise RuntimeError(
        "Incomplete site-held-out predictions."
    )

table = pd.DataFrame(
    rows
).sort_values("fold")

table.to_csv(
    OUT / "stage26_fold_results.csv",
    index=False
)

inner_table = pd.DataFrame(
    inner_rows
)

inner_table.to_csv(
    OUT / "stage26_inner_selection.csv",
    index=False
)

pd.DataFrame({
    "case_id": clinical.case_id.iloc[eligible],
    "site": site[eligible],
    "PFI_time": times[eligible],
    "PFI_event": events[eligible],
    **{
        name: values[eligible]
        for name, values in oof.items()
    }
}).to_csv(
    OUT / "stage26_training_only_oof.csv",
    index=False
)


# Exploratory site-cluster bootstrap
rng = np.random.default_rng(
    SEED + 26
)

deltas = []

for _ in range(N_BOOT):

    sampled_gains = []

    for _, fit, held, _, _ in outer:

        sites_in_fold = np.unique(
            site[held]
        )

        sampled = rng.choice(
            sites_in_fold,
            size=len(sites_in_fold),
            replace=True
        )

        indices = np.concatenate([
            held[site[held] == category]
            for category in sampled
        ])

        selected_c = cindex(
            times[indices],
            events[indices],
            oof["selected"][indices]
        )

        baseline_c = cindex(
            times[indices],
            events[indices],
            oof["rna_svd8"][indices]
        )

        if not (
            np.isfinite(selected_c)
            and np.isfinite(baseline_c)
        ):
            break

        sampled_gains.append(
            selected_c - baseline_c
        )

    if len(sampled_gains) == 4:

        deltas.append(
            np.mean(sampled_gains)
        )

if len(deltas) < 100:
    raise RuntimeError(
        "Insufficient site-bootstrap replicates."
    )


# Select and fit a frozen full-training model
ranking = (
    inner_table
    .groupby("model")
    .C_index.mean()
    .sort_values(ascending=False)
)

simple = max(
    ("Clinical_Cox", "RNA_SVD8"),
    key=lambda name: ranking[name]
)

extended = max(
    (
        name
        for name in SPECS
        if name not in (
            "Clinical_Cox",
            "RNA_SVD8"
        )
    ),
    key=lambda name: ranking[name]
)

final_choice = (
    extended
    if (
        ranking[extended] - ranking[simple]
    ) >= MIN_EXTENSION_GAIN
    else simple
)

with threadpool_limits(limits=THREADS):

    (
        anchor,
        _,
        latent,
        preprocessing
    ) = build_fold(
        eligible,
        eligible
    )

    _, model_state = predict_spec(
        SPECS[final_choice],
        eligible,
        eligible,
        anchor,
        anchor,
        latent
    )

model_path = (
    MODELS / "stage26_training_only_final.joblib"
)

temporary = model_path.with_name(
    model_path.name + ".tmp"
)

joblib.dump(
    {
        "version": VERSION,
        "signature": signature,
        "model": final_choice,
        "spec": SPECS[final_choice],
        "preprocessing": preprocessing,
        "model_state": model_state,
        "training_case_ids": (
            clinical.case_id.iloc[eligible].tolist()
        ),
        "pathway_mapping": str(
            S19 / "reactome_gene_pathway_weights.npz"
        ),
        "gene_reference": str(
            RAW / "gene_reference.csv"
        ),
        "validated_externally": False
    },
    temporary,
    compress=3
)

os.replace(
    temporary,
    model_path
)


# Final reproducibility summary
summary = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "stage": VERSION,
    "clinical_mean_C": float(
        table.clinical_C.mean()
    ),
    "fixed_RNA_SVD8_mean_C": float(
        table.rna_svd8_C.mean()
    ),
    "nested_selected_mean_C": float(
        table.selected_C.mean()
    ),
    "nested_minus_fixed_SVD8": float(
        table["delta_C_vs_SVD8"].mean()
    ),
    "site_bootstrap_exploratory_95pct": (
        np.percentile(
            deltas,
            [2.5, 97.5]
        ).tolist()
    ),
    "outer_model_usage": (
        table.winner.value_counts().to_dict()
    ),
    "final_choice_from_inner_CV": final_choice,
    "validation_test_unused": True,
    "caveats": [
        "Stage25 SVD8 was investigated after prior exploratory analyses.",
        "Repeated development on the same cohort limits confirmatory inference.",
        "Site-held-out CV is not independent external validation.",
        "No time-dependent calibration or IPCW C-index is evaluated yet."
    ]
}

(
    OUT / "stage26_summary.json"
).write_text(
    json.dumps(
        summary,
        indent=2
    ),
    encoding="utf-8"
)


# Final reporting
print("\n" + "=" * 72)
print(
    "STAGE 26 FINAL - "
    "HISTOLOGY-CONDITIONED MULTI-OMIC SURVIVAL"
)
print("=" * 72)

print(
    table.to_string(
        index=False,
        float_format=lambda v: f"{v:.4f}"
    )
)

print(
    "Clinical mean C:",
    round(
        summary["clinical_mean_C"],
        4
    )
)

print(
    "Fixed RNA SVD8 mean C:",
    round(
        summary["fixed_RNA_SVD8_mean_C"],
        4
    )
)

print(
    "Nested selected mean C:",
    round(
        summary["nested_selected_mean_C"],
        4
    )
)

print(
    "Nested minus RNA SVD8:",
    round(
        summary["nested_minus_fixed_SVD8"],
        4
    )
)

print(
    "Exploratory site-cluster 95% interval:",
    summary["site_bootstrap_exploratory_95pct"]
)

print(
    "Outer-fold choices:",
    summary["outer_model_usage"]
)

print(
    "Final training-only model:",
    final_choice
)

print(
    "Saved:",
    OUT,
    model_path
)

print("Validation/test evaluated: NO")
print("STAGE 26 COMPLETED")

STAGE 26 - HISTOLOGY-CONDITIONED MULTI-OMIC RESIDUAL COX
Training=667, PFI eligible=657, events=225
RNA pathways=1613; CNV signed/absolute=3226
Validation/test molecular features and outcomes: NOT READ

OUTER 1: train=491, held=166, events=56
  selected=Clinical_Cox | clinical=0.6489 | SVD8=0.6720 | selected=0.6489

OUTER 2: train=494, held=163, events=55
  selected=RNA_SVD8 | clinical=0.6421 | SVD8=0.6596 | selected=0.6596

OUTER 3: train=493, held=164, events=57
  selected=RNA_SVD8 | clinical=0.6660 | SVD8=0.7075 | selected=0.7075

OUTER 4: train=493, held=164, events=57
  selected=RNA_SVD8 | clinical=0.5696 | SVD8=0.5804 | selected=0.5804

STAGE 26 FINAL - HISTOLOGY-CONDITIONED MULTI-OMIC SURVIVAL
 fold  held  events       winner  inner_best_gain_vs_simple  clinical_C  rna_svd8_C  selected_C  LUAD_selected_C  LUSC_selected_C  clinical_ref_error  svd8_ref_error  delta_C_vs_SVD8
    1   166      56 Clinical_Cox                    -0.0129      0.6489      0.6720      0.6489           0

In [26]:

import hashlib
import json
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from scipy import sparse
from scipy.stats import rankdata

# ============================================================
# STAGE 27 - Frozen RNA-SVD8 PFI validation
# ============================================================

ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

RAW = ROOT / "data" / "processed" / "gene_level"
S19 = ROOT / "results" / "metrics" / "stage19"
S21 = ROOT / "results" / "metrics" / "stage21"

OUT = ROOT / "results" / "metrics" / "stage27_frozen_validation"

MODEL_FILE = (
    ROOT / "models" / "checkpoints" / "stage26"
    / "stage26_training_only_final.joblib"
)

CDR_FILE = (
    ROOT / "data" / "raw" / "clinical_references"
    / "TCGA-CDR-SupplementalTableS1.xlsx"
)

STAGE_FILE = (
    ROOT / "data" / "interim"
    / "tcga_clinical_stage_audit.csv"
)

SPLIT_FILE = (
    ROOT / "data" / "splits"
    / "locked_four_omics_v1.csv"
)

GENE_FILE = RAW / "gene_reference.csv"
CASE_FILE = RAW / "case_order.csv"
RNA_FILE = RAW / "rna_tpm.npy"
W_FILE = S19 / "reactome_gene_pathway_weights.npz"

CLINICAL_FILE = (
    S21 / "stage21_training_clinical_endpoints.csv"
)

PROTOCOL_FILE = (
    OUT / "stage27_preregistered_freeze.json"
)

REPORT_FILE = (
    OUT / "stage27_validation_summary.json"
)

CDR_MD5 = "a4591b2dcee39591f59e5e25a6ce75fa"

HORIZONS = (365, 1095)
N_BOOT = 1000
SEED = 2701

VERSION = "stage27_frozen_rna_svd8_pfi_validation_v1"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )


# ============================================================
# 1. Reproducibility and evaluation helpers
# ============================================================

def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as handle:

        for block in iter(
            lambda: handle.read(4 * 1024 * 1024),
            b""
        ):
            digest.update(block)

    return digest.hexdigest()


def json_safe(value):

    if isinstance(value, dict):
        return {
            str(k): json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, (tuple, list)):
        return [
            json_safe(v)
            for v in value
        ]

    if isinstance(value, np.generic):
        return json_safe(value.item())

    if isinstance(value, float) and not np.isfinite(value):
        return None

    return value


def cindex(t, event, score):

    t = np.asarray(t, dtype=float)
    event = np.asarray(event, dtype=int)
    score = np.asarray(score, dtype=float)

    comparable = (
        (t[:, None] < t[None, :])
        & (event[:, None] == 1)
    )

    n_pairs = int(comparable.sum())

    if n_pairs < 10:
        return float("nan"), n_pairs

    difference = (
        score[:, None] - score[None, :]
    )

    concordant = np.count_nonzero(
        comparable & (difference > 1e-10)
    )

    tied = np.count_nonzero(
        comparable
        & (np.abs(difference) <= 1e-10)
    )

    return (
        float((concordant + 0.5 * tied) / n_pairs),
        n_pairs
    )


def major_stage(original):

    match = re.search(
        r"STAGE\s*(IV|III|II|I)(?:[ABC])?\b",
        str(original).upper()
    )

    if not match:
        return np.nan

    return {
        "I": 1,
        "II": 2,
        "III": 3,
        "IV": 4
    }[match.group(1)]


# ============================================================
# 2. Read only validation patient records from TCGA-CDR
# ============================================================

def extract_cdr_validation(path, allowed):

    needed = (
        "bcr_patient_barcode",
        "type",
        "PFI",
        "PFI.time",
        "Redaction",
        "age_at_initial_pathologic_diagnosis",
        "gender"
    )

    workbook = load_workbook(
        path,
        read_only=True,
        data_only=True
    )

    try:

        if "TCGA-CDR" not in workbook.sheetnames:
            raise RuntimeError(
                "Missing TCGA-CDR worksheet."
            )

        worksheet = workbook["TCGA-CDR"]
        values = worksheet.iter_rows(
            values_only=True
        )

        headings = next(values)

        positions = {
            str(value).strip(): i
            for i, value in enumerate(headings)
        }

        absent = sorted(
            set(needed) - set(positions)
        )

        if absent:
            raise RuntimeError(
                f"TCGA-CDR missing fields: {absent}"
            )

        records = []

        patient_col = positions[
            "bcr_patient_barcode"
        ]

        for row in values:

            patient_id = (
                str(row[patient_col])
                .strip()
                .upper()[:12]
            )

            if patient_id in allowed:

                record = {
                    field: row[positions[field]]
                    for field in needed
                }

                record["patient_id"] = patient_id

                records.append(record)

    finally:
        workbook.close()

    frame = pd.DataFrame(records)

    if frame.empty or not frame.patient_id.is_unique:
        raise RuntimeError(
            "Missing or duplicated validation "
            "TCGA-CDR patient identifiers."
        )

    return frame


# ============================================================
# 3. Apply the frozen Stage 26 preprocessing
# ============================================================

def clinical_matrix(frame, stored):

    age = pd.to_numeric(
        frame.age_years,
        errors="coerce"
    ).to_numpy(float)

    sex = pd.to_numeric(
        frame.male,
        errors="coerce"
    ).to_numpy(float)

    stage = pd.to_numeric(
        frame.stage_major,
        errors="coerce"
    ).to_numpy(float)

    hist = frame.histology.eq(
        "LUSC"
    ).to_numpy(float)

    median = float(
        stored["age_median"]
    )

    imputed_age = np.where(
        np.isfinite(age),
        age,
        median
    )

    features = np.column_stack([
        (imputed_age - median) / 10.0,
        (~np.isfinite(age)).astype(float),
        np.where(
            np.isfinite(sex),
            sex,
            0.5
        ),
        (~np.isfinite(sex)).astype(float),
        hist,
        (stage == 2).astype(float),
        (stage == 3).astype(float),
        (stage == 4).astype(float)
    ])

    return stored["scaler"].transform(
        features
    )


def pathway_scores(source, rows, W, n_genes):

    values = np.asarray(
        source[rows],
        dtype=np.float32
    ).copy()

    acceptable = (
        np.isfinite(values)
        & (values >= 0)
    )

    if acceptable.mean() < 0.95:
        raise RuntimeError(
            "RNA completeness below the "
            "prespecified threshold."
        )

    values = np.log1p(
        np.where(
            acceptable,
            values,
            0
        )
    ).astype(np.float32)

    ranks = rankdata(
        values,
        axis=1,
        method="average"
    ).astype(np.float32)

    ranks = (
        ranks - (n_genes + 1) / 2.0
    ) / n_genes

    result = np.asarray(
        (W @ ranks.T).T,
        dtype=np.float32
    )

    if not np.isfinite(result).all():
        raise RuntimeError(
            "Invalid Reactome pathway scores."
        )

    return result


def frozen_scores(frame, pathway, model):

    state = model["preprocessing"]

    clinical = clinical_matrix(
        frame,
        state["clinical"]
    )

    clinical_risk = (
        clinical
        @ np.asarray(
            state["clinical"]["beta"],
            dtype=float
        )
    )

    molecular = np.clip(
        state["rna"]["scaler"].transform(
            pathway
        ),
        -7,
        7
    )

    molecular = state[
        "rna"
    ]["svd"].transform(molecular)

    molecular = np.clip(
        state[
            "rna"
        ]["latent_scaler"].transform(
            molecular
        ),
        -7,
        7
    )

    coefficients = np.asarray(
        model["model_state"]["coef"],
        dtype=float
    )

    if (
        molecular.shape[1] != 8
        or len(coefficients) != 8
    ):
        raise RuntimeError(
            "Frozen RNA-SVD8 latent dimension mismatch."
        )

    combined_risk = (
        clinical_risk
        + molecular @ coefficients
    )

    if not (
        np.isfinite(clinical_risk).all()
        and np.isfinite(combined_risk).all()
    ):
        raise RuntimeError(
            "Non-finite frozen survival risk."
        )

    return (
        clinical_risk.astype(float),
        combined_risk.astype(float)
    )


# ============================================================
# 4. Training-derived Cox baseline survival functions
# ============================================================

def baseline_hazard(t, e, risk):

    t = np.asarray(t, dtype=float)
    e = np.asarray(e, dtype=int)
    risk = np.asarray(risk, dtype=float)

    shift = float(risk.max())

    weight = np.exp(
        np.clip(
            risk - shift,
            -40,
            40
        )
    )

    event_times = np.unique(
        t[e == 1]
    )

    increments = []

    for moment in event_times:

        count = int(
            np.sum(
                (t == moment)
                & (e == 1)
            )
        )

        denominator = weight[
            t >= moment
        ].sum()

        increments.append(
            count / denominator
        )

    return (
        event_times,
        np.cumsum(increments),
        shift
    )


def survival_probability(
    curve,
    risk,
    horizon
):

    times, cumulative, shift = curve

    index = (
        np.searchsorted(
            times,
            horizon,
            side="right"
        ) - 1
    )

    hazard = (
        float(cumulative[index])
        if index >= 0
        else 0.0
    )

    multiplier = np.exp(
        np.clip(
            np.asarray(risk) - shift,
            -40,
            40
        )
    )

    return np.exp(
        -hazard * multiplier
    )


# ============================================================
# 5. Kaplan-Meier and IPCW Brier score
# ============================================================

def km_survival_at(t, e, horizon):

    t = np.asarray(t, dtype=float)
    e = np.asarray(e, dtype=int)

    survival = 1.0

    event_times = np.unique(
        t[
            (e == 1)
            & (t <= horizon)
        ]
    )

    for moment in event_times:

        at_risk = (
            t >= moment
        ).sum()

        failures = (
            (t == moment)
            & (e == 1)
        ).sum()

        survival *= (
            1.0 - failures / at_risk
        )

    return float(survival)


def censoring_distribution(t, e):

    t = np.asarray(t, dtype=float)
    e = np.asarray(e, dtype=int)

    distinct = np.unique(t)

    before = np.empty(
        len(distinct),
        dtype=float
    )

    after = np.empty(
        len(distinct),
        dtype=float
    )

    current = 1.0

    for j, moment in enumerate(distinct):

        before[j] = current

        at_risk = (
            t >= moment
        ).sum()

        n_censored = (
            (t == moment)
            & (e == 0)
        ).sum()

        current *= (
            1.0 - n_censored / at_risk
        )

        after[j] = current

    return distinct, before, after


def ipcw_brier(
    t,
    e,
    survival,
    horizon,
    censor_km
):

    distinct, before, after = censor_km

    index = (
        np.searchsorted(
            distinct,
            horizon,
            side="right"
        ) - 1
    )

    g_h = (
        float(after[index])
        if index >= 0
        else 1.0
    )

    # Suppress unstable late-horizon estimates
    if g_h < 0.10:
        return np.nan, g_h

    event_before = (
        (t <= horizon)
        & (e == 1)
    )

    observed_after = (
        t > horizon
    )

    event_index = np.searchsorted(
        distinct,
        t[event_before],
        side="left"
    )

    g_before = before[
        event_index
    ]

    if (
        g_before.size
        and g_before.min() < 0.10
    ):
        return np.nan, g_h

    total = (
        np.sum(
            survival[event_before] ** 2
            / g_before
        )
        + np.sum(
            (
                1 - survival[observed_after]
            ) ** 2
            / g_h
        )
    )

    return float(total / len(t)), g_h


# ============================================================
# 6. Frozen validation evaluation
# ============================================================

def run_stage27():

    required = (
        CASE_FILE,
        SPLIT_FILE,
        GENE_FILE,
        RNA_FILE,
        W_FILE,
        CLINICAL_FILE,
        STAGE_FILE,
        CDR_FILE,
        MODEL_FILE
    )

    for path in required:

        if not path.is_file():
            raise FileNotFoundError(
                f"Required file unavailable: {path}"
            )

    if hashlib.md5(
        CDR_FILE.read_bytes()
    ).hexdigest() != CDR_MD5:

        raise RuntimeError(
            "TCGA-CDR workbook checksum "
            "differs from Stage 21."
        )

    OUT.mkdir(
        parents=True,
        exist_ok=True
    )

    model = joblib.load(
        MODEL_FILE
    )

    if (
        model.get("model") != "RNA_SVD8"
        or model.get("spec", {}).get("cnv")
        or model.get("spec", {}).get("hist")
        or not model.get("spec", {}).get("rna")
    ):
        raise RuntimeError(
            "The Stage 26 frozen model is "
            "not RNA_SVD8."
        )

    cases = pd.read_csv(
        CASE_FILE,
        dtype=str,
        keep_default_na=False
    )

    split = pd.read_csv(
        SPLIT_FILE,
        dtype=str,
        keep_default_na=False
    )

    genes = pd.read_csv(
        GENE_FILE,
        dtype=str,
        keep_default_na=False
    )

    train_clinical = pd.read_csv(
        CLINICAL_FILE,
        dtype={"case_id": str}
    )

    if not (
        cases.case_id.is_unique
        and split.case_id.is_unique
        and genes.ensembl_id.is_unique
    ):
        raise RuntimeError(
            "Duplicate identifiers in original cohort."
        )

    cohort = cases[
        [
            "case_id",
            "patient_id",
            "histology",
            "stage_group"
        ]
    ].merge(
        split[
            [
                "case_id",
                "histology",
                "stage_group",
                "split"
            ]
        ],
        on="case_id",
        how="left",
        validate="one_to_one",
        suffixes=("_raw", "_lock")
    )

    if (
        not cohort.histology_raw.equals(
            cohort.histology_lock
        )
        or not cohort.stage_group_raw.equals(
            cohort.stage_group_lock
        )
    ):
        raise RuntimeError(
            "Source labels differ from locked split."
        )

    tr = np.flatnonzero(
        cohort.split.eq("train")
    )

    va = np.flatnonzero(
        cohort.split.eq("validation")
    )

    te = np.flatnonzero(
        cohort.split.eq("test")
    )

    if (
        len(tr),
        len(va),
        len(te),
        len(genes)
    ) != (667, 143, 143, 19944):
        raise RuntimeError(
            "Locked cohort dimensions changed."
        )

    if (
        len(train_clinical) != len(tr)
        or not np.array_equal(
            train_clinical.case_id.to_numpy(),
            cohort.case_id.iloc[tr].to_numpy()
        )
    ):
        raise RuntimeError(
            "Training clinical patient order mismatch."
        )

    eligible_train = np.flatnonzero(
        train_clinical.PFI_valid
        .astype(str)
        .str.lower()
        .eq("true")
        .to_numpy()
    )

    if (
        len(eligible_train) != 657
        or int(
            train_clinical.PFI.iloc[
                eligible_train
            ].sum()
        ) != 225
        or model.get("training_case_ids")
        != train_clinical.case_id.iloc[
            eligible_train
        ].tolist()
    ):
        raise RuntimeError(
            "Frozen model training identifiers "
            "or PFI events disagree."
        )

    # Freeze the exact model and evaluation protocol
    # before accessing validation outcomes.
    protocol = {
        "version": VERSION,
        "model_sha256": sha256_file(MODEL_FILE),
        "case_sha256": sha256_file(CASE_FILE),
        "split_sha256": sha256_file(SPLIT_FILE),
        "gene_sha256": sha256_file(GENE_FILE),
        "stage_sha256": sha256_file(STAGE_FILE),
        "cdr_md5": CDR_MD5,
        "endpoint": (
            "TCGA-CDR PFI; 0=censored, 1=event"
        ),
        "primary_metric": (
            "validation Harrell C-index paired difference "
            "RNA-SVD8 minus clinical"
        ),
        "horizons_days": list(HORIZONS),
        "bootstrap_sites": N_BOOT,
        "seed": SEED,
        "model_retraining_or_recalibration": False,
        "test_partition_scored": False
    }

    if PROTOCOL_FILE.exists():

        previous = json.loads(
            PROTOCOL_FILE.read_text(
                encoding="utf-8"
            )
        )

        previous.pop(
            "frozen_utc",
            None
        )

        if previous != protocol:
            raise RuntimeError(
                "Frozen validation protocol changed. "
                "Do not overwrite or retune."
            )

    else:

        stored = dict(protocol)

        stored["frozen_utc"] = (
            datetime.now(
                timezone.utc
            ).isoformat()
        )

        PROTOCOL_FILE.write_text(
            json.dumps(stored, indent=2),
            encoding="utf-8"
        )

    # Reuse the existing validation report
    # rather than repeating evaluation.
    if REPORT_FILE.exists():

        saved = json.loads(
            REPORT_FILE.read_text(
                encoding="utf-8"
            )
        )

        if (
            saved.get("model_sha256")
            != protocol["model_sha256"]
        ):
            raise RuntimeError(
                "Existing report uses a different model."
            )

        print(
            "Stage 27 already evaluated; "
            "frozen results reused:"
        )

        print(
            json.dumps(
                saved,
                indent=2
            )
        )

        return

    # ========================================================
    # 7. Recover validation clinical inputs
    # ========================================================

    validation = cohort.iloc[
        va
    ][
        [
            "case_id",
            "patient_id",
            "histology_raw",
            "stage_group_raw"
        ]
    ].copy()

    validation = validation.rename(
        columns={
            "histology_raw": "histology",
            "stage_group_raw": "stage_group"
        }
    )

    validation["patient_id"] = (
        validation.patient_id
        .astype(str)
        .str.upper()
        .str.slice(0, 12)
    )

    if not validation.patient_id.is_unique:
        raise RuntimeError(
            "Duplicated validation patient ID."
        )

    stage = pd.read_csv(
        STAGE_FILE,
        dtype=str,
        keep_default_na=False
    )

    if not {
        "case_id",
        "stage_original"
    }.issubset(stage.columns):
        raise RuntimeError(
            "Original pathological stage annotations "
            "are unavailable."
        )

    if (
        stage.groupby("case_id")
        .stage_original.nunique()
        .gt(1)
        .any()
    ):
        raise RuntimeError(
            "Conflicting original stage annotations."
        )

    validation = validation.merge(
        stage[
            ["case_id", "stage_original"]
        ].drop_duplicates("case_id"),
        on="case_id",
        how="left",
        validate="one_to_one"
    )

    validation["stage_major"] = (
        validation.stage_original.map(
            major_stage
        )
    )

    if (
        validation.stage_major.isna().any()
        or not np.array_equal(
            validation.stage_group.eq(
                "Advanced"
            ).to_numpy(),
            (
                validation.stage_major >= 3
            ).to_numpy()
        )
    ):
        raise RuntimeError(
            "Validation AJCC stage missing "
            "or conflicts with locked stage group."
        )

    val_cdr = extract_cdr_validation(
        CDR_FILE,
        set(validation.patient_id)
    )

    validation = validation.merge(
        val_cdr,
        on="patient_id",
        how="left",
        validate="one_to_one"
    )

    if (
        validation["type"].isna().any()
        or not np.array_equal(
            validation["type"]
            .astype(str)
            .str.upper(),
            validation.histology
        )
    ):
        raise RuntimeError(
            "Missing TCGA-CDR records or "
            "validation histology mismatch."
        )

    validation["age_years"] = pd.to_numeric(
        validation.age_at_initial_pathologic_diagnosis,
        errors="coerce"
    )

    validation.loc[
        ~validation.age_years.between(18, 105),
        "age_years"
    ] = np.nan

    sex = (
        validation.gender
        .astype(str)
        .str.lower()
        .str.strip()
    )

    validation["male"] = np.where(
        sex.eq("male"),
        1.0,
        np.where(
            sex.eq("female"),
            0.0,
            np.nan
        )
    )

    validation["PFI"] = pd.to_numeric(
        validation.PFI,
        errors="coerce"
    )

    validation["PFI.time"] = pd.to_numeric(
        validation["PFI.time"],
        errors="coerce"
    )

    validation["redacted"] = (
        validation.Redaction
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("redacted")
    )

    validation["PFI_valid"] = (
        validation.PFI.isin((0, 1))
        & np.isfinite(
            validation["PFI.time"]
        )
        & validation["PFI.time"].gt(0)
        & ~validation.redacted
    )

    valid_index = np.flatnonzero(
        validation.PFI_valid.to_numpy()
    )

    if (
        len(valid_index) < 50
        or validation.PFI.iloc[
            valid_index
        ].sum() < 12
    ):
        raise RuntimeError(
            "Insufficient validation PFI outcomes."
        )

    # ========================================================
    # 8. Apply frozen model to training and validation
    # ========================================================

    train_frame = (
        train_clinical.iloc[
            eligible_train
        ]
        .rename(
            columns={
                "histology_matrix": "histology"
            }
        )
        .copy()
    )

    W = sparse.load_npz(
        W_FILE
    ).tocsr().astype(
        np.float32
    )

    if W.shape != (1613, 19944):
        raise RuntimeError(
            "Reactome mapping changed."
        )

    source = np.load(
        RNA_FILE,
        mmap_mode="r"
    )

    if source.shape != (953, 19944):
        raise RuntimeError(
            "RNA source dimensions changed."
        )

    # Extract training and validation rows only.
    X_train = pathway_scores(
        source,
        tr[eligible_train],
        W,
        len(genes)
    )

    X_validation = pathway_scores(
        source,
        va,
        W,
        len(genes)
    )

    del source

    train_clin_risk, train_full_risk = frozen_scores(
        train_frame,
        X_train,
        model
    )

    val_clin_risk, val_full_risk = frozen_scores(
        validation,
        X_validation,
        model
    )

    validation["clinical_risk"] = (
        val_clin_risk
    )

    validation["rna_svd8_risk"] = (
        val_full_risk
    )

    val = validation.iloc[
        valid_index
    ].copy()

    t = val["PFI.time"].to_numpy(
        float
    )

    e = val.PFI.to_numpy(
        int
    )

    c_risk = val.clinical_risk.to_numpy(
        float
    )

    m_risk = val.rna_svd8_risk.to_numpy(
        float
    )

    c_clinical, comparable = cindex(
        t,
        e,
        c_risk
    )

    c_molecular, _ = cindex(
        t,
        e,
        m_risk
    )

    observed_delta = (
        c_molecular - c_clinical
    )

    # ========================================================
    # 9. Validation site and subtype robustness
    # ========================================================

    val["site"] = (
        val.patient_id.str.extract(
            r"^TCGA-([A-Z0-9]{2})-",
            expand=False
        )
    )

    if val.site.isna().any():
        raise RuntimeError(
            "Invalid tissue-source-site barcode."
        )

    all_train_sites = set(
        train_clinical.site.astype(str)
    )

    val["site_seen_in_training"] = (
        val.site.isin(
            all_train_sites
        )
    )

    rng = np.random.default_rng(
        SEED
    )

    sites_unique = val.site.unique()
    bootstrap = []

    for _ in range(N_BOOT):

        sampled_sites = rng.choice(
            sites_unique,
            size=len(sites_unique),
            replace=True
        )

        ix = np.concatenate([
            np.flatnonzero(
                val.site.to_numpy() == s
            )
            for s in sampled_sites
        ])

        if (
            e[ix].sum() < 10
            or (e[ix] == 0).sum() < 10
        ):
            continue

        molecular, _ = cindex(
            t[ix],
            e[ix],
            m_risk[ix]
        )

        clinical_c, _ = cindex(
            t[ix],
            e[ix],
            c_risk[ix]
        )

        if (
            np.isfinite(molecular)
            and np.isfinite(clinical_c)
        ):
            bootstrap.append((
                clinical_c,
                molecular,
                molecular - clinical_c
            ))

    if len(bootstrap) < 200:
        raise RuntimeError(
            "Insufficient valid site-cluster bootstrap draws."
        )

    interval = np.quantile(
        np.asarray(bootstrap),
        [0.025, 0.975],
        axis=0
    )

    # ========================================================
    # 10. Training-only calibration functions
    # ========================================================

    train_t = pd.to_numeric(
        train_frame["PFI.time"],
        errors="raise"
    ).to_numpy(float)

    train_e = pd.to_numeric(
        train_frame.PFI,
        errors="raise"
    ).to_numpy(int)

    clinical_curve = baseline_hazard(
        train_t,
        train_e,
        train_clin_risk
    )

    molecular_curve = baseline_hazard(
        train_t,
        train_e,
        train_full_risk
    )

    # Validation censoring distribution is used
    # for metric estimation, not model fitting.
    censor_km = censoring_distribution(
        t,
        e
    )

    brier_rows = []
    calibration_rows = []

    training_cuts = np.quantile(
        train_full_risk,
        [1 / 3, 2 / 3]
    )

    val_groups = np.digitize(
        m_risk,
        training_cuts
    )

    for horizon in HORIZONS:

        surv_clin = survival_probability(
            clinical_curve,
            c_risk,
            horizon
        )

        surv_mol = survival_probability(
            molecular_curve,
            m_risk,
            horizon
        )

        clinical_brier, g_h = ipcw_brier(
            t,
            e,
            surv_clin,
            horizon,
            censor_km
        )

        molecular_brier, _ = ipcw_brier(
            t,
            e,
            surv_mol,
            horizon,
            censor_km
        )

        brier_rows.append({
            "horizon_days": horizon,
            "validation_censoring_G": g_h,
            "clinical_IPCW_Brier": clinical_brier,
            "RNA_SVD8_IPCW_Brier": molecular_brier,
            "delta_Brier_lower_better": (
                molecular_brier - clinical_brier
            )
        })

        for group in range(3):

            selected = (
                val_groups == group
            )

            if selected.sum() < 8:
                continue

            calibration_rows.append({
                "horizon_days": horizon,
                "risk_tertile": (
                    "low",
                    "middle",
                    "high"
                )[group],
                "patients": int(
                    selected.sum()
                ),
                "events_by_horizon": int(
                    np.sum(
                        (t[selected] <= horizon)
                        & (e[selected] == 1)
                    )
                ),
                "predicted_event_probability": float(
                    1 - surv_mol[selected].mean()
                ),
                "KM_observed_event_probability": float(
                    1 - km_survival_at(
                        t[selected],
                        e[selected],
                        horizon
                    )
                )
            })

    # ========================================================
    # 11. Subgroup evaluation
    # ========================================================

    subgroup_rows = []

    subgroups = (
        (
            "LUAD",
            val.histology.eq("LUAD").to_numpy()
        ),
        (
            "LUSC",
            val.histology.eq("LUSC").to_numpy()
        ),
        (
            "site_seen_in_training",
            val.site_seen_in_training.to_numpy()
        ),
        (
            "site_not_seen_in_training",
            ~val.site_seen_in_training.to_numpy()
        )
    )

    for label, mask in subgroups:

        if (
            mask.sum() < 15
            or e[mask].sum() < 6
        ):
            subgroup_rows.append({
                "subgroup": label,
                "patients": int(mask.sum()),
                "events": int(e[mask].sum()),
                "clinical_C": np.nan,
                "RNA_SVD8_C": np.nan
            })

        else:

            subgroup_rows.append({
                "subgroup": label,
                "patients": int(mask.sum()),
                "events": int(e[mask].sum()),
                "clinical_C": cindex(
                    t[mask],
                    e[mask],
                    c_risk[mask]
                )[0],
                "RNA_SVD8_C": cindex(
                    t[mask],
                    e[mask],
                    m_risk[mask]
                )[0]
            })

    subgroup_table = pd.DataFrame(
        subgroup_rows
    )

    brier_table = pd.DataFrame(
        brier_rows
    )

    calibration_table = pd.DataFrame(
        calibration_rows
    )

    # ========================================================
    # 12. Save frozen evaluation artifacts
    # ========================================================

    validation[
        [
            "case_id",
            "patient_id",
            "histology",
            "stage_major",
            "PFI",
            "PFI.time",
            "PFI_valid",
            "redacted",
            "clinical_risk",
            "rna_svd8_risk"
        ]
    ].to_csv(
        OUT / "stage27_all_validation_predictions.csv",
        index=False
    )

    subgroup_table.to_csv(
        OUT / "stage27_subgroup_results.csv",
        index=False
    )

    brier_table.to_csv(
        OUT / "stage27_ipcw_brier.csv",
        index=False
    )

    calibration_table.to_csv(
        OUT / "stage27_km_calibration_tertiles.csv",
        index=False
    )

    summary = {
        "completed_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "model_sha256": protocol["model_sha256"],
        "frozen_model": (
            "Stage 26 RNA_SVD8, trained on "
            "657 PFI-eligible patients"
        ),
        "validation_total": len(validation),
        "validation_PFI_eligible": len(val),
        "validation_PFI_events": int(e.sum()),
        "validation_censored": int(
            (e == 0).sum()
        ),
        "validation_sites": int(
            len(sites_unique)
        ),
        "patients_from_training_seen_sites": int(
            val.site_seen_in_training.sum()
        ),
        "patients_from_unseen_sites": int(
            (~val.site_seen_in_training).sum()
        ),
        "Harrell_comparable_pairs": comparable,
        "clinical_C": c_clinical,
        "RNA_SVD8_C": c_molecular,
        "delta_C_RNA_minus_clinical": (
            observed_delta
        ),
        "exploratory_site_cluster_95pct_CI": {
            "clinical_C": interval[:, 0].tolist(),
            "RNA_SVD8_C": interval[:, 1].tolist(),
            "delta_C": interval[:, 2].tolist()
        },
        "IPCW_Brier": brier_rows,
        "calibration_tertiles": calibration_rows,
        "subgroup_metrics": subgroup_rows,
        "sealed_test_cases_scored_or_evaluated": 0,
        "notes": [
            "Frozen model evaluated once without refitting.",
            "Validation is an internal TCGA holdout.",
            "Site-cluster bootstrap intervals are exploratory.",
            "Breslow baseline hazards use training PFI only.",
            "IPCW Brier assumes independent censoring.",
            "Model must not be retuned using validation results."
        ]
    }

    temporary = REPORT_FILE.with_name(
        REPORT_FILE.name + ".tmp"
    )

    temporary.write_text(
        json.dumps(
            json_safe(summary),
            indent=2,
            allow_nan=False
        ),
        encoding="utf-8"
    )

    temporary.replace(
        REPORT_FILE
    )

    # ========================================================
    # 13. Final report
    # ========================================================

    print("\n" + "=" * 76)
    print(
        "STAGE 27 FINAL - FROZEN VALIDATION PFI SURVIVAL"
    )
    print("=" * 76)

    print(
        f"Validation: {len(validation)} total | "
        f"{len(val)} PFI eligible | "
        f"{int(e.sum())} events | "
        f"{int((e == 0).sum())} censored"
    )

    print(
        f"Sites: {len(sites_unique)} | "
        "patients at training-seen sites: "
        f"{int(val.site_seen_in_training.sum())}"
    )

    print(
        f"Clinical-only C-index: {c_clinical:.4f}"
    )

    print(
        f"Frozen RNA-SVD8 C-index: {c_molecular:.4f}"
    )

    print(
        f"Paired C-index difference: {observed_delta:+.4f}"
    )

    print(
        "Exploratory site-cluster 95% CI:",
        interval[:, 2].round(4).tolist()
    )

    print(
        "\nPFI IPCW Brier (lower is better):"
    )

    print(
        brier_table.to_string(
            index=False,
            float_format=lambda v: f"{v:.4f}"
        )
    )

    print("\nValidation subgroups:")

    print(
        subgroup_table.to_string(
            index=False,
            float_format=lambda v: f"{v:.4f}"
        )
    )

    print(
        "\nKM calibration by training-defined risk tertile:"
    )

    print(
        calibration_table.to_string(
            index=False,
            float_format=lambda v: f"{v:.4f}"
        )
    )

    print(
        "\nFrozen validation report:",
        OUT
    )

    print(
        "SEALED TEST: NOT SCORED; "
        "validation result must not be used for new tuning."
    )

    print("STAGE 27 COMPLETED")


run_stage27()


STAGE 27 FINAL - FROZEN VALIDATION PFI SURVIVAL
Validation: 143 total | 140 PFI eligible | 50 events | 90 censored
Sites: 46 | patients at training-seen sites: 136
Clinical-only C-index: 0.6339
Frozen RNA-SVD8 C-index: 0.6437
Paired C-index difference: +0.0098
Exploratory site-cluster 95% CI: [-0.0191, 0.0457]

PFI IPCW Brier (lower is better):
 horizon_days  validation_censoring_G  clinical_IPCW_Brier  RNA_SVD8_IPCW_Brier  delta_Brier_lower_better
          365                  0.8175               0.1362               0.1342                   -0.0020
         1095                  0.3871               0.2287               0.2292                    0.0005

Validation subgroups:
                 subgroup  patients  events  clinical_C  RNA_SVD8_C
                     LUAD        70      29      0.6604      0.6756
                     LUSC        70      21      0.6269      0.6338
    site_seen_in_training       136      48      0.6272      0.6353
site_not_seen_in_training         4    

In [27]:
import hashlib
import json
import re
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from scipy import sparse
from scipy.stats import rankdata

# Fixed, one-time sealed-test protocol. Run this entire script as one Jupyter cell.
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"
RAW = ROOT / "data" / "processed" / "gene_level"
S19 = ROOT / "results" / "metrics" / "stage19"
S21 = ROOT / "results" / "metrics" / "stage21"
S27 = ROOT / "results" / "metrics" / "stage27_frozen_validation"
OUT = ROOT / "results" / "metrics" / "stage28_sealed_test_final"
MODEL_FILE = ROOT / "models" / "checkpoints" / "stage26" / "stage26_training_only_final.joblib"
CDR_FILE = ROOT / "data" / "raw" / "clinical_references" / "TCGA-CDR-SupplementalTableS1.xlsx"
STAGE_FILE = ROOT / "data" / "interim" / "tcga_clinical_stage_audit.csv"
CASE_FILE = RAW / "case_order.csv"
GENE_FILE = RAW / "gene_reference.csv"
RNA_FILE = RAW / "rna_tpm.npy"
W_FILE = S19 / "reactome_gene_pathway_weights.npz"
SPLIT_FILE = ROOT / "data" / "splits" / "locked_four_omics_v1.csv"
TRAIN_FILE = S21 / "stage21_training_clinical_endpoints.csv"
VALIDATION_PROTOCOL = S27 / "stage27_preregistered_freeze.json"
VALIDATION_REPORT = S27 / "stage27_validation_summary.json"
PROTOCOL_FILE = OUT / "stage28_test_protocol_locked.json"
REPORT_FILE = OUT / "stage28_final_test_summary.json"
CDR_MD5 = "a4591b2dcee39591f59e5e25a6ce75fa"
VERSION = "stage28_one_time_sealed_pfi_test_v1"
HORIZONS = (365, 1095)
BOOTSTRAPS = 1500
SEED = 2801

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError("Select the NST NSCLC Research (Python 3.12) Jupyter kernel.")


def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(2**20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def c_index(t, e, risk):
    t, e, risk = np.asarray(t, float), np.asarray(e, int), np.asarray(risk, float)
    pairs = (t[:, None] < t[None, :]) & (e[:, None] == 1)
    n = int(pairs.sum())
    if n < 10:
        return float("nan"), n
    diff = risk[:, None] - risk[None, :]
    result = ((pairs & (diff > 1e-10)).sum() + 0.5 * (pairs & (np.abs(diff) <= 1e-10)).sum()) / n
    return float(result), n


def major_stage(label):
    match = re.search(r"STAGE\s*(IV|III|II|I)(?:[ABC])?\b", str(label).upper())
    return {"I": 1, "II": 2, "III": 3, "IV": 4}[match.group(1)] if match else np.nan


def fetch_test_cdr(path, allowed):
    wanted = (
        "bcr_patient_barcode", "type", "PFI", "PFI.time", "Redaction",
        "age_at_initial_pathologic_diagnosis", "gender"
    )
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        if "TCGA-CDR" not in workbook.sheetnames:
            raise RuntimeError("TCGA-CDR worksheet not found.")
        values = workbook["TCGA-CDR"].iter_rows(values_only=True)
        names = {str(v).strip(): i for i, v in enumerate(next(values))}
        if not set(wanted).issubset(names):
            raise RuntimeError("TCGA-CDR required survival or clinical columns missing.")
        records = []
        for row in values:
            patient = str(row[names["bcr_patient_barcode"]]).strip().upper()[:12]
            if patient in allowed:
                record = {name: row[names[name]] for name in wanted}
                record["patient_id"] = patient
                records.append(record)
    finally:
        workbook.close()
    df = pd.DataFrame(records)
    if df.empty or not df.patient_id.is_unique:
        raise RuntimeError("Missing or duplicate test clinical patient records.")
    return df


def pathway_features(matrix, indices, weights, n_genes):
    raw = np.asarray(matrix[indices], dtype=np.float32).copy()
    good = np.isfinite(raw) & (raw >= 0)
    if good.mean() < 0.95:
        raise RuntimeError("RNA missingness exceeds the preset threshold.")
    raw = np.log1p(np.where(good, raw, 0.0)).astype(np.float32)
    ranks = rankdata(raw, axis=1, method="average").astype(np.float32)
    ranks = (ranks - (n_genes + 1.0) / 2.0) / n_genes
    scores = np.asarray((weights @ ranks.T).T, dtype=np.float32)
    if not np.isfinite(scores).all():
        raise RuntimeError("Non-finite Reactome pathway scores.")
    return scores


def frozen_risk(data, pathways, model):
    state = model["preprocessing"]
    cstate = state["clinical"]
    ages = pd.to_numeric(data.age_years, errors="coerce").to_numpy(float)
    sex = pd.to_numeric(data.male, errors="coerce").to_numpy(float)
    stages = pd.to_numeric(data.stage_major, errors="coerce").to_numpy(float)
    hist = data.histology.astype(str).eq("LUSC").to_numpy(float)
    median = float(cstate["age_median"])
    a = np.where(np.isfinite(ages), ages, median)
    features = np.column_stack([
        (a - median) / 10.0, (~np.isfinite(ages)).astype(float),
        np.where(np.isfinite(sex), sex, 0.5), (~np.isfinite(sex)).astype(float),
        hist, (stages == 2).astype(float), (stages == 3).astype(float), (stages == 4).astype(float)
    ])
    clinical = cstate["scaler"].transform(features) @ np.asarray(cstate["beta"], float)
    rstate = state["rna"]
    z = np.clip(rstate["scaler"].transform(pathways), -7, 7)
    z = rstate["svd"].transform(z)
    z = np.clip(rstate["latent_scaler"].transform(z), -7, 7)
    coef = np.asarray(model["model_state"]["coef"], float)
    if z.shape[1] != 8 or coef.shape != (8,):
        raise RuntimeError("The frozen RNA-SVD8 feature state is inconsistent.")
    molecular = clinical + z @ coef
    if not (np.isfinite(clinical).all() and np.isfinite(molecular).all()):
        raise RuntimeError("Non-finite frozen model scores.")
    return clinical, molecular


def breslow_baseline(t, e, risk):
    t, e, risk = np.asarray(t, float), np.asarray(e, int), np.asarray(risk, float)
    shift = float(np.max(risk))
    w = np.exp(np.clip(risk - shift, -50, 0))
    moments, counts = np.unique(t[e == 1], return_counts=True)
    increments = np.array([d / np.sum(w[t >= moment]) for moment, d in zip(moments, counts)])
    return moments, np.cumsum(increments), shift


def predicted_survival(curve, score, horizon):
    moments, cumhaz, shift = curve
    j = np.searchsorted(moments, horizon, side="right") - 1
    h = float(cumhaz[j]) if j >= 0 else 0.0
    return np.exp(-h * np.exp(np.clip(np.asarray(score, float) - shift, -40, 40)))


def censor_km(t, e):
    # Kaplan-Meier for censoring events, with actual PFI events treated as censored.
    t, e = np.asarray(t, float), np.asarray(e, int)
    distinct = np.unique(t)
    survival = np.ones(len(distinct), float)
    current = 1.0
    for j, moment in enumerate(distinct):
        at_risk = int(np.sum(t >= moment))
        lost = int(np.sum((t == moment) & (e == 0)))
        current *= 1.0 - lost / at_risk
        survival[j] = current
    return distinct, survival


def g_at(km, t, left=False):
    distinct, survival = km
    t = np.asarray(t, float)
    index = np.searchsorted(distinct, t, side="left" if left else "right") - 1
    answer = np.ones_like(t, dtype=float)
    mask = index >= 0
    answer[mask] = survival[index[mask]]
    return answer


def ipcw_brier(t, e, survival, horizon, km):
    t, e, survival = np.asarray(t, float), np.asarray(e, int), np.asarray(survival, float)
    g_h = float(g_at(km, np.array([horizon]))[0])
    if g_h < 0.10:
        return float("nan"), g_h
    observed_events = (t <= horizon) & (e == 1)
    still_observed = t > horizon
    g_events = g_at(km, t[observed_events], left=True)
    if len(g_events) and np.min(g_events) < 0.10:
        return float("nan"), g_h
    score = np.sum(survival[observed_events] ** 2 / g_events)
    score += np.sum((1.0 - survival[still_observed]) ** 2 / g_h)
    return float(score / len(t)), g_h


def uno_ipcw_cindex(t, e, risk, km_train, tau=1095):
    # Supplementary truncated Uno estimator; training-derived censoring KM.
    g = g_at(km_train, t, left=True)
    valid_event = (e == 1) & (t <= tau) & (g >= 0.10)
    pairs = (t[:, None] < t[None, :]) & valid_event[:, None]
    weight = np.where(valid_event, 1.0 / np.maximum(g, 0.10) ** 2, 0.0)
    denom = float((pairs * weight[:, None]).sum())
    if denom < 1e-8:
        return float("nan"), 0
    difference = risk[:, None] - risk[None, :]
    success = (difference > 1e-10).astype(float) + 0.5 * (np.abs(difference) <= 1e-10)
    return float((pairs * weight[:, None] * success).sum() / denom), int(pairs.sum())


def km_event_risk(t, e, horizon):
    t, e = np.asarray(t, float), np.asarray(e, int)
    s = 1.0
    for moment in np.unique(t[(e == 1) & (t <= horizon)]):
        s *= 1.0 - np.sum((t == moment) & (e == 1)) / np.sum(t >= moment)
    return float(1.0 - s)


def safe_float(x):
    return float(x) if np.isfinite(x) else None


def run_stage28():
    required = (
        CASE_FILE, GENE_FILE, RNA_FILE, W_FILE, SPLIT_FILE, TRAIN_FILE,
        STAGE_FILE, CDR_FILE, MODEL_FILE, VALIDATION_REPORT, VALIDATION_PROTOCOL
    )
    for path in required:
        if not path.is_file():
            raise FileNotFoundError(f"Missing required frozen artifact: {path}")
    if hashlib.md5(CDR_FILE.read_bytes()).hexdigest() != CDR_MD5:
        raise RuntimeError("TCGA-CDR checksum changed; test remains sealed.")

    model_sha = file_sha256(MODEL_FILE)
    val_protocol = json.loads(VALIDATION_PROTOCOL.read_text(encoding="utf-8"))
    val_report = json.loads(VALIDATION_REPORT.read_text(encoding="utf-8"))
    if val_protocol.get("model_sha256") != model_sha or val_report.get("model_sha256") != model_sha:
        raise RuntimeError("Stage 27 and Stage 26 frozen model hashes disagree; do not unseal test.")

    model = joblib.load(MODEL_FILE)
    spec = model.get("spec", {})
    if model.get("model") != "RNA_SVD8" or not spec.get("rna") or spec.get("cnv") or spec.get("hist"):
        raise RuntimeError("The frozen model is not the prespecified RNA-SVD8 model.")

    cases = pd.read_csv(CASE_FILE, dtype=str, keep_default_na=False)
    splits = pd.read_csv(SPLIT_FILE, dtype=str, keep_default_na=False)
    genes = pd.read_csv(GENE_FILE, dtype=str, keep_default_na=False)
    train = pd.read_csv(TRAIN_FILE, dtype={"case_id": str})
    if not (cases.case_id.is_unique and splits.case_id.is_unique and genes.ensembl_id.is_unique):
        raise RuntimeError("Cohort contains duplicate identifiers.")
    cohort = cases[["case_id", "patient_id", "histology", "stage_group"]].merge(
        splits[["case_id", "histology", "stage_group", "split"]],
        on="case_id", how="left", validate="one_to_one", suffixes=("_raw", "_lock")
    )
    if not (cohort.histology_raw.equals(cohort.histology_lock) and cohort.stage_group_raw.equals(cohort.stage_group_lock)):
        raise RuntimeError("Clinical annotations changed since the locked split.")
    tr = np.flatnonzero(cohort.split.eq("train"))
    va = np.flatnonzero(cohort.split.eq("validation"))
    te = np.flatnonzero(cohort.split.eq("test"))
    if (len(tr), len(va), len(te), len(genes)) != (667, 143, 143, 19944):
        raise RuntimeError("The original 667/143/143 split or gene count changed.")
    if len(train) != 667 or not np.array_equal(train.case_id.to_numpy(), cohort.case_id.iloc[tr].to_numpy()):
        raise RuntimeError("Training patient order does not match Stage 21.")
    training_eligible = np.flatnonzero(train.PFI_valid.astype(str).str.lower().eq("true"))
    if len(training_eligible) != 657 or int(train.PFI.iloc[training_eligible].sum()) != 225:
        raise RuntimeError("Training PFI eligibility or events have changed.")
    if model.get("training_case_ids") != train.case_id.iloc[training_eligible].tolist():
        raise RuntimeError("Frozen model patient list differs from Stage 21 training.")

    # Lock the model, endpoint, metrics, and source checksums before test outcomes are read.
    protocol = {
        "version": VERSION, "model_sha256": model_sha,
        "validation_report_sha256": file_sha256(VALIDATION_REPORT),
        "endpoint": "TCGA-CDR PFI", "positive_event": 1,
        "primary_metric": "test Harrell C-index difference: RNA-SVD8 minus clinical Cox",
        "secondary_metrics": ["Uno IPCW C-index through day 1095", "IPCW Brier at 365 and 1095 days", "training-tertile KM calibration", "LUAD/LUSC subgroup C-index"],
        "risk_horizons_days": list(HORIZONS), "bootstrap_sampling_unit": "TCGA tissue-source site",
        "bootstrap_replicates": BOOTSTRAPS, "bootstrap_seed": SEED,
        "frozen_training_baseline_hazards": True, "refitting_or_recalibration": False,
        "data_sha256": {str(p): file_sha256(p) for p in (CASE_FILE, SPLIT_FILE, GENE_FILE, RNA_FILE, W_FILE, STAGE_FILE, TRAIN_FILE)},
        "cdr_md5": CDR_MD5, "test_used_before_protocol": False,
    }
    OUT.mkdir(parents=True, exist_ok=True)
    if PROTOCOL_FILE.exists():
        stored_protocol = json.loads(PROTOCOL_FILE.read_text(encoding="utf-8"))
        stored_protocol.pop("locked_utc", None)
        if stored_protocol != protocol:
            raise RuntimeError("Stage 28 test protocol changed; refuse to rerun a different analysis.")
    else:
        saved_protocol = dict(protocol)
        saved_protocol["locked_utc"] = datetime.now(timezone.utc).isoformat()
        PROTOCOL_FILE.write_text(json.dumps(saved_protocol, indent=2), encoding="utf-8")

    if REPORT_FILE.exists():
        old = json.loads(REPORT_FILE.read_text(encoding="utf-8"))
        if old.get("model_sha256") != model_sha or old.get("protocol_sha256") != file_sha256(PROTOCOL_FILE):
            raise RuntimeError("A previous sealed-test report does not match the locked protocol.")
        print("STAGE 28 ALREADY COMPLETED - EXISTING FINAL TEST REPORT REUSED")
        print(json.dumps(old, indent=2, ensure_ascii=False))
        return

    # Only the originally locked 143 test patients are looked up in TCGA-CDR.
    test = cohort.iloc[te][["case_id", "patient_id", "histology_raw", "stage_group_raw"]].copy()
    test = test.rename(columns={"histology_raw": "histology", "stage_group_raw": "stage_group"})
    test["patient_id"] = test.patient_id.astype(str).str.upper().str.slice(0, 12)
    if not test.patient_id.is_unique:
        raise RuntimeError("Duplicate test TCGA patient identifiers.")

    stages = pd.read_csv(STAGE_FILE, dtype=str, keep_default_na=False)
    if not {"case_id", "stage_original"}.issubset(stages.columns):
        raise RuntimeError("Original clinical stage audit cannot be found.")
    if stages.groupby("case_id").stage_original.nunique().gt(1).any():
        raise RuntimeError("Conflicting pathological stage observations.")
    test = test.merge(stages[["case_id", "stage_original"]].drop_duplicates("case_id"),
                      on="case_id", how="left", validate="one_to_one")
    test["stage_major"] = test.stage_original.map(major_stage)
    if test.stage_major.isna().any() or not np.array_equal(
        test.stage_group.eq("Advanced").to_numpy(), (test.stage_major >= 3).to_numpy()
    ):
        raise RuntimeError("Test clinical stage differs from the locked clinical stage.")

    selected_cdr = fetch_test_cdr(CDR_FILE, set(test.patient_id))
    test = test.merge(selected_cdr, on="patient_id", how="left", validate="one_to_one")
    if test["type"].isna().any() or not np.array_equal(test["type"].astype(str).str.upper(), test.histology):
        raise RuntimeError("Test CDR patient/histology mismatch.")
    test["age_years"] = pd.to_numeric(test.age_at_initial_pathologic_diagnosis, errors="coerce")
    test.loc[~test.age_years.between(18, 105), "age_years"] = np.nan
    gender = test.gender.astype(str).str.lower().str.strip()
    test["male"] = np.where(gender.eq("male"), 1.0, np.where(gender.eq("female"), 0.0, np.nan))
    test["PFI"] = pd.to_numeric(test.PFI, errors="coerce")
    test["PFI.time"] = pd.to_numeric(test["PFI.time"], errors="coerce")
    test["redacted"] = test.Redaction.astype(str).str.lower().str.strip().eq("redacted")
    test["PFI_valid"] = test.PFI.isin((0, 1)) & np.isfinite(test["PFI.time"]) & test["PFI.time"].gt(0) & ~test.redacted
    valid_idx = np.flatnonzero(test.PFI_valid.to_numpy())
    if len(valid_idx) < 50 or test.PFI.iloc[valid_idx].sum() < 12:
        raise RuntimeError("Insufficient eligible PFI test cases/events to report the planned metrics.")

    # Frozen transformer scores: no new fitting, and validation rows are never loaded.
    weights = sparse.load_npz(W_FILE).tocsr().astype(np.float32)
    if weights.shape != (1613, 19944):
        raise RuntimeError("Reactome mapping differs from the frozen Stage 26 model.")
    source = np.load(RNA_FILE, mmap_mode="r")
    if source.shape != (953, 19944):
        raise RuntimeError("RNA data dimensions changed.")
    rna_train = pathway_features(source, tr[training_eligible], weights, len(genes))
    rna_test = pathway_features(source, te, weights, len(genes))
    del source

    train_scoring = train.iloc[training_eligible].rename(columns={"histology_matrix": "histology"}).copy()
    train_clin, train_full = frozen_risk(train_scoring, rna_train, model)
    test_clin, test_full = frozen_risk(test, rna_test, model)
    test["clinical_risk"] = test_clin
    test["rna_svd8_risk"] = test_full
    test["site"] = test.patient_id.str.extract(r"^TCGA-([A-Z0-9]{2})-", expand=False)
    if test.site.isna().any():
        raise RuntimeError("Test site barcode cannot be extracted.")
    test["training_seen_site"] = test.site.isin(set(train.site.astype(str)))

    evaluated = test.iloc[valid_idx].copy()
    t = evaluated["PFI.time"].to_numpy(float)
    e = evaluated.PFI.to_numpy(int)
    c = evaluated.clinical_risk.to_numpy(float)
    m = evaluated.rna_svd8_risk.to_numpy(float)
    clinical_C, n_pairs = c_index(t, e, c)
    molecular_C, _ = c_index(t, e, m)
    delta_C = molecular_C - clinical_C

    # Site-cluster bootstrap: exploratory uncertainty for the frozen, paired C-index difference.
    test_sites = evaluated.site.to_numpy()
    sites_unique = np.unique(test_sites)
    rng = np.random.default_rng(SEED)
    draws = []
    for _ in range(BOOTSTRAPS):
        chosen = rng.choice(sites_unique, size=len(sites_unique), replace=True)
        ii = np.concatenate([np.flatnonzero(test_sites == s) for s in chosen])
        if np.sum(e[ii]) < 10 or np.sum(e[ii] == 0) < 10:
            continue
        ca, _ = c_index(t[ii], e[ii], c[ii])
        mo, _ = c_index(t[ii], e[ii], m[ii])
        if np.isfinite(ca) and np.isfinite(mo):
            draws.append((ca, mo, mo - ca))
    if len(draws) < 250:
        raise RuntimeError("Too few valid site-cluster bootstrap draws.")
    bounds = np.quantile(np.asarray(draws), (0.025, 0.975), axis=0)

    train_t = pd.to_numeric(train_scoring["PFI.time"], errors="raise").to_numpy(float)
    train_e = pd.to_numeric(train_scoring.PFI, errors="raise").to_numpy(int)
    km_train = censor_km(train_t, train_e)
    km_test = censor_km(t, e)
    baseline_clin = breslow_baseline(train_t, train_e, train_clin)
    baseline_mol = breslow_baseline(train_t, train_e, train_full)
    cuts = np.quantile(train_full, [1.0 / 3, 2.0 / 3])
    groups = np.digitize(m, cuts)

    brier_rows, calibration_rows = [], []
    for horizon in HORIZONS:
        spc = predicted_survival(baseline_clin, c, horizon)
        spm = predicted_survival(baseline_mol, m, horizon)
        # Same evaluation-cohort censoring estimator as the frozen Stage 27 report.
        bce, gh_test = ipcw_brier(t, e, spc, horizon, km_test)
        bme, _ = ipcw_brier(t, e, spm, horizon, km_test)
        # Sensitivity analysis: training-derived censoring estimator.
        bct, gh_train = ipcw_brier(t, e, spc, horizon, km_train)
        bmt, _ = ipcw_brier(t, e, spm, horizon, km_train)
        brier_rows.append({
            "horizon_days": horizon, "test_censoring_G": gh_test, "train_censoring_G": gh_train,
            "clinical_Brier_testKM": safe_float(bce), "RNA_SVD8_Brier_testKM": safe_float(bme),
            "delta_Brier_testKM": safe_float(bme - bce),
            "clinical_Brier_trainKM": safe_float(bct), "RNA_SVD8_Brier_trainKM": safe_float(bmt),
            "delta_Brier_trainKM": safe_float(bmt - bct)
        })
        for group in range(3):
            mask = groups == group
            if not mask.any():
                continue
            n_at_risk = int(np.sum(t[mask] >= horizon))
            observed = km_event_risk(t[mask], e[mask], horizon) if n_at_risk >= 3 else np.nan
            calibration_rows.append({
                "horizon_days": horizon, "risk_tertile": ("low", "middle", "high")[group],
                "patients": int(mask.sum()), "events_by_horizon": int(((t[mask] <= horizon) & (e[mask] == 1)).sum()),
                "at_risk_at_horizon": n_at_risk,
                "predicted_event_probability": float(np.mean(1 - spm[mask])),
                "KM_observed_event_probability": safe_float(observed)
            })

    uno_clin, uno_pairs = uno_ipcw_cindex(t, e, c, km_train, 1095)
    uno_mol, _ = uno_ipcw_cindex(t, e, m, km_train, 1095)
    subgroup_rows = []
    subsets = (
        ("LUAD", evaluated.histology.eq("LUAD").to_numpy()),
        ("LUSC", evaluated.histology.eq("LUSC").to_numpy()),
        ("seen_training_site", evaluated.training_seen_site.to_numpy()),
        ("unseen_training_site", ~evaluated.training_seen_site.to_numpy()),
    )
    for name, mask in subsets:
        enough = int(mask.sum()) >= 15 and int(e[mask].sum()) >= 6
        a, _ = c_index(t[mask], e[mask], c[mask]) if enough else (np.nan, 0)
        b, _ = c_index(t[mask], e[mask], m[mask]) if enough else (np.nan, 0)
        subgroup_rows.append({
            "subgroup": name, "patients": int(mask.sum()), "PFI_events": int(e[mask].sum()),
            "clinical_C": safe_float(a), "RNA_SVD8_C": safe_float(b), "delta_C": safe_float(b - a)
        })

    # Persist exactly one final report; no model selection or recalibration is performed.
    prediction_path = OUT / "stage28_all_locked_test_predictions.csv"
    test[[
        "case_id", "patient_id", "histology", "site", "training_seen_site", "PFI",
        "PFI.time", "PFI_valid", "redacted", "clinical_risk", "rna_svd8_risk"
    ]].to_csv(prediction_path, index=False)
    pd.DataFrame(brier_rows).to_csv(OUT / "stage28_brier.csv", index=False)
    pd.DataFrame(calibration_rows).to_csv(OUT / "stage28_km_calibration.csv", index=False)
    pd.DataFrame(subgroup_rows).to_csv(OUT / "stage28_subgroup_results.csv", index=False)

    report = {
        "version": VERSION, "completed_utc": datetime.now(timezone.utc).isoformat(),
        "model_sha256": model_sha, "protocol_sha256": file_sha256(PROTOCOL_FILE),
        "test_total": len(test), "PFI_eligible": len(evaluated), "PFI_events": int(e.sum()),
        "PFI_censored": int(np.sum(e == 0)), "test_sites": len(sites_unique),
        "eligible_patients_at_unseen_training_sites": int((~evaluated.training_seen_site).sum()),
        "comparable_pairs": n_pairs, "clinical_C": clinical_C, "RNA_SVD8_C": molecular_C,
        "paired_delta_C": delta_C,
        "exploratory_site_cluster_95pct_CI": {
            "clinical_C": bounds[:, 0].tolist(), "RNA_SVD8_C": bounds[:, 1].tolist(),
            "paired_delta_C": bounds[:, 2].tolist(), "valid_draws": len(draws)
        },
        "Uno_IPCW_3year_train_censor_KM": {
            "clinical_C": safe_float(uno_clin), "RNA_SVD8_C": safe_float(uno_mol),
            "paired_delta": safe_float(uno_mol - uno_clin), "comparable_pairs": uno_pairs
        },
        "Brier": brier_rows, "risk_tertile_KM_calibration": calibration_rows,
        "subgroup_metrics": subgroup_rows, "model_refitted": False,
        "research_caveats": [
            "This is an internal TCGA test partition, not an external institutional cohort.",
            "Stage 24-26 models were explored repeatedly on the same training data.",
            "Stage 27 validation was previously inspected, but this Stage 28 model was not retuned.",
            "The endpoint is PFI; survival discrimination does not establish clinical utility.",
            "Bootstrap confidence intervals are exploratory and site-cluster sizes are heterogeneous.",
            "IPCW Brier assumes non-informative censoring; 3-year results may be unstable."
        ]
    }
    tmp = REPORT_FILE.with_name(REPORT_FILE.name + ".tmp")
    tmp.write_text(json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")
    tmp.replace(REPORT_FILE)

    print("\n" + "=" * 77)
    print("STAGE 28 FINAL - ONE-TIME SEALED TCGA PFI TEST")
    print("=" * 77)
    print(f"Test: {len(test)} total; PFI eligible={len(evaluated)}, events={int(e.sum())}, censored={int(np.sum(e==0))}")
    print(f"Sites={len(sites_unique)}; patients from unseen training sites={int((~evaluated.training_seen_site).sum())}")
    print(f"Clinical-only Harrell C:   {clinical_C:.4f}")
    print(f"Frozen RNA-SVD8 Harrell C: {molecular_C:.4f}")
    print(f"Primary paired delta C:  {delta_C:+.4f}")
    print("Exploratory site-bootstrap 95% CI for delta C:", np.round(bounds[:, 2], 4).tolist())
    print(f"Secondary Uno IPCW C (1095 days): clinical={uno_clin:.4f}, molecular={uno_mol:.4f}")
    print("\nBrier scores (lower is better):")
    print(pd.DataFrame(brier_rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("\nSubgroup discrimination:")
    print(pd.DataFrame(subgroup_rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("\n1- and 3-year calibration:")
    print(pd.DataFrame(calibration_rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
    print("\nTest status: USED ONCE; no further model tuning against this partition.")
    print("Saved frozen report:", REPORT_FILE)
    print("STAGE 28 COMPLETED")


run_stage28()


STAGE 28 FINAL - ONE-TIME SEALED TCGA PFI TEST
Test: 143 total; PFI eligible=138, events=51, censored=87
Sites=43; patients from unseen training sites=1
Clinical-only Harrell C:   0.5863
Frozen RNA-SVD8 Harrell C: 0.6207
Primary paired delta C:  +0.0344
Exploratory site-bootstrap 95% CI for delta C: [-0.004, 0.0814]
Secondary Uno IPCW C (1095 days): clinical=0.5581, molecular=0.5983

Brier scores (lower is better):
 horizon_days  test_censoring_G  train_censoring_G  clinical_Brier_testKM  RNA_SVD8_Brier_testKM  delta_Brier_testKM  clinical_Brier_trainKM  RNA_SVD8_Brier_trainKM  delta_Brier_trainKM
          365            0.7944             0.8089                 0.1523                 0.1469             -0.0054                  0.1527                  0.1473              -0.0054
         1095            0.4185             0.4000                 0.2500                 0.2453             -0.0047                  0.2546                  0.2499              -0.0047

Subgroup discriminati

In [28]:

import hashlib
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


# Configuration
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
RAW = ROOT / "data" / "processed" / "gene_level"
S19 = ROOT / "results" / "metrics" / "stage19"
S21 = ROOT / "results" / "metrics" / "stage21"
S25 = ROOT / "results" / "metrics" / "stage25"
S26 = ROOT / "results" / "metrics" / "stage26"
OUT = ROOT / "results" / "metrics" / "spare_phase_a"

MODEL = (
    ROOT / "models" / "checkpoints" / "stage26"
    / "stage26_training_only_final.joblib"
)

SEED = 42
THREADS = min(4, os.cpu_count() or 1)
TOL = 1e-3
VERSION = "spare_phase_a_recap_reproduction_v1"

EXPECTED_PYTHON = (
    ROOT / "envs" / "nsclc" / "python.exe"
)

if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

OUT.mkdir(parents=True, exist_ok=True)


# File integrity and atomic checkpoint utilities
def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(4 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def atomic_joblib(value, path):
    temporary = path.with_name(path.name + ".tmp")
    joblib.dump(value, temporary, compress=3)
    os.replace(temporary, path)


def atomic_json(value, path):
    temporary = path.with_name(path.name + ".tmp")

    temporary.write_text(
        json.dumps(value, indent=2, allow_nan=False),
        encoding="utf-8"
    )

    os.replace(temporary, path)


# Harrell's concordance index
def cindex(t, e, risk):
    pair = (
        (t[:, None] < t[None, :])
        & (e[:, None] == 1)
    )

    n = int(pair.sum())

    if n < 10:
        raise RuntimeError(
            "Too few comparable survival pairs."
        )

    difference = risk[:, None] - risk[None, :]

    wins = np.count_nonzero(
        pair & (difference > 1e-10)
    )

    ties = np.count_nonzero(
        pair & (np.abs(difference) <= 1e-10)
    )

    return float(
        (wins + 0.5 * ties) / n
    )


# Breslow Cox partial likelihood and analytic gradient
def cox_objective(beta, z, t, e, penalty, offset):
    order = np.argsort(
        -t,
        kind="stable"
    )

    z = np.asarray(
        z,
        dtype=np.float64
    )[order]

    t = np.asarray(
        t,
        dtype=np.float64
    )[order]

    e = np.asarray(
        e,
        dtype=np.int64
    )[order]

    offset = np.asarray(
        offset,
        dtype=np.float64
    )[order]

    start = np.r_[
        0,
        np.flatnonzero(
            t[1:] != t[:-1]
        ) + 1
    ]

    end = np.r_[
        start[1:] - 1,
        len(t) - 1
    ]

    counts = np.add.reduceat(
        e,
        start
    )

    active = counts > 0
    end = end[active]
    counts = counts[active].astype(float)

    eta = z @ beta + offset
    shift = eta.max()

    weights = np.exp(eta - shift)
    cumulative = np.cumsum(weights)

    cumulative_z = np.cumsum(
        weights[:, None] * z,
        axis=0
    )

    denominator = np.maximum(
        cumulative[end],
        1e-300
    )

    n_events = float(e.sum())

    if n_events < 10:
        raise RuntimeError(
            "Insufficient events for Cox estimation."
        )

    loss = (
        np.sum(
            counts
            * (shift + np.log(denominator))
        )
        - eta[e == 1].sum()
    ) / n_events

    gradient = (
        np.sum(
            counts[:, None]
            * cumulative_z[end]
            / denominator[:, None],
            axis=0
        )
        - z[e == 1].sum(axis=0)
    ) / n_events

    penalty = np.broadcast_to(
        np.asarray(penalty, dtype=float),
        (z.shape[1],)
    )

    return (
        float(
            loss
            + 0.5 * np.dot(
                penalty,
                beta * beta
            )
        ),
        gradient + penalty * beta
    )


def fit_cox(z, t, e, penalty, offset=None):
    if offset is None:
        offset = np.zeros(
            len(t),
            dtype=float
        )

    if (
        z.ndim != 2
        or len(t) != len(e)
        or len(t) != len(offset)
        or len(t) != len(z)
    ):
        raise RuntimeError(
            "Cox input dimension mismatch."
        )

    outcome = minimize(
        lambda b: cox_objective(
            b, z, t, e, penalty, offset
        ),
        np.zeros(z.shape[1]),
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter": 250,
            "ftol": 1e-9,
            "gtol": 1e-6
        }
    )

    if (
        not np.isfinite(outcome.x).all()
        or not np.isfinite(outcome.fun)
    ):
        raise RuntimeError(
            "Non-finite Cox fit."
        )

    return outcome.x


# Verify the gradient, including tied event times
rng_check = np.random.default_rng(SEED)

check_z = rng_check.normal(
    size=(12, 4)
)

check_t = np.array(
    [9, 9, 8, 8, 7, 6, 5, 5, 4, 3, 2, 1],
    dtype=float
)

check_e = np.array(
    [1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1],
    dtype=float
)

check_off = (
    rng_check.normal(size=12) * 0.15
)

check_beta = (
    rng_check.normal(size=4) * 0.2
)

check_loss, analytic = cox_objective(
    check_beta,
    check_z,
    check_t,
    check_e,
    0.1,
    check_off
)

finite_diff = np.empty(4)

for k in range(4):
    step = np.zeros(4)
    step[k] = 1e-6

    plus = cox_objective(
        check_beta + step,
        check_z,
        check_t,
        check_e,
        0.1,
        check_off
    )[0]

    minus = cox_objective(
        check_beta - step,
        check_z,
        check_t,
        check_e,
        0.1,
        check_off
    )[0]

    finite_diff[k] = (
        (plus - minus) / 2e-6
    )

max_gradient_error = float(
    np.max(
        np.abs(analytic - finite_diff)
    )
)

if max_gradient_error > 1e-5:
    raise RuntimeError(
        f"Breslow Cox gradient check failed: "
        f"{max_gradient_error}"
    )


# Verify original training files
case_file = RAW / "case_order.csv"

lock_file = (
    ROOT / "data" / "splits"
    / "locked_four_omics_v1.csv"
)

clinical_file = (
    S21 / "stage21_training_clinical_endpoints.csv"
)

weight_file = (
    S19 / "reactome_gene_pathway_weights.npz"
)

rna_file = RAW / "rna_tpm.npy"

required = [
    case_file,
    lock_file,
    clinical_file,
    weight_file,
    rna_file,
    MODEL
]

required += [
    S21 / f"pfi_outer_fold{i}.joblib"
    for i in range(1, 5)
]

required += [
    S25 / f"fold{i}_pathway_graph.joblib"
    for i in range(1, 5)
]

required += [
    S26 / f"fold{i}_verified.joblib"
    for i in range(1, 5)
]

for path in required:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required project artifact not found: {path}"
        )


# Reconstruct the original training-only cohort
cases = pd.read_csv(
    case_file,
    dtype={"case_id": str}
)

locked = pd.read_csv(
    lock_file,
    dtype={"case_id": str}
)

clinical = pd.read_csv(
    clinical_file,
    dtype={"case_id": str}
)

if not (
    cases.case_id.is_unique
    and locked.case_id.is_unique
    and clinical.case_id.is_unique
):
    raise RuntimeError(
        "Duplicate TCGA sample identifiers."
    )

cohort = cases[
    ["case_id", "histology", "stage_group"]
].merge(
    locked[
        [
            "case_id",
            "histology",
            "stage_group",
            "split"
        ]
    ],
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_raw", "_locked")
)

if not (
    cohort.histology_raw.equals(
        cohort.histology_locked
    )
    and cohort.stage_group_raw.equals(
        cohort.stage_group_locked
    )
):
    raise RuntimeError(
        "Locked cohort labels do not match the source."
    )

train_rows = np.flatnonzero(
    cohort.split.eq("train")
)

val_rows = np.flatnonzero(
    cohort.split.eq("validation")
)

test_rows = np.flatnonzero(
    cohort.split.eq("test")
)

if (
    len(cases),
    len(train_rows),
    len(val_rows),
    len(test_rows)
) != (953, 667, 143, 143):
    raise RuntimeError(
        "Original TCGA splits changed."
    )

if not np.array_equal(
    clinical.case_id.to_numpy(),
    cohort.case_id.iloc[
        train_rows
    ].to_numpy()
):
    raise RuntimeError(
        "Stage 21 training case order does not "
        "match the locked split."
    )

eligible = np.flatnonzero(
    clinical.PFI_valid
    .astype(str)
    .str.lower()
    .eq("true")
    .to_numpy()
)

times = pd.to_numeric(
    clinical["PFI.time"],
    errors="coerce"
).to_numpy(float)

events = pd.to_numeric(
    clinical.PFI,
    errors="coerce"
).to_numpy(float)

hist = clinical.histology_matrix.eq(
    "LUSC"
).to_numpy(int)

site = clinical.site.astype(
    str
).to_numpy()

if (
    len(eligible),
    int(events[eligible].sum())
) != (657, 225):
    raise RuntimeError(
        "Training PFI eligibility or event counts changed."
    )

if (
    not np.isfinite(times[eligible]).all()
    or not np.isin(
        events[eligible],
        (0, 1)
    ).all()
    or np.any(
        times[eligible] <= 0
    )
):
    raise RuntimeError(
        "Training PFI times or event flags are invalid."
    )

if not np.isin(
    clinical.histology_matrix.iloc[eligible],
    ("LUAD", "LUSC")
).all():
    raise RuntimeError(
        "Unexpected histology in training patients."
    )


# Recreate the same Reactome RNA representation
W = sparse.load_npz(
    weight_file
).tocsr().astype(np.float32)

if W.shape != (1613, 19944):
    raise RuntimeError(
        f"Unexpected pathway membership shape: "
        f"{W.shape}"
    )

if not np.isfinite(W.data).all():
    raise RuntimeError(
        "Reactome mapping contains non-finite weights."
    )

source = np.load(
    rna_file,
    mmap_mode="r"
)

if source.shape != (953, 19944):
    raise RuntimeError(
        "RNA matrix shape changed."
    )

raw = np.asarray(
    source[train_rows],
    dtype=np.float32
).copy()

del source

valid = (
    np.isfinite(raw)
    & (raw >= 0)
)

if float(valid.mean()) < 0.95:
    raise RuntimeError(
        "Unexpected training RNA completeness."
    )

raw = np.log1p(
    np.where(valid, raw, 0)
).astype(np.float32)

del valid

rank = rankdata(
    raw,
    axis=1,
    method="average"
).astype(np.float32)

del raw

rank = (
    rank - (19944 + 1) / 2.0
) / 19944

X = np.asarray(
    (W @ rank.T).T,
    dtype=np.float32
)

del rank

if (
    not np.isfinite(X).all()
    or X.shape != (667, 1613)
):
    raise RuntimeError(
        "Reactome pathway construction failed."
    )


# Check frozen Stage 26 model identity
frozen = joblib.load(MODEL)

if (
    frozen.get("model") != "RNA_SVD8"
    or frozen.get("spec", {}).get("cnv")
    or frozen.get("spec", {}).get("hist")
):
    raise RuntimeError(
        "Frozen Stage 26 checkpoint is not "
        "the original RNA_SVD8 model."
    )

if frozen.get("training_case_ids") != (
    clinical.case_id.iloc[eligible].tolist()
):
    raise RuntimeError(
        "Frozen Stage 26 training IDs changed."
    )

files_signature = {
    str(p): {
        "size": p.stat().st_size,
        "mtime_ns": p.stat().st_mtime_ns
    }
    for p in required
}

model_sha = sha256(MODEL)

signature = hashlib.sha256(
    json.dumps(
        {
            "version": VERSION,
            "model_sha256": model_sha,
            "sources": files_signature,
            "eligible": eligible.tolist(),
            "seed": SEED,
            "sklearn_svd_dim": 8
        },
        sort_keys=True
    ).encode()
).hexdigest()

age = pd.to_numeric(
    clinical.age_years,
    errors="coerce"
).to_numpy(float)

male = pd.to_numeric(
    clinical.male,
    errors="coerce"
).to_numpy(float)

stage = pd.to_numeric(
    clinical.stage_major,
    errors="coerce"
).to_numpy(float)


# Use exactly the Stage 26 clinical feature construction
def raw_clinical(ids, median):
    a = np.where(
        np.isfinite(age[ids]),
        age[ids],
        median
    )

    return np.column_stack((
        (a - median) / 10.0,
        (~np.isfinite(age[ids])).astype(float),
        np.where(
            np.isfinite(male[ids]),
            male[ids],
            0.5
        ),
        (~np.isfinite(male[ids])).astype(float),
        hist[ids],
        (stage[ids] == 2).astype(float),
        (stage[ids] == 3).astype(float),
        (stage[ids] == 4).astype(float)
    ))


# Produce fitting-fold-only survival feature contexts
def make_context(fit, held, fold_label):
    available = age[fit][
        np.isfinite(age[fit])
    ]

    median = (
        float(np.median(available))
        if len(available)
        else 65.0
    )

    clinical_scaler = StandardScaler().fit(
        raw_clinical(fit, median)
    )

    c_fit = clinical_scaler.transform(
        raw_clinical(fit, median)
    )

    c_hold = clinical_scaler.transform(
        raw_clinical(held, median)
    )

    clinical_beta = fit_cox(
        c_fit,
        times[fit],
        events[fit],
        0.003 * 0.15
    )

    clinical_fit = c_fit @ clinical_beta
    clinical_hold = c_hold @ clinical_beta

    pathway_scaler = StandardScaler().fit(
        X[fit]
    )

    p_fit = np.clip(
        pathway_scaler.transform(X[fit]),
        -7,
        7
    ).astype(np.float32)

    p_hold = np.clip(
        pathway_scaler.transform(X[held]),
        -7,
        7
    ).astype(np.float32)

    svd = TruncatedSVD(
        n_components=8,
        n_iter=4,
        random_state=SEED
    )

    z_fit = svd.fit_transform(
        p_fit
    )

    z_hold = svd.transform(
        p_hold
    )

    latent_scaler = StandardScaler().fit(
        z_fit
    )

    z_fit = np.clip(
        latent_scaler.transform(z_fit),
        -7,
        7
    )

    z_hold = np.clip(
        latent_scaler.transform(z_hold),
        -7,
        7
    )

    molecular_beta = fit_cox(
        z_fit,
        times[fit],
        events[fit],
        0.10,
        offset=clinical_fit
    )

    risk_fit = (
        clinical_fit
        + z_fit @ molecular_beta
    )

    risk_hold = (
        clinical_hold
        + z_hold @ molecular_beta
    )

    return {
        "version": VERSION,
        "signature": signature,
        "label": fold_label,

        "fit_idx": fit.copy(),
        "held_idx": held.copy(),

        "z_fit": z_fit.astype(np.float32),
        "z_hold": z_hold.astype(np.float32),

        "clinical_fit": clinical_fit,
        "clinical_hold": clinical_hold,

        "baseline_fit": risk_fit,
        "baseline_hold": risk_hold,

        "fit_time": times[fit].copy(),
        "fit_event": events[fit].copy(),
        "fit_hist": hist[fit].copy(),
        "fit_site": site[fit].copy(),

        "held_time": times[held].copy(),
        "held_event": events[held].copy(),
        "held_hist": hist[held].copy(),
        "held_site": site[held].copy(),

        "clinical_state": {
            "scaler": clinical_scaler,
            "beta": clinical_beta,
            "age_median": median
        },

        "rna_state": {
            "scaler": pathway_scaler,
            "svd": svd,
            "latent_scaler": latent_scaler,
            "beta": molecular_beta
        }
    }


def get_context(path, fit, held, label):
    if path.exists():
        data = joblib.load(path)

        if (
            data.get("signature") != signature
            or data.get("label") != label
            or not np.array_equal(
                data["fit_idx"],
                fit
            )
            or not np.array_equal(
                data["held_idx"],
                held
            )
        ):
            raise RuntimeError(
                f"Incompatible cached feature context: {path}"
            )

        return data

    data = make_context(
        fit,
        held,
        label
    )

    atomic_joblib(data, path)
    return data


# Create site-disjoint inner folds for SPARE-Surv development
def inner_folds(fit):
    strata = (
        2 * events[fit].astype(int)
        + hist[fit]
    )

    choices = []

    for seed in (
        42, 59, 73, 89,
        101, 127, 151, 173
    ):
        try:
            trial = list(
                StratifiedGroupKFold(
                    n_splits=3,
                    shuffle=True,
                    random_state=seed
                ).split(
                    np.arange(len(fit)),
                    strata,
                    site[fit]
                )
            )
        except ValueError:
            continue

        valid = True
        imbalance = 0.0

        for a, b in trial:
            if (
                events[fit[b]].sum() < 15
                or events[fit[a]].sum() < 30
                or set(site[fit[a]])
                & set(site[fit[b]])
                or len(
                    np.unique(events[fit[b]])
                ) < 2
            ):
                valid = False
                break

            imbalance += (
                abs(
                    len(b) / len(fit) - 1 / 3
                )
                + abs(
                    events[fit[b]].mean()
                    - events[fit].mean()
                )
            )

        if valid:
            choices.append((
                imbalance,
                seed,
                [
                    (fit[a], fit[b])
                    for a, b in trial
                ]
            ))

    if not choices:
        raise RuntimeError(
            "Could not construct valid inner "
            "site-held-out folds."
        )

    return min(
        choices,
        key=lambda item: (
            item[0],
            item[1]
        )
    )[2]


# Verify frozen full-training model and matched outer folds
with threadpool_limits(limits=THREADS):
    full = get_context(
        OUT / "full_training_reference.joblib",
        eligible,
        eligible,
        "full_training"
    )

    c_state = frozen[
        "preprocessing"
    ]["clinical"]

    r_state = frozen[
        "preprocessing"
    ]["rna"]

    stored_c = (
        c_state["scaler"].transform(
            raw_clinical(
                eligible,
                float(c_state["age_median"])
            )
        )
        @ np.asarray(
            c_state["beta"],
            dtype=float
        )
    )

    stored_p = np.clip(
        r_state["scaler"].transform(
            X[eligible]
        ),
        -7,
        7
    ).astype(np.float32)

    stored_z = r_state["svd"].transform(
        stored_p
    )

    stored_z = np.clip(
        r_state["latent_scaler"].transform(
            stored_z
        ),
        -7,
        7
    )

    stored_risk = (
        stored_c
        + stored_z @ np.asarray(
            frozen["model_state"]["coef"],
            dtype=float
        )
    )

    full_error = float(
        np.max(
            np.abs(
                stored_risk
                - full["baseline_hold"]
            )
        )
    )

    if full_error > TOL:
        raise RuntimeError(
            "Frozen full-training model disagrees "
            f"with fresh fit: {full_error:.7f}"
        )

    rows = []
    inner_rows = []

    oof_c = np.full(
        667,
        np.nan
    )

    oof_m = np.full(
        667,
        np.nan
    )

    oof_z = np.full(
        (667, 8),
        np.nan
    )

    outer_held_all = []

    print("=" * 76)
    print(
        "SPARE PHASE A - FROZEN RECAP-COX "
        "REPRODUCTION AND SITE AUDIT"
    )
    print("=" * 76)

    print(
        f"Training=667, PFI eligible=657, "
        f"events=225, pathways={X.shape[1]}"
    )

    print(
        "Frozen checkpoint SHA256:",
        model_sha[:20] + "..."
    )

    print(
        "Cox gradient max numerical error:",
        f"{max_gradient_error:.2e}"
    )

    print(
        "Frozen full-training risk max error:",
        f"{full_error:.8f}"
    )

    print(
        "Stage27/Stage28 validation and "
        "test outcomes: NOT ACCESSED"
    )

    for fold in range(1, 5):
        saved21 = joblib.load(
            S21 / f"pfi_outer_fold{fold}.joblib"
        )

        saved25 = joblib.load(
            S25 / f"fold{fold}_pathway_graph.joblib"
        )

        saved26 = joblib.load(
            S26 / f"fold{fold}_verified.joblib"
        )

        held = np.asarray(
            saved21["predictions"]["index"],
            dtype=int
        )

        fit = np.setdiff1d(
            eligible,
            held
        )

        if (
            len(np.unique(held)) != len(held)
            or not np.isin(
                held,
                eligible
            ).all()
            or not np.array_equal(
                held,
                np.asarray(saved25["held"])
            )
            or not np.array_equal(
                held,
                np.asarray(saved26["held"])
            )
            or set(site[fit])
            & set(site[held])
        ):
            raise RuntimeError(
                f"Fold {fold}: patient identity "
                "or site separation failed."
            )

        ctx = get_context(
            OUT / f"outer_fold{fold}_features.joblib",
            fit,
            held,
            f"outer_{fold}"
        )

        ref_c21 = np.asarray(
            saved21["predictions"]["clinical_risk"],
            dtype=float
        )

        ref_c26 = np.asarray(
            saved26["predictions"]["clinical"],
            dtype=float
        )

        ref_m25 = np.asarray(
            saved25["predictions"]["svd8"],
            dtype=float
        )

        ref_m26 = np.asarray(
            saved26["predictions"]["rna_svd8"],
            dtype=float
        )

        errors = {
            "clinical_21": float(
                np.max(
                    np.abs(
                        ctx["clinical_hold"]
                        - ref_c21
                    )
                )
            ),

            "clinical_26": float(
                np.max(
                    np.abs(
                        ctx["clinical_hold"]
                        - ref_c26
                    )
                )
            ),

            "rna_svd8_25": float(
                np.max(
                    np.abs(
                        ctx["baseline_hold"]
                        - ref_m25
                    )
                )
            ),

            "rna_svd8_26": float(
                np.max(
                    np.abs(
                        ctx["baseline_hold"]
                        - ref_m26
                    )
                )
            )
        }

        if max(errors.values()) > TOL:
            raise RuntimeError(
                f"Fold {fold}: historical baseline "
                f"failed reproduction: {errors}"
            )

        held_c = cindex(
            times[held],
            events[held],
            ctx["clinical_hold"]
        )

        held_m = cindex(
            times[held],
            events[held],
            ctx["baseline_hold"]
        )

        rows.append({
            "fold": fold,
            "n_fit": len(fit),
            "n_held": len(held),
            "held_events": int(
                events[held].sum()
            ),
            "fit_sites": len(
                set(site[fit])
            ),
            "held_sites": len(
                set(site[held])
            ),
            "clinical_C": held_c,
            "recap_svd8_C": held_m,
            "delta_C": held_m - held_c,
            "max_reproduction_error": (
                max(errors.values())
            )
        })

        oof_c[held] = ctx["clinical_hold"]
        oof_m[held] = ctx["baseline_hold"]
        oof_z[held] = ctx["z_hold"]

        outer_held_all.append(held)

        print(
            f"Outer {fold}: clinical={held_c:.4f}, "
            f"ReCAP={held_m:.4f}, "
            f"delta={held_m-held_c:+.4f}, "
            f"max_error={max(errors.values()):.2e}",
            flush=True
        )

        for j, (
            inner_fit,
            inner_held
        ) in enumerate(
            inner_folds(fit),
            1
        ):
            if (
                set(site[inner_fit])
                & set(site[inner_held])
            ):
                raise RuntimeError(
                    f"Fold {fold}/{j}: inner-site overlap."
                )

            inner_ctx = get_context(
                OUT / (
                    f"outer_fold{fold}_"
                    f"inner{j}_features.joblib"
                ),
                inner_fit,
                inner_held,
                f"outer_{fold}_inner_{j}"
            )

            inner_rows.append({
                "outer_fold": fold,
                "inner_fold": j,
                "n_fit": len(inner_fit),
                "n_held": len(inner_held),
                "fit_events": int(
                    events[inner_fit].sum()
                ),
                "held_events": int(
                    events[inner_held].sum()
                ),
                "fit_sites": len(
                    set(site[inner_fit])
                ),
                "held_sites": len(
                    set(site[inner_held])
                ),
                "recap_inner_C": cindex(
                    times[inner_held],
                    events[inner_held],
                    inner_ctx["baseline_hold"]
                )
            })

    if not np.array_equal(
        np.sort(
            np.concatenate(
                outer_held_all
            )
        ),
        np.sort(eligible)
    ):
        raise RuntimeError(
            "Outer folds do not partition "
            "the entire PFI cohort."
        )

    if (
        not np.isfinite(
            oof_m[eligible]
        ).all()
        or not np.isfinite(
            oof_z[eligible]
        ).all()
    ):
        raise RuntimeError(
            "Incomplete ReCAP-Cox OOF reconstruction."
        )

    table = pd.DataFrame(
        rows
    ).sort_values("fold")

    inner_table = pd.DataFrame(
        inner_rows
    )

    mean_clinical = float(
        table.clinical_C.mean()
    )

    mean_recap = float(
        table.recap_svd8_C.mean()
    )

    if (
        abs(mean_recap - 0.6549) > 0.0006
        or abs(mean_clinical - 0.6317) > 0.0006
    ):
        raise RuntimeError(
            f"Benchmark drift: clinical "
            f"{mean_clinical:.6f}, "
            f"ReCAP {mean_recap:.6f}"
        )


# Assess site/histology overlap before site perturbations
site_frame = pd.DataFrame({
    "site": site[eligible],
    "histology": (
        clinical.histology_matrix
        .iloc[eligible]
        .to_numpy()
    ),
    "event": events[
        eligible
    ].astype(int),
    "stage": stage[eligible]
})

site_summary = site_frame.groupby(
    "site",
    as_index=False
).agg(
    patients=("event", "size"),
    events=("event", "sum"),
    fraction_LUSC=(
        "histology",
        lambda s: float(
            (s == "LUSC").mean()
        )
    ),
    fraction_stage_III_IV=(
        "stage",
        lambda s: float(
            (s >= 3).mean()
        )
    )
)

site_summary["single_histology"] = (
    site_summary.fraction_LUSC.isin(
        (0.0, 1.0)
    )
)

site_hist = (
    site_frame.groupby(
        ["site", "histology"]
    )
    .size()
    .reset_index(name="n")
)

site_pairs = {}

for subtype in ("LUAD", "LUSC"):
    matching_sites = int(
        (
            (site_hist.histology == subtype)
            & (site_hist.n >= 5)
        ).sum()
    )

    site_pairs[subtype] = {
        "sites_with_at_least_5_patients": (
            matching_sites
        ),
        "possible_site_pairs": (
            matching_sites
            * (matching_sites - 1)
            // 2
        )
    }


# Describe the frozen SVD component loadings
pathway_meta_file = (
    S19 / "reactome_pathway_reference.csv"
)

if pathway_meta_file.exists():
    pathways = pd.read_csv(
        pathway_meta_file
    )

    pathway_name_column = next(
        (
            c
            for c in (
                "pathway_name",
                "name",
                "display_name",
                "pathway",
                "term"
            )
            if c in pathways.columns
        ),
        None
    )

    if len(pathways) != X.shape[1]:
        raise RuntimeError(
            "Reactome pathway metadata rows "
            "do not match scoring matrix."
        )
else:
    pathways = None
    pathway_name_column = None

loading_rows = []

components = np.asarray(
    frozen["preprocessing"]["rna"]["svd"].components_
)

if components.shape != (8, 1613):
    raise RuntimeError(
        "Frozen SVD components are not 8 x 1613."
    )

for component_id in range(8):
    indices = np.argsort(
        -np.abs(
            components[component_id]
        )
    )[:10]

    for rank_id, pathway_id in enumerate(
        indices,
        1
    ):
        name = (
            str(
                pathways.iloc[pathway_id][
                    pathway_name_column
                ]
            )
            if pathway_name_column
            else f"Pathway_{pathway_id}"
        )

        loading_rows.append({
            "component": component_id + 1,
            "rank": rank_id,
            "pathway_index": int(
                pathway_id
            ),
            "pathway_name": name,
            "signed_loading": float(
                components[
                    component_id,
                    pathway_id
                ]
            )
        })


# Save verified training-only artifacts
table.to_csv(
    OUT / "recap_reproduced_outer_folds.csv",
    index=False
)

inner_table.to_csv(
    OUT / "recap_inner_folds.csv",
    index=False
)

site_summary.to_csv(
    OUT / "training_site_histology_audit.csv",
    index=False
)

pd.DataFrame(
    loading_rows
).to_csv(
    OUT / "svd8_top_pathway_loadings.csv",
    index=False
)

pd.DataFrame({
    "case_id": (
        clinical.case_id
        .iloc[eligible]
        .to_numpy()
    ),
    "site": site[eligible],
    "histology": (
        clinical.histology_matrix
        .iloc[eligible]
        .to_numpy()
    ),
    "PFI_time": times[eligible],
    "PFI_event": (
        events[eligible].astype(int)
    ),
    "clinical_risk": oof_c[eligible],
    "recap_risk": oof_m[eligible],
    **{
        f"svd_component_{j+1}": (
            oof_z[eligible, j]
        )
        for j in range(8)
    }
}).to_csv(
    OUT / "training_only_oof_and_svd8.csv",
    index=False
)


summary = {
    "version": VERSION,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "cache_signature": signature,
    "stage26_model_sha256": model_sha,

    "training_eligible": int(
        len(eligible)
    ),
    "training_events": int(
        events[eligible].sum()
    ),

    "clinical_mean_site_C": mean_clinical,
    "recap_mean_site_C": mean_recap,
    "mean_delta_C": float(
        table.delta_C.mean()
    ),

    "maximum_reference_risk_error": float(
        table.max_reproduction_error.max()
    ),

    "frozen_full_training_max_error": (
        full_error
    ),

    "cox_gradient_max_error": (
        max_gradient_error
    ),

    "training_sites": int(
        len(site_summary)
    ),

    "single_histology_sites": int(
        site_summary.single_histology.sum()
    ),

    "sites_with_at_least_5_patients": int(
        (site_summary.patients >= 5).sum()
    ),

    "site_histology_matched_pairs": (
        site_pairs
    ),

    "outer_contexts": 4,
    "inner_contexts": int(
        len(inner_table)
    ),

    "next_step": (
        "SPARE-Surv low-rank residual architecture "
        "with training-only nested site CV"
    ),

    "restrictions": [
        "All scalers, SVDs, and Cox parameters fit on context-specific fitting patients only.",
        "The outer and inner held-out patients were not used in fitted transformations.",
        "No Stage27/Stage28 validation or test outcomes were accessed.",
        "Stage27/Stage28 are no longer untouched evaluation cohorts for architecture development.",
        "Old site CV results are exploratory because prior architecture development used the same folds.",
        "The final external cohort remains one prespecified GSE50081 dataset, pending endpoint audit."
    ]
}

atomic_json(
    summary,
    OUT / "phase_a_manifest.json"
)


# Final audit report
print("\n" + "=" * 76)

print(
    "SPARE PHASE A FINAL - "
    "RECAP BENCHMARK VERIFIED"
)

print("=" * 76)

print(
    table.to_string(
        index=False,
        float_format=lambda v: f"{v:.4f}"
    )
)

print(
    f"Clinical Cox mean C: {mean_clinical:.4f}"
)

print(
    f"ReCAP-Cox mean C:    {mean_recap:.4f}"
)

print(
    "Mean improvement:   ",
    f"{mean_recap-mean_clinical:+.4f}"
)

print(
    f"Training sites: {len(site_summary)}; "
    "single-histology sites: "
    f"{int(site_summary.single_histology.sum())}"
)

print(
    "Histology-matched site perturbation feasibility:",
    site_pairs
)

print(
    f"Saved 4 outer and {len(inner_table)} "
    f"inner feature contexts: {OUT}"
)

print(
    "Validation/test outcomes read: NO"
)

print(
    "SPARE PHASE A COMPLETED"
)

SPARE PHASE A - FROZEN RECAP-COX REPRODUCTION AND SITE AUDIT
Training=667, PFI eligible=657, events=225, pathways=1613
Frozen checkpoint SHA256: 511fdab1c6487313d98d...
Cox gradient max numerical error: 3.55e-10
Frozen full-training risk max error: 0.00000000
Stage27/Stage28 validation and test outcomes: NOT ACCESSED
Outer 1: clinical=0.6489, ReCAP=0.6720, delta=+0.0231, max_error=4.44e-16
Outer 2: clinical=0.6421, ReCAP=0.6596, delta=+0.0174, max_error=2.22e-16
Outer 3: clinical=0.6660, ReCAP=0.7075, delta=+0.0415, max_error=7.28e-14
Outer 4: clinical=0.5696, ReCAP=0.5804, delta=+0.0108, max_error=8.06e-11

SPARE PHASE A FINAL - RECAP BENCHMARK VERIFIED
 fold  n_fit  n_held  held_events  fit_sites  held_sites  clinical_C  recap_svd8_C  delta_C  max_reproduction_error
    1    491     166           56         48          14      0.6489        0.6720   0.0231                  0.0000
    2    494     163           55         47          15      0.6421        0.6596   0.0174              

In [29]:

import hashlib
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from scipy.linalg import solve


# SPARE-Surv Phase B: prespecified residual architecture comparison.
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
A = ROOT / "results" / "metrics" / "spare_phase_a"
S21 = ROOT / "results" / "metrics" / "stage21"
OUT = ROOT / "results" / "metrics" / "spare_phase_b"
EXPECTED_PYTHON = ROOT / "envs" / "nsclc" / "python.exe"

VERSION = "spare_phase_b_frozen_recap_residual_v2"
SEEDS = (41, 137)
EPOCHS = (20, 50, 80)
GAMMAS = (0.5, 1.0)
ARCHS = ("mlp_residual", "lowrank_interaction", "spare_full")

MIN_INNER_GAIN = 0.005
MIN_POSITIVE_INNER_FOLDS = 2

RANK = 4
RHO = 0.75
LR = 0.002
WEIGHT_DECAY = 0.01
DELTA_PENALTY = 0.04
SITE_PENALTY = 0.20
PERTURB_SCALE = 0.35
PERTURB_LIMIT = 0.45
THREADS = min(4, os.cpu_count() or 1)

if Path(sys.executable).resolve() != EXPECTED_PYTHON.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

OUT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(THREADS)


# Reproducibility utilities
def sha256(path):
    digest = hashlib.sha256()

    with open(path, "rb") as handle:
        for chunk in iter(
            lambda: handle.read(4 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()


def atomic_joblib(value, path):
    temp = path.with_name(path.name + ".tmp")
    joblib.dump(value, temp, compress=3)
    os.replace(temp, path)


def atomic_json(value, path):
    temp = path.with_name(path.name + ".tmp")

    temp.write_text(
        json.dumps(value, indent=2, allow_nan=False),
        encoding="utf-8"
    )

    os.replace(temp, path)


# Harrell concordance with strict time ordering
def cindex(t, e, risk):
    t = np.asarray(t, dtype=float)
    e = np.asarray(e, dtype=int)
    risk = np.asarray(risk, dtype=float)

    if (
        not (len(t) == len(e) == len(risk))
        or not np.isfinite(risk).all()
    ):
        raise RuntimeError("Invalid concordance inputs.")

    comparable = (
        (t[:, None] < t[None, :])
        & (e[:, None] == 1)
    )

    pairs = int(comparable.sum())

    if pairs < 10:
        raise RuntimeError(
            "Insufficient comparable survival pairs."
        )

    delta = risk[:, None] - risk[None, :]

    wins = np.count_nonzero(
        comparable & (delta > 1e-10)
    )

    ties = np.count_nonzero(
        comparable & (np.abs(delta) <= 1e-10)
    )

    return float(
        (wins + 0.5 * ties) / pairs
    )


# Full-risk-set Breslow Cox partial likelihood
def cox_breslow(risk, t, e):
    order = torch.argsort(
        t,
        descending=True,
        stable=True
    )

    ts = t[order]
    es = e[order]
    rs = risk[order]

    last = torch.searchsorted(
        -ts.contiguous(),
        -ts.contiguous(),
        right=True
    ) - 1

    log_riskset = torch.logcumsumexp(
        rs,
        dim=0
    )[last]

    return (
        torch.sum(
            es * (log_riskset - rs)
        )
        / torch.clamp(es.sum(), min=1.0)
    )


# Compact clinical-anchored residual architecture
class SPARE(nn.Module):
    def __init__(self, interaction):
        super().__init__()

        self.interaction = interaction

        self.main = nn.Linear(8, 8)
        self.gate = nn.Linear(10, 1)

        if interaction:
            self.u = nn.Linear(
                8, RANK, bias=False
            )
            self.v = nn.Linear(
                8, RANK, bias=False
            )

        output_dim = (
            8 + RANK if interaction else 8
        )

        self.output = nn.Linear(
            output_dim, 1
        )

        # The initial model exactly reproduces ReCAP-Cox.
        nn.init.zeros_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, z, hist):
        hidden = torch.tanh(
            self.main(z)
        )

        hist_onehot = F.one_hot(
            hist,
            num_classes=2
        ).to(z.dtype)

        gate = torch.sigmoid(
            self.gate(
                torch.cat(
                    [z, hist_onehot],
                    dim=1
                )
            )
        )

        if self.interaction:
            interaction = (
                self.u(z) * self.v(z)
            ) / np.sqrt(RANK)

            hidden = torch.cat(
                [hidden, interaction],
                dim=1
            )

        return (
            RHO
            * gate.squeeze(1)
            * torch.tanh(
                self.output(hidden).squeeze(1)
            )
        )


# Generate clinically adjusted, same-histology site shifts
def make_shift_bank(ctx, clinical, seed, n_draws=4):
    z = np.asarray(
        ctx["z_fit"],
        dtype=float
    )

    sites = np.asarray(
        ctx["fit_site"],
        dtype=str
    )

    hist = np.asarray(
        ctx["fit_hist"],
        dtype=int
    )

    index = np.asarray(
        ctx["fit_idx"],
        dtype=int
    )

    stage = pd.to_numeric(
        clinical.stage_major,
        errors="coerce"
    ).to_numpy(float)[index]

    age = pd.to_numeric(
        clinical.age_years,
        errors="coerce"
    ).to_numpy(float)[index]

    male = pd.to_numeric(
        clinical.male,
        errors="coerce"
    ).to_numpy(float)[index]

    age_median = np.nanmedian(age)

    design = np.column_stack([
        np.ones(len(index)),
        hist,
        (stage == 2).astype(float),
        (stage == 3).astype(float),
        (stage == 4).astype(float),
        np.where(
            np.isfinite(age),
            (age - age_median) / 10.0,
            0
        ),
        np.where(
            np.isfinite(male),
            male,
            0.5
        ),
        (~np.isfinite(age)).astype(float),
        (~np.isfinite(male)).astype(float)
    ])

    ridge = (
        np.eye(design.shape[1]) * 0.01
    )

    ridge[0, 0] = 0

    coefficients = solve(
        design.T @ design + ridge,
        design.T @ z,
        assume_a="pos"
    )

    unexplained = (
        z - design @ coefficients
    )

    site_data = {}

    for site_id in np.unique(sites):
        mask = sites == site_id

        if len(np.unique(hist[mask])) != 1:
            raise RuntimeError(
                "A training site contains both histologies; "
                "review perturbation design."
            )

        if mask.sum() >= 5:
            site_data[site_id] = (
                int(hist[mask][0]),
                unexplained[mask].mean(axis=0)
            )

    options = {}

    for source_site, (h, _) in site_data.items():
        options[source_site] = [
            target_site
            for target_site, (target_hist, _) in site_data.items()
            if (
                target_site != source_site
                and target_hist == h
            )
        ]

    if any(
        not targets
        for targets in options.values()
    ):
        raise RuntimeError(
            "Insufficient same-histology sites "
            "for perturbation."
        )

    rng = np.random.default_rng(seed)

    participant = np.asarray([
        site_id in options
        for site_id in sites
    ], dtype=bool)

    if participant.mean() < 0.50:
        raise RuntimeError(
            "Too few patients belong to "
            "reliable perturbation sites."
        )

    bank = []

    for _ in range(n_draws):
        shift = np.zeros_like(
            z,
            dtype=np.float32
        )

        for source_site, targets in options.items():
            target_site = targets[
                rng.integers(len(targets))
            ]

            direction = (
                site_data[target_site][1]
                - site_data[source_site][1]
            )

            direction = np.clip(
                PERTURB_SCALE * direction,
                -PERTURB_LIMIT,
                PERTURB_LIMIT
            )

            shift[sites == source_site] = (
                direction
            )

        bank.append(
            torch.as_tensor(
                shift,
                dtype=torch.float32
            )
        )

    diagnostics = {
        "paired_patient_fraction": float(
            participant.mean()
        ),
        "shift_rms": float(
            np.sqrt(
                np.mean(
                    bank[0].numpy()[
                        participant
                    ] ** 2
                )
            )
        ),
        "eligible_sites": int(
            len(site_data)
        )
    }

    return (
        bank,
        torch.as_tensor(
            participant,
            dtype=torch.float32
        ),
        diagnostics
    )


# Train one residual branch with fixed Cox anchor
def train_branch(
    ctx,
    arch,
    seed,
    clinical,
    needed_epochs=EPOCHS
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    z = torch.as_tensor(
        ctx["z_fit"],
        dtype=torch.float32
    )

    h = torch.as_tensor(
        ctx["fit_hist"],
        dtype=torch.long
    )

    zh = torch.as_tensor(
        ctx["z_hold"],
        dtype=torch.float32
    )

    hh = torch.as_tensor(
        ctx["held_hist"],
        dtype=torch.long
    )

    baseline = torch.as_tensor(
        ctx["baseline_fit"],
        dtype=torch.float32
    )

    t = torch.as_tensor(
        ctx["fit_time"],
        dtype=torch.float32
    )

    e = torch.as_tensor(
        ctx["fit_event"],
        dtype=torch.float32
    )

    if (
        len(z) != len(h)
        or len(z) != len(baseline)
        or int(e.sum()) < 30
    ):
        raise RuntimeError(
            "Invalid residual training data."
        )

    model = SPARE(
        interaction=arch != "mlp_residual"
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    if arch == "spare_full":
        shifts, coverage, diagnostics = (
            make_shift_bank(
                ctx,
                clinical,
                seed
            )
        )
    else:
        shifts = ()
        coverage = None
        diagnostics = None

    snapshots = {}
    states = {}

    for epoch in range(
        1,
        max(needed_epochs) + 1
    ):
        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        residual = model(z, h)

        objective = (
            cox_breslow(
                baseline + residual,
                t,
                e
            )
            + DELTA_PENALTY
            * residual.square().mean()
        )

        if arch == "spare_full":
            shift = shifts[
                (epoch - 1) % len(shifts)
            ]

            perturbed = torch.clamp(
                z + shift,
                -7,
                7
            )

            alternate = model(
                perturbed,
                h
            )

            difference = (
                residual - alternate
            ).square()

            consistency = (
                (difference * coverage).sum()
                / coverage.sum().clamp_min(1)
            )

            objective = (
                objective
                + SITE_PENALTY * consistency
            )

        if not torch.isfinite(objective):
            raise RuntimeError(
                f"Nonfinite neural Cox objective: "
                f"{arch}, seed={seed}, epoch={epoch}"
            )

        objective.backward()

        nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=2.0
        )

        optimizer.step()

        if epoch in needed_epochs:
            model.eval()

            with torch.no_grad():
                output = model(
                    zh,
                    hh
                ).numpy().astype(float)

            if not np.isfinite(output).all():
                raise RuntimeError(
                    "Nonfinite residual predictions."
                )

            snapshots[epoch] = output

            states[epoch] = {
                name: tensor.detach()
                .cpu()
                .clone()
                for name, tensor
                in model.state_dict().items()
            }

    return (
        snapshots,
        states,
        diagnostics
    )


# Verify exact baseline initialization and Cox gradients
torch.manual_seed(11)

unit = SPARE(interaction=True)

with torch.no_grad():
    zero = unit(
        torch.randn(9, 8),
        torch.randint(0, 2, (9,))
    ).numpy()

if np.max(np.abs(zero)) != 0:
    raise RuntimeError(
        "Residual model does not initialize "
        "at the exact frozen baseline."
    )

ri = torch.tensor(
    [0.1, -0.2, 0.3, 0.6, -0.3, 0.4],
    dtype=torch.float64,
    requires_grad=True
)

ti = torch.tensor(
    [9, 9, 6, 5, 5, 2],
    dtype=torch.float64
)

ei = torch.tensor(
    [1, 0, 1, 1, 1, 1],
    dtype=torch.float64
)

loss = cox_breslow(
    ri, ti, ei
)

loss.backward()

finite = []

for k in range(len(ri)):
    shift = torch.zeros_like(ri)
    shift[k] = 1e-6

    plus = cox_breslow(
        ri + shift, ti, ei
    )

    minus = cox_breslow(
        ri - shift, ti, ei
    )

    finite.append(
        ((plus - minus) / 2e-6).item()
    )

gradient_error = float(
    np.max(
        np.abs(
            np.asarray(finite)
            - ri.grad.detach().numpy()
        )
    )
)

if gradient_error > 1e-6:
    raise RuntimeError(
        f"Censored Cox gradient check failed: "
        f"{gradient_error:.2e}"
    )


# Load verified Phase A results and immutable inputs
manifest_file = A / "phase_a_manifest.json"

clinical_file = (
    S21 / "stage21_training_clinical_endpoints.csv"
)

if (
    not manifest_file.exists()
    or not clinical_file.exists()
):
    raise FileNotFoundError(
        "SPARE Phase A manifest or "
        "Stage 21 clinical data unavailable."
    )

manifest = json.loads(
    manifest_file.read_text(
        encoding="utf-8"
    )
)

clinical = pd.read_csv(
    clinical_file,
    dtype={"case_id": str}
)

if (
    len(clinical) != 667
    or abs(
        manifest["recap_mean_site_C"]
        - 0.6549
    ) > 0.0006
):
    raise RuntimeError(
        "Verified Phase A baseline "
        "is missing or changed."
    )

frozen_model = (
    ROOT
    / "models"
    / "checkpoints"
    / "stage26"
    / "stage26_training_only_final.joblib"
)

if (
    manifest["stage26_model_sha256"]
    != sha256(frozen_model)
):
    raise RuntimeError(
        "Frozen ReCAP-Cox checkpoint changed."
    )


# Validate outer/inner feature contexts
contexts = {}
needed_paths = []

for fold in range(1, 5):
    needed_paths.append(
        A / f"outer_fold{fold}_features.joblib"
    )

    for inner in range(1, 4):
        needed_paths.append(
            A
            / f"outer_fold{fold}_inner{inner}_features.joblib"
        )

required_fields = (
    "z_fit", "z_hold",
    "baseline_fit", "baseline_hold",
    "fit_idx", "held_idx",
    "fit_time", "held_time",
    "fit_event", "held_event",
    "fit_hist", "held_hist",
    "fit_site", "held_site"
)

for path in needed_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing verified Phase A context: {path}"
        )

    ctx = joblib.load(path)

    if (
        ctx.get("signature")
        != manifest["cache_signature"]
    ):
        raise RuntimeError(
            f"Phase A context fingerprint mismatch: {path}"
        )

    for field in required_fields:
        if field not in ctx:
            raise RuntimeError(
                f"Missing context field {field}: {path}"
            )

    if (
        np.asarray(ctx["z_fit"]).shape
        != (len(ctx["fit_idx"]), 8)
        or np.asarray(ctx["z_hold"]).shape
        != (len(ctx["held_idx"]), 8)
        or not set(ctx["fit_site"]).isdisjoint(
            ctx["held_site"]
        )
    ):
        raise RuntimeError(
            f"Invalid site-held-out context: {path}"
        )

    contexts[path.name] = ctx


# Fingerprint all model-selection choices
settings = {
    "version": VERSION,
    "phase_a_signature": manifest["cache_signature"],
    "checkpoint_sha256": (
        manifest["stage26_model_sha256"]
    ),
    "seeds": SEEDS,
    "epochs": EPOCHS,
    "gammas": GAMMAS,
    "architectures": ARCHS,
    "min_inner_gain": MIN_INNER_GAIN,
    "min_positive_inner_folds": MIN_POSITIVE_INNER_FOLDS,
    "rank": RANK,
    "rho": RHO,
    "learning_rate": LR,
    "weight_decay": WEIGHT_DECAY,
    "delta_penalty": DELTA_PENALTY,
    "site_penalty": SITE_PENALTY,
    "perturb_scale": PERTURB_SCALE,
    "perturb_limit": PERTURB_LIMIT,
    "clinical_file_sha256": sha256(clinical_file)
}

signature = hashlib.sha256(
    json.dumps(
        settings,
        sort_keys=True
    ).encode()
).hexdigest()


print("=" * 78)
print(
    "SPARE PHASE B - PRESPECIFIED RESIDUAL "
    "ARCHITECTURE COMPARISON"
)
print("=" * 78)

print(
    "ReCAP-Cox frozen training-CV benchmark:",
    f"{manifest['recap_mean_site_C']:.4f}"
)

print(
    "Single-histology training sites:",
    f"{manifest['single_histology_sites']}"
    f"/{manifest['training_sites']}"
)

print(
    f"Models: {ARCHS}; "
    f"seeds={SEEDS}; checkpoints={EPOCHS}"
)

print(
    "Censored Cox gradient check:",
    f"{gradient_error:.2e}"
)

print("Exact anchor recovery: PASS")

print(
    "TCGA validation/test and GSE50081 "
    "outcomes: NOT ACCESSED"
)


def context(name):
    return contexts[name]


# Train two deterministic seeds per architecture
def ensemble_predictions(
    ctx,
    arch,
    fold,
    inner_label,
    needed_epochs=EPOCHS
):
    predictions = {
        epoch: []
        for epoch in needed_epochs
    }

    states_by_seed = []
    diagnostics = []

    for j, seed in enumerate(SEEDS):
        actual_seed = (
            seed
            + fold * 1000
            + inner_label * 100
            + j * 7
        )

        predictions_seed, state, diag = (
            train_branch(
                ctx,
                arch,
                actual_seed,
                clinical,
                needed_epochs=needed_epochs
            )
        )

        for epoch in needed_epochs:
            predictions[epoch].append(
                predictions_seed[epoch]
            )

        states_by_seed.append(state)

        if diag is not None:
            diagnostics.append(diag)

    ensemble = {
        epoch: np.mean(
            values,
            axis=0
        )
        for epoch, values in predictions.items()
    }

    return (
        ensemble,
        states_by_seed,
        diagnostics
    )


# Nested training-only model selection
fold_rows = []
inner_rows = []

all_oof = {
    name: np.full(667, np.nan)
    for name in (
        "recap",
        "selected",
        *ARCHS
    )
}

for outer in range(1, 5):
    checkpoint = (
        OUT / f"outer_fold{outer}_evaluation.joblib"
    )

    ctx_outer = context(
        f"outer_fold{outer}_features.joblib"
    )

    held_outer = np.asarray(
        ctx_outer["held_idx"],
        dtype=int
    )

    if checkpoint.exists():
        prior = joblib.load(checkpoint)

        if (
            prior.get("signature") != signature
            or not np.array_equal(
                prior["held"],
                held_outer
            )
        ):
            raise RuntimeError(
                f"Incompatible Phase B checkpoint: "
                f"{checkpoint}"
            )

        row = prior["row"]
        inner_saved = prior["inner"]
        pred = prior["predictions"]

        print(
            f"Outer {outer}: verified checkpoint reused",
            flush=True
        )

    else:
        inner_contexts = [
            context(
                f"outer_fold{outer}_inner{j}_features.joblib"
            )
            for j in range(1, 4)
        ]

        reference_inner = np.asarray([
            cindex(
                ctx["held_time"],
                ctx["held_event"],
                ctx["baseline_hold"]
            )
            for ctx in inner_contexts
        ])

        scores = {
            (arch, epoch, gamma): []
            for arch in ARCHS
            for epoch in EPOCHS
            for gamma in GAMMAS
        }

        inner_saved = []

        print(
            f"\nOUTER {outer}: "
            f"train={len(ctx_outer['fit_idx'])}, "
            f"held={len(held_outer)}, "
            f"events={int(np.sum(ctx_outer['held_event']))}",
            flush=True
        )

        for inner_no, ctx in enumerate(
            inner_contexts,
            1
        ):
            for arch in ARCHS:
                snapshots, _, diagnostics = (
                    ensemble_predictions(
                        ctx,
                        arch,
                        outer,
                        inner_no
                    )
                )

                for epoch, residual in snapshots.items():
                    for gamma in GAMMAS:
                        risk = (
                            np.asarray(
                                ctx["baseline_hold"]
                            )
                            + gamma * residual
                        )

                        score = cindex(
                            ctx["held_time"],
                            ctx["held_event"],
                            risk
                        )

                        key = (
                            arch,
                            epoch,
                            gamma
                        )

                        scores[key].append(score)

                        inner_saved.append({
                            "outer": outer,
                            "inner": inner_no,
                            "architecture": arch,
                            "epoch": epoch,
                            "gamma": gamma,
                            "recap_C": float(
                                reference_inner[
                                    inner_no - 1
                                ]
                            ),
                            "candidate_C": score,
                            "gain": (
                                score
                                - reference_inner[
                                    inner_no - 1
                                ]
                            )
                        })

                if diagnostics:
                    coverage = np.mean([
                        d["paired_patient_fraction"]
                        for d in diagnostics
                    ])

                    print(
                        f"  inner {inner_no}, "
                        f"site-shift coverage={coverage:.1%}",
                        flush=True
                    )

        candidate_ranks = []

        for (
            arch,
            epoch,
            gamma
        ), values in scores.items():
            gains = (
                np.asarray(values)
                - reference_inner
            )

            candidate_ranks.append({
                "arch": arch,
                "epoch": epoch,
                "gamma": gamma,
                "inner_mean_C": float(
                    np.mean(values)
                ),
                "inner_delta": float(
                    np.mean(gains)
                ),
                "positive_inner": int(
                    np.sum(gains > 0)
                )
            })

        priority = {
            "mlp_residual": 0,
            "lowrank_interaction": 1,
            "spare_full": 2
        }

        def sort_key(item):
            return (
                -item["inner_mean_C"],
                priority[item["arch"]],
                item["epoch"],
                item["gamma"]
            )

        best_by_arch = {
            arch: sorted(
                (
                    item
                    for item in candidate_ranks
                    if item["arch"] == arch
                ),
                key=sort_key
            )[0]
            for arch in ARCHS
        }

        acceptable = [
            item
            for item in candidate_ranks
            if (
                item["inner_delta"]
                >= MIN_INNER_GAIN
                and item["positive_inner"]
                >= MIN_POSITIVE_INNER_FOLDS
            )
        ]

        winner = (
            sorted(
                acceptable,
                key=sort_key
            )[0]
            if acceptable
            else None
        )

        print(
            "  inner gains:",
            {
                arch: round(
                    best_by_arch[arch]["inner_delta"],
                    4
                )
                for arch in ARCHS
            },
            flush=True
        )

        print(
            "  inner selection:",
            (
                winner["arch"]
                if winner is not None
                else "ReCAP baseline"
            ),
            flush=True
        )

        # Refit outer-training branches at fixed inner choices.
        pred = {
            "recap": np.asarray(
                ctx_outer["baseline_hold"],
                dtype=float
            )
        }

        trained_models = {}
        raw_snapshots = {}

        for arch in ARCHS:
            snapshots, states, diagnostics = (
                ensemble_predictions(
                    ctx_outer,
                    arch,
                    outer,
                    9
                )
            )

            raw_snapshots[arch] = snapshots
            best = best_by_arch[arch]

            pred[arch] = (
                pred["recap"]
                + best["gamma"]
                * snapshots[best["epoch"]]
            )

            trained_models[arch] = {
                "epoch": best["epoch"],
                "gamma": best["gamma"],
                "states": [
                    state[best["epoch"]]
                    for state in states
                ],
                "parameters": sum(
                    p.numel()
                    for p in SPARE(
                        arch != "mlp_residual"
                    ).parameters()
                ),
                "site_shift_diagnostics": diagnostics,
                "inner": best
            }

        # Evaluate the exact configuration chosen by inner CV.
        pred["selected"] = (
            pred["recap"]
            if winner is None
            else (
                pred["recap"]
                + winner["gamma"]
                * raw_snapshots[
                    winner["arch"]
                ][winner["epoch"]]
            )
        )

        # Matched epoch/gamma comparisons isolate modules.
        full_setting = best_by_arch[
            "spare_full"
        ]

        interaction_setting = best_by_arch[
            "lowrank_interaction"
        ]

        matched_full_risk = (
            pred["recap"]
            + full_setting["gamma"]
            * raw_snapshots[
                "spare_full"
            ][full_setting["epoch"]]
        )

        matched_no_site_risk = (
            pred["recap"]
            + full_setting["gamma"]
            * raw_snapshots[
                "lowrank_interaction"
            ][full_setting["epoch"]]
        )

        matched_with_interaction = (
            pred["recap"]
            + interaction_setting["gamma"]
            * raw_snapshots[
                "lowrank_interaction"
            ][interaction_setting["epoch"]]
        )

        matched_without_interaction = (
            pred["recap"]
            + interaction_setting["gamma"]
            * raw_snapshots[
                "mlp_residual"
            ][interaction_setting["epoch"]]
        )

        row = {
            "fold": outer,
            "held": len(held_outer),
            "events": int(
                np.sum(ctx_outer["held_event"])
            ),
            "selected_architecture": (
                winner["arch"]
                if winner
                else "ReCAP"
            ),
            "selected_epoch": (
                winner["epoch"]
                if winner
                else 0
            ),
            "selected_gamma": (
                winner["gamma"]
                if winner
                else 0.0
            ),
            "inner_selected_gain": (
                winner["inner_delta"]
                if winner
                else 0.0
            )
        }

        for name, risk in pred.items():
            row[name + "_C"] = cindex(
                ctx_outer["held_time"],
                ctx_outer["held_event"],
                risk
            )

        row["selected_minus_recap"] = (
            row["selected_C"]
            - row["recap_C"]
        )

        row["full_minus_interaction"] = (
            row["spare_full_C"]
            - row["lowrank_interaction_C"]
        )

        row["matched_site_module_delta"] = (
            cindex(
                ctx_outer["held_time"],
                ctx_outer["held_event"],
                matched_full_risk
            )
            - cindex(
                ctx_outer["held_time"],
                ctx_outer["held_event"],
                matched_no_site_risk
            )
        )

        row["matched_interaction_module_delta"] = (
            cindex(
                ctx_outer["held_time"],
                ctx_outer["held_event"],
                matched_with_interaction
            )
            - cindex(
                ctx_outer["held_time"],
                ctx_outer["held_event"],
                matched_without_interaction
            )
        )

        atomic_joblib(
            {
                "signature": signature,
                "held": held_outer,
                "row": row,
                "inner": inner_saved,
                "predictions": pred,
                "trained_models": trained_models,
                "selected_candidate": winner,
                "note": (
                    "Ablations use inner-selected settings; "
                    "matched epoch/gamma contrasts also saved."
                )
            },
            checkpoint
        )

        print(
            f"  ReCAP={row['recap_C']:.4f}, "
            f"selected={row['selected_C']:.4f}, "
            f"delta={row['selected_minus_recap']:+.4f}",
            flush=True
        )

    fold_rows.append(row)
    inner_rows.extend(inner_saved)

    for name, values in pred.items():
        all_oof[name][held_outer] = values


# Verify matched outer-fold evaluation
outer_table = pd.DataFrame(
    fold_rows
).sort_values("fold")

inner_table = pd.DataFrame(
    inner_rows
)

numerical = outer_table.select_dtypes(
    include="number"
).to_numpy()

if not np.isfinite(numerical).all():
    raise RuntimeError(
        "Nonfinite fold-level results."
    )

eligible_ids = np.concatenate([
    np.asarray(
        context(
            f"outer_fold{i}_features.joblib"
        )["held_idx"],
        dtype=int
    )
    for i in range(1, 5)
])

if (
    len(eligible_ids)
    != len(np.unique(eligible_ids))
    or len(eligible_ids) != 657
):
    raise RuntimeError(
        "Outer folds do not partition the "
        "original eligible training patients."
    )

if any(
    not np.isfinite(
        risks[eligible_ids]
    ).all()
    for risks in all_oof.values()
):
    raise RuntimeError(
        "Incomplete outer OOF risk predictions."
    )

mean_baseline = float(
    outer_table.recap_C.mean()
)

mean_selected = float(
    outer_table.selected_C.mean()
)

if abs(
    mean_baseline
    - manifest["recap_mean_site_C"]
) > 2e-5:
    raise RuntimeError(
        "ReCAP benchmark changed during "
        "architecture evaluation."
    )


# Exploratory paired site-cluster bootstrap
bootstrap = []
rng = np.random.default_rng(2026)

for _ in range(300):
    differences = []

    for i in range(1, 5):
        held = np.asarray(
            context(
                f"outer_fold{i}_features.joblib"
            )["held_idx"],
            dtype=int
        )

        fold_sites = (
            clinical.site
            .iloc[held]
            .astype(str)
            .to_numpy()
        )

        distinct = np.unique(
            fold_sites
        )

        sampled = rng.choice(
            distinct,
            size=len(distinct),
            replace=True
        )

        indices = np.concatenate([
            held[
                fold_sites == site_id
            ]
            for site_id in sampled
        ])

        t = pd.to_numeric(
            clinical["PFI.time"].iloc[
                indices
            ],
            errors="raise"
        ).to_numpy(float)

        e = pd.to_numeric(
            clinical.PFI.iloc[
                indices
            ],
            errors="raise"
        ).to_numpy(float)

        try:
            selected_c = cindex(
                t,
                e,
                all_oof["selected"][indices]
            )

            recap_c = cindex(
                t,
                e,
                all_oof["recap"][indices]
            )

            difference = (
                selected_c - recap_c
            )

        except RuntimeError:
            break

        differences.append(
            difference
        )

    if len(differences) == 4:
        bootstrap.append(
            float(np.mean(differences))
        )

if len(bootstrap) < 100:
    raise RuntimeError(
        "Too few valid site-cluster bootstrap replicates."
    )


# Save reproducible evaluation artifacts
outer_table.to_csv(
    OUT / "spare_phase_b_outer_results.csv",
    index=False
)

inner_table.to_csv(
    OUT / "spare_phase_b_inner_ablation_scores.csv",
    index=False
)

pd.DataFrame({
    "case_id": (
        clinical.case_id.iloc[
            eligible_ids
        ].to_numpy()
    ),
    "site": (
        clinical.site.iloc[
            eligible_ids
        ].to_numpy()
    ),
    "histology": (
        clinical.histology_matrix.iloc[
            eligible_ids
        ].to_numpy()
    ),
    "PFI_time": pd.to_numeric(
        clinical["PFI.time"],
        errors="raise"
    ).iloc[
        eligible_ids
    ].to_numpy(),
    "PFI_event": pd.to_numeric(
        clinical.PFI,
        errors="raise"
    ).iloc[
        eligible_ids
    ].to_numpy(),
    **{
        name + "_risk": risk[
            eligible_ids
        ]
        for name, risk in all_oof.items()
    }
}).to_csv(
    OUT / "spare_phase_b_training_site_oof.csv",
    index=False
)


# Research summary and prespecified go/no-go decision
summary = {
    "version": VERSION,
    "completed_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "phase_a_signature": (
        manifest["cache_signature"]
    ),
    "signature": signature,
    "fixed_recap_mean_C": mean_baseline,
    "selected_mean_C": mean_selected,
    "selected_delta_C": (
        mean_selected - mean_baseline
    ),
    "outer_positive_folds": int(
        (
            outer_table.selected_minus_recap > 0
        ).sum()
    ),
    "model_usage": (
        outer_table.selected_architecture
        .value_counts()
        .to_dict()
    ),
    "ablation_mean_C": {
        name: float(
            outer_table[
                name + "_C"
            ].mean()
        )
        for name in ARCHS
    },
    "site_consistency_mean_C_delta": float(
        outer_table.matched_site_module_delta.mean()
    ),
    "interaction_mean_C_delta": float(
        outer_table.matched_interaction_module_delta.mean()
    ),
    "exploratory_site_cluster_95pct_delta": (
        np.quantile(
            bootstrap,
            [0.025, 0.975]
        ).tolist()
    ),
    "development_decision": (
        "PROMISING_FOR_INDEPENDENT_CONFIRMATION"
        if (
            mean_selected - mean_baseline >= 0.01
            and (
                outer_table.selected_minus_recap > 0
            ).sum() >= 3
        )
        else "NOT_YET_BETTER_THAN_RECAP"
    ),
    "limitations": [
        (
            "The same training sites were previously used "
            "in architecture development; estimates are exploratory."
        ),
        (
            "All training sites contain one histology, "
            "so site and histology are structurally confounded."
        ),
        (
            "Clinically adjusted site shifts may still "
            "contain biological variation."
        ),
        (
            "Ablations differ in parameter count; "
            "matched epoch/gamma contrasts are reported."
        ),
        (
            "Old TCGA validation and test are not fresh "
            "evaluation cohorts for a new architecture."
        ),
        (
            "The single planned external cohort is GSE50081; "
            "its outcomes were not accessed."
        )
    ]
}

atomic_json(
    summary,
    OUT / "spare_phase_b_summary.json"
)


# Final report
print("\n" + "=" * 78)
print(
    "SPARE PHASE B FINAL - TRAINING-ONLY "
    "SITE HELD-OUT ABLATIONS"
)
print("=" * 78)

columns = [
    "fold",
    "selected_architecture",
    "selected_epoch",
    "selected_gamma",
    "recap_C",
    "mlp_residual_C",
    "lowrank_interaction_C",
    "spare_full_C",
    "selected_C",
    "selected_minus_recap",
    "matched_interaction_module_delta",
    "matched_site_module_delta"
]

print(
    outer_table[columns].to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print(
    "ReCAP mean C:",
    round(mean_baseline, 4)
)

print(
    "SPARE selected mean C:",
    round(mean_selected, 4)
)

print(
    "Selected minus ReCAP:",
    round(
        mean_selected - mean_baseline,
        4
    )
)

print(
    "Outer positive folds:",
    summary["outer_positive_folds"],
    "/ 4"
)

print(
    "Architecture ablation means:",
    {
        name: round(value, 4)
        for name, value
        in summary["ablation_mean_C"].items()
    }
)

print(
    "Interaction module matched delta:",
    round(
        summary["interaction_mean_C_delta"],
        4
    )
)

print(
    "Site module matched delta:",
    round(
        summary["site_consistency_mean_C_delta"],
        4
    )
)

print(
    "Exploratory site-cluster 95% paired delta CI:",
    [
        round(float(x), 4)
        for x in summary[
            "exploratory_site_cluster_95pct_delta"
        ]
    ]
)

print(
    "Development decision:",
    summary["development_decision"]
)

print("Saved:", OUT)

print(
    "Old validation, sealed test and "
    "external outcomes evaluated: NO"
)

print("SPARE PHASE B COMPLETED")

SPARE PHASE B - PRESPECIFIED RESIDUAL ARCHITECTURE COMPARISON
ReCAP-Cox frozen training-CV benchmark: 0.6549
Single-histology training sites: 62/62
Models: ('mlp_residual', 'lowrank_interaction', 'spare_full'); seeds=(41, 137); checkpoints=(20, 50, 80)
Censored Cox gradient check: 1.83e-10
Exact anchor recovery: PASS
TCGA validation/test and GSE50081 outcomes: NOT ACCESSED

OUTER 1: train=491, held=166, events=56
  inner 1, site-shift coverage=91.5%
  inner 2, site-shift coverage=93.0%
  inner 3, site-shift coverage=93.5%
  inner gains: {'mlp_residual': 0.002, 'lowrank_interaction': 0.0019, 'spare_full': 0.002}
  inner selection: ReCAP baseline
  ReCAP=0.6720, selected=0.6720, delta=+0.0000

OUTER 2: train=494, held=163, events=55
  inner 1, site-shift coverage=92.7%
  inner 2, site-shift coverage=90.3%
  inner 3, site-shift coverage=92.1%
  inner gains: {'mlp_residual': 0.0009, 'lowrank_interaction': 0.0006, 'spare_full': 0.0006}
  inner selection: ReCAP baseline
  ReCAP=0.6596, selec

In [30]:

import hashlib
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.optimize import minimize
from scipy.stats import rankdata
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

# Phase C: fixed biological-prior controls; no architecture search.
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
RAW = ROOT / "data" / "processed" / "gene_level"
A = ROOT / "results" / "metrics" / "spare_phase_a"
S19 = ROOT / "results" / "metrics" / "stage19"
S21 = ROOT / "results" / "metrics" / "stage21"
OUT = ROOT / "results" / "metrics" / "spare_phase_c"

MODEL = (
    ROOT / "models" / "checkpoints" / "stage26"
    / "stage26_training_only_final.joblib"
)

SEED = 42
NULL_SEEDS = (101, 211, 307, 419, 523)
BOOTSTRAPS = 300
THREADS = min(4, os.cpu_count() or 1)
VERSION = "spare_phase_c_reactome_prior_ablation_v1"

if Path(sys.executable).resolve() != (
    ROOT / "envs" / "nsclc" / "python.exe"
).resolve():
    raise RuntimeError(
        "Select NST NSCLC Research (Python 3.12) kernel."
    )

OUT.mkdir(parents=True, exist_ok=True)


# Reproducibility and checkpoint utilities
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for data in iter(
            lambda: f.read(4 * 1024 * 1024), b""
        ):
            h.update(data)
    return h.hexdigest()


def atomic_dump(obj, path):
    temporary = path.with_name(path.name + ".tmp")
    joblib.dump(obj, temporary, compress=3)
    os.replace(temporary, path)


def atomic_json(obj, path):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(
        json.dumps(obj, indent=2, allow_nan=False),
        encoding="utf-8"
    )
    os.replace(temporary, path)


# Harrell concordance
def cindex(t, e, risk):
    t, e, risk = map(
        np.asarray, (t, e, risk)
    )

    pairs = (
        (t[:, None] < t[None, :])
        & (e[:, None] == 1)
    )

    n = int(pairs.sum())

    if n < 10:
        return np.nan

    differences = (
        risk[:, None] - risk[None, :]
    )

    concordant = np.count_nonzero(
        pairs & (differences > 1e-10)
    )

    tied = np.count_nonzero(
        pairs & (np.abs(differences) <= 1e-10)
    )

    return float(
        (concordant + 0.5 * tied) / n
    )


# Penalized Cox regression with fixed clinical offset
def cox_fit(z, time, event, offset, ridge=0.10):
    z = np.asarray(z, dtype=np.float64)
    time = np.asarray(time, dtype=np.float64)
    event = np.asarray(event, dtype=np.int64)
    offset = np.asarray(offset, dtype=np.float64)

    if (
        z.ndim != 2
        or not (
            len(z) == len(time) == len(event)
            == len(offset)
        )
        or event.sum() < 10
    ):
        raise RuntimeError(
            "Invalid Cox input or insufficient events."
        )

    ordering = np.argsort(
        -time, kind="stable"
    )

    z = z[ordering]
    time = time[ordering]
    event = event[ordering]
    offset = offset[ordering]

    starts = np.r_[
        0,
        np.flatnonzero(
            time[1:] != time[:-1]
        ) + 1
    ]

    ends = np.r_[
        starts[1:] - 1,
        len(time) - 1
    ]

    deaths = np.add.reduceat(
        event, starts
    )

    ends = ends[deaths > 0]
    deaths = deaths[
        deaths > 0
    ].astype(float)

    n_events = float(event.sum())

    def objective(beta):
        eta = z @ beta + offset
        shift = eta.max()
        weight = np.exp(eta - shift)

        cum = np.cumsum(weight)

        cum_z = np.cumsum(
            weight[:, None] * z,
            axis=0
        )

        denom = np.maximum(
            cum[ends], 1e-300
        )

        loss = (
            np.sum(
                deaths * (
                    shift + np.log(denom)
                )
            )
            - eta[event == 1].sum()
        ) / n_events

        grad = (
            np.sum(
                deaths[:, None]
                * cum_z[ends]
                / denom[:, None],
                axis=0
            )
            - z[event == 1].sum(axis=0)
        ) / n_events

        return (
            float(
                loss
                + 0.5 * ridge
                * np.dot(beta, beta)
            ),
            grad + ridge * beta
        )

    fitted = minimize(
        objective,
        np.zeros(z.shape[1]),
        method="L-BFGS-B",
        jac=True,
        options={
            "maxiter": 250,
            "ftol": 1e-9,
            "gtol": 1e-6
        }
    )

    if (
        not np.isfinite(fitted.x).all()
        or not np.isfinite(fitted.fun)
    ):
        raise RuntimeError(
            "Cox optimization returned non-finite values."
        )

    return fitted.x


# Reproduce the fixed SVD8 and offset-Cox protocol
def fixed_svd8_offset_scores(
    features, ctx, fit, held
):
    scaler = StandardScaler().fit(
        features[fit]
    )

    z_fit = np.clip(
        scaler.transform(features[fit]),
        -7, 7
    ).astype(np.float32)

    z_held = np.clip(
        scaler.transform(features[held]),
        -7, 7
    ).astype(np.float32)

    svd = TruncatedSVD(
        n_components=8,
        n_iter=4,
        random_state=SEED
    )

    s_fit = svd.fit_transform(
        z_fit
    )

    s_held = svd.transform(
        z_held
    )

    latent_scaler = StandardScaler().fit(
        s_fit
    )

    s_fit = np.clip(
        latent_scaler.transform(s_fit),
        -7, 7
    )

    s_held = np.clip(
        latent_scaler.transform(s_held),
        -7, 7
    )

    beta = cox_fit(
        s_fit,
        ctx["fit_time"],
        ctx["fit_event"],
        ctx["clinical_fit"]
    )

    scores = (
        np.asarray(
            ctx["clinical_hold"],
            dtype=float
        )
        + s_held @ beta
    )

    if not np.isfinite(scores).all():
        raise RuntimeError(
            "Non-finite molecular risk predictions."
        )

    return scores


# Verify original cohort and Phase A reference
manifest_file = A / "phase_a_manifest.json"

clinical_file = (
    S21 / "stage21_training_clinical_endpoints.csv"
)

case_file = RAW / "case_order.csv"

locked_file = (
    ROOT / "data" / "splits"
    / "locked_four_omics_v1.csv"
)

genes_file = RAW / "gene_reference.csv"
rna_file = RAW / "rna_tpm.npy"

w_file = (
    S19 / "reactome_gene_pathway_weights.npz"
)

contexts_files = [
    A / f"outer_fold{k}_features.joblib"
    for k in range(1, 5)
]

inputs = [
    manifest_file,
    clinical_file,
    case_file,
    locked_file,
    genes_file,
    rna_file,
    w_file,
    MODEL
] + contexts_files

for file in inputs:
    if not file.is_file():
        raise FileNotFoundError(
            f"Missing project input: {file}"
        )

manifest = json.loads(
    manifest_file.read_text(
        encoding="utf-8"
    )
)

if (
    manifest["stage26_model_sha256"]
    != sha256(MODEL)
):
    raise RuntimeError(
        "Frozen ReCAP-Cox checkpoint changed."
    )

if abs(
    manifest["recap_mean_site_C"] - 0.6549
) > 0.0006:
    raise RuntimeError(
        "Phase A benchmark unexpectedly changed."
    )

clinical = pd.read_csv(
    clinical_file,
    dtype={"case_id": str}
)

cases = pd.read_csv(
    case_file,
    dtype={"case_id": str}
)

locked = pd.read_csv(
    locked_file,
    dtype={"case_id": str}
)

genes = pd.read_csv(
    genes_file,
    dtype={"ensembl_id": str}
)

if not (
    clinical.case_id.is_unique
    and cases.case_id.is_unique
    and locked.case_id.is_unique
):
    raise RuntimeError(
        "Duplicated patient identifiers."
    )

cohort = cases[
    ["case_id", "histology", "stage_group"]
].merge(
    locked[
        [
            "case_id",
            "histology",
            "stage_group",
            "split"
        ]
    ],
    on="case_id",
    how="left",
    validate="one_to_one",
    suffixes=("_raw", "_lock")
)

if not (
    cohort.histology_raw.equals(
        cohort.histology_lock
    )
    and cohort.stage_group_raw.equals(
        cohort.stage_group_lock
    )
):
    raise RuntimeError(
        "Locked cohort labels do not match case order."
    )

training_rows = np.flatnonzero(
    cohort.split.eq("train").to_numpy()
)

if (
    len(cases),
    len(training_rows),
    len(genes)
) != (953, 667, 19944):
    raise RuntimeError(
        "Unexpected cohort or gene reference dimensions."
    )

if not np.array_equal(
    clinical.case_id.to_numpy(),
    cohort.case_id.iloc[
        training_rows
    ].to_numpy()
):
    raise RuntimeError(
        "Stage 21 training patient order mismatch."
    )

eligible = np.flatnonzero(
    clinical.PFI_valid.astype(str)
    .str.lower()
    .eq("true")
    .to_numpy()
)

if (
    len(eligible) != 657
    or int(
        clinical.PFI.iloc[eligible].sum()
    ) != 225
):
    raise RuntimeError(
        "Unexpected training PFI outcome counts."
    )


# Verify site-held-out outer folds
ctx_list = [
    joblib.load(p)
    for p in contexts_files
]

for k, ctx in enumerate(ctx_list, 1):
    fit = np.asarray(
        ctx["fit_idx"], int
    )

    held = np.asarray(
        ctx["held_idx"], int
    )

    if (
        ctx["signature"]
        != manifest["cache_signature"]
    ):
        raise RuntimeError(
            f"Phase A fold {k} signature mismatch."
        )

    if not (
        len(np.unique(fit)) == len(fit)
        and len(np.unique(held)) == len(held)
    ):
        raise RuntimeError(
            f"Repeated IDs in fold {k}."
        )

    if (
        not set(fit).isdisjoint(held)
        or not set(
            ctx["fit_site"]
        ).isdisjoint(
            ctx["held_site"]
        )
    ):
        raise RuntimeError(
            f"Data or site leakage in fold {k}."
        )

if not np.array_equal(
    np.sort(
        np.concatenate([
            c["held_idx"]
            for c in ctx_list
        ])
    ),
    eligible
):
    raise RuntimeError(
        "Outer folds do not partition all "
        "657 PFI-eligible patients."
    )


# Fingerprint the fixed comparison protocol
settings = {
    "version": VERSION,
    "null_seeds": NULL_SEEDS,
    "n_components": 8,
    "ridge": 0.10,
    "phase_a_signature": (
        manifest["cache_signature"]
    ),
    "source_sha256": {
        str(p): sha256(p)
        for p in inputs
        if p != rna_file
    },
    "rna_source_size_and_mtime": (
        rna_file.stat().st_size,
        rna_file.stat().st_mtime_ns
    )
}

signature = hashlib.sha256(
    json.dumps(
        settings,
        sort_keys=True
    ).encode()
).hexdigest()

print("=" * 79)
print(
    "SPARE PHASE C - "
    "REACTOME BIOLOGICAL-PRIOR ABLATION"
)
print("=" * 79)

print(
    "Goal: test whether Reactome membership "
    "adds value beyond generic RNA SVD8."
)

print(
    "Reference ReCAP mean site C: 0.6549"
)

print(
    "Fixed controls: gene-rank SVD8 and "
    "five shuffled pathway-gene label maps."
)

print(
    "Stage27/28 and GSE50081 outcomes: NOT ACCESSED"
)


# Construct pathway scores from locked training RNA only
W = sparse.load_npz(
    w_file
).tocsr().astype(np.float32)

if (
    W.shape != (1613, 19944)
    or not np.isfinite(W.data).all()
):
    raise RuntimeError(
        "Reactome gene weight matrix changed."
    )

raw_source = np.load(
    rna_file,
    mmap_mode="r"
)

if raw_source.shape != (953, 19944):
    raise RuntimeError(
        "Original RNA matrix shape changed."
    )

log_rna = np.asarray(
    raw_source[training_rows],
    dtype=np.float32
).copy()

del raw_source

present = (
    np.isfinite(log_rna)
    & (log_rna >= 0)
)

if present.mean() < 0.95:
    raise RuntimeError(
        "Unexpected raw RNA completeness."
    )

log_rna = np.log1p(
    np.where(present, log_rna, 0)
).astype(np.float32)

del present

ranks = rankdata(
    log_rna,
    axis=1,
    method="average"
).astype(np.float32)

del log_rna

ranks = (
    ranks - (len(genes) + 1) / 2.0
) / len(genes)

real_pathways = np.asarray(
    (W @ ranks.T).T,
    dtype=np.float32
)


# Randomize gene-to-pathway correspondence globally
# while retaining the original pathway membership structure.
shuffled_pathways = {}

for seed in NULL_SEEDS:
    perm = np.random.default_rng(
        seed
    ).permutation(
        len(genes)
    )

    shuffled_pathways[seed] = np.asarray(
        (W @ ranks[:, perm].T).T,
        dtype=np.float32
    )

if (
    not np.isfinite(real_pathways).all()
    or any(
        not np.isfinite(x).all()
        for x in shuffled_pathways.values()
    )
):
    raise RuntimeError(
        "Invalid biological-prior feature matrix."
    )


# Evaluate fixed representations with identical Cox settings
rows = []

names = (
    "clinical",
    "recap",
    "gene_rank_svd8",
    *(
        f"shuffled_{s}"
        for s in NULL_SEEDS
    )
)

risks = {
    name: np.full(667, np.nan)
    for name in names
}

with threadpool_limits(
    limits=THREADS
):
    for fold, ctx in enumerate(
        ctx_list, 1
    ):
        output = (
            OUT
            / f"outer_fold{fold}_controls.joblib"
        )

        held = np.asarray(
            ctx["held_idx"],
            dtype=int
        )

        fit = np.asarray(
            ctx["fit_idx"],
            dtype=int
        )

        if output.exists():
            cached = joblib.load(
                output
            )

            if (
                cached.get("signature")
                != signature
                or not np.array_equal(
                    cached["held"], held
                )
            ):
                raise RuntimeError(
                    f"Incompatible cached fold: {output}"
                )

            predictions = cached[
                "predictions"
            ]

            row = cached["row"]

            print(
                f"Outer {fold}: "
                "verified checkpoint reused",
                flush=True
            )

        else:
            reproduced_recap = (
                fixed_svd8_offset_scores(
                    real_pathways,
                    ctx,
                    fit,
                    held
                )
            )

            error = float(
                np.max(
                    np.abs(
                        reproduced_recap
                        - ctx["baseline_hold"]
                    )
                )
            )

            if error > 1e-3:
                raise RuntimeError(
                    f"Fold {fold}: ReCAP reproduction "
                    f"mismatch {error:.5g}"
                )

            predictions = {
                "clinical": np.asarray(
                    ctx["clinical_hold"],
                    dtype=float
                ),
                "recap": np.asarray(
                    ctx["baseline_hold"],
                    dtype=float
                ),
                "gene_rank_svd8": (
                    fixed_svd8_offset_scores(
                        ranks,
                        ctx,
                        fit,
                        held
                    )
                )
            }

            for seed, features in (
                shuffled_pathways.items()
            ):
                predictions[
                    f"shuffled_{seed}"
                ] = fixed_svd8_offset_scores(
                    features,
                    ctx,
                    fit,
                    held
                )

            tt = np.asarray(
                ctx["held_time"],
                dtype=float
            )

            ee = np.asarray(
                ctx["held_event"],
                dtype=int
            )

            scores = {
                name: cindex(
                    tt, ee, risk
                )
                for name, risk
                in predictions.items()
            }

            null_scores = [
                scores[f"shuffled_{seed}"]
                for seed in NULL_SEEDS
            ]

            row = {
                "fold": fold,
                "held": len(held),
                "events": int(
                    ee.sum()
                ),
                "max_reference_error": error,
                "clinical_C": (
                    scores["clinical"]
                ),
                "recap_C": (
                    scores["recap"]
                ),
                "gene_rank_svd8_C": (
                    scores["gene_rank_svd8"]
                ),
                "shuffled_mean_C": float(
                    np.mean(null_scores)
                ),
                "shuffled_max_C": float(
                    np.max(null_scores)
                ),
                "recap_minus_gene": (
                    scores["recap"]
                    - scores["gene_rank_svd8"]
                ),
                "recap_minus_shuffled_mean": (
                    scores["recap"]
                    - float(
                        np.mean(null_scores)
                    )
                )
            }

            atomic_dump(
                {
                    "signature": signature,
                    "held": held,
                    "predictions": predictions,
                    "row": row
                },
                output
            )

            print(
                f"Outer {fold}: "
                f"ReCAP={row['recap_C']:.4f}; "
                f"gene={row['gene_rank_svd8_C']:.4f}; "
                f"shuffled={row['shuffled_mean_C']:.4f}",
                flush=True
            )

        rows.append(row)

        for name, risk in (
            predictions.items()
        ):
            risks[name][held] = risk


# Verify complete matched evaluation
results = pd.DataFrame(
    rows
).sort_values("fold")

if (
    not np.isfinite(
        results.select_dtypes(
            include="number"
        ).to_numpy()
    ).all()
    or any(
        not np.isfinite(
            r[eligible]
        ).all()
        for r in risks.values()
    )
):
    raise RuntimeError(
        "Incomplete or invalid outer-fold predictions."
    )

if abs(
    float(results.recap_C.mean())
    - manifest["recap_mean_site_C"]
) > 2e-5:
    raise RuntimeError(
        "ReCAP baseline changed."
    )


# Exploratory paired site-cluster uncertainty
rng = np.random.default_rng(2026)
bootstrap_differences = []

for _ in range(BOOTSTRAPS):
    fold_delta_gene = []
    fold_delta_shuffled = []

    for ctx in ctx_list:
        held = np.asarray(
            ctx["held_idx"],
            dtype=int
        )

        site_ids = np.asarray(
            ctx["held_site"],
            dtype=str
        )

        distinct_sites = np.unique(
            site_ids
        )

        sampled_sites = rng.choice(
            distinct_sites,
            size=len(distinct_sites),
            replace=True
        )

        selected = np.concatenate([
            held[site_ids == sid]
            for sid in sampled_sites
        ])

        t = pd.to_numeric(
            clinical["PFI.time"].iloc[
                selected
            ],
            errors="raise"
        ).to_numpy(float)

        e = pd.to_numeric(
            clinical.PFI.iloc[
                selected
            ],
            errors="raise"
        ).to_numpy(int)

        base = cindex(
            t,
            e,
            risks["recap"][selected]
        )

        gene = cindex(
            t,
            e,
            risks["gene_rank_svd8"][selected]
        )

        null_scores = np.asarray([
            cindex(
                t,
                e,
                risks[
                    f"shuffled_{s}"
                ][selected]
            )
            for s in NULL_SEEDS
        ])

        if not np.isfinite(
            [base, gene, *null_scores]
        ).all():
            break

        fold_delta_gene.append(
            base - gene
        )

        fold_delta_shuffled.append(
            base - float(
                null_scores.mean()
            )
        )

    if len(fold_delta_gene) == 4:
        bootstrap_differences.append(
            (
                float(
                    np.mean(
                        fold_delta_gene
                    )
                ),
                float(
                    np.mean(
                        fold_delta_shuffled
                    )
                )
            )
        )

if len(bootstrap_differences) < 100:
    raise RuntimeError(
        "Too few valid site-bootstrap replicates."
    )

intervals = np.quantile(
    bootstrap_differences,
    [0.025, 0.975],
    axis=0
)


# Save results and transparent limitations
summary = {
    "version": VERSION,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "clinical_mean_C": float(
        results.clinical_C.mean()
    ),
    "recap_mean_C": float(
        results.recap_C.mean()
    ),
    "gene_rank_svd8_mean_C": float(
        results.gene_rank_svd8_C.mean()
    ),
    "shuffled_pathway_mean_C": float(
        results.shuffled_mean_C.mean()
    ),
    "recap_minus_gene": float(
        results.recap_minus_gene.mean()
    ),
    "recap_minus_shuffled": float(
        results.recap_minus_shuffled_mean.mean()
    ),
    "recap_minus_gene_exploratory_site_CI": (
        intervals[:, 0].tolist()
    ),
    "recap_minus_shuffled_exploratory_site_CI": (
        intervals[:, 1].tolist()
    ),
    "recap_better_than_gene_folds": int(
        (
            results.recap_minus_gene > 0
        ).sum()
    ),
    "recap_better_than_shuffled_folds": int(
        (
            results.recap_minus_shuffled_mean > 0
        ).sum()
    ),
    "random_gene_mapping_seeds": list(
        NULL_SEEDS
    ),
    "limitations": [
        "Exploratory reanalysis of repeatedly used training-only outer folds, not unbiased confirmation.",
        "Shuffling gene-to-pathway correspondence preserves membership structure but is a negative control, not a formal permutation test.",
        "No model selection on Stage27/28 validation or test outcomes.",
        "External cohort remains GSE50081 only; endpoint and platform transfer require separate audit."
    ]
}

results.to_csv(
    OUT / "phase_c_fold_comparisons.csv",
    index=False
)

pd.DataFrame({
    "case_id": (
        clinical.case_id.iloc[
            eligible
        ].to_numpy()
    ),
    "PFI_time": pd.to_numeric(
        clinical["PFI.time"].iloc[
            eligible
        ]
    ).to_numpy(),
    "PFI_event": pd.to_numeric(
        clinical.PFI.iloc[
            eligible
        ]
    ).to_numpy(),
    "site": (
        clinical.site.iloc[
            eligible
        ].to_numpy()
    ),
    **{
        name + "_risk": risk[
            eligible
        ]
        for name, risk in (
            risks.items()
        )
    }
}).to_csv(
    OUT / "phase_c_training_only_oof.csv",
    index=False
)

atomic_json(
    summary,
    OUT / "phase_c_summary.json"
)


# Final research report
print("\n" + "=" * 79)
print(
    "SPARE PHASE C FINAL - "
    "BIOLOGICAL PRIOR CONTRIBUTION"
)
print("=" * 79)

print(
    results.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print(
    f"Clinical mean C: "
    f"{summary['clinical_mean_C']:.4f}"
)

print(
    f"ReCAP mean C: "
    f"{summary['recap_mean_C']:.4f}"
)

print(
    f"Gene-level rank-SVD8 mean C: "
    f"{summary['gene_rank_svd8_mean_C']:.4f}"
)

print(
    f"Shuffled-pathway mean C: "
    f"{summary['shuffled_pathway_mean_C']:.4f}"
)

print(
    f"ReCAP - gene SVD8: "
    f"{summary['recap_minus_gene']:+.4f}"
)

print(
    f"ReCAP - shuffled: "
    f"{summary['recap_minus_shuffled']:+.4f}"
)

print(
    "Exploratory site-bootstrap CI "
    "(ReCAP - gene):",
    np.round(
        intervals[:, 0], 4
    ).tolist()
)

print(
    "Exploratory site-bootstrap CI "
    "(ReCAP - shuffled):",
    np.round(
        intervals[:, 1], 4
    ).tolist()
)

print(
    "ReCAP positive folds vs gene:",
    summary[
        "recap_better_than_gene_folds"
    ],
    "/4"
)

print(
    "ReCAP positive folds vs shuffled:",
    summary[
        "recap_better_than_shuffled_folds"
    ],
    "/4"
)

print(
    "Development interpretation only; "
    "previous TCGA validation/test not reused."
)

print("Saved:", OUT)
print("SPARE PHASE C COMPLETED")

SPARE PHASE C - REACTOME BIOLOGICAL-PRIOR ABLATION
Goal: test whether Reactome membership adds value beyond generic RNA SVD8.
Reference ReCAP mean site C: 0.6549
Fixed controls: gene-rank SVD8 and five shuffled pathway-gene label maps.
Stage27/28 and GSE50081 outcomes: NOT ACCESSED
Outer 1: ReCAP=0.6720; gene=0.6666; shuffled=0.6686
Outer 2: ReCAP=0.6596; gene=0.6286; shuffled=0.6617
Outer 3: ReCAP=0.7075; gene=0.6932; shuffled=0.7042
Outer 4: ReCAP=0.5804; gene=0.5772; shuffled=0.5858

SPARE PHASE C FINAL - BIOLOGICAL PRIOR CONTRIBUTION
 fold  held  events  max_reference_error  clinical_C  recap_C  gene_rank_svd8_C  shuffled_mean_C  shuffled_max_C  recap_minus_gene  recap_minus_shuffled_mean
    1   166      56               0.0000      0.6489   0.6720            0.6666           0.6686          0.6834            0.0054                     0.0034
    2   163      55               0.0000      0.6421   0.6596            0.6286           0.6617          0.6787            0.0309          

In [31]:

import csv
import gzip
import hashlib
import json
import os
import re
import sys
import urllib.request

from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse


# External cohort configuration
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
RAW = ROOT / "data" / "processed" / "gene_level"
S19 = ROOT / "results" / "metrics" / "stage19"

MODEL = (
    ROOT / "models" / "checkpoints" / "stage26"
    / "stage26_training_only_final.joblib"
)

OUT = ROOT / "data" / "external" / "GSE50081"
REPORT = (
    ROOT / "results" / "metrics"
    / "gse50081_external_audit"
)

VERSION = "gse50081_external_feasibility_v1"

SERIES_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/series/"
    "GSE50nnn/GSE50081/matrix/"
    "GSE50081_series_matrix.txt.gz"
)

ANNOT_URL = (
    "https://ftp.ncbi.nlm.nih.gov/geo/platforms/"
    "GPLnnn/GPL570/annot/GPL570.annot.gz"
)

SERIES_FILE = OUT / "GSE50081_series_matrix.txt.gz"
ANNOT_FILE = OUT / "GPL570.annot.gz"

EXPECTED = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != EXPECTED.resolve():
    raise RuntimeError(
        "Select NST NSCLC Research (Python 3.12) kernel."
    )

OUT.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(parents=True, exist_ok=True)


# Reproducibility utilities
def digest(path):
    h = hashlib.sha256()

    with open(path, "rb") as stream:
        for block in iter(
            lambda: stream.read(4 * 1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def save_json(path, data):
    temp = path.with_name(path.name + ".tmp")

    temp.write_text(
        json.dumps(
            data,
            indent=2,
            allow_nan=False
        ),
        encoding="utf-8"
    )

    os.replace(temp, path)


def normalize(value):
    return re.sub(
        r"[^a-z0-9]+",
        " ",
        str(value).lower()
    ).strip()


def split_tab(line):
    return next(
        csv.reader(
            [line],
            delimiter="\t",
            quotechar='"'
        )
    )


# Download official GEO files with bounded size
def download_geo(url, path, limit_mb=180):

    if path.exists():
        if path.stat().st_size == 0:
            raise RuntimeError(
                f"Empty existing archive: {path}"
            )

        print(
            f"Cached: {path.name} "
            f"({path.stat().st_size / 1048576:.1f} MiB)"
        )

        return

    temp = path.with_name(
        path.name + ".part"
    )

    if temp.exists():
        temp.unlink()

    request = urllib.request.Request(
        url,
        headers={
            "User-Agent": (
                "AcademicResearch-GEO-Audit/1.0"
            )
        }
    )

    try:
        with (
            urllib.request.urlopen(
                request,
                timeout=90
            ) as response,
            open(temp, "wb") as output
        ):
            total = 0

            while True:
                block = response.read(
                    1024 * 1024
                )

                if not block:
                    break

                total += len(block)

                if total > limit_mb * 1048576:
                    raise RuntimeError(
                        "Unexpectedly large GEO file."
                    )

                output.write(block)

        with open(temp, "rb") as handle:
            if handle.read(2) != b"\x1f\x8b":
                raise RuntimeError(
                    "Downloaded file is not gzip."
                )

        os.replace(temp, path)

    except Exception as error:
        if temp.exists():
            temp.unlink()

        raise RuntimeError(
            f"Download failed: {url}\n"
            "Download the official .gz file manually "
            f"to {path} and rerun.\n"
            f"Original error: {error}"
        ) from error

    print(
        f"Downloaded: {path.name} "
        f"({path.stat().st_size / 1048576:.1f} MiB)"
    )


# Read GEO metadata and inspect the matrix
def parse_series_matrix(path):

    metadata = defaultdict(list)
    probes = []
    numeric_sample = []

    with gzip.open(
        path,
        "rt",
        encoding="utf-8",
        errors="replace"
    ) as stream:

        for line in stream:

            if line.startswith(
                "!series_matrix_table_begin"
            ):
                break

            if (
                line.startswith("!Sample_")
                or line.startswith('"!Sample_')
            ):
                parts = split_tab(
                    line.rstrip("\r\n")
                )

                metadata[
                    parts[0].strip('"')
                ].append(
                    parts[1:]
                )

        else:
            raise RuntimeError(
                "Series matrix table was not found."
            )

        header = split_tab(
            next(stream).rstrip("\r\n")
        )

        if header[0].strip('"') != "ID_REF":
            raise RuntimeError(
                "Invalid GEO expression header."
            )

        sample_ids = [
            value.strip('"').strip()
            for value in header[1:]
        ]

        if (
            len(sample_ids) != 181
            or len(set(sample_ids)) != 181
        ):
            raise RuntimeError(
                "GSE50081 sample count/order unexpected."
            )

        found_end = False

        for line in stream:

            if line.startswith(
                "!series_matrix_table_end"
            ):
                found_end = True
                break

            if not line.strip():
                continue

            probe = (
                line.partition("\t")[0]
                .strip()
                .strip('"')
            )

            probes.append(probe)

            # Sample only a subset of expression
            # rows for numerical quality control.
            if len(probes) % 101 == 1:

                row = split_tab(
                    line.rstrip("\r\n")
                )

                if len(row) != 182:
                    raise RuntimeError(
                        "Expression column count mismatch."
                    )

                values = pd.to_numeric(
                    pd.Series(row[1:]),
                    errors="coerce"
                ).to_numpy(float)

                numeric_sample.extend(
                    values
                )

    if (
        not found_end
        or len(probes) < 10000
        or len(set(probes)) != len(probes)
    ):
        raise RuntimeError(
            "Incomplete or duplicated GEO probe table."
        )

    accession = metadata.get(
        "!Sample_geo_accession",
        []
    )

    if (
        len(accession) != 1
        or accession[0] != sample_ids
    ):
        raise RuntimeError(
            "GEO metadata does not align with expression."
        )

    platforms = metadata.get(
        "!Sample_platform_id",
        []
    )

    if (
        not platforms
        or any(
            value != "GPL570"
            for value in platforms[0]
        )
    ):
        raise RuntimeError(
            "Unexpected GEO microarray platform."
        )

    records = []

    for i, sample_id in enumerate(
        sample_ids
    ):
        entry = {"GSM": sample_id}

        for row in metadata.get(
            "!Sample_characteristics_ch1",
            []
        ):
            if len(row) != 181:
                raise RuntimeError(
                    "Incomplete sample characteristics."
                )

            value = row[i].strip()

            if ":" not in value:
                continue

            label, content = value.split(
                ":",
                1
            )

            field = normalize(label)
            content = content.strip()

            if (
                field in entry
                and entry[field] != content
            ):
                raise RuntimeError(
                    f"Conflicting metadata for {sample_id}: "
                    f"{field}"
                )

            entry[field] = content

        records.append(entry)

    sampled = np.asarray(
        numeric_sample,
        dtype=float
    )

    statistics = {
        "samples": len(sample_ids),
        "probes": len(probes),
        "sampled_finite_fraction": float(
            np.isfinite(sampled).mean()
        ),
        "sampled_median": float(
            np.nanmedian(sampled)
        ),
        "sampled_min": float(
            np.nanmin(sampled)
        ),
        "sampled_max": float(
            np.nanmax(sampled)
        )
    }

    return (
        pd.DataFrame(records),
        probes,
        statistics
    )


# Parse GPL570 probes; exclude ambiguous gene mappings
def parse_gpl570(path, known_probes):

    known_probes = set(
        known_probes
    )

    mapping = {}
    ambiguous = 0
    header_found = False

    with gzip.open(
        path,
        "rt",
        encoding="utf-8",
        errors="replace"
    ) as stream:

        for line in stream:

            if (
                not line.strip()
                or line.startswith(
                    ("#", "^", "!")
                )
            ):
                continue

            row = split_tab(
                line.rstrip("\r\n")
            )

            if not header_found:

                headings = [
                    normalize(value)
                    for value in row
                ]

                if (
                    "id" in headings
                    and "gene symbol" in headings
                ):
                    id_col = headings.index("id")
                    gene_col = headings.index(
                        "gene symbol"
                    )

                    header_found = True

                continue

            if len(row) <= max(
                id_col,
                gene_col
            ):
                continue

            probe = row[id_col].strip()

            if probe not in known_probes:
                continue

            symbols = list(
                dict.fromkeys(
                    [
                        x.strip()
                        for x in re.split(
                            r"\s*///\s*",
                            row[gene_col].strip()
                        )
                        if x.strip()
                    ]
                )
            )

            if (
                len(symbols) != 1
                or symbols[0] in ("---", "NA")
                or re.search(
                    r"\s+//\s+|;",
                    symbols[0]
                )
            ):
                ambiguous += 1
                continue

            mapping[probe] = (
                symbols[0].upper()
            )

    if not header_found or len(mapping) < 10000:
        raise RuntimeError(
            "Insufficient valid GPL570 probe annotations."
        )

    return mapping, ambiguous


# Conservative clinical metadata interpretation
def pick_field(columns, candidates):

    for name in candidates:
        if name in columns:
            return name

    return None


def classify_histology(text):

    value = normalize(text)

    if re.search(
        r"\b(adenocarcinoma|adc|adeno|luad)\b",
        value
    ):
        return "LUAD"

    if re.search(
        r"\b(squamous|sqc|scc|lusc)\b",
        value
    ):
        return "LUSC"

    return "OTHER_OR_UNKNOWN"


def parse_stage(text):

    match = re.search(
        r"\b(?:stage\s*)?"
        r"(iv|iii|ii|i|[1-4])"
        r"(?:a|b|c)?\b",
        normalize(text)
    )

    if not match:
        return np.nan

    return {
        "i": 1,
        "ii": 2,
        "iii": 3,
        "iv": 4,
        "1": 1,
        "2": 2,
        "3": 3,
        "4": 4
    }[match.group(1)]


def parse_number(text):

    value = str(text).strip()

    if not re.fullmatch(
        r"[+-]?(?:\d+(?:\.\d*)?|\.\d+)"
        r"(?:[eE][+-]?\d+)?"
        r"(?:\s*(?:days?|months?|years?))?",
        value,
        flags=re.IGNORECASE
    ):
        return np.nan

    return float(
        value.split()[0]
    )


# Freeze the cohort choice before processing
def main():

    for path in (
        MODEL,
        RAW / "gene_reference.csv",
        S19 / "reactome_gene_pathway_weights.npz"
    ):
        if not path.exists():
            raise FileNotFoundError(
                f"Missing required file: {path}"
            )

    frozen = joblib.load(
        MODEL
    )

    if frozen.get("model") != "RNA_SVD8":
        raise RuntimeError(
            "Frozen model is not RNA_SVD8."
        )

    protocol = {
        "version": VERSION,
        "external_cohort": "GSE50081",
        "platform": "GPL570",
        "eligible_histologies": [
            "LUAD",
            "LUSC"
        ],
        "required_clinical_variables": [
            "age",
            "sex",
            "stage",
            "histology"
        ],
        "candidate_endpoint": (
            "Recurrence event with paired "
            "disease-free survival time"
        ),
        "training_endpoint": "TCGA-CDR PFI",
        "endpoint_equivalence_assumed": False,
        "frozen_model_sha256": digest(MODEL),
        "external_model_fitting": False,
        "external_performance_scoring": False
    }

    protocol_file = (
        REPORT / "frozen_external_cohort_protocol.json"
    )

    if protocol_file.exists():

        existing = json.loads(
            protocol_file.read_text(
                encoding="utf-8"
            )
        )

        if existing != protocol:
            raise RuntimeError(
                "Previously frozen protocol differs."
            )

    else:
        save_json(
            protocol_file,
            protocol
        )

    print("=" * 76)
    print(
        "EXTERNAL PHASE D - "
        "GSE50081 FEASIBILITY AUDIT"
    )
    print("=" * 76)

    print(
        "Single external cohort: GSE50081"
    )

    print(
        "Frozen model: RNA_SVD8"
    )

    print(
        "Outcome-based fitting and performance "
        "evaluation: NOT PERFORMED"
    )

    download_geo(
        SERIES_URL,
        SERIES_FILE
    )

    download_geo(
        ANNOT_URL,
        ANNOT_FILE
    )

    samples, probes, expression_stats = (
        parse_series_matrix(
            SERIES_FILE
        )
    )

    mapping, ambiguous = (
        parse_gpl570(
            ANNOT_FILE,
            probes
        )
    )

    genes = pd.read_csv(
        RAW / "gene_reference.csv"
    )

    if (
        "gene_symbol" not in genes.columns
        or len(genes) != 19944
    ):
        raise RuntimeError(
            "Training gene reference changed."
        )

    training_symbols = (
        genes.gene_symbol
        .astype(str)
        .str.upper()
        .str.strip()
    )

    available_symbols = set(
        mapping.values()
    )

    gene_coverage = (
        training_symbols
        .isin(available_symbols)
        .to_numpy(dtype=np.float64)
    )

    W = sparse.load_npz(
        S19 / "reactome_gene_pathway_weights.npz"
    ).tocsr()

    if W.shape != (1613, 19944):
        raise RuntimeError(
            "Reactome membership dimensions changed."
        )

    abs_W = W.copy()

    abs_W.data = np.abs(
        abs_W.data
    )

    pathway_size = np.asarray(
        abs_W.sum(axis=1)
    ).ravel()

    pathway_coverage = (
        np.asarray(
            abs_W @ gene_coverage
        ).ravel()
        / np.maximum(
            pathway_size,
            1e-12
        )
    )

    components = np.asarray(
        frozen["preprocessing"]["rna"]["svd"].components_
    )

    if components.shape != (8, 1613):
        raise RuntimeError(
            "Frozen SVD dimensions changed."
        )

    important_pathways = np.unique(
        np.argsort(
            -np.abs(components),
            axis=1
        )[:, :20].ravel()
    )

    # Identify clinical fields without assuming their presence.
    fields = set(samples.columns)

    selected = {
        "histology": pick_field(
            fields,
            ("histology", "cell type")
        ),
        "stage": pick_field(
            fields,
            (
                "stage",
                "pathological stage",
                "pathologic stage",
                "disease stage"
            )
        ),
        "age": pick_field(
            fields,
            ("age", "age years")
        ),
        "sex": pick_field(
            fields,
            ("sex", "gender")
        ),
        "recurrence": pick_field(
            fields,
            (
                "recurrence",
                "recurrence status",
                "relapse"
            )
        ),
        "dfs_time": pick_field(
            fields,
            (
                "disease free survival time",
                "disease free survival time months",
                "disease free survival months",
                "dfs time",
                "time to recurrence"
            )
        )
    }

    n = len(samples)

    hist = (
        samples[
            selected["histology"]
        ].map(classify_histology)
        if selected["histology"]
        else pd.Series(
            ["OTHER_OR_UNKNOWN"] * n
        )
    )

    stage = (
        samples[
            selected["stage"]
        ].map(parse_stage)
        if selected["stage"]
        else pd.Series(
            [np.nan] * n
        )
    )

    age = (
        samples[
            selected["age"]
        ].map(parse_number)
        if selected["age"]
        else pd.Series(
            [np.nan] * n
        )
    )

    sex = (
        samples[
            selected["sex"]
        ].astype(str).str.lower().str.strip()
        if selected["sex"]
        else pd.Series([""] * n)
    )

    recurrence = (
        samples[
            selected["recurrence"]
        ].astype(str).str.lower().str.strip()
        if selected["recurrence"]
        else pd.Series([""] * n)
    )

    dfs_time = (
        samples[
            selected["dfs_time"]
        ].map(parse_number)
        if selected["dfs_time"]
        else pd.Series(
            [np.nan] * n
        )
    )

    clinical_complete = (
        hist.isin(("LUAD", "LUSC"))
        & stage.isin((1, 2))
        & age.between(18, 105)
        & sex.isin(
            ("male", "female", "m", "f")
        )
    )

    known_recurrence = recurrence.isin(
        (
            "y", "n",
            "yes", "no",
            "1", "0",
            "true", "false"
        )
    )

    endpoint_pair_available = (
        known_recurrence
        & dfs_time.gt(0)
    )

    preliminary_eligible = (
        clinical_complete
        & endpoint_pair_available
    )

    report = {
        "version": VERSION,
        "created_utc": datetime.now(
            timezone.utc
        ).isoformat(),
        "cohort": "GSE50081",
        "model_sha256": digest(MODEL),
        "series_sha256": digest(SERIES_FILE),
        "annotation_sha256": digest(ANNOT_FILE),
        "expression_summary": expression_stats,
        "clinical_field_mapping": selected,
        "histology_counts": (
            hist.value_counts().to_dict()
        ),
        "recurrence_counts": (
            recurrence.value_counts().to_dict()
        ),
        "clinical_complete": int(
            clinical_complete.sum()
        ),
        "recurrence_time_pair_available": int(
            endpoint_pair_available.sum()
        ),
        "preliminary_eligible": int(
            preliminary_eligible.sum()
        ),
        "unambiguous_probes": len(mapping),
        "ambiguous_probes_excluded": ambiguous,
        "training_gene_coverage": float(
            gene_coverage.mean()
        ),
        "pathways_80pct_covered": int(
            (pathway_coverage >= 0.80).sum()
        ),
        "important_pathway_median_coverage": float(
            np.median(
                pathway_coverage[
                    important_pathways
                ]
            )
        ),
        "endpoint_time_unit_verified": False,
        "tcga_PFI_equivalent": False,
        "external_risk_scored": False,
        "model_updated": False
    }

    samples.assign(
        audit_histology=hist,
        audit_stage=stage,
        audit_clinical_complete=clinical_complete,
        audit_endpoint_pair_available=(
            endpoint_pair_available
        )
    ).to_csv(
        REPORT / "gse50081_metadata_audit.csv",
        index=False
    )

    pd.DataFrame({
        "pathway_index": np.arange(
            len(pathway_coverage)
        ),
        "gene_coverage": pathway_coverage
    }).to_csv(
        REPORT / "reactome_external_gene_coverage.csv",
        index=False
    )

    save_json(
        REPORT / "gse50081_feasibility_summary.json",
        report
    )

    print("\n" + "=" * 76)
    print(
        "EXTERNAL PHASE D FINAL - "
        "GSE50081 DATA COMPATIBILITY"
    )
    print("=" * 76)

    print(
        "Expression matrix:",
        expression_stats
    )

    print(
        "Detected clinical fields:",
        selected
    )

    print(
        "Histology:",
        report["histology_counts"]
    )

    print(
        "Recurrence metadata:",
        report["recurrence_counts"]
    )

    print(
        "Clinical-complete patients:",
        report["clinical_complete"]
    )

    print(
        "Usable recurrence/time pairs:",
        report["recurrence_time_pair_available"]
    )

    print(
        "Preliminary eligible patients:",
        report["preliminary_eligible"]
    )

    print(
        "Mapped training genes:",
        f"{int(gene_coverage.sum())}/19944",
        f"({gene_coverage.mean():.1%})"
    )

    print(
        "Pathways with >=80% coverage:",
        report["pathways_80pct_covered"],
        "/1613"
    )

    print(
        "Important pathway median coverage:",
        f"{report['important_pathway_median_coverage']:.1%}"
    )

    if not all(
        selected.values()
    ):
        print(
            "STATUS: Metadata fields require manual audit."
        )

    elif (
        report["preliminary_eligible"] < 50
        or report["training_gene_coverage"] < 0.65
    ):
        print(
            "STATUS: External compatibility needs review."
        )

    else:
        print(
            "STATUS: Ready for endpoint-unit and "
            "cross-platform inference protocol review."
        )

    print(
        "Endpoint time unit verified: NO"
    )

    print(
        "External C-index calculated: NO"
    )

    print(
        "Frozen model modified: NO"
    )

    print(
        "Saved:",
        REPORT
    )

    print("EXTERNAL PHASE D COMPLETED")


main()

EXTERNAL PHASE D - GSE50081 FEASIBILITY AUDIT
Single external cohort: GSE50081
Frozen model: RNA_SVD8
Outcome-based fitting and performance evaluation: NOT PERFORMED
Downloaded: GSE50081_series_matrix.txt.gz (51.3 MiB)
Downloaded: GPL570.annot.gz (8.1 MiB)

EXTERNAL PHASE D FINAL - GSE50081 DATA COMPATIBILITY
Expression matrix: {'samples': 181, 'probes': 54675, 'sampled_finite_fraction': 1.0, 'sampled_median': 4.714740341000001, 'sampled_min': 1.95714717, 'sampled_max': 13.71365215}
Detected clinical fields: {'histology': 'histology', 'stage': 'stage', 'age': 'age', 'sex': 'sex', 'recurrence': 'recurrence', 'dfs_time': 'disease free survival time'}
Histology: {'LUAD': 128, 'LUSC': 43, 'OTHER_OR_UNKNOWN': 10}
Recurrence metadata: {'n': 126, 'y': 51, 'u': 4}
Clinical-complete patients: 171
Usable recurrence/time pairs: 177
Preliminary eligible patients: 167
Mapped training genes: 16483/19944 (82.6%)
Pathways with >=80% coverage: 1483 /1613
Important pathway median coverage: 95.1%
STATUS:

In [32]:

import csv
import gzip
import hashlib
import json
import os
import re
import sys
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.stats import rankdata, spearmanr


# Outcome-blind external assay and transfer-compatibility audit.
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
EXT = ROOT / "data" / "external" / "GSE50081"
A = ROOT / "results" / "metrics" / "spare_phase_a"
D = ROOT / "results" / "metrics" / "gse50081_external_audit"
OUT = ROOT / "results" / "metrics" / "gse50081_transfer_readiness"
RAW = ROOT / "data" / "processed" / "gene_level"

MODEL = (
    ROOT / "models" / "checkpoints" / "stage26"
    / "stage26_training_only_final.joblib"
)

SERIES = EXT / "GSE50081_series_matrix.txt.gz"
ANNOT = EXT / "GPL570.annot.gz"
VERSION = "gse50081_transfer_readiness_v1"

if Path(sys.executable).resolve() != (
    ROOT / "envs" / "nsclc" / "python.exe"
).resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

OUT.mkdir(parents=True, exist_ok=True)


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def atomic_json(path, data):
    temp = path.with_name(path.name + ".tmp")

    temp.write_text(
        json.dumps(
            data,
            indent=2,
            allow_nan=False
        ),
        encoding="utf-8"
    )

    os.replace(temp, path)


def atomic_npz(path, **arrays):
    temp = path.with_name(
        path.stem + ".partial.npz"
    )

    np.savez_compressed(
        temp,
        **arrays
    )

    os.replace(temp, path)


def tabline(line):
    return next(
        csv.reader(
            [line.rstrip("\r\n")],
            delimiter="\t",
            quotechar='"'
        )
    )


# Parse GPL570, keeping only unambiguous probe-to-gene mappings.
def read_gpl(path):
    mapping = {}
    found = False

    with gzip.open(
        path,
        "rt",
        encoding="utf-8",
        errors="replace"
    ) as f:

        for line in f:
            if (
                line.startswith(("#", "!", "^"))
                or not line.strip()
            ):
                continue

            values = tabline(line)

            if not found:
                heads = [
                    re.sub(
                        r"[^a-z0-9]+",
                        " ",
                        value.lower()
                    ).strip()
                    for value in values
                ]

                if (
                    "id" in heads
                    and "gene symbol" in heads
                ):
                    pi = heads.index("id")
                    gi = heads.index("gene symbol")
                    found = True

                continue

            if len(values) <= max(pi, gi):
                continue

            probe = values[pi].strip()
            symbol = values[gi].strip().upper()

            if (
                not symbol
                or symbol in {
                    "---", "NA", "N/A", "NULL"
                }
                or "///" in symbol
                or ";" in symbol
                or " // " in symbol
                or not re.fullmatch(
                    r"[A-Z0-9][A-Z0-9_.-]*",
                    symbol
                )
            ):
                continue

            if (
                probe in mapping
                and mapping[probe] != symbol
            ):
                raise RuntimeError(
                    f"Conflicting GPL570 annotation: {probe}"
                )

            mapping[probe] = symbol

    if not found or len(mapping) < 10000:
        raise RuntimeError(
            "GPL570 gene annotations are unavailable."
        )

    return mapping


# Read the complete series matrix without changing sample order.
def read_expression(
    path,
    expected_samples,
    expected_probes
):
    metadata_accessions = None

    with gzip.open(
        path,
        "rt",
        encoding="utf-8",
        errors="replace"
    ) as f:

        for line in f:
            if line.startswith(
                "!Sample_geo_accession"
            ):
                metadata_accessions = (
                    tabline(line)[1:]
                )

            if line.startswith(
                "!series_matrix_table_begin"
            ):
                break

        else:
            raise RuntimeError(
                "GEO expression table was not found."
            )

        header = tabline(next(f))

        ids = [
            value.strip().strip('"')
            for value in header[1:]
        ]

        if (
            header[0].strip('"') != "ID_REF"
            or len(ids) != expected_samples
            or len(set(ids)) != len(ids)
            or ids != metadata_accessions
        ):
            raise RuntimeError(
                "GEO expression sample alignment failed."
            )

        expression = np.empty(
            (expected_probes, expected_samples),
            dtype=np.float32
        )

        probes = []
        reached_end = False

        for line in f:

            if line.startswith(
                "!series_matrix_table_end"
            ):
                reached_end = True
                break

            if not line.strip():
                continue

            probe, separator, values = (
                line.partition("\t")
            )

            if not separator:
                raise RuntimeError(
                    "Invalid expression row delimiter."
                )

            probe = probe.strip().strip('"')

            row = np.fromstring(
                values,
                sep="\t",
                dtype=np.float32
            )

            if len(row) != expected_samples:
                raise RuntimeError(
                    f"Invalid expression row: {probe}"
                )

            index = len(probes)

            if index >= expected_probes:
                raise RuntimeError(
                    "More expression probes than expected."
                )

            expression[index] = row
            probes.append(probe)

    if (
        not reached_end
        or len(probes) != expected_probes
        or len(set(probes)) != len(probes)
    ):
        raise RuntimeError(
            "Incomplete or duplicated GEO probe table."
        )

    if not np.isfinite(expression).all():
        raise RuntimeError(
            "External expression contains nonfinite values."
        )

    if not (
        0 < np.percentile(expression, 1)
        and np.percentile(expression, 99) < 25
    ):
        raise RuntimeError(
            "Unexpected GPL570 signal range."
        )

    return ids, probes, expression


# Apply the same frozen pathway projection with a fixed
# neutral-rank policy for unavailable microarray genes.
def label_free_pathway_transform(
    values,
    mask,
    W,
    rna_state
):
    if (
        values.ndim != 2
        or values.shape[1] != len(mask)
    ):
        raise RuntimeError(
            "Gene matrix and availability mask mismatch."
        )

    if not np.isfinite(
        values[:, mask]
    ).all():
        raise RuntimeError(
            "Measured gene signals are not finite."
        )

    count = int(mask.sum())

    centered = (
        rankdata(
            values[:, mask],
            axis=1,
            method="average"
        ).astype(np.float32)
        - (count + 1) / 2.0
    ) / count

    gene_ranks = np.zeros(
        values.shape,
        dtype=np.float32
    )

    gene_ranks[:, mask] = centered

    pathway = np.asarray(
        (W @ gene_ranks.T).T,
        dtype=np.float32
    )

    scaled = np.clip(
        rna_state["scaler"].transform(
            pathway
        ),
        -7,
        7
    ).astype(np.float32)

    latent = rna_state["svd"].transform(
        scaled
    )

    latent = np.clip(
        rna_state["latent_scaler"].transform(
            latent
        ),
        -7,
        7
    )

    if (
        latent.shape != (len(values), 8)
        or not np.isfinite(latent).all()
    ):
        raise RuntimeError(
            "Frozen SVD8 transfer produced invalid features."
        )

    return (
        pathway,
        latent.astype(np.float32)
    )


# Verify all existing sources and the previous audit.
needed = [
    SERIES,
    ANNOT,
    D / "gse50081_feasibility_summary.json",
    D / "gse50081_metadata_audit.csv",
    A / "phase_a_manifest.json",
    A / "full_training_reference.joblib",
    MODEL,
    RAW / "gene_reference.csv",
    RAW / "case_order.csv",
    RAW / "rna_tpm.npy",
    ROOT / "data" / "splits"
    / "locked_four_omics_v1.csv",
    ROOT / "results" / "metrics"
    / "stage19" / "reactome_gene_pathway_weights.npz"
]

for path in needed:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing verified project file: {path}"
        )

previous = json.loads(
    (
        D / "gse50081_feasibility_summary.json"
    ).read_text(encoding="utf-8")
)

phase_a = json.loads(
    (
        A / "phase_a_manifest.json"
    ).read_text(encoding="utf-8")
)

if (
    sha256(SERIES) != previous["series_sha256"]
    or sha256(ANNOT)
    != previous["annotation_sha256"]
    or sha256(MODEL)
    != previous["model_sha256"]
    or sha256(MODEL)
    != phase_a["stage26_model_sha256"]
):
    raise RuntimeError(
        "External sources or frozen model changed."
    )

if previous.get("preliminary_eligible") != 167:
    raise RuntimeError(
        "Unexpected Phase D eligibility count."
    )

frozen = joblib.load(MODEL)

if frozen.get("model") != "RNA_SVD8":
    raise RuntimeError(
        "The frozen model is not RNA_SVD8."
    )


# Preserve the clinical eligibility established in Phase D.
meta = pd.read_csv(
    D / "gse50081_metadata_audit.csv",
    dtype={"GSM": str}
)

if len(meta) != 181 or not meta.GSM.is_unique:
    raise RuntimeError(
        "Invalid GSE50081 metadata sample identities."
    )


def verified_bool(series):
    values = (
        series.astype(str)
        .str.strip()
        .str.lower()
    )

    if not values.isin(
        ["true", "false"]
    ).all():
        raise RuntimeError(
            "Invalid Phase D eligibility indicator."
        )

    return values.eq("true").to_numpy()


eligible = (
    verified_bool(
        meta["audit_clinical_complete"]
    )
    & verified_bool(
        meta["audit_endpoint_pair_available"]
    )
)

if int(eligible.sum()) != 167:
    raise RuntimeError(
        "Predeclared eligibility changed."
    )


# Audit survival metadata without fitting or scoring risks.
for field in (
    "histology",
    "stage",
    "age",
    "sex",
    "recurrence",
    "disease free survival time"
):
    if field not in meta.columns:
        raise RuntimeError(
            f"Missing GEO clinical field: {field}"
        )

time_raw = (
    meta["disease free survival time"]
    .astype(str)
    .str.strip()
)

time_num = pd.to_numeric(
    time_raw.str.extract(
        r"^\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))"
    )[0],
    errors="coerce"
)

units = (
    time_raw.str.extract(
        r"(?i)\b(days?|months?|years?)\b"
    )[0]
    .dropna()
    .str.lower()
    .unique()
    .tolist()
)

if not (
    time_num.to_numpy()[eligible] > 0
).all():
    raise RuntimeError(
        "Eligible DFS time is invalid or nonpositive."
    )

rec = (
    meta["recurrence"]
    .astype(str)
    .str.strip()
    .str.lower()
)

if not rec[eligible].isin(
    [
        "y", "n",
        "yes", "no",
        "1", "0",
        "true", "false"
    ]
).all():
    raise RuntimeError(
        "Ambiguous recurrence status entered eligibility."
    )

print("=" * 78)
print(
    "EXTERNAL PHASE E - "
    "OUTCOME-BLIND CROSS-PLATFORM TRANSFER AUDIT"
)
print("=" * 78)

print(
    "Selected external cohort: GSE50081"
)

print(
    "Predeclared eligible patients:",
    int(eligible.sum())
)

print(
    "Assay: GPL570 log2 RMA microarray."
)

print(
    "External outcomes will not be "
    "used for fitting or scoring."
)

print(
    "First 10 eligible DFS times:",
    time_raw[eligible].head(10).tolist()
)

print(
    "Explicit time units:",
    units if units else "NONE - UNVERIFIED"
)

print(
    "DFS raw time quantiles:",
    time_num[eligible].quantile(
        [0, .25, .5, .75, 1]
    ).round(3).to_dict()
)

print(
    "Eligible recurrence categories:",
    rec[eligible].value_counts().to_dict()
)

print(
    "Selected histology:",
    meta.loc[
        eligible, "audit_histology"
    ].value_counts().to_dict()
)

print(
    "Selected stage:",
    meta.loc[
        eligible, "audit_stage"
    ].value_counts().to_dict()
)

if "survival time" in meta.columns:
    os_time = pd.to_numeric(
        meta["survival time"],
        errors="coerce"
    )

    print(
        "OS raw time quantiles:",
        os_time[eligible].quantile(
            [0, .25, .5, .75, 1]
        ).round(3).to_dict()
    )


# Construct gene-level external expression.
gsm, probes, expr = read_expression(
    SERIES,
    181,
    54675
)

if gsm != meta.GSM.tolist():
    raise RuntimeError(
        "Expression and metadata sample order mismatch."
    )

probe_to_symbol = read_gpl(
    ANNOT
)

genes = pd.read_csv(
    RAW / "gene_reference.csv"
)

if (
    len(genes) != 19944
    or "gene_symbol" not in genes
):
    raise RuntimeError(
        "Training gene reference changed."
    )

symbols = (
    genes.gene_symbol
    .astype(str)
    .str.upper()
    .str.strip()
    .to_numpy()
)

valid_symbols = (
    set(symbols)
    - {"", "NAN", "NA", "NONE", "---"}
)

symbol_probes = defaultdict(list)

for i, probe in enumerate(probes):
    symbol = probe_to_symbol.get(
        probe
    )

    if symbol in valid_symbols:
        symbol_probes[symbol].append(
            i
        )

gene_expr = np.full(
    (181, 19944),
    np.nan,
    dtype=np.float32
)

for gene_idx, symbol in enumerate(
    symbols
):
    indices = symbol_probes.get(
        symbol
    )

    if indices:
        gene_expr[:, gene_idx] = (
            np.median(
                expr[indices, :],
                axis=0
            )
        )

mask = np.isfinite(
    gene_expr
).all(axis=0)

if not np.all(
    np.isnan(gene_expr[:, ~mask])
):
    raise RuntimeError(
        "Partial gene missingness needs separate review."
    )

covered = int(mask.sum())

if (
    covered < 15000
    or abs(covered - 16483) > 400
):
    raise RuntimeError(
        f"Unexpected gene coverage: {covered}/19944"
    )

print(
    "Mapped training genes:",
    f"{covered}/19944 ({covered/19944:.1%})"
)

print(
    "Duplicate training gene symbols:",
    int(
        pd.Series(symbols)
        .duplicated(keep=False)
        .sum()
    )
)

print(
    "Matched genes with multiple probes:",
    sum(
        len(indices) >= 2
        for indices in symbol_probes.values()
    )
)

print(
    "Expression 1st/99th percentiles:",
    np.round(
        np.quantile(expr, [.01, .99]),
        3
    ).tolist()
)

del expr


# Load the exact frozen Reactome and SVD8 transformation.
W = sparse.load_npz(
    ROOT / "results" / "metrics"
    / "stage19" / "reactome_gene_pathway_weights.npz"
).tocsr().astype(np.float32)

if (
    W.shape != (1613, 19944)
    or not np.isfinite(W.data).all()
):
    raise RuntimeError(
        "Frozen Reactome mapping changed."
    )

rna_state = frozen["preprocessing"]["rna"]

if np.asarray(
    rna_state["svd"].components_
).shape != (8, 1613):
    raise RuntimeError(
        "Frozen RNA representation is not SVD8."
    )


# Freeze one deterministic cross-platform input policy.
protocol = {
    "version": VERSION,
    "external_cohort": "GSE50081_only",
    "series_sha256": sha256(SERIES),
    "annotation_sha256": sha256(ANNOT),
    "frozen_checkpoint_sha256": sha256(MODEL),
    "gene_reference_sha256": sha256(
        RAW / "gene_reference.csv"
    ),
    "reactome_weights_sha256": sha256(
        ROOT / "results" / "metrics"
        / "stage19" / "reactome_gene_pathway_weights.npz"
    ),
    "mapping": (
        "GPL570 unambiguous gene-symbol annotation; "
        "median across probes per gene"
    ),
    "rna_scale": (
        "Published log2 RMA signals; "
        "no TPM conversion or external fitting"
    ),
    "ranking": (
        "Within-sample ranks among mapped genes, "
        "average ties, centered to [-0.5, 0.5]"
    ),
    "missing_genes": (
        "Neutral centered rank zero"
    ),
    "pathways": (
        "Original frozen Reactome mapping; "
        "no pathway-weight refitting"
    ),
    "projection": (
        "Frozen scaler, clip +/-7, SVD8, "
        "frozen latent scaler, clip +/-7"
    ),
    "clinical": (
        "Frozen clinical feature definitions and scaler"
    ),
    "eligibility": (
        "Original Phase D clinical completeness "
        "and paired recurrence/DFS availability"
    ),
    "endpoint": (
        "Recurrence with DFS time; "
        "not identical to TCGA-CDR PFI"
    ),
    "endpoint_time_units": "UNVERIFIED",
    "external_model_fitting": False,
    "external_performance_scoring": False
}

protocol_path = (
    OUT / "locked_transfer_protocol.json"
)

if protocol_path.exists():
    prior = json.loads(
        protocol_path.read_text(
            encoding="utf-8"
        )
    )

    if prior != protocol:
        raise RuntimeError(
            "Frozen transfer protocol was modified."
        )
else:
    atomic_json(
        protocol_path,
        protocol
    )


# Compute external features without using prediction coefficients.
external_pathway, external_z = (
    label_free_pathway_transform(
        gene_expr,
        mask,
        W,
        rna_state
    )
)

del gene_expr


# Verify the frozen clinical encoder.
clinical_state = frozen[
    "preprocessing"
]["clinical"]

median_age = float(
    clinical_state["age_median"]
)

age = pd.to_numeric(
    meta["age"].astype(str).str.extract(
        r"^\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))"
    )[0],
    errors="coerce"
).to_numpy(float)

sex = (
    meta["sex"]
    .astype(str)
    .str.strip()
    .str.lower()
    .to_numpy()
)

male = np.where(
    np.isin(sex, ["m", "male"]),
    1.0,
    np.where(
        np.isin(sex, ["f", "female"]),
        0.0,
        np.nan
    )
)

hist = (
    meta["audit_histology"]
    .astype(str)
    .to_numpy()
)

stage = pd.to_numeric(
    meta["audit_stage"],
    errors="coerce"
).to_numpy(float)

if (
    not np.isfinite(age[eligible]).all()
    or not np.isfinite(male[eligible]).all()
    or not np.isin(
        hist[eligible],
        ["LUAD", "LUSC"]
    ).all()
    or not np.isin(
        stage[eligible],
        [1, 2]
    ).all()
):
    raise RuntimeError(
        "External clinical variables cannot be "
        "encoded without assumptions."
    )

age_imputed = np.where(
    np.isfinite(age),
    age,
    median_age
)

raw_clinical = np.column_stack([
    (age_imputed - median_age) / 10.0,
    (~np.isfinite(age)).astype(float),
    np.where(
        np.isfinite(male),
        male,
        0.5
    ),
    (~np.isfinite(male)).astype(float),
    (hist == "LUSC").astype(float),
    (stage == 2).astype(float),
    (stage == 3).astype(float),
    (stage == 4).astype(float)
])

clinical_scaled = (
    clinical_state["scaler"]
    .transform(raw_clinical)
    .astype(np.float32)
)

if (
    clinical_scaled.shape != (181, 8)
    or not np.isfinite(
        clinical_scaled[eligible]
    ).all()
):
    raise RuntimeError(
        "Frozen clinical encoding failed."
    )


# Apply the external missing-gene mask to TCGA training RNA.
# This measures missing-feature distortion without external outcomes.
original_full = joblib.load(
    A / "full_training_reference.joblib"
)

if (
    original_full.get("signature")
    != phase_a["cache_signature"]
):
    raise RuntimeError(
        "Phase A training reference changed."
    )

cases = pd.read_csv(
    RAW / "case_order.csv",
    dtype={"case_id": str}
)

locked = pd.read_csv(
    ROOT / "data" / "splits"
    / "locked_four_omics_v1.csv",
    dtype={"case_id": str}
)

cohort = cases[
    ["case_id"]
].merge(
    locked[
        ["case_id", "split"]
    ],
    on="case_id",
    how="left",
    validate="one_to_one"
)

train_rows = np.flatnonzero(
    cohort.split.eq("train").to_numpy()
)

if len(train_rows) != 667:
    raise RuntimeError(
        "Original training cohort changed."
    )

fit_idx = np.asarray(
    original_full["fit_idx"],
    dtype=int
)

if (
    len(fit_idx) != 657
    or len(np.unique(fit_idx)) != 657
):
    raise RuntimeError(
        "Phase A training patient order changed."
    )

rna = np.load(
    RAW / "rna_tpm.npy",
    mmap_mode="r"
)

if rna.shape != (953, 19944):
    raise RuntimeError(
        "TCGA RNA source dimensions changed."
    )

train_values = np.asarray(
    rna[
        train_rows[fit_idx]
    ],
    dtype=np.float32
)

train_values = np.where(
    np.isfinite(train_values)
    & (train_values >= 0),
    train_values,
    0
).astype(np.float32)

_, train_mask_z = (
    label_free_pathway_transform(
        train_values,
        mask,
        W,
        rna_state
    )
)

reference_z = np.asarray(
    original_full["z_fit"],
    dtype=np.float32
)

if reference_z.shape != (657, 8):
    raise RuntimeError(
        "Original SVD8 training representation changed."
    )

coef = np.asarray(
    frozen["model_state"]["coef"],
    dtype=float
)

ref_molecular = reference_z @ coef
mask_molecular = train_mask_z @ coef

rank_stability = float(
    spearmanr(
        ref_molecular,
        mask_molecular
    ).statistic
)

if not np.isfinite(rank_stability):
    raise RuntimeError(
        "Missing-gene stability is undefined."
    )


# Compare unlabeled external features with mask-matched TCGA features.
z_mean = train_mask_z.mean(
    axis=0
)

z_sd = np.maximum(
    train_mask_z.std(axis=0),
    1e-6
)

normalized_mean_shift = np.abs(
    (
        external_z[eligible].mean(axis=0)
        - z_mean
    ) / z_sd
)

normalized_external_sd = (
    external_z[eligible].std(axis=0)
    / z_sd
)


# Save outcome-blind inputs for the later one-time evaluation.
feature_path = (
    OUT / "external_frozen_input_features.npz"
)

atomic_npz(
    feature_path,
    gsm=np.asarray(gsm),
    eligible=eligible.astype(np.uint8),
    gene_available=mask.astype(np.uint8),
    pathway_scores=external_pathway,
    z_svd8=external_z,
    clinical_scaled=clinical_scaled
)

summary = {
    "version": VERSION,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "total_samples": int(len(gsm)),
    "predeclared_eligible": int(
        eligible.sum()
    ),
    "eligible_recurrence_events": int(
        rec[eligible].isin(
            ["y", "yes", "1", "true"]
        ).sum()
    ),
    "dfs_raw_values_first_ten": (
        time_raw[eligible].head(10).tolist()
    ),
    "explicit_dfs_units": units,
    "dfs_time_units_verified": False,
    "gene_coverage": covered,
    "gene_coverage_fraction": float(
        mask.mean()
    ),
    "training_molecular_rank_correlation_after_gene_mask": (
        rank_stability
    ),
    "training_molecular_median_abs_shift_due_to_mask": float(
        np.median(
            np.abs(
                ref_molecular - mask_molecular
            )
        )
    ),
    "external_vs_masked_training_svd_mean_shift_in_training_sd": (
        normalized_mean_shift.tolist()
    ),
    "external_vs_masked_training_svd_sd_ratio": (
        normalized_external_sd.tolist()
    ),
    "feature_file": str(
        feature_path
    ),
    "feature_file_sha256": sha256(
        feature_path
    ),
    "external_risk_or_performance_computed": False,
    "external_outcomes_used_for_preprocessing": False,
    "notes": [
        "External DFS is not equivalent to TCGA-CDR PFI.",
        "External missing genes receive a neutral centered rank.",
        "Frozen survival coefficients and SVD are unchanged.",
        "Missing-gene stability is not predictive accuracy.",
        "Endpoint-unit verification precedes external scoring."
    ]
}

atomic_json(
    OUT / "phase_e_summary.json",
    summary
)


# Final audit report
print("\n" + "=" * 78)
print(
    "EXTERNAL PHASE E FINAL - "
    "FROZEN INPUT COMPATIBILITY"
)
print("=" * 78)

print(
    "Predeclared eligible:",
    int(eligible.sum())
)

print(
    "Recurrence events:",
    summary["eligible_recurrence_events"]
)

print(
    "Genes represented:",
    f"{covered}/19944 ({mask.mean():.1%})"
)

print(
    "Training molecular-risk rank stability "
    "after missing-gene mask:",
    f"{rank_stability:.4f}"
)

print(
    "Median absolute training RNA-risk shift "
    "due to missing genes:",
    f"{summary['training_molecular_median_abs_shift_due_to_mask']:.4f}"
)

print(
    "External vs mask-matched training SVD8 "
    "mean shift (training SD units):",
    np.round(
        normalized_mean_shift,
        2
    ).tolist()
)

print(
    "External vs mask-matched training "
    "SVD8 SD ratio:",
    np.round(
        normalized_external_sd,
        2
    ).tolist()
)

print(
    "DFS time units explicitly established:",
    "YES" if units else "NO"
)

print(
    "External outcome-based model "
    "performance computed: NO"
)

print(
    "Frozen checkpoint modified: NO"
)

print(
    "Saved:",
    OUT
)

print("EXTERNAL PHASE E COMPLETED")

EXTERNAL PHASE E - OUTCOME-BLIND CROSS-PLATFORM TRANSFER AUDIT
Selected external cohort: GSE50081
Predeclared eligible patients: 167
Assay: GPL570 log2 RMA microarray.
External outcomes will not be used for fitting or scoring.
First 10 eligible DFS times: ['7.521', '7.324', '6.842', '1.613', '7.034', '6.85', '4.156', '6.218', '3.743', '6.311']
Explicit time units: NONE - UNVERIFIED
DFS raw time quantiles: {0.0: 0.093, 0.25: 1.664, 0.5: 4.156, 0.75: 5.65, 1.0: 12.036}
Eligible recurrence categories: {'n': 119, 'y': 48}
Selected histology: {'LUAD': 125, 'LUSC': 42}
Selected stage: {1: 116, 2: 51}
OS raw time quantiles: {0.0: 0.09, 0.25: 2.065, 0.5: 4.79, 0.75: 5.84, 1.0: 12.04}
Mapped training genes: 16483/19944 (82.6%)
Duplicate training gene symbols: 12
Matched genes with multiple probes: 9750
Expression 1st/99th percentiles: [2.247, 11.464]

EXTERNAL PHASE E FINAL - FROZEN INPUT COMPATIBILITY
Predeclared eligible: 167
Recurrence events: 48
Genes represented: 16483/19944 (82.6%)
Traini

In [33]:

import hashlib
import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


# One-time external recurrence-related endpoint transfer evaluation.
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
D = ROOT / "results" / "metrics" / "gse50081_external_audit"
E = ROOT / "results" / "metrics" / "gse50081_transfer_readiness"
OUT = (
    ROOT / "results" / "metrics"
    / "gse50081_one_time_external_evaluation"
)

MODEL = (
    ROOT / "models" / "checkpoints" / "stage26"
    / "stage26_training_only_final.joblib"
)

VERSION = "gse50081_frozen_external_one_time_v1"
SEED = 20261121
BOOTSTRAPS = 2000

expected_python = ROOT / "envs" / "nsclc" / "python.exe"

if Path(sys.executable).resolve() != expected_python.resolve():
    raise RuntimeError(
        "Select the NST NSCLC Research (Python 3.12) kernel."
    )

OUT.mkdir(parents=True, exist_ok=True)


# File integrity and atomic output utilities
def file_hash(path):
    h = hashlib.sha256()

    with open(path, "rb") as stream:
        for block in iter(
            lambda: stream.read(4 * 1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def atomic_json(path, payload):
    temporary = path.with_suffix(".partial.json")

    temporary.write_text(
        json.dumps(
            payload,
            indent=2,
            allow_nan=False
        ),
        encoding="utf-8"
    )

    os.replace(temporary, path)


def atomic_csv(path, frame):
    temporary = path.with_suffix(".partial.csv")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, path)


# Harrell concordance using strict earlier event times
def cindex(time, event, risk):
    time = np.asarray(time, dtype=np.float64)
    event = np.asarray(event, dtype=bool)
    risk = np.asarray(risk, dtype=np.float64)

    comparable = (
        (time[:, None] < time[None, :])
        & event[:, None]
    )

    count = int(
        np.count_nonzero(comparable)
    )

    if count < 10:
        return np.nan

    delta = risk[:, None] - risk[None, :]

    concordant = np.count_nonzero(
        comparable & (delta > 1e-10)
    )

    tied = np.count_nonzero(
        comparable & (
            np.abs(delta) <= 1e-10
        )
    )

    return float(
        (concordant + 0.5 * tied) / count
    )


# Left-limit Kaplan-Meier estimator for censoring
def censoring_km_left(time, event):
    unique, inverse, counts = np.unique(
        time,
        return_inverse=True,
        return_counts=True
    )

    censored = np.bincount(
        inverse,
        weights=(1 - event),
        minlength=len(unique)
    )

    at_risk = (
        len(time)
        - np.r_[
            0,
            np.cumsum(counts[:-1])
        ]
    )

    left = np.empty(
        len(unique),
        dtype=np.float64
    )

    survival = 1.0

    for i, (n, d) in enumerate(
        zip(at_risk, censored)
    ):
        left[i] = survival

        survival *= max(
            0.0,
            1.0 - d / n
        )

    return left[inverse]


# Uno IPCW concordance at a fixed horizon
def uno_cindex(time, event, risk, tau):
    event = np.asarray(
        event,
        dtype=np.int64
    )

    g = censoring_km_left(
        time,
        event
    )

    selected = (
        (event == 1)
        & (time <= tau)
    )

    if (
        selected.sum() < 10
        or np.min(g[selected]) < 0.10
    ):
        return np.nan

    pair = (
        (time[:, None] < time[None, :])
        & selected[:, None]
    )

    weight = (
        selected.astype(float)
        / np.maximum(g, 1e-12) ** 2
    )

    weight_pairs = (
        weight[:, None] * pair
    )

    denominator = weight_pairs.sum()

    if denominator < 1e-10:
        return np.nan

    delta = risk[:, None] - risk[None, :]

    concordance = (
        (delta > 1e-10)
        + 0.5 * (
            np.abs(delta) <= 1e-10
        )
    )

    return float(
        np.sum(
            weight_pairs * concordance
        ) / denominator
    )


# Paired, patient-level bootstrap uncertainty
def paired_bootstrap(
    time,
    event,
    baseline,
    augmented,
    n_boot,
    seed
):
    rng = np.random.default_rng(seed)

    n = len(time)
    draws = []

    for _ in range(n_boot):
        index = rng.integers(
            0,
            n,
            size=n
        )

        base_c = cindex(
            time[index],
            event[index],
            baseline[index]
        )

        model_c = cindex(
            time[index],
            event[index],
            augmented[index]
        )

        if (
            np.isfinite(base_c)
            and np.isfinite(model_c)
        ):
            draws.append(
                (
                    base_c,
                    model_c,
                    model_c - base_c
                )
            )

    if len(draws) < max(
        200,
        n_boot * 0.80
    ):
        raise RuntimeError(
            "Insufficient valid paired bootstrap samples."
        )

    array = np.asarray(draws)

    return {
        "valid_resamples": len(draws),

        "clinical_C_95pct": np.quantile(
            array[:, 0],
            [0.025, 0.975]
        ).tolist(),

        "recap_C_95pct": np.quantile(
            array[:, 1],
            [0.025, 0.975]
        ).tolist(),

        "delta_C_95pct": np.quantile(
            array[:, 2],
            [0.025, 0.975]
        ).tolist()
    }


# Read verified Phase D/E artifacts
summary_d = (
    D / "gse50081_feasibility_summary.json"
)

summary_e = (
    E / "phase_e_summary.json"
)

protocol_e = (
    E / "locked_transfer_protocol.json"
)

meta_file = (
    D / "gse50081_metadata_audit.csv"
)

feature_file = (
    E / "external_frozen_input_features.npz"
)

needed = [
    MODEL,
    summary_d,
    summary_e,
    protocol_e,
    meta_file,
    feature_file
]

for path in needed:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing verified external audit artifact: {path}"
        )

d = json.loads(
    summary_d.read_text(
        encoding="utf-8"
    )
)

e = json.loads(
    summary_e.read_text(
        encoding="utf-8"
    )
)

p = json.loads(
    protocol_e.read_text(
        encoding="utf-8"
    )
)

if not (
    d["preliminary_eligible"]
    == e["predeclared_eligible"]
    == 167
    and e["eligible_recurrence_events"] == 48
    and d["model_sha256"] == file_hash(MODEL)
    and e["feature_file_sha256"] == file_hash(feature_file)
    and p["frozen_checkpoint_sha256"] == file_hash(MODEL)
    and p["external_performance_scoring"] is False
    and e["external_risk_or_performance_computed"] is False
):
    raise RuntimeError(
        "Pre-evaluation audit or frozen checkpoint mismatch."
    )


# Verify sample identity and the original eligibility mask
meta = pd.read_csv(
    meta_file,
    dtype={"GSM": str},
    keep_default_na=False
)

with np.load(
    feature_file,
    allow_pickle=False
) as archive:
    ids = archive["gsm"].astype(str)

    eligible = archive[
        "eligible"
    ].astype(bool)

    clinical_scaled = archive[
        "clinical_scaled"
    ].astype(np.float64)

    z_svd8 = archive[
        "z_svd8"
    ].astype(np.float64)

if (
    len(meta) != 181
    or not meta.GSM.is_unique
    or ids.tolist() != meta.GSM.tolist()
    or int(eligible.sum()) != 167
    or clinical_scaled.shape != (181, 8)
    or z_svd8.shape != (181, 8)
):
    raise RuntimeError(
        "GEO sample identity, inclusion, or feature shape mismatch."
    )

for field in (
    "audit_clinical_complete",
    "audit_endpoint_pair_available"
):
    values = (
        meta[field].astype(str)
        .str.lower()
        .str.strip()
    )

    if not values.isin(
        ("true", "false")
    ).all():
        raise RuntimeError(
            f"Invalid inclusion field: {field}"
        )

recomputed = (
    meta.audit_clinical_complete
    .astype(str).str.lower().eq("true")
    & meta.audit_endpoint_pair_available
    .astype(str).str.lower().eq("true")
).to_numpy()

if not np.array_equal(
    recomputed,
    eligible
):
    raise RuntimeError(
        "The frozen inclusion mask differs from Phase D."
    )


# Publication-supported units: years, not explicit GEO metadata.
# Recurrence status is not identical to TCGA-CDR PFI.
raw_time = (
    meta.loc[
        eligible,
        "disease free survival time"
    ]
    .astype(str)
    .str.strip()
)

if not raw_time.str.fullmatch(
    r"(?:\d+(?:\.\d*)?|\.\d+)"
).all():
    raise RuntimeError(
        "Unexpected DFS time values."
    )

time_years = raw_time.astype(
    float
).to_numpy()

rec_raw = (
    meta.loc[
        eligible,
        "recurrence"
    ]
    .astype(str)
    .str.lower()
    .str.strip()
)

if not rec_raw.isin(
    ("y", "n")
).all():
    raise RuntimeError(
        "Ambiguous recurrence status in eligible cohort."
    )

event = (
    rec_raw.eq("y")
    .to_numpy(dtype=np.int64)
)

histology = (
    meta.loc[
        eligible,
        "audit_histology"
    ]
    .astype(str)
    .to_numpy()
)

stage = pd.to_numeric(
    meta.loc[
        eligible,
        "audit_stage"
    ],
    errors="coerce"
).to_numpy()

if (
    not np.isfinite(
        time_years
    ).all()
    or not np.all(
        (time_years > 0)
        & (time_years < 25)
    )
    or int(event.sum()) != 48
    or not np.isin(
        histology,
        ("LUAD", "LUSC")
    ).all()
    or not np.isin(
        stage,
        (1, 2)
    ).all()
):
    raise RuntimeError(
        "External recurrence endpoint or cohort definition mismatch."
    )


# Retrieve the unchanged final Stage 26 coefficients
frozen = joblib.load(MODEL)

if frozen.get("model") != "RNA_SVD8":
    raise RuntimeError(
        "Incorrect frozen survival model."
    )

clinical_coef = np.asarray(
    frozen["preprocessing"]["clinical"]["beta"],
    dtype=float
)

molecular_coef = np.asarray(
    frozen["model_state"]["coef"],
    dtype=float
)

if (
    clinical_coef.shape != (8,)
    or molecular_coef.shape != (8,)
):
    raise RuntimeError(
        "Frozen coefficient dimensions changed."
    )

if not (
    np.isfinite(
        clinical_scaled[eligible]
    ).all()
    and np.isfinite(
        z_svd8[eligible]
    ).all()
):
    raise RuntimeError(
        "Nonfinite frozen external inputs."
    )


# Lock the one-time analysis plan before computing performance
protocol = {
    "version": VERSION,
    "external_cohort": "GSE50081_ONLY",

    "external_endpoint": (
        "Recurrence-labelled DFS follow-up; "
        "not TCGA-CDR PFI"
    ),

    "endpoint_years": (
        "Publication-supported inference; "
        "not explicit GEO sample metadata"
    ),

    "unit_evidence": [
        "https://doi.org/10.3389/fgene.2022.893511",
        "https://pmc.ncbi.nlm.nih.gov/articles/PMC11586294/"
    ],

    "eligible_N": 167,
    "eligible_events": 48,

    "inputs_sha256": {
        str(path): file_hash(path)
        for path in needed
    },

    "baseline": (
        "Stage26 frozen clinical Cox coefficients"
    ),

    "augmented": (
        "Stage26 frozen clinical Cox "
        "plus RNA-SVD8 coefficients"
    ),

    "preprocessing": (
        "Phase E frozen cross-platform neutral-rank policy"
    ),

    "primary_metric": (
        "Harrell C-index; strict earlier event times"
    ),

    "primary_comparison": (
        "Paired delta C = ReCAP - clinical Cox"
    ),

    "uncertainty": (
        f"{BOOTSTRAPS} patient-level paired "
        "percentile bootstrap replicates"
    ),

    "secondary_metrics": [
        "Uno IPCW C-index at 3 and 5 years",
        "LUAD/LUSC and stage subgroup C-index"
    ],

    "no_external_parameter_fitting": True,
    "no_outcome_dependent_modification": True,
    "seed": SEED
}

protocol_path = (
    OUT / "locked_one_time_evaluation_protocol.json"
)

if protocol_path.exists():
    existing = json.loads(
        protocol_path.read_text(
            encoding="utf-8"
        )
    )

    if existing != protocol:
        raise RuntimeError(
            "Locked external evaluation protocol cannot be changed."
        )

else:
    atomic_json(
        protocol_path,
        protocol
    )


# Do not rescore an already completed external evaluation
final_path = (
    OUT / "final_external_results.json"
)

if final_path.exists():
    result = json.loads(
        final_path.read_text(
            encoding="utf-8"
        )
    )

    if (
        result["protocol_sha256"]
        != file_hash(protocol_path)
    ):
        raise RuntimeError(
            "Previously completed evaluation differs."
        )

    print(
        "ONE-TIME EXTERNAL EVALUATION ALREADY COMPLETED."
    )

    print(
        "No models or outcomes were rescored."
    )

    print(
        json.dumps(
            result,
            indent=2
        )
    )

else:
    # Apply exactly the frozen clinical and molecular predictors.
    clinical_risk = (
        clinical_scaled[eligible]
        @ clinical_coef
    )

    molecular_risk = (
        z_svd8[eligible]
        @ molecular_coef
    )

    recap_risk = (
        clinical_risk + molecular_risk
    )

    if not (
        np.isfinite(
            clinical_risk
        ).all()
        and np.isfinite(
            recap_risk
        ).all()
    ):
        raise RuntimeError(
            "External frozen risk scores are not finite."
        )

    # Primary paired discrimination comparison.
    clinical_c = cindex(
        time_years,
        event,
        clinical_risk
    )

    recap_c = cindex(
        time_years,
        event,
        recap_risk
    )

    if not (
        np.isfinite(clinical_c)
        and np.isfinite(recap_c)
    ):
        raise RuntimeError(
            "External C-index could not be estimated."
        )

    bootstrap = paired_bootstrap(
        time_years,
        event,
        clinical_risk,
        recap_risk,
        BOOTSTRAPS,
        SEED
    )

    # Descriptive subgroup comparisons only.
    subgroups = []

    for variable, values, categories in (
        ("histology", histology, ("LUAD", "LUSC")),
        ("stage", stage, (1, 2))
    ):
        for category in categories:
            take = values == category

            cc = cindex(
                time_years[take],
                event[take],
                clinical_risk[take]
            )

            cm = cindex(
                time_years[take],
                event[take],
                recap_risk[take]
            )

            subgroups.append({
                "variable": variable,
                "category": str(category),

                "n": int(
                    take.sum()
                ),

                "events": int(
                    event[take].sum()
                ),

                "clinical_C": (
                    float(cc)
                    if np.isfinite(cc)
                    else None
                ),

                "recap_C": (
                    float(cm)
                    if np.isfinite(cm)
                    else None
                ),

                "delta_C": (
                    float(cm - cc)
                    if (
                        np.isfinite(cc)
                        and np.isfinite(cm)
                    )
                    else None
                )
            })

    # Secondary censoring-adjusted discrimination.
    uno = {}

    for tau in (3.0, 5.0):
        cc = uno_cindex(
            time_years,
            event,
            clinical_risk,
            tau
        )

        cm = uno_cindex(
            time_years,
            event,
            recap_risk,
            tau
        )

        uno[str(tau)] = {
            "clinical": (
                float(cc)
                if np.isfinite(cc)
                else None
            ),

            "recap": (
                float(cm)
                if np.isfinite(cm)
                else None
            ),

            "delta": (
                float(cm - cc)
                if (
                    np.isfinite(cc)
                    and np.isfinite(cm)
                )
                else None
            )
        }

    # Retain patient-level predictions and outcomes for audit.
    output = pd.DataFrame({
        "GSM": ids[eligible],
        "histology": histology,
        "stage": stage.astype(int),
        "recurrence_event": event,
        "followup_years": time_years,
        "clinical_logrisk": clinical_risk,
        "molecular_logrisk": molecular_risk,
        "recap_logrisk": recap_risk
    })

    atomic_csv(
        OUT / "external_frozen_patient_predictions.csv",
        output
    )

    result = {
        "version": VERSION,

        "completed_utc": datetime.now(
            timezone.utc
        ).isoformat(),

        "protocol_sha256": file_hash(
            protocol_path
        ),

        "cohort": "GSE50081",

        "outcome": (
            "Recurrence-labelled DFS follow-up; "
            "distinct from TCGA-CDR PFI"
        ),

        "n": int(
            len(time_years)
        ),

        "events": int(
            event.sum()
        ),

        "clinical_C": float(
            clinical_c
        ),

        "recap_C": float(
            recap_c
        ),

        "delta_C": float(
            recap_c - clinical_c
        ),

        "paired_patient_bootstrap": bootstrap,
        "uno_C_by_tau_years": uno,
        "subgroups": subgroups,

        "assay_shift_warning": (
            "Substantial cross-platform feature shift, "
            "including component 7 SD ratio about 0.03."
        ),

        "calibration": (
            "Not assessed: no verified transportable "
            "survival baseline for probability calibration."
        ),

        "interpretation": (
            "External cross-platform, related-endpoint "
            "transfer evaluation; not identical-PFI validation."
        ),

        "checkpoint_refitted": False,
        "external_outcomes_used_for_tuning": False,
        "single_external_cohort": True
    }

    atomic_json(
        final_path,
        result
    )

    print("=" * 76)
    print(
        "FINAL EXTERNAL PHASE F - "
        "ONE-TIME GSE50081 RESULT"
    )
    print("=" * 76)

    print(
        f"Eligible patients: {len(time_years)}; "
        f"recurrence events: {int(event.sum())}"
    )

    print(
        f"Frozen clinical Cox Harrell C: "
        f"{clinical_c:.4f}"
    )

    print(
        f"Frozen ReCAP-Cox Harrell C:   "
        f"{recap_c:.4f}"
    )

    print(
        f"Paired delta C:               "
        f"{recap_c - clinical_c:+.4f}"
    )

    print(
        "Bootstrap delta C 95% CI:",
        np.round(
            bootstrap["delta_C_95pct"],
            4
        ).tolist()
    )

    print(
        "Uno IPCW C at 3/5 years:",
        uno
    )

    print("\nSubgroup results:")

    print(
        pd.DataFrame(
            subgroups
        ).to_string(
            index=False,
            float_format=lambda x: f"{x:.4f}"
        )
    )

    print(
        "\nExternal outcomes used for tuning: NO"
    )

    print(
        "Frozen checkpoint modified: NO"
    )

    print(
        "RESULT LOCKED; DO NOT RETUNE "
        "AFTER THIS EVALUATION."
    )

    print(
        "Saved:",
        OUT
    )

FINAL EXTERNAL PHASE F - ONE-TIME GSE50081 RESULT
Eligible patients: 167; recurrence events: 48
Frozen clinical Cox Harrell C: 0.5721
Frozen ReCAP-Cox Harrell C:   0.6225
Paired delta C:               +0.0504
Bootstrap delta C 95% CI: [0.0017, 0.102]
Uno IPCW C at 3/5 years: {'3.0': {'clinical': 0.5589000185813173, 'recap': 0.6035599269883108, 'delta': 0.04465990840699352}, '5.0': {'clinical': 0.5947968516983025, 'recap': 0.6393941174800023, 'delta': 0.04459726578169976}}

Subgroup results:
 variable category   n  events  clinical_C  recap_C  delta_C
histology     LUAD 125      37      0.5801   0.6452   0.0650
histology     LUSC  42      11      0.5817   0.5788  -0.0029
    stage        1 116      29      0.5071   0.5826   0.0755
    stage        2  51      19      0.5559   0.6151   0.0592

External outcomes used for tuning: NO
Frozen checkpoint modified: NO
RESULT LOCKED; DO NOT RETUNE AFTER THIS EVALUATION.
Saved: D:\NST_NSCLC_PPI_GAT\results\metrics\gse50081_one_time_external_evalua

In [34]:

import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd


# Phase G: audit locked results without rescoring outcomes.
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
METRICS = ROOT / "results" / "metrics"

A = METRICS / "spare_phase_a"
B = METRICS / "spare_phase_b"
C = METRICS / "spare_phase_c"
D = METRICS / "gse50081_external_audit"
E = METRICS / "gse50081_transfer_readiness"
F = METRICS / "gse50081_one_time_external_evaluation"

OUT = METRICS / "recap_phase_g_manuscript_audit"

MODEL = (
    ROOT / "models" / "checkpoints"
    / "stage26" / "stage26_training_only_final.joblib"
)

VERSION = "recap_phase_g_locked_results_integrity_v1"

paths = {
    "model": MODEL,
    "phase_a": A / "phase_a_manifest.json",
    "phase_b": B / "spare_phase_b_summary.json",
    "phase_c": C / "phase_c_summary.json",
    "phase_d": D / "gse50081_feasibility_summary.json",
    "phase_e": E / "phase_e_summary.json",
    "phase_f": F / "final_external_results.json",
    "locked_protocol": (
        F / "locked_one_time_evaluation_protocol.json"
    ),
    "external_predictions": (
        F / "external_frozen_patient_predictions.csv"
    ),
    "external_features": (
        E / "external_frozen_input_features.npz"
    )
}

for name, path in paths.items():
    if not path.is_file():
        raise FileNotFoundError(f"{name}: {path}")


def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as handle:
        for block in iter(
            lambda: handle.read(4 * 1024 * 1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


def read_json(name):
    return json.loads(
        paths[name].read_text(encoding="utf-8")
    )


def require(condition, message):
    if not condition:
        raise RuntimeError(
            "PHASE G AUDIT FAILED: " + message
        )


phase_a = read_json("phase_a")
phase_b = read_json("phase_b")
phase_c = read_json("phase_c")
phase_d = read_json("phase_d")
phase_e = read_json("phase_e")
result = read_json("phase_f")
protocol = read_json("locked_protocol")


# Verify the original frozen model and external protocol.
require(
    result["protocol_sha256"]
    == sha256(paths["locked_protocol"]),
    "External analysis protocol hash mismatch"
)

require(
    phase_a["stage26_model_sha256"] == sha256(MODEL),
    "Stage 26 model changed"
)

require(
    phase_d["model_sha256"] == sha256(MODEL),
    "External model provenance mismatch"
)

require(
    protocol["eligible_N"] == result["n"] == 167,
    "External patient count mismatch"
)

require(
    protocol["eligible_events"]
    == result["events"] == 48,
    "External event count mismatch"
)

require(
    protocol["no_external_parameter_fitting"] is True,
    "External fitting is not excluded by the protocol"
)

require(
    result["checkpoint_refitted"] is False
    and result["external_outcomes_used_for_tuning"] is False,
    "External refitting or tuning flag mismatch"
)

require(
    result["single_external_cohort"] is True
    and result["cohort"] == "GSE50081",
    "External cohort lock mismatch"
)

require(
    abs(
        result["recap_C"]
        - result["clinical_C"]
        - result["delta_C"]
    ) < 1e-10,
    "External delta-C arithmetic mismatch"
)


# Verify development benchmarks remain unchanged.
require(
    abs(
        phase_a["recap_mean_site_C"] - 0.6549
    ) < 0.0006,
    "Training ReCAP benchmark mismatch"
)

require(
    abs(
        phase_b["fixed_recap_mean_C"]
        - phase_a["recap_mean_site_C"]
    ) < 1e-8,
    "Phase B baseline mismatch"
)

require(
    abs(
        phase_c["recap_mean_C"]
        - phase_a["recap_mean_site_C"]
    ) < 1e-8,
    "Phase C baseline mismatch"
)


# All inputs named in the locked protocol must be unchanged.
for source_path, expected_hash in (
    protocol["inputs_sha256"].items()
):
    path = Path(source_path)

    require(
        path.is_file(),
        "Protocol source missing: " + source_path
    )

    require(
        sha256(path) == expected_hash,
        "Protocol source changed: " + source_path
    )

require(
    sha256(paths["external_features"])
    == phase_e["feature_file_sha256"],
    "External feature archive changed"
)

ci = result[
    "paired_patient_bootstrap"
]["delta_C_95pct"]

require(
    len(ci) == 2
    and all(np.isfinite(ci))
    and ci[0] < ci[1],
    "Invalid saved bootstrap interval"
)


# Reconstruct frozen risk values, without using outcomes.
with np.load(
    paths["external_features"],
    allow_pickle=False
) as data:

    ids = data["gsm"].astype(str)
    included = data["eligible"].astype(bool)

    clinical_input = np.asarray(
        data["clinical_scaled"][included],
        dtype=float
    )

    molecular_input = np.asarray(
        data["z_svd8"][included],
        dtype=float
    )

frozen = joblib.load(MODEL)

require(
    frozen.get("model") == "RNA_SVD8",
    "Unexpected frozen model"
)

clinical_beta = np.asarray(
    frozen["preprocessing"]["clinical"]["beta"],
    dtype=float
)

molecular_beta = np.asarray(
    frozen["model_state"]["coef"],
    dtype=float
)

require(
    clinical_input.shape
    == molecular_input.shape
    == (167, 8),
    "Frozen feature shape mismatch"
)

require(
    clinical_beta.shape
    == molecular_beta.shape
    == (8,),
    "Frozen coefficient shape mismatch"
)

saved = pd.read_csv(
    paths["external_predictions"],
    dtype={"GSM": str}
)

require(
    len(saved) == 167
    and saved.GSM.is_unique,
    "External prediction identities invalid"
)

require(
    saved.GSM.tolist()
    == ids[included].tolist(),
    "External patient order mismatch"
)

require(
    int(saved.recurrence_event.sum()) == 48,
    "Patient-level event count mismatch"
)

require(
    (saved.followup_years > 0).all(),
    "Invalid external follow-up values"
)

expected_clinical = (
    clinical_input @ clinical_beta
)

expected_molecular = (
    molecular_input @ molecular_beta
)

errors = {
    "clinical_logrisk": float(
        np.max(
            np.abs(
                saved.clinical_logrisk.to_numpy(float)
                - expected_clinical
            )
        )
    ),
    "molecular_logrisk": float(
        np.max(
            np.abs(
                saved.molecular_logrisk.to_numpy(float)
                - expected_molecular
            )
        )
    ),
    "recap_logrisk": float(
        np.max(
            np.abs(
                saved.recap_logrisk.to_numpy(float)
                - expected_clinical
                - expected_molecular
            )
        )
    )
}

require(
    max(errors.values()) < 1e-6,
    f"Frozen external risk mismatch: {errors}"
)


# Verify that published subgroup counts are consistent.
subgroups = {}

for variable in (
    "histology",
    "stage"
):
    groups = [
        item
        for item in result["subgroups"]
        if item["variable"] == variable
    ]

    require(
        sum(item["n"] for item in groups) == 167
        and sum(
            item["events"]
            for item in groups
        ) == 48,
        f"{variable} subgroup counts mismatch"
    )

    subgroups[variable] = {
        str(item["category"]): {
            "n": item["n"],
            "events": item["events"],
            "delta_C": item["delta_C"]
        }
        for item in groups
    }


# Build the manuscript's matched discrimination table.
# TCGA values are previously reported, not rescored here.
historical = [
    (
        "TCGA site-held-out CV",
        0.6317, 0.6549
    ),
    (
        "TCGA frozen validation",
        0.6339, 0.6437
    ),
    (
        "TCGA sealed test",
        0.5863, 0.6207
    ),
    (
        "GSE50081 external DFS",
        result["clinical_C"],
        result["recap_C"]
    )
]

table = pd.DataFrame(
    historical,
    columns=[
        "Evaluation",
        "Clinical Cox",
        "ReCAP-Cox"
    ]
)

table["Delta C"] = (
    table["ReCAP-Cox"]
    - table["Clinical Cox"]
).round(4)


# Record both supportive and negative findings.
conclusions = {
    "phase_b": (
        "Nonlinear residual, low-rank interaction, "
        "and site-consistency modules did not "
        "improve ReCAP-Cox in development CV."
    ),
    "phase_c": (
        "Reactome representation outperformed "
        "gene-level rank-SVD8, but performed "
        "similarly to shuffled gene-pathway mappings."
    ),
    "external": (
        "The frozen ReCAP-Cox model improved "
        "discrimination over frozen clinical Cox "
        "in the single GSE50081 cohort."
    ),
    "endpoint_caveat": (
        "Recurrence-labelled DFS is not identical "
        "to TCGA-CDR PFI. Follow-up years are "
        "literature-supported, not explicitly "
        "verified in GEO sample metadata."
    ),
    "platform_caveat": (
        "RNA-seq TPM to GPL570 log2 RMA introduces "
        "substantial latent-feature distribution shift."
    ),
    "publication_caveat": (
        "Only one external cohort and 48 events; "
        "limited LUSC subgroup, no external calibration, "
        "and no demonstrated curated Reactome-specific "
        "or superior neural architecture contribution."
    )
}


# Prepare a reproducible, read-only result package.
OUT.mkdir(
    parents=True,
    exist_ok=True
)

payload = {
    "version": VERSION,
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "external_result_sha256": sha256(
        paths["phase_f"]
    ),
    "model_sha256": sha256(MODEL),
    "external_n": result["n"],
    "external_events": result["events"],
    "external_delta_C": result["delta_C"],
    "external_delta_C_95pct": ci,
    "locked_risk_max_absolute_errors": errors,
    "subgroups": subgroups,
    "phase_b_selected_delta_C": (
        phase_b["selected_delta_C"]
    ),
    "phase_c_recap_minus_gene": (
        phase_c["recap_minus_gene"]
    ),
    "phase_c_recap_minus_shuffled": (
        phase_c["recap_minus_shuffled"]
    ),
    "source_sha256": {
        name: sha256(path)
        for name, path in paths.items()
    },
    "notes": conclusions,
    "result_rescored": False,
    "model_refitted": False
}


def atomic_text(path, content):
    temp = path.with_name(
        path.name + ".tmp"
    )
    temp.write_text(
        content,
        encoding="utf-8"
    )
    os.replace(temp, path)


table_md = "\n".join([
    "| Evaluation | Clinical Cox | ReCAP-Cox | Delta C |",
    "|---|---:|---:|---:|",
    *[
        (
            f"| {row[0]} | {row[1]:.4f} | "
            f"{row[2]:.4f} | {row[3]:+.4f} |"
        )
        for row in table.itertuples(
            index=False,
            name=None
        )
    ]
])

md = f"""# ReCAP-Cox — Locked Research Evidence

## Discrimination Results

{table_md}

TCGA validation and test values are previously reported
locked results. This audit does not recompute them.

## GSE50081 External Evaluation

Patients: {result['n']}
Recurrence events: {result['events']}

Clinical Cox Harrell C: {result['clinical_C']:.4f}
ReCAP-Cox Harrell C: {result['recap_C']:.4f}
Paired improvement: {result['delta_C']:+.4f}
95% bootstrap CI: [{ci[0]:.4f}, {ci[1]:.4f}]

## Scientific Findings and Limitations

""" + "\n".join(
    f"- **{key.replace('_', ' ').title()}:** {value}"
    for key, value in conclusions.items()
)

md += (
    "\n\nFrozen score and provenance checks: PASS.\n"
    "No outcome-performance metric was recomputed "
    "during Phase G.\n"
)

atomic_text(
    OUT / "locked_results_evidence_summary.md",
    md
)

atomic_text(
    OUT / "phase_g_integrity_summary.json",
    json.dumps(
        payload,
        indent=2,
        allow_nan=False
    )
)

table.to_csv(
    OUT / "matched_discrimination_table.csv",
    index=False
)


# Final integrity report
print("=" * 76)
print(
    "PHASE G - LOCKED EXTERNAL "
    "RESULT INTEGRITY VERIFIED"
)
print("=" * 76)

print(
    table.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}"
    )
)

print(
    "External paired delta C 95% CI:",
    [round(float(x), 4) for x in ci]
)

print(
    "Frozen prediction maximum discrepancy:",
    f"{max(errors.values()):.2e}"
)

print(
    "Phase B selected minus ReCAP:",
    f"{phase_b['selected_delta_C']:+.4f}"
)

print(
    "Phase C Reactome minus shuffled:",
    f"{phase_c['recap_minus_shuffled']:+.4f}"
)

print(
    "External result recalculated: NO"
)

print(
    "Frozen model modified: NO"
)

print("Saved:", OUT)
print("PHASE G COMPLETED")

PHASE G - LOCKED EXTERNAL RESULT INTEGRITY VERIFIED
            Evaluation  Clinical Cox  ReCAP-Cox  Delta C
 TCGA site-held-out CV        0.6317     0.6549   0.0232
TCGA frozen validation        0.6339     0.6437   0.0098
      TCGA sealed test        0.5863     0.6207   0.0344
 GSE50081 external DFS        0.5721     0.6225   0.0504
External paired delta C 95% CI: [0.0017, 0.102]
Frozen prediction maximum discrepancy: 2.22e-16
Phase B selected minus ReCAP: -0.0024
Phase C Reactome minus shuffled: -0.0002
External result recalculated: NO
Frozen model modified: NO
Saved: D:\NST_NSCLC_PPI_GAT\results\metrics\recap_phase_g_manuscript_audit
PHASE G COMPLETED


In [38]:

from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from matplotlib.font_manager import FontProperties, findfont

from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


# Phase H: publication figures and two monochrome Word tables.
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
METRICS = ROOT / "results" / "metrics"
OUT = ROOT / "results" / "manuscript_assets" / "phase_h"

OUT.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "phase_a": METRICS / "spare_phase_a" / "phase_a_manifest.json",
    "phase_b": METRICS / "spare_phase_b" / "spare_phase_b_summary.json",
    "phase_c": METRICS / "spare_phase_c" / "phase_c_summary.json",
    "phase_d": METRICS / "gse50081_external_audit" / "gse50081_feasibility_summary.json",
    "phase_e": METRICS / "gse50081_transfer_readiness" / "phase_e_summary.json",
    "phase_f": METRICS / "gse50081_one_time_external_evaluation" / "final_external_results.json",
    "phase_g": METRICS / "recap_phase_g_manuscript_audit" / "phase_g_integrity_summary.json",
    "summary_csv": METRICS / "recap_phase_g_manuscript_audit" / "matched_discrimination_table.csv",
}

for name, path in SOURCES.items():
    if not path.is_file():
        raise FileNotFoundError(f"Missing research artifact [{name}]: {path}")


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def load_json(name):
    with open(SOURCES[name], "r", encoding="utf-8") as f:
        return json.load(f)


a, b, c, d, e, f, g = (
    load_json(k)
    for k in (
        "phase_a", "phase_b", "phase_c",
        "phase_d", "phase_e", "phase_f", "phase_g"
    )
)

perf = pd.read_csv(SOURCES["summary_csv"])

expected_columns = {
    "Evaluation", "Clinical Cox", "ReCAP-Cox", "Delta C"
}

expected_rows = [
    "TCGA site-held-out CV",
    "TCGA frozen validation",
    "TCGA sealed test",
    "GSE50081 external DFS",
]

if set(perf.columns) != expected_columns or len(perf) != 4:
    raise RuntimeError("Unexpected locked performance table schema.")

if perf["Evaluation"].astype(str).tolist() != expected_rows:
    raise RuntimeError("Locked evaluation row order changed.")

for column in ("Clinical Cox", "ReCAP-Cox", "Delta C"):
    perf[column] = pd.to_numeric(perf[column], errors="raise")

if not np.isfinite(
    perf[["Clinical Cox", "ReCAP-Cox", "Delta C"]].to_numpy()
).all():
    raise RuntimeError("Non-finite performance values.")

if not np.allclose(
    perf["ReCAP-Cox"] - perf["Clinical Cox"],
    perf["Delta C"],
    atol=1.1e-4,
):
    raise RuntimeError("Locked performance arithmetic mismatch.")

if g["external_result_sha256"] != sha256(SOURCES["phase_f"]):
    raise RuntimeError("External results differ from the Phase G audit.")

if abs(
    float(perf.iloc[-1]["ReCAP-Cox"]) - float(f["recap_C"])
) > 1e-6:
    raise RuntimeError("External performance mismatch.")

if abs(
    float(a["recap_mean_site_C"]) -
    float(perf.iloc[0]["ReCAP-Cox"])
) > 1.1e-4:
    raise RuntimeError("Training baseline changed.")

if (
    f["n"] != 167
    or f["events"] != 48
    or d["preliminary_eligible"] != 167
):
    raise RuntimeError("External cohort counts changed.")


# Enforce Times New Roman in all figure exports.
try:
    font_path = findfont(
        FontProperties(family="Times New Roman"),
        fallback_to_default=False,
    )
except Exception as exc:
    raise RuntimeError(
        "Times New Roman was not found in the system fonts."
    ) from exc

if "times" not in Path(font_path).name.lower():
    raise RuntimeError("Incorrect Times New Roman font resolution.")


# Journal figure typography and color system.
NAVY = "#244A70"
TEAL = "#087E80"
CORAL = "#D06B46"
AMBER = "#D69A36"
PURPLE = "#70639A"

INK = "#24313D"
GRID = "#D9E2EA"
PALE = "#F5F8FA"

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 10.3,
    "axes.labelsize": 10.5,
    "xtick.labelsize": 9.2,
    "ytick.labelsize": 9.2,
    "legend.fontsize": 9.0,
    "axes.edgecolor": "#8495A4",
    "axes.linewidth": 0.65,
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
    "figure.dpi": 150,
})


def export_figure(fig, filename):
    for extension in ("pdf", "svg", "png"):
        fig.savefig(
            OUT / f"{filename}.{extension}",
            dpi=600 if extension == "png" else 300,
            bbox_inches="tight",
            pad_inches=0.12,
            facecolor="white",
        )
    plt.close(fig)


def format_axis(ax, grid_axis="x"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(
        axis=grid_axis,
        color=GRID,
        linewidth=0.65,
        alpha=0.85,
    )
    ax.tick_params(length=0, pad=5)


def diagram_box(
    ax, x, y, w, h, text,
    background, border, fontsize=9.2,
    weight="normal",
):
    patch = FancyBboxPatch(
        (x, y),
        w,
        h,
        boxstyle="round,pad=0.008,rounding_size=0.018",
        transform=ax.transAxes,
        facecolor=background,
        edgecolor=border,
        linewidth=1.15,
        clip_on=False,
    )

    ax.add_patch(patch)

    ax.text(
        x + w / 2,
        y + h / 2,
        text,
        ha="center",
        va="center",
        transform=ax.transAxes,
        fontsize=fontsize,
        fontweight=weight,
        color=INK,
        linespacing=1.25,
    )


def diagram_arrow(ax, start, end, color=NAVY):
    ax.add_patch(
        FancyArrowPatch(
            start,
            end,
            transform=ax.transAxes,
            arrowstyle="-|>",
            mutation_scale=12,
            linewidth=1.2,
            color=color,
            shrinkA=1,
            shrinkB=1,
            clip_on=False,
        )
    )


# Figure 1: study design and independent cohort allocation.
fig, ax = plt.subplots(figsize=(7.15, 3.75))
ax.axis("off")

diagram_box(
    ax, .34, .77, .32, .16,
    "TCGA LUAD + LUSC\nn = 953",
    PALE, NAVY, 11, "bold",
)

for center in (.17, .50, .83):
    diagram_arrow(ax, (.50, .76), (center, .63))

diagram_box(
    ax, .035, .40, .27, .21,
    "Development cohort\n667 assigned\n657 PFI eligible\n225 events",
    "#ECF3FA", NAVY, 9.3,
)

diagram_box(
    ax, .365, .40, .27, .21,
    "Internal validation\n143 assigned\n140 PFI eligible\n50 events",
    "#E9F5F2", TEAL, 9.3,
)

diagram_box(
    ax, .695, .40, .27, .21,
    "One-time TCGA test\n143 assigned\n138 PFI eligible\n51 events",
    "#FCF2E7", CORAL, 9.3,
)

ax.text(
    .5, .31,
    "Independent external cohort",
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontweight="bold",
    fontsize=10,
    color=NAVY,
)

diagram_box(
    ax, .24, .07, .52, .18,
    "GSE50081  |  GPL570 microarray\n"
    "181 samples  |  167 eligible  |  48 recurrence events\n"
    "Related DFS endpoint (not TCGA-CDR PFI)",
    "#F3F0FA", PURPLE, 9.6,
)

export_figure(fig, "Figure_1_cohort_flow")


# Figure 2: frozen ReCAP-Cox modeling workflow.
fig, ax = plt.subplots(figsize=(7.2, 3.65))
ax.axis("off")

diagram_box(
    ax, .025, .67, .185, .19,
    "RNA expression\n19,944 genes",
    "#ECF3FA", NAVY, 10,
)

diagram_box(
    ax, .275, .67, .175, .19,
    "Within-sample\ngene ranks",
    "#ECF3FA", NAVY, 10,
)

diagram_box(
    ax, .515, .67, .18, .19,
    "Reactome scores\n1,613 pathways",
    "#ECF3FA", NAVY, 10,
)

diagram_box(
    ax, .755, .67, .20, .19,
    "Frozen projection\nglobal SVD (8)",
    "#ECF3FA", NAVY, 10,
)

for start, end in (
    (.21, .275),
    (.45, .515),
    (.695, .755),
):
    diagram_arrow(ax, (start, .765), (end, .765))

diagram_box(
    ax, .11, .29, .29, .19,
    "Clinical covariates\nfixed Cox log-risk",
    "#E9F5F2", TEAL, 10,
)

diagram_box(
    ax, .59, .29, .30, .19,
    "Ridge-penalized Cox\nmolecular correction",
    "#FCF1E8", CORAL, 10,
)

diagram_arrow(ax, (.855, .665), (.74, .49), NAVY)
diagram_arrow(ax, (.255, .28), (.43, .13), TEAL)
diagram_arrow(ax, (.74, .28), (.57, .13), CORAL)

diagram_box(
    ax, .35, .035, .30, .14,
    "Combined log-risk\nclinical + RNA",
    "#F0F2F7", PURPLE, 10, "bold",
)

export_figure(fig, "Figure_2_model_workflow")


# Figure 3: matched C-index and incremental discrimination.
clinical_values = perf["Clinical Cox"].to_numpy(float)
recap_values = perf["ReCAP-Cox"].to_numpy(float)
delta_values = recap_values - clinical_values

external_ci = np.asarray(
    f["paired_patient_bootstrap"]["delta_C_95pct"],
    dtype=float,
)

if external_ci.shape != (2,) or not np.isfinite(external_ci).all():
    raise RuntimeError("External paired confidence interval is invalid.")

positions = np.arange(4)[::-1]

display_labels = [
    "TCGA development CV",
    "TCGA validation",
    "TCGA sealed test",
    "GSE50081 external",
]

fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(7.25, 3.5),
    gridspec_kw={"width_ratios": [1.25, 1]},
)

for position, clinical_c, recap_c in zip(
    positions, clinical_values, recap_values
):
    ax1.plot(
        [clinical_c, recap_c],
        [position, position],
        color="#A8B8C7",
        linewidth=2.5,
        zorder=1,
    )

ax1.scatter(
    clinical_values,
    positions,
    s=54,
    color=NAVY,
    label="Clinical Cox",
    zorder=3,
)

ax1.scatter(
    recap_values,
    positions,
    s=62,
    color=TEAL,
    marker="D",
    label="ReCAP-Cox",
    zorder=3,
)

ax1.set_yticks(positions, display_labels)
ax1.set_xlim(.50, .77)
ax1.set_ylim(-.55, 3.55)
ax1.set_xlabel("Harrell C-index")

format_axis(ax1)

ax1.legend(
    loc="lower center",
    bbox_to_anchor=(.55, 1.02),
    frameon=False,
    ncol=2,
    handletextpad=.4,
    columnspacing=1.2,
)

ax1.text(
    -.08, 1.045, "A",
    transform=ax1.transAxes,
    fontsize=12,
    fontweight="bold",
)

ax2.axvline(
    0,
    color="#8793A0",
    linewidth=1,
    linestyle="--",
)

ax2.scatter(
    delta_values,
    positions,
    s=58,
    color=[TEAL, TEAL, TEAL, CORAL],
    zorder=3,
)

external_error = np.array([
    [delta_values[-1] - external_ci[0]],
    [external_ci[1] - delta_values[-1]],
])

ax2.errorbar(
    delta_values[-1],
    positions[-1],
    xerr=external_error,
    fmt="none",
    ecolor=CORAL,
    elinewidth=2,
    capsize=4,
    zorder=2,
)

ax2.set_xlim(-.035, .115)
ax2.set_ylim(-.55, 3.55)
ax2.set_yticks(positions, ["", "", "", ""])
ax2.set_xlabel("Incremental C-index (ReCAP - clinical)")

format_axis(ax2)

ax2.text(
    .98, .96,
    "External 95% CI only",
    transform=ax2.transAxes,
    fontsize=8.5,
    ha="right",
    va="top",
)

ax2.text(
    -.08, 1.045, "B",
    transform=ax2.transAxes,
    fontsize=12,
    fontweight="bold",
)

fig.subplots_adjust(
    left=.23,
    right=.98,
    top=.78,
    bottom=.21,
    wspace=.22,
)

export_figure(fig, "Figure_3_discrimination")


# Figure 4: external subgroup and IPCW concordance panels.
subgroups = [
    r for r in f["subgroups"]
    if r["variable"] == "histology"
] + [
    r for r in f["subgroups"]
    if r["variable"] == "stage"
]

if len(subgroups) != 4:
    raise RuntimeError("Unexpected external subgroup schema.")

group_names = []
group_deltas = []

for row in subgroups:
    name = (
        str(row["category"])
        if row["variable"] == "histology"
        else f'Stage {row["category"]}'
    )

    group_names.append(
        f'{name}  (n={row["n"]}, events={row["events"]})'
    )

    group_deltas.append(float(row["delta_C"]))

uno = f["uno_C_by_tau_years"]

if not all(key in uno for key in ("3.0", "5.0")):
    raise RuntimeError("Missing external Uno C-index values.")

time_points = [3, 5]

clinical_uno = [
    float(uno[str(float(t))]["clinical"])
    for t in time_points
]

recap_uno = [
    float(uno[str(float(t))]["recap"])
    for t in time_points
]

fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(7.25, 3.25),
    gridspec_kw={"width_ratios": [1.23, 1]},
)

y_groups = np.arange(4)[::-1]

ax1.axvline(
    0,
    color="#8999A5",
    linewidth=1,
    linestyle="--",
)

ax1.scatter(
    group_deltas,
    y_groups,
    c=[
        TEAL if value >= 0 else CORAL
        for value in group_deltas
    ],
    s=68,
    zorder=3,
)

ax1.set_yticks(y_groups, group_names)
ax1.set_xlim(-.025, .105)
ax1.set_ylim(-.5, 3.5)
ax1.set_xlabel("Subgroup incremental C-index")

format_axis(ax1)

ax1.text(
    -.08, 1.06, "A",
    transform=ax1.transAxes,
    fontsize=12,
    fontweight="bold",
)

ax2.plot(
    time_points,
    clinical_uno,
    marker="o",
    markersize=7,
    linewidth=1.8,
    color=NAVY,
    label="Clinical Cox",
)

ax2.plot(
    time_points,
    recap_uno,
    marker="D",
    markersize=6.5,
    linewidth=1.8,
    color=TEAL,
    label="ReCAP-Cox",
)

ax2.set_xticks(time_points, ["3 years", "5 years"])
ax2.set_xlim(2.65, 5.35)

ax2.set_ylim(
    max(.45, min(clinical_uno + recap_uno) - .055),
    min(.75, max(clinical_uno + recap_uno) + .055),
)

ax2.set_ylabel("Uno IPCW C-index")

format_axis(ax2, "y")

ax2.legend(
    loc="lower right",
    frameon=False,
)

ax2.text(
    -.10, 1.06, "B",
    transform=ax2.transAxes,
    fontsize=12,
    fontweight="bold",
)

fig.subplots_adjust(
    left=.275,
    right=.97,
    top=.84,
    bottom=.25,
    wspace=.37,
)

export_figure(fig, "Figure_4_external_evaluation")


# Figure S1: compact supplementary comparison.
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(7.1, 2.9),
)

names_a = [
    "ReCAP",
    "MLP residual",
    "Low-rank",
    "Full SPARE",
]

values_a = [
    float(a["recap_mean_site_C"]),
    float(b["ablation_mean_C"]["mlp_residual"]),
    float(b["ablation_mean_C"]["lowrank_interaction"]),
    float(b["ablation_mean_C"]["spare_full"]),
]

names_b = [
    "Gene SVD8",
    "Reactome SVD8",
    "Shuffled map",
]

values_b = [
    float(c["gene_rank_svd8_mean_C"]),
    float(c["recap_mean_C"]),
    float(c["shuffled_pathway_mean_C"]),
]

panel_data = [
    (
        ax1,
        names_a,
        values_a,
        [TEAL, NAVY, PURPLE, CORAL],
        "A",
    ),
    (
        ax2,
        names_b,
        values_b,
        [NAVY, TEAL, AMBER],
        "B",
    ),
]

for ax, names, values, colors, letter in panel_data:
    y_loc = np.arange(len(names))

    ax.barh(
        y_loc,
        values,
        color=colors,
        height=.56,
    )

    ax.set_yticks(y_loc, names)
    ax.invert_yaxis()
    ax.set_xlim(0, .75)
    ax.set_xlabel("Mean site-held-out C-index")

    ax.text(
        -.09, 1.055, letter,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
    )

    format_axis(ax)

    for i, value in enumerate(values):
        ax.text(
            value + .01,
            i,
            f"{value:.4f}",
            va="center",
            fontsize=9,
        )

fig.subplots_adjust(
    left=.18,
    right=.97,
    top=.83,
    bottom=.23,
    wspace=.43,
)

export_figure(fig, "Figure_S1_development_controls")


# Generate exactly two black-and-white Word tables.
doc = Document()
section = doc.sections[0]

section.page_width = Inches(8.5)
section.page_height = Inches(11)

section.top_margin = Inches(.75)
section.bottom_margin = Inches(.72)
section.left_margin = Inches(.72)
section.right_margin = Inches(.72)

normal = doc.styles["Normal"]
normal.font.name = "Times New Roman"
normal.font.size = Pt(10)
normal.font.color.rgb = RGBColor(0, 0, 0)
normal.paragraph_format.space_after = Pt(4)


def style_cell(
    cell,
    value,
    bold=False,
    alignment=WD_ALIGN_PARAGRAPH.LEFT,
):
    cell.text = ""

    paragraph = cell.paragraphs[0]
    paragraph.alignment = alignment
    paragraph.paragraph_format.space_before = Pt(0)
    paragraph.paragraph_format.space_after = Pt(0)

    run = paragraph.add_run(str(value))
    run.bold = bold
    run.font.name = "Times New Roman"
    run.font.size = Pt(9)
    run.font.color.rgb = RGBColor(0, 0, 0)

    cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

    tcpr = cell._tc.get_or_add_tcPr()

    margins = tcpr.first_child_found_in("w:tcMar")

    if margins is None:
        margins = OxmlElement("w:tcMar")
        tcpr.append(margins)

    for side in ("top", "start", "bottom", "end"):
        node = margins.find(qn("w:" + side))

        if node is None:
            node = OxmlElement("w:" + side)
            margins.append(node)

        node.set(qn("w:w"), "100")
        node.set(qn("w:type"), "dxa")

    borders = tcpr.first_child_found_in("w:tcBorders")

    if borders is None:
        borders = OxmlElement("w:tcBorders")
        tcpr.append(borders)

    for side in (
        "top", "left", "bottom",
        "right", "insideH", "insideV"
    ):
        border = borders.find(qn("w:" + side))

        if border is None:
            border = OxmlElement("w:" + side)
            borders.append(border)

        border.set(qn("w:val"), "single")
        border.set(qn("w:sz"), "5")
        border.set(qn("w:color"), "000000")


def add_caption(text):
    paragraph = doc.add_paragraph()

    paragraph.paragraph_format.space_before = Pt(6)
    paragraph.paragraph_format.space_after = Pt(5)
    paragraph.paragraph_format.keep_with_next = True

    run = paragraph.add_run(text)
    run.bold = True
    run.font.name = "Times New Roman"
    run.font.size = Pt(10)
    run.font.color.rgb = RGBColor(0, 0, 0)


def add_note(text):
    paragraph = doc.add_paragraph()

    paragraph.paragraph_format.space_before = Pt(3)
    paragraph.paragraph_format.space_after = Pt(10)

    run = paragraph.add_run(text)
    run.font.name = "Times New Roman"
    run.font.size = Pt(8.5)
    run.font.color.rgb = RGBColor(0, 0, 0)


def add_grid_table(headers, records, widths):
    table = doc.add_table(
        rows=1,
        cols=len(headers),
    )

    table.style = "Table Grid"
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False

    for j, (header, width) in enumerate(
        zip(headers, widths)
    ):
        table.columns[j].width = Inches(width)

        cell = table.rows[0].cells[j]
        cell.width = Inches(width)

        style_cell(
            cell,
            header,
            bold=True,
            alignment=(
                WD_ALIGN_PARAGRAPH.CENTER
                if j else WD_ALIGN_PARAGRAPH.LEFT
            ),
        )

    for record in records:
        cells = table.add_row().cells

        for j, (value, width) in enumerate(
            zip(record, widths)
        ):
            cells[j].width = Inches(width)

            style_cell(
                cells[j],
                value,
                alignment=(
                    WD_ALIGN_PARAGRAPH.CENTER
                    if j else WD_ALIGN_PARAGRAPH.LEFT
                ),
            )

    return table


# Table 1: cohort characteristics.
add_caption(
    "Table 1. Cohort allocation and endpoint eligibility"
)

cohort_rows = [
    (
        "TCGA development",
        "667", "657", "225",
        "PFI; RNA-seq",
    ),
    (
        "TCGA validation",
        "143", "140", "50",
        "PFI; RNA-seq",
    ),
    (
        "TCGA sealed test",
        "143", "138", "51",
        "PFI; RNA-seq",
    ),
    (
        "GSE50081 external",
        "181", "167", "48",
        "Recurrence-labelled DFS; microarray",
    ),
]

add_grid_table(
    [
        "Cohort",
        "Assigned, n",
        "Eligible, n",
        "Events, n",
        "Endpoint and assay",
    ],
    cohort_rows,
    [1.56, 1.00, 1.00, .92, 2.58],
)

add_note(
    "PFI, progression-free interval. "
    "GSE50081 has a related recurrence-labelled DFS "
    "endpoint, not an identical TCGA-CDR PFI endpoint."
)


# Table 2: frozen clinical and molecular performance.
add_caption(
    "Table 2. Clinical-only and ReCAP-Cox discrimination"
)

performance_rows = []

short_names = [
    "TCGA development CV",
    "TCGA validation",
    "TCGA sealed test",
    "GSE50081 external",
]

for i, row in perf.iterrows():
    ci = (
        f"[{external_ci[0]:+.4f}, {external_ci[1]:+.4f}]"
        if i == 3 else "—"
    )

    performance_rows.append((
        short_names[i],
        f'{row["Clinical Cox"]:.4f}',
        f'{row["ReCAP-Cox"]:.4f}',
        f'{row["Delta C"]:+.4f}',
        ci,
    ))

add_grid_table(
    [
        "Evaluation",
        "Clinical Cox C",
        "ReCAP-Cox C",
        "Delta C",
        "95% CI for delta C",
    ],
    performance_rows,
    [2.02, 1.04, 1.04, .80, 2.16],
)

add_note(
    "C, Harrell concordance; delta C = ReCAP-Cox "
    "minus clinical Cox. The displayed 95% CI is a "
    "paired patient-bootstrap interval for GSE50081. "
    "Development results are exploratory."
)

word_path = OUT / "Main_Tables_1_to_2.docx"
doc.save(word_path)


# Keep figure legends separate from exported figures.
captions = {
    "Figure 1": (
        "Study cohorts and endpoint-eligible sample flow. "
        "The external cohort is independent and uses a related, "
        "not identical, survival endpoint."
    ),
    "Figure 2": (
        "Schematic of the frozen ReCAP-Cox clinical-anchor "
        "and low-rank transcriptomic risk correction. "
        "Model coefficients are not refitted on external outcomes."
    ),
    "Figure 3": (
        "Clinical-only versus ReCAP-Cox Harrell concordance "
        "in development and locked evaluations (A), and paired "
        "differences (B). The error bar denotes the external "
        "95% patient-bootstrap interval."
    ),
    "Figure 4": (
        "Descriptive GSE50081 subgroup differences in "
        "concordance (A) and time-restricted Uno concordance "
        "at 3 and 5 years (B). The LUSC subgroup contains "
        "only 11 events."
    ),
    "Figure S1": (
        "Exploratory development comparisons for additional "
        "residual architectures (A) and pathway representation "
        "controls (B). These comparisons are not independent "
        "external confirmation."
    ),
}

caption_text = "\n\n".join(
    f"{key}. {value}"
    for key, value in captions.items()
)

(OUT / "Figure_Captions.txt").write_text(
    caption_text,
    encoding="utf-8",
)


# Verify exports and record immutable source provenance.
figure_names = [
    "Figure_1_cohort_flow",
    "Figure_2_model_workflow",
    "Figure_3_discrimination",
    "Figure_4_external_evaluation",
    "Figure_S1_development_controls",
]

for name in figure_names:
    for extension in ("pdf", "svg", "png"):
        file = OUT / f"{name}.{extension}"

        if not file.is_file() or file.stat().st_size < 1000:
            raise RuntimeError(
                f"Figure export failed: {file}"
            )

if len(doc.tables) != 2:
    raise RuntimeError(
        "The Word file must contain exactly two tables."
    )

manifest = {
    "source_sha256": {
        key: sha256(path)
        for key, path in SOURCES.items()
    },
    "model": "Frozen ReCAP-Cox",
    "main_figures": 4,
    "supplementary_figures": 1,
    "tables": 2,
    "figure_font": "Times New Roman",
    "figure_formats": ["PDF", "SVG", "PNG 600 DPI"],
    "figure_titles_embedded": False,
    "tables_font": "Times New Roman",
    "tables_colors": "Black and white only",
    "tables_borders": "Full black grid",
    "survival_performance_recalculated": False,
}

(OUT / "phase_h_artifact_manifest.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("=" * 76)
print("PHASE H - PUBLICATION FIGURES AND TABLES GENERATED")
print("=" * 76)

print("Main figures: 4")
print("Supplementary figures: 1")
print("Figure formats: PDF, SVG, PNG (600 DPI)")
print("Figure font: Times New Roman")
print("Figure titles embedded: NO")

print("\nWord tables: 2")
print("Table font: Times New Roman")
print("Table colors: black and white")
print("Table borders: complete black grid")

print("\nExternal performance recalculated: NO")
print("Frozen model modified: NO")

print("\nOutput directory:")
print(OUT)

print("\nGenerated files:")
for file in sorted(OUT.iterdir()):
    print(" ", file.name)

print("\nPHASE H COMPLETED")

PHASE H - PUBLICATION FIGURES AND TABLES GENERATED
Main figures: 4
Supplementary figures: 1
Figure formats: PDF, SVG, PNG (600 DPI)
Figure font: Times New Roman
Figure titles embedded: NO

Word tables: 2
Table font: Times New Roman
Table colors: black and white
Table borders: complete black grid

External performance recalculated: NO
Frozen model modified: NO

Output directory:
D:\NST_NSCLC_PPI_GAT\results\manuscript_assets\phase_h

Generated files:
  Figure_1_cohort_flow.pdf
  Figure_1_cohort_flow.png
  Figure_1_cohort_flow.svg
  Figure_2_model_workflow.pdf
  Figure_2_model_workflow.png
  Figure_2_model_workflow.svg
  Figure_3_discrimination.pdf
  Figure_3_discrimination.png
  Figure_3_discrimination.svg
  Figure_4_external_evaluation.pdf
  Figure_4_external_evaluation.png
  Figure_4_external_evaluation.svg
  Figure_Captions.txt
  Figure_S1_development_controls.pdf
  Figure_S1_development_controls.png
  Figure_S1_development_controls.svg
  Main_Tables_1_to_2.docx
  phase_h_artifact_ma

In [37]:
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT

In [39]:

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties, findfont

from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import (
    WD_TABLE_ALIGNMENT,
    WD_CELL_VERTICAL_ALIGNMENT,
)
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


# Phase H2: result-focused manuscript figures and tables.
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
M = ROOT / "results" / "metrics"
OUT = ROOT / "results" / "manuscript_assets" / "phase_h_results"
OUT.mkdir(parents=True, exist_ok=True)

FILES = {
    "phase_a": M / "spare_phase_a" / "phase_a_manifest.json",
    "phase_b": M / "spare_phase_b" / "spare_phase_b_summary.json",
    "phase_c": M / "spare_phase_c" / "phase_c_summary.json",
    "phase_e": M / "gse50081_transfer_readiness" / "phase_e_summary.json",
    "phase_f": M / "gse50081_one_time_external_evaluation" / "final_external_results.json",
    "phase_g": M / "recap_phase_g_manuscript_audit" / "phase_g_integrity_summary.json",
    "performance": M / "recap_phase_g_manuscript_audit" / "matched_discrimination_table.csv",
    "cv_folds": M / "spare_phase_a" / "recap_reproduced_outer_folds.csv",
    "external_patients": M / "gse50081_one_time_external_evaluation" / "external_frozen_patient_predictions.csv",
}

for key, path in FILES.items():
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing saved research result [{key}]: {path}"
        )


def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(
            lambda: fh.read(4 * 1024 * 1024), b""
        ):
            h.update(chunk)
    return h.hexdigest()


def read_json(key):
    return json.loads(
        FILES[key].read_text(encoding="utf-8")
    )


a, b, c, e, f, g = (
    read_json(k)
    for k in (
        "phase_a", "phase_b", "phase_c",
        "phase_e", "phase_f", "phase_g"
    )
)

perf = pd.read_csv(FILES["performance"])

folds = pd.read_csv(
    FILES["cv_folds"]
).sort_values("fold")

patients = pd.read_csv(
    FILES["external_patients"],
    dtype={"GSM": str},
)


# Verify locked research results before plotting.
assert (
    g["external_result_sha256"]
    == digest(FILES["phase_f"])
), "External result lock mismatch"

assert perf["Evaluation"].tolist() == [
    "TCGA site-held-out CV",
    "TCGA frozen validation",
    "TCGA sealed test",
    "GSE50081 external DFS",
], "Unexpected evaluation rows"

assert (
    len(folds) == 4
    and folds.fold.tolist() == [1, 2, 3, 4]
)

assert abs(
    folds.recap_svd8_C.mean()
    - a["recap_mean_site_C"]
) < 6e-4

assert abs(
    float(perf.iloc[-1]["ReCAP-Cox"])
    - f["recap_C"]
) < 1.1e-4

assert (
    len(patients) == f["n"] == 167
    and patients.GSM.is_unique
)

assert (
    int(patients.recurrence_event.sum())
    == f["events"] == 48
)

assert np.isfinite(
    patients[
        [
            "followup_years",
            "clinical_logrisk",
            "recap_logrisk",
        ]
    ].to_numpy()
).all()

assert (patients.followup_years > 0).all()

assert set(
    patients.recurrence_event.unique()
).issubset({0, 1})

assert np.allclose(
    patients.clinical_logrisk
    + patients.molecular_logrisk,
    patients.recap_logrisk,
    atol=1e-8,
)

assert np.isclose(
    float(perf.iloc[-1]["Delta C"]),
    f["delta_C"],
    atol=1.1e-4,
)

assert (
    f["checkpoint_refitted"] is False
    and f["external_outcomes_used_for_tuning"] is False
)


# Enforce Times New Roman.
try:
    resolved = findfont(
        FontProperties(family="Times New Roman"),
        fallback_to_default=False,
    )

    if "times" not in Path(resolved).name.lower():
        raise RuntimeError("Incorrect resolved font")

except Exception as exc:
    raise RuntimeError(
        "Times New Roman must be installed on Windows."
    ) from exc


# Publication figure style.
NAVY = "#274A70"
TEAL = "#087E83"
CORAL = "#D46B49"
GOLD = "#C88C28"
PURPLE = "#7462A2"

INK = "#263746"
LIGHT = "#DEE6EC"

COLORS = [TEAL, GOLD, CORAL]

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 10,
    "axes.labelsize": 10.3,
    "xtick.labelsize": 8.8,
    "ytick.labelsize": 9,
    "legend.fontsize": 8.8,
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.linewidth": 0.75,
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})


def clean_axis(ax, axis="x"):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(
        axis=axis,
        color=LIGHT,
        linewidth=0.65,
        zorder=0,
    )
    ax.set_axisbelow(True)
    ax.tick_params(length=0, pad=5)


def save_figure(fig, stem):
    for ext in ("pdf", "svg", "png"):
        fig.savefig(
            OUT / f"{stem}.{ext}",
            dpi=600 if ext == "png" else 300,
            bbox_inches="tight",
            pad_inches=0.10,
            facecolor="white",
        )
    plt.close(fig)


# ============================================================
# FIGURE 1 - Locked discrimination and paired improvements
# ============================================================

clinical_c = perf["Clinical Cox"].to_numpy(float)
recap_c = perf["ReCAP-Cox"].to_numpy(float)

ci = np.asarray(
    f["paired_patient_bootstrap"]["delta_C_95pct"],
    dtype=float,
)

assert (
    ci.shape == (2,)
    and np.isfinite(ci).all()
    and ci[0] < f["delta_C"] < ci[1]
)

labels = [
    "TCGA site-held-out CV",
    "TCGA validation",
    "TCGA sealed test",
    "GSE50081 external",
]

fig, (ax, bx) = plt.subplots(
    1, 2,
    figsize=(8.2, 3.55),
    gridspec_kw={"width_ratios": [1.15, 1]},
)

y = np.arange(4)[::-1]

for yi, x1, x2 in zip(
    y, clinical_c, recap_c
):
    ax.plot(
        [x1, x2],
        [yi, yi],
        color="#A8B8C5",
        linewidth=2.7,
        zorder=1,
    )

ax.scatter(
    clinical_c, y,
    s=62,
    color=NAVY,
    label="Clinical Cox",
    zorder=3,
)

ax.scatter(
    recap_c, y,
    s=68,
    marker="D",
    color=TEAL,
    label="ReCAP-Cox",
    zorder=3,
)

ax.set_yticks(y, labels)
ax.set_ylim(-0.45, 3.45)
ax.set_xlim(0.49, 0.76)
ax.set_xlabel("Harrell C-index")

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 1.18),
    frameon=False,
    ncol=2,
)

ax.text(
    -0.13, 1.10, "A",
    transform=ax.transAxes,
    fontweight="bold",
    fontsize=12,
)

clean_axis(ax)


# Fold-level effects and external bootstrap uncertainty.
fold_delta = folds["delta_C"].to_numpy(float)

for yi, value in zip(
    [4, 3, 2, 1], fold_delta
):
    bx.plot(
        [0, value],
        [yi, yi],
        color="#C1D0D9",
        linewidth=2,
        zorder=1,
    )

    bx.scatter(
        value, yi,
        s=54,
        color=TEAL,
        zorder=3,
    )

bx.errorbar(
    f["delta_C"],
    0,
    xerr=[
        [f["delta_C"] - ci[0]],
        [ci[1] - f["delta_C"]],
    ],
    fmt="D",
    color=CORAL,
    ecolor=CORAL,
    elinewidth=1.9,
    capsize=4,
    markersize=6.5,
)

bx.axvline(
    0,
    linestyle="--",
    color="#7A8793",
    linewidth=1,
)

bx.set_yticks(
    [4, 3, 2, 1, 0],
    [
        "CV fold 1",
        "CV fold 2",
        "CV fold 3",
        "CV fold 4",
        "GSE50081 external",
    ],
)

bx.set_ylim(-0.5, 4.5)
bx.set_xlim(-0.015, 0.115)
bx.set_xticks([0, 0.025, 0.05, 0.075, 0.10])

bx.set_xlabel(
    "Paired improvement (delta C)"
)

bx.text(
    0.98, 0.97,
    "95% CI: external only",
    transform=bx.transAxes,
    ha="right",
    va="top",
    fontsize=8,
)

bx.text(
    -0.12, 1.10, "B",
    transform=bx.transAxes,
    fontweight="bold",
    fontsize=12,
)

clean_axis(bx)

fig.subplots_adjust(
    left=0.22,
    right=0.98,
    bottom=0.20,
    top=0.80,
    wspace=0.48,
)

save_figure(
    fig,
    "Figure_1_locked_discrimination",
)


# ============================================================
# FIGURE 2 - External Kaplan-Meier risk stratification
# ============================================================

# Risk groups use frozen predictions only, not outcomes.
def tertiles(values):
    values = np.asarray(values, dtype=float)

    order = np.argsort(
        values,
        kind="mergesort",
    )

    group = np.empty(
        len(values),
        dtype=np.int64,
    )

    group[order] = (
        3 * np.arange(len(values))
        // len(values)
    )

    return group


def km_curve(time, event, tmax):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event, dtype=int)

    event_times = np.unique(
        time[
            (event == 1)
            & (time <= tmax)
        ]
    )

    x = [0.0]
    survival = [1.0]
    s = 1.0

    for t in event_times:
        at_risk = int(
            np.sum(time >= t)
        )

        failures = int(
            np.sum(
                (time == t)
                & (event == 1)
            )
        )

        if at_risk <= 0:
            raise RuntimeError(
                "Invalid Kaplan-Meier risk set."
            )

        s *= 1.0 - failures / at_risk

        x.append(float(t))
        survival.append(float(s))

    x.append(float(tmax))
    survival.append(float(s))

    return (
        np.asarray(x),
        np.asarray(survival),
    )


T = patients.followup_years.to_numpy(float)
E = patients.recurrence_event.to_numpy(int)

X_MAX = 8.0
TICKS = np.array([0, 2, 4, 6, 8])

fig = plt.figure(figsize=(8.0, 4.65))

gs = fig.add_gridspec(
    2, 2,
    height_ratios=[4.0, 1.12],
    hspace=0.38,
    wspace=0.26,
)

for panel, variable in enumerate(
    ["clinical_logrisk", "recap_logrisk"]
):
    ax = fig.add_subplot(gs[0, panel])
    rax = fig.add_subplot(gs[1, panel])

    groups = tertiles(
        patients[variable].to_numpy(float)
    )

    group_labels = [
        "Low risk",
        "Medium risk",
        "High risk",
    ]

    for j, (label, color) in enumerate(
        zip(group_labels, COLORS)
    ):
        use = groups == j

        x, surv = km_curve(
            T[use],
            E[use],
            X_MAX,
        )

        ax.step(
            x, surv,
            where="post",
            color=color,
            linewidth=2.0,
            label=f"{label} (n={int(use.sum())})",
        )

    ax.set_xlim(0, X_MAX)
    ax.set_ylim(0, 1.03)
    ax.set_xticks(TICKS)

    ax.set_ylabel(
        "Recurrence-free probability"
        if panel == 0 else ""
    )

    ax.set_xlabel("Follow-up (years)")

    ax.legend(
        loc="lower left",
        frameon=False,
        fontsize=8.3,
        handlelength=2,
    )

    ax.text(
        -0.09, 1.07,
        "A" if panel == 0 else "B",
        transform=ax.transAxes,
        fontweight="bold",
        fontsize=12,
    )

    clean_axis(ax, "both")

    # Publication-style numbers-at-risk table.
    rax.axis("off")

    risk_columns = (
        0.37, 0.52, 0.67, 0.82, 0.97
    )

    rax.text(
        0.01, 0.90,
        "At risk",
        transform=rax.transAxes,
        fontsize=8.4,
    )

    for xloc, tick in zip(
        risk_columns, TICKS
    ):
        rax.text(
            xloc, 0.90,
            str(int(tick)),
            transform=rax.transAxes,
            ha="center",
            fontsize=8.2,
        )

    for j, (label, color) in enumerate(
        zip(["Low", "Medium", "High"], COLORS)
    ):
        use = groups == j
        yy = 0.63 - j * 0.235

        rax.text(
            0.01, yy,
            label,
            color=color,
            transform=rax.transAxes,
            ha="left",
            fontsize=8.4,
            weight="bold",
        )

        for xloc, tick in zip(
            risk_columns, TICKS
        ):
            rax.text(
                xloc, yy,
                str(
                    int(
                        np.sum(
                            use & (T >= tick)
                        )
                    )
                ),
                transform=rax.transAxes,
                ha="center",
                fontsize=8.4,
            )

fig.subplots_adjust(
    left=0.09,
    right=0.985,
    top=0.91,
    bottom=0.08,
)

save_figure(
    fig,
    "Figure_2_external_km_risk_tertiles",
)


# ============================================================
# FIGURE 3 - External subgroup effects and Uno C-index
# ============================================================

subgroups = [
    r for r in f["subgroups"]
    if r["variable"] == "histology"
] + [
    r for r in f["subgroups"]
    if r["variable"] == "stage"
]

assert len(subgroups) == 4

group_names = [
    (
        str(r["category"])
        if r["variable"] == "histology"
        else f'Stage {r["category"]}'
    )
    + f' (n={r["n"]}, events={r["events"]})'
    for r in subgroups
]

group_delta = np.asarray(
    [
        float(r["delta_C"])
        for r in subgroups
    ]
)

uno = f["uno_C_by_tau_years"]

taus = [3, 5]

clinical_uno = [
    float(
        uno[str(float(t))]["clinical"]
    )
    for t in taus
]

recap_uno = [
    float(
        uno[str(float(t))]["recap"]
    )
    for t in taus
]

fig, (ax, bx) = plt.subplots(
    1, 2,
    figsize=(8.05, 3.15),
    gridspec_kw={
        "width_ratios": [1.23, 1]
    },
)

yg = np.arange(4)[::-1]

ax.axvline(
    0,
    linestyle="--",
    color="#8996A2",
    linewidth=1,
)

for yi, value in zip(
    yg, group_delta
):
    ax.plot(
        [0, value],
        [yi, yi],
        linewidth=2,
        color="#C4D1D9",
    )

ax.scatter(
    group_delta,
    yg,
    s=73,
    color=[
        TEAL if value >= 0 else CORAL
        for value in group_delta
    ],
    zorder=3,
)

ax.set_yticks(yg, group_names)
ax.set_ylim(-0.5, 3.5)
ax.set_xlim(-0.045, 0.12)

ax.set_xlabel(
    "Incremental C-index (ReCAP - clinical)"
)

ax.text(
    -0.10, 1.09, "A",
    transform=ax.transAxes,
    fontweight="bold",
    fontsize=12,
)

clean_axis(ax)


# Censoring-adjusted discrimination at fixed horizons.
bx.plot(
    taus,
    clinical_uno,
    color=NAVY,
    marker="o",
    linewidth=2,
    markersize=7,
    label="Clinical Cox",
)

bx.plot(
    taus,
    recap_uno,
    color=TEAL,
    marker="D",
    linewidth=2,
    markersize=6.5,
    label="ReCAP-Cox",
)

bx.set_xticks(
    taus,
    ["3 years", "5 years"],
)

bx.set_xlim(2.75, 5.25)
bx.set_ylim(0.50, 0.70)

bx.set_ylabel("Uno IPCW C-index")

bx.legend(
    loc="lower right",
    frameon=False,
)

bx.text(
    -0.13, 1.09, "B",
    transform=bx.transAxes,
    fontweight="bold",
    fontsize=12,
)

clean_axis(bx, "y")

fig.subplots_adjust(
    left=0.30,
    right=0.98,
    bottom=0.23,
    top=0.83,
    wspace=0.38,
)

save_figure(
    fig,
    "Figure_3_external_subgroups_uno",
)


# ============================================================
# FIGURE 4 - Cross-platform representation shift
# ============================================================

shift = np.asarray(
    e[
        "external_vs_masked_training_svd_mean_shift_in_training_sd"
    ],
    dtype=float,
)

ratio = np.asarray(
    e[
        "external_vs_masked_training_svd_sd_ratio"
    ],
    dtype=float,
)

assert (
    shift.shape == (8,)
    and ratio.shape == (8,)
    and np.isfinite(shift).all()
    and np.isfinite(ratio).all()
)

fig, (ax, bx) = plt.subplots(
    1, 2,
    figsize=(7.55, 3.35),
    sharey=True,
)

components = np.arange(1, 9)
yc = np.arange(8)

ax.barh(
    yc,
    shift,
    height=0.58,
    color=TEAL,
)

ax.set_yticks(
    yc,
    [f"SVD {j}" for j in components],
)

ax.invert_yaxis()

ax.set_xlabel(
    "Absolute mean shift / training SD"
)

ax.set_xlim(
    0,
    max(
        3.7,
        float(shift.max()) + 0.45,
    ),
)

ax.text(
    -0.10, 1.08, "A",
    transform=ax.transAxes,
    fontweight="bold",
    fontsize=12,
)

clean_axis(ax)


bx.barh(
    yc,
    ratio,
    height=0.58,
    color=PURPLE,
)

bx.axvline(
    1,
    linestyle="--",
    color="#7D8995",
    linewidth=1.2,
)

bx.set_xlim(
    0,
    max(
        1.18,
        float(ratio.max()) + 0.2,
    ),
)

bx.set_xlabel(
    "External / training SD ratio"
)

bx.text(
    -0.10, 1.08, "B",
    transform=bx.transAxes,
    fontweight="bold",
    fontsize=12,
)

clean_axis(bx)

fig.subplots_adjust(
    left=0.17,
    right=0.97,
    bottom=0.22,
    top=0.84,
    wspace=0.20,
)

save_figure(
    fig,
    "Figure_4_cross_platform_shift",
)


# ============================================================
# SUPPLEMENTARY FIGURE S1 - Essential method controls
# ============================================================

comparison_a = [
    a["recap_mean_site_C"],
    b["ablation_mean_C"]["mlp_residual"],
    b["ablation_mean_C"]["lowrank_interaction"],
    b["ablation_mean_C"]["spare_full"],
]

comparison_b = [
    c["gene_rank_svd8_mean_C"],
    c["recap_mean_C"],
    c["shuffled_pathway_mean_C"],
]

fig, axes = plt.subplots(
    1, 2,
    figsize=(7.6, 3.05),
)

panels = [
    (
        axes[0],
        [
            "ReCAP",
            "MLP residual",
            "Low-rank residual",
            "Full SPARE",
        ],
        comparison_a,
        [TEAL, NAVY, PURPLE, CORAL],
        "A",
    ),
    (
        axes[1],
        [
            "Gene SVD8",
            "Reactome SVD8",
            "Shuffled pathways",
        ],
        comparison_b,
        [NAVY, TEAL, GOLD],
        "B",
    ),
]

for (
    ax, labels, values, colors, letter
) in panels:
    yy = np.arange(len(values))

    ax.barh(
        yy,
        values,
        color=colors,
        height=0.55,
    )

    ax.set_yticks(yy, labels)
    ax.invert_yaxis()
    ax.set_xlim(0, 0.76)

    ax.set_xlabel(
        "Mean site-held-out C-index"
    )

    for i, value in enumerate(values):
        ax.text(
            value + 0.008,
            i,
            f"{value:.4f}",
            va="center",
            fontsize=8.6,
        )

    ax.text(
        -0.10, 1.08, letter,
        transform=ax.transAxes,
        fontweight="bold",
        fontsize=12,
    )

    clean_axis(ax)

fig.subplots_adjust(
    left=0.18,
    right=0.97,
    bottom=0.20,
    top=0.82,
    wspace=0.46,
)

save_figure(
    fig,
    "Figure_S1_method_controls",
)


# ============================================================
# TWO WORD TABLES - Black, white, full grid
# ============================================================

doc = Document()
sec = doc.sections[0]

sec.page_width = Inches(8.5)
sec.page_height = Inches(11)

sec.top_margin = Inches(0.7)
sec.bottom_margin = Inches(0.7)
sec.left_margin = Inches(0.72)
sec.right_margin = Inches(0.72)

style = doc.styles["Normal"]
style.font.name = "Times New Roman"
style.font.size = Pt(10)
style.font.color.rgb = RGBColor(0, 0, 0)
style.paragraph_format.space_after = Pt(4)


def style_cell(
    cell,
    value,
    bold=False,
    center=False,
):
    cell.text = ""
    p = cell.paragraphs[0]

    p.alignment = (
        WD_ALIGN_PARAGRAPH.CENTER
        if center else WD_ALIGN_PARAGRAPH.LEFT
    )

    p.paragraph_format.space_after = Pt(0)

    r = p.add_run(str(value))
    r.bold = bold
    r.font.name = "Times New Roman"
    r.font.size = Pt(9)
    r.font.color.rgb = RGBColor(0, 0, 0)

    cell.vertical_alignment = (
        WD_CELL_VERTICAL_ALIGNMENT.CENTER
    )

    tcPr = cell._tc.get_or_add_tcPr()

    margins = tcPr.find(qn("w:tcMar"))

    if margins is None:
        margins = OxmlElement("w:tcMar")
        tcPr.append(margins)

    for side in (
        "top", "start", "bottom", "end"
    ):
        item = margins.find(qn("w:" + side))

        if item is None:
            item = OxmlElement(
                "w:" + side
            )
            margins.append(item)

        item.set(qn("w:w"), "100")
        item.set(qn("w:type"), "dxa")

    borders = tcPr.find(qn("w:tcBorders"))

    if borders is None:
        borders = OxmlElement("w:tcBorders")
        tcPr.append(borders)

    for side in (
        "top", "left", "bottom",
        "right", "insideH", "insideV"
    ):
        item = borders.find(qn("w:" + side))

        if item is None:
            item = OxmlElement("w:" + side)
            borders.append(item)

        item.set(qn("w:val"), "single")
        item.set(qn("w:sz"), "5")
        item.set(qn("w:color"), "000000")


def caption(text):
    p = doc.add_paragraph()
    p.paragraph_format.keep_with_next = True
    p.paragraph_format.space_after = Pt(5)

    r = p.add_run(text)
    r.bold = True
    r.font.name = "Times New Roman"
    r.font.size = Pt(10)
    r.font.color.rgb = RGBColor(0, 0, 0)


def add_table(headers, rows, widths):
    t = doc.add_table(
        rows=1,
        cols=len(headers),
    )

    t.style = "Table Grid"
    t.alignment = WD_TABLE_ALIGNMENT.CENTER
    t.autofit = False

    for j, (heading, width) in enumerate(
        zip(headers, widths)
    ):
        t.columns[j].width = Inches(width)
        t.rows[0].cells[j].width = Inches(width)

        style_cell(
            t.rows[0].cells[j],
            heading,
            bold=True,
            center=j > 0,
        )

    for row in rows:
        cells = t.add_row().cells

        for j, (value, width) in enumerate(
            zip(row, widths)
        ):
            cells[j].width = Inches(width)

            style_cell(
                cells[j],
                value,
                center=j > 0,
            )

    return t


def note(text):
    p = doc.add_paragraph()

    p.paragraph_format.space_before = Pt(4)
    p.paragraph_format.space_after = Pt(12)

    r = p.add_run(text)
    r.font.name = "Times New Roman"
    r.font.size = Pt(8.5)
    r.font.color.rgb = RGBColor(0, 0, 0)


# Table 1
caption(
    "Table 1. Cohort allocation and survival endpoint eligibility"
)

add_table(
    [
        "Cohort",
        "Assigned n",
        "Eligible n",
        "Events",
        "Endpoint; platform",
    ],
    [
        (
            "TCGA development",
            "667", "657", "225",
            "PFI; RNA-seq",
        ),
        (
            "TCGA validation",
            "143", "140", "50",
            "PFI; RNA-seq",
        ),
        (
            "TCGA sealed test",
            "143", "138", "51",
            "PFI; RNA-seq",
        ),
        (
            "GSE50081 external",
            "181", "167", "48",
            "Recurrence-labelled DFS; GPL570",
        ),
    ],
    [1.55, 0.92, 0.92, 0.80, 2.85],
)

note(
    "PFI: progression-free interval. "
    "The external recurrence-labelled disease-free "
    "survival endpoint is not identical to TCGA-CDR PFI."
)


# Table 2
caption(
    "Table 2. Locked discrimination of clinical Cox and ReCAP-Cox"
)

short_labels = [
    "TCGA development CV",
    "TCGA validation",
    "TCGA sealed test",
    "GSE50081 external",
]

rows = []

for j, r in perf.iterrows():
    interval = (
        f"[{ci[0]:+.4f}, {ci[1]:+.4f}]"
        if j == 3
        else "Not shown"
    )

    rows.append((
        short_labels[j],
        f'{r["Clinical Cox"]:.4f}',
        f'{r["ReCAP-Cox"]:.4f}',
        f'{r["Delta C"]:+.4f}',
        interval,
    ))

add_table(
    [
        "Evaluation",
        "Clinical C",
        "ReCAP C",
        "Delta C",
        "95% CI for delta C",
    ],
    rows,
    [2.08, 0.98, 0.98, 0.82, 2.18],
)

note(
    "C: Harrell C-index. The external 95% CI is a "
    "paired patient-bootstrap interval. Other "
    "confidence intervals are not displayed here. "
    "Development comparisons remain exploratory."
)

assert len(doc.tables) == 2

word_path = (
    OUT / "Main_Tables_1_to_2.docx"
)

doc.save(word_path)


# ============================================================
# Separate journal captions and reproducibility manifest
# ============================================================

captions = {
    "Figure 1": (
        "Performance across the TCGA development, frozen "
        "validation and sealed test, and GSE50081 external "
        "cohort (A). Paired C-index gains for the four "
        "TCGA development folds and external cohort (B). "
        "Only the external estimate is displayed with a "
        "95% bootstrap interval. TCGA PFI and external "
        "recurrence-related DFS have different definitions."
    ),

    "Figure 2": (
        "Descriptive Kaplan-Meier curves for GSE50081 "
        "patients grouped into outcome-blind, balanced "
        "tertiles of frozen clinical Cox risk (A) and "
        "frozen ReCAP-Cox risk (B). Numbers at risk are "
        "shown below each panel. These exploratory curves "
        "do not establish calibration or clinical utility. "
        "Follow-up time is interpreted as years based on "
        "publication evidence, not explicit GEO metadata."
    ),

    "Figure 3": (
        "External C-index differences within LUAD, LUSC, "
        "stage I and stage II (A), and Uno IPCW C-index "
        "at 3 and 5 years for clinical Cox and ReCAP-Cox "
        "(B). Subgroup differences are descriptive and "
        "are displayed without confidence intervals. "
        "The LUSC subgroup contains only 11 events."
    ),

    "Figure 4": (
        "Outcome-blind cross-platform latent representation "
        "shift. Absolute differences between external and "
        "mask-matched training SVD component means, scaled "
        "by training SD (A), and ratios of external to "
        "training component SDs (B). These measures quantify "
        "distribution shift, not predictive performance."
    ),

    "Figure S1": (
        "Exploratory training-only comparisons of residual "
        "architecture extensions (A) and transcriptomic "
        "representation controls (B). Reactome and shuffled "
        "pathway mappings showed similar mean performance, "
        "and neural extensions did not improve the incumbent."
    ),
}

(
    OUT / "Figure_Captions.txt"
).write_text(
    "\n\n".join(
        f"{key}. {value}"
        for key, value in captions.items()
    ),
    encoding="utf-8",
)

manifest = {
    "input_sha256": {
        name: digest(path)
        for name, path in FILES.items()
    },
    "main_results_figures": 4,
    "supplementary_figures": 1,
    "word_tables": 2,
    "figure_font": "Times New Roman",
    "figure_embedded_titles": False,
    "external_survival_model_refitted": False,
    "external_primary_Cindex_rescored": False,
    "external_KM_curves_are_descriptive": True,
}

(
    OUT / "phase_h_results_manifest.json"
).write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)


# Final artifact verification.
figure_stems = [
    "Figure_1_locked_discrimination",
    "Figure_2_external_km_risk_tertiles",
    "Figure_3_external_subgroups_uno",
    "Figure_4_cross_platform_shift",
    "Figure_S1_method_controls",
]

for stem in figure_stems:
    for ext in ("png", "pdf", "svg"):
        path = OUT / f"{stem}.{ext}"

        if (
            not path.is_file()
            or path.stat().st_size < 1000
        ):
            raise RuntimeError(
                f"Figure export failed: {path}"
            )

print("=" * 75)
print("PHASE H2 COMPLETED - RESULT-FOCUSED MANUSCRIPT FIGURES")
print("=" * 75)

print("Main results figures:", 4)
print("Supplementary figures:", 1)
print("Figure formats: 600-DPI PNG, SVG, PDF")
print("Figure font: Times New Roman")
print("Embedded figure titles: NONE")

print("\nWord tables:", len(doc.tables))
print("Table style: black text, white background, full grid")

print("\nExternal primary C-index recalculated: NO")
print("Frozen survival model modified: NO")
print("New KM curves: descriptive analysis only")

print("\nSaved to:")
print(OUT)

print("\nGenerated files:")
for path in sorted(OUT.iterdir()):
    print(" ", path.name)

print("\nPHASE H2 COMPLETED")

PHASE H2 COMPLETED - RESULT-FOCUSED MANUSCRIPT FIGURES
Main results figures: 4
Supplementary figures: 1
Figure formats: 600-DPI PNG, SVG, PDF
Figure font: Times New Roman
Embedded figure titles: NONE

Word tables: 2
Table style: black text, white background, full grid

External primary C-index recalculated: NO
Frozen survival model modified: NO
New KM curves: descriptive analysis only

Saved to:
D:\NST_NSCLC_PPI_GAT\results\manuscript_assets\phase_h_results

Generated files:
  Figure_1_locked_discrimination.pdf
  Figure_1_locked_discrimination.png
  Figure_1_locked_discrimination.svg
  Figure_2_external_km_risk_tertiles.pdf
  Figure_2_external_km_risk_tertiles.png
  Figure_2_external_km_risk_tertiles.svg
  Figure_3_external_subgroups_uno.pdf
  Figure_3_external_subgroups_uno.png
  Figure_3_external_subgroups_uno.svg
  Figure_4_cross_platform_shift.pdf
  Figure_4_cross_platform_shift.png
  Figure_4_cross_platform_shift.svg
  Figure_Captions.txt
  Figure_S1_method_controls.pdf
  Figure_S

In [40]:

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties, findfont
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from docx import Document
from docx.shared import Inches, Pt, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import OxmlElement
from docx.oxml.ns import qn


# Final manuscript assets: five main figures, one supplementary figure, two tables.
ROOT = Path(r"D:\NST_NSCLC_PPI_GAT")
M = ROOT / "results" / "metrics"
OUT = ROOT / "results" / "manuscript_assets" / "recap_final_figures_tables"

FILES = {
    "a": M / "spare_phase_a" / "phase_a_manifest.json",
    "c": M / "spare_phase_c" / "phase_c_summary.json",
    "c_folds": M / "spare_phase_c" / "phase_c_fold_comparisons.csv",
    "d": M / "gse50081_external_audit" / "gse50081_feasibility_summary.json",
    "e": M / "gse50081_transfer_readiness" / "phase_e_summary.json",
    "f": M / "gse50081_one_time_external_evaluation" / "final_external_results.json",
    "g": M / "recap_phase_g_manuscript_audit" / "phase_g_integrity_summary.json",
    "performance": M / "recap_phase_g_manuscript_audit" / "matched_discrimination_table.csv",
    "patients": M / "gse50081_one_time_external_evaluation" / "external_frozen_patient_predictions.csv",
}

for label, path in FILES.items():
    if not path.is_file():
        raise FileNotFoundError(f"Research source missing [{label}]: {path}")


def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def read_json(key):
    return json.loads(FILES[key].read_text(encoding="utf-8"))


a, c, d, e, f, g = [
    read_json(key)
    for key in ("a", "c", "d", "e", "f", "g")
]

performance = pd.read_csv(FILES["performance"])
folds = pd.read_csv(FILES["c_folds"]).sort_values("fold")
patients = pd.read_csv(FILES["patients"], dtype={"GSM": str})

needed = {
    "fold", "held", "events", "clinical_C",
    "recap_C", "gene_rank_svd8_C", "shuffled_mean_C"
}

if not needed.issubset(folds.columns):
    raise RuntimeError(
        f"Phase C fold columns missing: {sorted(needed - set(folds.columns))}"
    )

if (
    folds.fold.tolist() != [1, 2, 3, 4]
    or int(folds.held.sum()) != 657
    or int(folds.events.sum()) != 225
):
    raise RuntimeError("Development folds differ from the locked cohort.")

if performance.Evaluation.tolist() != [
    "TCGA site-held-out CV",
    "TCGA frozen validation",
    "TCGA sealed test",
    "GSE50081 external DFS"
]:
    raise RuntimeError("Unexpected evaluation order in the Phase G table.")

if not np.allclose(
    performance["ReCAP-Cox"] - performance["Clinical Cox"],
    performance["Delta C"],
    atol=1.1e-4
):
    raise RuntimeError("Discrimination table is arithmetically inconsistent.")

if not np.isclose(
    folds.recap_C.mean(),
    a["recap_mean_site_C"],
    atol=6e-4
):
    raise RuntimeError("Original development performance changed.")

if not np.isclose(
    folds.recap_C.mean(),
    c["recap_mean_C"],
    atol=6e-4
):
    raise RuntimeError("Phase C ReCAP-Cox performance changed.")

if g["external_result_sha256"] != digest(FILES["f"]):
    raise RuntimeError(
        "One-time external result hash does not match Phase G."
    )

if not (
    len(patients) == f["n"] == 167
    and patients.GSM.is_unique
    and int(patients.recurrence_event.sum()) == f["events"] == 48
):
    raise RuntimeError(
        "External patient identities, count, or events changed."
    )

if not (
    f["checkpoint_refitted"] is False
    and f["external_outcomes_used_for_tuning"] is False
):
    raise RuntimeError("External-model lock has changed.")

if not np.allclose(
    patients.clinical_logrisk + patients.molecular_logrisk,
    patients.recap_logrisk,
    atol=1e-8
):
    raise RuntimeError("External log-risk components are inconsistent.")

if not np.isfinite(
    patients[
        ["followup_years", "clinical_logrisk", "recap_logrisk"]
    ].to_numpy(float)
).all():
    raise RuntimeError("Nonfinite external values.")

if (
    not (patients.followup_years > 0).all()
    or not set(patients.recurrence_event.unique()).issubset({0, 1})
):
    raise RuntimeError("External recurrence status or follow-up is invalid.")

if not np.isclose(
    performance.iloc[-1]["ReCAP-Cox"],
    f["recap_C"],
    atol=1e-4
):
    raise RuntimeError("Saved external C-index differs from Phase G.")

ci = np.asarray(
    f["paired_patient_bootstrap"]["delta_C_95pct"],
    float
)

if (
    ci.shape != (2,)
    or not np.isfinite(ci).all()
    or not ci[0] < f["delta_C"] < ci[1]
):
    raise RuntimeError("Saved external paired interval is invalid.")


# Enforce the journal figure font.
try:
    font_file = findfont(
        FontProperties(family="Times New Roman"),
        fallback_to_default=False
    )

    if "times" not in Path(font_file).name.lower():
        raise RuntimeError("Incorrect font resolution.")

except Exception as exc:
    raise RuntimeError(
        "Times New Roman must be installed and visible to Matplotlib."
    ) from exc


OUT.mkdir(parents=True, exist_ok=True)

BLUE = "#27639B"
TEAL = "#008C83"
ORANGE = "#D97C35"
VIOLET = "#7567A6"

INK = "#263541"
GRID = "#DCE4E9"
MUTED = "#738390"

mpl.rcParams.update({
    "font.family": "Times New Roman",
    "font.size": 10.0,
    "axes.labelsize": 10.1,
    "xtick.labelsize": 9.1,
    "ytick.labelsize": 9.1,
    "legend.fontsize": 8.7,
    "axes.edgecolor": MUTED,
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "mathtext.fontset": "custom",
    "mathtext.rm": "Times New Roman",
    "mathtext.it": "Times New Roman:italic",
    "mathtext.bf": "Times New Roman:bold",
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
})


def clean(ax, grid="x"):
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(length=0, pad=5)
    ax.set_axisbelow(True)
    ax.grid(axis=grid, color=GRID, linewidth=0.7)


def panel(ax, letter, x=-0.08, y=1.06):
    ax.text(
        x, y, letter,
        transform=ax.transAxes,
        fontsize=12,
        weight="bold",
        va="center"
    )


def export(fig, name):
    for ext in ("pdf", "svg", "png"):
        fig.savefig(
            OUT / f"{name}.{ext}",
            dpi=600 if ext == "png" else 300,
            bbox_inches="tight",
            pad_inches=0.12,
            facecolor="white"
        )
    plt.close(fig)


def box(ax, x, y, w, h, label, fill, edge, size=9.1):
    ax.add_patch(
        FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.01,rounding_size=0.024",
            transform=ax.transAxes,
            facecolor=fill,
            edgecolor=edge,
            linewidth=1.1
        )
    )

    ax.text(
        x + w / 2,
        y + h / 2,
        label,
        transform=ax.transAxes,
        ha="center",
        va="center",
        fontsize=size,
        linespacing=1.15
    )


def arrow(ax, x1, y1, x2, y2, color=MUTED):
    ax.add_patch(
        FancyArrowPatch(
            (x1, y1),
            (x2, y2),
            transform=ax.transAxes,
            arrowstyle="-|>",
            mutation_scale=11,
            linewidth=1.25,
            color=color
        )
    )


# ==========================================================
# FIGURE 1: Clinical anchor and low-rank RNA correction
# ==========================================================

fig = plt.figure(figsize=(7.45, 3.35))
ax = fig.add_axes([0.06, 0.08, 0.90, 0.86])
ax.axis("off")

panel(ax, "A", x=-0.045, y=0.96)

box(
    ax, 0.04, 0.67, 0.21, 0.18,
    "Clinical covariates\nage, sex, histology, stage",
    "#EAF3FB", BLUE, 9.2
)

box(
    ax, 0.34, 0.67, 0.22, 0.18,
    "Clinical Cox\nfixed coefficient vector",
    "#EAF3FB", BLUE, 9.2
)

box(
    ax, 0.77, 0.62, 0.19, 0.26,
    "Final log-risk\nclinical + RNA",
    "#E9F4F0", TEAL, 10
)

arrow(ax, 0.26, 0.76, 0.33, 0.76, BLUE)
arrow(ax, 0.57, 0.76, 0.76, 0.76, BLUE)

labels = [
    "RNA gene\nexpression",
    "Patient-wise\nRNA ranks",
    "Reactome\npathway scores",
    "Global SVD\n8 dimensions",
    "Penalized Cox\nRNA correction"
]

xs = [0.025, 0.225, 0.425, 0.625, 0.825]

for j, (x, label) in enumerate(zip(xs, labels)):
    box(
        ax, x, 0.30, 0.15, 0.18,
        label, "#FFF2E9", ORANGE, 8.6
    )

    if j:
        arrow(
            ax,
            xs[j - 1] + 0.155,
            0.39,
            x - 0.005,
            0.39,
            ORANGE
        )

arrow(ax, 0.90, 0.50, 0.90, 0.61, ORANGE)

panel(ax, "B", x=-0.045, y=0.185)

ax.text(
    0.045, 0.12,
    r"$z_i=\mathrm{SVD}_8\!\left[\mathrm{Scale}\left(W\,r(x_i)\right)\right]$",
    transform=ax.transAxes,
    fontsize=12.2,
    color=BLUE
)

ax.text(
    0.52, 0.12,
    r"$\eta_i=\widehat{\beta}_c^{\mathsf{T}}c_i+\widehat{\theta}^{\mathsf{T}}z_i$",
    transform=ax.transAxes,
    fontsize=12.2,
    color=TEAL
)

ax.text(
    0.50, 0.015,
    "RNA offset correction fitted on TCGA training only; external model fully frozen",
    transform=ax.transAxes,
    ha="center",
    fontsize=8.6,
    color=MUTED
)

export(fig, "Figure_1_recap_method")


# ==========================================================
# FIGURE 2: Locked discrimination and external uncertainty
# ==========================================================

clinical = performance["Clinical Cox"].to_numpy(float)
recap = performance["ReCAP-Cox"].to_numpy(float)

fig, (ax, bx) = plt.subplots(
    1, 2,
    figsize=(7.8, 3.50),
    gridspec_kw={"width_ratios": [1.3, 1]}
)

y = np.arange(4)[::-1]

for pos, cc, rr in zip(y, clinical, recap):
    ax.plot(
        [cc, rr],
        [pos, pos],
        lw=2.8,
        color="#B2C0CA",
        zorder=1
    )

ax.scatter(
    clinical, y,
    s=64,
    color=BLUE,
    label="Clinical Cox",
    zorder=3
)

ax.scatter(
    recap, y,
    s=68,
    marker="D",
    color=TEAL,
    label="ReCAP-Cox",
    zorder=3
)

ax.set_yticks(
    y,
    [
        "TCGA development CV",
        "TCGA validation",
        "TCGA sealed test",
        "GSE50081 external"
    ]
)

ax.set(
    xlim=(0.52, 0.75),
    ylim=(-0.6, 3.6),
    xlabel="Harrell C-index"
)

ax.legend(
    loc="lower center",
    bbox_to_anchor=(0.52, 1.02),
    ncol=2,
    frameon=False
)

clean(ax)
panel(ax, "A", -0.12, 1.065)

fold_delta = (
    folds.recap_C - folds.clinical_C
).to_numpy(float)

for yy, dd in zip([4, 3, 2, 1], fold_delta):
    bx.plot(
        [0, dd],
        [yy, yy],
        color="#C5D2DC",
        lw=2
    )

    bx.scatter(
        dd, yy,
        s=52,
        color=TEAL,
        zorder=4
    )

bx.errorbar(
    f["delta_C"],
    0,
    xerr=[
        [f["delta_C"] - ci[0]],
        [ci[1] - f["delta_C"]]
    ],
    fmt="D",
    color=ORANGE,
    ecolor=ORANGE,
    capsize=4,
    ms=6,
    lw=1.9
)

bx.axvline(
    0,
    color=MUTED,
    linestyle="--",
    lw=1
)

bx.set_yticks(
    [4, 3, 2, 1, 0],
    [
        "CV fold 1",
        "CV fold 2",
        "CV fold 3",
        "CV fold 4",
        "GSE50081"
    ]
)

bx.set(
    xlim=(-0.015, 0.115),
    ylim=(-0.6, 4.6),
    xlabel="ReCAP - clinical (C-index)"
)

bx.text(
    0.98, 0.98,
    "External: paired 95% CI",
    transform=bx.transAxes,
    fontsize=8.3,
    ha="right",
    va="top",
    color=MUTED
)

clean(bx)
panel(bx, "B", -0.12, 1.065)

fig.subplots_adjust(
    left=0.245,
    right=0.985,
    top=0.81,
    bottom=0.21,
    wspace=0.49
)

export(fig, "Figure_2_locked_performance")


# ==========================================================
# FIGURE 3: External Kaplan-Meier risk stratification
# ==========================================================

def rank_tertiles(values):
    values = np.asarray(values, float)

    order = np.argsort(
        values,
        kind="mergesort"
    )

    categories = np.empty(
        len(values),
        dtype=np.int64
    )

    categories[order] = (
        3 * np.arange(len(values))
        // len(values)
    )

    return categories


def km(time, events, horizon):
    observed_times = np.unique(
        time[
            (events == 1)
            & (time <= horizon)
        ]
    )

    X = [0.0]
    Y = [1.0]
    S = 1.0

    for t in observed_times:
        n = np.count_nonzero(time >= t)

        d_events = np.count_nonzero(
            (time == t)
            & (events == 1)
        )

        S *= (1 - d_events / n)

        X.append(float(t))
        Y.append(float(S))

    X.append(float(horizon))
    Y.append(float(S))

    return np.asarray(X), np.asarray(Y)


T = patients.followup_years.to_numpy(float)
E = patients.recurrence_event.to_numpy(int)

HORIZON = 6.0
TICKS = [0, 2, 4, 6]

fig = plt.figure(figsize=(7.7, 4.45))

grid = fig.add_gridspec(
    2, 2,
    height_ratios=[4.2, 1.28],
    hspace=0.36,
    wspace=0.25
)

for k, predictor in enumerate(
    ["clinical_logrisk", "recap_logrisk"]
):
    ax = fig.add_subplot(grid[0, k])
    risk_ax = fig.add_subplot(grid[1, k])

    strata = rank_tertiles(
        patients[predictor].to_numpy(float)
    )

    for j, (name, color, style) in enumerate(
        zip(
            ["Low risk", "Intermediate risk", "High risk"],
            [TEAL, BLUE, ORANGE],
            ["-", "--", "-."]
        )
    ):
        use = strata == j

        xx, yy = km(
            T[use],
            E[use],
            HORIZON
        )

        ax.step(
            xx,
            yy,
            where="post",
            color=color,
            ls=style,
            lw=2.0,
            label=f"{name} (n={int(use.sum())})"
        )

    ax.set(
        xlim=(0, HORIZON),
        ylim=(0, 1.02),
        xticks=TICKS,
        xlabel="DFS follow-up (years)"
    )

    if k == 0:
        ax.set_ylabel(
            "Recurrence-free probability"
        )

    ax.legend(
        loc="lower left",
        frameon=False,
        fontsize=8.1
    )

    clean(ax, "both")

    panel(
        ax,
        "A" if k == 0 else "B",
        -0.09,
        1.065
    )

    risk_ax.axis("off")

    risk_ax.text(
        0.02, 0.90,
        "At risk",
        transform=risk_ax.transAxes,
        fontsize=8.6
    )

    xlocs = [0.43, 0.60, 0.77, 0.94]

    for xpos, tick in zip(xlocs, TICKS):
        risk_ax.text(
            xpos, 0.90,
            str(tick),
            transform=risk_ax.transAxes,
            ha="center",
            fontsize=8.4
        )

    for j, (name, color) in enumerate(
        zip(
            ["Low", "Intermediate", "High"],
            [TEAL, BLUE, ORANGE]
        )
    ):
        use = strata == j
        ypos = 0.62 - 0.24 * j

        risk_ax.text(
            0.02, ypos,
            name,
            transform=risk_ax.transAxes,
            color=color,
            fontsize=8.4
        )

        for xpos, tick in zip(xlocs, TICKS):
            risk_ax.text(
                xpos,
                ypos,
                str(
                    int(
                        np.sum(
                            use & (T >= tick)
                        )
                    )
                ),
                transform=risk_ax.transAxes,
                ha="center",
                fontsize=8.4
            )

fig.subplots_adjust(
    left=0.10,
    right=0.98,
    top=0.91,
    bottom=0.075
)

export(fig, "Figure_3_external_km")


# ==========================================================
# FIGURE 4: RNA representation and biological-prior controls
# ==========================================================

models = [
    "Clinical Cox",
    "Gene SVD8",
    "ReCAP-Cox",
    "Shuffled pathway map"
]

columns = [
    "clinical_C",
    "gene_rank_svd8_C",
    "recap_C",
    "shuffled_mean_C"
]

values = np.array(
    [
        folds[col].to_numpy(float)
        for col in columns
    ]
).T

means = values.mean(axis=0)

if not np.isclose(
    means[1],
    c["gene_rank_svd8_mean_C"],
    atol=6e-4
):
    raise RuntimeError(
        "Gene-level SVD control differs from Phase C summary."
    )

if not np.isclose(
    means[3],
    c["shuffled_pathway_mean_C"],
    atol=6e-4
):
    raise RuntimeError(
        "Randomized mapping control differs from Phase C summary."
    )

fig, (ax, bx) = plt.subplots(
    1, 2,
    figsize=(7.8, 3.55),
    gridspec_kw={"width_ratios": [1.08, 1]}
)

colors = [MUTED, BLUE, TEAL, VIOLET]

for i in range(4):
    ax.scatter(
        [i] * 4,
        values[:, i],
        s=25,
        color=colors[i],
        alpha=0.48,
        zorder=2
    )

    ax.scatter(
        i,
        means[i],
        s=85,
        marker="D",
        color=colors[i],
        edgecolor="white",
        linewidth=0.7,
        zorder=4
    )

ax.set_xticks(
    range(4),
    ["Clinical", "Gene SVD8", "ReCAP", "Shuffled"]
)

ax.set(
    xlim=(-0.45, 3.45),
    ylim=(0.53, 0.75),
    ylabel="Site-held-out C-index"
)

ax.text(
    0.98, 0.98,
    "Dots: folds; diamonds: means",
    transform=ax.transAxes,
    ha="right",
    va="top",
    fontsize=8.0,
    color=MUTED
)

clean(ax, "y")
panel(ax, "A", -0.10, 1.06)

xx = np.arange(1, 5)

vs_gene = values[:, 2] - values[:, 1]
vs_null = values[:, 2] - values[:, 3]

bx.axhline(
    0,
    linestyle="--",
    color=MUTED,
    lw=1
)

bx.plot(
    xx,
    vs_gene,
    color=TEAL,
    lw=1.7,
    marker="D",
    ms=6,
    label="ReCAP - gene SVD8"
)

bx.plot(
    xx,
    vs_null,
    color=VIOLET,
    lw=1.7,
    marker="o",
    ms=6,
    label="ReCAP - shuffled"
)

bx.set(
    xticks=xx,
    xlabel="Site-held-out fold",
    ylabel="Paired difference in C-index"
)

bx.legend(
    loc="upper right",
    frameon=False,
    fontsize=8.3
)

clean(bx, "y")
panel(bx, "B", -0.10, 1.06)

fig.subplots_adjust(
    left=0.11,
    right=0.97,
    top=0.83,
    bottom=0.20,
    wspace=0.29
)

export(
    fig,
    "Figure_4_representation_comparison"
)


# ==========================================================
# FIGURE 5: External subgroup and Uno IPCW concordance
# ==========================================================

subgroups = [
    x
    for kind in ("histology", "stage")
    for x in f["subgroups"]
    if x["variable"] == kind
]

if len(subgroups) != 4:
    raise RuntimeError(
        "Unexpected saved external subgroup result schema."
    )

sub_labels = [
    (
        str(x["category"])
        if x["variable"] == "histology"
        else f'Stage {x["category"]}'
    )
    + f' (n={x["n"]}; events={x["events"]})'
    for x in subgroups
]

sub_delta = [
    float(x["delta_C"])
    for x in subgroups
]

uno = f["uno_C_by_tau_years"]

clinical_uno = [
    float(
        uno[str(float(t))]["clinical"]
    )
    for t in (3, 5)
]

recap_uno = [
    float(
        uno[str(float(t))]["recap"]
    )
    for t in (3, 5)
]

fig, (ax, bx) = plt.subplots(
    1, 2,
    figsize=(7.75, 3.3),
    gridspec_kw={"width_ratios": [1.26, 1]}
)

y = np.arange(4)[::-1]

ax.axvline(
    0,
    linestyle="--",
    color=MUTED,
    lw=1
)

for pos, delta in zip(y, sub_delta):
    ax.plot(
        [0, delta],
        [pos, pos],
        color="#BDCDD6",
        lw=2
    )

ax.scatter(
    sub_delta,
    y,
    c=[
        TEAL if x >= 0 else ORANGE
        for x in sub_delta
    ],
    s=68,
    zorder=3
)

ax.set_yticks(y, sub_labels)

ax.set(
    xlim=(-0.03, 0.11),
    ylim=(-0.5, 3.5),
    xlabel="Subgroup ReCAP - clinical (C-index)"
)

clean(ax)
panel(ax, "A", -0.12, 1.07)

bx.plot(
    [3, 5],
    clinical_uno,
    color=BLUE,
    marker="o",
    lw=2,
    ms=7,
    label="Clinical Cox"
)

bx.plot(
    [3, 5],
    recap_uno,
    color=TEAL,
    marker="D",
    lw=2,
    ms=6,
    label="ReCAP-Cox"
)

bx.set(
    xticks=[3, 5],
    xlim=(2.7, 5.3),
    ylim=(0.50, 0.70),
    xlabel="Evaluation horizon (years)",
    ylabel="Uno IPCW C-index"
)

bx.legend(
    loc="lower right",
    frameon=False
)

clean(bx, "y")
panel(bx, "B", -0.13, 1.07)

fig.subplots_adjust(
    left=0.315,
    right=0.98,
    top=0.83,
    bottom=0.24,
    wspace=0.35
)

export(fig, "Figure_5_external_robustness")


# ==========================================================
# SUPPLEMENTARY FIGURE S1: Cross-platform shift
# ==========================================================

shift = np.asarray(
    e["external_vs_masked_training_svd_mean_shift_in_training_sd"],
    float
)

ratio = np.asarray(
    e["external_vs_masked_training_svd_sd_ratio"],
    float
)

if (
    shift.shape != (8,)
    or ratio.shape != (8,)
    or not np.isfinite(shift).all()
    or not np.isfinite(ratio).all()
):
    raise RuntimeError(
        "External SVD shift diagnostics changed."
    )

fig, (ax, bx) = plt.subplots(
    1, 2,
    figsize=(7.1, 3.2),
    sharey=True
)

y = np.arange(8)

ax.barh(
    y,
    shift,
    height=0.55,
    color=TEAL
)

ax.set_yticks(
    y,
    [f"SVD {i}" for i in range(1, 9)]
)

ax.invert_yaxis()
ax.set_xlabel(
    "Absolute mean shift / training SD"
)

clean(ax)
panel(ax, "A", -0.10, 1.06)

bx.barh(
    y,
    ratio,
    height=0.55,
    color=VIOLET
)

bx.axvline(
    1,
    linestyle="--",
    color=MUTED,
    lw=1
)

bx.set_xlabel(
    "External / training SD"
)

clean(bx)
panel(bx, "B", -0.10, 1.06)

fig.subplots_adjust(
    left=0.16,
    right=0.97,
    bottom=0.21,
    top=0.83,
    wspace=0.18
)

export(fig, "Figure_S1_transfer_shift")


# ==========================================================
# TWO WORD TABLES: White background and full black grid
# ==========================================================

doc = Document()
section = doc.sections[0]

section.page_width = Inches(8.5)
section.page_height = Inches(11)

section.top_margin = Inches(0.73)
section.bottom_margin = Inches(0.73)
section.left_margin = Inches(0.69)
section.right_margin = Inches(0.69)

normal = doc.styles["Normal"]
normal.font.name = "Times New Roman"
normal.font.size = Pt(10)
normal.font.color.rgb = RGBColor(0, 0, 0)
normal.paragraph_format.space_after = Pt(4)


def styled_cell(cell, value, is_header, centered):
    cell.text = ""

    cell.vertical_alignment = (
        WD_CELL_VERTICAL_ALIGNMENT.CENTER
    )

    p = cell.paragraphs[0]

    p.alignment = (
        WD_ALIGN_PARAGRAPH.CENTER
        if centered else WD_ALIGN_PARAGRAPH.LEFT
    )

    p.paragraph_format.space_after = Pt(0)

    r = p.add_run(str(value))
    r.bold = is_header
    r.font.name = "Times New Roman"
    r.font.size = Pt(9)
    r.font.color.rgb = RGBColor(0, 0, 0)

    tcpr = cell._tc.get_or_add_tcPr()

    mar = tcpr.find(qn("w:tcMar"))

    if mar is None:
        mar = OxmlElement("w:tcMar")
        tcpr.append(mar)

    for side in ("top", "start", "bottom", "end"):
        item = mar.find(
            qn("w:" + side)
        )

        if item is None:
            item = OxmlElement(
                "w:" + side
            )
            mar.append(item)

        item.set(
            qn("w:w"),
            "105"
        )

        item.set(
            qn("w:type"),
            "dxa"
        )

    border = tcpr.find(
        qn("w:tcBorders")
    )

    if border is None:
        border = OxmlElement(
            "w:tcBorders"
        )
        tcpr.append(border)

    for side in (
        "top", "left", "bottom",
        "right", "insideH", "insideV"
    ):
        item = border.find(
            qn("w:" + side)
        )

        if item is None:
            item = OxmlElement(
                "w:" + side
            )
            border.append(item)

        item.set(
            qn("w:val"),
            "single"
        )

        item.set(
            qn("w:sz"),
            "6"
        )

        item.set(
            qn("w:color"),
            "000000"
        )


def add_table(
    caption_text,
    headers,
    rows,
    widths,
    note
):
    p = doc.add_paragraph()

    p.paragraph_format.keep_with_next = True
    p.paragraph_format.space_after = Pt(5)

    run = p.add_run(caption_text)
    run.bold = True
    run.font.name = "Times New Roman"
    run.font.size = Pt(10)

    tbl = doc.add_table(
        rows=1,
        cols=len(headers)
    )

    tbl.style = "Table Grid"
    tbl.autofit = False
    tbl.alignment = WD_TABLE_ALIGNMENT.CENTER

    for j, (header, width) in enumerate(
        zip(headers, widths)
    ):
        tbl.columns[j].width = Inches(width)
        tbl.rows[0].cells[j].width = Inches(width)

        styled_cell(
            tbl.rows[0].cells[j],
            header,
            True,
            j > 0
        )

    for record in rows:
        cells = tbl.add_row().cells

        for j, (value, width) in enumerate(
            zip(record, widths)
        ):
            cells[j].width = Inches(width)

            styled_cell(
                cells[j],
                value,
                False,
                j > 0
            )

    p = doc.add_paragraph()

    p.paragraph_format.space_before = Pt(4)
    p.paragraph_format.space_after = Pt(11)

    run = p.add_run(note)
    run.font.name = "Times New Roman"
    run.font.size = Pt(8.5)

    return tbl


# Table 1: locked cohort information.
# TCGA validation/test counts are from the previous locked audits.
add_table(
    "Table 1. Cohort composition and endpoint availability",
    [
        "Study cohort",
        "Assigned",
        "Eligible",
        "Events",
        "Endpoint / assay"
    ],
    [
        (
            "TCGA development",
            "667",
            "657",
            str(int(folds.events.sum())),
            "PFI / RNA-seq"
        ),
        (
            "TCGA validation",
            "143",
            "140",
            "50",
            "PFI / RNA-seq"
        ),
        (
            "TCGA sealed test",
            "143",
            "138",
            "51",
            "PFI / RNA-seq"
        ),
        (
            "GSE50081 external",
            "181",
            str(f["n"]),
            str(f["events"]),
            "Recurrence-DFS / GPL570"
        )
    ],
    [1.75, 0.83, 0.84, 0.77, 2.91],
    (
        "PFI, progression-free interval. External recurrence-labelled "
        "DFS differs from TCGA-CDR PFI; its follow-up unit is "
        "publication-supported rather than explicit in GEO metadata."
    )
)


# Table 2: all locked discrimination results.
rows = []

for idx, row in performance.iterrows():
    ci_text = (
        f"[{ci[0]:+.4f}, {ci[1]:+.4f}]"
        if idx == 3
        else "Not shown"
    )

    rows.append(
        (
            str(row["Evaluation"]),
            f'{row["Clinical Cox"]:.4f}',
            f'{row["ReCAP-Cox"]:.4f}',
            f'{row["Delta C"]:+.4f}',
            ci_text
        )
    )

add_table(
    "Table 2. Discrimination of clinical Cox and frozen ReCAP-Cox models",
    [
        "Evaluation",
        "Clinical C",
        "ReCAP C",
        "Delta C",
        "95% CI of delta C"
    ],
    rows,
    [2.25, 0.94, 0.94, 0.79, 2.18],
    (
        "Harrell C-index; delta C = ReCAP-Cox minus clinical Cox. "
        "Only the external paired patient-bootstrap 95% interval "
        "is displayed. Development CV is exploratory; GSE50081 "
        "uses recurrence-labelled DFS."
    )
)

if len(doc.tables) != 2:
    raise RuntimeError(
        "Expected exactly two main manuscript tables."
    )

doc.save(
    OUT / "Main_Tables_1_to_2.docx"
)


# ==========================================================
# SEPARATE FIGURE CAPTIONS
# ==========================================================

captions = {
    "Figure 1": (
        "Clinically anchored ReCAP-Cox representation (A) and "
        "compact risk formulation (B). RNA pathway aggregation "
        "and global SVD8 yield a penalized molecular correction "
        "to a fixed clinical Cox log-risk. All coefficients and "
        "projections were frozen for GSE50081."
    ),

    "Figure 2": (
        "Locked clinical Cox and ReCAP-Cox discrimination across "
        "TCGA development, validation, sealed test and GSE50081 (A). "
        "Site-held-out development fold differences and the one-time "
        "external estimate with its paired patient-bootstrap 95% CI (B). "
        "External recurrence-related DFS is distinct from TCGA-CDR PFI."
    ),

    "Figure 3": (
        "Descriptive Kaplan-Meier recurrence-free curves for GSE50081 "
        "stratified into outcome-blind, sample-balanced frozen-risk "
        "tertiles from clinical Cox (A) or ReCAP-Cox (B). Numbers at "
        "risk shown at prespecified follow-up horizons. Plots do not "
        "establish probability calibration or clinical utility. "
        "Follow-up is interpreted as years based on published information."
    ),

    "Figure 4": (
        "Development fold distributions and means for clinical Cox, "
        "gene-rank SVD8, the ReCAP-Cox representation, and shuffled "
        "pathway-map controls (A). Fold-matched contrasts of ReCAP-Cox "
        "against gene SVD8 and shuffled mappings (B). Shuffled controls "
        "provide a biological-prior specificity check; no Reactome-specific "
        "advantage was established."
    ),

    "Figure 5": (
        "Descriptive frozen external subgroup discrimination differences "
        "by histology and stage (A) and Uno IPCW C-index at three and five "
        "years (B). Small subgroup event counts, especially LUSC "
        "(11 events), limit interpretation."
    ),

    "Figure S1": (
        "Outcome-blind GPL570 to TCGA RNA-seq representation shift: "
        "per-component external mean displacement relative to "
        "mask-matched training SD (A) and external/training SD ratios (B)."
    )
}

(
    OUT / "Figure_Captions.txt"
).write_text(
    "\n\n".join(
        f"{k}. {v}"
        for k, v in captions.items()
    ),
    encoding="utf-8"
)


# ==========================================================
# ARTIFACT VALIDATION AND PROVENANCE
# ==========================================================

manifest = {
    "version": "recap_phase_h_final_v1",
    "main_figures": 5,
    "supplementary_figures": 1,
    "word_tables": 2,
    "font": "Times New Roman",
    "titles_inside_figures": False,
    "external_Cindex_rescored": False,
    "frozen_model_refitted": False,
    "external_KM_descriptive_only": True,
    "source_sha256": {
        name: digest(path)
        for name, path in FILES.items()
    }
}

(
    OUT / "artifact_manifest.json"
).write_text(
    json.dumps(
        manifest,
        indent=2
    ),
    encoding="utf-8"
)

for stem in (
    "Figure_1_recap_method",
    "Figure_2_locked_performance",
    "Figure_3_external_km",
    "Figure_4_representation_comparison",
    "Figure_5_external_robustness",
    "Figure_S1_transfer_shift"
):
    for extension in ("png", "svg", "pdf"):
        p = OUT / f"{stem}.{extension}"

        if (
            not p.is_file()
            or p.stat().st_size < 1000
        ):
            raise RuntimeError(
                f"Figure export failed: {p}"
            )


print("=" * 73)
print("PHASE H FINAL - PUBLICATION FIGURES AND TABLES COMPLETED")
print("=" * 73)

print("5 main figures + 1 supplementary figure | PDF, SVG, 600-dpi PNG")
print("2 Word tables | Times New Roman | white fill | full black grid")
print("Re-scored external C-index: NO | Refitted survival model: NO")

print("Saved:", OUT)

for p in sorted(OUT.iterdir()):
    print(" ", p.name)

No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
'created' timestamp seems very low; regarding as unix timestamp
'modified' timestamp seems very low; regarding as unix timestamp
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping for '\\__hatwide__'.
No TeX to Unicode mapping

PHASE H FINAL - PUBLICATION FIGURES AND TABLES COMPLETED
5 main figures + 1 supplementary figure | PDF, SVG, 600-dpi PNG
2 Word tables | Times New Roman | white fill | full black grid
Re-scored external C-index: NO | Refitted survival model: NO
Saved: D:\NST_NSCLC_PPI_GAT\results\manuscript_assets\recap_final_figures_tables
  artifact_manifest.json
  Figure_1_recap_method.pdf
  Figure_1_recap_method.png
  Figure_1_recap_method.svg
  Figure_2_locked_performance.pdf
  Figure_2_locked_performance.png
  Figure_2_locked_performance.svg
  Figure_3_external_km.pdf
  Figure_3_external_km.png
  Figure_3_external_km.svg
  Figure_4_representation_comparison.pdf
  Figure_4_representation_comparison.png
  Figure_4_representation_comparison.svg
  Figure_5_external_robustness.pdf
  Figure_5_external_robustness.png
  Figure_5_external_robustness.svg
  Figure_Captions.txt
  Figure_S1_transfer_shift.pdf
  Figure_S1_transfer_shift.png
  Figure_S1_transfer_shift.svg
  Main_Tables_1_to_2.docx
